# Single-pass row mean and variance (Welford)

Normalization layers need the mean and variance of every row. The textbook shortcut $\mathrm{Var} = \mathbb{E}[x^2] - \mathbb{E}[x]^2$ reads the row once but cancels catastrophically in float32 when the mean is large compared to the spread. Compute both statistics in **one pass** over the row, stably, by merging per-chunk statistics.

Write `row_mean_var(x)`.

## Inputs
- `x`: contiguous `float32` CUDA tensor of shape `(M, N)`, $M \ge 1$, $N \ge 1$. `N` can be larger than the block size.

## Output
- A tuple `(mean, var)` of `float32` CUDA tensors of shape `(M,)`: the row means and the **population** variances (divide by $N$).

## Algorithm
One program per row loops over the row in chunks of `BLOCK` columns, reading each element exactly once. For every chunk compute its count $n_b$, mean $\bar x_b$ and centred sum of squares $M_{2,b} = \sum (x - \bar x_b)^2$ (masked lanes excluded), then merge into the running $(n, \bar x, M_2)$:

$$\delta = \bar x_b - \bar x, \quad n' = n + n_b, \quad \bar x' = \bar x + \delta\,\frac{n_b}{n'}, \quad M_2' = M_2 + M_{2,b} + \delta^2\,\frac{n\,n_b}{n'}$$

At the end, `var = M2 / N`.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def row_mean_var(x):
    """x: (M, N) float32 CUDA tensor. Return (mean (M,), population var (M,)) in one pass per row."""
    raise NotImplementedError


@triton.jit
def _row_mean_var_kernel(x_ptr, mean_ptr, var_ptr, N, stride_row, BLOCK: tl.constexpr):
    # TODO: one program per row; loop over chunks and merge (count, mean, M2)
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0, 4.0], [2.0, 2.0, 2.0, 2.0]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [2.5, 2.0], "dtype": "float32"}, {"__nd__": [1.25, 0.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 10.0], [-1.0, 1.0], [5.0, 7.0]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [5.0, 0.0, 6.0], "dtype": "float32"}, {"__nd__": [25.0, 1.0, 1.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[3.5]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [3.5], "dtype": "float32"}, {"__nd__": [0.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[10000.3583984375, 10001.5107421875, 9998.2138671875, 10001.6865234375, 9999.953125, 9999.2001953125, 9999.197265625, 9998.9169921875, 9999.7763671875, 10000.833984375, 10000.583984375, 10000.638671875, 9998.3056640625, 9998.4287109375, 10001.5537109375, 10000.96875, 10002.18359375, 10001.2099609375, 9998.9755859375, 10001.28515625, 10000.6279296875, 10000.21484375, 9999.1796875, 10000.0029296875, 9999.8525390625, 10000.892578125, 10000.1240234375, 10001.5849609375, 9999.2255859375, 10000.470703125, 10000.4990234375, 9999.1357421875, 9998.818359375, 10001.013671875, 10000.517578125, 9999.837890625, 10000.39453125, 9999.32421875, 10000.23828125, 9999.39453125, 10000.1884765625, 10000.986328125, 9999.66015625, 10000.849609375, 9998.9658203125, 9999.7119140625, 9998.6357421875, 9999.8994140625, 10000.0791015625, 9999.802734375, 9999.732421875, 10001.33984375, 10001.423828125, 9998.7841796875, 10001.173828125, 9998.27734375, 9999.265625, 9999.826171875, 10000.111328125, 9998.9521484375, 9999.27734375, 10000.076171875, 9999.306640625, 9999.3583984375, 10000.666015625, 10000.931640625, 9999.9404296875, 10000.0185546875, 9999.98046875, 9999.5458984375, 9999.48828125, 9998.9619140625, 10000.6484375, 9999.916015625, 9999.8623046875, 10000.2001953125, 10000.5302734375, 10001.4208984375, 9999.7265625, 9999.5986328125, 10000.2001953125, 9999.154296875, 10001.2724609375, 10000.92578125, 10000.6943359375, 9999.880859375, 10000.7392578125, 10001.0498046875, 10000.5, 9999.7197265625, 9998.314453125, 10000.947265625, 9999.646484375, 9999.1572265625, 10000.890625, 10000.337890625, 9998.5810546875, 9999.9013671875, 9999.73046875, 10000.9921875, 9998.1044921875, 9999.6142578125, 9999.861328125, 10001.4189453125, 9998.9169921875, 9997.0595703125, 9998.6162109375, 10000.2626953125, 10000.8095703125, 10001.7333984375, 10000.4013671875, 9999.197265625, 10000.884765625, 9999.9892578125, 9999.798828125, 10000.017578125, 10001.2744140625, 9999.0986328125, 9998.9755859375, 9999.93359375, 9999.93359375, 9999.185546875, 9999.1201171875, 10000.3779296875, 10001.337890625, 10000.712890625, 10000.685546875, 10001.24609375, 9998.0087890625, 9998.4892578125, 9999.294921875, 9998.8046875, 9999.0107421875, 10000.0703125, 9998.7158203125, 9999.556640625, 9997.95703125, 9999.0263671875, 10001.345703125, 9999.45703125, 10000.32421875, 10000.7080078125, 9999.5087890625, 10000.5791015625, 10000.21484375, 10000.697265625, 9999.3642578125, 9998.5595703125, 9998.9775390625, 9999.4453125, 10000.5361328125, 9999.7958984375, 9999.6015625, 9999.8203125, 10001.099609375, 9999.4248046875, 10001.51171875, 9998.7724609375, 9999.408203125, 10000.85546875, 10000.517578125, 9999.14453125, 9999.890625, 9999.291015625, 10000.8505859375, 10000.5400390625, 9999.287109375, 10000.3974609375, 9999.6328125, 9999.927734375, 10000.064453125, 9999.435546875, 10000.4248046875, 10000.4599609375, 10000.42578125, 10001.1044921875, 9999.728515625, 9998.517578125, 9999.5107421875, 10000.201171875, 10000.0224609375, 10000.5791015625, 10000.806640625, 9999.8828125, 9999.8896484375, 9999.0126953125, 10001.033203125, 9999.736328125, 9998.9169921875, 9998.8857421875, 9999.5849609375, 10001.5556640625, 10000.7724609375, 9998.3056640625, 9999.8759765625, 10000.263671875, 10000.626953125, 10000.34375, 9999.359375, 10000.412109375, 9999.4462890625, 9997.759765625, 9999.291015625, 9999.314453125, 10000.9541015625, 10001.0556640625, 9998.955078125, 10000.173828125, 10000.08984375, 9998.564453125, 9997.947265625, 9999.9072265625, 10000.13671875, 9999.4052734375, 9997.2890625, 9999.970703125, 9999.4208984375, 9999.1103515625, 9999.67578125, 9998.87109375, 10000.5595703125, 10000.0107421875, 9999.4404296875, 10000.2822265625, 9999.9677734375, 9998.4814453125, 9999.85546875, 9999.54296875, 9998.259765625, 9999.34765625, 10000.4267578125, 9999.380859375, 10001.4072265625, 10001.6982421875, 9999.693359375, 10000.5537109375, 10001.9921875, 9999.6689453125, 9999.1005859375, 9999.578125, 9998.6259765625, 9998.650390625, 10000.208984375, 9999.5830078125, 10000.24609375, 10000.48828125, 9999.2353515625, 9999.8056640625, 9999.73046875, 9999.3310546875, 9999.1630859375, 10000.3154296875, 9999.859375, 10000.33203125, 10001.0009765625, 10001.0146484375, 9998.8876953125, 9999.14453125, 9999.744140625, 10000.1240234375, 10000.9384765625, 9999.0380859375, 9999.5634765625, 10000.2470703125, 10001.228515625, 10001.37890625, 9998.994140625, 9999.7880859375, 9999.1923828125, 10000.84765625, 10000.447265625, 9999.642578125, 9999.388671875, 10000.294921875, 10000.98046875, 9999.232421875, 9999.6767578125, 10000.11328125, 10000.017578125, 10000.337890625, 9999.0, 9999.4052734375, 9997.8994140625, 10000.0556640625, 10001.0703125, 9997.9423828125, 9999.3681640625, 10000.8984375, 9999.857421875, 9998.1044921875, 10001.1689453125, 9998.57421875, 10001.37109375, 9999.4150390625, 9999.29296875, 10000.02734375, 9999.9775390625, 10000.6005859375, 10000.5029296875, 9999.8525390625, 10001.322265625, 10000.7138671875, 10000.2734375, 10000.134765625, 9999.904296875, 9999.9228515625, 10000.6650390625, 10000.474609375, 10000.0400390625, 10001.0146484375, 9999.7373046875, 10000.435546875, 10000.859375, 9998.884765625, 9999.736328125, 10001.6689453125, 9999.04296875, 10001.15234375, 9998.9404296875, 10000.4052734375, 9998.935546875, 9999.3837890625, 9999.625, 9999.451171875, 10001.0087890625, 10000.8330078125, 10000.8701171875, 10000.193359375, 10001.63671875, 10001.0283203125, 10000.3251953125, 9998.951171875, 10000.685546875, 10000.65234375, 10000.0947265625, 10000.712890625, 10000.76953125, 9999.701171875, 10000.5654296875, 10000.3466796875, 10000.3134765625, 9999.658203125, 9999.537109375, 10000.1181640625, 10001.431640625, 10002.326171875, 9999.6298828125, 9998.9453125, 9999.658203125, 9999.1650390625, 9999.5830078125, 10000.396484375, 9999.0859375, 10000.609375, 10000.7919921875, 10001.5166015625, 10001.8193359375, 10000.998046875, 10002.267578125, 9999.5, 10000.2763671875, 10000.171875, 10000.439453125, 10001.0244140625, 9999.6767578125, 10001.712890625, 10001.2763671875, 10000.353515625, 10000.4716796875, 9999.0361328125, 9997.5244140625, 9999.9189453125, 10000.32421875, 10000.2353515625, 9998.91015625, 9999.5576171875, 10001.7587890625, 10000.736328125, 10000.6240234375, 9999.701171875, 10000.7470703125, 10001.2919921875, 10000.7666015625, 9999.6064453125, 10000.294921875, 9999.1357421875, 9999.7080078125, 10000.6015625, 9999.48046875, 9999.7822265625, 10001.4404296875, 9999.845703125, 10000.775390625, 9999.71484375, 9998.8828125, 10000.0185546875, 9999.65625, 9998.0166015625, 9998.8203125, 9999.5966796875, 10001.48046875, 9999.443359375, 9999.982421875, 10000.6396484375, 10000.3037109375, 9999.6787109375, 10001.703125, 10001.3447265625, 9997.59375, 10000.9677734375, 10000.5546875, 10000.9873046875, 10000.103515625, 10000.833984375, 10001.9267578125, 10001.712890625, 10001.7900390625, 10001.0595703125, 10000.9306640625, 10000.9501953125, 9998.9052734375, 10002.26953125, 10000.2509765625, 10000.337890625, 10000.134765625, 10000.4853515625, 9997.9111328125, 10000.244140625, 9998.7587890625, 9999.037109375, 9998.1708984375, 9998.310546875, 10000.595703125, 9999.5283203125, 9999.9111328125, 9999.8427734375, 9998.8603515625, 9999.30859375, 9998.7607421875, 9999.7021484375, 9997.8056640625, 9999.6181640625, 10000.9130859375, 10001.2060546875, 10001.1435546875, 10000.4990234375, 9999.0078125, 10000.90625, 10000.5205078125, 10000.2685546875, 9998.7998046875, 10000.3916015625, 10001.40625, 10000.740234375, 9999.775390625, 10000.544921875, 10000.005859375, 10000.1796875, 10000.8427734375, 10000.34375, 10000.07421875, 10000.3837890625, 9999.1875, 9999.01171875, 10000.4833984375, 9999.412109375, 10000.556640625, 9998.171875, 10001.19140625, 9999.9013671875, 10001.65625, 9997.541015625, 9998.330078125, 10001.396484375, 10000.158203125, 10000.3046875, 10000.126953125, 10000.1630859375, 10000.5380859375, 9999.0009765625, 9999.8388671875, 10001.296875, 10000.1845703125, 9999.388671875, 10000.1005859375, 10001.0439453125, 9999.8076171875, 9999.8291015625, 10000.494140625, 10000.8515625, 10001.4736328125, 9999.8779296875, 10000.353515625, 10001.0458984375, 9998.1171875, 10000.15234375, 10000.4697265625, 9999.927734375, 9999.1640625, 10000.5947265625, 10001.2490234375, 9999.92578125, 9999.53515625, 9999.8017578125, 10001.564453125, 10001.2548828125, 10000.0615234375, 9998.9287109375, 9998.76171875, 9999.7001953125, 9999.70703125, 10000.287109375, 9998.8974609375, 9999.8984375, 10000.5537109375, 9998.3447265625, 10002.310546875, 10000.5986328125, 10000.55078125, 9998.7294921875, 9999.005859375, 10001.0419921875, 10000.9130859375, 9997.603515625, 10001.181640625, 10000.7568359375, 10000.6533203125, 9998.92578125, 10000.291015625, 9999.4296875, 10000.5947265625, 10001.87109375, 10000.67578125, 10001.0517578125, 9999.7275390625, 9999.01953125, 10000.0537109375, 10001.076171875, 10001.5078125, 9999.9833984375, 10000.2958984375, 9999.6455078125, 9998.3291015625, 10000.1298828125, 10000.3916015625, 9999.6318359375, 9999.873046875, 9998.248046875, 10000.1025390625, 10001.42578125, 10000.0234375, 10000.640625, 10000.4892578125, 9999.2900390625, 10000.255859375, 9999.2294921875, 10002.958984375, 10000.251953125, 9998.9599609375, 9998.5146484375, 10001.5361328125, 10000.66015625, 10000.4423828125, 9998.6875, 9999.484375, 9998.833984375, 10001.142578125, 9999.5224609375, 10001.96875, 10000.5634765625, 9999.724609375, 10001.3837890625, 9997.951171875, 9999.2138671875, 9999.64453125, 10001.1123046875, 10001.1201171875, 10000.400390625, 9998.2724609375, 9999.517578125, 9998.943359375, 10001.2919921875, 9999.6044921875, 10000.6328125, 10001.8984375, 9999.970703125, 10000.7431640625, 9999.662109375, 10000.091796875, 10000.087890625, 10000.0595703125, 10000.11328125, 9999.65234375, 10001.91015625, 9999.396484375, 10000.291015625, 9999.7470703125, 9997.7861328125, 10001.013671875, 10000.2734375, 9998.6689453125, 9999.3134765625, 9998.4287109375, 10000.32421875, 10001.5986328125, 10001.2548828125, 9998.744140625, 9998.314453125, 10000.5634765625, 10001.7177734375, 10000.083984375, 10001.0830078125, 9998.638671875, 9999.919921875, 10000.4716796875, 9999.6181640625, 9999.802734375, 10000.7119140625, 9999.4033203125, 9999.033203125, 10000.7724609375, 10000.2685546875, 9999.0283203125, 10000.45703125, 9999.4619140625, 10000.6630859375, 10000.013671875, 9999.7353515625, 9999.2490234375, 10000.2236328125, 9999.9755859375, 10001.9130859375, 9999.9765625, 9998.673828125, 10002.08984375, 9997.982421875, 10001.6708984375, 9999.1572265625, 10001.408203125, 9999.833984375, 10000.5615234375, 10000.0478515625, 9999.90625, 10000.537109375, 9999.3740234375, 10000.4931640625, 9998.9345703125, 9999.283203125, 9999.505859375, 9999.9140625, 10001.2275390625, 9999.17578125, 9999.1513671875, 9998.88671875, 9999.603515625, 10000.5712890625, 9999.2626953125, 9999.71484375, 9999.4189453125, 10000.091796875, 9997.9921875, 10000.6005859375, 10000.9306640625, 9999.1708984375, 9999.9365234375, 9999.0966796875, 10001.0078125, 10000.6376953125, 10002.9052734375, 9999.7998046875, 9999.1328125, 9999.529296875, 9998.7529296875, 10000.2197265625, 10000.3212890625, 10000.9365234375, 10000.41796875, 9999.6005859375, 10001.30859375, 10000.5966796875, 10000.8486328125, 9998.84765625, 9999.1845703125, 9999.37890625, 9999.6298828125, 9997.8369140625, 10001.2255859375, 9999.8154296875, 10000.875, 10000.8017578125, 10001.525390625, 10000.60546875, 10000.4111328125, 9999.296875, 10000.1630859375, 10001.796875, 9998.2822265625, 9999.3408203125, 10000.3232421875, 10001.6669921875, 9998.9560546875, 9999.60546875, 10000.2119140625, 10003.166015625, 9999.197265625, 9999.9091796875, 10000.8125, 9999.365234375, 10000.4580078125, 10000.4873046875, 9999.7158203125, 10000.4287109375, 10001.8017578125, 10000.20703125, 9999.6572265625, 9999.337890625, 10000.3173828125, 10001.125, 10000.2353515625, 10000.400390625, 10002.7392578125, 9999.328125, 9999.013671875, 10000.3662109375, 9999.8779296875, 10000.666015625, 10001.3310546875, 10000.451171875, 9999.8974609375, 10000.611328125, 10000.576171875, 10000.001953125, 9999.5185546875, 10000.7607421875, 9999.8916015625, 9998.9482421875, 10000.7724609375, 9999.5712890625, 9999.5517578125, 10000.73828125, 9999.3056640625, 9999.9111328125, 9999.10546875, 9998.560546875, 10000.3759765625, 10001.6376953125, 9999.208984375, 10001.2421875, 10001.1455078125, 10001.583984375, 9999.01171875, 9999.4287109375, 10001.7236328125, 10000.0302734375, 10000.0751953125, 10000.7919921875, 10000.6962890625, 9999.5400390625, 9999.876953125, 10000.7265625, 10000.359375, 9999.365234375, 10001.369140625, 10000.943359375, 9999.7783203125, 9999.546875, 10001.017578125, 9997.6904296875, 10000.2060546875, 9999.5087890625, 9999.427734375, 10000.9833984375, 10000.4091796875, 10003.15625, 9999.8935546875, 10000.4765625, 9998.716796875, 10001.12109375, 9999.5234375, 10000.3369140625, 10001.412109375, 10001.2998046875, 10000.6123046875, 10001.3994140625, 9999.6806640625, 9999.1943359375, 9998.6064453125, 10000.6787109375, 10002.083984375, 9999.7998046875, 10000.1328125, 9999.80859375, 10000.6064453125, 9998.30078125, 10000.9736328125, 10001.1982421875, 9999.4404296875, 9999.11328125, 9999.240234375, 10000.669921875, 10001.5615234375, 9999.125, 9998.5751953125, 10000.2177734375, 10000.380859375, 10000.8203125, 9999.4208984375, 10000.841796875, 10001.20703125, 9998.4736328125, 9999.8095703125, 10000.716796875, 10000.8681640625, 9999.548828125, 9998.158203125, 9999.3173828125, 9998.671875, 9999.568359375, 10001.115234375, 9999.455078125, 10000.1689453125, 9998.875, 10001.009765625, 9999.4541015625, 9999.0361328125, 10001.9423828125, 10000.412109375, 10000.7587890625, 9998.083984375, 10001.498046875, 10001.7900390625, 9999.9375, 9999.7734375, 9999.4912109375, 10001.146484375, 9998.50390625, 10000.3955078125, 10002.0634765625, 10000.2060546875, 10002.0390625, 10000.0927734375, 9999.7646484375, 10000.9794921875, 9999.994140625, 10000.1669921875, 9999.6640625, 9998.861328125, 9999.759765625, 10000.7724609375, 10001.1455078125, 9999.99609375, 10000.375, 9999.1396484375, 9999.8916015625, 9999.9130859375, 9999.931640625, 10000.1298828125, 10000.421875, 10001.2880859375, 10000.83984375, 10000.5849609375, 9998.8544921875, 9999.2666015625, 10000.119140625, 10000.173828125, 10000.8017578125, 9998.3193359375, 10000.251953125, 10000.1982421875, 9999.73828125, 9999.2861328125, 10000.30078125, 9999.9169921875, 10000.5458984375, 9998.076171875, 9999.17578125, 10001.5966796875, 10000.3046875, 9999.30859375, 9999.76171875, 9998.7900390625, 10001.4990234375, 10001.5068359375, 10000.962890625, 10001.283203125, 10000.552734375, 10001.3974609375, 10000.7197265625, 10001.4951171875, 10000.0693359375, 10000.0078125, 9999.537109375, 9999.6298828125, 9998.5302734375, 10001.4169921875, 10001.4912109375, 10000.8251953125, 10001.091796875, 10000.326171875, 10000.5869140625, 9998.7626953125, 10000.02734375, 9998.3056640625, 9999.4267578125, 9999.81640625, 9999.7890625, 9999.046875, 10001.373046875, 10001.0810546875, 9999.5771484375, 9998.880859375, 9999.61328125, 10001.0263671875, 9998.205078125, 10000.8916015625, 9999.9658203125, 9999.701171875, 10001.103515625, 10000.11328125, 9999.744140625, 10001.3212890625, 9998.51953125, 10001.6923828125, 9999.2568359375, 9999.267578125, 10001.6669921875, 10003.388671875, 10001.31640625, 9999.2666015625, 9997.9375, 10000.3369140625, 10000.1005859375, 9999.51953125, 9999.970703125, 9999.4267578125, 10000.931640625, 10000.01171875, 9999.8896484375, 9997.9189453125, 10000.345703125, 9998.654296875, 10000.822265625, 10000.0576171875, 9998.08203125, 9999.7578125, 9998.908203125, 10000.638671875, 10000.53125, 10001.1328125, 9999.3212890625, 10000.5732421875, 9998.7216796875, 10000.4052734375, 9999.8525390625, 10000.314453125, 10000.068359375, 10001.08984375, 9999.736328125, 9999.896484375, 10001.462890625, 10000.0087890625, 9999.5498046875, 9999.18359375, 10000.08984375, 10000.041015625, 10000.7119140625, 9999.9072265625, 10000.755859375, 9999.8134765625, 10001.654296875, 10000.0263671875, 10000.7470703125, 10000.45703125, 9998.771484375, 10001.09765625, 9999.658203125, 10000.9853515625, 9998.09375, 9998.2373046875, 9998.08984375, 9998.2880859375, 9999.6767578125, 10000.0263671875, 9999.087890625, 9999.7177734375, 9999.3798828125, 10000.7861328125, 10000.7578125, 10000.3505859375, 10000.6943359375, 10001.7509765625, 10000.6943359375, 9998.546875, 9999.8427734375, 9999.755859375, 10000.859375, 10000.5234375, 10000.9033203125, 9998.521484375, 10001.4306640625, 9999.900390625, 10000.9990234375, 10000.5224609375, 9999.1728515625, 9999.2333984375, 10000.63671875, 10001.1689453125, 9999.1376953125, 9998.90625, 9998.0, 9999.10546875, 9999.2880859375, 9999.73046875, 10000.6171875, 9999.53125, 9998.9111328125, 9999.634765625, 9999.9326171875, 10000.15234375, 10000.13671875, 9999.8291015625, 10000.544921875, 9999.7119140625, 9998.9677734375, 9999.8271484375, 10001.83203125, 9999.1884765625, 10002.4189453125, 9999.5, 10000.705078125, 9998.4091796875, 9999.8984375, 10001.4189453125, 9998.841796875, 9999.283203125, 10000.3115234375, 10000.3466796875, 9998.5380859375, 9998.7587890625, 10000.283203125, 10000.291015625, 10000.7236328125, 9999.8369140625, 10000.6181640625, 10001.1904296875, 9998.7978515625, 10001.0029296875, 10000.74609375, 10001.263671875, 9999.623046875, 9999.970703125, 10000.08203125, 9999.5322265625, 10000.9970703125, 10000.2275390625, 9999.0263671875, 10000.2607421875, 9998.958984375, 9998.4697265625, 9998.984375, 10000.0244140625, 10000.119140625, 9999.8984375, 10000.5361328125, 10001.8896484375, 9998.8994140625, 10000.423828125, 10000.275390625, 9998.8388671875, 9999.5126953125, 10000.2041015625, 10000.03515625, 9999.189453125, 10000.7548828125, 10000.23828125, 10000.6298828125, 10001.5576171875, 9998.763671875, 10000.96875, 10000.46875, 9999.15234375, 9999.947265625, 9999.59765625, 9998.861328125, 10000.7900390625, 9999.25, 9998.01953125, 10000.8037109375, 9999.52734375, 10000.3623046875, 10000.2021484375, 9999.7021484375, 9999.6298828125, 9997.53515625, 9999.6015625, 9998.892578125, 10000.4921875, 9999.240234375, 9999.36328125, 10000.0947265625, 9998.87890625, 9999.306640625, 9999.5244140625, 10000.3896484375, 9998.5859375, 10000.7421875, 9999.8583984375, 9999.5087890625, 10000.3310546875, 10000.9814453125, 10001.1171875, 9998.982421875, 10001.3037109375, 10000.03125, 10001.26953125, 9998.966796875, 9999.830078125, 9999.888671875, 9998.7607421875, 9999.5537109375, 10000.8740234375, 10001.0791015625, 10000.6337890625, 10000.267578125, 9999.5634765625, 10001.99609375, 9998.6845703125, 9998.8203125, 9999.2421875, 10000.162109375, 9999.54296875, 10000.740234375, 9998.462890625, 10000.796875, 10001.69921875, 10000.837890625, 9999.697265625, 9999.400390625, 10001.044921875, 9998.3310546875, 9999.212890625, 9998.3525390625, 9999.84375, 10001.7724609375, 9999.564453125, 10001.02734375, 9998.93359375, 9999.4482421875, 9999.4267578125, 10001.0068359375, 9999.130859375, 9999.3291015625, 9999.9892578125, 10001.4873046875, 10000.0419921875, 10001.7109375, 10000.369140625, 9999.7373046875, 10001.1005859375, 9999.4189453125, 9999.0966796875, 10000.0869140625, 9998.484375, 9998.8271484375, 9999.2001953125, 10000.017578125, 10000.9931640625, 10001.09765625, 9999.794921875, 10000.2109375, 9999.68359375, 9998.607421875, 10000.37890625, 9999.2734375, 10000.7138671875, 9999.45703125, 9999.00390625, 10000.3056640625, 9999.4501953125, 9998.708984375, 10000.98828125, 10000.2919921875, 10000.2880859375, 10000.05859375, 9999.4384765625, 9997.259765625, 9999.4365234375, 9999.552734375, 10000.3916015625, 9998.7109375, 10000.443359375, 10000.0126953125, 10001.4111328125, 9999.279296875, 9999.9970703125, 10000.2255859375, 10000.5576171875, 10001.2412109375, 10000.4501953125, 9998.6259765625, 10001.5966796875, 9998.05859375, 10000.6142578125, 10001.2734375, 9999.3291015625, 9999.94921875, 9999.923828125, 9998.91015625, 10000.357421875, 9999.0634765625, 10000.3125, 10000.5859375, 10001.0625, 10001.3212890625, 10000.4130859375, 10000.4775390625, 9999.787109375, 9999.939453125, 10000.86328125, 10000.76171875, 10000.462890625, 9999.9931640625, 10001.009765625, 9998.9072265625, 9999.4873046875, 10000.2197265625, 10001.1015625, 9998.80078125, 9998.96484375, 9998.8271484375, 10000.7109375, 10000.7822265625, 10001.498046875, 10002.216796875, 9998.8369140625, 9999.8232421875, 9999.4697265625, 9998.943359375, 10000.04296875, 9999.4697265625, 10000.1455078125, 9999.658203125, 9998.5947265625, 10000.5791015625, 10000.5517578125, 9999.5771484375, 9997.896484375, 9997.7451171875, 9998.6572265625, 10000.361328125, 10000.6455078125, 10000.06640625, 10000.064453125, 9997.525390625, 10000.34765625, 9999.32421875, 9999.7236328125, 9999.052734375, 9998.046875, 10000.728515625, 10000.3544921875, 9998.951171875, 9998.4814453125, 9999.3740234375, 10000.830078125, 10000.5126953125, 10000.0361328125, 10000.8720703125, 10000.3134765625, 9999.0595703125, 10000.61328125, 9999.9228515625, 9999.17578125, 10001.0888671875, 10000.3984375, 9998.5810546875, 10000.7275390625, 10000.17578125, 9999.6298828125, 9999.53515625, 9998.9853515625, 9998.560546875, 10000.2509765625, 10000.4404296875, 9999.65625, 10000.2314453125, 9999.615234375, 9999.638671875, 10000.4765625, 10000.45703125, 10001.3603515625, 9999.533203125, 10000.322265625, 9999.822265625, 10000.1962890625, 10001.1982421875, 10000.3720703125, 9999.23046875, 9999.5400390625, 9999.4130859375, 9999.7314453125, 9999.341796875, 9999.5048828125, 9999.8193359375, 10000.2978515625, 9999.162109375, 10000.048828125, 10000.0234375, 10000.796875, 10000.2275390625, 10000.4833984375, 10000.24609375, 10001.080078125, 9999.8603515625, 9998.546875, 10000.31640625, 9999.03515625, 9997.8427734375, 10000.431640625, 9999.9345703125, 10000.1552734375, 10000.404296875, 9999.7958984375, 10000.9951171875, 10000.0234375, 9999.6689453125, 9999.9873046875, 9998.01171875, 10000.4453125, 9998.5537109375, 10000.3330078125, 9999.828125, 10000.02734375, 9999.18359375, 10001.548828125, 9998.01171875, 9999.826171875, 9999.099609375, 9998.4384765625, 10001.3212890625, 10000.1328125, 10002.267578125, 10000.6953125, 10000.4306640625, 10000.49609375, 9998.7373046875, 9998.7099609375, 10001.322265625, 10000.736328125, 9999.2646484375, 10000.0712890625, 10000.125, 10001.1298828125, 10000.302734375, 9999.318359375, 10000.75, 9999.4736328125, 10000.5283203125, 9999.4326171875, 9997.0029296875, 10001.525390625, 9999.1044921875, 10000.9404296875, 10000.5927734375, 9999.4931640625, 9998.892578125, 10001.8251953125, 10000.2236328125, 10001.2265625, 10000.3271484375, 10000.9072265625, 10000.2294921875, 9998.7890625, 9999.27734375, 10000.376953125, 9998.8828125, 9999.689453125, 9999.9169921875, 10001.068359375, 10001.556640625, 9998.056640625, 10000.4150390625, 10001.7353515625, 9998.724609375, 9998.98828125, 10001.134765625, 10000.1826171875, 9999.4921875, 10000.21484375, 10000.7109375, 10000.1142578125, 10000.396484375, 10000.236328125, 10000.4208984375, 10000.2470703125, 10000.455078125, 9999.9091796875, 10001.12109375, 9999.783203125, 9999.59375, 9998.7099609375, 9999.37109375, 10000.2255859375, 9999.91015625, 10000.6484375, 9998.9716796875, 9999.4755859375, 9998.201171875, 9999.8193359375, 10000.986328125, 10000.517578125, 9999.6904296875, 10000.525390625, 10000.328125, 9999.21875, 9999.1025390625, 9999.1845703125, 10000.4833984375, 10000.609375, 10000.4365234375, 9999.8095703125, 10001.8525390625, 9999.865234375, 10000.5224609375, 10000.23046875, 10001.0458984375, 9998.46484375, 10000.5517578125, 9999.876953125, 9999.9736328125, 9999.0654296875, 9999.3662109375, 10001.2080078125, 10000.8828125, 9998.701171875, 10000.291015625, 9999.595703125, 10001.0751953125, 10000.484375, 10000.78125, 10000.0029296875, 10000.1015625, 9999.4404296875, 9998.6328125, 9999.517578125, 10001.2197265625, 10001.328125, 9999.22265625, 9999.2392578125, 9999.337890625, 9999.2998046875, 9998.9248046875, 9999.380859375, 10000.6494140625, 10000.720703125, 10001.6318359375, 9999.8681640625, 9997.982421875, 9999.4814453125, 9999.0, 9999.68359375, 9999.30859375, 10000.9228515625, 9999.2080078125, 10000.005859375, 10001.703125, 9999.9873046875, 9998.4296875, 10000.8486328125, 10000.4404296875, 10000.513671875, 10000.6494140625, 9999.099609375, 9999.056640625, 10000.138671875, 9998.392578125, 9999.2021484375, 10001.0859375, 9998.7939453125, 10000.3251953125, 10000.99609375, 9999.9365234375, 9999.5703125, 10001.2314453125, 9998.697265625, 10001.5654296875, 9999.146484375, 10000.021484375, 9998.953125, 9998.9736328125, 10000.9267578125, 10000.060546875, 10000.482421875, 9998.76171875, 9999.654296875, 9998.8193359375, 9999.96484375, 10000.42578125, 10000.4638671875, 10000.478515625, 9999.37109375, 10000.5341796875, 9999.578125, 10001.2705078125, 9999.830078125, 10001.1220703125, 9999.65234375, 10000.6181640625, 10000.208984375, 9998.603515625, 10000.6259765625, 9999.125, 9998.298828125, 10000.125, 10001.0634765625, 10000.6904296875, 9999.1904296875, 9998.3701171875, 9997.44921875, 9999.3798828125, 9999.4931640625, 9999.833984375, 10001.16015625, 9999.7998046875, 9998.9609375, 9998.92578125, 9999.0810546875, 9999.349609375, 10000.2939453125, 9999.6845703125, 9999.9150390625, 10000.083984375, 10000.185546875, 10000.6044921875, 10001.1005859375, 9999.044921875, 9999.7109375, 9999.8798828125, 9999.84375, 10000.7685546875, 10000.6806640625, 9999.39453125, 10001.5791015625, 10000.4892578125, 10000.6552734375, 9999.427734375, 10000.7490234375, 9999.6943359375, 9999.099609375, 10000.1171875, 9999.033203125, 9998.9697265625, 10000.4296875, 10000.755859375, 9999.3505859375, 10000.9111328125, 10001.1142578125, 10001.0947265625, 10000.328125, 9999.716796875, 10001.46875, 9999.6474609375, 10000.4599609375, 10001.0771484375, 10001.6396484375, 10000.3056640625, 10000.2763671875, 10000.529296875, 10000.240234375, 10001.0908203125, 10001.05078125, 10001.67578125, 9999.9931640625, 10000.7490234375, 9999.3740234375, 9999.6630859375, 10001.4033203125, 9999.7978515625, 9998.2568359375, 9999.5380859375, 9999.5537109375, 9998.3173828125, 10000.2353515625, 10000.8447265625, 9999.5576171875, 10000.9716796875, 9998.0888671875, 10000.7568359375, 10000.03515625, 10002.18359375, 10000.0673828125, 10000.0224609375, 10000.0234375, 9999.720703125, 9999.6884765625, 10000.703125, 10000.3056640625, 10000.33203125, 10000.3115234375, 9999.40625, 10001.8369140625, 9999.10546875, 10000.931640625, 9999.892578125, 10001.0361328125, 9999.8427734375, 10001.0859375, 10000.525390625, 10001.166015625, 10001.9345703125, 10000.2666015625, 10000.212890625, 9997.8125, 10000.087890625, 10000.912109375, 9999.8349609375, 10000.130859375, 9999.919921875, 9998.2587890625, 10000.8818359375, 9999.3984375, 9998.7978515625, 10000.9951171875, 10000.4462890625, 10000.6064453125, 10000.1962890625, 9999.4599609375, 10000.9150390625, 9999.2666015625, 10000.318359375, 9999.1435546875, 9999.046875, 10000.8447265625, 9999.353515625, 10001.5791015625, 9998.8671875, 10001.0830078125, 9999.8076171875, 10000.8095703125, 10000.7412109375, 9998.384765625, 10000.658203125, 9999.7939453125, 9999.2099609375, 10001.0654296875, 10000.43359375, 10000.4765625, 10001.3994140625, 10001.0380859375, 10000.1806640625, 9999.3349609375, 10001.462890625, 9999.486328125, 9998.5947265625, 10001.8857421875, 10002.0439453125, 9998.5703125, 10000.447265625, 9998.7705078125, 10000.6923828125, 9999.673828125, 9998.7919921875, 10000.98828125, 9999.283203125, 10000.470703125, 9999.4716796875, 10001.6953125, 10001.1064453125, 9999.6796875, 9998.0888671875, 10000.4423828125, 9999.21875, 10001.2060546875, 10001.568359375, 10000.107421875, 10000.94140625, 9998.865234375, 10000.6357421875, 10001.6279296875, 9998.8017578125, 10000.775390625, 10000.3115234375, 10000.892578125, 10000.1337890625, 10000.0029296875, 10001.1904296875, 10000.099609375, 10000.51171875, 9998.7890625, 10000.431640625, 10000.7080078125, 9999.92578125, 10000.0166015625, 9998.193359375, 9999.08203125, 9999.806640625, 9999.76171875, 10002.0908203125, 10000.7822265625, 9998.9404296875, 10002.185546875, 9998.744140625, 9998.9716796875, 9999.2275390625, 9999.9990234375, 10002.474609375, 9999.8076171875, 10000.5693359375, 9998.6796875, 9999.6357421875, 10002.4677734375, 10001.287109375, 10000.9228515625, 10001.49609375, 10000.6005859375, 10000.5693359375, 9999.0478515625, 9998.64453125, 9999.4345703125, 9999.7509765625, 9999.302734375, 10000.7412109375, 10001.7958984375, 10000.0537109375, 9998.583984375, 9998.8916015625, 10001.0595703125, 10000.8203125, 10000.2880859375, 10000.6796875, 10001.1455078125, 10000.1123046875, 10000.3623046875, 9999.7412109375, 9998.1875, 10000.16796875, 10000.029296875, 10000.4404296875, 9999.5361328125, 10000.4482421875, 10000.4228515625, 9999.6513671875, 9999.0927734375, 9999.390625, 10000.373046875, 10000.8779296875, 10001.7890625, 9999.9580078125, 10000.529296875, 10001.203125, 9999.0908203125, 10000.650390625, 9999.609375, 9999.6884765625, 10000.7822265625, 9999.8232421875, 10000.37109375, 9998.9853515625, 9999.9873046875, 9999.208984375, 9999.75390625, 10002.79296875, 10000.072265625, 9999.8427734375, 9999.6669921875, 9999.306640625, 10001.7958984375, 9999.3828125, 10002.3701171875, 9999.8408203125, 10001.2529296875, 10000.1689453125, 9999.4345703125, 9999.873046875, 10001.3427734375, 9999.8896484375, 9998.1708984375, 10001.884765625, 9999.369140625, 9999.732421875, 10001.484375, 9998.412109375, 9999.5576171875, 10000.6728515625, 10000.6689453125, 10000.6142578125, 9999.11328125, 10000.96484375, 9999.6103515625, 10000.234375, 9999.14453125, 10001.755859375, 9999.6884765625, 9999.9833984375, 9999.51953125, 10000.4443359375, 9998.7412109375, 9999.9638671875, 9999.837890625, 10001.66015625, 9999.7353515625, 9999.8798828125, 10000.2080078125, 10001.0859375, 10002.0732421875, 10000.51171875, 10000.5859375, 10001.689453125, 10000.1484375, 10000.5107421875, 9999.498046875, 9999.9423828125, 9999.318359375, 10002.2607421875, 10000.8701171875, 10000.7802734375, 9999.0908203125, 10000.203125, 9999.40234375, 9999.3974609375, 9998.7353515625, 9999.435546875, 10002.7099609375, 9999.4814453125, 10000.7685546875, 10001.041015625, 10000.5546875, 10000.84765625, 9999.0966796875, 10001.6337890625, 10002.189453125, 10000.130859375, 9999.388671875, 9999.2587890625, 9998.7587890625, 10000.693359375, 10000.9267578125, 10001.7822265625, 9998.7841796875, 10000.93359375, 10000.7705078125, 9999.8212890625, 9999.2041015625, 10000.3720703125, 10002.287109375, 9998.80078125, 9999.4814453125, 9999.544921875, 9999.9267578125, 10001.1640625, 9999.5908203125, 9997.7919921875, 9998.80078125, 9998.783203125, 10000.3173828125, 10000.2138671875, 9998.912109375, 10000.7783203125, 9998.2255859375, 9999.86328125, 10001.3564453125, 9998.4765625, 10000.6484375, 10001.0810546875, 9999.587890625, 10000.2060546875, 9998.55859375, 10000.3818359375, 9998.83984375, 9999.4814453125, 10000.462890625, 9999.6591796875, 9998.935546875, 9999.994140625, 9999.7578125, 9999.220703125, 9998.04296875, 9999.818359375, 10000.9013671875, 9999.400390625, 9999.4599609375, 9999.5361328125, 10000.419921875, 9998.5009765625, 10000.76953125, 10002.0615234375, 10000.4873046875, 10000.0751953125, 10000.455078125, 10000.498046875, 9999.677734375, 9999.888671875, 9999.8125, 10000.5458984375, 10001.892578125, 10000.1552734375, 10002.0419921875, 9999.1552734375, 10000.7705078125, 9999.3125, 10000.15234375, 9999.93359375, 9998.13671875, 9999.783203125, 10000.4248046875, 10000.326171875, 9998.5576171875, 9998.591796875, 10000.63671875, 9999.0146484375, 10002.068359375, 10000.3359375, 10000.359375, 9999.8515625, 9999.2451171875, 10000.2998046875, 9999.8056640625, 10001.5302734375, 10000.19140625, 10000.27734375, 10001.0947265625, 10001.01171875, 9998.009765625, 10001.9970703125, 9998.509765625, 9999.0205078125, 9999.736328125, 9999.2373046875, 10001.84765625, 10000.0673828125, 9998.9033203125, 9999.6572265625, 9999.23828125, 9999.353515625, 10000.423828125, 10000.5732421875, 9999.8779296875, 9998.7177734375, 9999.4013671875, 10000.9580078125, 10000.568359375, 9999.7216796875, 10000.7939453125, 9998.173828125, 9999.3447265625, 10000.072265625, 10000.34375, 10001.0908203125, 10000.1220703125, 10000.3369140625, 10000.1943359375, 9998.9765625, 9999.8525390625, 10001.466796875, 10000.7587890625, 9999.732421875, 10000.0556640625, 10001.765625, 10001.625, 10001.919921875, 10001.337890625, 9999.912109375, 9999.8583984375, 9998.4423828125, 10001.333984375, 10001.28125, 10002.2841796875, 9999.2900390625, 10001.7861328125, 10000.619140625, 10000.2109375, 10000.34375, 9998.9365234375, 10000.01171875, 9999.095703125, 9999.6025390625, 10000.3017578125, 9999.6123046875, 10000.7998046875, 9999.3935546875, 9999.54296875, 9999.51953125, 10000.2978515625, 10000.3515625, 10001.095703125, 9999.791015625, 9999.439453125, 10000.3935546875, 10001.8994140625, 10000.9931640625, 9998.244140625, 10000.2265625, 9999.7216796875, 9999.927734375, 10000.9501953125, 10000.177734375, 10000.0283203125, 9998.533203125, 10000.01953125, 9999.3837890625, 10000.3701171875, 10001.1982421875, 9999.255859375, 9999.8994140625, 10000.732421875, 9998.5, 9999.421875, 9998.6767578125, 9999.50390625, 10000.0185546875, 9999.7763671875, 10001.5400390625, 10000.90234375, 10000.306640625, 10001.23828125, 10002.0302734375, 9999.75390625, 9999.951171875, 10000.466796875, 10001.1845703125, 9998.12109375, 10000.0693359375, 10002.80859375, 10000.169921875, 10002.0634765625, 10002.6162109375, 10000.9228515625, 10002.45703125, 10000.2021484375, 9998.4033203125, 9999.6796875, 10001.6357421875, 10002.8515625, 10000.76171875, 9998.5390625, 10000.171875, 9998.162109375, 10000.25390625, 10001.80078125, 9999.2421875, 9999.2822265625, 9999.6923828125, 9999.8125, 10000.759765625, 10000.51953125, 10000.92578125, 9999.154296875, 10001.37890625, 10000.828125, 9999.28125, 9999.634765625, 9999.513671875, 9998.4267578125, 10002.123046875, 10000.80078125, 10000.234375, 10001.330078125, 10000.0078125, 10000.009765625, 10001.4912109375, 9999.1279296875, 9999.279296875, 9998.4560546875, 9999.4580078125, 10001.208984375, 9999.6240234375, 10001.703125, 9999.6162109375, 10000.3681640625, 9999.056640625, 10000.8759765625, 10002.1953125, 9999.8427734375, 9998.5224609375, 10000.041015625, 10000.8642578125, 9999.2900390625, 10002.099609375, 9998.82421875, 10000.7197265625, 10001.5849609375, 10000.431640625, 10001.435546875, 10000.6298828125, 9999.7080078125, 9999.6484375, 10001.990234375, 9998.3251953125, 10000.3642578125, 9999.302734375, 9999.6611328125, 10000.974609375, 9999.2216796875, 9998.515625, 10001.3251953125, 10001.38671875, 10000.1337890625, 9999.17578125, 9999.0654296875, 10000.41015625, 10000.51953125, 10000.5732421875, 10000.7490234375, 10000.05859375, 10000.689453125, 10000.029296875, 10000.4404296875, 9999.828125, 10000.359375, 9999.10546875, 9999.833984375, 9998.8828125, 10000.62890625, 9999.4423828125, 9999.3515625, 9999.3857421875, 9999.1748046875, 9999.0576171875, 10000.6279296875, 10000.5498046875, 10000.9501953125, 9999.4189453125, 10000.294921875, 9998.041015625, 10000.931640625, 9998.9990234375, 10001.318359375, 10000.7451171875, 10002.166015625, 10000.556640625, 9999.3759765625, 10002.2578125, 10001.369140625, 10001.2255859375, 10001.8740234375, 10000.0322265625, 9997.7021484375, 9999.861328125, 9999.486328125, 10000.7255859375, 9999.677734375, 10000.6591796875, 9999.21484375, 10001.23046875, 10000.720703125, 9999.359375, 10001.4794921875, 10001.6494140625, 10000.7060546875, 9999.4931640625, 10000.453125, 10000.1982421875, 9999.6416015625, 10000.0146484375, 9999.9345703125, 9998.65625, 10000.025390625, 9998.9892578125, 10000.041015625, 9999.8642578125, 10000.9267578125, 9999.572265625, 10000.21484375, 9998.5859375, 10001.2509765625, 10002.26953125, 9999.951171875, 10000.6025390625, 10000.77734375, 9999.91015625, 10000.99609375, 9999.5380859375, 9999.9658203125, 10000.0048828125, 9999.4609375, 9998.65625, 10001.09765625, 10000.1923828125, 9999.3056640625, 9998.90234375, 9999.9296875, 9999.0654296875, 10000.6025390625, 10000.1044921875, 10000.484375, 10001.1240234375, 10000.2626953125, 10000.68359375, 10001.2548828125, 10000.5830078125, 9997.763671875, 10001.2685546875, 10000.326171875, 10000.6484375, 10001.6572265625, 9999.8642578125, 10000.560546875, 9997.9326171875, 10001.595703125, 10000.0048828125, 9999.8154296875, 9999.80859375, 10000.662109375, 9998.1962890625, 10001.96484375, 10000.1875, 9999.96484375, 10001.4873046875, 10000.9619140625, 10000.58984375, 9999.1484375, 10001.1875, 10000.333984375, 9999.2314453125, 9999.888671875, 10000.3564453125, 9999.310546875, 10000.9296875, 9999.501953125, 10000.962890625, 10000.3603515625, 10000.5400390625, 9999.052734375, 10000.1552734375, 10000.68359375, 10000.404296875, 9999.689453125, 10000.416015625, 9999.7158203125, 10000.0126953125, 10001.2744140625, 10001.74609375, 10001.853515625, 10000.66015625, 9999.4912109375, 9999.7958984375, 10001.1875, 10000.197265625, 10001.6259765625, 9999.72265625, 10000.9755859375, 9998.7548828125, 9999.75, 10000.58984375, 10000.4794921875, 10000.234375, 9998.9384765625, 10000.177734375, 9999.76171875, 9999.400390625, 10000.2861328125, 9999.2294921875, 9999.41015625, 10002.2529296875, 10000.423828125, 10001.3056640625, 9999.5400390625, 9999.5927734375, 9998.7236328125, 9998.71484375, 9998.765625, 10000.5244140625, 9999.0986328125, 9999.1669921875, 9999.42578125, 10000.7548828125, 9999.634765625, 10000.640625, 9998.9873046875, 10000.625, 10000.00390625, 10000.626953125, 9998.87890625, 9999.8662109375, 10000.0078125, 9999.3173828125, 10000.6015625, 10000.0751953125, 10000.5498046875, 9999.42578125, 10000.2255859375, 9998.6064453125, 10000.7041015625, 9998.6630859375, 9999.330078125, 10000.09765625, 10000.5498046875, 10000.1298828125, 9997.98046875, 9998.73828125, 9999.16015625, 9999.869140625, 9998.615234375, 9999.123046875, 10000.3427734375, 10000.0537109375, 10000.4658203125, 10000.3291015625, 9998.486328125, 9998.912109375, 9999.9970703125, 10000.26953125, 9999.3486328125, 10000.8642578125, 9999.21484375, 9999.3779296875, 9999.9052734375, 10000.171875, 9999.29296875, 9999.8193359375, 9998.93359375, 9999.169921875, 10001.435546875, 9998.7607421875, 10001.431640625, 9999.3193359375, 10001.158203125, 9999.4482421875, 9998.8466796875, 9999.82421875, 9999.8955078125, 10000.1591796875, 9998.908203125, 9998.423828125, 9999.68359375, 9999.4384765625, 9999.126953125, 9999.8193359375, 10001.27734375, 10001.1630859375, 10000.25, 10001.291015625, 10001.916015625, 10001.4423828125, 9998.23828125, 9999.2236328125, 10000.279296875, 10000.9482421875, 9998.728515625, 10002.2041015625, 9999.3232421875, 9999.0078125, 9997.8857421875, 10000.9150390625, 10001.0107421875, 10000.927734375, 10000.23828125, 10000.37109375, 10000.119140625, 10001.8388671875, 10000.75, 10001.09375, 9998.333984375, 9998.3544921875, 10001.177734375, 10000.6376953125, 10000.544921875, 9999.9248046875, 10000.3193359375, 9999.560546875, 10000.666015625, 9998.326171875, 10000.11328125, 9999.3720703125, 9999.1357421875, 9999.78125, 9999.8076171875, 9999.8837890625, 10001.6962890625, 9998.763671875, 9999.0341796875, 10000.021484375, 9999.9912109375, 10000.3974609375, 10000.9423828125, 10000.3525390625, 9999.1123046875, 10002.3671875, 10000.203125, 10000.0458984375, 10000.0791015625, 9999.1416015625, 9999.23046875, 10000.0205078125, 10000.271484375, 10000.5439453125, 9998.9462890625, 10000.7138671875, 9998.89453125, 10001.0859375, 10002.6123046875, 9998.6962890625, 10000.5048828125, 10001.056640625, 9999.509765625, 10001.484375, 9998.4541015625, 9999.3544921875, 10000.5888671875, 10001.5810546875, 9999.4794921875, 9998.513671875, 10000.54296875, 10001.69921875, 10000.5439453125, 9999.2275390625, 9998.5419921875, 10002.857421875, 10000.2265625, 9998.2626953125, 9998.1259765625, 10001.306640625, 10001.369140625, 9999.91015625, 9999.5234375, 10000.798828125, 10000.89453125, 10000.2587890625, 9998.68359375, 10000.3271484375, 10002.5576171875, 9998.888671875, 9999.205078125, 9999.16796875, 9999.150390625, 10001.6220703125, 10000.7744140625, 9998.9248046875, 9998.783203125, 9999.9072265625, 9999.2431640625, 10000.359375, 10000.220703125, 9998.3681640625, 10001.994140625, 9999.154296875, 10001.509765625, 10000.1669921875, 9999.6826171875, 9997.751953125, 9999.30078125, 10001.1640625, 10000.876953125, 9999.0615234375, 9997.2333984375, 10000.25390625, 9998.62890625, 9999.7470703125, 9998.841796875, 10001.3203125, 9999.3681640625, 10000.1142578125, 9999.79296875, 9999.2939453125, 10000.7666015625, 10000.3720703125, 10000.3330078125, 9999.576171875, 10000.5048828125, 9998.8291015625, 10000.27734375, 9999.7275390625, 9996.814453125, 9998.5283203125, 9999.5439453125, 10002.2060546875, 10001.326171875, 9999.3525390625, 10001.1865234375, 9998.97265625, 10001.2099609375, 9999.5966796875, 9999.0458984375, 9998.3037109375, 9999.7958984375, 10001.0, 9999.921875, 10003.5009765625, 9999.3115234375, 10001.2392578125, 10000.97265625, 10000.12109375, 9999.7568359375, 9999.26953125, 10000.748046875, 10000.8291015625, 9999.76171875, 9999.224609375, 9998.619140625, 9999.87109375, 9998.1591796875, 9999.1669921875, 10002.017578125, 9998.9921875, 9999.4560546875, 10000.8779296875, 10000.443359375, 9999.6748046875, 10000.416015625, 9999.8173828125, 10000.0029296875, 9999.0576171875, 10000.5126953125, 10000.169921875, 10000.0390625, 10000.3544921875, 10000.0556640625, 9999.5478515625, 10000.0283203125, 9999.365234375, 9997.50390625, 9999.0771484375, 9999.5361328125, 9999.9677734375, 10000.291015625, 10001.296875, 10001.4736328125, 10001.54296875, 9999.6884765625, 10001.015625, 9997.673828125, 9998.7392578125, 10001.62890625, 9999.7724609375, 9999.798828125, 10000.349609375, 10000.462890625, 9999.65234375, 9999.9072265625, 9998.4755859375, 10000.2958984375, 10000.85546875, 10000.876953125, 10000.140625, 9999.3779296875, 9999.5947265625, 9999.126953125, 9998.4931640625, 10000.423828125, 9999.259765625, 10002.1982421875, 10000.244140625, 10001.37109375, 9998.3896484375, 9999.80078125, 10000.84765625, 10000.7880859375, 10000.35546875, 10000.5185546875, 9999.173828125, 9999.083984375, 9999.4677734375, 9998.4990234375, 10000.9814453125, 9998.8427734375, 9998.919921875, 9999.74609375, 10001.6416015625, 10000.232421875, 10000.4970703125, 10000.078125, 10001.58203125, 10000.591796875, 9998.5, 9998.7412109375, 10000.154296875, 10000.2470703125, 9999.4267578125, 10000.3056640625, 10001.298828125, 9999.4736328125, 10001.3896484375, 10001.400390625, 9996.9365234375, 9999.3310546875, 10000.74609375, 9999.875, 10000.0126953125, 10000.896484375, 10000.791015625, 9999.2490234375, 9997.3837890625, 10000.5322265625, 10001.013671875, 10000.4599609375, 10000.3466796875, 10000.724609375, 10000.4677734375, 9999.29296875, 10000.22265625, 10000.5693359375, 9999.3388671875, 10000.0732421875, 10001.0986328125, 10001.9755859375, 10000.7080078125, 10002.0693359375, 9999.9228515625, 9999.80859375, 10001.3427734375, 9999.7958984375, 9999.56640625, 9998.830078125, 9999.94921875, 9999.677734375, 9998.65234375, 9998.64453125, 10000.34375, 10000.7900390625, 9999.4736328125, 9999.240234375, 9999.7314453125, 9995.716796875, 9999.369140625, 10001.130859375, 10000.95703125, 9999.4013671875, 9999.7724609375, 9998.2529296875, 9998.7275390625, 10000.0234375, 9998.8671875, 10001.6201171875, 10001.046875, 10000.455078125, 10000.283203125, 9999.3720703125, 10000.041015625, 9998.064453125, 10000.65625, 10000.701171875, 10000.80859375, 9999.8349609375, 10001.59765625, 10000.0830078125, 10000.1943359375, 9999.47265625, 9999.525390625, 9998.4052734375, 9999.4560546875, 9999.6630859375, 9999.7861328125, 9998.91015625, 9998.556640625, 9999.1025390625, 9999.6162109375, 10000.9951171875, 9998.99609375, 10000.3330078125, 9999.6396484375, 9999.6279296875, 9999.013671875, 10002.109375, 10000.1728515625, 9999.87890625, 10000.6787109375, 9998.935546875, 9999.083984375, 10000.197265625, 9998.7978515625, 9999.90625, 10001.41015625, 9998.7978515625, 9999.9853515625, 10000.748046875, 9998.525390625, 10000.2744140625, 10000.013671875, 9998.462890625, 9999.94921875, 10000.6767578125, 10000.9697265625, 9999.7685546875, 10001.3603515625, 10000.1982421875, 10000.0205078125, 9999.1572265625, 9999.546875, 10000.4912109375, 10000.021484375, 10001.2197265625, 10000.2587890625, 9999.9912109375, 10002.279296875, 9999.5439453125, 10000.4248046875, 10001.4326171875, 10000.2685546875, 10000.82421875, 10000.7744140625, 9998.3369140625, 10001.5908203125, 9999.5703125, 10000.0380859375, 10001.58984375, 9999.8505859375, 9999.97265625, 10000.5576171875, 9999.32421875, 10000.6796875, 10000.0654296875, 9999.654296875, 9999.1943359375, 10000.5439453125, 9997.9345703125, 9999.390625, 9999.2265625, 10000.5390625, 10002.349609375, 10000.54296875, 10001.2412109375, 9998.779296875, 10001.224609375, 9998.9189453125, 9999.3251953125, 9999.716796875, 9998.6357421875, 10000.625, 10000.966796875, 10000.6943359375, 9998.5029296875, 9998.0751953125, 9999.5390625, 10001.6982421875, 9998.0185546875, 9998.890625, 10001.47265625, 10000.74609375, 10000.87890625, 10000.10546875, 10000.2880859375, 10000.1591796875, 9999.7236328125, 9997.806640625, 9999.7138671875, 10001.130859375, 9999.041015625, 10000.7392578125, 9999.8251953125, 9999.3447265625, 10001.013671875, 9999.775390625, 9998.572265625, 9998.5751953125, 10001.068359375, 10000.3125, 9999.0146484375, 10000.5517578125, 9999.8251953125, 10001.591796875, 9999.208984375, 9998.6884765625, 9998.634765625, 9999.0361328125, 9999.3193359375, 9999.224609375, 9999.416015625, 9998.921875, 10000.8056640625, 10001.150390625, 9998.88671875, 10001.435546875, 10001.04296875, 10001.2626953125, 9999.412109375, 9998.9501953125, 10001.279296875, 9997.765625, 9999.638671875, 9999.46484375, 9999.3984375, 9999.1474609375, 9999.7646484375, 9999.734375, 10000.1962890625, 9998.9970703125, 9999.8671875, 10000.4833984375, 10000.052734375, 9998.9580078125, 9999.34765625, 10000.490234375, 9999.732421875, 9999.728515625, 9999.8271484375, 10000.751953125, 10000.4130859375, 10003.3671875, 10001.466796875, 10000.6513671875, 10000.8349609375, 9999.447265625, 10000.0771484375, 10000.78515625, 10001.408203125, 9999.94140625, 9999.4931640625, 10000.2451171875, 9999.669921875, 9999.6318359375, 10000.4873046875, 10001.3017578125, 9999.41015625, 10001.7490234375, 9999.9716796875, 9998.505859375, 10000.4921875, 10000.2265625, 10000.5302734375, 9998.431640625, 10000.62890625, 10000.923828125, 9998.90234375, 9998.9970703125, 10000.9267578125, 10002.2177734375, 10001.6435546875, 9998.6416015625, 10001.7021484375, 9998.8759765625, 9999.88671875, 9999.7333984375, 10001.3125, 10001.220703125, 9999.958984375, 10000.453125, 9999.1240234375, 10000.166015625, 10001.4765625, 10000.193359375, 9999.498046875, 10001.3212890625, 9999.662109375, 10000.228515625, 10000.3642578125, 10000.2705078125, 9999.5146484375, 9999.9599609375, 9999.7099609375, 9999.8427734375, 9997.8193359375, 9999.6025390625, 9999.662109375, 10001.31640625, 10001.158203125, 9998.408203125, 9998.7861328125, 10000.6318359375, 10000.0947265625, 9999.3310546875, 9999.076171875, 9999.5126953125, 9999.466796875, 9998.75, 10000.45703125, 10001.4658203125, 9999.857421875, 10000.2578125, 9997.9873046875, 9998.96484375, 9998.3203125, 10000.5986328125, 10000.9794921875, 9998.908203125, 9999.4580078125, 9999.001953125, 9999.5615234375, 9998.5615234375, 10000.78515625, 9999.181640625, 9998.8701171875, 9998.935546875, 9999.6328125, 10001.1943359375, 10000.2470703125, 10000.1123046875, 9999.33203125, 9999.1005859375, 10000.2626953125, 10000.857421875, 10001.1171875, 9999.9951171875, 9998.787109375, 9999.3671875, 9999.955078125, 10000.302734375, 10000.109375, 10001.0146484375, 10000.576171875, 10000.689453125, 10001.173828125, 10000.294921875, 10001.23046875, 9999.701171875, 9999.564453125, 10001.65234375, 9999.9931640625, 10002.1826171875, 9998.0537109375, 10000.333984375, 10000.5634765625, 10002.3330078125, 10001.072265625, 9999.6181640625, 10000.314453125, 9999.291015625, 9999.3935546875, 10000.599609375, 10000.3212890625, 10001.876953125, 9997.33984375, 9999.8896484375, 10000.798828125, 10001.3876953125, 10000.232421875, 10000.11328125, 10000.607421875, 10002.970703125, 10000.853515625, 9999.0458984375, 9999.267578125, 10000.4482421875, 9999.25, 10000.853515625, 10000.2236328125, 9999.9267578125, 10000.2998046875, 9998.6796875, 9999.306640625, 9999.765625, 10000.04296875, 10000.134765625, 10000.1474609375, 10002.1455078125, 9998.41015625, 10001.314453125, 10000.5673828125, 9999.294921875, 10000.1708984375, 9999.4482421875, 9999.76953125, 10000.359375, 10000.55859375, 10000.25, 10001.0498046875, 9998.693359375, 10001.3212890625, 10000.240234375, 9999.0546875, 10001.2275390625, 10000.23828125, 9999.76171875, 9999.9013671875, 9999.51953125, 9998.5166015625, 10000.0595703125, 10000.3759765625, 9999.447265625, 9999.845703125, 9997.4599609375, 10001.09375, 9999.2763671875, 9998.857421875, 10000.6640625, 10001.1279296875, 10000.3603515625, 10000.8779296875, 10000.033203125, 9999.556640625, 10001.50390625, 9997.6025390625, 9998.7705078125, 9999.81640625, 10000.8720703125, 10000.37109375, 9999.8408203125, 9999.865234375, 10000.703125, 9999.7158203125, 10000.1923828125, 10000.2939453125, 9998.875, 9999.185546875, 9999.60546875, 10000.30078125, 10001.1044921875, 9999.25, 9998.1396484375, 10001.11328125, 9998.6142578125, 10001.046875, 9999.2392578125, 9999.3076171875, 9999.884765625, 9999.068359375, 10000.8583984375, 10000.6494140625, 9997.4853515625, 9999.9091796875, 9998.513671875, 10000.1142578125, 10000.509765625, 9999.2060546875, 10000.05078125, 9999.9287109375, 10000.41015625, 9999.9697265625, 9999.09765625, 10000.607421875, 10000.33984375, 9998.14453125, 9999.6142578125, 9999.03515625, 9999.82421875, 9997.44140625, 9999.4033203125, 10001.880859375, 9998.5810546875, 9999.990234375, 10001.8408203125, 10000.392578125, 10001.0009765625, 9999.818359375, 9998.6162109375, 9998.9892578125, 10000.478515625, 9999.126953125, 9999.775390625, 9999.234375, 9998.189453125, 9997.193359375, 10000.7900390625, 9998.5537109375, 9998.8076171875, 10000.0029296875, 9999.7880859375, 9999.57421875, 10000.3779296875, 10000.494140625, 9999.630859375, 9999.095703125, 10000.912109375, 9999.9140625, 9999.4599609375, 9999.69921875, 10001.162109375, 10002.1494140625, 9999.7998046875, 9999.6318359375, 10000.6181640625, 10000.55859375, 10001.0322265625, 10000.015625, 9999.1796875, 10001.1953125, 9999.2314453125, 9999.4111328125, 9999.216796875, 10000.2197265625, 9998.5576171875, 9999.19140625, 10001.7880859375, 10000.9521484375, 10000.9033203125, 10000.978515625, 10001.978515625, 9999.3056640625, 10000.07421875, 10000.919921875, 9999.6787109375, 10000.3974609375, 9999.498046875, 10000.33984375, 10000.5107421875, 10000.876953125, 10000.7861328125, 10000.5546875, 9997.82421875, 10000.2626953125, 9999.93359375, 9999.9697265625, 9999.1513671875, 10000.734375, 10000.1962890625, 9999.6279296875, 10000.1240234375, 10000.5712890625, 9999.060546875, 10000.9384765625, 10000.26171875, 10000.73828125, 9999.55078125, 9999.62890625, 9999.029296875, 9999.373046875, 10000.322265625, 10000.052734375, 10000.2841796875, 9998.58984375, 9999.91796875, 10001.2685546875, 9999.78125, 10000.439453125, 9999.4951171875, 10000.9501953125, 9999.962890625, 10000.384765625, 10000.1796875, 10002.2236328125, 9999.599609375, 10001.78515625, 9998.6689453125, 10000.2353515625, 10000.837890625, 9999.650390625, 10001.0537109375, 10001.54296875, 9999.953125, 10000.0400390625, 9999.6240234375, 9998.5205078125, 9999.5390625, 10000.361328125, 10000.712890625, 9999.8828125, 9999.45703125, 9999.77734375, 9999.63671875, 10000.9619140625, 10000.3388671875, 9997.7939453125, 9999.9326171875, 10000.6953125, 9997.68359375, 10000.4404296875, 9999.8740234375, 10001.0673828125, 9999.453125, 9999.6650390625, 9999.60546875, 10000.1376953125, 10001.0712890625, 10000.32421875, 10001.07421875, 9999.7841796875, 10000.0283203125, 10000.3388671875, 10000.71484375, 10000.63671875, 10001.6376953125, 9999.26953125, 9999.7705078125, 10000.40234375, 10001.376953125, 10000.3642578125, 9999.2392578125, 10000.978515625, 9999.48828125, 9999.7744140625, 10001.2841796875, 10001.095703125, 9998.7099609375, 10001.69921875, 10000.3544921875, 10000.51171875, 9996.8486328125, 9999.2783203125, 9999.8359375, 9999.251953125, 9999.345703125, 9999.9619140625, 10000.2509765625, 10000.3544921875, 9998.6826171875, 9999.9521484375, 9999.48046875, 10002.5283203125, 10000.244140625, 9999.3037109375, 9999.80078125, 9998.2861328125, 9999.955078125, 10001.03125, 10001.087890625, 10002.0810546875, 9999.5263671875, 9998.857421875, 9999.962890625, 10000.25, 9999.9140625, 10001.2099609375, 9999.6337890625, 10000.1982421875, 10001.01953125, 10000.259765625, 10001.388671875, 10001.91796875, 10000.5166015625, 9999.06640625, 10001.52734375, 9999.17578125, 9999.822265625, 9998.658203125, 9999.248046875, 9998.8779296875, 9999.9892578125, 9998.5439453125, 9999.0576171875, 9999.6875, 10001.400390625, 9999.6162109375, 9998.591796875, 9999.0830078125, 10002.3720703125, 9998.75, 9998.556640625, 10000.2177734375, 10000.232421875, 10000.669921875, 9998.4736328125, 9999.8857421875, 9999.5859375, 10001.482421875, 10001.166015625, 9998.0849609375, 10000.98046875, 9999.5556640625, 9999.83984375, 9999.546875, 9998.2099609375, 9999.712890625, 9998.82421875, 10000.1826171875, 9999.4443359375, 10000.162109375, 9998.9208984375, 9998.587890625, 10000.70703125, 9998.302734375, 10000.9375, 10000.7841796875, 9999.8056640625, 10000.111328125, 9999.0673828125, 10000.0458984375, 9998.0400390625, 10000.28125, 9999.544921875, 9999.830078125, 9998.1962890625, 10000.7666015625, 10001.6689453125, 9999.8779296875, 9999.8134765625, 10001.79296875, 10000.19921875, 9999.560546875, 9997.83984375, 10001.19921875, 9999.5087890625, 10000.6455078125, 10000.95703125, 10000.37109375, 10000.2763671875, 9999.515625, 10000.2060546875, 10000.1474609375, 10000.244140625, 9998.60546875, 9998.8720703125, 9999.1044921875, 10001.115234375, 10000.2060546875, 9998.1220703125, 10001.001953125, 10001.22265625, 10000.0615234375, 10002.046875, 10000.697265625, 10000.994140625, 10001.60546875, 10000.5439453125, 9999.404296875, 9999.173828125, 9999.06640625, 9999.845703125, 9999.78515625, 9999.3125, 10001.8447265625, 10000.6962890625, 10000.7119140625, 10000.05859375, 10000.517578125, 10000.1279296875, 9999.5517578125, 10000.158203125, 9998.431640625, 9999.7685546875, 10000.1171875, 10000.10546875, 9999.5205078125, 9999.8017578125, 10000.6845703125, 9999.3701171875, 9999.248046875, 10001.5849609375, 9999.5263671875, 9999.7353515625, 9999.75390625, 10000.6904296875, 9999.9013671875, 9998.935546875, 9998.8515625, 9998.5166015625, 10000.26171875, 9999.5625, 10000.6298828125, 10000.47265625, 9999.3486328125, 10000.7216796875, 9999.7783203125, 10001.2431640625, 10000.5966796875, 10000.4755859375, 10001.4091796875, 9999.8837890625, 9999.224609375, 10000.1962890625, 9999.5576171875, 10002.669921875, 9999.576171875, 9999.5126953125, 9999.205078125, 10001.759765625, 9998.4736328125, 9999.552734375, 9999.673828125, 9999.5029296875, 9999.8134765625, 10000.791015625, 9998.5419921875, 9999.44140625, 10000.4970703125, 10000.6396484375, 10001.15234375, 9999.5078125, 9997.986328125, 10000.193359375, 10001.4248046875, 10000.0263671875, 10000.587890625, 9999.53125, 9999.3427734375, 10000.4091796875, 9998.2138671875, 10000.357421875, 10000.1181640625, 10000.7900390625, 9999.5009765625, 10000.927734375, 10000.576171875, 10000.625, 9998.91796875, 9998.8447265625, 9998.4072265625, 9999.9736328125, 9999.01171875, 10000.4990234375, 9999.4970703125, 9998.744140625, 9999.4814453125, 10000.314453125, 9999.6396484375, 9999.8017578125, 10000.12890625, 9999.6181640625, 9999.2900390625, 10000.0634765625, 9998.1650390625, 9999.4140625, 9999.0673828125, 9999.8291015625, 10000.2265625, 9999.861328125, 10000.2197265625, 9999.7109375, 9999.6708984375, 10000.642578125, 9998.904296875, 9999.3876953125, 10000.28125, 10000.46875, 10000.1943359375, 9999.2392578125, 10000.107421875, 10000.5849609375, 9999.013671875, 10001.7783203125, 10000.625, 9998.7275390625, 10001.90625, 9999.1875, 10001.044921875, 9998.8583984375, 9999.5595703125, 9998.013671875, 9999.0322265625, 9998.7275390625, 9999.9150390625, 10001.150390625, 10000.0654296875, 9998.33203125, 10001.73828125, 9999.31640625, 10000.1201171875, 10000.7451171875, 10000.9482421875, 9999.4033203125, 10001.3974609375, 9999.87109375, 10000.1953125, 9997.470703125, 10001.1748046875, 9999.1240234375, 9998.716796875, 10001.189453125, 9999.8896484375, 9999.1298828125, 9999.5927734375, 9999.3818359375, 9999.23828125, 10001.08984375, 9999.4716796875, 9999.9873046875, 9999.04296875, 9999.013671875, 9999.453125, 9999.13671875, 9999.9501953125, 10001.2998046875, 10002.294921875, 9999.302734375, 10001.1044921875, 9999.4716796875, 10000.873046875, 9999.8984375, 9999.611328125, 10000.0322265625, 9998.51953125, 9998.7080078125, 9997.470703125, 10001.6787109375, 10001.603515625, 10000.0439453125, 10001.3984375, 10000.1162109375, 9999.0, 9999.765625, 9998.4951171875, 9999.9853515625, 10000.4873046875, 10000.2998046875, 9998.8857421875, 10000.119140625, 9998.779296875, 9999.0712890625, 9998.1357421875, 10000.3837890625, 10000.056640625, 10000.4189453125, 10001.4892578125, 10001.6689453125, 9999.6357421875, 10001.1884765625, 9999.7587890625, 9999.6220703125, 10000.2314453125, 10000.1064453125, 10000.0625, 9999.75, 10000.4638671875, 9998.1728515625, 10001.0009765625, 9999.4365234375, 9999.619140625, 9999.4599609375, 9998.8525390625, 10000.8544921875, 9999.3876953125, 9999.5888671875, 10000.703125, 10000.9619140625, 9999.802734375, 9999.9931640625, 10000.421875, 10000.173828125, 10000.44921875, 9999.3525390625, 9999.974609375, 9999.1767578125, 10001.0751953125, 10000.62890625, 10000.478515625, 10000.234375, 9999.515625, 9998.6962890625, 9999.271484375, 10000.52734375, 9999.1787109375, 10001.4482421875, 10000.2685546875, 9999.3154296875, 10000.306640625, 10002.044921875, 10000.271484375, 10002.0361328125, 9999.931640625, 9999.3212890625, 10000.40625, 9998.9111328125, 9998.5341796875, 9998.2939453125, 10001.3193359375, 9999.84375, 9998.05078125, 10000.97265625, 9998.61328125, 9999.2724609375, 10001.298828125, 9999.09375, 10001.0703125, 9999.421875, 10000.2294921875, 10001.0, 10000.1669921875, 10000.0947265625, 10000.3271484375, 9998.7099609375, 10000.1923828125, 10000.0986328125, 10001.5986328125, 9999.779296875, 10000.8720703125, 10000.3662109375, 9998.533203125, 9999.431640625, 10000.0478515625, 10000.4716796875, 9999.4189453125, 9999.591796875, 10000.556640625, 10000.6162109375, 9999.5712890625, 9999.6845703125, 10000.6689453125, 9999.302734375, 10000.4111328125, 10000.4638671875, 9998.20703125, 10001.5068359375, 10001.3359375, 10000.7587890625, 10001.0361328125, 10000.091796875, 10000.9677734375, 9999.314453125, 9998.6630859375, 10000.0859375, 10000.287109375, 9999.955078125, 9999.498046875, 9999.3837890625, 9999.427734375, 9999.9150390625, 9998.421875, 10000.0380859375, 9999.6943359375, 10000.73828125, 9998.9111328125, 10000.013671875, 9999.734375, 9999.625, 10000.3798828125, 10000.42578125, 10000.9384765625, 10001.5390625, 10001.23828125, 9999.107421875, 9997.3759765625, 9999.580078125, 9999.892578125, 10000.7529296875, 10000.52734375, 10000.609375, 9999.36328125, 9998.939453125, 10000.1259765625, 10002.0849609375, 9999.638671875, 9998.69921875, 10000.556640625, 9999.9638671875, 10000.0146484375, 10000.8720703125, 9999.8603515625, 9997.9541015625, 9999.1435546875, 9999.8271484375, 10000.89453125, 9999.470703125, 10000.5087890625, 10001.06640625, 10001.40625, 9999.2939453125, 10000.0634765625, 10000.1572265625, 9999.767578125, 10001.5166015625, 9999.65234375, 10001.0224609375, 9999.4931640625, 10000.7314453125, 10001.412109375, 10001.39453125, 9999.720703125, 9999.7509765625, 10000.5185546875, 10001.1240234375, 9999.8515625, 10000.900390625, 9998.5576171875, 10000.373046875, 10000.640625, 9998.9267578125, 9999.9609375, 10000.974609375, 9998.552734375, 9999.490234375, 9998.5498046875, 10001.955078125, 9998.4697265625, 9998.33984375, 10000.677734375, 10000.5478515625, 9998.9775390625, 10000.77734375, 10001.1005859375, 9999.376953125, 10000.7978515625, 10001.9912109375, 9999.828125, 9998.1787109375, 9999.60546875, 10002.40625, 9997.6259765625, 10000.2841796875, 9999.3818359375, 10000.994140625, 9999.8115234375, 10000.1611328125, 9999.37109375, 9999.560546875, 10001.005859375, 10001.6142578125, 10001.22265625, 9999.310546875, 9999.3408203125, 9999.8857421875, 10000.1328125, 9998.84765625, 9999.3125, 9999.318359375, 9999.3330078125, 9999.0703125, 10000.3427734375, 9998.7255859375, 9998.7626953125, 10000.9267578125, 9999.46484375, 9999.1435546875, 9999.5302734375, 10000.23828125, 10000.46484375, 10001.5205078125, 10000.212890625, 9998.12109375, 9999.7626953125, 9998.462890625, 9999.92578125, 9999.888671875, 9998.6416015625, 10001.2939453125, 10000.4765625, 10001.2431640625, 9998.7490234375, 10000.978515625, 10000.302734375, 10000.9794921875, 10000.7060546875, 9999.7138671875, 9999.1103515625, 10000.40234375, 10000.28515625, 9998.7421875, 9998.5751953125, 10002.591796875, 9999.6044921875, 9998.5849609375, 9998.7021484375, 9999.0419921875, 10000.1474609375, 9998.8876953125, 10000.4130859375, 9999.3076171875, 9998.357421875, 10001.17578125, 10000.7138671875, 10000.19921875, 10001.1181640625, 9999.6328125, 10000.27734375, 9998.400390625, 9998.3955078125, 10000.755859375, 9999.1083984375, 10000.3388671875, 10000.966796875, 9999.517578125, 9998.8896484375, 9999.7255859375, 10000.4267578125, 10001.1865234375, 10000.712890625, 9998.8291015625, 10001.181640625, 9998.6533203125, 9999.6015625, 9999.404296875, 9999.0556640625, 9997.759765625, 9998.0986328125, 9999.830078125, 9999.31640625, 10001.0947265625, 9999.6689453125, 9999.255859375, 10000.1318359375, 9998.5595703125, 10001.0205078125, 10000.0927734375, 10001.7021484375, 9998.7724609375, 9998.205078125, 9998.2275390625, 10000.4990234375, 9999.6259765625, 10000.61328125, 9998.91796875, 9998.56640625, 9999.1943359375, 9999.58203125, 10000.353515625, 10002.1494140625, 10000.796875, 10001.1806640625, 9999.3984375, 9999.044921875, 9998.7236328125, 9999.51171875, 9998.875, 10001.279296875, 10000.337890625, 9998.2939453125, 9999.232421875, 10001.5810546875, 10000.10546875, 10000.7236328125, 10000.22265625, 9999.6787109375, 9997.064453125, 9999.6396484375, 9998.0380859375, 10000.5830078125, 10000.0732421875, 10000.0244140625, 10001.294921875, 10000.42578125, 9998.98046875, 10000.6328125, 9999.615234375, 10002.033203125, 9999.18359375, 10000.1455078125, 9999.4287109375, 9999.8759765625, 10000.2705078125, 10000.5751953125, 9998.6689453125, 9999.314453125, 10000.5498046875, 10000.0634765625, 9999.9013671875, 9998.6513671875, 10000.7705078125, 9998.4541015625, 10001.1669921875, 9998.275390625, 10001.5703125, 9999.626953125, 9999.2109375, 10000.9697265625, 9999.2685546875, 10000.380859375, 10000.2119140625, 9999.669921875, 9999.2314453125, 9999.859375, 9998.7529296875, 9999.93359375, 10000.798828125, 9999.20703125, 10000.1142578125, 9999.892578125, 9998.837890625, 10000.162109375, 10003.380859375, 9999.0205078125, 9999.064453125, 10001.5869140625, 10001.765625, 10000.9541015625, 10000.0908203125, 9999.931640625, 9999.904296875, 10000.513671875, 9999.0703125, 9999.6943359375, 9998.376953125, 10000.146484375, 9997.859375, 10000.4677734375, 9999.1015625, 9999.61328125, 10002.3876953125, 9999.04296875, 9999.0791015625, 10000.390625, 10000.021484375, 10000.8876953125, 10000.484375, 10000.53515625, 10001.396484375, 9999.46484375, 9999.767578125, 10000.7001953125, 10000.1396484375, 9999.109375, 9997.4716796875, 10000.498046875, 9999.990234375, 9999.04296875, 10000.15234375, 10001.203125, 9999.1044921875, 10000.412109375, 9999.455078125, 9999.69921875, 9999.1875, 9999.5322265625, 10000.296875, 10000.0947265625, 10000.4443359375, 10000.255859375, 10000.8056640625, 10001.22265625, 10000.208984375, 10000.6513671875, 10000.650390625, 10000.55078125, 9999.84765625, 9999.861328125, 9999.654296875, 10001.12109375, 10000.35546875, 10000.322265625, 9999.3486328125, 10000.3359375, 10000.876953125, 9998.623046875, 10000.244140625, 10001.57421875, 9998.8857421875, 9999.49609375, 10000.8623046875, 10002.1513671875, 9999.779296875, 10000.828125, 9998.294921875, 9998.580078125, 9999.8154296875, 10000.33984375, 9999.240234375, 10001.2060546875, 9999.9912109375, 9997.9541015625, 10000.51171875, 10000.4345703125, 10000.482421875, 9998.265625, 9998.8779296875, 10000.130859375, 9999.35546875, 10001.6640625, 9998.822265625, 9998.7548828125, 10001.1982421875, 10002.3818359375, 10001.845703125, 9998.7578125, 10000.5859375, 9998.1904296875, 10001.529296875, 9999.0498046875, 9999.7978515625, 9998.7607421875, 10001.611328125, 9998.162109375, 10000.37109375, 10000.4091796875, 10000.515625, 9999.2783203125, 9999.0751953125, 9999.845703125, 9999.9189453125, 10000.14453125, 9999.802734375, 9999.390625, 9999.1044921875, 9999.9609375, 10000.5615234375, 9998.8203125, 9999.7822265625, 10000.7685546875, 9999.501953125, 10000.103515625, 9999.5234375, 9999.6103515625, 9999.890625, 9998.634765625, 9999.255859375, 9998.328125, 9999.380859375, 10000.908203125, 9999.818359375, 10001.642578125, 10000.404296875, 9998.5234375, 10000.693359375, 10000.052734375, 10000.4599609375, 10000.4013671875, 9999.466796875, 9999.275390625, 9999.5791015625, 9999.5478515625, 10001.091796875, 10000.4140625, 10000.0048828125, 9999.0517578125, 10001.4375, 10000.8115234375, 10001.54296875, 9999.5859375, 10002.501953125, 10000.7607421875, 10000.8623046875, 10000.9111328125, 10000.23828125, 9998.59375, 9998.5419921875, 10001.2861328125, 10001.193359375, 9999.20703125, 9999.5400390625, 10001.2802734375, 9999.9599609375, 9999.5576171875, 9999.0849609375, 9997.8740234375, 9999.541015625, 9999.95703125, 9999.7353515625, 10000.525390625, 9998.6484375, 10001.4423828125, 10001.16015625, 9998.625, 9997.841796875, 9999.466796875, 9999.4853515625, 10001.396484375, 9999.4130859375, 9999.7451171875, 9998.4326171875, 9998.791015625, 10000.841796875, 9997.7763671875, 9999.3701171875, 9999.1884765625, 10001.3505859375, 10003.0234375, 10000.7080078125, 9997.751953125, 10000.30859375, 10001.25, 9999.201171875, 10000.9794921875, 10000.560546875, 9999.8134765625, 9997.4677734375, 9998.7822265625, 10001.3642578125, 10001.4140625, 9998.96484375, 10000.630859375, 9998.267578125, 9999.5751953125, 10001.4560546875, 10000.751953125, 9999.814453125, 9999.8564453125, 10002.50390625, 10000.2392578125, 10000.173828125, 9999.9638671875, 10000.6962890625, 10000.166015625, 10000.0634765625, 10000.6796875, 9999.72265625, 10000.078125, 10000.8818359375, 10000.4296875, 9998.8828125, 10000.3623046875, 9998.9326171875, 9999.583984375, 9998.5859375, 9999.53515625, 10000.779296875, 10000.2333984375, 9999.5556640625, 10000.7568359375, 10001.0302734375, 9999.384765625, 9997.814453125, 9999.1513671875, 9999.7783203125, 9999.62890625, 9999.3388671875, 10000.9208984375, 9999.9599609375, 10001.294921875, 10001.349609375, 9999.6201171875, 10000.5634765625, 10000.9619140625, 10000.6728515625, 10000.1748046875, 10000.2255859375, 9999.8359375, 10002.17578125, 10000.66015625, 10001.693359375, 10000.447265625, 9999.759765625, 9999.404296875, 10000.1240234375, 9998.0322265625, 9999.6328125, 9999.771484375, 9998.810546875, 10001.0908203125, 9998.6201171875, 10000.5869140625, 9999.6279296875, 10000.1875, 10000.2470703125, 9999.7958984375, 9998.4658203125, 10000.1376953125, 9998.5673828125, 10000.37890625, 10000.1787109375, 9998.5009765625, 9999.546875, 9999.51953125, 10000.9814453125, 9998.3671875, 9999.6748046875, 9999.9306640625, 10001.572265625, 10001.244140625, 10000.9375, 9999.1064453125, 10000.4765625, 10000.2041015625, 10000.3349609375, 10000.06640625, 10000.3203125, 9999.9296875, 10000.42578125, 10001.8369140625, 10000.412109375, 10001.1865234375, 10000.8427734375, 9999.2041015625, 10000.5908203125, 10000.333984375, 10000.375, 9998.548828125, 9999.453125, 9999.736328125, 9999.1435546875, 9998.7705078125, 10000.3857421875, 9999.3232421875, 9998.7333984375, 9999.623046875, 10000.1416015625, 9999.1474609375, 9999.287109375, 9998.880859375, 9999.8837890625, 10000.498046875, 10001.431640625, 10000.314453125, 9998.859375, 9999.705078125, 9999.7607421875, 10003.09765625, 10001.94921875, 9999.509765625, 9999.564453125, 9997.509765625, 9999.611328125, 10001.208984375, 9999.2119140625, 10000.2646484375, 9999.865234375, 10001.6484375, 10001.681640625, 10000.818359375, 9999.3662109375, 9999.009765625, 9999.42578125, 10002.5419921875, 10000.3203125, 10000.8486328125, 9997.78515625, 9999.4716796875, 10000.62890625, 10000.8212890625, 10001.4345703125, 10000.8232421875, 10003.2001953125, 10001.119140625, 10002.095703125, 9999.640625, 10000.7880859375, 9999.6533203125, 10000.916015625, 9997.3408203125, 9998.9599609375, 9999.92578125, 9999.7548828125, 10000.4853515625, 9999.6533203125, 9999.6953125, 9999.548828125, 10001.873046875, 9998.74609375, 10000.2255859375, 10001.0869140625, 9999.1943359375, 9998.837890625, 10001.39453125, 10000.1767578125, 9998.88671875, 10002.05859375, 9999.7822265625, 10000.1455078125, 10001.556640625, 9998.8310546875, 9999.9140625, 9999.6513671875, 10001.125, 9998.69921875, 10000.7509765625, 10000.5166015625, 10000.62890625, 9998.830078125, 9998.0849609375, 10000.517578125, 10001.447265625, 10001.279296875, 10000.2431640625, 10001.125, 10001.990234375, 10000.2314453125, 10000.73828125, 9999.1884765625, 10002.828125, 9999.2119140625, 10000.7392578125, 9999.935546875, 10001.3935546875, 9998.978515625, 9998.1748046875, 10000.8173828125, 9998.763671875, 10001.08203125, 10000.2099609375, 10000.4365234375, 9998.873046875, 10000.2744140625, 9999.994140625, 10000.9306640625, 10000.1923828125, 10000.21484375, 10001.0205078125, 10001.798828125, 9998.6845703125, 9999.6748046875, 10000.677734375, 9999.7041015625, 9998.9404296875, 9999.234375, 9999.8955078125, 9999.9873046875, 9999.6884765625, 10001.005859375, 10001.0439453125, 10000.46484375, 9998.9267578125, 10000.7041015625, 9998.8134765625, 10002.150390625, 9999.1640625, 9999.0244140625, 9999.802734375, 9999.599609375, 9998.3701171875, 9999.78515625, 10000.9814453125, 9998.904296875, 9999.9423828125, 9999.1689453125, 10000.9775390625, 9999.875, 9998.9951171875, 9998.771484375, 10000.0947265625, 9999.9462890625, 10001.7294921875, 9999.2568359375, 10000.5380859375, 9999.5595703125, 9998.76171875, 9999.41796875, 10000.9765625, 9999.177734375, 10000.47265625, 10001.015625, 10000.49609375, 10000.248046875, 10001.359375, 10001.4384765625, 9998.380859375, 10000.0146484375, 9999.3623046875, 9999.505859375, 9998.9658203125, 9999.4775390625, 10002.080078125, 9999.095703125, 10000.8828125, 10000.6318359375, 9998.017578125, 10000.865234375, 10001.193359375, 10001.8369140625, 10000.556640625, 10001.73046875, 10000.2822265625, 10001.1337890625, 9998.9892578125, 9999.2783203125, 10000.7216796875, 9998.3974609375, 9998.4482421875, 10000.8056640625, 9999.9521484375, 9999.041015625, 9999.56640625, 9999.2861328125, 9999.2392578125, 9999.3251953125, 9999.4150390625, 9998.27734375, 9998.5498046875, 10000.2099609375, 9999.9140625, 9998.8623046875, 9999.9375, 10000.435546875, 10000.3955078125, 9999.44140625, 10000.1845703125, 10000.7880859375, 10000.3671875, 10000.015625, 9999.2919921875, 10000.1357421875, 9999.84375, 10000.984375, 9999.771484375, 10000.0322265625, 9999.1376953125, 10000.5654296875, 10000.5205078125, 9999.541015625, 10000.4140625, 9999.978515625, 9999.2275390625, 9998.75390625, 10000.1015625, 9999.1044921875, 10001.8056640625, 10001.0400390625, 10000.134765625, 10000.5478515625, 10000.8564453125, 9997.48046875, 9999.6123046875, 9998.96484375, 9999.7841796875, 9998.119140625, 10000.625, 9998.4794921875, 10000.34375, 10000.5859375, 9998.96875, 9999.587890625, 9999.3134765625, 9999.6142578125, 9999.7177734375, 9999.091796875, 9998.6884765625, 9998.599609375, 9999.83203125, 10000.7548828125, 10001.4736328125, 10000.125, 10000.0126953125, 10000.40625, 9998.8984375, 10000.9765625, 10001.34765625, 9999.4365234375, 10000.6337890625, 10000.134765625, 9999.33203125, 10000.9423828125, 9999.6494140625, 10000.7099609375, 10000.09765625, 9999.1953125, 10000.2822265625, 9998.056640625, 10000.666015625, 10000.4638671875, 10000.814453125, 9999.6513671875, 10000.017578125, 10000.32421875, 10000.8759765625, 10001.33984375, 10000.220703125, 10001.4208984375, 9999.56640625, 9998.5859375, 9999.7783203125, 9998.9501953125, 9999.7998046875, 9999.685546875, 9999.7548828125, 9999.6748046875, 9998.6708984375, 9999.7919921875, 9999.9716796875, 10000.6826171875, 10001.4892578125, 9999.4326171875, 9998.111328125, 9998.72265625, 9999.9658203125, 9999.4921875, 10001.275390625, 10000.2939453125, 9998.9775390625, 9999.2177734375, 9998.9638671875, 10002.6865234375, 9999.01953125, 9999.2353515625, 9999.0693359375, 9999.34375, 9999.0830078125, 10000.048828125, 9999.4130859375, 9999.349609375, 10000.4423828125, 10001.2294921875, 10000.63671875, 10000.33203125, 9999.7978515625, 9999.5888671875, 9999.9580078125, 10000.6826171875, 9999.3955078125, 10000.1279296875, 10000.1162109375, 9999.61328125, 9998.1650390625, 9997.615234375, 10000.009765625, 10000.84765625, 9999.0654296875, 9998.9794921875, 10000.876953125, 10000.44921875, 9999.85546875, 9999.2021484375, 10001.1572265625, 9999.86328125, 10001.6259765625, 10000.255859375, 10000.8017578125, 9999.9873046875, 9999.337890625, 9997.630859375, 9998.337890625, 10001.453125, 9998.9892578125, 10001.890625, 10000.708984375, 10001.8427734375, 9999.37109375, 10000.3466796875, 10000.6025390625, 9999.791015625, 9998.357421875, 10000.00390625, 10001.6103515625, 9998.673828125, 9999.751953125, 10002.349609375, 9999.84375, 10000.64453125, 10000.3974609375, 10000.6220703125, 9999.6240234375, 9997.96484375, 10000.2109375, 10001.947265625, 10001.248046875, 9999.2578125, 10000.96875, 9999.63671875, 9999.9072265625, 9998.080078125, 9998.9052734375, 10000.201171875, 9998.9208984375, 10000.15625, 9999.2998046875, 10000.05859375, 10000.9775390625, 9999.943359375, 9998.369140625, 9999.37890625, 9997.947265625, 9998.5576171875, 10000.3681640625, 10000.6083984375, 10000.5625, 10000.4365234375, 10000.783203125, 10001.4453125, 10002.0478515625, 9999.66015625, 9999.4580078125, 10000.0537109375, 10001.0751953125, 10000.720703125, 9999.8544921875, 9999.65625, 10000.2392578125, 9999.3544921875, 10002.486328125, 10000.21875, 9998.5234375, 10000.07421875, 9998.76953125, 9999.7412109375, 9999.359375, 9998.740234375, 9999.869140625, 10000.1259765625, 9999.7275390625, 10000.1552734375, 9997.900390625, 10001.8828125, 10001.201171875, 9999.5244140625, 9999.02734375, 10000.7646484375, 9999.2734375, 9999.677734375, 9999.4443359375, 10000.7470703125, 9999.599609375, 9999.3466796875, 10001.638671875, 9998.708984375, 9999.9775390625, 10000.4873046875, 10000.1435546875, 10000.888671875, 10000.6875, 9998.1416015625, 9999.5224609375, 10000.4111328125, 9998.919921875, 10000.390625, 10000.138671875, 9998.666015625, 10001.4453125, 9999.064453125, 9999.892578125, 9999.6044921875, 9999.927734375, 10002.064453125, 10000.443359375, 10001.435546875, 9999.7763671875, 9999.501953125, 10000.00390625, 10000.6103515625, 9999.56640625, 10001.5185546875, 10001.041015625, 10000.880859375, 10001.1181640625, 9998.765625, 9999.3037109375, 9998.57421875, 10000.349609375, 10001.7958984375, 10001.0380859375, 9999.2060546875, 9999.5732421875, 10002.0615234375, 9999.4775390625, 9998.5654296875, 9998.62890625, 10000.2333984375, 9999.4296875, 9999.611328125, 10000.2255859375, 9999.357421875, 10000.1630859375, 9999.9892578125, 10000.001953125, 10001.62109375, 9997.82421875, 10000.2158203125, 10000.75390625, 10000.7099609375, 10000.3359375, 10000.650390625, 9998.572265625, 9999.9931640625, 10001.8427734375, 10000.5224609375, 10000.7490234375, 10001.75, 10001.572265625, 10000.3251953125, 9999.646484375, 10001.3212890625, 10000.5888671875, 10000.798828125, 10000.5263671875, 9999.6865234375, 9999.224609375, 10000.87109375, 9998.888671875, 10000.4228515625, 10000.697265625, 10000.6572265625, 10000.2509765625, 10000.1806640625, 10000.529296875, 10001.6025390625, 9998.8212890625, 9999.6728515625, 9998.8232421875, 9999.8212890625, 9999.3994140625, 9999.4765625, 9999.31640625, 9999.783203125, 10000.3359375, 10000.3271484375, 10001.69921875, 9999.48046875, 10001.228515625, 10000.5107421875, 9999.767578125, 9999.68359375, 9998.9384765625, 10000.2197265625, 10000.154296875, 9999.9755859375, 10000.974609375, 9998.6826171875, 9998.5673828125, 9999.7353515625, 10000.8525390625, 9999.4638671875, 10001.1806640625, 10000.0947265625, 10000.8916015625, 9999.826171875, 9998.6279296875, 10000.0869140625, 9998.9169921875, 9999.697265625, 10000.2939453125, 9999.486328125, 9999.1376953125, 10001.16796875, 10001.439453125, 10001.119140625, 9998.67578125, 10000.5556640625, 9999.373046875, 9998.7783203125, 9997.45703125, 9998.7158203125, 10000.61328125, 9999.67578125, 9999.7451171875, 10001.73046875, 9999.05859375, 9997.998046875, 9998.8564453125, 9997.8720703125, 9999.4609375, 9999.69140625, 10001.3203125, 9999.2666015625, 9998.111328125, 9999.474609375, 9999.7041015625, 10001.3701171875, 9999.287109375, 9999.376953125, 10000.189453125, 9996.8916015625, 10000.4365234375, 10000.783203125, 10000.767578125, 9999.0576171875, 10002.0791015625, 10000.3447265625, 10002.74609375, 10000.6044921875, 10000.0498046875, 10000.986328125, 9998.5234375, 10000.4091796875, 9997.9697265625, 9998.486328125, 10000.9638671875, 9999.3046875, 9999.8232421875, 10000.40234375, 9999.3876953125, 10000.1474609375, 10000.5107421875, 9999.8896484375, 10000.7978515625, 10002.0107421875, 9999.1455078125, 9998.830078125, 9998.6875, 10000.06640625, 10001.658203125, 9998.7060546875, 9998.732421875, 9999.197265625, 10000.427734375, 9998.9033203125, 9998.9169921875, 9999.0595703125, 9999.05078125, 9999.90625, 10000.294921875, 9998.6591796875, 9999.07421875, 9999.6640625, 10000.365234375, 10001.251953125, 10000.2294921875, 10000.5458984375, 9998.302734375, 10000.107421875, 9999.7802734375, 9999.3125, 10000.3076171875, 9998.2421875, 9999.693359375, 9997.9775390625, 9999.759765625, 10000.0078125, 10001.080078125, 9999.96484375, 9999.0205078125, 9999.1748046875, 10001.822265625, 10000.55078125, 10000.0595703125, 9999.8623046875, 10000.845703125, 9998.072265625, 9999.8798828125, 9999.7353515625, 10001.154296875, 9998.9130859375, 9999.328125, 10001.6904296875, 10001.3916015625, 9997.9921875, 9999.1484375, 10000.57421875, 10000.6083984375, 9998.900390625, 9998.7890625, 10001.6220703125, 10000.474609375, 10000.669921875, 9999.4306640625, 9999.31640625, 9999.4609375, 10000.755859375, 10000.1650390625, 10001.099609375, 9999.9599609375, 9998.3603515625, 10000.2724609375, 9999.4765625, 10000.787109375, 10000.3388671875, 9999.4306640625, 10001.1669921875, 10000.3291015625, 10000.6357421875, 10000.0009765625, 10000.3388671875, 10000.306640625, 9999.7978515625, 9999.8251953125, 10000.24609375, 10001.177734375, 10000.236328125, 9997.6552734375, 9998.828125, 9998.1923828125, 9998.693359375, 9999.81640625, 9999.1806640625, 10000.14453125, 9998.3154296875, 9999.2900390625, 9998.447265625, 9999.5029296875, 10000.46875, 9998.2890625, 9999.7587890625, 10000.830078125, 10000.4296875, 9999.2568359375, 9999.712890625, 9999.2607421875, 10001.283203125, 10000.19140625, 10001.2802734375, 9998.734375, 10001.4326171875, 9999.2021484375, 9999.9892578125, 10001.103515625, 9999.58984375, 10000.3154296875, 9999.28515625, 10000.6376953125, 9999.84375, 10001.265625, 10000.0244140625, 10000.185546875, 9999.05078125, 9998.9912109375, 9999.310546875, 10000.4443359375, 10001.0791015625, 10002.4189453125, 10000.0654296875, 9999.4697265625, 9999.37890625, 10000.73828125, 10000.2666015625, 10000.3720703125, 10002.0517578125, 9999.9326171875, 9999.234375, 9999.9560546875, 9999.0107421875, 10001.2490234375, 10000.0029296875, 9999.80859375, 10000.8837890625, 10000.0654296875, 9999.421875, 9999.1279296875, 10001.05078125, 10000.5068359375, 10001.0615234375, 10000.373046875, 10001.1318359375, 9998.84375, 9999.3447265625, 9999.818359375, 9999.658203125, 9999.2197265625, 10000.001953125, 10001.0009765625, 10002.1728515625, 9999.9794921875, 10000.77734375, 9998.4306640625, 9998.2783203125, 10001.5126953125, 10000.1298828125, 9999.3935546875, 9999.869140625, 9998.4404296875, 10000.7607421875, 9998.5009765625, 10000.2978515625, 10000.1943359375, 9999.564453125, 10000.751953125, 9998.54296875, 10000.0693359375, 10001.2822265625, 10000.2109375, 10000.658203125, 9999.97265625, 10000.2451171875, 9999.7802734375, 9998.4287109375, 9999.1474609375, 10000.2880859375, 10000.0537109375, 10000.5673828125, 9999.6064453125, 10000.57421875, 10001.5830078125, 9999.5966796875, 9999.9404296875, 10000.60546875, 10001.3720703125, 10000.328125, 10000.7060546875, 10000.0244140625, 10000.3525390625, 10001.015625, 10000.197265625, 10000.87890625, 10000.3251953125, 9999.0048828125, 9999.6533203125, 9999.4814453125, 10000.39453125, 9999.685546875, 9999.5078125, 10000.81640625, 10000.8583984375, 9999.8212890625, 10001.4736328125, 10002.30859375, 10000.7099609375, 9999.3564453125, 9999.58984375, 10001.45703125, 10000.5859375, 9999.33203125, 10000.12109375, 10000.6591796875, 10001.0400390625, 10000.4443359375, 9999.619140625, 10000.68359375, 9998.3408203125, 10000.7939453125, 9998.5830078125, 9999.138671875, 9998.7841796875, 10000.0224609375, 9998.0771484375, 9998.1875, 10001.14453125, 9999.7490234375, 10001.6123046875, 9998.7763671875, 9998.2919921875, 9998.498046875, 10000.5419921875, 10001.1298828125, 9999.96484375, 10000.328125, 10001.3251953125, 10001.96875, 9999.3857421875, 9998.865234375, 10001.5654296875, 10000.6748046875, 10000.21875, 9999.9423828125, 9999.48828125, 9998.1669921875, 9999.076171875, 10000.3662109375, 10000.25390625, 9999.447265625, 9998.615234375, 10000.2587890625, 9999.4931640625, 9999.8056640625, 10000.34765625, 10000.3056640625, 10000.7158203125, 9999.3251953125, 10000.263671875, 10000.181640625, 10000.53125, 10000.0693359375, 10000.21484375, 9999.064453125, 9998.015625, 9999.2041015625, 9998.5703125, 10000.1376953125, 10000.1572265625, 10001.7578125, 9999.4853515625, 10000.0712890625, 9999.7705078125, 9997.98046875, 10001.029296875, 10000.5576171875, 9998.1875, 10001.240234375, 9998.826171875, 10001.24609375, 9999.5869140625, 9998.740234375, 9998.66796875, 10000.794921875, 10000.515625, 10000.03125, 9999.6669921875, 10001.529296875, 9998.0673828125, 9998.8125, 9999.701171875, 9999.9404296875], [10000.0966796875, 10000.85546875, 10000.1728515625, 9999.76953125, 10000.6953125, 10000.0087890625, 9998.4345703125, 10000.30078125, 9998.4990234375, 9999.095703125, 10000.1748046875, 9998.609375, 10000.4501953125, 10001.1259765625, 10001.1103515625, 10000.208984375, 10000.3955078125, 10000.6181640625, 10000.484375, 10000.037109375, 10000.4169921875, 9999.53125, 10000.181640625, 10001.45703125, 9999.44921875, 9999.9833984375, 9998.9072265625, 9998.50390625, 10000.5859375, 9997.8779296875, 10001.0380859375, 9999.2685546875, 10000.642578125, 9999.587890625, 9999.6416015625, 10000.5830078125, 9999.51953125, 10000.3671875, 9997.6904296875, 9998.08203125, 10001.005859375, 9999.0595703125, 9999.8662109375, 10001.921875, 9998.4658203125, 9999.9775390625, 9999.203125, 10000.8994140625, 10000.5439453125, 9999.560546875, 10001.6708984375, 10000.525390625, 9998.80859375, 10001.1103515625, 9999.0283203125, 10001.3876953125, 10001.7421875, 9998.681640625, 10000.375, 9999.2314453125, 9999.2060546875, 10000.0029296875, 10000.3798828125, 10001.94140625, 9999.9423828125, 10000.208984375, 9998.1943359375, 10000.63671875, 10000.373046875, 9999.8515625, 10000.0498046875, 9998.0908203125, 10000.671875, 9999.9921875, 9999.837890625, 10000.373046875, 9999.662109375, 9999.9501953125, 9999.68359375, 10001.236328125, 9998.796875, 10001.0048828125, 10001.8125, 10000.056640625, 10000.404296875, 10001.4287109375, 9999.3720703125, 9999.50390625, 9999.439453125, 10000.736328125, 10001.13671875, 10001.171875, 10000.291015625, 9999.5615234375, 9999.7783203125, 10000.7646484375, 9998.8828125, 9999.0634765625, 9998.8271484375, 9999.623046875, 9998.9677734375, 10000.4443359375, 10001.3154296875, 9998.8828125, 10000.81640625, 10000.4951171875, 10000.3046875, 10000.19921875, 9998.52734375, 9999.55859375, 9998.548828125, 10000.3056640625, 9999.4384765625, 10000.07421875, 9997.376953125, 9999.6875, 9999.822265625, 9999.294921875, 10000.7119140625, 10000.1845703125, 9999.84375, 10000.28125, 9999.8603515625, 10000.7548828125, 9998.2890625, 9998.58203125, 9999.3037109375, 10000.994140625, 10000.5693359375, 10001.53125, 10001.615234375, 9998.8017578125, 9999.98046875, 9999.6416015625, 9999.927734375, 10000.0830078125, 9999.9541015625, 9999.8046875, 10000.1044921875, 9999.2890625, 10001.052734375, 9999.7314453125, 10000.27734375, 10000.095703125, 9998.966796875, 9999.494140625, 10000.626953125, 10001.7890625, 9999.6875, 9999.923828125, 9999.5849609375, 9998.076171875, 9999.587890625, 9999.6083984375, 10001.5888671875, 10000.59375, 10000.1416015625, 9999.525390625, 9999.5888671875, 10000.20703125, 9999.66796875, 10000.4599609375, 10001.20703125, 9998.103515625, 10000.1962890625, 10001.767578125, 9997.90625, 10001.0771484375, 10001.806640625, 10001.1005859375, 9999.3486328125, 10000.328125, 10000.0068359375, 9999.2783203125, 10000.1357421875, 10000.994140625, 10000.919921875, 10000.5654296875, 9999.716796875, 9999.4208984375, 10001.7861328125, 9997.5947265625, 9999.2822265625, 9997.669921875, 9998.681640625, 9999.4287109375, 9999.2001953125, 10000.6728515625, 9999.8671875, 9999.9033203125, 10000.3466796875, 10002.62109375, 9999.5185546875, 9999.349609375, 10000.1005859375, 9999.2099609375, 9998.21875, 9999.48828125, 10002.13671875, 10000.6318359375, 9999.740234375, 9998.2900390625, 10001.44921875, 10000.6103515625, 9999.94921875, 10000.9384765625, 9999.642578125, 9999.482421875, 10000.365234375, 9999.083984375, 10000.0478515625, 9999.8740234375, 9999.7998046875, 10001.31640625, 9998.2578125, 9999.310546875, 10000.162109375, 10000.69140625, 10001.314453125, 10000.7861328125, 10001.3759765625, 10001.037109375, 10000.1513671875, 9999.9912109375, 9999.482421875, 9999.5537109375, 9999.3876953125, 10001.5869140625, 10000.189453125, 9998.90234375, 9999.330078125, 9999.3203125, 10001.150390625, 9999.810546875, 9998.75390625, 10000.025390625, 9999.31640625, 9998.8681640625, 10001.2861328125, 10000.703125, 10001.2490234375, 9999.1728515625, 9999.8896484375, 10000.2724609375, 9998.7333984375, 9999.3330078125, 10000.1904296875, 9999.72265625, 10000.7724609375, 10000.5341796875, 9999.47265625, 10000.0830078125, 10000.3173828125, 10000.2216796875, 10000.5908203125, 10000.7275390625, 9998.1845703125, 9998.5546875, 9997.91015625, 10000.828125, 9999.68359375, 10001.4921875, 10000.6591796875, 9999.373046875, 10001.076171875, 9999.138671875, 9999.5693359375, 9998.6328125, 10000.6337890625, 9998.29296875, 10001.3564453125, 10001.375, 10001.6767578125, 9999.6171875, 10000.26171875, 9999.0009765625, 10000.6767578125, 9998.5595703125, 9999.603515625, 9998.30859375, 10000.39453125, 9999.1962890625, 9999.2646484375, 10000.017578125, 9999.77734375, 9999.427734375, 9998.654296875, 10001.3212890625, 10000.3603515625, 10000.5009765625, 10000.044921875, 10000.2509765625, 10000.0732421875, 10000.0224609375, 10000.9384765625, 9998.9443359375, 9999.376953125, 9997.9462890625, 10001.375, 9999.580078125, 9999.384765625, 9999.900390625, 9999.2734375, 9999.9384765625, 10000.27734375, 9999.53125, 9998.798828125, 9999.890625, 10000.39453125, 10000.8212890625, 9999.28125, 9998.02734375, 9999.416015625, 10000.728515625, 9999.52734375, 10000.212890625, 10000.0498046875, 9999.1689453125, 10001.3896484375, 10001.9560546875, 9999.548828125, 9999.798828125, 10000.2119140625, 10000.5302734375, 9999.8251953125, 9998.50390625, 10001.4560546875, 9999.7705078125, 9999.9365234375, 10001.7998046875, 10000.3388671875, 9999.9970703125, 9999.94140625, 10000.77734375, 10000.2705078125, 10000.4150390625, 10000.9296875, 10000.1103515625, 9999.849609375, 9998.107421875, 10003.2626953125, 9999.9287109375, 9999.669921875, 10000.2763671875, 9999.9560546875, 10000.9189453125, 9999.2275390625, 9999.4814453125, 9999.296875, 10000.9658203125, 9999.814453125, 10001.3798828125, 10001.09375, 9998.7529296875, 9999.1630859375, 9999.29296875, 9999.609375, 10000.1982421875, 9998.8837890625, 9999.9306640625, 9999.7880859375, 9999.642578125, 9999.3759765625, 10000.67578125, 9999.1171875, 9999.857421875, 9998.5556640625, 10001.3369140625, 9999.6357421875, 10001.31640625, 9999.2509765625, 10000.744140625, 9998.234375, 10001.083984375, 10000.2275390625, 9998.8544921875, 9998.8212890625, 9999.8095703125, 10000.0634765625, 9998.828125, 10000.6748046875, 10000.5654296875, 10000.4306640625, 10000.900390625, 9998.115234375, 10001.4033203125, 9999.1630859375, 9999.6083984375, 9999.5830078125, 10000.7490234375, 10000.4482421875, 9999.69921875, 10000.552734375, 9999.447265625, 9998.5205078125, 10000.408203125, 10000.4384765625, 10000.8681640625, 9999.57421875, 10001.8115234375, 9998.7783203125, 10001.931640625, 10000.755859375, 9999.6552734375, 9998.86328125, 9998.9892578125, 10000.341796875, 9998.72265625, 9999.5234375, 10000.2568359375, 10000.9716796875, 10000.6435546875, 10000.876953125, 10000.3583984375, 9999.306640625, 10000.009765625, 9999.5859375, 9998.759765625, 9999.4560546875, 9998.7236328125, 9998.8310546875, 10001.673828125, 9998.958984375, 10001.890625, 10000.1396484375, 10000.2421875, 9999.8212890625, 10000.435546875, 10000.5048828125, 9998.7587890625, 10000.1787109375, 9999.451171875, 9999.556640625, 9999.8046875, 10000.349609375, 10002.2158203125, 10001.06640625, 9999.3115234375, 9999.6015625, 9998.8896484375, 10000.255859375, 9998.0048828125, 9999.1484375, 9999.9755859375, 10002.2236328125, 10000.294921875, 10001.3984375, 10001.9228515625, 9999.9619140625, 10000.91015625, 9999.171875, 10000.765625, 9999.482421875, 10000.490234375, 9999.20703125, 9999.630859375, 10002.021484375, 9999.5830078125, 10000.173828125, 10000.3154296875, 10000.2666015625, 10001.373046875, 10001.4697265625, 9998.7216796875, 9999.7216796875, 9999.38671875, 10001.1748046875, 10001.7890625, 9999.6064453125, 9999.501953125, 9999.025390625, 10002.1181640625, 10000.91015625, 10000.0576171875, 10000.337890625, 9999.9658203125, 9999.5458984375, 9999.564453125, 10000.19140625, 9999.748046875, 9999.3525390625, 9999.4462890625, 10000.7353515625, 10001.951171875, 9999.4296875, 10000.0478515625, 10000.875, 9999.3544921875, 10001.3525390625, 10000.84375, 10000.7431640625, 9999.857421875, 10001.5263671875, 9999.7158203125, 10000.4404296875, 10000.2421875, 10001.140625, 9997.9775390625, 9998.5234375, 10001.181640625, 9998.310546875, 10000.2021484375, 9998.361328125, 10000.701171875, 9998.6630859375, 10001.25, 9998.5302734375, 9999.873046875, 10000.8564453125, 10001.6865234375, 10000.0732421875, 10000.216796875, 9998.701171875, 9999.146484375, 10000.4970703125, 9999.384765625, 9998.287109375, 9999.56640625, 10000.3134765625, 9998.88671875, 10000.8974609375, 10000.634765625, 9998.900390625, 10000.412109375, 9997.814453125, 9998.4453125, 10000.6298828125, 9999.529296875, 9998.9404296875, 10000.45703125, 10000.2451171875, 10000.5810546875, 10000.4521484375, 10000.984375, 10000.244140625, 10001.0419921875, 10002.1025390625, 9999.2255859375, 10000.3740234375, 9999.5869140625, 10001.2109375, 10001.1357421875, 10000.5146484375, 10001.6591796875, 10000.583984375, 9998.1064453125, 9999.822265625, 10000.443359375, 10000.587890625, 10000.2041015625, 10001.9267578125, 9999.5390625, 10001.3046875, 10000.4990234375, 10000.0185546875, 10000.24609375, 10001.37109375, 10000.8740234375, 9998.4033203125, 10001.4345703125, 9997.78125, 10000.1494140625, 10000.2509765625, 10001.0830078125, 9999.85546875, 9998.5048828125, 9999.9345703125, 9999.2509765625, 10000.3173828125, 9998.6328125, 9999.6904296875, 9999.7548828125, 10002.912109375, 10000.1640625, 10000.7138671875, 9999.642578125, 10001.109375, 9999.693359375, 10000.4814453125, 9999.681640625, 10000.421875, 10000.802734375, 9999.7841796875, 10003.4365234375, 9999.6767578125, 9999.548828125, 9999.1826171875, 10000.025390625, 10000.8447265625, 9999.80859375, 9999.5498046875, 9999.2041015625, 10000.5908203125, 9999.951171875, 9999.8193359375, 9999.8642578125, 9999.5517578125, 9999.6083984375, 9999.1748046875, 10000.25, 9999.4638671875, 10000.75390625, 10000.2490234375, 9998.3701171875, 10001.5, 9999.603515625, 10000.8330078125, 9999.80859375, 10000.1884765625, 10000.9111328125, 9999.6259765625, 9998.88671875, 9999.2236328125, 10001.3603515625, 9999.8291015625, 9998.603515625, 10000.046875, 9998.96875, 9998.689453125, 10000.712890625, 10000.978515625, 9999.771484375, 9998.857421875, 9998.005859375, 10000.30078125, 10000.82421875, 9998.734375, 10001.7275390625, 9997.1015625, 9998.9775390625, 10001.3525390625, 10001.326171875, 9998.361328125, 10000.955078125, 10001.4453125, 10000.87890625, 10000.080078125, 10000.197265625, 9998.474609375, 10000.041015625, 9999.8984375, 9998.91015625, 9998.90234375, 9999.328125, 9999.71875, 10001.1142578125, 10000.1875, 10000.59375, 10001.2060546875, 9998.8330078125, 10000.021484375, 9999.5791015625, 9999.2763671875, 10000.78515625, 9999.935546875, 10001.66796875, 9999.5673828125, 10001.4306640625, 9998.9658203125, 10000.7548828125, 9999.451171875, 10000.8134765625, 10000.841796875, 10001.2294921875, 9999.5439453125, 10000.59375, 9999.29296875, 9999.3505859375, 9999.9521484375, 9997.9833984375, 10000.625, 9999.173828125, 10000.42578125, 9999.03515625, 10000.0869140625, 9998.7900390625, 10000.86328125, 10000.47265625, 9999.826171875, 9999.837890625, 9998.765625, 9999.5849609375, 10001.638671875, 10000.2314453125, 10001.4482421875, 10001.5400390625, 9997.1015625, 10000.4423828125, 10000.46875, 10001.3046875, 10000.1806640625, 9997.5732421875, 10000.4990234375, 10002.1181640625, 9999.5205078125, 9999.13671875, 10000.271484375, 10000.279296875, 9998.76953125, 10001.55078125, 10002.36328125, 9999.880859375, 9998.8193359375, 9999.3623046875, 10000.7255859375, 9997.421875, 10000.052734375, 9998.8857421875, 9998.322265625, 9999.39453125, 10000.5966796875, 9999.654296875, 10000.0595703125, 10001.8896484375, 9999.7958984375, 9998.8916015625, 10000.0009765625, 9999.5927734375, 10001.064453125, 9999.001953125, 9998.9072265625, 9999.3828125, 10000.1533203125, 10000.90625, 10000.9853515625, 10001.4794921875, 10000.4287109375, 9999.208984375, 10000.12890625, 10001.6435546875, 9999.111328125, 9999.4052734375, 9998.78125, 9999.724609375, 9999.3076171875, 9999.1494140625, 9999.49609375, 10000.24609375, 10001.2333984375, 10000.142578125, 10000.1484375, 10000.0146484375, 9999.8115234375, 10000.2236328125, 10000.2724609375, 9999.3955078125, 9999.8564453125, 9999.548828125, 10000.5888671875, 10000.2314453125, 10000.4365234375, 10002.49609375, 9999.259765625, 10002.4150390625, 10001.05859375, 10001.076171875, 10000.1865234375, 10001.1044921875, 10001.0234375, 10000.8984375, 9997.6337890625, 9998.8564453125, 10001.4775390625, 10001.5703125, 10000.337890625, 9999.87890625, 10000.4189453125, 9998.9794921875, 10001.5927734375, 10000.2314453125, 10000.162109375, 10000.9619140625, 9999.029296875, 9999.6904296875, 10000.8544921875, 10000.7607421875, 10000.8828125, 9999.740234375, 9999.5556640625, 10000.2373046875, 9999.982421875, 10001.2763671875, 9999.2685546875, 10000.419921875, 10000.724609375, 10002.6513671875, 9999.0478515625, 10001.2578125, 10001.0888671875, 9999.93359375, 10000.4619140625, 10000.802734375, 9999.880859375, 9997.814453125, 9999.9873046875, 9999.73828125, 9998.8271484375, 10001.5517578125, 9998.7119140625, 10000.3388671875, 9999.4521484375, 9999.240234375, 9999.7060546875, 9999.48046875, 9998.1865234375, 9998.921875, 9998.98828125, 9999.6376953125, 10001.197265625, 9998.5205078125, 9998.708984375, 10000.263671875, 9999.8681640625, 9999.7392578125, 10000.82421875, 10001.9873046875, 10000.6728515625, 9999.255859375, 9999.998046875, 9999.787109375, 9999.875, 9999.8232421875, 9999.6748046875, 9999.6513671875, 10000.1044921875, 10001.3818359375, 10000.53125, 10001.35546875, 10000.150390625, 10000.5390625, 9998.982421875, 10000.1474609375, 10001.0439453125, 9999.4755859375, 10000.4267578125, 10000.4638671875, 9998.689453125, 9999.7138671875, 10001.7119140625, 9997.958984375, 10001.1845703125, 10000.26171875, 10002.294921875, 9999.7626953125, 10000.888671875, 9999.0625, 10001.0556640625, 10000.8291015625, 9999.6025390625, 10000.8681640625, 10000.37109375, 9999.1787109375, 9999.81640625, 9998.10546875, 9999.2587890625, 9999.3779296875, 10001.666015625, 9999.4482421875, 10000.546875, 10000.50390625, 10001.3466796875, 9999.4931640625, 10000.1552734375, 9999.10546875, 10000.51953125, 10001.0546875, 10000.8505859375, 9998.8671875, 9998.212890625, 9999.1025390625, 9999.4306640625, 10000.63671875, 10000.46875, 10000.6298828125, 9997.603515625, 9999.853515625, 10000.54296875, 9999.7109375, 9998.7919921875, 9999.138671875, 10002.1982421875, 9999.8525390625, 10000.482421875, 9999.2861328125, 9999.626953125, 9999.4638671875, 10000.38671875, 9999.7353515625, 9999.3662109375, 9998.2548828125, 9999.4814453125, 9999.462890625, 9997.873046875, 9998.859375, 9997.9267578125, 10000.22265625, 10000.359375, 10000.9931640625, 9999.9228515625, 10000.6689453125, 10000.3955078125, 9999.4501953125, 10000.357421875, 9999.48828125, 9998.53125, 10001.0537109375, 10001.3642578125, 10000.5498046875, 9999.953125, 9999.0859375, 10000.09765625, 9998.765625, 9999.57421875, 10000.28515625, 9998.53515625, 9998.142578125, 9999.330078125, 9997.923828125, 9999.34765625, 10000.2744140625, 9999.0654296875, 10000.7822265625, 10001.392578125, 10000.9921875, 10000.2294921875, 9999.4462890625, 10000.8974609375, 9997.6806640625, 9999.5341796875, 9998.900390625, 9998.7333984375, 10001.3310546875, 10000.171875, 9999.0283203125, 9997.9755859375, 10000.6025390625, 9999.7294921875, 9999.6162109375, 9999.173828125, 10000.1689453125, 10000.314453125, 9997.6796875, 9999.5380859375, 9999.615234375, 9998.7431640625, 9999.3017578125, 10001.767578125, 9998.3974609375, 9999.87109375, 10001.3203125, 10001.2841796875, 10001.1240234375, 9999.9736328125, 10000.39453125, 10002.3837890625, 9998.3232421875, 9997.490234375, 9999.1103515625, 9999.7900390625, 10000.7880859375, 9998.912109375, 9998.4892578125, 9999.2099609375, 10000.0771484375, 9999.5537109375, 9999.6455078125, 10000.7900390625, 9998.833984375, 9998.8330078125, 9999.8544921875, 10000.3564453125, 10001.6123046875, 9999.890625, 10000.859375, 10001.9736328125, 10001.6318359375, 9999.521484375, 10000.568359375, 9998.1025390625, 9999.599609375, 10000.5146484375, 10001.59375, 10000.2177734375, 10001.677734375, 10000.7236328125, 10000.716796875, 10000.791015625, 9998.3017578125, 10000.97265625, 9999.9248046875, 10000.5390625, 10000.5107421875, 9999.3369140625, 9999.2578125, 10000.048828125, 9999.263671875, 9999.9912109375, 10000.1435546875, 9999.212890625, 9999.4228515625, 9999.126953125, 10000.375, 9999.4140625, 10000.8369140625, 9999.0751953125, 10000.294921875, 10000.1455078125, 9998.123046875, 9999.7822265625, 10002.41796875, 10000.4541015625, 9999.658203125, 10000.373046875, 9998.904296875, 9999.408203125, 10001.0625, 10000.546875, 9999.486328125, 9999.58203125, 9999.453125, 10000.384765625, 10000.9052734375, 9999.478515625, 9999.150390625, 9998.5849609375, 9998.546875, 10001.0517578125, 9999.552734375, 9997.486328125, 10001.0986328125, 10000.8291015625, 9999.7021484375, 10001.6142578125, 10000.6357421875, 9998.5673828125, 9999.6865234375, 9999.8212890625, 10000.4619140625, 9999.2568359375, 9997.9599609375, 10000.0341796875, 10001.1611328125, 9997.9404296875, 9999.1474609375, 9997.8525390625, 9999.0048828125, 9999.62890625, 9998.6103515625, 9999.96484375, 9998.1787109375, 9999.2275390625, 10000.2841796875, 9999.9990234375, 10001.48828125, 9999.0224609375, 10000.9541015625, 9999.759765625, 9998.5634765625, 9999.873046875, 10000.755859375, 9999.68359375, 10000.0673828125, 10000.4453125, 9997.8671875, 10000.1826171875, 10000.4697265625, 9999.69140625, 10000.8876953125, 10000.1181640625, 9998.39453125, 9998.9638671875, 9999.5419921875, 9999.3525390625, 10000.94921875, 10000.7392578125, 9998.1005859375, 10000.4248046875, 10001.7998046875, 10001.458984375, 9999.7919921875, 9999.2529296875, 9997.7861328125, 9999.96875, 10000.396484375, 9999.1328125, 9999.5654296875, 10001.333984375, 9999.94921875, 10000.9052734375, 9999.9697265625, 9998.4609375, 10000.7744140625, 10000.1181640625, 10000.71484375, 9999.5400390625, 9999.3759765625, 10000.0771484375, 10001.6005859375, 10000.6728515625, 10000.6640625, 9999.3525390625, 10001.6142578125, 10000.318359375, 10001.0, 10001.2392578125, 9999.13671875, 9999.78515625, 10000.322265625, 10000.4697265625, 9999.0703125, 9999.3349609375, 9998.0224609375, 9998.5068359375, 10000.146484375, 10001.6767578125, 10000.2060546875, 10000.4404296875, 10002.1142578125, 9999.4091796875, 9998.7119140625, 10001.7158203125, 10000.412109375, 10000.060546875, 9999.9912109375, 10001.3349609375, 9999.517578125, 10000.8984375, 9999.056640625, 9998.3701171875, 10000.0869140625, 10000.158203125, 10000.2783203125, 9999.033203125, 9998.7900390625, 9999.775390625, 10001.1513671875, 10000.6962890625, 10000.3330078125, 10000.4814453125, 10001.08984375, 9998.767578125, 10000.0244140625, 10000.802734375, 10001.0625, 10001.0087890625, 10000.2890625, 9998.7197265625, 9999.8837890625, 10000.8310546875, 9999.1474609375, 9998.890625, 9998.30078125, 10000.251953125, 9999.91015625, 10000.2294921875, 10002.236328125, 10001.015625, 10000.482421875, 10001.6552734375, 10000.048828125, 10000.5771484375, 9999.9755859375, 10000.4521484375, 9998.8203125, 9999.6044921875, 9999.138671875, 9998.0673828125, 9998.236328125, 9998.302734375, 9998.23046875, 10001.0615234375, 10001.62890625, 10001.5634765625, 10000.3134765625, 9999.68359375, 9998.5048828125, 9999.095703125, 10002.4365234375, 9999.357421875, 9998.8662109375, 10000.29296875, 9998.6064453125, 10001.0390625, 9998.4931640625, 10000.220703125, 9998.892578125, 9999.7763671875, 10000.328125, 10000.9765625, 10001.390625, 10001.0869140625, 10000.779296875, 10000.9482421875, 10001.1103515625, 10000.9345703125, 10001.4951171875, 9999.541015625, 10000.166015625, 9999.8720703125, 10000.90234375, 9999.3056640625, 9999.138671875, 9999.634765625, 9999.0078125, 10000.4091796875, 10000.6123046875, 9999.310546875, 10001.7802734375, 10000.7744140625, 10000.693359375, 9999.9931640625, 10000.490234375, 10002.01171875, 9999.837890625, 10000.6396484375, 10000.015625, 9998.57421875, 9998.3642578125, 9998.974609375, 9998.9921875, 10001.3095703125, 9999.51171875, 10000.3828125, 9999.7197265625, 10000.1416015625, 10000.130859375, 9997.7392578125, 9999.3525390625, 9997.859375, 9999.873046875, 10001.291015625, 9999.5576171875, 9999.9873046875, 10000.4814453125, 9999.1494140625, 10000.833984375, 9999.1455078125, 10001.35546875, 10000.2939453125, 9998.796875, 9999.068359375, 10001.189453125, 9997.8017578125, 9998.83203125, 9999.7900390625, 10000.5439453125, 9999.013671875, 9998.623046875, 10001.6357421875, 9999.2470703125, 9998.6826171875, 10001.62109375, 9999.2275390625, 9999.697265625, 9999.9384765625, 10000.2412109375, 10000.4736328125, 10000.220703125, 10000.1826171875, 10000.2509765625, 10000.796875, 10000.8701171875, 9998.9892578125, 9999.8505859375, 10001.34765625, 9999.75, 10000.6728515625, 9999.8427734375, 9999.125, 9999.4677734375, 10000.1064453125, 10001.33203125, 10000.533203125, 9999.2978515625, 10000.904296875, 10000.2158203125, 9997.8857421875, 10001.498046875, 10001.2646484375, 10001.0791015625, 10001.2138671875, 9998.853515625, 10000.095703125, 10001.6376953125, 10000.068359375, 9999.28125, 9998.8974609375, 9999.6474609375, 9998.96875, 9999.0703125, 9998.8564453125, 10000.74609375, 9998.953125, 10000.896484375, 9999.484375, 10001.111328125, 9999.1962890625, 9999.44921875, 10001.486328125, 9999.0361328125, 10000.34375, 10001.984375, 9999.4775390625, 10000.7255859375, 9999.322265625, 9999.333984375, 9999.396484375, 10000.5927734375, 10000.2421875, 9999.57421875, 9999.357421875, 10000.1181640625, 10000.6533203125, 10000.3662109375, 9999.8447265625, 10000.943359375, 10002.64453125, 10001.7880859375, 10002.0205078125, 9999.8369140625, 9999.216796875, 10000.1552734375, 10000.0302734375, 10000.1787109375, 10000.982421875, 10000.103515625, 10000.4130859375, 10000.7607421875, 9999.5107421875, 10000.60546875, 10000.6513671875, 9999.3798828125, 9999.7724609375, 10001.53515625, 9999.6904296875, 9999.724609375, 9999.638671875, 9999.8828125, 9999.75390625, 10000.916015625, 10000.994140625, 10000.3486328125, 10000.603515625, 10001.0, 9999.6171875, 9999.517578125, 10001.0712890625, 10001.3681640625, 9998.8525390625, 10001.0283203125, 10000.2607421875, 10001.4638671875, 10000.4404296875, 9998.9140625, 9998.9921875, 10000.359375, 10001.103515625, 9999.06640625, 9998.5751953125, 10000.1767578125, 9999.9482421875, 10002.787109375, 9999.2021484375, 9999.447265625, 9999.0048828125, 9999.263671875, 9999.7275390625, 10000.94140625, 10000.5771484375, 10000.826171875, 9999.076171875, 9999.482421875, 10000.6015625, 10000.1142578125, 9999.455078125, 9999.8701171875, 10000.9794921875, 10001.2841796875, 10001.7041015625, 10000.2041015625, 10002.1904296875, 10000.06640625, 10000.5546875, 10001.0244140625, 9999.42578125, 10000.21484375, 9999.72265625, 10001.94921875, 10001.0517578125, 9998.68359375, 10000.3642578125, 10000.978515625, 10001.97265625, 10000.2021484375, 9997.658203125, 10000.1796875, 10001.4541015625, 10000.423828125, 9999.1162109375, 9999.650390625, 9999.646484375, 10001.9921875, 10000.2060546875, 9998.86328125, 9999.484375, 10001.6201171875, 9997.078125, 9998.052734375, 9999.134765625, 9999.8837890625, 9998.294921875, 10001.283203125, 9999.3505859375, 9998.6435546875, 9999.4658203125, 9997.7314453125, 9999.390625, 10001.7578125, 9998.7900390625, 9999.9658203125, 9999.880859375, 9999.380859375, 10001.3583984375, 10003.080078125, 9998.4189453125, 9999.8994140625, 9999.1142578125, 9999.705078125, 10000.9150390625, 10003.302734375, 10000.21875, 9999.98046875, 9998.775390625, 9999.548828125, 10001.134765625, 9999.6123046875, 9998.7607421875, 9999.1162109375, 9998.228515625, 9999.205078125, 10000.9580078125, 10001.4296875, 10001.6357421875, 9999.56640625, 9999.310546875, 10001.4130859375, 9998.3388671875, 10001.171875, 10001.1318359375, 9999.6181640625, 9999.7314453125, 10003.61328125, 10000.345703125, 10000.1708984375, 10002.1533203125, 10000.1171875, 9999.0888671875, 10000.4384765625, 10001.0, 9999.9677734375, 10000.6875, 10001.6474609375, 9999.654296875, 9999.6201171875, 10000.4716796875, 10001.1533203125, 9998.5830078125, 10001.0625, 9997.3447265625, 10000.73046875, 9998.3447265625, 10000.50390625, 10000.919921875, 9998.6220703125, 9998.7314453125, 10000.423828125, 10000.5302734375, 10000.2705078125, 9998.8818359375, 10000.1591796875, 9999.9384765625, 10000.8779296875, 10000.6044921875, 10000.6416015625, 9998.85546875, 10001.091796875, 10001.658203125, 10000.556640625, 9999.951171875, 9998.9462890625, 9999.57421875, 9998.91015625, 10000.8515625, 10000.2861328125, 9999.3701171875, 10001.2744140625, 10000.1640625, 10000.5478515625, 9999.158203125, 9999.7509765625, 9999.912109375, 9999.7822265625, 10001.1474609375, 9999.98828125, 9999.70703125, 10001.6337890625, 10000.640625, 10001.5341796875, 9999.5673828125, 9998.3212890625, 9998.8759765625, 10000.3916015625, 10000.2802734375, 10000.83984375, 10001.0732421875, 10000.005859375, 9999.68359375, 10000.72265625, 9999.41796875, 9999.23828125, 9999.9208984375, 10000.6396484375, 10000.2890625, 9999.529296875, 10000.3076171875, 9999.02734375, 10000.3984375, 9998.2626953125, 10001.16796875, 9999.0146484375, 10000.048828125, 9999.0322265625, 9999.8994140625, 10000.5869140625, 9998.0126953125, 10001.541015625, 9998.9677734375, 9998.630859375, 10000.587890625, 9998.9072265625, 9999.95703125, 9999.9287109375, 10001.390625, 10000.0595703125, 9999.9326171875, 9998.3466796875, 9999.384765625, 9999.3291015625, 10000.142578125, 10000.873046875, 9998.3359375, 9999.8349609375, 10001.2626953125, 9999.1337890625, 10000.2587890625, 10000.5087890625, 10002.24609375, 10000.84765625, 9999.697265625, 9999.3974609375, 9999.998046875, 10001.0556640625, 10001.5185546875, 9999.7783203125, 10000.2470703125, 9999.2861328125, 10000.38671875, 9998.9990234375, 10000.154296875, 9999.6728515625, 9999.580078125, 10000.98828125, 9998.3857421875, 10000.7080078125, 9999.82421875, 10000.552734375, 9999.4970703125, 9998.6796875, 9999.6025390625, 9998.9765625, 9998.7744140625, 9998.79296875, 10000.0244140625, 9998.1298828125, 9999.5302734375, 10001.4306640625, 10001.0771484375, 9999.3544921875, 10001.1484375, 9998.9736328125, 10000.232421875, 10000.1357421875, 10001.09375, 9998.5078125, 10002.5009765625, 10000.6650390625, 10000.755859375, 10000.544921875, 10000.5078125, 10001.2900390625, 10000.52734375, 9999.5419921875, 9998.556640625, 10001.6005859375, 10000.8857421875, 9999.755859375, 9999.4873046875, 10000.5029296875, 9999.060546875, 10000.4677734375, 10001.349609375, 9998.482421875, 9999.65234375, 10000.640625, 10001.1416015625, 10000.6845703125, 9999.98828125, 9999.1328125, 9999.9921875, 9997.3935546875, 9998.9208984375, 10000.7353515625, 9999.708984375, 9999.3876953125, 10001.3798828125, 9999.2900390625, 10000.654296875, 10000.8203125, 10000.1591796875, 10000.08984375, 9999.0966796875, 10001.7333984375, 10000.056640625, 9999.5341796875, 10001.1689453125, 10000.3828125, 10000.857421875, 9999.205078125, 10001.0732421875, 9999.720703125, 9999.1201171875, 10000.7607421875, 9999.5439453125, 9999.50390625, 9999.2109375, 10001.4033203125, 9998.8095703125, 9999.3408203125, 10000.3232421875, 10001.02734375, 9998.546875, 10001.5439453125, 10000.4208984375, 9998.6123046875, 10000.25, 10000.296875, 9999.83203125, 9999.9326171875, 9998.958984375, 10000.77734375, 10000.84765625, 10001.90625, 9997.775390625, 10000.466796875, 9999.6123046875, 10001.1259765625, 9998.3544921875, 9997.8955078125, 10000.41796875, 9999.8876953125, 10000.025390625, 10000.6416015625, 10000.111328125, 9998.1220703125, 10000.0712890625, 10000.724609375, 10001.685546875, 10000.1015625, 10000.1298828125, 9998.876953125, 10000.4052734375, 9999.5595703125, 9999.544921875, 9999.05859375, 10001.9091796875, 10000.794921875, 9999.0185546875, 10000.40625, 9999.701171875, 9999.20703125, 10001.59765625, 9999.583984375, 9999.1875, 9998.4833984375, 10000.1494140625, 10000.814453125, 10000.0048828125, 10001.0380859375, 9999.548828125, 9998.37890625, 9999.009765625, 10000.9677734375, 9999.0556640625, 9997.265625, 9999.4931640625, 9998.97265625, 9999.7490234375, 10000.6953125, 10000.9697265625, 10000.005859375, 9999.318359375, 10000.4306640625, 10000.216796875, 10000.8134765625, 10000.107421875, 9999.9560546875, 9999.6025390625, 9999.7041015625, 10000.69140625, 10000.8662109375, 9999.525390625, 10000.3720703125, 10002.1396484375, 9999.7607421875, 9999.2724609375, 9998.8583984375, 9999.7861328125, 10001.59375, 9998.9208984375, 9998.5966796875, 10000.76171875, 9999.15625, 10001.400390625, 10001.3466796875, 10000.6162109375, 9999.25390625, 9999.228515625, 9999.8330078125, 10000.708984375, 9999.0283203125, 10000.146484375, 9998.8759765625, 9999.611328125, 10000.2861328125, 10000.0205078125, 10000.4716796875, 9999.267578125, 9998.0859375, 10000.6875, 10001.5751953125, 9997.76953125, 10000.9140625, 10000.076171875, 10000.8115234375, 9999.921875, 9999.853515625, 10000.02734375, 10000.41015625, 10000.0205078125, 10000.892578125, 9998.712890625, 9999.9638671875, 9999.9287109375, 9999.9619140625, 10000.78515625, 10000.1943359375, 10001.6357421875, 10001.12109375, 10000.224609375, 10000.7509765625, 10000.134765625, 9998.8583984375, 9999.5771484375, 10001.689453125, 10000.96875, 9999.044921875, 10000.6689453125, 9999.255859375, 9999.2373046875, 10000.0078125, 10001.52734375, 9998.982421875, 10000.7021484375, 10001.1220703125, 9997.7880859375, 9999.6044921875, 10000.44140625, 9998.8955078125, 10001.666015625, 9999.0654296875, 10001.583984375, 9999.4912109375, 9999.482421875, 10001.7734375, 10000.4052734375, 9999.6015625, 9999.126953125, 10001.2509765625, 10000.9091796875, 9998.21875, 9999.525390625, 9999.607421875, 10000.029296875, 10000.0009765625, 9999.0859375, 9998.8720703125, 9998.8232421875, 10000.7685546875, 9999.92578125, 10000.5791015625, 10000.1953125, 10000.900390625, 10000.9873046875, 9999.998046875, 9999.4150390625, 10000.12890625, 9999.68359375, 10001.251953125, 9999.947265625, 9999.751953125, 10000.0078125, 10001.3291015625, 9999.201171875, 9999.8056640625, 10000.1875, 10002.0556640625, 9999.591796875, 9999.7802734375, 9999.0166015625, 10001.6240234375, 10000.634765625, 10000.4208984375, 10000.9306640625, 10000.0576171875, 9999.6640625, 9998.53125, 10000.0341796875, 9999.255859375, 10001.4013671875, 9999.4716796875, 9999.25, 9999.1103515625, 10000.5517578125, 10001.8359375, 10000.9716796875, 9997.7646484375, 10000.2216796875, 10000.2802734375, 10002.3681640625, 10000.4951171875, 9998.62109375, 9999.3701171875, 9998.833984375, 10000.7626953125, 10001.0048828125, 10001.4794921875, 10001.0673828125, 9999.0048828125, 9999.4130859375, 10000.0712890625, 9999.9404296875, 10000.2119140625, 10002.0869140625, 9999.36328125, 9999.7021484375, 9999.611328125, 9999.107421875, 9998.7177734375, 9999.2578125, 9998.55859375, 10001.0849609375, 9998.693359375, 9999.08984375, 9998.361328125, 10000.1611328125, 9999.0185546875, 10001.076171875, 9998.794921875, 10000.5087890625, 9997.416015625, 9999.232421875, 9999.212890625, 9999.6787109375, 9998.8125, 10000.4287109375, 9999.119140625, 9999.8095703125, 10000.529296875, 10000.0859375, 10000.970703125, 9999.59765625, 9999.16015625, 10000.5400390625, 9999.908203125, 9999.3564453125, 10000.4189453125, 10000.5361328125, 10000.724609375, 10000.8740234375, 9999.16796875, 9999.7626953125, 9999.4296875, 9999.8330078125, 9998.75, 10000.1064453125, 10001.0185546875, 10000.447265625, 9998.8818359375, 9999.58984375, 10000.2880859375, 10001.6376953125, 9999.5390625, 10000.0078125, 9999.5498046875, 9999.0654296875, 10000.1298828125, 10000.478515625, 9999.7607421875, 10001.0859375, 9999.748046875, 9999.650390625, 10000.5419921875, 10001.130859375, 9999.8017578125, 10001.2119140625, 10000.9873046875, 9999.984375, 10001.5283203125, 10000.9150390625, 10000.9287109375, 10000.7578125, 9997.8701171875, 9999.6298828125, 10002.1005859375, 9999.5400390625, 10001.0859375, 9998.5244140625, 10000.5751953125, 9999.203125, 9999.7412109375, 9998.9931640625, 10000.193359375, 9999.5302734375, 10000.2294921875, 9999.7763671875, 10000.48828125, 9999.3330078125, 9999.94140625, 9997.6953125, 10000.40625, 9999.6337890625, 10000.478515625, 10000.3759765625, 10001.244140625, 9998.5380859375, 9999.267578125, 9998.2685546875, 9999.58984375, 10000.4482421875, 10000.1923828125, 10000.328125, 10001.041015625, 9999.74609375, 9999.5810546875, 10000.9833984375, 10000.9912109375, 9998.0439453125, 10000.263671875, 10000.7998046875, 9999.568359375, 9999.794921875, 10000.421875, 9998.775390625, 9999.3349609375, 9999.4677734375, 9999.4189453125, 10001.0166015625, 10000.91015625, 9999.2958984375, 10002.0556640625, 9998.392578125, 10000.193359375, 9999.31640625, 9999.109375, 9999.6796875, 10001.6796875, 10000.6240234375, 9999.5849609375, 9999.3349609375, 9999.87109375, 9999.126953125, 10001.5390625, 9998.919921875, 9998.392578125, 10000.4482421875, 10002.787109375, 9999.84765625, 9999.53125, 10000.13671875, 9999.8173828125, 9999.9873046875, 10000.80859375, 10000.76953125, 9999.046875, 9999.7353515625, 10001.091796875, 9998.0751953125, 9999.0078125, 10000.15234375, 9999.9326171875, 10001.830078125, 9999.6376953125, 10001.3515625, 10000.0498046875, 9999.341796875, 10001.689453125, 9999.376953125, 10000.998046875, 10000.7509765625, 10000.8779296875, 10000.06640625, 10000.4521484375, 10000.6298828125, 9999.7216796875, 9999.8193359375, 9999.376953125, 10000.0078125, 10001.7607421875, 10000.7265625, 10000.236328125, 9998.935546875, 9999.7333984375, 9998.9287109375, 9999.5849609375, 10000.0712890625, 9998.7587890625, 9999.22265625, 10000.3681640625, 9999.9189453125, 9999.4794921875, 10000.732421875, 10000.7236328125, 9997.6279296875, 9999.6015625, 10001.8701171875, 9999.8720703125, 9998.435546875, 9998.609375, 9998.9072265625, 10000.8427734375, 10000.59375, 9999.78125, 9999.2197265625, 9998.7607421875, 10000.771484375, 10001.0126953125, 9998.6318359375, 10002.53125, 10001.0390625, 10000.4501953125, 9996.2744140625, 10001.076171875, 10000.15625, 10001.0888671875, 9999.3623046875, 9998.5146484375, 9999.48828125, 10001.7509765625, 10000.00390625, 10000.134765625, 9999.3291015625, 10000.5458984375, 10002.4423828125, 9999.6142578125, 9999.5400390625, 9998.1591796875, 10000.2568359375, 10000.1923828125, 9998.16796875, 10000.3828125, 9999.0263671875, 10001.38671875, 10000.7744140625, 10000.017578125, 9998.6943359375, 9999.9521484375, 9999.201171875, 10001.8349609375, 10000.869140625, 10000.30859375, 9999.439453125, 10000.1650390625, 9999.5, 10002.0908203125, 10000.4833984375, 9998.919921875, 9998.8037109375, 10001.4345703125, 10000.107421875, 9999.8212890625, 9998.1494140625, 10000.681640625, 9998.5625, 10000.4697265625, 10000.4140625, 10000.7802734375, 10000.2744140625, 10000.0693359375, 9997.958984375, 9999.7431640625, 10001.638671875, 10000.216796875, 10000.92578125, 10000.0361328125, 9999.544921875, 10000.28125, 9998.6865234375, 10000.2998046875, 9998.8916015625, 10001.0, 9999.26171875, 9999.8359375, 9999.560546875, 9998.5966796875, 9999.6845703125, 10000.3349609375, 10000.322265625, 9999.3916015625, 9998.19140625, 10000.3203125, 10002.4345703125, 10001.212890625, 9999.5888671875, 10000.10546875, 10000.5224609375, 9999.728515625, 10000.7353515625, 10000.98828125, 9999.814453125, 10000.1513671875, 10002.34765625, 9999.4755859375, 9999.8505859375, 9999.033203125, 10000.6435546875, 9999.427734375, 9999.001953125, 10000.3076171875, 9999.72265625, 9999.40234375, 10001.2412109375, 10001.2607421875, 9999.7041015625, 9999.2744140625, 10000.435546875, 10001.44140625, 9999.34765625, 9999.6806640625, 9999.13671875, 9998.998046875, 9998.71484375, 9999.392578125, 9998.40625, 9999.515625, 9998.3896484375, 9999.5126953125, 9998.69140625, 9998.7431640625, 9998.3505859375, 9999.896484375, 10001.7275390625, 10001.658203125, 10000.51171875, 9999.9677734375, 9998.9541015625, 10001.4736328125, 9999.111328125, 10001.8935546875, 10000.7373046875, 10000.42578125, 10000.3564453125, 10000.9697265625, 9997.2001953125, 9999.642578125, 10001.84375, 10000.18359375, 10000.1640625, 9998.736328125, 10000.17578125, 10000.849609375, 9999.3427734375, 9998.84765625, 9999.509765625, 9999.267578125, 10000.328125, 9999.4013671875, 10000.5390625, 10002.4736328125, 10001.67578125, 10000.5478515625, 10000.1435546875, 10000.1650390625, 9999.599609375, 10001.0458984375, 9999.6474609375, 9998.265625, 10001.744140625, 9999.525390625, 9998.9150390625, 10000.0341796875, 10000.203125, 10000.6962890625, 9999.607421875, 10000.642578125, 10000.2685546875, 10000.3154296875, 10000.2109375, 10000.39453125, 10000.3076171875, 9998.4775390625, 9998.5888671875, 9998.810546875, 10000.5263671875, 9999.8828125, 9999.017578125, 10000.4990234375, 10001.7666015625, 10000.431640625, 9999.32421875, 10001.826171875, 9998.9833984375, 10000.6533203125, 10001.140625, 10000.8427734375, 10000.033203125, 9999.7353515625, 10000.16796875, 10000.11328125, 10000.2353515625, 9999.8564453125, 10001.9755859375, 9997.8779296875, 10002.0283203125, 9999.45703125, 10000.216796875, 10001.1474609375, 9998.7744140625, 10001.26171875, 9999.7998046875, 9999.662109375, 9999.828125, 10001.470703125, 9999.7607421875, 10001.33984375, 9999.5927734375, 10000.359375, 10001.0498046875, 10001.9599609375, 9999.5771484375, 10000.2431640625, 10000.4072265625, 9999.8330078125, 9998.6513671875, 10001.51171875, 10001.8505859375, 9998.80859375, 10001.08984375, 9999.9248046875, 9999.1005859375, 10001.447265625, 10000.140625, 10001.55859375, 10000.3388671875, 10001.3388671875, 10001.57421875, 10000.0517578125, 9999.685546875, 10000.986328125, 10000.2412109375, 9999.0625, 9998.5634765625, 9999.3857421875, 9997.49609375, 9999.9716796875, 9999.0009765625, 9998.03515625, 10000.0615234375, 10000.064453125, 9999.65625, 10000.5927734375, 9999.33203125, 9999.556640625, 9999.9697265625, 9999.5146484375, 9999.8427734375, 10000.5537109375, 9999.37890625, 10000.076171875, 10001.48828125, 10000.3642578125, 10000.8271484375, 9999.296875, 10000.6982421875, 9999.40625, 9998.8720703125, 9997.9541015625, 9998.380859375, 10001.0546875, 10000.6025390625, 9999.75, 9999.828125, 9999.6416015625, 10000.119140625, 9999.498046875, 10000.0380859375, 10000.552734375, 9999.6142578125, 10000.408203125, 10001.390625, 10000.2841796875, 10002.17578125, 10000.8681640625, 9999.9443359375, 9999.5283203125, 9999.1572265625, 9998.470703125, 9998.9853515625, 9999.6962890625, 9999.7548828125, 9999.6337890625, 10001.8955078125, 9998.9794921875, 10001.04296875, 9999.1767578125, 9997.6708984375, 9999.1494140625, 10001.4365234375, 10000.021484375, 9998.791015625, 10001.703125, 9998.7197265625, 10001.982421875, 10000.2060546875, 10001.23046875, 10000.4423828125, 9999.2333984375, 10000.6767578125, 9999.3671875, 9999.6513671875, 10000.013671875, 10001.0087890625, 9998.7236328125, 9999.330078125, 10000.931640625, 10000.3466796875, 9999.0029296875, 9998.9541015625, 9999.6689453125, 9999.4765625, 10000.716796875, 10001.0703125, 9999.6083984375, 9999.91796875, 10000.380859375, 10001.04296875, 9998.84765625, 9999.466796875, 10000.08984375, 10000.3212890625, 10000.5380859375, 10001.18359375, 9999.013671875, 9999.9873046875, 10000.3046875, 10000.0009765625, 10001.0029296875, 10000.3935546875, 10000.0771484375, 9999.8837890625, 10000.5439453125, 10000.537109375, 9999.6123046875, 9999.5556640625, 9999.814453125, 10000.771484375, 9999.2158203125, 10000.6220703125, 9998.5205078125, 10000.82421875, 10001.96875, 10000.314453125, 10000.1591796875, 10000.9111328125, 9999.7822265625, 10000.8212890625, 9998.4091796875, 10000.8310546875, 9999.8671875, 9998.79296875, 10001.38671875, 9999.6044921875, 9998.41796875, 10001.7802734375, 10000.994140625, 10002.279296875, 10000.2041015625, 10000.4921875, 9999.0283203125, 9999.966796875, 9999.048828125, 10000.0791015625, 10001.4267578125, 9999.5869140625, 9999.05859375, 9999.83203125, 10000.0654296875, 9999.427734375, 10000.0302734375, 10000.654296875, 9999.3681640625, 9999.1552734375, 10000.0703125, 9998.732421875, 9998.0927734375, 9997.888671875, 10001.0078125, 9999.3388671875, 10002.087890625, 10000.4931640625, 10000.029296875, 10001.5712890625, 10001.146484375, 9999.423828125, 10000.3486328125, 10000.6484375, 9999.2216796875, 9999.9775390625, 10000.6796875, 9999.58203125, 9999.3623046875, 9999.5361328125, 10001.2275390625, 10000.21484375, 9999.6337890625, 10000.15234375, 9999.779296875, 10000.00390625, 9999.9697265625, 9999.853515625, 9999.1708984375, 10000.3701171875, 10001.375, 9999.1591796875, 10000.109375, 9999.521484375, 9999.287109375, 9999.40234375, 10000.2421875, 10000.73828125, 9999.7529296875, 9999.69140625, 10000.021484375, 9998.7900390625, 10000.4619140625, 9998.7822265625, 9998.4453125, 10001.1572265625, 10000.44921875, 10000.208984375, 10001.3359375, 10000.2666015625, 10001.556640625, 9998.43359375, 9999.35546875, 10000.501953125, 9999.0859375, 9999.841796875, 9998.525390625, 10000.748046875, 9998.876953125, 9999.5087890625, 9998.416015625, 9999.0146484375, 10000.1083984375, 9999.6748046875, 10000.3349609375, 10000.716796875, 9997.40625, 9998.302734375, 9998.697265625, 10001.537109375, 9999.7236328125, 9998.8369140625, 9999.1572265625, 10000.9462890625, 9999.65234375, 10000.2919921875, 9999.3876953125, 10000.26171875, 9998.8642578125, 9999.7529296875, 9998.556640625, 10001.4033203125, 10000.34375, 9998.83984375, 10000.69921875, 9999.9521484375, 10000.216796875, 9999.8505859375, 9998.203125, 10000.279296875, 9999.90625, 9999.7392578125, 9999.578125, 9998.734375, 10000.2919921875, 10000.1142578125, 10000.5478515625, 9999.927734375, 10000.5830078125, 9997.783203125, 9998.685546875, 10000.365234375, 10000.626953125, 10000.767578125, 9998.6787109375, 10001.4609375, 10000.470703125, 9999.6240234375, 9998.1220703125, 9999.892578125, 9999.16015625, 9999.224609375, 10000.1826171875, 9999.9228515625, 9999.9453125, 9999.833984375, 10001.21875, 10000.7080078125, 9999.142578125, 10000.994140625, 9998.0751953125, 9996.8349609375, 9999.248046875, 10003.240234375, 9999.8564453125, 10001.77734375, 9999.927734375, 10000.796875, 9997.26171875, 10000.5263671875, 10000.0634765625, 10000.892578125, 10000.4541015625, 9998.7568359375, 10001.0693359375, 10000.662109375, 10001.650390625, 9999.24609375, 9997.7958984375, 9998.66796875, 9998.5283203125, 9998.7216796875, 10000.787109375, 9998.5654296875, 9998.8984375, 10001.607421875, 9999.869140625, 9999.796875, 10000.119140625, 10001.6845703125, 10001.0517578125, 10000.7236328125, 10001.681640625, 10001.66015625, 10000.9287109375, 9999.86328125, 9999.271484375, 9999.8984375, 9999.74609375, 9998.341796875, 10000.3876953125, 10001.1845703125, 10001.2744140625, 10001.470703125, 10000.55859375, 10000.0869140625, 10001.2548828125, 9998.9755859375, 10000.0078125, 10000.1220703125, 10001.6669921875, 10000.9296875, 10000.951171875, 10001.2197265625, 9999.43359375, 10001.4248046875, 9998.3154296875, 10000.36328125, 10000.1767578125, 9999.7177734375, 10000.212890625, 10000.2958984375, 10000.52734375, 9998.388671875, 9999.126953125, 9999.130859375, 10000.0810546875, 9999.5849609375, 10000.68359375, 9998.630859375, 9999.826171875, 10001.0537109375, 9999.447265625, 9998.5107421875, 9998.4775390625, 9999.35546875, 10000.666015625, 10000.7802734375, 9999.189453125, 10000.40234375, 9998.9794921875, 9999.111328125, 10001.0947265625, 9999.8125, 9999.34375, 9999.724609375, 10000.6806640625, 10000.2099609375, 9999.2080078125, 9999.3486328125, 10000.240234375, 10002.4384765625, 9998.9912109375, 10002.1494140625, 9999.1767578125, 9999.806640625, 10001.373046875, 10000.3642578125, 9999.6767578125, 10000.3291015625, 9998.3759765625, 9998.673828125, 10001.109375, 9999.3916015625, 9999.080078125, 9999.703125, 10000.794921875, 10002.01171875, 9999.380859375, 10000.0576171875, 10001.271484375, 10000.33984375, 10000.767578125, 9998.60546875, 9999.6943359375, 9999.646484375, 10000.666015625, 10000.423828125, 9999.3984375, 10000.34375, 10000.7099609375, 9999.021484375, 10001.322265625, 9999.63671875, 10000.296875, 10001.0908203125, 9999.08203125, 10000.92578125, 9999.3525390625, 10001.408203125, 9999.2744140625, 9997.8974609375, 9998.9091796875, 10001.44140625, 9999.921875, 10000.908203125, 10001.767578125, 10001.19140625, 10001.6953125, 9998.5498046875, 10000.8427734375, 9999.7119140625, 9996.83984375, 10001.1484375, 10001.2861328125, 10000.634765625, 9999.3251953125, 9999.560546875, 10000.35546875, 10001.724609375, 9998.802734375, 9997.6494140625, 9999.103515625, 9999.72265625, 10000.1455078125, 10001.083984375, 9999.09375, 9998.0068359375, 9998.94140625, 10000.8994140625, 10000.4296875, 9999.1787109375, 9997.40625, 9999.974609375, 9999.767578125, 10000.5166015625, 9999.8583984375, 10001.076171875, 9999.7421875, 9999.7841796875, 10000.236328125, 9999.169921875, 9999.8154296875, 10000.6728515625, 9998.42578125, 10000.052734375, 10001.208984375, 10000.966796875, 10000.443359375, 9999.4873046875, 10000.529296875, 9999.36328125, 9999.7705078125, 10000.1845703125, 10000.3505859375, 10000.2939453125, 9999.1640625, 10000.0126953125, 9999.08203125, 9999.998046875, 10001.337890625, 9997.7646484375, 9998.9248046875, 9998.609375, 10000.0673828125, 9999.6982421875, 9999.6025390625, 9999.9677734375, 9999.658203125, 10000.912109375, 10000.86328125, 10001.359375, 10000.8662109375, 10000.0478515625, 9999.650390625, 9999.431640625, 10000.306640625, 10000.705078125, 9999.7490234375, 9999.8271484375, 10001.9453125, 10001.2490234375, 9999.4638671875, 10001.5595703125, 9999.533203125, 9999.2041015625, 10000.3798828125, 10000.162109375, 10001.19140625, 10000.7529296875, 10001.8984375, 9998.169921875, 10001.0400390625, 10000.603515625, 9999.65625, 9999.1748046875, 10000.6611328125, 10000.1826171875, 9999.98828125, 10001.4619140625, 10000.39453125, 10001.830078125, 10000.6171875, 10001.5712890625, 9998.828125, 9997.8603515625, 10000.806640625, 10002.2314453125, 9999.115234375, 10000.181640625, 10001.03515625, 10000.23828125, 9999.912109375, 9999.919921875, 9999.1845703125, 9999.7568359375, 10002.3349609375, 9999.2431640625, 9999.7021484375, 9998.7255859375, 10001.1376953125, 10000.84375, 9999.841796875, 9998.6142578125, 9999.5126953125, 9999.7265625, 10000.6728515625, 9999.8623046875, 10000.77734375, 10000.5400390625, 10000.7587890625, 9999.4736328125, 10001.0673828125, 10001.048828125, 10001.08203125, 10000.0439453125, 10001.2275390625, 9999.4736328125, 9999.115234375, 10000.978515625, 9999.888671875, 9998.7568359375, 9999.2763671875, 10001.8623046875, 9999.58984375, 10000.53125, 10002.103515625, 9998.6826171875, 10000.4599609375, 9999.7314453125, 9999.912109375, 10001.146484375, 9997.3525390625, 10000.8271484375, 9998.736328125, 9999.4755859375, 9998.6162109375, 10000.685546875, 10000.4345703125, 10001.158203125, 10000.9462890625, 10000.150390625, 10001.21875, 10000.17578125, 10000.607421875, 10000.7275390625, 10000.1533203125, 10000.5830078125, 10000.349609375, 9999.0712890625, 9998.189453125, 9999.34375, 9999.7509765625, 10000.3994140625, 10000.1259765625, 10000.927734375, 9998.8837890625, 10000.060546875, 10000.75390625, 10000.54296875, 10000.8359375, 9997.6923828125, 9999.4697265625, 9997.50390625, 10000.1591796875, 10001.8876953125, 9999.65625, 10000.0595703125, 9999.0146484375, 9999.154296875, 10002.0751953125, 9998.7138671875, 9999.9462890625, 10000.078125, 10002.091796875, 10000.6611328125, 10000.912109375, 9999.8017578125, 9999.9541015625, 10001.19140625, 9999.9208984375, 10001.4482421875, 9998.94921875, 9999.7392578125, 10000.072265625, 10000.79296875, 9999.2197265625, 10000.5009765625, 9999.5224609375, 10000.931640625, 9998.5947265625, 10000.01953125, 9999.1611328125, 10000.365234375, 10000.482421875, 9999.4755859375, 9999.2255859375, 9998.6796875, 9999.5712890625, 10001.126953125, 10000.51953125, 9999.23828125, 10001.8466796875, 10000.8779296875, 10001.1142578125, 10000.734375, 10001.27734375, 9999.703125, 10001.0361328125, 10000.4658203125, 9998.982421875, 10000.5693359375, 10002.5439453125, 10000.1826171875, 9999.9990234375, 10000.26171875, 10001.17578125, 10001.693359375, 9999.5205078125, 10000.3896484375, 9999.9443359375, 9999.0478515625, 10000.509765625, 10000.998046875, 9999.7021484375, 10001.400390625, 10000.6787109375, 9999.8857421875, 10000.3779296875, 10000.3408203125, 10002.3046875, 10001.447265625, 10000.8935546875, 9999.490234375, 9999.490234375, 10000.0205078125, 10000.36328125, 10000.7001953125, 10000.8798828125, 10000.0869140625, 10000.1181640625, 10002.1015625, 9999.830078125, 10000.00390625, 9999.55859375, 9999.208984375, 10000.7578125, 9998.87109375, 9999.3173828125, 10000.595703125, 10000.7421875, 10000.8173828125, 10001.0361328125, 10001.0068359375, 10000.4619140625, 9998.41796875, 10000.47265625, 10000.099609375, 9999.6396484375, 10000.125, 10001.4716796875, 10000.5830078125, 10001.1220703125, 9999.8759765625, 10000.8525390625, 9999.7216796875, 9998.458984375, 10001.3701171875, 9999.4287109375, 9998.6630859375, 9999.984375, 9998.78515625, 9999.6064453125, 9999.6884765625, 10000.2138671875, 9999.0478515625, 10001.4208984375, 10000.6201171875, 9999.3701171875, 9999.814453125, 10000.3681640625, 10000.646484375, 9998.81640625, 9998.5048828125, 10000.208984375, 9999.861328125, 10000.0078125, 9999.2236328125, 10000.8232421875, 10000.5615234375, 10001.0009765625, 10001.259765625, 9999.9912109375, 9999.6142578125, 9997.3134765625, 10000.6533203125, 10001.5078125, 10000.4150390625, 9999.888671875, 10000.7880859375, 10001.20703125, 10001.4853515625, 10000.3583984375, 9999.6767578125, 9999.7880859375, 10000.3212890625, 9999.060546875, 9999.6953125, 10001.0380859375, 10001.5078125, 10001.6259765625, 9999.1416015625, 10001.224609375, 10000.99609375, 9999.2919921875, 10000.3603515625, 10001.013671875, 9999.6591796875, 9999.1044921875, 10000.1455078125, 10000.85546875, 9999.060546875, 9999.27734375, 9998.150390625, 9999.75390625, 9999.798828125, 9999.9736328125, 9998.5517578125, 10000.7978515625, 9999.7490234375, 10000.234375, 10000.88671875, 10000.3203125, 10000.00390625, 10001.4296875, 9999.3916015625, 9999.345703125, 10000.12890625, 10000.3564453125, 10000.15625, 9998.7958984375, 9998.3349609375, 9999.2314453125, 9999.5234375, 10001.7626953125, 10000.912109375, 10000.40625, 9999.154296875, 9999.6259765625, 9998.13671875, 9997.9423828125, 9998.697265625, 10001.34765625, 9998.6923828125, 10000.6455078125, 9999.169921875, 10000.7763671875, 9999.7685546875, 9999.376953125, 9999.8359375, 10000.6630859375, 10000.7060546875, 10000.390625, 10001.587890625, 9998.2099609375, 9999.38671875, 10001.767578125, 9999.3310546875, 9999.8701171875, 9999.9111328125, 10000.15625, 9999.6298828125, 10000.3662109375, 10001.0322265625, 9999.341796875, 10001.0302734375, 9999.365234375, 10001.5390625, 10000.4921875, 9999.84375, 10000.2900390625, 10000.10546875, 10001.2490234375, 9998.7421875, 10000.4375, 10001.046875, 9999.5986328125, 10000.4228515625, 9999.5458984375, 10000.3671875, 10001.779296875, 9999.408203125, 9997.70703125, 9999.6416015625, 9998.3623046875, 9998.150390625, 9999.37890625, 10000.080078125, 9999.6533203125, 9998.048828125, 10000.59765625, 10000.1259765625, 9997.494140625, 10000.7666015625, 9998.8828125, 10000.0048828125, 9999.267578125, 9999.3642578125, 9999.685546875, 9999.61328125, 10000.44140625, 10000.1943359375, 9999.0712890625, 10000.431640625, 10000.90234375, 9999.95703125, 10000.701171875, 9998.91015625, 10001.275390625, 9999.916015625, 9999.8427734375, 9999.01171875, 10001.14453125, 9999.859375, 10001.4970703125, 9999.1904296875, 10000.40234375, 9999.3349609375, 9999.59765625, 10000.0244140625, 10000.2236328125, 9999.3408203125, 9998.4189453125, 9999.8544921875, 9998.9306640625, 10001.4677734375, 10000.2255859375, 10000.6904296875, 9999.6318359375, 9999.875, 10002.0595703125, 9999.7138671875, 9999.3486328125, 9998.2060546875, 10000.8681640625, 10001.2841796875, 9999.7666015625, 10000.6669921875, 10000.654296875, 9998.583984375, 10001.2646484375, 9999.8798828125, 10001.11328125, 10000.9365234375, 9999.6376953125, 10002.0234375, 9999.841796875, 9998.970703125, 10001.6025390625, 9999.994140625, 9999.05859375, 9998.1669921875, 10001.0087890625, 9999.4580078125, 9999.7783203125, 10000.4296875, 10001.5185546875, 9997.666015625, 9999.2265625, 9999.744140625, 9999.7421875, 9999.359375, 10000.8740234375, 9999.529296875, 10000.380859375, 10000.439453125, 9998.7890625, 10001.021484375, 10000.0810546875, 10000.3330078125, 10000.1123046875, 9998.42578125, 9999.5068359375, 10001.66796875, 9999.94140625, 9999.173828125, 9998.1083984375, 10000.0380859375, 10000.001953125, 9999.513671875, 10000.2451171875, 9998.84765625, 9998.0986328125, 9999.22265625, 9999.6005859375, 10001.2939453125, 9998.34765625, 9999.8349609375, 10002.3408203125, 10001.1689453125, 10000.2373046875, 9999.8447265625, 10000.2373046875, 10000.6904296875, 10000.720703125, 9999.84765625, 9998.2763671875, 9998.0634765625, 9998.302734375, 10000.3525390625, 9999.1259765625, 10000.1025390625, 9998.9169921875, 10000.580078125, 9999.88671875, 9999.05078125, 9998.5654296875, 10001.025390625, 9998.2509765625, 10001.1025390625, 9999.3251953125, 10000.28515625, 9999.83203125, 9999.5234375, 9999.3798828125, 10000.373046875, 9999.8232421875, 10000.091796875, 10000.458984375, 10000.5126953125, 10000.6259765625, 9999.5771484375, 9998.525390625, 9999.2841796875, 10001.11328125, 9998.7978515625, 10000.2119140625, 10000.7265625, 9999.919921875, 10000.5205078125, 10001.43359375, 10000.8994140625, 10001.7646484375, 10000.4892578125, 9999.62109375, 9999.77734375, 9998.5888671875, 9998.8515625, 10000.72265625, 10000.2353515625, 10000.126953125, 10000.2314453125, 9999.7431640625, 9999.0302734375, 10001.0966796875, 9999.525390625, 9999.25390625, 9999.6884765625, 10000.279296875, 10001.1259765625, 9999.3095703125, 10000.1279296875, 10000.9697265625, 9999.654296875, 10001.6630859375, 9999.92578125, 10000.177734375, 9999.0263671875, 10001.0791015625, 10000.8134765625, 10000.1103515625, 9999.287109375, 10000.248046875, 9999.9521484375, 9999.265625, 10000.4775390625, 9998.7802734375, 9997.75390625, 10001.66796875, 10000.2724609375, 9998.732421875, 9999.513671875, 10000.685546875, 9999.5380859375, 10000.5380859375, 9997.7890625, 9999.9052734375, 9999.9423828125, 9998.791015625, 10001.1611328125, 10001.037109375, 9999.4326171875, 9998.7900390625, 9998.7841796875, 10001.34375, 9999.4677734375, 9998.9169921875, 9998.3291015625, 10001.9013671875, 10001.6259765625, 9999.7861328125, 10000.1220703125, 10001.220703125, 10000.4931640625, 9998.5703125, 10000.197265625, 9999.3232421875, 10000.3583984375, 10000.1982421875, 9999.6845703125, 9998.87109375, 10000.9970703125, 10000.486328125, 9998.83984375, 10000.1845703125, 9999.380859375, 9999.990234375, 9999.66015625, 10000.4140625, 10000.72265625, 9999.06640625, 9999.9814453125, 9999.7998046875, 9999.53125, 9999.724609375, 9999.451171875, 10001.451171875, 9999.490234375, 10000.220703125, 10000.0126953125, 10000.6357421875, 9999.97265625, 9999.076171875, 9999.4130859375, 9999.33203125, 10000.5478515625, 9999.7900390625, 10000.26171875, 10000.05859375, 9999.21484375, 10002.2998046875, 10000.5478515625, 10000.9560546875, 9999.4462890625, 10000.453125, 9999.919921875, 9998.072265625, 10000.4208984375, 9999.2158203125, 9999.4033203125, 10001.578125, 10000.1806640625, 9999.8076171875, 9999.9599609375, 9999.6298828125, 9999.421875, 10000.603515625, 10000.4951171875, 10000.5, 9997.568359375, 9999.806640625, 10000.7021484375, 9997.6171875, 10000.8564453125, 10000.21484375, 9999.875, 10000.3134765625, 9999.853515625, 10001.1953125, 10000.1611328125, 10001.15234375, 10000.0859375, 9997.6591796875, 9999.41015625, 9999.8525390625, 9998.916015625, 9999.099609375, 9998.2109375, 10001.2294921875, 10000.9013671875, 10000.642578125, 9998.7353515625, 9998.6279296875, 9998.2890625, 10002.4599609375, 10001.1943359375, 10001.451171875, 9999.5849609375, 10000.900390625, 9999.91015625, 9999.1884765625, 9998.775390625, 9999.265625, 10000.291015625, 10001.3681640625, 9998.9296875, 9999.7275390625, 9999.6650390625, 9998.158203125, 10000.513671875, 10001.9462890625, 9998.556640625, 9998.779296875, 10000.7099609375, 9997.740234375, 9999.7119140625, 9999.734375, 10000.1982421875, 9998.6669921875, 10000.587890625, 10000.720703125, 10000.4970703125, 9999.177734375, 9998.6767578125, 9999.630859375, 10000.486328125, 9998.43359375, 10000.591796875, 10000.25, 10001.03125, 9999.67578125, 9997.84765625, 9998.9912109375, 9997.0830078125, 10000.451171875, 9999.7919921875, 10001.03515625, 10000.7421875, 9998.490234375, 9998.99609375, 10000.71875, 9998.3583984375, 9999.607421875, 9998.681640625, 10000.18359375, 9999.28125, 10000.3427734375, 10000.78125, 9999.794921875, 9999.931640625, 9999.453125, 9999.73828125, 9999.92578125, 10000.1552734375, 9999.2578125, 9998.8134765625, 10001.123046875, 10001.287109375, 9999.0166015625, 9998.6767578125, 10000.3720703125, 9999.537109375, 10000.88671875, 9999.68359375, 9998.4765625, 9999.38671875, 9999.2705078125, 9999.96484375, 10000.2822265625, 9999.609375, 9998.390625, 9999.1806640625, 9999.6669921875, 10000.96875, 9999.6513671875, 10000.5107421875, 9999.9736328125, 10001.15625, 9999.3818359375, 9997.087890625, 10001.9248046875, 10002.3330078125, 10000.5498046875, 9999.4912109375, 10000.19921875, 9999.6787109375, 9998.791015625, 10000.423828125, 10000.71484375, 10001.5556640625, 9997.7412109375, 9999.185546875, 9999.212890625, 9998.80078125, 10000.423828125, 9999.9755859375, 9999.630859375, 10000.857421875, 10001.8173828125, 9999.8134765625, 9999.5322265625, 9998.4951171875, 9999.1513671875, 9999.8056640625, 10000.12109375, 9999.4130859375, 9999.5166015625, 10000.068359375, 9998.1337890625, 10001.30078125, 9998.5556640625, 10001.1953125, 9999.2705078125, 9999.3896484375, 10000.6787109375, 10000.20703125, 9998.78515625, 9998.4501953125, 10000.5966796875, 10001.240234375, 9999.88671875, 10000.732421875, 10000.6650390625, 10000.6611328125, 10000.3349609375, 10000.091796875, 10001.0849609375, 9999.64453125, 10000.1298828125, 10000.947265625, 10001.2939453125, 10000.2158203125, 10001.1015625, 10001.1884765625, 10001.03515625, 9999.138671875, 9999.89453125, 10000.2666015625, 10001.587890625, 9999.560546875, 9999.427734375, 9998.9169921875, 9999.3466796875, 9999.693359375, 10001.5185546875, 9999.7685546875, 9997.1923828125, 10001.0537109375, 9999.150390625, 9999.7197265625, 10000.3134765625, 10000.4130859375, 10001.6455078125, 9998.4091796875, 10001.7744140625, 10000.787109375, 9999.78125, 10000.8466796875, 10001.4912109375, 9999.794921875, 9997.8408203125, 10001.443359375, 10000.9775390625, 10000.130859375, 10000.4990234375, 9998.470703125, 10000.2431640625, 9997.8876953125, 10000.0341796875, 9999.373046875, 10000.2890625, 9997.9296875, 10000.7109375, 9998.8642578125, 9999.123046875, 9999.4404296875, 9999.705078125, 9998.8603515625, 9998.8544921875, 10001.2685546875, 9999.6982421875, 9999.2548828125, 10000.7275390625, 9997.830078125, 10000.546875, 9999.619140625, 10000.197265625, 10000.9697265625, 10000.8974609375, 9999.9619140625, 10001.6123046875, 9998.4765625, 9999.8349609375, 9999.8837890625, 10000.81640625, 10000.55859375, 10000.2236328125, 10000.13671875, 9999.4296875, 9999.8505859375, 9999.888671875, 10000.2578125, 10001.4345703125, 10000.8212890625, 9998.1552734375, 9999.1162109375, 9999.7861328125, 9998.6513671875, 9999.328125, 9999.0576171875, 9999.8349609375, 10000.0703125, 10003.298828125, 9998.8056640625, 10000.4755859375, 9998.431640625, 9998.6064453125, 10000.9384765625, 9998.359375, 10001.13671875, 9999.87890625, 9997.77734375, 9998.9189453125, 9998.3271484375, 10000.5166015625, 10000.322265625, 10000.0732421875, 9998.7275390625, 10000.7578125, 10002.9052734375, 10000.943359375, 9999.71875, 10000.111328125, 9999.8486328125, 10001.4345703125, 10000.3388671875, 9999.3046875, 10000.2802734375, 9999.294921875, 10001.2841796875, 10000.8603515625, 9999.6435546875, 10000.71484375, 9999.693359375, 10001.4326171875, 10000.779296875, 9999.4091796875, 9999.880859375, 10000.275390625, 9999.779296875, 9999.9658203125, 10001.060546875, 10000.462890625, 10002.3369140625, 9998.189453125, 10000.5595703125, 9999.3291015625, 9999.888671875, 10000.7216796875, 10000.1044921875, 9999.3916015625, 9999.8203125, 9999.5283203125, 9998.08203125, 9999.9267578125, 10001.33984375, 10000.5654296875, 9999.4560546875, 9995.84765625, 9997.15625, 10001.3427734375, 9999.880859375, 10000.4599609375, 10000.8271484375, 9998.626953125, 10000.8056640625, 9998.6044921875, 9999.830078125, 9999.462890625, 10000.8876953125, 10002.2998046875, 9999.2587890625, 10000.6552734375, 10000.060546875, 9999.439453125, 9999.904296875, 9999.66015625, 9999.4130859375, 10001.283203125, 10000.365234375, 10001.1865234375, 9998.9404296875, 9999.9970703125, 10000.2861328125, 9997.9345703125, 10000.0185546875, 10000.4267578125, 9999.8134765625, 10000.2587890625, 10000.1484375, 10001.33984375, 10000.6396484375, 9999.7197265625, 9999.7275390625, 10000.484375, 9999.21875, 9999.771484375, 9999.666015625, 10001.3857421875, 10001.724609375, 10000.02734375, 10000.8544921875, 10000.068359375, 10000.9541015625, 9998.8955078125, 9999.8056640625, 10001.708984375, 10001.6669921875, 10000.48828125, 10000.3515625, 9999.021484375, 10000.220703125, 10000.154296875, 10000.2216796875, 9999.7314453125, 9999.1162109375, 9999.611328125, 10000.15234375, 10000.748046875, 9999.015625, 9999.884765625, 9999.0546875, 9999.9580078125, 10000.404296875, 9998.537109375, 9999.9375, 9998.9482421875, 9998.8544921875, 10000.7060546875, 9999.3427734375, 10000.4169921875, 9997.73828125, 10000.4443359375, 10000.6328125, 9998.814453125, 10000.787109375, 9999.779296875, 10001.091796875, 10000.2353515625, 9999.5693359375, 9999.947265625, 9998.0029296875, 9998.02734375, 10000.6181640625, 9999.6015625, 10000.623046875, 10000.134765625, 9998.546875, 9999.2548828125, 9999.8515625, 9998.9423828125, 9999.2724609375, 10001.4697265625, 10000.2138671875, 10000.3759765625, 10000.6240234375, 10001.5263671875, 10000.734375, 9999.75, 9998.0908203125, 10001.416015625, 9999.9521484375, 10000.5595703125, 9998.416015625, 10000.609375, 10001.11328125, 10000.1513671875, 9999.8974609375, 10000.76171875, 10000.6904296875, 10001.662109375, 9999.8974609375, 9999.3837890625, 9998.4228515625, 10000.6005859375, 9998.7939453125, 9997.671875, 10001.56640625, 9998.73828125, 9999.82421875, 10000.0966796875, 9999.2919921875, 10000.091796875, 9998.6337890625, 10000.208984375, 9999.4609375, 10001.6357421875, 9999.5830078125, 10001.9580078125, 10001.3896484375, 9999.2470703125, 10000.3095703125, 9999.5009765625, 9999.2783203125, 9999.578125, 9998.87890625, 9998.5224609375, 9999.7529296875, 10001.1337890625, 9998.0166015625, 9998.744140625, 9999.4365234375, 9999.9140625, 10000.7939453125, 9997.5283203125, 10000.564453125, 10001.896484375, 9999.0576171875, 10000.7265625, 9998.638671875, 9999.798828125, 10000.185546875, 9999.634765625, 9999.5517578125, 9998.9560546875, 9997.74609375, 9999.6328125, 10000.087890625, 10000.1083984375, 9999.50390625, 10000.1796875, 9999.8037109375, 9998.85546875, 9999.69921875, 9999.5556640625, 9999.50390625, 10000.9677734375, 10000.64453125, 9998.5322265625, 9999.4697265625, 10000.9052734375, 9998.90234375, 10000.84765625, 9999.7587890625, 9999.0341796875, 9998.9599609375, 9999.208984375, 10000.787109375, 10000.25390625, 9999.56640625, 10000.2060546875, 9999.953125, 10002.6689453125, 9999.361328125, 9999.0947265625, 9999.4345703125, 10000.6298828125, 10000.408203125, 10000.2666015625, 9999.3857421875, 9999.412109375, 10000.859375, 10000.01171875, 10001.064453125, 9997.4501953125, 10000.4580078125, 9999.0859375, 9999.2841796875, 10000.4130859375, 9996.7490234375, 9999.3759765625, 10000.63671875, 9999.482421875, 10001.5419921875, 9999.8291015625, 9999.490234375, 10001.4130859375, 10000.6240234375, 10000.171875, 10000.025390625, 9999.7861328125, 9999.8671875, 10000.080078125, 10000.541015625, 10001.083984375, 10000.69140625, 9999.333984375, 9999.533203125, 10000.7587890625, 9999.5771484375, 10000.986328125, 10000.5595703125, 9999.5830078125, 9997.509765625, 10000.3876953125, 9999.01171875, 9999.6806640625, 9997.7861328125, 9999.197265625, 9999.58984375, 10000.107421875, 9999.48828125, 10000.6259765625, 9999.6474609375, 9999.3486328125, 9999.724609375, 10000.5625, 9999.7265625, 9998.880859375, 9999.669921875, 10000.4521484375, 9998.2080078125, 9998.4462890625, 10000.4248046875, 10001.271484375, 9998.6240234375, 10000.0234375, 10000.1083984375, 9999.734375, 9998.892578125, 10001.1865234375, 9999.8076171875, 10001.09375, 9998.359375, 9999.7724609375, 9998.87109375, 9999.3212890625, 9998.5458984375, 10000.2001953125, 10001.390625, 10001.9384765625, 9998.9091796875, 9999.3388671875, 10000.2431640625, 10000.3798828125, 9999.4013671875, 9999.92578125, 9998.31640625, 9998.3212890625, 9999.6318359375, 10000.439453125, 9999.759765625, 10001.0029296875, 10000.3916015625, 10000.8955078125, 9999.884765625, 9998.88671875, 10000.0869140625, 9999.318359375, 10000.2314453125, 9998.82421875, 10000.84375, 10001.1025390625, 10000.63671875, 10001.3203125, 10001.92578125, 10001.162109375, 9999.4150390625, 9998.7890625, 10000.3701171875, 10000.9208984375, 10000.6376953125, 9999.9560546875, 9999.625, 10000.5869140625, 10000.3720703125, 9998.8291015625, 9999.9287109375, 9999.46484375, 10000.7412109375, 9999.5458984375, 9999.283203125, 9999.447265625, 10000.2490234375, 10002.10546875, 10001.333984375, 9999.3154296875, 9999.603515625, 9998.3974609375, 10000.8779296875, 10001.7666015625, 9999.8828125, 9998.173828125, 10002.1337890625, 10000.9150390625, 9998.048828125, 9997.986328125, 10000.857421875, 9999.3427734375, 9999.2724609375, 9998.82421875, 9999.2939453125, 10000.2001953125, 10000.8056640625, 9999.5537109375, 9999.3388671875, 10000.3173828125, 10000.4677734375, 9999.58203125, 9999.7158203125, 9999.064453125, 10001.8671875, 9998.791015625, 9999.5322265625, 9998.08984375, 9999.44140625, 9998.9541015625, 10000.06640625, 10000.9287109375, 9999.18359375, 10001.0693359375, 9999.720703125, 9998.6181640625, 10000.0751953125, 10000.955078125, 9999.82421875, 9999.24609375, 10001.4658203125, 10001.0166015625, 9998.68359375, 10001.2275390625, 10000.6396484375, 10000.236328125, 9999.857421875, 9999.744140625, 9999.2744140625, 10000.11328125, 10000.59375, 9998.66015625, 9998.984375, 10001.1826171875, 9999.056640625, 9999.0380859375, 9999.2353515625, 10000.1240234375, 10000.2333984375, 9999.822265625, 10001.298828125, 10001.515625, 10000.873046875, 10000.837890625, 10000.6494140625, 9998.68359375, 10000.25390625, 9998.92578125, 9998.3154296875, 9998.48828125, 9998.8408203125, 10000.4716796875, 9999.6923828125, 9999.591796875, 10000.1923828125, 10001.1796875, 10001.03515625, 10000.41015625, 9999.7421875, 10000.65234375, 9999.373046875, 9998.837890625, 9998.640625, 10000.13671875, 9999.435546875, 10000.4384765625, 10000.4677734375, 9999.3994140625, 9999.6787109375, 10000.810546875, 10001.52734375, 9999.5361328125, 10000.701171875, 9999.587890625, 9999.078125, 9999.2099609375, 10001.5087890625, 9999.1748046875, 9998.7275390625, 10002.1845703125, 10000.841796875, 10000.5888671875, 10001.2919921875, 10001.31640625, 10001.337890625, 10000.8203125, 9999.9453125, 9998.6259765625, 10001.609375, 9998.9560546875, 10000.4833984375, 10001.005859375, 9998.0234375, 10000.3515625, 10001.5478515625, 9999.2705078125, 10000.703125, 10000.791015625, 9998.841796875, 10001.0107421875, 9999.6875, 9998.771484375, 10001.9169921875, 10000.0322265625, 9999.3603515625, 9999.849609375, 9999.9189453125, 10000.9638671875, 10000.599609375, 10000.6884765625, 9998.783203125, 10000.2197265625, 9999.1162109375, 10001.9931640625, 10000.3896484375, 9998.935546875, 9999.5126953125, 9999.5263671875, 9998.6982421875, 9998.7138671875, 9999.134765625, 10000.421875, 9998.037109375, 10000.0615234375, 10001.3115234375, 10001.1484375, 10000.2578125, 10001.162109375, 10001.001953125, 10000.0654296875, 10001.3037109375, 10001.0595703125, 9999.640625, 10001.6376953125, 9999.953125, 10000.095703125, 10000.158203125, 9998.634765625, 9999.8857421875, 10000.517578125, 10001.9013671875, 9999.1728515625, 9999.861328125, 9999.935546875, 10002.125, 9999.029296875, 9999.833984375, 9999.5400390625, 9999.6494140625, 9999.04296875, 9999.783203125, 10001.1162109375, 9999.3505859375, 9999.271484375, 9999.654296875, 10001.736328125, 10000.408203125, 10000.2841796875, 10000.5537109375, 9999.236328125, 10002.59765625, 10000.53125, 9999.5986328125, 9998.7509765625, 10001.041015625, 10001.1279296875, 9999.265625, 9998.87890625, 10000.166015625, 10000.6044921875, 9998.689453125, 9999.357421875, 10000.8837890625, 10000.658203125, 10000.0478515625, 10000.423828125, 9999.2412109375, 9998.845703125, 10000.5048828125, 10000.9443359375, 9998.6376953125, 9999.7490234375, 10000.0859375, 10000.314453125, 9999.85546875, 10000.3828125, 9999.9921875, 10001.1826171875, 10000.666015625, 9998.419921875, 10000.306640625, 10000.91015625, 10002.1103515625, 9999.8955078125, 9999.646484375, 10001.5087890625, 9997.607421875, 9998.662109375, 9999.177734375, 10000.0888671875, 9999.5478515625, 9999.7763671875, 9999.900390625, 9999.5849609375, 9998.97265625, 9999.5673828125, 9999.078125, 9998.447265625, 9999.560546875, 10001.140625, 10000.2802734375, 10001.4306640625, 9999.9365234375, 9999.611328125, 9999.72265625, 10000.044921875, 10001.47265625, 10000.6396484375, 10000.697265625, 9998.875, 9999.9375, 10000.4677734375, 10000.5263671875, 9999.1953125, 9999.4033203125, 9999.99609375, 10001.830078125, 9998.8408203125, 9999.9638671875, 9999.31640625, 10001.080078125, 9999.1201171875, 9998.6689453125, 10001.6953125, 10000.1630859375, 10000.072265625, 10000.7861328125, 10000.177734375, 9999.501953125, 10000.142578125, 9998.6416015625, 9999.9638671875, 9999.630859375, 10000.3017578125, 10000.05078125, 9999.66796875, 10001.2333984375, 9999.0712890625, 10001.2568359375, 10001.0322265625, 10002.0576171875, 10000.6708984375, 10000.1552734375, 9998.8017578125, 9999.4033203125, 10001.1650390625, 9999.4140625, 9998.2802734375, 10000.6298828125, 10000.57421875, 10000.6787109375, 9998.685546875, 10000.5205078125, 9998.412109375, 10001.171875, 10000.8916015625, 9998.88671875, 9998.8193359375, 10000.1298828125, 10000.7041015625, 10001.0078125, 9998.8564453125, 10001.279296875, 9999.142578125, 9999.4677734375, 9999.0673828125, 10000.9384765625, 9999.931640625, 10000.109375, 10000.1162109375, 10000.572265625, 9999.0654296875, 9998.537109375, 9998.9560546875, 9999.4833984375, 10001.3974609375, 9999.8779296875, 9999.982421875, 10001.7216796875, 9999.79296875, 9998.86328125, 9999.576171875, 10000.69921875, 9999.0322265625, 9999.3115234375, 10000.7333984375, 9999.5703125, 10001.46875, 10000.896484375, 9999.857421875, 9998.990234375, 10000.4599609375, 10000.01953125, 10000.7451171875, 10000.1640625, 10000.1435546875, 9999.2548828125, 10001.021484375, 9999.3623046875, 10001.078125, 10000.5810546875, 10000.515625, 9999.828125, 10000.7861328125, 9998.6904296875, 9998.5107421875, 10000.654296875, 9998.9375, 10000.62109375, 9999.11328125, 10000.0634765625, 9999.451171875, 10000.20703125, 10001.4638671875, 9999.2099609375, 10002.359375, 9999.2197265625, 10000.498046875, 9998.7666015625, 9999.2783203125, 9999.52734375, 10000.150390625, 10000.85546875, 10000.28515625, 10000.6826171875, 9999.5439453125, 9999.9609375, 10000.330078125, 9999.24609375, 9998.82421875, 9999.224609375, 9999.3251953125, 9999.224609375, 9999.81640625, 9999.7373046875, 9999.7998046875, 9999.5078125, 9999.6328125, 9999.1376953125, 10000.0556640625, 10000.888671875, 9999.1025390625, 9998.9833984375, 10000.142578125, 10001.23046875, 9997.0009765625, 9997.9794921875, 10000.2841796875, 10000.4970703125, 10000.01953125, 10001.0771484375, 10001.6259765625, 10000.3779296875, 9999.7548828125, 9999.0126953125, 10000.4892578125, 9999.958984375, 10000.787109375, 9999.2548828125, 9998.4189453125, 9999.71875, 10000.0029296875, 10000.056640625, 9998.1015625, 9999.1376953125, 10000.1015625, 10000.177734375, 10000.1396484375, 9999.3916015625, 9999.86328125, 10000.759765625, 10000.2880859375, 10000.2109375, 10000.1728515625, 10000.3759765625, 9998.419921875, 10001.203125, 10000.2001953125, 9999.681640625, 9998.5029296875, 9999.7587890625, 9999.484375, 9997.4228515625, 10002.4169921875, 9998.697265625, 10001.0390625, 9999.361328125, 9999.4208984375, 10000.359375, 9999.59765625, 10000.69921875, 9999.634765625, 9999.4140625, 10000.759765625, 10000.4619140625, 9998.4716796875, 9998.5595703125, 9999.578125, 10000.1376953125, 10000.5634765625, 10001.5751953125, 9999.4853515625, 9998.515625, 10000.1396484375, 9998.25390625, 9998.8466796875, 10000.2080078125, 9999.7509765625, 9999.904296875, 10001.5732421875, 10000.9453125, 9998.9755859375, 10000.3671875, 9999.3564453125, 9999.6953125, 10000.041015625, 10001.2490234375, 9998.0576171875, 9999.1826171875, 9998.400390625, 9999.6875, 9999.5068359375, 9998.7861328125, 10000.1484375, 9999.74609375, 10000.7431640625, 10001.0849609375, 10000.87109375, 9999.169921875, 9999.138671875, 10000.6943359375, 9998.9921875, 9999.396484375, 9998.0595703125, 10000.5185546875, 9999.2431640625, 9999.7958984375, 9999.783203125, 9999.859375, 9999.2607421875, 9999.005859375, 9999.9443359375, 9999.974609375, 9999.982421875, 9998.642578125, 10001.482421875, 9999.2080078125, 10000.3056640625, 10000.8359375, 9999.4892578125, 9999.548828125, 10000.4716796875, 10000.29296875, 9999.810546875, 9999.2568359375, 9998.5576171875, 9998.9716796875, 9999.32421875, 9998.4580078125, 9999.591796875, 10000.1279296875, 10000.0283203125, 10000.3681640625, 9998.087890625, 10001.099609375, 9999.560546875, 9999.0947265625, 10000.166015625, 9998.861328125, 9999.435546875, 10001.1142578125, 9999.111328125, 10002.3681640625, 10000.3505859375, 10001.302734375, 9999.5771484375, 9999.953125, 10000.091796875, 10001.083984375, 10000.052734375, 9999.962890625, 10000.57421875, 10000.908203125, 10000.9541015625, 10001.482421875, 9999.9169921875, 9999.3681640625, 10001.75390625, 10000.8173828125, 10001.7958984375, 10000.2802734375, 9999.787109375, 9999.1943359375, 9999.5771484375, 9999.4150390625, 10000.5712890625, 9999.908203125, 10000.5380859375, 9998.1376953125, 10000.185546875, 10001.466796875, 9999.8037109375, 9998.576171875, 10001.1708984375, 10000.0771484375, 10000.533203125, 9999.3994140625, 10001.48828125, 9999.5068359375, 10001.5283203125, 9999.482421875, 10000.0361328125, 10000.6328125, 10001.9521484375, 9998.408203125, 9998.3310546875, 10001.658203125, 9999.5126953125, 9998.8056640625, 10000.283203125, 9999.1767578125, 9999.1572265625, 10000.79296875, 10002.767578125, 10000.6376953125, 10000.498046875, 9999.9306640625, 10000.9287109375, 9997.6962890625, 9999.5556640625, 9998.9931640625, 10001.76953125, 10000.3232421875, 9998.541015625, 9999.4580078125, 10000.333984375, 10000.7412109375, 9998.7548828125, 9999.3896484375, 9998.6103515625, 9999.7939453125, 10000.91015625, 9997.3935546875, 9999.1845703125, 9999.1484375, 10001.513671875, 9998.4716796875, 10000.7080078125, 10001.521484375, 10000.6162109375, 9999.671875, 10001.005859375, 9999.345703125, 10000.83984375, 10001.8369140625, 10000.984375, 10001.91796875, 9997.9814453125, 9999.375, 10000.6318359375, 9999.638671875, 10001.4208984375, 10002.4306640625, 10000.654296875, 9999.359375, 10001.494140625, 10001.1689453125, 9999.884765625, 9999.7412109375, 10001.509765625, 10001.40234375, 9997.796875, 9998.9169921875, 10001.341796875, 9998.6376953125, 10000.1552734375, 10001.3134765625, 9999.8759765625, 10000.7314453125, 10000.001953125, 9999.4970703125, 9999.6552734375, 10001.0341796875, 10000.2470703125, 9999.787109375, 10001.845703125, 9999.37890625, 9999.75, 10000.1982421875, 10000.0654296875, 9999.978515625, 9997.63671875, 10000.6845703125, 10000.3701171875, 9998.6513671875, 9999.962890625, 9999.515625, 9999.0185546875, 10001.3359375, 10002.4404296875, 9999.83984375, 9999.4951171875, 10000.1015625, 9998.6630859375, 10001.50390625, 9999.5732421875, 10000.8212890625, 10000.53515625, 10000.23046875, 10000.48046875, 9999.9599609375, 10000.5380859375, 10000.5869140625, 9998.9619140625, 9999.607421875, 9998.95703125, 9998.8466796875, 9999.29296875, 9999.189453125, 9999.98046875, 10000.11328125, 10001.2548828125, 10000.6982421875, 10001.08984375, 9998.7099609375, 10000.3359375, 10001.048828125, 10000.978515625, 9999.1572265625, 10000.4111328125, 9997.7333984375, 9998.3916015625, 9998.458984375, 9999.9765625, 9999.2275390625, 10001.552734375, 10000.1328125, 9999.8740234375, 9999.53125, 9999.5068359375, 9997.5986328125, 10000.5322265625, 10001.412109375, 9999.8984375, 9999.634765625, 10000.3525390625, 9998.3662109375, 10001.7294921875, 9998.150390625, 9999.5556640625, 9999.4501953125, 9999.794921875, 9999.78515625, 10000.80078125, 9999.521484375, 10002.1240234375, 10000.28515625, 9999.86328125, 10000.611328125, 9999.427734375, 9999.08203125, 10000.763671875, 9999.4599609375, 9998.6328125, 9998.8828125, 9998.9677734375, 10000.1279296875, 10000.9853515625, 9999.9970703125, 9999.884765625, 9999.3076171875, 9998.3369140625, 9999.146484375, 9998.20703125, 10000.2451171875, 10000.8740234375, 10000.4541015625, 9998.0654296875, 9999.8232421875, 10000.6123046875, 9999.6279296875, 9999.7109375, 10001.6748046875, 10001.279296875, 9999.2783203125, 10000.8779296875, 9998.7666015625, 10000.44921875, 10000.98046875, 9998.8466796875, 9999.705078125, 9998.58984375, 10000.41796875, 10001.69140625, 9997.2451171875, 9999.3046875, 10001.947265625, 9999.212890625, 9999.7783203125, 10000.5654296875, 9999.2763671875, 10000.1552734375, 10000.306640625, 9999.318359375, 10000.154296875, 9999.1142578125, 9999.232421875, 9999.3955078125, 9998.8349609375, 9999.474609375, 10000.90625, 9999.9873046875, 10001.0068359375, 9999.2109375, 9999.9541015625, 9998.865234375, 9999.736328125, 9999.4443359375, 10000.951171875, 9998.8798828125, 10000.171875, 9999.28515625, 10000.4345703125, 10000.9638671875, 9998.7685546875, 10001.158203125, 9999.46484375, 9997.9912109375, 10000.767578125, 9999.9990234375, 10000.6796875, 10000.6005859375, 9999.6650390625, 10000.88671875, 9998.7548828125, 10000.2890625, 9999.0908203125, 10000.458984375, 9998.7294921875, 10000.955078125, 9999.146484375, 9998.3876953125, 10000.6357421875, 10000.17578125, 9998.24609375, 9999.8212890625, 9999.876953125, 10000.4921875, 9999.0, 10000.3408203125, 9998.3447265625, 10000.2890625, 10000.62109375, 10000.232421875, 9999.681640625, 9998.7607421875, 10000.4091796875, 9999.98828125, 10000.384765625, 10001.8564453125, 9998.5341796875, 10001.123046875, 10001.3408203125, 9998.8173828125, 9999.0205078125, 10000.6484375, 9999.83984375, 10000.6337890625, 10001.333984375, 9999.068359375, 10000.3798828125, 9997.13671875, 10000.30078125, 10000.4970703125, 10001.4970703125, 9997.689453125, 9999.4990234375, 10001.4384765625, 10000.6630859375, 9999.34765625, 10003.1806640625, 10001.4677734375, 10000.07421875, 9999.705078125, 9999.2568359375, 10000.1650390625, 10000.80859375, 9999.775390625, 9999.8642578125, 9999.912109375, 10000.6298828125, 9999.197265625, 9998.1171875, 9999.2978515625, 10000.3974609375, 9999.419921875, 10000.1611328125, 9999.123046875, 9999.6591796875, 10000.6826171875, 9999.4140625, 10002.3056640625, 9999.9765625, 10001.27734375, 9998.49609375, 10001.0400390625, 10001.861328125, 10000.373046875, 10000.8916015625, 9998.6767578125, 9999.896484375, 9997.5732421875, 9999.291015625, 10001.388671875, 9996.9541015625, 9999.935546875, 10001.345703125, 10001.38671875, 9999.7138671875, 9999.2578125, 10000.1318359375, 9999.994140625, 10001.1875, 10000.314453125, 9998.9375, 10000.224609375, 9999.9306640625, 10000.51953125, 9999.1640625, 9999.95703125, 10000.212890625, 9999.78515625, 10000.3935546875, 10000.3720703125, 10000.1962890625, 10001.3740234375, 10000.7939453125, 10000.267578125, 9999.8125, 9999.29296875, 9999.310546875, 9999.166015625, 9998.6416015625, 9999.5185546875, 10001.37890625, 10000.0458984375, 9998.810546875, 9999.861328125, 10000.18359375, 9999.791015625, 10000.1337890625, 10000.755859375, 9999.6806640625, 9999.8662109375, 10000.263671875, 10000.54296875, 9999.59765625, 9999.6845703125, 10001.8720703125, 10000.6123046875, 9999.16796875, 10001.1015625, 10002.0009765625, 9999.5107421875, 10000.3369140625, 9999.6826171875, 10001.650390625, 9998.7001953125, 10000.2626953125, 10000.3779296875, 9999.7666015625, 9999.392578125, 10000.5439453125, 9999.990234375, 9999.357421875, 10001.1005859375, 10000.1689453125, 10000.0595703125, 10001.1640625, 10000.91015625, 9999.443359375, 10000.0263671875, 10000.671875, 10002.150390625, 9998.8359375, 10000.173828125, 9999.4912109375, 9998.9443359375, 9999.83203125, 10000.001953125, 9999.66796875, 10000.7880859375, 9999.33203125, 10000.90234375, 9999.0634765625, 10000.255859375, 10001.494140625, 10000.2802734375, 9999.3623046875, 9999.0107421875, 9999.4296875, 9999.501953125, 10000.1328125, 9998.2880859375, 9999.1318359375, 10000.2978515625, 9998.708984375, 9999.8779296875, 9999.31640625, 10000.501953125, 9998.9267578125, 10001.1435546875, 10000.1923828125, 10000.8603515625, 10000.13671875, 9999.1982421875, 10001.7626953125, 9999.6650390625, 10000.228515625, 9999.1279296875, 9999.947265625, 9999.029296875, 9999.396484375, 9999.265625, 10002.146484375, 10000.203125, 10000.6923828125, 9998.8251953125, 10000.13671875, 10001.58203125, 9999.8759765625, 9998.03515625, 10000.3505859375, 9998.896484375, 10000.4619140625, 10000.810546875, 10000.3740234375, 9998.7783203125, 9999.953125, 10001.888671875, 10000.3798828125, 10001.4365234375, 10000.21875, 10000.0322265625, 10000.91015625, 9999.3525390625, 10001.296875, 9999.91015625, 10000.255859375, 9999.5625, 10000.255859375, 10000.537109375, 9999.04296875, 10001.375, 10001.4384765625, 9999.59375, 10001.890625, 10000.2734375, 9998.3994140625, 10000.8681640625, 10000.5966796875, 10001.2177734375, 10000.224609375, 10000.9697265625, 10001.103515625, 10000.4228515625, 9998.6328125, 9999.1484375, 10000.240234375, 9998.8095703125, 9999.26953125, 10001.0205078125, 9998.23828125, 9999.7626953125, 10000.6767578125, 10000.369140625, 10000.591796875, 9998.7529296875, 10001.98046875, 10000.9208984375, 9999.451171875, 10000.6376953125, 9999.6171875, 10001.31640625, 9999.1552734375, 9999.6376953125, 10001.6416015625, 10000.5771484375, 9999.6103515625, 9999.3564453125, 10000.052734375, 10000.26953125, 9999.76953125, 10001.744140625, 10001.150390625, 10000.392578125, 9998.4931640625, 10000.458984375, 9999.025390625, 9998.99609375, 10001.51171875, 9999.849609375, 10000.576171875, 10000.40234375, 9999.533203125, 9999.44921875, 9999.890625, 9999.7724609375, 9997.716796875, 9999.01171875, 9999.849609375, 10000.5341796875], [9999.9765625, 9999.673828125, 9999.09375, 9999.015625, 9999.4404296875, 9999.2099609375, 9999.455078125, 10000.2119140625, 9999.0205078125, 9998.880859375, 9999.6943359375, 10000.134765625, 10000.7216796875, 10000.07421875, 10000.4091796875, 9999.2236328125, 10001.1259765625, 10000.4599609375, 9999.01171875, 10000.4462890625, 9999.7353515625, 10000.3427734375, 9998.671875, 10000.1875, 9999.9013671875, 10000.9990234375, 9999.7724609375, 9997.830078125, 10000.171875, 10000.015625, 10000.2763671875, 10000.0771484375, 9998.470703125, 10000.6103515625, 9999.552734375, 10001.296875, 10000.232421875, 9999.3798828125, 10000.96484375, 10001.314453125, 9999.4072265625, 10000.1376953125, 9997.8212890625, 9999.111328125, 9999.8369140625, 9998.9814453125, 10001.2998046875, 9999.8564453125, 9998.6357421875, 10000.98046875, 9998.9638671875, 10001.025390625, 9999.318359375, 10000.2587890625, 9999.88671875, 9999.658203125, 10000.9677734375, 10000.2919921875, 9997.904296875, 9999.58984375, 10001.5732421875, 10000.37890625, 9999.0107421875, 10000.5927734375, 10000.220703125, 9998.7958984375, 9999.2763671875, 10000.61328125, 10000.2744140625, 10000.970703125, 9999.1337890625, 10000.8251953125, 10002.095703125, 10000.7724609375, 10000.248046875, 10001.2841796875, 9999.544921875, 9997.76171875, 9999.5322265625, 9997.494140625, 10000.5419921875, 10000.9033203125, 9999.318359375, 9998.73046875, 9999.8984375, 9999.841796875, 9999.7080078125, 9999.4091796875, 9998.7021484375, 9999.0126953125, 9998.7822265625, 9999.81640625, 10000.8984375, 9998.2646484375, 10000.02734375, 9998.578125, 9999.1806640625, 10000.2080078125, 9998.04296875, 9999.3310546875, 10001.134765625, 9999.939453125, 9997.9658203125, 10000.859375, 9999.767578125, 9998.9453125, 9999.3837890625, 9999.9423828125, 10001.20703125, 9999.7529296875, 9999.4033203125, 10000.14453125, 9997.95703125, 9998.9951171875, 9999.8408203125, 9997.9580078125, 9999.8056640625, 10000.1396484375, 9999.4482421875, 9999.21875, 10001.2763671875, 10000.0634765625, 9999.9453125, 9999.6904296875, 10000.1337890625, 10000.9423828125, 10001.0078125, 9999.3193359375, 9999.6875, 10000.64453125, 9999.86328125, 10000.162109375, 10000.7158203125, 9999.642578125, 10000.2626953125, 9999.5771484375, 10000.8544921875, 10000.8125, 10001.416015625, 9999.6494140625, 10000.130859375, 10002.2109375, 9998.3447265625, 10001.716796875, 9999.1337890625, 10001.15625, 10002.40625, 10001.0263671875, 9999.744140625, 10000.3037109375, 9997.1064453125, 9999.447265625, 10000.0888671875, 9999.83984375, 9998.4091796875, 10000.04296875, 10000.861328125, 9999.38671875, 9999.2509765625, 9998.7841796875, 10001.2333984375, 10000.0068359375, 10000.5927734375, 10000.138671875, 9999.90625, 10000.62890625, 10000.54296875, 10000.017578125, 10000.0625, 10000.3154296875, 10000.2021484375, 9999.5126953125, 10000.5068359375, 9999.177734375, 10000.302734375, 10000.2060546875, 10000.6865234375, 10000.70703125, 9999.333984375, 10002.408203125, 9999.4658203125, 10000.8310546875, 9998.2021484375, 9998.7900390625, 9999.1171875, 10000.091796875, 10000.3466796875, 9997.7666015625, 10000.1611328125, 9999.32421875, 9999.6103515625, 10000.220703125, 10002.0869140625, 10001.9541015625, 10001.361328125, 10000.4404296875, 10001.7568359375, 10001.3193359375, 9998.3759765625, 9998.8251953125, 9999.939453125, 9999.341796875, 10000.400390625, 10001.5244140625, 10000.181640625, 9999.82421875, 9998.470703125, 9999.890625, 9998.580078125, 9999.072265625, 10000.6220703125, 9999.9853515625, 10000.40625, 9999.646484375, 9999.8515625, 10000.2373046875, 9998.84765625, 10000.0693359375, 9999.3271484375, 10000.03515625, 10001.5458984375, 9998.701171875, 10001.9736328125, 10000.3115234375, 10000.060546875, 10001.5849609375, 10001.951171875, 10000.681640625, 9997.9228515625, 9999.1376953125, 9999.388671875, 10000.0791015625, 10000.7431640625, 10000.134765625, 10001.8857421875, 10001.6767578125, 10001.4560546875, 9999.1796875, 10001.4501953125, 9998.98046875, 9999.8330078125, 9999.6953125, 10000.0380859375, 10000.5771484375, 9999.166015625, 9998.7861328125, 10000.1533203125, 10001.3193359375, 9999.7958984375, 9999.17578125, 9998.9716796875, 9997.822265625, 9999.333984375, 9999.6962890625, 9998.58984375, 10000.0185546875, 9999.09375, 10000.1591796875, 10000.173828125, 10000.7373046875, 10000.443359375, 9998.8466796875, 10000.748046875, 9999.279296875, 10001.650390625, 9999.451171875, 9999.556640625, 10001.6484375, 9999.4150390625, 9998.8037109375, 9999.1025390625, 9999.4580078125, 9999.09765625, 9998.806640625, 10000.455078125, 10000.0947265625, 9998.9130859375, 10000.0859375, 10000.0087890625, 9999.1787109375, 9998.486328125, 10000.978515625, 10000.41015625, 10000.84765625, 10001.298828125, 9998.13671875, 10000.0849609375, 10000.14453125, 9999.138671875, 10000.91796875, 9999.2705078125, 10001.1396484375, 9999.9306640625, 10001.6044921875, 10000.984375, 10000.1083984375, 10000.978515625, 10000.5712890625, 10001.185546875, 9999.6962890625, 9999.341796875, 10000.017578125, 10000.5029296875, 10001.6025390625, 9998.8310546875, 10000.642578125, 10000.04296875, 10000.779296875, 10000.875, 9999.4814453125, 10000.44921875, 9999.6171875, 9998.9931640625, 10000.830078125, 9999.7939453125, 10000.44921875, 10000.6181640625, 10001.6162109375, 9997.392578125, 10000.86328125, 10001.2412109375, 10000.646484375, 10000.8427734375, 9998.92578125, 9997.0830078125, 10001.529296875, 10000.509765625, 9999.669921875, 9999.107421875, 9998.201171875, 9999.8271484375, 10000.232421875, 9999.0869140625, 9999.6357421875, 9998.4150390625, 9998.521484375, 10000.287109375, 9999.810546875, 9998.361328125, 9999.689453125, 9999.8388671875, 9999.375, 9999.9921875, 10000.1845703125, 9999.8154296875, 10000.92578125, 9998.482421875, 10001.4833984375, 10000.0849609375, 10000.0771484375, 10000.4140625, 10000.5419921875, 9999.3515625, 10000.6552734375, 10000.1611328125, 10000.5205078125, 9999.3564453125, 10000.2666015625, 9999.9072265625, 9998.9462890625, 9998.8681640625, 10000.3916015625, 10000.244140625, 10000.2890625, 10000.8291015625, 9999.9130859375, 10001.5712890625, 10000.1728515625, 10000.2021484375, 10001.0205078125, 10000.7548828125, 10000.5361328125, 9998.5810546875, 9999.853515625, 9998.09375, 10000.4326171875, 9999.9521484375, 9999.17578125, 9999.966796875, 9999.927734375, 10001.986328125, 9999.1884765625, 10001.283203125, 9998.3828125, 10000.4697265625, 10002.1337890625, 10000.6943359375, 10000.0771484375, 9998.4287109375, 10000.189453125, 9998.314453125, 10000.4404296875, 10000.1806640625, 10000.8271484375, 9999.802734375, 10002.2080078125, 10000.34765625, 10001.8154296875, 9999.3408203125, 10001.81640625, 9999.5390625, 10000.259765625, 9997.5859375, 9999.1611328125, 10000.4130859375, 10000.548828125, 10000.3203125, 9999.671875, 9999.4873046875, 10002.0849609375, 10001.708984375, 9999.474609375, 10000.5966796875, 9998.9990234375, 10001.0234375, 10001.263671875, 10000.34765625, 10000.875, 9999.2734375, 9999.4697265625, 10000.171875, 9998.521484375, 9999.9794921875, 10001.19921875, 9997.5009765625, 10001.4990234375, 10001.2392578125, 9999.345703125, 10001.0908203125, 9998.8076171875, 9999.861328125, 9999.197265625, 9999.7978515625, 9998.8203125, 9999.7978515625, 10000.1259765625, 10000.4765625, 10000.830078125, 9998.99609375, 9999.2578125, 10000.98046875, 10000.076171875, 10001.59765625, 10000.3349609375, 10000.267578125, 9998.44921875, 9999.0751953125, 9999.646484375, 9999.8759765625, 10000.3994140625, 9999.626953125, 9997.9619140625, 10000.45703125, 9999.7216796875, 9999.08203125, 10001.41015625, 9999.98828125, 10001.3984375, 9998.7998046875, 9997.7890625, 10000.462890625, 9999.2724609375, 10001.125, 9998.759765625, 9999.6826171875, 9999.861328125, 10001.3056640625, 10000.00390625, 10000.05859375, 9999.7314453125, 10000.35546875, 10000.7861328125, 10000.29296875, 10000.564453125, 9996.615234375, 9999.216796875, 10001.322265625, 10001.671875, 10000.41796875, 10001.6728515625, 9999.05078125, 9999.2275390625, 10000.564453125, 9999.5341796875, 9999.2021484375, 10000.9091796875, 10000.890625, 9999.8251953125, 9998.0283203125, 10000.6494140625, 10000.1328125, 9998.6103515625, 9998.5859375, 10000.9033203125, 10001.3017578125, 10000.0068359375, 10000.4033203125, 10001.5224609375, 10000.27734375, 10000.1796875, 10000.4443359375, 10000.1953125, 9999.615234375, 10001.4130859375, 9999.3720703125, 10001.080078125, 9999.94140625, 10000.9765625, 9998.095703125, 10002.125, 10001.140625, 10000.5263671875, 9999.7373046875, 9998.61328125, 10000.6806640625, 10001.8154296875, 9999.4921875, 9999.8623046875, 10000.099609375, 10000.642578125, 10001.185546875, 9999.8623046875, 9999.1826171875, 10002.4296875, 9998.96484375, 9999.1748046875, 9999.515625, 9999.3427734375, 10000.478515625, 9998.9345703125, 10001.1298828125, 9998.9814453125, 9999.2705078125, 10001.6171875, 9999.9150390625, 10000.7685546875, 9999.939453125, 10000.66796875, 9999.8671875, 10002.0869140625, 9998.798828125, 9999.732421875, 9999.8291015625, 10001.7802734375, 9998.9404296875, 9998.9423828125, 10001.525390625, 10000.6318359375, 9999.9453125, 9999.8603515625, 9999.837890625, 10000.3583984375, 10000.453125, 10000.3369140625, 9999.998046875, 9999.1201171875, 9998.9013671875, 9999.53515625, 10000.3818359375, 10002.1201171875, 9999.7255859375, 9998.8583984375, 10000.466796875, 9999.484375, 10001.068359375, 10000.6904296875, 10000.8642578125, 10000.3564453125, 9999.8740234375, 10000.7763671875, 10001.2177734375, 9999.51171875, 9999.6904296875, 9999.1630859375, 9998.3935546875, 9999.0546875, 9998.041015625, 9999.4453125, 10001.23828125, 9999.876953125, 10001.1669921875, 10000.162109375, 10002.5390625, 10000.1416015625, 9999.908203125, 9999.244140625, 10000.0517578125, 9999.76953125, 10000.5478515625, 9999.1494140625, 9999.51953125, 9999.1806640625, 9999.9482421875, 10000.869140625, 10000.240234375, 10000.54296875, 10000.9404296875, 10000.4384765625, 10001.130859375, 10001.5048828125, 9998.734375, 9998.798828125, 9999.7109375, 9999.6474609375, 10000.3798828125, 9998.9619140625, 10000.6328125, 10000.6982421875, 10000.4853515625, 10000.388671875, 10000.912109375, 9999.1845703125, 9999.013671875, 9998.830078125, 10000.2412109375, 10000.2900390625, 9998.046875, 9997.84765625, 10000.0400390625, 9998.5693359375, 9998.4560546875, 10000.6298828125, 10000.591796875, 9999.654296875, 9998.689453125, 10001.287109375, 10002.2177734375, 10002.3857421875, 10000.548828125, 10000.052734375, 10000.330078125, 10000.0869140625, 9998.5517578125, 9999.5869140625, 10001.0634765625, 10001.1298828125, 9999.236328125, 10000.5703125, 9998.6787109375, 10000.9921875, 9999.1806640625, 10000.677734375, 10000.541015625, 10000.908203125, 9999.009765625, 9997.6943359375, 9999.8232421875, 9999.970703125, 9999.755859375, 9998.1025390625, 9999.3896484375, 10000.7373046875, 10000.19921875, 9999.3291015625, 10000.49609375, 10000.109375, 10001.2802734375, 10000.8955078125, 9999.4189453125, 10000.232421875, 9999.9755859375, 10000.421875, 10001.2744140625, 10000.5625, 10000.462890625, 9999.009765625, 9999.2138671875, 10001.0546875, 10000.76171875, 9999.501953125, 9999.7890625, 9999.42578125, 10001.46875, 9999.564453125, 9998.703125, 9998.935546875, 10000.451171875, 9999.103515625, 10001.173828125, 10000.40625, 10000.8525390625, 9999.89453125, 10000.7587890625, 10001.6865234375, 9999.3017578125, 9999.5693359375, 9999.4833984375, 9999.0078125, 10001.80859375, 10000.2197265625, 10000.7099609375, 9999.4072265625, 9998.72265625, 9997.7294921875, 10000.0107421875, 9998.9990234375, 10000.033203125, 9999.8740234375, 10000.0439453125, 10002.1162109375, 9999.2470703125, 10000.013671875, 9999.7802734375, 9998.802734375, 9999.759765625, 10000.0048828125, 9998.7060546875, 10000.294921875, 9999.244140625, 10000.173828125, 10000.76953125, 10000.806640625, 10002.142578125, 9999.904296875, 9999.072265625, 10000.8662109375, 10000.63671875, 9999.669921875, 9999.2958984375, 9998.5537109375, 9999.5048828125, 9999.65625, 10000.09765625, 10001.1455078125, 9999.0947265625, 9997.9921875, 10000.591796875, 10000.7451171875, 9999.9677734375, 9998.9052734375, 9999.1259765625, 9998.8896484375, 10001.396484375, 9997.9248046875, 9999.5908203125, 10000.505859375, 10000.076171875, 9999.650390625, 10000.4208984375, 10000.0078125, 9998.9892578125, 10000.4990234375, 9999.9794921875, 9998.66015625, 10000.67578125, 10000.9970703125, 9999.0810546875, 9998.755859375, 10002.021484375, 9999.9306640625, 9999.6025390625, 9999.3720703125, 10000.9443359375, 9999.7177734375, 9999.1953125, 10000.03125, 10000.1298828125, 9999.6953125, 10000.9716796875, 9999.4892578125, 10000.623046875, 10000.404296875, 9999.0224609375, 9999.6142578125, 9999.0986328125, 9999.6083984375, 10000.36328125, 10001.091796875, 9998.77734375, 9999.9931640625, 9998.1259765625, 10000.5107421875, 9999.4697265625, 9998.396484375, 9998.5830078125, 9999.65625, 9998.8837890625, 10001.5029296875, 10000.65234375, 9998.552734375, 9999.2744140625, 10001.6162109375, 10000.1923828125, 9998.8701171875, 10000.0810546875, 9999.79296875, 10000.6337890625, 10000.5751953125, 10000.0107421875, 10000.6494140625, 10001.12109375, 10000.966796875, 10001.51171875, 9998.611328125, 9998.7626953125, 10000.1572265625, 9999.8369140625, 10000.7216796875, 9999.025390625, 9999.5283203125, 10001.7041015625, 10000.888671875, 10001.1845703125, 9999.7197265625, 10000.712890625, 9998.8349609375, 10000.67578125, 10001.216796875, 9999.39453125, 10000.166015625, 10000.232421875, 9999.77734375, 9999.6904296875, 9998.1220703125, 9999.5380859375, 10000.2158203125, 9999.7822265625, 10001.369140625, 9996.837890625, 10000.2412109375, 9997.7919921875, 9999.46484375, 9999.80859375, 9999.251953125, 9999.1943359375, 9998.95703125, 9997.880859375, 10000.2978515625, 9999.8896484375, 9998.5166015625, 10000.5908203125, 9997.85546875, 10001.6337890625, 9998.5654296875, 10000.2421875, 9999.8369140625, 9999.4921875, 9998.5576171875, 10000.6416015625, 10000.3837890625, 9999.4765625, 10002.2841796875, 9999.4013671875, 9999.13671875, 9997.9990234375, 9999.849609375, 10000.0361328125, 9999.7041015625, 10001.2861328125, 9998.912109375, 10000.3388671875, 10000.5390625, 10000.1162109375, 9999.6669921875, 9999.634765625, 10000.9833984375, 10000.4404296875, 9999.2978515625, 9999.4599609375, 10000.0400390625, 9998.7783203125, 10000.30078125, 10001.9931640625, 9999.9306640625, 10000.3505859375, 10000.548828125, 9999.029296875, 10001.2919921875, 9999.2978515625, 9999.5791015625, 9998.48046875, 9998.1552734375, 10002.53125, 10000.3623046875, 10001.1826171875, 10001.080078125, 9998.1005859375, 9999.501953125, 9999.953125, 9998.7548828125, 10001.81640625, 10000.2685546875, 10000.208984375, 9997.7939453125, 10000.37890625, 9998.7509765625, 10000.6728515625, 10000.04296875, 10002.4072265625, 10001.9638671875, 9998.939453125, 9999.681640625, 10002.13671875, 10000.4111328125, 10001.0361328125, 9998.7890625, 9999.8349609375, 10000.2294921875, 10000.08984375, 9999.7568359375, 9999.2451171875, 9998.8173828125, 10000.2744140625, 10001.5322265625, 10000.814453125, 9998.8876953125, 9999.5634765625, 10001.296875, 10001.181640625, 10000.5546875, 9999.1943359375, 9999.6240234375, 10000.025390625, 10000.12109375, 10001.6005859375, 9999.67578125, 9999.1513671875, 9998.265625, 10001.2197265625, 10000.5322265625, 9999.0322265625, 10000.59375, 9999.6962890625, 10000.8916015625, 10000.7802734375, 9998.1513671875, 10000.2421875, 10000.333984375, 10000.826171875, 9999.171875, 9999.4423828125, 10000.359375, 9999.4443359375, 9999.287109375, 9999.052734375, 9999.53125, 9998.677734375, 10000.8232421875, 9998.748046875, 10000.470703125, 10000.8173828125, 9999.2119140625, 9999.572265625, 10000.6083984375, 9999.0068359375, 10000.70703125, 10000.7236328125, 10000.6474609375, 10000.203125, 9998.5419921875, 9999.2177734375, 10001.1826171875, 9999.1796875, 9999.947265625, 9999.7890625, 9997.703125, 10001.1591796875, 10000.39453125, 10001.4775390625, 9999.7236328125, 9999.931640625, 9999.1494140625, 9998.57421875, 10000.537109375, 9999.5009765625, 9999.044921875, 9999.6298828125, 9999.859375, 9999.927734375, 9998.8115234375, 9999.556640625, 9999.541015625, 9998.19921875, 9999.64453125, 9998.369140625, 9997.595703125, 9999.4609375, 10001.4326171875, 9999.9423828125, 10001.3603515625, 10000.6181640625, 9999.830078125, 9999.5224609375, 9999.2744140625, 10000.5986328125, 10000.474609375, 10000.4033203125, 9999.744140625, 10002.3515625, 10000.7587890625, 10000.0537109375, 10001.40625, 10000.0966796875, 9998.4619140625, 9999.111328125, 10001.0810546875, 10000.9404296875, 9998.9365234375, 10000.9736328125, 10000.5234375, 10000.1025390625, 9999.734375, 10000.21484375, 10001.125, 10000.423828125, 10000.7666015625, 9999.8173828125, 10000.728515625, 10000.1591796875, 10000.22265625, 9999.3076171875, 10001.2421875, 10001.7216796875, 10000.6806640625, 9999.2705078125, 10001.8291015625, 9998.3310546875, 9997.982421875, 10000.5185546875, 10001.4091796875, 10000.0068359375, 10001.2890625, 9999.970703125, 9999.490234375, 9999.16015625, 10000.5380859375, 9999.275390625, 9999.28515625, 10000.6728515625, 10000.0654296875, 10000.046875, 9998.9306640625, 9999.056640625, 9999.83984375, 9999.6884765625, 10000.5810546875, 9999.796875, 9999.6142578125, 9999.7265625, 9999.564453125, 10000.013671875, 10001.49609375, 10000.3466796875, 10001.001953125, 10000.0986328125, 10000.615234375, 9999.9404296875, 9998.212890625, 10000.568359375, 9998.7333984375, 9999.3603515625, 9999.5146484375, 10000.8642578125, 9999.078125, 10000.3095703125, 9999.1669921875, 9999.958984375, 10000.134765625, 9999.2568359375, 9999.7744140625, 9999.685546875, 10001.2958984375, 10000.5234375, 10000.482421875, 10000.404296875, 9999.837890625, 10000.888671875, 10000.775390625, 10000.615234375, 10000.4765625, 9998.1513671875, 9999.0517578125, 10001.0341796875, 9999.025390625, 10000.5234375, 10000.015625, 9999.0185546875, 9999.0498046875, 9998.9453125, 9998.376953125, 9999.443359375, 9997.978515625, 9998.9072265625, 10001.15625, 9999.419921875, 10001.142578125, 10001.68359375, 9999.6875, 9999.7177734375, 9999.939453125, 10000.791015625, 9999.759765625, 10000.603515625, 9999.95703125, 9999.1171875, 10000.923828125, 10000.056640625, 9998.74609375, 9999.5576171875, 10000.33203125, 10001.2607421875, 10001.173828125, 9999.9384765625, 10001.001953125, 9999.552734375, 10000.720703125, 9999.7216796875, 9999.8173828125, 10000.1005859375, 9998.8935546875, 10000.1494140625, 10000.73828125, 9999.7666015625, 9998.58984375, 10000.392578125, 10002.29296875, 10000.658203125, 9999.7177734375, 10001.337890625, 9999.1875, 9997.6455078125, 9998.4833984375, 9999.498046875, 9998.755859375, 10001.2373046875, 9999.8544921875, 9999.814453125, 10000.462890625, 9998.9697265625, 9999.9345703125, 9999.8359375, 9999.80078125, 9998.4267578125, 10000.7783203125, 9999.38671875, 9999.845703125, 9998.6201171875, 9999.9697265625, 9999.79296875, 9999.78515625, 10001.6630859375, 10000.671875, 9999.033203125, 9999.69921875, 9999.81640625, 10000.3837890625, 10000.2744140625, 10000.765625, 10001.064453125, 10001.3759765625, 10001.0283203125, 10001.3271484375, 10000.5400390625, 9999.296875, 9999.548828125, 10000.359375, 10000.189453125, 10000.43359375, 9999.583984375, 10000.4150390625, 10000.4951171875, 9999.4638671875, 10000.3330078125, 10001.20703125, 9998.6767578125, 9999.7421875, 9999.619140625, 9999.9287109375, 9999.6669921875, 10000.2763671875, 9999.205078125, 10001.9716796875, 9998.7421875, 10000.791015625, 9999.9833984375, 9999.939453125, 9998.8232421875, 9998.318359375, 10000.001953125, 9999.73828125, 9998.845703125, 9999.669921875, 10000.2998046875, 9998.6015625, 9998.16796875, 9998.8857421875, 10000.1044921875, 9999.296875, 9999.4072265625, 10000.427734375, 10000.615234375, 9999.337890625, 10000.615234375, 10000.373046875, 9999.265625, 9999.048828125, 10000.515625, 9998.6806640625, 10000.4619140625, 10001.0791015625, 10000.201171875, 9998.453125, 10001.6953125, 9999.5927734375, 9999.6396484375, 9998.28125, 9999.806640625, 9999.8994140625, 10001.9501953125, 9998.251953125, 10000.7744140625, 10000.0751953125, 10001.3984375, 9999.048828125, 10000.4072265625, 9999.0751953125, 10001.259765625, 9998.966796875, 10000.74609375, 10000.8642578125, 10001.0537109375, 9999.138671875, 10000.2568359375, 9998.77734375, 10000.6357421875, 10001.4443359375, 9998.84375, 10001.0498046875, 10001.1875, 9999.66796875, 10001.7998046875, 9999.2431640625, 9999.6796875, 10000.8515625, 9999.779296875, 10000.2255859375, 9999.3857421875, 9998.71484375, 10001.189453125, 9999.115234375, 10000.3720703125, 10000.5, 9999.71875, 10000.427734375, 9998.1181640625, 9999.6767578125, 10001.119140625, 9999.544921875, 10000.3525390625, 10000.396484375, 10000.0439453125, 9999.64453125, 10001.45703125, 9999.4580078125, 10001.0625, 9999.86328125, 10001.818359375, 10000.904296875, 10000.591796875, 9999.5185546875, 10000.4765625, 9999.193359375, 9999.2646484375, 10002.294921875, 9999.501953125, 9999.4140625, 10001.541015625, 10000.5654296875, 10001.5595703125, 9998.8916015625, 9999.103515625, 10000.509765625, 10001.9228515625, 9999.3994140625, 9999.6845703125, 10000.1064453125, 10000.7919921875, 10002.138671875, 10000.1337890625, 10001.0693359375, 10002.189453125, 9999.462890625, 10000.2138671875, 9999.212890625, 9998.408203125, 9999.3427734375, 10000.443359375, 9997.6103515625, 10000.408203125, 9999.244140625, 10002.19921875, 9999.96484375, 10001.578125, 9999.0556640625, 9999.37890625, 10000.7529296875, 10002.0126953125, 10000.2470703125, 10000.78125, 9998.177734375, 10000.11328125, 9999.388671875, 9998.00390625, 10001.41796875, 10002.6669921875, 10001.4833984375, 10000.892578125, 10000.345703125, 9999.4443359375, 9999.4990234375, 9999.5810546875, 9999.8486328125, 10000.044921875, 10001.513671875, 9998.8759765625, 9998.7138671875, 10000.283203125, 10000.3515625, 9999.6103515625, 9999.41015625, 10000.267578125, 10000.2021484375, 9999.7978515625, 10001.39453125, 10000.0546875, 9999.4501953125, 9997.9296875, 9999.556640625, 10000.4912109375, 9999.283203125, 9998.7763671875, 10000.7587890625, 9998.18359375, 10001.0166015625, 10000.5322265625, 9998.9189453125, 9999.921875, 10000.322265625, 9999.873046875, 9999.736328125, 10000.494140625, 10001.6240234375, 9998.5556640625, 9999.494140625, 9999.173828125, 10000.5361328125, 9998.931640625, 10000.5400390625, 10001.1748046875, 10001.5009765625, 10002.2666015625, 10000.328125, 9999.4765625, 9999.8388671875, 10000.453125, 10001.125, 9999.4609375, 9999.630859375, 10000.3876953125, 10000.333984375, 10000.5087890625, 10001.837890625, 9999.9814453125, 9999.7060546875, 9998.5537109375, 10000.5107421875, 9999.548828125, 10000.2412109375, 9999.09375, 9998.3427734375, 9999.7197265625, 10000.0634765625, 9998.5654296875, 10001.0263671875, 10000.4189453125, 9999.99609375, 9999.005859375, 9999.5791015625, 10000.140625, 10001.765625, 10000.1435546875, 9999.765625, 9998.166015625, 9999.029296875, 9999.7236328125, 9999.7109375, 9999.384765625, 9998.1181640625, 10000.302734375, 10000.13671875, 9999.27734375, 10000.3583984375, 9999.7587890625, 9999.359375, 9999.5341796875, 9998.1533203125, 9998.189453125, 10000.935546875, 9999.4697265625, 9998.263671875, 9999.78515625, 9999.1787109375, 9998.9833984375, 10000.931640625, 9998.9287109375, 9999.72265625, 9999.5888671875, 9999.5126953125, 10000.2578125, 10000.822265625, 10000.08984375, 9999.1962890625, 10001.02734375, 10000.69140625, 10000.1513671875, 10000.6474609375, 10000.6796875, 10002.1611328125, 9999.064453125, 9999.74609375, 10000.3974609375, 10000.7919921875, 10000.6953125, 10001.00390625, 10001.591796875, 9999.220703125, 10001.3828125, 10000.6474609375, 10000.4267578125, 10000.0458984375, 10001.3193359375, 9999.1259765625, 10000.134765625, 10000.60546875, 10000.7138671875, 9999.6259765625, 9999.5078125, 10000.3125, 10001.0126953125, 9999.1259765625, 10000.0703125, 9999.0400390625, 10001.1611328125, 10000.515625, 9998.9580078125, 9998.75, 10000.3349609375, 10000.8076171875, 9998.3046875, 9999.66015625, 10000.498046875, 9998.3671875, 9997.80859375, 9998.728515625, 10000.9072265625, 10000.294921875, 9998.6875, 10000.01171875, 10000.1103515625, 9999.4140625, 10001.8642578125, 10000.6357421875, 9999.9541015625, 10001.033203125, 9999.630859375, 10000.48046875, 9999.7216796875, 9998.830078125, 10000.765625, 10000.7587890625, 9999.634765625, 9999.369140625, 10000.6171875, 10000.9140625, 9999.0419921875, 10000.3525390625, 10001.7734375, 9999.7685546875, 10001.5888671875, 9999.6328125, 10000.466796875, 10001.09765625, 10000.306640625, 10000.0625, 9998.943359375, 10000.0576171875, 9999.90234375, 9999.5390625, 9999.4111328125, 9998.8955078125, 10001.337890625, 9999.9599609375, 9998.6337890625, 10000.6630859375, 9998.8505859375, 10000.26171875, 10000.8525390625, 9999.8857421875, 9999.40234375, 9999.8408203125, 10001.3525390625, 10001.0947265625, 9999.75390625, 10000.177734375, 9998.970703125, 10000.72265625, 9999.4189453125, 9999.5732421875, 10001.201171875, 10002.546875, 9998.2880859375, 10002.123046875, 9999.298828125, 9999.248046875, 10000.2265625, 10001.4384765625, 10000.552734375, 10000.333984375, 10000.71484375, 10000.19140625, 9998.380859375, 9998.7978515625, 10001.361328125, 10001.0537109375, 10001.125, 10000.3828125, 9998.7900390625, 9999.5126953125, 10000.5390625, 10000.2578125, 9999.671875, 10001.5830078125, 10000.2109375, 10000.591796875, 9998.5380859375, 9998.9677734375, 9999.8837890625, 10000.0439453125, 9999.5615234375, 10000.3125, 10000.3251953125, 10000.4482421875, 10000.283203125, 10001.2314453125, 9999.5908203125, 9999.3662109375, 10000.576171875, 10000.2705078125, 9998.671875, 10000.1435546875, 10001.7373046875, 9999.4189453125, 10000.1025390625, 10000.3974609375, 10001.6611328125, 9996.8291015625, 10000.8994140625, 10000.60546875, 9999.609375, 10000.1953125, 10001.310546875, 9999.3720703125, 10001.673828125, 10000.6162109375, 9999.1484375, 9999.287109375, 10000.640625, 9999.7265625, 10000.91015625, 10000.107421875, 9999.15234375, 10000.265625, 10000.5693359375, 10001.08984375, 10000.6533203125, 9999.5908203125, 10001.046875, 10001.103515625, 10001.2939453125, 10001.2421875, 10001.2734375, 10000.49609375, 9999.392578125, 10000.794921875, 10001.7939453125, 9999.3837890625, 9998.837890625, 10000.4912109375, 10001.07421875, 9999.7998046875, 9998.998046875, 10002.0693359375, 9998.7353515625, 10000.29296875, 9999.6435546875, 10000.1787109375, 9999.4326171875, 10000.0810546875, 10001.091796875, 9999.5673828125, 10000.939453125, 10000.73828125, 9999.787109375, 10000.7041015625, 9999.322265625, 10001.4091796875, 9998.2109375, 9999.587890625, 9999.0224609375, 9999.556640625, 9997.41015625, 10001.509765625, 10000.8408203125, 10000.69921875, 9999.8291015625, 10000.294921875, 10000.470703125, 9999.6435546875, 10001.138671875, 10000.1298828125, 10000.9072265625, 9999.169921875, 9998.337890625, 10001.654296875, 10000.0478515625, 9999.6396484375, 9999.8447265625, 10001.94140625, 10001.9208984375, 10000.1064453125, 10000.4130859375, 9999.515625, 9999.548828125, 9998.9287109375, 10000.6796875, 9999.82421875, 10000.03125, 10002.169921875, 9999.4345703125, 9998.7607421875, 10001.1953125, 9998.9609375, 9999.662109375, 9999.8740234375, 9997.7255859375, 9999.7490234375, 10000.927734375, 9999.166015625, 9999.9609375, 10000.494140625, 10000.671875, 10000.62109375, 9998.9716796875, 9999.572265625, 9997.884765625, 10001.798828125, 9999.974609375, 10000.365234375, 9998.1943359375, 9999.62890625, 9998.84375, 9998.7392578125, 9999.8037109375, 10000.5, 9999.83984375, 9999.3642578125, 9998.630859375, 9999.607421875, 9999.9267578125, 9999.607421875, 9999.326171875, 10000.1767578125, 10000.525390625, 9997.8369140625, 10000.40234375, 9999.1474609375, 9999.091796875, 10000.4580078125, 10000.90625, 9998.291015625, 9998.658203125, 10000.3427734375, 9998.318359375, 10000.3271484375, 10002.1572265625, 9998.2490234375, 9998.5791015625, 10002.669921875, 9999.0205078125, 10000.740234375, 10000.4580078125, 10000.23828125, 10000.5947265625, 9998.42578125, 10001.5537109375, 9999.9599609375, 9998.4931640625, 10001.1689453125, 10003.140625, 10000.4970703125, 9999.0185546875, 10000.55078125, 9998.7451171875, 10001.40234375, 9997.8115234375, 9998.7568359375, 10001.171875, 9999.3486328125, 10001.8125, 10000.001953125, 9999.9033203125, 9998.8857421875, 10000.19921875, 10001.783203125, 9998.623046875, 9999.3427734375, 10001.658203125, 10000.185546875, 10000.3701171875, 9999.458984375, 9998.69921875, 10001.810546875, 9998.5849609375, 9998.4580078125, 10000.1337890625, 9999.4921875, 9999.1484375, 9999.603515625, 10000.861328125, 10000.1123046875, 9999.462890625, 9998.583984375, 10000.66796875, 10002.0380859375, 9999.0244140625, 9999.6513671875, 9998.0498046875, 10000.251953125, 9999.6806640625, 10000.4013671875, 10000.0595703125, 9999.4853515625, 9998.568359375, 10000.517578125, 9999.154296875, 10000.287109375, 10000.271484375, 10000.8037109375, 10000.462890625, 9999.109375, 10001.1494140625, 10000.736328125, 10001.318359375, 10000.685546875, 9997.8251953125, 9999.572265625, 9999.5302734375, 10000.1591796875, 9999.2587890625, 10002.0029296875, 9999.2509765625, 9999.201171875, 10002.142578125, 10001.7841796875, 9998.6884765625, 9999.521484375, 10001.51953125, 9999.822265625, 9998.91015625, 10000.537109375, 10000.826171875, 9998.6865234375, 10000.185546875, 9999.7607421875, 9998.99609375, 10000.79296875, 9998.8779296875, 10001.6318359375, 10000.330078125, 9997.9091796875, 9999.7021484375, 10001.8447265625, 10000.1435546875, 10000.8486328125, 10000.40234375, 9999.72265625, 9998.5185546875, 10000.8818359375, 10002.9951171875, 10000.4658203125, 10000.251953125, 9999.513671875, 10002.1171875, 9999.0849609375, 10000.98046875, 10000.3662109375, 9999.916015625, 10000.689453125, 10000.8505859375, 9999.203125, 10000.875, 9998.8369140625, 10000.359375, 10000.0322265625, 9999.2353515625, 10000.4599609375, 10001.3154296875, 10000.8955078125, 10000.1201171875, 9999.37890625, 9999.716796875, 9998.193359375, 10000.478515625, 10001.384765625, 9999.2119140625, 9998.92578125, 9998.9169921875, 10000.603515625, 10000.1962890625, 9998.7568359375, 10000.7138671875, 10000.7919921875, 9999.2802734375, 9998.623046875, 9998.9716796875, 9999.66015625, 10000.0048828125, 9999.4853515625, 10000.0859375, 9999.83984375, 10001.685546875, 9999.658203125, 10002.005859375, 10000.8447265625, 9999.0478515625, 9999.4306640625, 9997.7939453125, 10001.015625, 10000.2001953125, 9999.9599609375, 10000.431640625, 10000.74609375, 10000.7490234375, 9999.1640625, 9999.6240234375, 9999.0068359375, 9998.521484375, 10000.8759765625, 10000.4482421875, 9999.3369140625, 10000.2509765625, 10000.248046875, 10000.34765625, 9999.1513671875, 9999.7333984375, 9999.4287109375, 9997.56640625, 10000.06640625, 9999.521484375, 9998.3017578125, 9998.3701171875, 9999.173828125, 9998.2060546875, 10002.1162109375, 9999.490234375, 10000.9375, 10000.3291015625, 9999.5732421875, 10000.7197265625, 10001.0146484375, 9999.2724609375, 9999.5361328125, 10001.31640625, 9999.345703125, 9999.931640625, 10001.615234375, 9999.5244140625, 10001.0810546875, 9997.80859375, 10001.1484375, 9999.072265625, 9999.9716796875, 9998.1572265625, 10000.728515625, 10002.388671875, 9999.4091796875, 9999.8359375, 9999.7763671875, 10002.291015625, 10001.1015625, 9999.6162109375, 10000.7841796875, 9999.53515625, 10000.921875, 9998.955078125, 10000.626953125, 10002.8984375, 9998.6923828125, 10001.123046875, 10001.1162109375, 10000.4248046875, 9999.7685546875, 10000.2685546875, 10000.5341796875, 10000.7314453125, 10000.7890625, 10000.77734375, 9999.798828125, 9999.7529296875, 9999.3232421875, 9998.9697265625, 10001.0654296875, 10000.3857421875, 9999.0908203125, 9998.291015625, 10001.3583984375, 9999.5078125, 9999.87109375, 10002.4892578125, 9999.8564453125, 9999.380859375, 9999.986328125, 10000.8798828125, 10001.408203125, 9999.447265625, 10000.46484375, 10001.0224609375, 9999.1982421875, 10000.421875, 9999.58203125, 10000.21484375, 9999.166015625, 9998.669921875, 9998.099609375, 9997.7734375, 10000.4873046875, 10001.916015625, 9998.6123046875, 9999.75390625, 10000.49609375, 10001.4765625, 9998.115234375, 9998.6943359375, 9999.931640625, 10000.0849609375, 10001.349609375, 9999.859375, 9999.3359375, 9999.4892578125, 10001.1669921875, 10000.6181640625, 9999.2763671875, 9999.0634765625, 10000.5859375, 10000.171875, 10001.2861328125, 9998.875, 9999.763671875, 10001.6083984375, 10000.708984375, 10001.552734375, 10000.3837890625, 10000.08984375, 9998.87109375, 9997.943359375, 10000.7431640625, 10000.931640625, 10000.8291015625, 9999.896484375, 10000.095703125, 9999.685546875, 9999.5478515625, 10000.90625, 9998.49609375, 9998.1064453125, 9999.2802734375, 10000.2275390625, 9999.1142578125, 10001.140625, 10000.1220703125, 9999.5283203125, 9999.91015625, 9997.8232421875, 10001.2470703125, 9999.564453125, 9998.4833984375, 10000.0068359375, 9998.61328125, 10000.3056640625, 9999.8544921875, 9998.197265625, 9998.53515625, 10000.7548828125, 9999.03125, 10001.8349609375, 10000.8466796875, 9998.9453125, 9999.7568359375, 9999.34375, 10000.14453125, 9999.9638671875, 10000.771484375, 10000.4892578125, 10000.4345703125, 9999.3955078125, 9999.2431640625, 9998.4521484375, 9999.5703125, 10000.69921875, 9999.298828125, 9997.16015625, 10000.3525390625, 9998.6884765625, 9999.9423828125, 9999.564453125, 10001.5595703125, 10001.037109375, 10000.546875, 9999.5712890625, 9998.861328125, 10000.3662109375, 9999.2236328125, 9997.67578125, 9999.9248046875, 10001.7412109375, 10001.91796875, 10000.9150390625, 10000.1357421875, 10000.5546875, 9999.701171875, 9999.5302734375, 10000.3212890625, 9999.375, 9999.6083984375, 9999.849609375, 10000.3037109375, 9999.626953125, 10000.654296875, 10000.046875, 9998.9013671875, 10001.1513671875, 10000.001953125, 10001.2119140625, 10000.318359375, 9998.81640625, 10000.98828125, 10000.6826171875, 10000.3125, 10000.9873046875, 10001.517578125, 9998.8662109375, 10000.0126953125, 9999.849609375, 9998.7763671875, 9999.6484375, 10001.0908203125, 10000.8291015625, 9999.087890625, 9999.8212890625, 9997.23828125, 9999.865234375, 9999.2626953125, 10000.3203125, 9999.3857421875, 9999.74609375, 9999.6513671875, 9998.0654296875, 10000.0341796875, 9998.9609375, 10000.8798828125, 9998.015625, 9999.7275390625, 9999.138671875, 10000.81640625, 10001.5546875, 10001.552734375, 10000.26953125, 10000.3125, 9999.0234375, 9999.78515625, 10001.5263671875, 10000.875, 9999.60546875, 9999.884765625, 9999.541015625, 10002.0498046875, 10001.685546875, 10002.0283203125, 10000.8837890625, 9999.9580078125, 9998.6103515625, 9999.5927734375, 9999.1416015625, 9999.5419921875, 10001.119140625, 10000.6044921875, 10000.986328125, 10000.6943359375, 9999.927734375, 10001.0791015625, 9999.822265625, 10000.2939453125, 10000.2548828125, 10001.5146484375, 10001.15625, 9999.6171875, 9999.3310546875, 10000.041015625, 9999.1083984375, 10002.1708984375, 9999.5869140625, 9999.2841796875, 9999.376953125, 10000.771484375, 10000.8388671875, 9998.8994140625, 10000.8623046875, 9999.548828125, 10000.572265625, 10000.11328125, 10000.6806640625, 10000.4248046875, 9998.0732421875, 10000.7744140625, 9999.0439453125, 9999.8095703125, 9998.966796875, 9999.0068359375, 10000.517578125, 10001.70703125, 10001.0419921875, 9999.556640625, 10000.3291015625, 10000.9580078125, 9999.0400390625, 9999.466796875, 10000.5634765625, 9999.171875, 10000.1904296875, 9999.8583984375, 9998.70703125, 9999.36328125, 10000.5986328125, 10000.53125, 9999.576171875, 9999.1884765625, 10001.2041015625, 10000.7255859375, 10001.6611328125, 10001.091796875, 9999.166015625, 10000.1904296875, 10001.0146484375, 10000.396484375, 10000.453125, 9999.9501953125, 9999.21875, 10000.236328125, 10000.048828125, 9998.37890625, 9999.455078125, 9999.4765625, 9999.1962890625, 9999.7216796875, 9999.5087890625, 10002.0087890625, 10001.7373046875, 10000.7861328125, 10000.36328125, 10000.19921875, 9997.4599609375, 9999.9765625, 10001.28515625, 10002.005859375, 10000.728515625, 9999.7119140625, 10001.3583984375, 9999.5087890625, 9999.404296875, 9998.6455078125, 10000.0556640625, 10002.056640625, 9998.515625, 10000.2353515625, 10001.4873046875, 10000.0634765625, 10000.248046875, 9999.93359375, 10000.845703125, 9998.2021484375, 10000.14453125, 9999.8447265625, 10001.3720703125, 10000.4716796875, 9999.6806640625, 9998.9130859375, 9999.5400390625, 10000.2685546875, 10000.740234375, 10000.71875, 10000.5625, 10000.1669921875, 10000.6162109375, 9999.787109375, 10000.8603515625, 9997.876953125, 9999.583984375, 10000.02734375, 9998.529296875, 9999.8154296875, 10000.921875, 10000.4560546875, 10000.380859375, 10000.2255859375, 9998.4599609375, 9999.4580078125, 10000.1357421875, 9999.578125, 9999.6572265625, 10000.388671875, 9999.4072265625, 10000.6279296875, 9999.013671875, 10000.15234375, 9999.8310546875, 10000.6279296875, 9999.7275390625, 9998.7724609375, 9999.9345703125, 10000.9248046875, 9998.37890625, 10000.7080078125, 10000.6376953125, 9999.0048828125, 10001.8720703125, 9999.9248046875, 10000.091796875, 10001.1337890625, 10001.4912109375, 9999.794921875, 9998.716796875, 10000.1044921875, 9998.9990234375, 10000.1484375, 9999.6845703125, 9999.5244140625, 9999.7841796875, 9999.8125, 9999.3955078125, 10000.392578125, 10000.4580078125, 9999.1552734375, 9999.6201171875, 10000.0234375, 10000.4736328125, 9999.84765625, 10001.013671875, 9999.2109375, 9998.0244140625, 10000.1875, 10001.84765625, 9998.69921875, 10001.8349609375, 9999.486328125, 9999.9521484375, 10000.12890625, 10000.1015625, 9998.775390625, 10001.673828125, 9998.4658203125, 10000.9892578125, 10000.3896484375, 9999.744140625, 10000.4453125, 10001.296875, 9998.53125, 9999.111328125, 9997.7802734375, 10001.7939453125, 10000.6884765625, 10002.0830078125, 10000.0791015625, 9999.4814453125, 9998.0400390625, 9998.5693359375, 10000.880859375, 9998.4365234375, 10000.150390625, 9999.6865234375, 10000.5498046875, 9999.544921875, 9999.8623046875, 9998.6337890625, 9999.76171875, 9999.26953125, 9999.3291015625, 9998.9912109375, 10000.35546875, 10000.751953125, 9999.65234375, 10000.447265625, 10000.1845703125, 9999.5634765625, 10000.8125, 10000.8662109375, 9999.7333984375, 9999.6083984375, 10000.2216796875, 10000.177734375, 9999.720703125, 10000.0107421875, 9998.7548828125, 9998.6103515625, 10000.12109375, 9999.056640625, 10000.2373046875, 9998.9931640625, 10000.8173828125, 10001.4970703125, 10000.5048828125, 9999.1025390625, 10000.0, 10001.18359375, 10001.32421875, 10000.4501953125, 10001.134765625, 10000.166015625, 9999.7314453125, 9999.396484375, 9998.0458984375, 9999.453125, 9999.505859375, 10001.2822265625, 9998.958984375, 9998.654296875, 10003.0556640625, 9998.708984375, 10000.5537109375, 10000.1064453125, 9998.4150390625, 9999.892578125, 10000.0126953125, 9998.904296875, 10000.287109375, 9998.0947265625, 9999.591796875, 9999.8720703125, 9999.4765625, 9999.2880859375, 9999.8115234375, 9998.822265625, 10000.69140625, 9999.7490234375, 10001.072265625, 10001.072265625, 9998.9775390625, 10002.0791015625, 10001.6650390625, 9998.8193359375, 9998.208984375, 9999.2158203125, 9998.1845703125, 9999.5390625, 10001.1171875, 9999.7509765625, 10000.0966796875, 10001.0751953125, 10001.162109375, 10002.37890625, 9999.8623046875, 10000.115234375, 10000.7666015625, 10000.248046875, 10000.1318359375, 9999.1884765625, 9999.310546875, 10001.1630859375, 9999.5185546875, 10000.5361328125, 9999.62109375, 10001.3759765625, 9999.1220703125, 9999.9931640625, 10000.259765625, 10001.5703125, 10000.2568359375, 9998.5205078125, 10000.1875, 10000.08984375, 9998.1884765625, 9998.7138671875, 10000.7958984375, 9999.427734375, 10000.0849609375, 9999.9873046875, 10000.3857421875, 9999.9423828125, 10000.71484375, 9998.5244140625, 10000.390625, 10000.4150390625, 10001.779296875, 9997.4970703125, 9999.5576171875, 10000.724609375, 10001.7705078125, 9996.4375, 9998.1064453125, 10001.1943359375, 9999.931640625, 9999.7021484375, 10000.7431640625, 9998.830078125, 9999.8671875, 9999.7333984375, 10000.83203125, 10000.392578125, 9999.201171875, 9999.353515625, 9999.6171875, 10000.376953125, 9998.845703125, 10001.345703125, 10001.4912109375, 10000.83984375, 9998.4541015625, 9999.759765625, 9999.2451171875, 10000.6201171875, 9998.3623046875, 9999.9228515625, 9999.0302734375, 10001.3017578125, 10000.390625, 9998.755859375, 10000.6015625, 10000.2119140625, 9998.884765625, 9999.083984375, 10000.1123046875, 9999.443359375, 10001.3115234375, 9998.8828125, 9999.9541015625, 10000.033203125, 10001.1796875, 10000.7294921875, 9999.6435546875, 10001.8271484375, 10000.8173828125, 9999.2158203125, 10000.8701171875, 9999.7822265625, 9999.1875, 10000.9228515625, 9999.998046875, 9998.322265625, 9999.158203125, 10000.6748046875, 10000.6982421875, 10002.4189453125, 9998.94921875, 10000.3857421875, 10000.115234375, 9999.4609375, 9999.564453125, 9999.87109375, 10001.748046875, 9999.8212890625, 10001.9765625, 10000.529296875, 9998.71484375, 9999.4560546875, 10002.701171875, 9999.392578125, 10002.6650390625, 9999.45703125, 9999.44140625, 9999.9853515625, 9999.701171875, 10000.33984375, 9999.9560546875, 10000.37109375, 10000.0234375, 9998.3505859375, 10000.4775390625, 9998.1142578125, 10000.1708984375, 9998.9208984375, 9999.109375, 10000.953125, 10000.9013671875, 9999.47265625, 9999.8251953125, 9998.916015625, 10000.4931640625, 9999.0947265625, 10001.4736328125, 9999.3525390625, 10000.5537109375, 9997.7705078125, 10000.638671875, 9998.78515625, 9999.587890625, 9999.1943359375, 10000.98046875, 10001.267578125, 9999.7255859375, 9998.21875, 10000.94921875, 10000.6220703125, 10000.3642578125, 10000.6083984375, 9999.6181640625, 10000.3359375, 10000.0830078125, 9999.34375, 9999.77734375, 9998.3525390625, 10001.40234375, 9999.5205078125, 10000.1171875, 9998.423828125, 9999.8837890625, 10000.271484375, 10000.833984375, 10000.0078125, 10000.4189453125, 10000.541015625, 9999.134765625, 9998.6806640625, 9999.16796875, 9998.3701171875, 9998.7412109375, 10001.0830078125, 10000.630859375, 10001.2373046875, 10000.2734375, 10000.4208984375, 10000.9736328125, 10000.150390625, 9999.611328125, 10000.998046875, 9998.8740234375, 9999.966796875, 10000.04296875, 9999.0, 10001.0107421875, 9999.333984375, 10002.0439453125, 9999.2001953125, 9998.37109375, 9998.3671875, 10000.3271484375, 10000.6396484375, 10001.8486328125, 9999.91796875, 10001.05859375, 10001.33203125, 10001.212890625, 9999.9765625, 9999.509765625, 9999.251953125, 10000.6650390625, 10001.3154296875, 10000.0283203125, 9998.5927734375, 9999.83984375, 9999.537109375, 9999.0185546875, 9999.98828125, 10000.2978515625, 10001.5576171875, 9999.4365234375, 10000.6064453125, 9999.6884765625, 9999.08984375, 10000.1923828125, 10000.9404296875, 10000.57421875, 10000.765625, 10000.0185546875, 10002.583984375, 9999.8359375, 9998.7138671875, 10000.673828125, 9999.5478515625, 9998.666015625, 9999.814453125, 10000.208984375, 9999.4609375, 9998.7255859375, 9999.2666015625, 9999.4375, 9999.8701171875, 10000.4013671875, 9999.0009765625, 9998.658203125, 10000.48046875, 9999.4912109375, 9999.427734375, 10001.3994140625, 10001.12109375, 10001.173828125, 9997.978515625, 9998.4677734375, 9999.91796875, 10000.3212890625, 10000.91015625, 9999.4912109375, 9998.50390625, 10001.765625, 9998.90625, 9999.00390625, 10000.4619140625, 10000.41796875, 9999.12890625, 9998.5283203125, 10000.0166015625, 9998.4501953125, 10001.5322265625, 10000.8134765625, 10000.4755859375, 10000.2275390625, 9998.7109375, 9999.6328125, 10000.193359375, 9999.33984375, 9998.615234375, 9999.4560546875, 10000.427734375, 9999.041015625, 9999.236328125, 9999.6806640625, 10000.447265625, 10000.6494140625, 10000.3232421875, 10001.4013671875, 10000.3212890625, 10001.0986328125, 10000.5859375, 9999.5234375, 10000.8896484375, 10000.3857421875, 10001.2705078125, 9999.9345703125, 10000.2353515625, 9999.75, 10000.384765625, 10000.6796875, 9999.6025390625, 10002.076171875, 10000.8525390625, 9999.5595703125, 9998.056640625, 10002.0, 10000.58984375, 9999.5078125, 10000.158203125, 10000.7197265625, 9998.7333984375, 10000.0966796875, 10002.2470703125, 9998.9794921875, 9997.3466796875, 10000.423828125, 9999.6376953125, 10000.572265625, 9999.6064453125, 9998.7861328125, 10001.10546875, 10000.0703125, 9999.451171875, 9999.1181640625, 10001.0771484375, 9999.599609375, 9999.01171875, 9999.5361328125, 9999.890625, 10000.1240234375, 10000.8525390625, 9999.4365234375, 10000.408203125, 10001.9599609375, 9998.888671875, 10000.060546875, 10001.2490234375, 9999.005859375, 10000.9560546875, 10000.8056640625, 10000.669921875, 9997.5166015625, 9999.5791015625, 10000.23828125, 10000.5556640625, 9999.4208984375, 9999.6416015625, 10001.462890625, 9997.9931640625, 10000.2548828125, 9998.9833984375, 10001.916015625, 9999.55859375, 10001.369140625, 9999.6025390625, 10000.03515625, 9999.8056640625, 10000.8486328125, 9999.1865234375, 10000.6845703125, 9999.5625, 9999.7255859375, 10000.587890625, 10002.0078125, 10001.5458984375, 9998.720703125, 10000.291015625, 10001.013671875, 9998.2265625, 10001.0, 9999.7333984375, 9998.8955078125, 10001.3701171875, 10002.02734375, 10000.7626953125, 9998.185546875, 10001.0107421875, 9999.205078125, 10000.0400390625, 9999.31640625, 10001.2255859375, 10001.845703125, 9999.8681640625, 9999.9150390625, 9998.259765625, 9999.375, 10000.2373046875, 10000.2587890625, 10000.0517578125, 9997.7939453125, 9998.7890625, 9999.458984375, 9998.3310546875, 10000.2998046875, 9999.8369140625, 9998.6962890625, 10000.4892578125, 9999.720703125, 10000.962890625, 9998.30859375, 10000.7265625, 10001.0009765625, 9998.0361328125, 9999.2236328125, 9999.4140625, 9999.34375, 10001.9189453125, 10001.9912109375, 10000.7509765625, 10000.3115234375, 9998.5966796875, 10000.1884765625, 9998.703125, 9998.9609375, 9998.4853515625, 9999.01953125, 9999.185546875, 9999.1533203125, 9998.423828125, 10000.6357421875, 10001.28125, 10000.423828125, 10000.8125, 9999.6455078125, 10000.314453125, 9998.47265625, 10000.3779296875, 10001.822265625, 9999.439453125, 10000.923828125, 10000.35546875, 10002.107421875, 10001.033203125, 9999.1787109375, 9999.2001953125, 10001.125, 9999.9443359375, 9999.8955078125, 9999.5849609375, 9998.8837890625, 10000.90625, 10000.8466796875, 9999.9033203125, 9997.294921875, 9999.5703125, 10000.8173828125, 9999.6474609375, 10000.7060546875, 10002.3623046875, 9999.96875, 10001.193359375, 10000.341796875, 9998.0927734375, 10000.044921875, 10000.1875, 9998.4326171875, 9998.8330078125, 9998.4072265625, 9999.3203125, 10001.0595703125, 10000.9990234375, 10000.5576171875, 10000.6923828125, 10000.462890625, 10000.6572265625, 10001.2578125, 9999.501953125, 10000.337890625, 10002.7109375, 10001.4541015625, 9999.673828125, 10000.21875, 10001.1044921875, 9999.5693359375, 9997.2412109375, 9999.4296875, 9999.9296875, 10000.326171875, 9999.677734375, 10001.482421875, 10003.0166015625, 10001.8486328125, 9998.591796875, 9998.8916015625, 9999.1748046875, 9999.5703125, 10000.2080078125, 9999.552734375, 9998.6298828125, 9999.8193359375, 10000.95703125, 9997.955078125, 9998.921875, 10000.71484375, 9999.0830078125, 10001.306640625, 10000.107421875, 9999.8984375, 9998.830078125, 9999.486328125, 9999.17578125, 10000.3427734375, 9999.9345703125, 10000.833984375, 10000.234375, 9997.82421875, 10001.013671875, 9999.5, 10000.650390625, 9999.970703125, 9999.0224609375, 9998.8505859375, 10001.921875, 10001.16796875, 9998.294921875, 10001.2529296875, 10000.9072265625, 9997.7978515625, 10000.5595703125, 9999.26171875, 10000.8837890625, 9999.7158203125, 9999.375, 9999.001953125, 9999.58984375, 9999.943359375, 10000.076171875, 10001.73828125, 9999.625, 9999.7041015625, 9998.884765625, 9999.587890625, 10000.619140625, 10001.8720703125, 9998.5986328125, 10000.8291015625, 9998.8291015625, 9999.029296875, 9999.4921875, 9998.802734375, 10000.3349609375, 10001.1953125, 10000.08984375, 9999.75, 9999.126953125, 10000.94140625, 9999.453125, 10000.3759765625, 9999.501953125, 9999.8212890625, 9998.0126953125, 10000.3935546875, 10000.8564453125, 9999.287109375, 9999.8798828125, 9999.587890625, 9998.6787109375, 10000.046875, 10000.9921875, 10000.048828125, 9999.8779296875, 9999.185546875, 10000.2958984375, 9999.5849609375, 10000.3427734375, 9998.7001953125, 9999.94140625, 9999.1630859375, 10000.1162109375, 9999.9482421875, 9999.853515625, 9999.052734375, 9998.640625, 10000.2265625, 9999.9287109375, 9998.97265625, 9998.5810546875, 10001.9384765625, 10000.49609375, 9999.931640625, 10000.16015625, 10002.1015625, 10000.2158203125, 9999.7705078125, 10001.697265625, 9998.7744140625, 10000.109375, 10000.27734375, 10000.6455078125, 10000.3779296875, 9999.7333984375, 10001.7529296875, 9999.1884765625, 10000.150390625, 10000.6591796875, 9998.8623046875, 10000.126953125, 9998.345703125, 9998.255859375, 10000.115234375, 9999.7392578125, 10000.7421875, 10000.53515625, 10000.0791015625, 9998.478515625, 9998.93359375, 10001.6748046875, 9998.3408203125, 9999.0732421875, 10000.263671875, 9999.3896484375, 10000.6025390625, 10000.271484375, 10000.626953125, 10000.130859375, 9999.7607421875, 9999.970703125, 10000.076171875, 9998.193359375, 10000.818359375, 10000.3388671875, 9999.3818359375, 9999.2783203125, 9999.6865234375, 9998.076171875, 9998.8759765625, 10000.587890625, 10000.6044921875, 10000.1884765625, 10000.6015625, 9999.4921875, 10000.2958984375, 10001.173828125, 9999.57421875, 10000.4326171875, 10000.21875, 9998.5712890625, 10000.4677734375, 9997.533203125, 9998.349609375, 10000.337890625, 9998.8896484375, 10001.6533203125, 10000.46484375, 10001.13671875, 10000.7353515625, 10001.7607421875, 9998.943359375, 9999.8662109375, 10000.7626953125, 9999.8193359375, 9999.869140625, 9998.7705078125, 9998.3671875, 9998.65234375, 9998.2060546875, 9999.646484375, 9999.6728515625, 9998.830078125, 10001.2275390625, 10000.34765625, 9998.572265625, 10000.244140625, 9997.9384765625, 10000.9072265625, 9999.794921875, 10000.072265625, 10001.662109375, 10000.6171875, 10000.328125, 10000.05078125, 9999.5478515625, 10000.23046875, 10000.6240234375, 10000.0478515625, 10000.400390625, 10002.470703125, 10001.2841796875, 9999.720703125, 10000.478515625, 9999.1669921875, 9998.2373046875, 9999.1787109375, 9999.3427734375, 9998.8671875, 9999.0859375, 9999.505859375, 9999.3994140625, 9999.5986328125, 10001.6416015625, 9999.25, 9999.318359375, 9999.8408203125, 10001.0712890625, 9999.9921875, 9999.76953125, 10001.2080078125, 10000.0517578125, 9999.55859375, 10000.8076171875, 10001.201171875, 9998.9130859375, 9999.9853515625, 10001.5048828125, 9998.9521484375, 9999.8642578125, 10001.302734375, 10000.0703125, 9998.4462890625, 9999.0595703125, 9998.5556640625, 10000.2158203125, 10000.5673828125, 9998.9736328125, 10001.8564453125, 9999.296875, 9999.419921875, 10001.255859375, 10002.810546875, 9999.494140625, 10001.005859375, 9999.109375, 9999.271484375, 10001.6328125, 9998.44140625, 10000.080078125, 10000.1708984375, 9999.9892578125, 10000.5302734375, 10000.806640625, 10000.8583984375, 10000.2685546875, 10001.5361328125, 10000.13671875, 9999.5361328125, 10001.5654296875, 10000.259765625, 9999.90234375, 9998.552734375, 10000.1171875, 9999.4814453125, 10000.5849609375, 10002.158203125, 10001.4365234375, 9998.5390625, 10000.5546875, 10001.3740234375, 9998.1953125, 10001.609375, 10000.2119140625, 10000.7109375, 9999.30859375, 10000.2822265625, 9999.82421875, 10001.740234375, 10000.0986328125, 10000.015625, 9999.546875, 9998.9951171875, 10001.451171875, 10000.623046875, 10000.26953125, 10000.369140625, 9999.46875, 10000.4072265625, 9998.2666015625, 10000.962890625, 10000.935546875, 10000.1767578125, 10000.7177734375, 9999.0712890625, 9999.3173828125, 9998.2021484375, 9998.146484375, 9997.6826171875, 9999.9609375, 10000.17578125, 9998.892578125, 9998.4921875, 10000.849609375, 10000.9677734375, 10000.134765625, 9999.46875, 9998.1201171875, 10002.193359375, 9999.150390625, 10000.1943359375, 9999.7724609375, 10000.7060546875, 10000.076171875, 9999.8544921875, 10001.658203125, 9998.435546875, 9998.556640625, 10000.4873046875, 10001.71484375, 9999.90234375, 10000.0732421875, 10001.8798828125, 9998.8974609375, 9999.29296875, 10000.283203125, 10001.4609375, 9999.3515625, 10001.7314453125, 9999.4501953125, 10000.59765625, 10000.904296875, 9999.658203125, 10000.3154296875, 10001.7392578125, 10001.0849609375, 9999.998046875, 10001.6376953125, 10000.3994140625, 9999.6240234375, 9998.4931640625, 9998.7509765625, 9999.447265625, 10000.3291015625, 9999.5966796875, 10000.10546875, 10000.330078125, 10000.962890625, 10000.4462890625, 10001.4033203125, 10001.640625, 10001.537109375, 9999.9873046875, 9998.7197265625, 10001.533203125, 10000.5341796875, 10001.068359375, 9999.6826171875, 9998.4453125, 10001.419921875, 10001.1748046875, 9999.51171875, 10000.2392578125, 10000.560546875, 10000.1083984375, 10001.3271484375, 9998.744140625, 9999.44921875, 10002.08984375, 10000.029296875, 10000.1708984375, 10000.7890625, 10002.0, 9998.822265625, 9999.826171875, 10000.06640625, 9999.9755859375, 9999.6865234375, 10002.5078125, 10000.4755859375, 10000.69140625, 10000.6845703125, 9998.8330078125, 10002.07421875, 9999.1611328125, 9998.3173828125, 9998.955078125, 9999.5498046875, 9998.72265625, 10000.810546875, 10001.66015625, 10001.634765625, 10000.1494140625, 10001.0205078125, 10000.703125, 10000.2431640625, 10000.4580078125, 9998.7451171875, 10001.0771484375, 10000.193359375, 9998.4287109375, 9999.8974609375, 9999.900390625, 9998.4140625, 10000.1103515625, 10000.4423828125, 10000.1533203125, 10000.57421875, 9999.9775390625, 9999.5107421875, 9999.5126953125, 9998.1611328125, 10001.1904296875, 10000.544921875, 9999.7509765625, 10000.30078125, 9998.8828125, 9999.990234375, 10000.6640625, 10000.5, 9999.6650390625, 9997.8046875, 9999.431640625, 10000.5615234375, 9999.736328125, 9999.34765625, 10001.01953125, 10000.4189453125, 10000.3994140625, 10000.0048828125, 9998.828125, 10000.5546875, 9999.9248046875, 10001.0927734375, 9999.8017578125, 9999.6767578125, 10000.275390625, 10001.078125, 9999.4169921875, 9998.068359375, 9999.78125, 9998.3701171875, 10000.0400390625, 10000.087890625, 10000.9375, 10001.2177734375, 10000.140625, 10002.2861328125, 10000.865234375, 9999.6826171875, 10000.3154296875, 10000.576171875, 10000.64453125, 10000.9501953125, 10001.376953125, 10000.8505859375, 10000.7939453125, 9999.1005859375, 9999.2578125, 10001.544921875, 10002.123046875, 9998.6025390625, 9999.837890625, 10000.767578125, 10001.7998046875, 9999.5283203125, 9998.3984375, 9998.564453125, 10000.818359375, 10000.439453125, 9997.62109375, 9999.994140625, 10001.6279296875, 10002.529296875, 10000.3486328125, 10000.6708984375, 10000.2978515625, 10001.357421875, 10000.572265625, 10000.43359375, 10000.3037109375, 9999.9658203125, 10000.0341796875, 9999.4990234375, 10000.6142578125, 9998.90625, 9998.7216796875, 9997.3623046875, 10002.2138671875, 10000.4404296875, 9998.07421875, 10000.8076171875, 9999.736328125, 9998.11328125, 10000.9296875, 9998.9775390625, 9998.638671875, 10000.7646484375, 10001.5751953125, 9999.939453125, 9999.1962890625, 10000.205078125, 9999.5986328125, 10001.5087890625, 10000.0751953125, 10001.9580078125, 9999.42578125, 10000.109375, 10000.5615234375, 9999.5078125, 10000.70703125, 10001.2080078125, 9999.6904296875, 9999.458984375, 9998.283203125, 9999.6123046875, 10000.80078125, 9999.6357421875, 9998.9052734375, 9999.8173828125, 9999.8212890625, 9999.7861328125, 10000.0029296875, 10000.294921875, 9999.07421875, 10000.6806640625, 9999.0224609375, 10000.69140625, 10000.296875, 10001.1962890625, 10000.068359375, 9998.8671875, 10000.693359375, 10000.0986328125, 9999.388671875, 10000.2275390625, 9997.865234375, 10000.3212890625, 10000.37109375, 10000.1005859375, 9997.966796875, 10001.2685546875, 10000.212890625, 9999.576171875, 10000.87890625, 9998.6083984375, 10000.67578125, 10001.88671875, 9999.9365234375, 10000.8994140625, 9999.3271484375, 10000.568359375, 9998.7685546875, 10001.041015625, 10001.2216796875, 10000.2890625, 10000.912109375, 10000.609375, 10001.5634765625, 10000.0078125, 10000.396484375, 9998.9072265625, 9999.390625, 10000.08203125, 9998.9892578125, 9998.65234375, 10000.2734375, 10001.1162109375, 10000.9150390625, 10000.197265625, 10000.05078125, 10001.9873046875, 9999.0927734375, 10001.466796875, 9999.70703125, 9999.7197265625, 9998.62109375, 10000.8525390625, 10001.423828125, 9999.7216796875, 9998.9462890625, 9999.24609375, 10000.6787109375, 10000.71484375, 9999.2734375, 10001.095703125, 10001.5791015625, 9998.3896484375, 9998.8798828125, 10000.810546875, 10001.1435546875, 9998.892578125, 9999.7529296875, 9999.064453125, 10002.13671875, 9998.0830078125, 9998.076171875, 9999.7998046875, 9999.5849609375, 10000.4228515625, 10002.2822265625, 9999.7119140625, 10001.2646484375, 10000.40234375, 10000.875, 9998.8759765625, 9999.7509765625, 10000.9873046875, 10000.8203125, 10002.4697265625, 10001.1611328125, 9999.6552734375, 9998.3837890625, 10000.2734375, 9999.05078125, 10000.56640625, 9999.2490234375, 9999.5859375, 9997.8603515625, 10001.3310546875, 10000.2607421875, 9998.625, 9998.951171875, 10000.1611328125, 9998.189453125, 10000.8720703125, 9998.6416015625, 10000.1083984375, 9999.5947265625, 9999.892578125, 9998.6044921875, 9999.408203125, 10002.12890625, 9998.8916015625, 10000.0205078125, 10000.880859375, 10000.818359375, 10000.990234375, 9998.4228515625, 9999.7861328125, 10000.0576171875, 10000.58203125, 9998.826171875, 9998.6826171875, 10000.091796875, 10000.8369140625, 9999.4267578125, 9999.3232421875, 9999.96484375, 10000.87890625, 9998.7919921875, 10000.4814453125, 10000.0009765625, 10000.0146484375, 10001.271484375, 9998.5400390625, 10001.1552734375, 9998.447265625, 10000.5634765625, 10000.6884765625, 9999.0302734375, 9999.15234375, 10000.2294921875, 9999.0234375, 10000.1806640625, 10000.0390625, 10000.5908203125, 9999.8095703125, 9999.240234375, 9999.37890625, 10000.5634765625, 10001.14453125, 10001.14453125, 9999.037109375, 9999.1328125, 9999.7724609375, 9999.6806640625, 10001.3974609375, 10001.1064453125, 10001.0966796875, 9999.4072265625, 10000.154296875, 10000.390625, 10000.8681640625, 9998.2724609375, 10001.7978515625, 10000.8798828125, 10000.72265625, 10000.5419921875, 9999.05078125, 10001.8828125, 9999.3779296875, 10000.240234375, 9997.787109375, 9999.6474609375, 10001.888671875, 9999.044921875, 10000.5185546875, 9999.5546875, 10000.28515625, 10000.27734375, 10000.1220703125, 10000.220703125, 10000.4306640625, 9998.6572265625, 9999.1787109375, 9998.82421875, 9999.861328125, 10002.5693359375, 10001.3662109375, 10001.3310546875, 9998.087890625, 9999.6083984375, 10000.8583984375, 10001.4345703125, 9998.869140625, 10000.572265625, 9998.435546875, 10000.4775390625, 9999.048828125, 9997.9033203125, 9999.015625, 9998.810546875, 10000.26953125, 10000.171875, 9999.3857421875, 9999.421875, 10001.0322265625, 9998.20703125, 9999.921875, 10001.611328125, 9999.90625, 9999.70703125, 9999.5537109375, 9999.91015625, 9999.166015625, 10001.197265625, 9999.90234375, 9999.42578125, 9999.1787109375, 10000.8076171875, 10000.521484375, 10000.5947265625, 9999.287109375, 9999.37890625, 9999.6943359375, 10001.2177734375, 9999.7705078125, 9999.9912109375, 9999.345703125, 10000.224609375, 10000.01953125, 9998.4033203125, 9998.1083984375, 9999.2099609375, 9998.791015625, 10001.029296875, 9999.505859375, 10002.4853515625, 10001.6083984375, 9999.6396484375, 9998.7275390625, 10000.4345703125, 9999.72265625, 9999.001953125, 9998.4443359375, 10001.515625, 10000.8974609375, 9998.8759765625, 9999.12890625, 10000.3203125, 10001.97265625, 9998.80078125, 9997.6181640625, 10002.400390625, 10000.2822265625, 9999.875, 10000.623046875, 10000.9345703125, 10000.11328125, 9998.6953125, 9998.583984375, 9999.5341796875, 9999.0078125, 10000.80859375, 10000.8662109375, 9999.474609375, 10000.205078125, 9998.5634765625, 10000.3330078125, 9999.0771484375, 10001.26171875, 9999.3681640625, 10000.3544921875, 9999.8798828125, 9999.1279296875, 9999.2431640625, 10000.3427734375, 10000.0849609375, 10000.3076171875, 9998.7177734375, 9999.55078125, 9999.3125, 10001.0146484375, 9998.341796875, 10001.2880859375, 10000.703125, 9999.0712890625, 9998.4453125, 9999.7431640625, 9999.3369140625, 10000.138671875, 9999.734375, 9999.5546875, 10000.8662109375, 9999.296875, 10001.50390625, 10000.5517578125, 10001.4794921875, 10001.095703125, 10000.751953125, 10000.1689453125, 9999.3681640625, 9999.3310546875, 10000.55078125, 10000.82421875, 9998.419921875, 9998.40625, 10002.0712890625, 9999.8056640625, 9998.83203125, 10000.052734375, 9998.48046875, 9999.3671875, 9998.9296875, 10001.5205078125, 10000.814453125, 9999.982421875, 10001.99609375, 10001.013671875, 10000.498046875, 10000.3857421875, 10001.150390625, 9999.1962890625, 10000.291015625, 9998.650390625, 10001.1796875, 10000.630859375, 10000.130859375, 9999.7646484375, 10000.0400390625, 9999.2177734375, 10000.271484375, 9999.3193359375, 10000.12890625, 9999.6826171875, 10001.2041015625, 10001.47265625, 10001.3173828125, 10000.0126953125, 9997.6337890625, 9997.0732421875, 10000.2392578125, 9997.1982421875, 9999.111328125, 9999.3173828125, 9999.1904296875, 9996.88671875, 9999.9951171875, 9999.4833984375, 9999.759765625, 9999.8037109375, 9999.2333984375, 9999.1962890625, 9998.8134765625, 9999.583984375, 9999.1240234375, 9999.646484375, 9998.2265625, 9999.42578125, 10001.669921875, 10000.3046875, 10001.5693359375, 10003.083984375, 10002.734375, 10000.2451171875, 9998.556640625, 10000.5244140625, 9999.57421875, 10000.6240234375, 10000.9228515625, 10000.541015625, 9998.6904296875, 10000.01171875, 10001.0185546875, 9999.5703125, 10000.7353515625, 10001.2373046875, 10000.3779296875, 9997.978515625, 9997.91796875, 9998.560546875, 10001.5400390625, 10001.1201171875, 9999.5869140625, 10000.6025390625, 9999.9462890625, 10000.955078125, 10000.7626953125, 9999.431640625, 9999.1201171875, 9998.4736328125, 10001.57421875, 9999.123046875, 10000.05078125, 10000.5166015625, 9998.6162109375, 9999.560546875, 10000.990234375, 9998.3671875, 9998.9404296875, 9999.978515625, 9999.4443359375, 10000.4521484375, 10000.9560546875, 10001.71484375, 9999.4931640625, 9999.1552734375, 9999.4189453125, 10000.9150390625, 9999.1533203125, 9999.59765625, 9999.9453125, 10000.8505859375, 10000.595703125, 9998.9775390625, 10002.705078125, 9999.521484375, 9999.1845703125, 10000.359375, 10000.9375, 9998.4384765625, 10000.72265625, 10000.3017578125, 9999.5693359375, 9999.1337890625, 9999.44921875, 10001.42578125, 9997.712890625, 10000.712890625, 10000.30078125, 9999.7978515625, 10000.4873046875, 9999.7265625, 10001.8583984375, 10000.041015625, 10000.1259765625, 9999.5546875, 10000.7548828125, 10001.1240234375, 9999.826171875, 9999.2490234375, 9999.7392578125, 9999.9580078125, 9998.8935546875, 9999.0234375, 10000.4814453125, 9999.69921875, 9999.8896484375, 9998.9150390625, 10000.0185546875, 10000.919921875, 9999.8193359375, 9999.4814453125, 10000.181640625, 9999.8359375, 10000.6630859375, 9999.044921875, 9999.9990234375, 10001.0546875, 10001.6650390625, 9999.0615234375, 9998.1064453125, 9998.8896484375, 10000.580078125, 10000.4208984375, 9999.5146484375, 9998.787109375, 9999.6650390625, 9999.298828125, 9998.6357421875, 9999.24609375, 10001.0556640625, 10000.0517578125, 10000.3974609375, 10001.5048828125, 9999.8974609375, 9999.9267578125, 10000.07421875, 9999.84765625, 9999.6904296875, 10000.892578125, 10000.7255859375, 9999.751953125, 10000.34765625, 10000.654296875, 10000.2666015625, 10000.484375, 9998.8193359375, 9999.212890625, 10000.43359375, 10002.025390625, 9999.310546875, 9998.72265625, 10000.7001953125, 10000.669921875, 10000.345703125, 10000.2119140625, 10001.3037109375, 9999.533203125, 10000.552734375, 9999.4599609375, 9999.28515625, 10000.837890625, 9999.7822265625, 9998.955078125, 9998.5693359375, 9999.46875, 9999.91015625, 10000.98828125, 10001.0712890625, 10000.625, 10000.2548828125, 10000.1357421875, 9999.8740234375, 10001.1416015625, 9999.1787109375, 9999.275390625, 10000.265625, 10000.7626953125, 9999.9404296875, 9999.939453125, 9999.4267578125, 9999.7900390625, 9999.421875, 10000.703125, 9999.5517578125, 10000.97265625, 10000.408203125, 9997.1181640625, 10000.01171875, 10000.7734375, 9999.0439453125, 10000.455078125, 10000.74609375, 10000.1669921875, 10000.7373046875, 9999.423828125, 10000.111328125, 9998.4326171875, 10000.669921875, 9999.1435546875, 10000.123046875, 10000.1728515625, 10000.5712890625, 10001.060546875, 10001.1953125, 10000.19921875, 9999.8798828125, 9998.8525390625, 10000.826171875, 10001.2294921875, 10000.5087890625, 10000.2626953125, 10000.2294921875, 10000.4638671875, 9999.4423828125, 10001.4658203125, 9999.8115234375, 10000.47265625, 9998.90234375, 9999.9482421875, 9998.9736328125, 9997.43359375, 10001.095703125, 9999.7646484375, 9999.4853515625, 9999.3056640625, 9999.1669921875, 9999.7783203125, 9999.701171875, 10000.2314453125, 10001.0673828125, 9998.59765625, 10000.7314453125, 9998.830078125, 10001.5380859375, 9999.7392578125, 9999.9541015625, 10000.2275390625, 10000.9267578125, 10001.48046875, 10000.23046875, 10000.2646484375, 10000.65234375, 9999.4404296875, 10000.4111328125, 10000.0576171875, 9999.79296875, 9997.0029296875, 9998.6728515625, 10000.146484375, 10001.548828125, 10000.107421875, 9999.9404296875, 10000.7646484375, 10000.60546875, 9999.603515625, 9999.2822265625, 10000.880859375, 10001.5400390625, 10000.6591796875, 9999.31640625, 9999.998046875, 10000.8046875, 10000.90234375, 9999.4365234375, 10000.8447265625, 9998.955078125, 10000.5986328125, 9999.9326171875, 10000.9150390625, 10000.0087890625, 10001.5869140625, 10000.6357421875, 10001.796875, 9999.513671875, 9999.947265625, 9998.712890625, 10000.294921875, 9999.998046875, 9997.1015625, 9999.7705078125, 10001.4609375, 10000.728515625, 10001.08203125, 9999.1728515625, 10001.5439453125, 9999.662109375, 10000.1748046875, 9999.01171875, 9999.451171875, 10000.93359375, 9999.4150390625, 10001.0341796875, 9998.1875, 9998.630859375, 10001.2783203125, 9999.4404296875, 10000.2724609375, 10000.7490234375, 9998.1337890625, 9999.2607421875, 9998.8076171875, 10000.2958984375, 10001.5029296875, 10000.359375, 10000.5244140625, 10001.8408203125, 10000.5126953125, 10000.736328125, 10001.3046875, 10000.251953125, 9999.921875, 10000.4814453125, 9999.8076171875, 9999.943359375, 10000.9306640625, 10000.744140625, 10001.43359375, 10000.1044921875, 9999.431640625, 10000.62890625, 10000.30078125, 10000.5869140625, 10002.1826171875, 10000.2177734375, 10000.2275390625, 10001.66015625, 10000.3056640625, 9999.5859375, 9999.5107421875, 9999.1279296875, 9998.8876953125, 10000.982421875, 10000.478515625, 9999.4404296875, 10000.50390625, 10000.3603515625, 9998.8525390625, 9999.66015625, 9999.7109375, 10000.2978515625, 9999.0107421875, 10001.185546875, 9998.7529296875, 9999.5478515625, 10000.4296875, 10000.5771484375, 10000.806640625, 10000.3515625, 10000.529296875, 9998.60546875, 9998.599609375, 10001.0859375, 9999.96875, 10001.064453125, 10000.1728515625, 9999.515625, 10000.58203125, 10001.287109375, 10000.24609375, 10002.8720703125, 9998.9130859375, 9999.708984375, 10000.4755859375, 10001.0, 9999.9443359375, 10000.40234375, 10000.833984375, 10000.4697265625, 10001.419921875, 9999.626953125, 9998.4404296875, 9999.2265625, 10001.361328125, 10000.9345703125, 9998.4921875, 10000.94921875, 10000.3662109375, 9999.978515625, 10000.205078125, 10000.556640625, 10001.2099609375, 10000.28515625, 10000.9208984375, 10001.3486328125, 10001.4560546875, 10000.9306640625, 10002.80859375, 10001.2333984375, 9998.6494140625, 9999.03515625, 10001.548828125, 10000.0322265625, 10000.3681640625, 10000.7958984375, 10000.078125, 9999.927734375, 10000.97265625, 10000.0537109375, 10000.4853515625, 9997.48828125, 10001.0185546875, 10003.1572265625, 10000.8662109375, 9999.3740234375, 10000.056640625, 9999.494140625, 9999.4365234375, 10001.73828125, 10000.82421875, 10000.1884765625, 10001.421875, 9998.87109375, 9998.5888671875, 10000.697265625, 9999.533203125, 10000.5205078125, 10001.677734375, 10000.130859375, 10000.0634765625, 10001.130859375, 10000.2646484375, 10000.185546875, 10001.0771484375, 9999.853515625, 10000.5263671875, 9999.5322265625, 10001.634765625, 10000.0693359375, 10000.599609375, 10000.3212890625, 9999.96484375, 10000.7548828125, 9998.2529296875, 9999.1953125, 9999.8212890625, 9998.3916015625, 9999.9833984375, 9999.478515625, 10000.0888671875, 10000.0927734375, 10000.765625, 10001.6513671875, 10000.3447265625, 9999.228515625, 10001.58984375, 10000.2685546875, 10001.1640625, 10000.3994140625, 10000.1572265625, 10000.119140625, 9999.9755859375, 10002.451171875, 9998.79296875, 9999.8046875, 9998.439453125, 9999.7568359375, 9998.2080078125, 10000.4208984375, 10000.30859375, 9999.9912109375, 10000.9921875, 10001.5869140625, 9999.8427734375, 10000.2529296875, 9999.53125, 9999.4501953125, 10002.3193359375, 10000.7431640625, 10000.0771484375, 9999.3408203125, 10000.490234375, 9999.7763671875, 9999.3974609375, 9999.22265625, 9998.6923828125, 9998.8623046875, 10001.2666015625, 9999.0498046875, 9999.919921875, 9998.681640625, 9999.994140625, 10000.7783203125, 9999.662109375, 10000.3154296875, 10000.876953125, 10000.4521484375, 9999.572265625, 9999.298828125, 9999.251953125, 10000.2236328125, 9999.5390625, 10000.1025390625, 10000.21875, 9998.478515625, 10000.2822265625, 10001.23046875, 10000.7919921875, 10000.1611328125, 10001.1142578125, 10001.123046875, 10000.091796875, 9999.8505859375, 10000.1171875, 10000.44921875, 9998.798828125, 9999.7421875, 10002.265625, 9999.763671875, 10000.408203125, 9999.84765625, 9999.8173828125, 10001.3623046875, 10001.091796875, 10000.640625, 10000.830078125, 10000.546875, 10000.0234375, 9999.5751953125, 10000.802734375, 10000.59765625, 10000.6103515625, 10000.0869140625, 9999.8125, 10001.328125, 9999.2607421875, 9999.068359375, 10000.5361328125, 9998.796875, 10000.5, 10000.5205078125, 9998.251953125, 9999.9453125, 9999.708984375, 10000.0732421875, 9998.771484375, 9999.701171875, 9999.8515625, 10000.0771484375, 9999.4345703125, 10001.0791015625, 9999.5419921875, 10000.60546875, 10000.443359375, 9999.9990234375, 9999.765625, 10000.8544921875, 9999.1826171875, 10000.8203125, 10001.400390625, 10000.408203125, 10001.4736328125, 9999.7587890625, 9999.3671875, 9998.888671875, 10000.2216796875, 9998.19140625, 9998.6474609375, 10001.4521484375, 9999.1943359375, 10000.3759765625, 9998.9970703125, 10001.134765625, 10000.419921875, 9999.1982421875, 10001.837890625, 10000.3916015625, 10001.337890625, 10000.53125, 9999.4150390625, 9999.3681640625, 9998.1142578125, 10001.583984375, 9999.1728515625, 10000.7236328125, 10000.205078125, 10000.482421875, 9999.2666015625, 10000.1611328125, 9998.5048828125, 10000.8408203125, 10000.6494140625, 10001.5634765625, 10001.8076171875, 10000.46875, 9999.609375, 9998.873046875, 10001.00390625, 9999.9638671875, 10001.251953125, 10001.833984375, 9999.134765625, 9999.0439453125, 10001.302734375, 9999.4267578125, 9999.9345703125, 10002.0068359375, 9999.9873046875, 10000.0478515625, 9998.953125, 9999.3212890625, 9999.869140625, 10001.060546875, 9999.3076171875, 9999.751953125, 9998.759765625, 9999.34375, 10000.1552734375, 9998.59765625, 10000.5654296875, 9999.7685546875, 10001.513671875, 9999.359375, 9999.12109375, 9999.2783203125, 10001.1181640625, 9999.8837890625, 9999.7666015625, 9998.5439453125, 9999.0712890625, 10000.1142578125, 9999.7900390625, 10001.9931640625, 9999.333984375, 9999.4375, 9999.9658203125, 9999.8818359375, 9998.7216796875, 9999.421875, 9999.3037109375, 10000.6640625, 10000.2822265625, 9999.091796875, 9999.953125, 9999.8291015625, 10000.3857421875, 10001.517578125, 9999.62890625, 10001.0185546875, 9999.8359375, 10000.2939453125, 9999.75, 10000.111328125, 9998.8212890625, 10000.9580078125, 9999.4833984375, 9999.796875, 10001.181640625, 10000.9609375, 9999.9755859375, 10000.4658203125, 10001.072265625, 10001.3017578125, 10001.2578125, 10001.662109375, 9999.9404296875, 9998.8525390625, 9998.8046875, 9999.857421875, 10001.326171875, 9998.8984375, 10000.0986328125, 10001.4599609375, 9999.8896484375, 9999.580078125, 9999.4599609375, 9999.2509765625, 10000.5234375, 9998.5732421875, 9999.8994140625, 9999.705078125, 9999.8232421875, 10000.3974609375, 10000.5634765625, 9999.625, 10001.5263671875, 9999.326171875, 9999.197265625, 10001.138671875, 9999.4423828125, 10000.623046875, 10000.80078125, 10000.2958984375, 9999.8369140625, 10000.380859375, 9997.607421875, 10001.82421875, 9999.2998046875, 9998.990234375, 10001.5830078125, 10000.9345703125, 9999.1171875, 10000.5244140625, 10002.2841796875, 10000.4228515625, 9999.505859375, 10000.158203125, 10000.171875, 9999.283203125, 9998.771484375, 10001.6123046875, 10001.84765625, 10000.1689453125, 9999.0546875, 9999.96484375, 10001.267578125, 9999.3134765625, 10001.427734375, 10001.158203125, 10000.8251953125, 10000.083984375, 10000.015625, 10001.1357421875, 9999.6533203125, 10000.4892578125, 10001.0625, 10000.8837890625, 10001.7255859375, 9999.765625, 10000.6923828125, 10001.9638671875, 9999.4462890625, 9999.0234375, 10000.08203125, 10001.1591796875, 9999.6337890625, 10000.2431640625, 9999.2470703125, 9999.9580078125, 9998.966796875, 9999.5498046875, 9998.044921875, 10000.708984375, 9998.4912109375, 10000.873046875, 9999.9228515625, 10001.1279296875, 9999.8447265625, 10000.4853515625, 10001.6025390625, 9999.8828125, 9998.6650390625, 9998.4033203125, 10000.37890625, 10000.2509765625, 10001.0380859375, 10001.16796875, 9997.4912109375, 9999.2490234375, 10000.14453125, 10001.1982421875, 9998.7880859375, 10000.5830078125, 9999.822265625, 10000.0048828125, 9999.67578125, 10001.1455078125, 9999.85546875, 10001.00390625, 10002.091796875, 10000.51171875, 10000.203125, 10000.2119140625, 9998.4375, 9999.935546875, 9999.5615234375, 9999.9462890625, 9998.9287109375, 10000.486328125, 10001.310546875, 10000.6826171875, 10001.361328125, 10000.3046875, 9998.310546875, 10000.9814453125, 10001.025390625, 9999.8408203125, 10001.130859375, 10001.25, 9999.7001953125, 9998.3408203125, 9999.7431640625, 9998.662109375, 9999.8056640625, 9998.529296875, 9999.134765625, 10000.255859375, 10000.1884765625, 9999.6806640625, 10000.994140625, 9999.5234375, 10000.4716796875, 10000.4912109375, 10000.8623046875, 9999.521484375, 10001.4892578125, 10000.78515625, 10000.900390625, 9999.044921875, 9999.1748046875, 9999.40234375, 9999.974609375, 9999.4833984375, 9999.8486328125, 10001.419921875, 10000.6708984375, 9999.9521484375, 10000.673828125, 9999.1240234375, 9999.177734375, 9999.912109375, 10001.166015625, 10000.7734375, 10000.7939453125, 10002.4306640625, 9999.830078125, 9999.4326171875, 10000.5654296875, 10000.3896484375, 9999.3115234375, 9997.720703125, 10001.2138671875, 10000.353515625, 10000.439453125, 9999.2890625, 10000.0703125, 9999.5791015625, 10000.494140625, 10001.1298828125, 10000.1484375, 9999.705078125, 10000.8505859375, 10001.55859375, 10001.1669921875, 10001.5966796875, 10001.5, 9999.123046875, 10000.205078125, 10000.6904296875, 9998.4677734375, 9998.1162109375, 9999.7451171875, 9999.8828125, 9998.7783203125, 9999.443359375, 10000.1396484375, 10000.23046875, 10000.5263671875, 10000.81640625, 9999.896484375, 9999.6845703125, 10001.25390625, 10001.0283203125, 10000.7734375, 10001.533203125, 10000.232421875, 10000.419921875, 10001.0458984375, 9999.2607421875, 10000.751953125, 9998.34375, 9998.833984375, 10000.9990234375, 10000.869140625, 10001.1396484375, 10002.099609375, 9999.6650390625, 10000.8857421875, 10001.8955078125, 10001.2841796875, 10001.2568359375, 9999.943359375, 9999.9150390625, 10000.123046875, 10001.1337890625, 10001.51953125, 10001.9521484375, 9999.169921875, 9999.70703125, 9999.337890625, 10001.2578125, 9999.6748046875, 10000.2060546875, 9999.619140625, 10000.46484375, 10001.1875, 10000.1728515625, 10000.0380859375, 10000.16796875, 10000.6796875, 10000.681640625, 9999.443359375, 9999.7666015625, 10000.5966796875, 10000.43359375, 10000.03125, 9998.6259765625, 10000.3857421875, 10000.4599609375, 10000.51171875, 9998.82421875, 10000.912109375, 9998.64453125, 9999.4443359375, 10000.486328125, 10000.5126953125, 10002.2041015625, 10000.7041015625, 9999.4501953125, 10001.0078125, 10001.0869140625, 9998.99609375, 9999.75, 9998.833984375, 9999.9560546875, 9999.662109375, 10000.8212890625, 9999.4228515625, 9999.548828125, 10000.123046875, 10001.4091796875, 9999.5576171875, 9999.5576171875, 9999.2890625, 10000.4521484375, 10000.5263671875, 9998.4970703125, 10000.138671875, 9998.6123046875, 9999.0830078125, 9998.140625, 9999.6904296875, 9999.443359375, 10000.3388671875, 10000.4755859375, 10001.015625, 9999.3291015625, 10000.2587890625, 10001.2890625, 9999.3828125, 10000.7626953125, 10000.94921875, 9997.5595703125, 10000.0966796875, 10000.6513671875, 9999.080078125, 10000.27734375, 10000.2412109375, 9999.771484375, 9999.865234375, 10002.34765625, 9999.0419921875, 9999.1201171875, 10000.703125, 10001.2646484375, 9999.2626953125, 10000.5205078125, 10000.771484375, 10000.5537109375, 9999.787109375, 10000.1171875, 10000.0478515625, 10001.9736328125, 9999.8173828125, 9999.7421875, 9999.408203125, 10000.3662109375, 10001.2138671875, 9999.1142578125, 9999.763671875, 9999.6669921875, 9999.2099609375, 9999.19921875, 9999.91796875, 10000.1533203125, 10000.42578125, 9999.49609375, 10000.7607421875, 9999.6630859375, 10000.4990234375, 9999.2578125, 9999.0439453125, 10000.3623046875, 10000.669921875, 9999.564453125, 10000.4169921875, 9999.8623046875, 9999.7470703125, 10000.271484375, 10001.53515625, 9999.45703125, 9999.708984375, 9998.1572265625, 9999.525390625, 10000.0390625, 10000.7001953125, 9998.966796875, 10000.48828125, 10001.0947265625, 9999.6171875, 9999.8984375, 10002.0703125, 9999.8349609375, 9999.4775390625, 10000.599609375, 10000.41015625, 9997.9755859375, 10000.26953125, 10000.421875, 9999.70703125, 10001.5556640625, 10000.3154296875, 10000.0029296875, 10000.58203125, 10000.1064453125, 9999.5244140625, 9999.6025390625, 9998.708984375, 9999.7958984375, 10000.12890625, 9997.96875, 9999.80859375, 10001.1474609375, 10000.603515625, 9999.0625, 10001.3779296875, 10001.9541015625, 9998.830078125, 9999.1904296875, 10002.01953125, 10001.3447265625, 10000.9775390625, 10000.314453125, 10001.642578125, 9999.546875, 10001.12890625, 9998.4619140625, 10001.2919921875, 10000.6904296875, 10000.419921875, 9999.35546875, 10001.4599609375, 9999.7265625, 9999.7373046875, 10000.517578125, 9999.041015625, 10000.525390625, 9998.44921875, 10001.146484375, 9999.0458984375, 10002.1123046875, 9998.904296875, 10001.517578125, 10000.720703125, 10000.8203125, 10000.533203125, 9999.7255859375, 9998.90234375, 9999.0966796875, 9999.3876953125, 10001.5888671875, 10000.0185546875, 10000.98828125, 10001.9384765625, 9999.115234375, 10000.4228515625, 9998.7841796875, 10000.9384765625, 9999.4501953125, 9999.5234375, 9999.994140625, 9999.681640625, 9999.7353515625, 10000.8876953125, 10000.572265625, 10001.6015625, 9999.224609375, 9999.7294921875, 9999.9794921875, 10000.513671875, 10000.3095703125, 10000.8203125, 10001.4267578125, 9999.6787109375, 9998.6923828125, 10000.5234375, 10000.478515625, 10001.55078125, 10000.1123046875, 9999.376953125, 10000.0390625, 9999.79296875, 9999.2548828125, 10000.63671875, 10000.04296875, 10001.251953125, 10000.2080078125, 10000.5107421875, 9998.1337890625, 10001.7724609375, 9999.830078125, 9998.65625, 10000.0634765625, 9999.0126953125, 10000.669921875, 9999.8505859375, 10000.22265625, 10000.935546875, 10000.1455078125, 9999.4619140625, 9999.140625, 10000.8408203125, 10000.7197265625, 9999.4609375, 10001.685546875, 9999.8466796875, 9999.658203125, 10001.0908203125, 9999.94921875, 9998.6904296875, 10000.5634765625, 10002.0146484375, 9998.7265625, 9999.3857421875, 10000.8935546875, 9999.2060546875, 10000.4365234375, 10000.5126953125, 9998.9990234375, 10000.630859375, 10000.158203125, 10001.2021484375, 10000.458984375, 9997.7822265625, 10002.5732421875, 10000.4619140625, 9999.541015625, 10000.3857421875, 10000.23046875, 9999.84765625, 10000.6455078125, 9999.7431640625, 9999.21875, 9999.009765625, 10000.5244140625, 10002.732421875, 10000.1376953125, 10000.455078125, 10001.017578125, 9998.6220703125, 10000.6591796875, 10000.9345703125, 9999.79296875, 9999.7080078125, 10000.0673828125, 9999.6669921875, 10000.7822265625, 10000.0849609375, 10000.5224609375, 10000.4638671875, 9998.3681640625, 9999.21875, 9999.87109375, 9998.697265625, 9998.4990234375, 9997.62109375, 9999.9853515625, 9999.8837890625, 10000.6748046875, 10000.4658203125, 10000.763671875, 10000.9130859375, 10000.07421875, 10000.3486328125, 10000.9658203125, 9999.2880859375, 10000.033203125, 10001.2998046875, 10001.2724609375, 9997.6884765625, 10000.4052734375, 10001.080078125, 9999.26171875, 10000.7294921875, 10000.0517578125, 9999.9375, 10000.677734375, 9999.625, 10000.5234375, 9999.5634765625, 9999.248046875, 10000.3642578125, 9999.279296875, 10000.6044921875, 9999.0029296875, 10000.30859375, 10000.421875, 10001.0517578125, 10001.0751953125, 9999.197265625, 9998.443359375, 10002.2099609375, 10002.4541015625, 10000.36328125, 10001.7177734375, 9998.9013671875, 9998.55859375, 10000.16796875, 9999.8251953125, 10001.681640625, 10000.7568359375, 10000.3896484375, 9999.234375, 10001.126953125, 10002.7490234375, 10000.6318359375, 9999.9697265625, 10001.3310546875, 10000.59765625, 10000.5439453125, 9999.5615234375, 10001.4736328125, 10000.7802734375, 10001.328125, 9999.798828125, 9998.876953125, 9998.66015625, 10000.8046875, 10000.3408203125, 10000.4912109375, 10000.123046875, 9998.609375, 10001.1669921875, 10000.37890625, 10002.255859375, 10000.8427734375, 10000.916015625, 10001.22265625, 9999.931640625, 10000.6083984375, 9999.6279296875, 9999.7177734375, 9999.8466796875, 9999.859375, 10001.6298828125, 9998.49609375, 10000.060546875, 9999.3232421875, 9999.189453125, 9999.6083984375, 9999.03515625, 9998.4560546875, 9998.1826171875, 9998.359375, 9999.2685546875, 10001.4775390625, 9999.669921875, 9999.8212890625, 9999.9580078125, 9997.552734375, 9999.369140625, 10000.2373046875, 9999.908203125, 10001.12890625, 9998.2900390625, 9998.4130859375, 9999.986328125, 9999.78125, 10000.458984375, 9999.255859375, 9998.3984375, 10000.7548828125, 9999.5419921875, 9999.5771484375, 10000.009765625, 9999.4658203125, 9998.5146484375, 9998.98046875, 10000.7939453125, 10000.7421875, 9999.568359375, 10000.7890625, 9999.341796875, 9999.509765625, 9999.3935546875, 9999.7939453125, 10001.392578125, 10000.3779296875, 10000.08984375, 10001.30859375, 9999.7841796875, 10000.33203125, 9999.7412109375, 10000.2216796875, 10000.162109375, 10000.123046875, 9998.4169921875, 10000.4267578125, 9999.232421875, 9999.1953125, 10000.7138671875, 9999.7373046875, 9999.748046875, 10000.28515625, 9999.9521484375, 10001.19921875, 10000.36328125, 10000.0478515625, 9997.45703125, 10000.072265625, 10000.9208984375, 10000.7236328125]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [10000.005859375, 9999.9873046875, 10000.0205078125], "dtype": "float32"}, {"__nd__": [0.9739676117897034, 0.9971923232078552, 0.9913285374641418], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.8950013518333435, -0.8831895589828491, 1.7445942163467407, -1.1662755012512207, 0.5720545649528503, 0.029575521126389503, 0.8338935375213623, 1.4243801832199097, -0.16400258243083954, 0.25069403648376465, 0.8180318474769592, -0.49829602241516113, -0.81483393907547, 0.5236600041389465, 1.3403639793395996, -1.6917515993118286, -0.23993471264839172, -0.28277286887168884, 0.06206958368420601, 0.4728505611419678, -0.4491310119628906, 0.7395175099372864, -1.7875984907150269, -0.9653618931770325, -1.8330367803573608, -0.41662004590034485, -0.008617806248366833, 0.3414776623249054, -0.4669875502586365, 0.08959410339593887, -0.5189201235771179, -0.2845749855041504, -1.8310950994491577, -1.4179105758666992, -0.23487108945846558, -1.470278024673462, 0.62632817029953, 0.8880252242088318, -0.4858105182647705, 0.8380498886108398, 0.6252244114875793, 1.2855889797210693, -0.8422362208366394, -0.28006067872047424, 0.24877505004405975, 0.22968128323554993, 0.3878573775291443, -1.5885837078094482, -0.2467086762189865, -0.07992654293775558, -1.9306007623672485, 0.13003592193126678, -0.9285198450088501, 1.9053353071212769, 0.15097585320472717, 0.8231984972953796, 0.8891940116882324, -1.0549700260162354, -1.4177132844924927, -1.1656490564346313, -1.5199538469314575, 1.0661520957946777, -0.7335365414619446, 1.541983723640442, 2.4973552227020264, 0.43389731645584106, -0.5977352261543274, -0.3874760866165161, 0.5112825036048889, -0.9155266284942627, 1.4863166809082031, -0.14158476889133453, 1.206803798675537, -0.03500773757696152, -0.9427066445350647, -0.1518602818250656, 0.637001097202301, 0.3634718954563141, -0.4247623383998871, -0.47626832127571106, -0.11497772485017776, 1.6245951652526855, -2.3833260536193848, -2.291696786880493, -0.07545075565576553, 0.42287251353263855, -0.06412781774997711, 2.0647635459899902, -0.09588126093149185, -0.043444424867630005, 0.8355596661567688, -1.1069844961166382, 0.9818705320358276, 0.8238506317138672, 0.34644827246665955, -0.6054546236991882, -0.34446147084236145, -0.9490306377410889, 0.41038796305656433, -0.003551456378772855, -2.257340431213379, -1.0767461061477661, -0.4773709774017334, -0.660788357257843, 0.8330498337745667, -2.4925646781921387, 0.7729089856147766, 0.7687857747077942, 2.068406343460083, -0.14662525057792664, 0.9019249677658081, 0.3380744755268097, -1.0524779558181763, 0.20652088522911072, -0.5637900233268738, -1.249507188796997, -1.5494078397750854, -0.1779838502407074, 1.232246994972229, 0.9284917712211609, -0.49868273735046387, -1.2415410280227661, -2.462961435317993, 1.869390606880188, 0.8870709538459778, 0.4277547001838684, -0.05172332003712654, 1.3441718816757202, 1.0812190771102905, -1.9520403146743774, 0.026200367137789726, 0.1373986303806305, -0.7857357859611511, 0.251972496509552, 0.3599013388156891, 1.444493055343628, 0.6830493211746216, 1.1071094274520874, 0.6452557444572449, -0.32502490282058716, -0.605536162853241, 2.4271328449249268, 1.042052149772644, -0.22591879963874817, -0.578068733215332, 1.2762744426727295, -1.0870169401168823, -0.9877637028694153, -1.5197242498397827, 0.39361926913261414, -1.1614444255828857, 0.28860315680503845, -0.6597527265548706, 0.7407019734382629, -1.1844685077667236, 0.6691887378692627, 0.7035619020462036, -0.45822370052337646, 0.6817063093185425, -1.9420113563537598, 2.6770386695861816, -0.28390493988990784, -0.17623873054981232, 0.5693044066429138, 0.7439444065093994, 0.4409419596195221, -1.3396306037902832, -0.5625504851341248, 1.2440581321716309, -0.7933403253555298, 3.34948468208313, -1.585366129875183, 0.8960181474685669, -0.047958455979824066, 0.5019239187240601, -0.5065333247184753, 0.6882523894309998, 0.5381847620010376, -1.650229811668396, 1.5845921039581299, -0.9581682085990906, 0.8820112347602844, 0.2179674208164215, 0.7956151962280273, -0.6303537487983704, 0.4736005365848541, 1.1060137748718262, -0.5284869074821472, 0.10938136279582977, -2.065913438796997, -0.3907833993434906, -0.508418619632721, -1.9012067317962646, -1.1521323919296265, 1.2005949020385742, 0.10690546035766602, -0.8128855228424072, 0.2949329614639282, -0.373566597700119, -0.2236194759607315, -1.2475519180297852, -1.6762953996658325, -0.20733028650283813, 0.28286516666412354, 0.5069754123687744, -0.4008741080760956, -0.007273941300809383, 0.22111032903194427, 0.1059773713350296, -0.6661162376403809, 0.8087701201438904, -1.1443092823028564, 1.5787791013717651, -0.7333540916442871, -1.3268288373947144, -1.5451467037200928, 0.9975569248199463, 1.2392358779907227, -0.3468668758869171, 0.43072226643562317, -0.23412533104419708, -0.012144280597567558, -0.2241651564836502, 1.361558198928833, -0.37933629751205444, -1.0780874490737915, 0.5189621448516846, 1.2233200073242188, -0.06241484731435776, -0.2902866303920746, 0.1463998407125473, -0.34266096353530884, -1.0039105415344238, 1.9183636903762817, 0.418993204832077, 0.028945574536919594, -0.5501952171325684, 1.1011946201324463, -1.7835056781768799, -0.6437520980834961, -0.5457308888435364, -0.4964459240436554, -0.6093086004257202, 0.4089907705783844, -0.2768712639808655, -1.549792766571045, 1.9469614028930664, -1.3959448337554932, -1.121201992034912, 0.8171266317367554, -0.0825611799955368, 0.8086242079734802, -0.15970464050769806, 0.070870541036129, -0.3512253761291504, -0.468110591173172, -0.11452741920948029, 1.1236571073532104, -1.6369010210037231, 0.14407780766487122, -1.0494047403335571, -0.4246441423892975, -0.012093144468963146, 1.2740941047668457, 2.0701186656951904, 0.8862760663032532, 1.6601834297180176, 1.477607011795044, 2.2613298892974854, 2.0312790870666504, -2.055250644683838, 0.463003933429718, 0.8776757717132568, 0.7602050304412842, -0.9510877728462219, -0.5112394094467163, 0.7171338200569153, 1.4843626022338867, -0.255093514919281, 0.854767382144928, 0.794657289981842, 0.034116823226213455, 0.28259000182151794, 0.809441328048706, 1.1699076890945435, 0.06053170934319496, 1.3369311094284058, -0.695853590965271, 1.2028169631958008, 0.1027379259467125, 0.32877209782600403, 0.12699079513549805, -0.4436926245689392, -1.565047264099121, -0.4965214729309082, 0.8369370102882385, 0.07635433971881866, 0.34851008653640747, 0.4228679835796356, -0.10987864434719086, 1.694240927696228, -0.037728097289800644, -0.7035906314849854, -2.230135679244995, -0.31145405769348145, -2.003734827041626, 2.7572197914123535, -1.227584958076477, -0.7531580328941345, 0.44273194670677185, 0.2624123990535736, 0.28854814171791077, 0.16470937430858612, 1.1033844947814941, 0.4857739508152008, 0.016948390752077103, 1.4821038246154785, -0.9719148874282837, -0.2914121150970459, -0.4328998625278473, -0.18080542981624603, 0.6441662907600403, -0.8673131465911865, 1.4225432872772217, -0.5956588387489319, 0.7861423492431641, 0.2299676239490509, 0.4832119941711426, -0.6057695746421814, 0.4039037823677063, -0.7989916801452637, 0.8275191187858582, 0.043542731553316116, 0.7239013314247131, -0.8565098643302917, -0.4007696807384491, 0.5861397981643677, -0.5462011098861694, -0.6408566236495972, -0.9915863871574402, 0.8954097032546997, 0.19775919616222382, 0.5525076389312744, 1.3301955461502075, 0.7027153968811035, 0.9700899124145508, -0.11865533888339996, -1.3372445106506348, 1.402254581451416, 1.4558757543563843, -0.6951120495796204, -1.468654751777649, 0.7231912612915039, -0.23723264038562775, -0.9894813895225525, 0.19846110045909882, -0.7328455448150635, -1.7293603420257568, -0.1219731792807579, -0.3737832307815552, 0.09253650158643723, -0.022894658148288727, 0.9419956207275391, -1.6563503742218018, 1.243592381477356, 1.6532905101776123, -1.1416596174240112, 0.8128545880317688, -0.4132165014743805, 0.11485018581151962, -1.1036162376403809, 0.3950808048248291, -0.972692608833313, -0.705197811126709, -1.1205309629440308, 0.11288657784461975, -0.8951496481895447, 2.036773204803467, -0.03938702866435051, -0.3387368321418762, -0.10578883439302444, -1.8136895895004272, -2.1211585998535156, 0.7974295616149902, -0.12390947341918945, -1.5615931749343872, 1.6341164112091064, -1.2389341592788696, -0.3490121364593506, -1.1121447086334229, -1.0269347429275513, -0.9409680366516113, -1.9997129440307617, -1.872178077697754, -1.0227519273757935, -1.5805786848068237, 0.536605954170227, 0.2595619559288025, 0.9318351149559021, -1.1831188201904297, -1.1864635944366455, -0.9143136143684387, -0.1700105369091034, -0.04453587904572487, 0.8718691468238831, 0.415731817483902, -0.2901652753353119, -0.3573126792907715, 0.4605191946029663, -0.8572934865951538, 0.7153884172439575, 0.6887211799621582, 0.324098140001297, -0.10837500542402267, -0.6163409352302551, -0.002312600612640381, 0.03146451339125633, 1.3541908264160156, 0.6228088736534119, -1.1335457563400269, 0.8749101758003235, -0.9598215818405151, 0.3386290967464447, 0.02209874801337719, -0.5247179269790649, 0.3371548652648926, -0.5249925851821899, -2.8725409507751465, -0.5624821782112122, -0.757944643497467, 0.31754615902900696, 0.9210584163665771, 0.06480372697114944, 0.31038472056388855, -1.7115994691848755, 2.4599978923797607, -0.5478244423866272, 1.0879541635513306, 3.123288154602051, -0.012620512396097183, 0.708721935749054, -1.8287099599838257, -0.9663940668106079, 0.21369202435016632, -0.003925146535038948, 0.3168119192123413, 2.500213384628296, 0.5822041034698486, -2.101027727127075, 0.08900035917758942, 0.6788265705108643, -0.08370355516672134, 0.7027950286865234, -1.1530139446258545, -1.5648374557495117, -0.5127443075180054, -1.7403987646102905, -0.08144840598106384, 0.49454718828201294, 0.38968783617019653, 1.6420964002609253, 1.49433171749115, 1.033939242362976, -0.06237398833036423, 0.2016303539276123, 2.0118205547332764, 0.46547603607177734, -0.49082666635513306, 0.1799398958683014, -0.6480451226234436, -0.7964743375778198, 0.8899105787277222, -1.2880871295928955, 0.02950819954276085, 0.057255975902080536, 0.3059667646884918, 0.4839085042476654, 0.35029712319374084, -1.0130784511566162, -0.9353435635566711, -0.3858809173107147, -0.441951721906662, 0.40560224652290344, 1.9316256046295166, -0.4053904414176941, 1.1642251014709473, 0.13092470169067383, -0.11199864745140076, -0.45003971457481384, -0.3411276340484619, 0.6391772031784058, 0.4918057918548584, -0.8520680665969849, -0.13384756445884705, 0.10648486018180847, -1.6162681579589844, 0.09496691077947617, 1.1450990438461304, 0.7056365013122559, -0.4915120303630829, -1.5491527318954468, -0.836022138595581, 1.8497421741485596, -0.2029591202735901, 0.051988061517477036, 1.0207929611206055, -0.8661860227584839, -0.7851723432540894, 0.8608755469322205, 1.7422823905944824, -1.2030963897705078, 0.45628228783607483, 0.16798709332942963, 0.4004274308681488, -0.12891985476016998, 1.2108429670333862, -0.04573734477162361, -0.43742427229881287, 0.042807020246982574, 0.21718163788318634, -1.349271535873413, 0.08374293148517609, -2.4890639781951904, -0.11746272444725037, 1.5925285816192627, 0.42934465408325195, -0.6139755845069885, 0.9736841917037964, -1.3340387344360352, 2.5849907398223877, 0.08197139203548431, 1.1323214769363403, 0.18495170772075653, -0.5483283996582031, -0.9624066948890686, 1.3939028978347778, -0.4445781707763672, 1.334558367729187, -1.4217020273208618, -1.7204005718231201, -0.6143754124641418, -0.3660949468612671, -1.237908959388733, 2.9249448776245117, 0.22425523400306702, 2.000380754470825, -0.2879750430583954, -0.5932884812355042, 0.17320306599140167, -0.516066312789917, 0.043699633330106735, -0.8342412710189819, 2.0158984661102295, 1.381935715675354, 2.130812883377075, 0.99677574634552, 0.7847715020179749, -0.5496519207954407, -0.5846631526947021, 0.880567193031311, -1.43218195438385, -1.218427300453186, 0.2693770229816437, 0.7619297504425049, -1.1139373779296875, 1.0224833488464355, 0.2925744652748108, -0.42329102754592896, -0.7916774749755859, -1.7819850444793701, 1.1566815376281738, -0.7905927896499634, -1.1524659395217896, -1.2382363080978394, -1.3303014039993286, 0.39981016516685486, -0.17969800531864166, -1.755163550376892, 0.8384950160980225, -0.8921731114387512, -1.1731764078140259, 0.19409684836864471, -0.0961650088429451, -1.2051411867141724, -0.11371834576129913, 0.25070565938949585, -0.5236033797264099, 1.399432897567749, -0.5534642338752747, 1.2750601768493652, 0.4631049335002899, -0.8684375882148743, -0.026464711874723434, 0.8253404498100281, 2.436981678009033, 0.23403914272785187, 2.1342458724975586, -0.5015634894371033, 0.2253989428281784, 1.5423988103866577, 0.9240959882736206, 0.8610872030258179, -0.09369752556085587, -0.5907250642776489, 0.8184011578559875, -0.9885557293891907, 1.3500126600265503, 0.5467410683631897, -0.2096341997385025, 1.4629894495010376, -0.5534582734107971, -0.5692377686500549, 1.420977234840393, -2.121514081954956, 0.4765574038028717, 1.2995319366455078, -0.2320169359445572, -1.237727165222168, 0.7831284403800964, -1.496310830116272, -0.6496111154556274, -0.038568466901779175, 0.00982662569731474, -1.4580813646316528, 0.12345083802938461, 0.5026980638504028, 0.697462260723114, 1.1567094326019287, -0.5252916812896729, 1.3860822916030884, -1.9255787134170532, 1.7954213619232178, -0.22933237254619598, 0.7175303101539612, -1.761254072189331, -1.9485071897506714, -0.13061848282814026, -0.7853994965553284, -0.26472771167755127, 0.5292261838912964, 1.6384522914886475, -1.950248122215271, 0.09294168651103973, -1.5085989236831665, -0.22612600028514862, 1.53293776512146, -1.3412163257598877, -2.1546900272369385, -0.15746036171913147, -1.3291141986846924, 0.8556405901908875, -1.3710764646530151, -0.41799578070640564, -0.4370530843734741, -0.9065364599227905, -0.35823842883110046, -1.6790943145751953, 0.7160117626190186, -0.10251187533140182, 0.2803681790828705, -0.7185907959938049, -0.19669662415981293, 0.9446325898170471, 1.004051923751831, -0.31501904129981995, 0.27057188749313354, 1.1503092050552368, -0.6856324672698975, 0.042643241584300995, -1.7751215696334839, -1.0885988473892212, 1.654744029045105, -1.040655493736267, 0.12939505279064178, -1.7053842544555664, 0.07816386222839355, 1.7076481580734253, 1.916119933128357, -0.218075230717659, 0.6419610381126404, 0.9672468304634094, -0.8913173675537109, 1.0969947576522827, -0.44961121678352356, 1.837667465209961, 0.2969013750553131, 0.35074207186698914, -1.4929730892181396, -1.0716991424560547, 0.8576482534408569, -0.4255589544773102, -0.6371153593063354, 0.6564825773239136, 0.12190257012844086, 1.2474483251571655, 0.21372556686401367, 0.6311593055725098, -2.8770408630371094, -1.07847261428833, 0.7722761631011963, 0.11586898565292358, 1.6290931701660156, -1.263971209526062, -0.8410223722457886, 0.16794176399707794, -1.5448132753372192, 1.360730528831482, -0.2944161295890808, -0.9392228126525879, -0.4773070514202118, 1.7135236263275146, 0.8844276070594788, -0.4443761706352234, -0.9078236222267151, 0.5049669146537781, -0.025092678144574165, 1.131484031677246, -0.7216254472732544, 0.9446688294410706, 0.7223191261291504, 0.36916086077690125, 0.784935712814331, 0.43931683897972107, -1.4990788698196411, 0.05793412774801254, -1.5385923385620117, 0.5228161811828613, -0.08883576840162277, -1.5519893169403076, -0.2081276774406433, -0.5955103039741516, 1.8699538707733154, -0.5212528705596924, 0.621661901473999, 0.9261783957481384, -0.9669936895370483, -0.6490753293037415, -0.036061495542526245, -1.144036054611206, -0.4249703884124756, 0.6353301405906677, 1.4881211519241333, -0.9675940871238708, 0.3475518822669983, 2.166578769683838, 1.6804977655410767, -1.6774636507034302, -0.34811195731163025, -2.6133291721343994, -1.7035106420516968, 0.38020047545433044, -0.269616961479187, 0.31635621190071106, -0.7683696150779724, -1.1434061527252197, -0.20100949704647064, 0.505520761013031, -0.1383453607559204, 0.23157185316085815, 1.1481924057006836, 1.7209253311157227, 0.7910286784172058, -0.23541022837162018, -0.5823266506195068, -1.4575475454330444, 0.8738281726837158, 0.2838411033153534, 1.0047166347503662, 1.015870451927185, 1.274242639541626, 0.015786845237016678, 0.8550227880477905, -1.7121598720550537, -0.5740044116973877, 2.191535234451294, -0.34044352173805237, 1.799560308456421, 0.8809595704078674, -0.10933016985654831, 1.0404984951019287, 1.4207440614700317, 0.672020435333252, 2.3151140213012695, 0.9250651597976685, 0.13352425396442413, -1.2314772605895996, -1.1409525871276855, -0.747085690498352, -1.2276370525360107, -1.2213225364685059, -0.22416968643665314, -0.17774122953414917, 0.596421480178833, 0.1669544130563736, -0.01348047610372305, -0.7985946536064148, 2.5016133785247803, 0.25348371267318726, -0.7008751034736633, -0.145237535238266, -0.16003988683223724, 0.5484449863433838, -0.12651802599430084, 0.15692053735256195, 0.270080029964447, -0.4765935242176056, 0.6331302523612976, 0.027456384152173996, -0.17254094779491425, -1.108730435371399, 0.22610138356685638, 0.036388099193573, 0.19712482392787933, -1.1724354028701782, -1.0513498783111572, 2.969503164291382, -0.06654670834541321, -0.2771749496459961, -0.167026549577713, -1.0608839988708496, -1.5557692050933838, -0.9172630310058594, -0.5300375819206238, 0.9604531526565552, -1.9294462203979492, 0.891685426235199, -1.0026278495788574, -0.6102368235588074, 1.114583134651184, 0.3389807343482971, -0.33711135387420654, 1.0937155485153198, -0.007259409409016371, 0.23562224209308624, -0.17249290645122528, 1.1432679891586304, 0.8953399062156677, -0.1893470734357834, -0.5709390044212341, -1.0024129152297974, -0.7631111741065979, 0.024015601724386215, -0.582737386226654, 1.0909178256988525, 0.07629059255123138, -1.5473618507385254, 1.105377197265625, -0.8915997743606567, -0.1404459923505783, 0.04447094351053238, 0.7713305354118347, 0.63113933801651, 0.36209797859191895, -1.1724045276641846, -1.4008904695510864, 1.261029839515686, 2.0631120204925537, 0.48817959427833557, 1.8183566331863403, 1.324262022972107, -1.2752947807312012, -1.9348702430725098, 0.9054692387580872, -0.11433470994234085, 0.15919838845729828, 0.1699225753545761, -1.5431636571884155, 1.1684178113937378, -1.1076403856277466, 1.124763011932373, -0.5579310059547424, -0.9360961318016052, 2.117482900619507, 0.09588450193405151, 0.3192170560359955, 1.034732460975647, -0.4760676324367523, -0.16970612108707428, 2.744478702545166, 0.5277820229530334, 0.8256539702415466, -0.21499882638454437, 0.7226828932762146, 2.1505401134490967, 0.47353050112724304, -0.44912847876548767, 0.7985402345657349, -0.2722775936126709, -0.16849619150161743, -0.11913187056779861, 0.22312447428703308, -0.3319219648838043, -0.5404409170150757, -1.0242204666137695, 0.6488791704177856, 0.5236029624938965, -1.2454314231872559, 0.41177794337272644, 0.9641225337982178, 0.2803255319595337, -1.691361904144287, -0.9339216947555542, -0.1087876483798027, 0.11142510175704956, -2.1626787185668945, 0.969573974609375, 0.9403798580169678, -1.1935487985610962, 0.4010457396507263, -0.10439891368150711, -1.5075980424880981, -1.6592578887939453, -0.44498923420906067, -0.7504867911338806, 0.030369801446795464, -1.499564290046692, 0.6115707755088806, 0.5605565905570984, -1.2032519578933716, -1.3594214916229248, -0.5060979723930359, 0.7174341678619385, 0.8410705924034119, 0.6154645681381226, 0.47952592372894287, 0.4125330150127411, -0.7006476521492004, -0.46205922961235046, -0.36668315529823303, 1.5793702602386475, 0.7227675318717957, -0.5618317723274231, -0.5333642959594727, 0.40957316756248474, 0.9551876783370972, -0.16849836707115173, 0.14704930782318115, -3.3199667930603027, 1.1438755989074707, -0.9672090411186218, -0.17736686766147614, 0.8032914996147156, -0.2771967053413391, 0.48584821820259094, 1.0465452671051025, -0.810379147529602, -0.8321104645729065, 0.9330134391784668, 0.0920538529753685, -0.5690355896949768, -1.8529003858566284, -0.12118753790855408, -0.39518964290618896, 0.2965143918991089, 0.21506787836551666, -0.20469540357589722, -0.2955860197544098, -1.6961092948913574, 0.001010124571621418, 0.2568225860595703, -1.7462023496627808, -0.557856559753418, 0.11808817088603973, 0.2570232152938843, -1.783612847328186, 0.9456532001495361, 0.955885648727417, 0.46134790778160095, -0.5815982818603516, 0.5034233927726746, 0.2568921148777008, 0.6319263577461243, -0.7350848913192749, -0.09135071933269501, -0.15485185384750366, 0.6924819946289062, 0.05338545888662338, -0.16972127556800842, -0.6761114597320557, -1.1561830043792725, -1.0597056150436401, 0.2703324854373932, 0.01653597131371498, -1.091469168663025, 0.26063522696495056, 0.8231850862503052, -0.8598800897598267, 0.19414952397346497, 0.868571400642395, -1.3539165258407593, -0.36514192819595337, -0.0976652130484581, 0.10345351696014404, 1.443048357963562, -0.4771571159362793, -0.172610342502594, 1.1477750539779663, 0.03148354962468147, 0.22345000505447388, 1.6965854167938232, 0.07057087868452072, -0.5571238398551941, -3.284130573272705, -0.617764949798584, -0.6445690393447876, 0.8663879036903381, 0.6903103590011597, 0.86639803647995, -1.6278806924819946, -0.13979534804821014, -1.9302276372909546, -1.1272639036178589, -1.1212027072906494, -2.711414337158203, 0.9329141974449158, 1.2290966510772705, 1.7561964988708496, -0.219767227768898, 0.22891956567764282, 0.2342505306005478, 1.266066551208496, 0.7347167730331421, -0.7565797567367554, 1.8821426630020142, 0.5244070291519165], [0.6021079421043396, 0.5885196924209595, -0.07238912582397461, 1.3019531965255737, 2.5781328678131104, 1.4284342527389526, -1.690289855003357, -1.1560695171356201, -0.1370801478624344, 0.07582706958055496, 1.1266871690750122, -0.7003784775733948, 0.6053408980369568, -0.9377398490905762, -0.3908444941043854, -1.336959719657898, 0.44286829233169556, 0.35104551911354065, 0.10759127140045166, 0.11985495686531067, 1.3356555700302124, -0.996505081653595, 0.5037549734115601, -0.14937742054462433, -1.0679439306259155, 0.9810582995414734, -1.0087486505508423, 1.0653325319290161, -0.19360579550266266, 0.4092160165309906, 0.7312635183334351, -1.2497433423995972, 1.0338003635406494, 1.5200583934783936, -0.603547215461731, -0.23427891731262207, -0.6256018877029419, -0.7426169514656067, -0.22431683540344238, -0.8942211270332336, -0.2770121097564697, 0.4357392191886902, -0.25103187561035156, 0.32260656356811523, -0.15345728397369385, -1.0193084478378296, 1.297469139099121, 0.734009325504303, 1.1970934867858887, 1.283193588256836, -2.95810604095459, 0.40741652250289917, -1.343899130821228, 0.3138933777809143, -0.18883168697357178, -0.9648263454437256, 0.7696608304977417, -0.7906695008277893, 0.8502480387687683, 0.08897387981414795, -0.4663907289505005, -0.14330655336380005, 0.1068209633231163, -0.895758867263794, -0.5807109475135803, 0.7610751390457153, 2.7008047103881836, 0.44845181703567505, -0.46421197056770325, 2.157857656478882, 0.6978796720504761, -0.5134952664375305, -1.2873574495315552, 0.5283656716346741, 0.4833435118198395, 1.2313114404678345, -0.45880255103111267, 2.061458110809326, -0.550930917263031, -0.16536813974380493, 0.6360880732536316, 0.4782254695892334, 0.271646112203598, -0.8809008598327637, 0.4971139430999756, -0.34912997484207153, 1.0306897163391113, -0.7436699271202087, -1.392067790031433, -1.3925455808639526, 0.1996559202671051, -1.3514257669448853, 0.343591570854187, -0.8427078127861023, 0.5115907192230225, -1.531963586807251, 0.8291752934455872, -1.6866415739059448, 1.4093949794769287, 1.420961856842041, 1.0750302076339722, -0.3659505844116211, -1.1844754219055176, 1.2756941318511963, -2.032170295715332, -0.9821498394012451, -0.1108446791768074, 0.8469268679618835, -1.4984408617019653, -0.8734241724014282, 0.5203467607498169, -0.2869056165218353, 2.070876359939575, 1.3810964822769165, -0.6924388408660889, 1.0793757438659668, 0.9798412919044495, 1.2427244186401367, -2.267686605453491, 0.30522480607032776, 0.369373083114624, 0.23077070713043213, -1.5535846948623657, 2.0169801712036133, -1.0463484525680542, 1.2791205644607544, -0.05219540745019913, -0.9797357320785522, -0.36266571283340454, 1.1446120738983154, -0.2176201045513153, 0.13050997257232666, -1.9607512950897217, -0.8138720989227295, -0.7748212218284607, -0.2765345871448517, -1.8806545734405518, -0.24105626344680786, -0.38765406608581543, 0.975246787071228, 1.7186064720153809, -0.8788488507270813, -1.118912935256958, 2.2881383895874023, -1.4359898567199707, 1.057881236076355, 0.028450392186641693, 0.2221808135509491, 0.6754259467124939, -0.8891115784645081, 0.6225748062133789, -2.0809922218322754, 0.07504992187023163, -0.016715480014681816, 0.9712024331092834, 0.39171257615089417, -0.3633843958377838, 0.5541942119598389, -0.9274469614028931, -0.8452681303024292, -2.041386365890503, 0.08556670695543289, -1.5205191373825073, 1.175490140914917, -1.1783277988433838, 1.47934889793396, 0.4346306622028351, 1.365519642829895, 0.4226182699203491, -1.0223838090896606, -0.2893536686897278, -0.19972757995128632, -0.049086879938840866, -0.7987754940986633, 1.7396363019943237, 0.8197157979011536, 0.3236715793609619, -0.15768036246299744, 0.3911046087741852, 0.16066500544548035, 0.24983510375022888, 0.6552920937538147, -0.23677358031272888, 0.1474652886390686, 0.667431652545929, -1.3893804550170898, -0.2642103433609009, 0.22237733006477356, 0.8674399852752686, 0.5051777958869934, -0.3195377588272095, 0.9340583086013794, -0.01332103181630373, -0.4317004680633545, 0.5079262256622314, 0.5922406911849976, 0.22577610611915588, -0.08193639665842056, 0.4404621124267578, -0.33984750509262085, 0.30961889028549194, -1.7768206596374512, -1.0860368013381958, 0.7866084575653076, -0.8204250335693359, 0.9164530038833618, -1.5778892040252686, -1.1143304109573364, -0.056076779961586, 1.4077715873718262, 0.0006839202833361924, 0.20554889738559723, -0.12069296091794968, -1.2585355043411255, 1.335744857788086, 1.542472243309021, -0.2806755006313324, -0.6152344346046448, -0.5460814237594604, -0.8605954647064209, 0.21848677098751068, -0.29779982566833496, -0.4714371860027313, -1.4199544191360474, 1.0557441711425781, 0.30017051100730896, -0.37928682565689087, -0.049120187759399414, 0.3368426263332367, -1.9210797548294067, 0.07457681745290756, 0.6678446531295776, 0.41975733637809753, -1.3409500122070312, -0.3993920385837555, 1.1991537809371948, 2.6227879524230957, -0.22499120235443115, -1.1870473623275757, 1.5411194562911987, -0.18219804763793945, 1.6284340620040894, 0.9463743567466736, 1.6478208303451538, 0.3569751977920532, 1.5922813415527344, 0.39361515641212463, 0.18780353665351868, 0.8634808659553528, 0.18894673883914948, 2.0069656372070312, -0.7200452089309692, 0.4833218455314636, 1.0730034112930298, 1.5903176069259644, -0.9700048565864563, 0.2375566065311432, -0.19595402479171753, 0.24748224020004272, 1.2557367086410522, 0.2636308968067169, -0.8909515142440796, 0.009401730261743069, 0.30838507413864136, -0.18173658847808838, 1.2873013019561768, -0.8970012068748474, -0.5827099680900574, -0.7359315752983093, -2.8282487392425537, -0.5670073628425598, -1.0932159423828125, -1.7010327577590942, -0.33685949444770813, 0.09181239455938339, 0.7552882432937622, -1.068824291229248, 0.4854559898376465, -0.6857307553291321, 0.5066011548042297, 0.3265226483345032, 0.3051237165927887, -0.08716654777526855, -0.8389526605606079, -0.558948278427124, 0.35956886410713196, -0.7416387796401978, 0.1286395639181137, 0.909882664680481, 0.5893151760101318, -1.7348706722259521, -0.3911255896091461, -0.017767522484064102, 1.5375336408615112, 0.4236159026622772, -0.6020240187644958, -0.18389363586902618, 2.188035726547241, 2.0848512649536133, 0.18890638649463654, 0.7048249244689941, -0.125161811709404, 0.6819535493850708, -0.24878446757793427, -0.043552201241254807, -0.48873379826545715, -0.6116884350776672, 1.1091375350952148, 0.16279256343841553, -1.8564081192016602, 2.2266507148742676, 0.2522575259208679, -0.11479458212852478, 0.6584498882293701, 0.11516716331243515, 0.5910482406616211, -0.3283385634422302, 0.7071605920791626, -1.2399934530258179, 0.03209756687283516, -0.522826611995697, 2.4253287315368652, -0.17449530959129333, -0.524177074432373, -0.0037202618550509214, 0.3623308837413788, 0.29347580671310425, 0.25203078985214233, 1.282090425491333, -0.6775913834571838, -0.2835088074207306, -1.0761452913284302, -0.5516759753227234, 1.4601349830627441, -0.3329883813858032, 0.3821825087070465, 0.907086968421936, -0.10006115585565567, -0.12269926071166992, 1.9867242574691772, 1.9527345895767212, -0.521126925945282, -0.6207583546638489, 0.5065426826477051, 0.5649610757827759, 1.1325798034667969, 0.82607102394104, -0.9594012498855591, -2.135647773742676, 1.371023178100586, 1.3157892227172852, -0.2178771197795868, -1.268462896347046, -0.582313597202301, 0.8248076438903809, 0.36494532227516174, -0.08151514083147049, -0.9925877451896667, -1.751550555229187, -1.970141053199768, -0.9823973774909973, 0.255631685256958, -0.017857763916254044, -0.642753005027771, -0.5449499487876892, -1.4233791828155518, -0.13246014714241028, -0.3526601791381836, -0.14595675468444824, 0.637373685836792, -0.20550736784934998, 1.0696369409561157, -1.2183098793029785, -1.3313628435134888, -1.1604409217834473, 1.648088812828064, -0.897289514541626, 0.014201474376022816, -0.7739289402961731, 2.5871732234954834, -0.04113025963306427, -1.6333179473876953, 1.609748363494873, 2.4152870178222656, -0.062106989324092865, 1.2254043817520142, 0.8868473768234253, -0.1782945692539215, -1.214954137802124, -0.2967790961265564, 1.4270868301391602, 1.6675142049789429, -0.1722036898136139, -0.3273811340332031, 0.17757610976696014, 0.343755304813385, 0.14021898806095123, -0.9544591903686523, 0.12683409452438354, 0.7842720746994019, 0.1527518481016159, -1.91728937625885, -0.4401366412639618, 0.06392419338226318, 0.14308033883571625, -1.7621955871582031, 0.22170013189315796, 0.2505435347557068, -1.905532717704773, -1.0908335447311401, 1.0634034872055054, 1.5400362014770508, 0.15583816170692444, 1.2432119846343994, -0.9594016075134277, 0.23616042733192444, -0.9667278528213501, 0.6365585923194885, 1.0371779203414917, 0.055042341351509094, -0.39104923605918884, -1.426600456237793, -0.8931892514228821, -0.40848469734191895, -0.87856525182724, 1.0366802215576172, 0.7560201287269592, 0.6665487289428711, -1.7957528829574585, -0.4028428792953491, -0.5994889736175537, 0.14353330433368683, 0.44624629616737366, 1.1212447881698608, 0.47827285528182983, 0.6915674805641174, -0.6805168390274048, 2.673877477645874, 0.8212005496025085, 1.0991724729537964, 0.37687763571739197, 0.2978080213069916, -0.06387712061405182, 0.20927536487579346, -0.017060445621609688, 0.07904475182294846, -2.1441659927368164, -0.3923085629940033, 1.079290747642517, 0.9580378532409668, 0.3152337074279785, -1.4773552417755127, -0.08616048842668533, 0.9542912244796753, -0.5732093453407288, 0.8730575442314148, -0.3039868474006653, 1.562077522277832, -1.1358778476715088, 0.8911901116371155, 1.3392442464828491, -0.7079593539237976, 1.1926181316375732, 0.7986077070236206, 0.5717358589172363, -0.3544589877128601, -0.7411819100379944, 0.45470690727233887, -0.2312549650669098, 0.6741580367088318, -1.2138805389404297, -1.1607742309570312, 1.9315903186798096, -0.12412355840206146, -0.8905079960823059, 0.8578740358352661, 0.1523812711238861, -1.4127553701400757, -0.3864161968231201, 0.17947369813919067, -0.9815434813499451, 0.8895529508590698, -0.6799948811531067, 0.4488275945186615, -0.8101443648338318, 0.6414819359779358, 1.153466820716858, 1.477107286453247, -1.3928569555282593, 1.0975642204284668, -1.4376723766326904, -0.2657499313354492, 0.35641470551490784, -0.4382385015487671, 1.5490007400512695, 0.863014280796051, -0.28005191683769226, 0.3417558968067169, -0.9550826549530029, -0.03917044401168823, 0.10963650792837143, 0.3823889195919037, 2.908493757247925, -1.6174958944320679, 0.8807430267333984, -1.4328950643539429, 0.6088306307792664, -0.8663985729217529, 1.364557147026062, -0.30557748675346375, 0.6873283386230469, -0.6192686557769775, 0.8489501476287842, -0.3516657054424286, -1.4085522890090942, -0.14543041586875916, -0.21781420707702637, 0.8028358817100525, 0.5999942421913147, -0.2647882103919983, 2.0099036693573, -1.4409103393554688, 1.0591892004013062, -0.4703032672405243, -0.5101585388183594, -0.6524564027786255, 0.021150648593902588, 0.3551156520843506, -1.787531852722168, -0.5804572701454163, 0.35193362832069397, -1.383185625076294, -0.6591269969940186, -0.9298238158226013, 0.577977180480957, -0.5740617513656616, 1.1231426000595093, -2.1267294883728027, -0.8394966125488281, 0.11609549820423126, 0.8030864000320435, 0.5569379925727844, 1.008602499961853, 0.13937656581401825, 1.2707531452178955, -0.278095006942749, 0.15382437407970428, 1.1453933715820312, 0.10946774482727051, -1.0266172885894775, 0.7241372466087341, -1.042515754699707, 1.6783528327941895, 0.17134422063827515, -0.9128832221031189, 0.3180995583534241, 0.06946372985839844, 0.32208356261253357, -1.0147581100463867, 0.5976097583770752, 0.005037585739046335, 1.175894856452942, -0.33922889828681946, -2.2810416221618652, -0.23496399819850922, -0.7644041180610657, 1.95730459690094, 0.8233094811439514, -1.9282399415969849, 0.8798143267631531, -1.653302788734436, 0.25674939155578613, 2.084519863128662, -1.050472617149353, 1.5459215641021729, 0.7315697073936462, 0.058139998465776443, 1.4222476482391357, 1.0760529041290283, -0.9278987646102905, 0.7904259562492371, 0.2967337965965271, -0.13959191739559174, 0.5017136931419373, -0.16809827089309692, 1.4657806158065796, -0.54145348072052, 0.8962085247039795, -1.5844966173171997, -0.6426953673362732, -2.121450662612915, -1.2712737321853638, 1.9396817684173584, 1.1793030500411987, 0.7618087530136108, 0.7203848958015442, 0.09332156926393509, 0.36544328927993774, -0.12314080446958542, 0.7354195713996887, 0.7855801582336426, -1.2494544982910156, 0.8168307542800903, 1.7376339435577393, 1.1927812099456787, 0.3196103870868683, 0.1198410764336586, -1.101603627204895, -0.7389147877693176, -1.1559009552001953, -0.30751246213912964, -0.6627797484397888, 2.001478910446167, -0.41614511609077454, -0.7788481712341309, -0.410419225692749, 0.31716758012771606, -0.2057807445526123, 0.9773401021957397, 0.018919559195637703, -0.22975170612335205, -0.6976010799407959, -0.39237749576568604, -1.006848692893982, 0.7948417067527771, 0.3435923159122467, 1.0286359786987305, -0.8808585405349731, 2.1871447563171387, -0.7615646123886108, 0.8475692272186279, -0.7998666167259216, 1.664997935295105, -1.3664921522140503, -1.5975693464279175, 0.26489007472991943, -0.4562067985534668, 0.22041115164756775, 0.5138402581214905, -0.9659405946731567, 1.084173321723938, -1.4448273181915283, 0.1319238245487213, 0.566724419593811, -1.4800128936767578, -1.0879591703414917, -0.1661367118358612, 0.7089515328407288, -0.6443300843238831, 0.24121038615703583, -1.0204646587371826, 1.1233161687850952, -0.1792452037334442, -0.4502648115158081, -0.8078210353851318, 0.9916824102401733, 0.04417014867067337, 0.8827691078186035, 2.0022192001342773, 0.7791928052902222, -0.5186078548431396, 2.280545949935913, 0.6176184415817261, 0.3984602391719818, 0.8408750891685486, -0.3055708706378937, 0.7382916212081909, -0.25009945034980774, -1.6608998775482178, -1.214572787284851, 1.0696642398834229, 0.4279786944389343, -0.16582010686397552, -0.839891791343689, -0.3810271620750427, 1.0239081382751465, -1.5542927980422974, -0.25818321108818054, -0.5497031807899475, 0.3174392282962799, -0.0187175665050745, 1.419127345085144, -1.4373944997787476, 2.095421075820923, -0.17173345386981964, -3.186838150024414, 1.1343926191329956, 0.02420441061258316, 0.5539826154708862, -0.6646414995193481, -0.6185204386711121, 0.027728252112865448, 0.29518648982048035, 1.3960928916931152, 1.9836721420288086, 0.39250731468200684, -0.0012257456546649337, 1.3417836427688599, -0.46189945936203003, 0.9572567939758301, -0.8810678124427795, -2.348257303237915, -1.1679697036743164, -0.8891187906265259, 0.9680615663528442, 0.19923169910907745, -1.6425424814224243, 0.7647415995597839, 0.33579424023628235, 0.53616863489151, -0.5401718020439148, 0.7849437594413757, 0.9144293069839478, -1.246156930923462, 1.0149592161178589, 0.5556056499481201, 0.3658220171928406, -0.006302541587501764, 1.4157358407974243, -0.6900818943977356, 0.5861892700195312, 0.7416383028030396, 1.5375784635543823, 0.9586761593818665, 0.07116270810365677, -1.8347374200820923, -0.37577518820762634, 0.8293971419334412, 0.38884347677230835, -1.1701022386550903, 0.48040786385536194, -0.6278336048126221, 0.2892039716243744, 1.126673936843872, -0.3119640350341797, 0.48127424716949463, -1.020655632019043, -0.19065719842910767, -0.009518997743725777, -0.8104256391525269, -1.5119811296463013, 0.7402134537696838, 0.5055892467498779, -1.4426188468933105, -1.391661524772644, 0.05587262287735939, -0.3162904679775238, 0.6165709495544434, -0.9453017711639404, -0.3849007189273834, 0.9279370307922363, 0.542424738407135, 0.7929266691207886, 0.5915922522544861, 0.09994380921125412, 0.0743657723069191, -0.5203091502189636, 1.5625509023666382, 0.2912442982196808, -0.10657782852649689, 1.969771146774292, 0.21739791333675385, 1.151069164276123, -2.231419324874878, -1.3592455387115479, -0.26680701971054077, 0.21573619544506073, -1.101524829864502, -0.33196961879730225, 0.4598804712295532, -0.7765439748764038, 0.8898857831954956, -0.9148352146148682, 0.11017921566963196, -0.6655448079109192, 0.6037348508834839, 0.6557273864746094, 0.06867647171020508, 0.819502055644989, -0.7289261221885681, -0.5107972025871277, -1.929166316986084, 0.28532645106315613, -0.36369380354881287, -1.3733205795288086, 0.9745016694068909, 0.06641247123479843, -1.482720136642456, 0.43708571791648865, -0.3406407833099365, -0.7910687327384949, 0.10701847076416016, -0.2983110845088959, -0.4151727557182312, -0.1181093081831932, 0.8108194470405579, -0.1576843112707138, -0.2376602739095688, 0.2653457820415497, 1.0704585313796997, -1.6874102354049683, -0.2804359197616577, 0.5212545394897461, 1.5598158836364746, -1.2466832399368286, -0.02254606783390045, 0.5862020254135132, 0.7862647175788879, -0.508847713470459, -0.5253257155418396, 0.35323378443717957, -0.23968477547168732, -0.9748564958572388, -0.06532406061887741, -2.199740409851074, 1.2903857231140137, 2.7031898498535156, 1.6871799230575562, 0.42671895027160645, -2.8060526847839355, -0.11433626711368561, -0.49681636691093445, 0.28205716609954834, -0.7466464042663574, 0.5611106157302856, -0.01355751883238554, 0.5662474632263184, -0.13244694471359253, 0.25395557284355164, -0.08008644729852676, -0.38534659147262573, 0.9691898822784424, -0.6067402362823486, 1.5913180112838745, -0.18639393150806427, 2.285165309906006, -0.9283767342567444, 0.47161027789115906, 0.612485408782959, 0.7097338438034058, -0.3849063217639923, 1.0230814218521118, -1.609910011291504, 0.2562333941459656, 0.21172727644443512, -1.5275189876556396, 0.04222918301820755, -0.9373408555984497, 0.12502101063728333, -0.5056241154670715, 0.4833756983280182, -1.4909621477127075, -0.6815075278282166, 0.498354434967041, 0.09614511579275131, -0.17212368547916412, -0.9108491539955139, 0.11056113243103027, 0.15243981778621674, -0.2104320526123047, 0.19674664735794067, 1.288187026977539, 0.1363101601600647, -0.3911823630332947, -0.5278224349021912, -1.101922869682312, 0.9560546875, -0.40411731600761414, 1.0474083423614502, 1.5780029296875, -0.8563026785850525, 0.475723534822464, 0.28515011072158813, 1.0388469696044922, -0.8814173936843872, -0.20918183028697968, -1.1726521253585815, -0.38771262764930725, -1.1271525621414185, 1.386827826499939, -0.5797030329704285, -1.0947846174240112, 0.46957680583000183, 0.2039431631565094, -0.9152319431304932, 0.7981361150741577, 0.5470491051673889, -0.8826503753662109, -2.0774266719818115, -0.3813599944114685, -0.4434199035167694, 0.12595568597316742, 1.3816916942596436, -2.002101421356201, 0.8260234594345093, -0.7984606027603149, -0.9196755290031433, 1.30377995967865, -0.29010847210884094, 0.476380318403244, 0.08989612758159637, 0.9873654842376709, -0.8640190362930298, -0.7769317030906677, 0.7623802423477173, -2.4208340644836426, -1.9814833402633667, 0.5997312068939209, -1.4240140914916992, 0.42261263728141785, 0.3533574640750885, 1.9120221138000488, 0.560841977596283, -1.3900750875473022, -0.2937406301498413, 0.2328605353832245, -0.756619930267334, -1.2277345657348633, 0.6384673118591309, -1.9840892553329468, -0.3564903140068054, -1.7633084058761597, 0.4516649544239044, -0.3678615391254425, 0.7594378590583801, 0.01876222714781761, -0.9157739281654358, 0.09829852730035782, -0.6982123851776123, -2.049973964691162, 2.333299398422241, -0.3988521695137024, 0.6366757154464722, 0.7656664848327637, -0.3667046129703522, -2.276608467102051, -1.0179109573364258, 0.48044466972351074, 0.47913989424705505, -0.1766500622034073, 0.5304875373840332, 1.7646684646606445, -0.4626924395561218, -0.4141309857368469, 0.3189128339290619, -1.1640068292617798, -0.03599085658788681, -0.8802823424339294, -0.47149038314819336, -0.43678018450737, 0.3416433036327362, -0.5607911944389343, 1.1769274473190308, 0.9061287045478821, 0.6626136898994446, 0.1571158617734909, -0.32857394218444824, 0.3997747302055359, 0.921918511390686, -0.33952826261520386, 1.1111247539520264, -0.2803296148777008, 1.408034086227417, 0.04802769050002098, 1.3667055368423462, -0.7069133520126343, -1.2803966999053955, -0.30127301812171936, 0.1462920606136322, -0.8737336993217468, 0.3962281346321106, 0.24824853241443634, -1.4970020055770874, -1.0300519466400146, -0.4017466604709625, 0.4571758508682251, -0.8982533812522888, 0.5851848125457764, 0.0972372442483902, 1.0612683296203613, 1.1749240159988403, -0.6516239047050476, 1.52065110206604, 0.9368642568588257, -2.3528428077697754, -0.45552244782447815, 0.40907400846481323, 1.4701963663101196, 0.15054497122764587, 0.32509395480155945, 0.6889846324920654, 0.012139925733208656, 1.8140257596969604, 0.7494502663612366, -0.09481759369373322, 0.3750101327896118, -0.9539543986320496, -1.4735534191131592, 1.1394976377487183, 1.8401108980178833, 0.5987173318862915, -0.16953709721565247, 0.9066435694694519, 1.5468244552612305, -0.5936877727508545, -0.8351201415061951, -0.8105896711349487, -0.5173641443252563, 0.740788996219635, -0.1814921349287033, -0.4112197458744049, 0.4082655608654022, 1.2668757438659668, 1.482812762260437, 1.734643816947937, -1.0991919040679932, -0.708267331123352, -0.3265441358089447, -0.23735937476158142, -2.1939029693603516, -1.399794340133667, 0.8233946561813354, 0.8743497729301453, 1.0582389831542969, -0.15349209308624268], [1.7604577541351318, -2.28620982170105, 1.5360603332519531, -0.9831602573394775, -0.6815540194511414, -0.3371778726577759, -0.04098936542868614, -1.9597129821777344, 0.6673652529716492, -1.0139724016189575, -1.9150118827819824, 1.2531886100769043, 1.5030505657196045, -0.5862651467323303, -0.1415356993675232, -0.9945091009140015, -0.33372730016708374, 0.6369214057922363, 1.0130276679992676, -0.07038993388414383, 1.8891093730926514, 0.3475951552391052, 1.9609609842300415, 0.10713112354278564, 1.9057952165603638, 0.7411404848098755, 0.13606762886047363, 1.8649102449417114, -0.6150475740432739, -1.5599175691604614, -0.9145128726959229, -0.020515354350209236, 1.535330891609192, -1.8658409118652344, -0.9369868040084839, -0.6663218140602112, 0.7297866940498352, -0.2142743468284607, -0.9167715907096863, 0.5634004473686218, 0.8252793550491333, 1.3160419464111328, 0.010816207155585289, 0.7571869492530823, -0.8340821266174316, -0.8414531946182251, -1.190440058708191, 0.00141876598354429, -0.2784852087497711, -0.5538502931594849, -1.5837401151657104, -0.03657791391015053, -0.5012871026992798, 1.8129996061325073, -0.4635687470436096, -0.4985962212085724, -0.2283025085926056, 0.13595789670944214, 1.0313024520874023, -0.1296234130859375, -1.4926714897155762, -0.03279748186469078, -1.621206283569336, -0.02630545198917389, -0.17572857439517975, 1.1005704402923584, 1.5292837619781494, -2.139749765396118, -2.320880889892578, 1.942386269569397, 0.8269739747047424, 1.133069396018982, -1.6524699926376343, 1.5089607238769531, -0.12832142412662506, 2.392427921295166, -1.0787501335144043, 0.8144552111625671, 0.4564985930919647, 2.895418882369995, 2.9972798824310303, -1.7706602811813354, -0.8712939620018005, -1.608540654182434, 0.6193849444389343, -0.5620474815368652, 0.5096645355224609, 0.16802053153514862, 1.244431972503662, 0.0725540965795517, -0.9459421634674072, 0.2514623701572418, 1.871928334236145, 2.3197011947631836, 0.5719213485717773, 0.7448236346244812, -0.6505770087242126, 1.053704023361206, -0.4320613741874695, 0.20790991187095642, 0.3751564025878906, 0.832302987575531, 0.10207916796207428, 0.7805482149124146, 0.19455862045288086, 0.18163512647151947, 0.7721130847930908, -1.0461890697479248, 0.1743221879005432, -0.6857733726501465, -0.7000702619552612, -0.38430681824684143, -0.8089354038238525, -0.4922386407852173, -0.014504699036478996, -0.0594356432557106, -0.4192028343677521, 2.8156580924987793, 1.1238826513290405, -0.0797177106142044, 0.27102288603782654, 0.14362961053848267, -1.3704607486724854, -1.3536920547485352, 1.552794337272644, 0.9207983016967773, 0.326558917760849, 1.3101898431777954, -0.27150028944015503, 1.1279761791229248, 0.15569624304771423, -0.28523916006088257, -1.4855989217758179, -1.5523945093154907, 0.02322087250649929, -1.2882148027420044, -0.1505320817232132, -0.8091399073600769, -0.21123670041561127, -0.08507370203733444, 1.158715844154358, 0.6605139970779419, -0.36754223704338074, -1.1157541275024414, -0.22229088842868805, 0.1493072509765625, -0.3055066466331482, -2.164952516555786, -1.3680579662322998, 0.053694307804107666, -1.1986719369888306, 0.8533357977867126, 0.7330235838890076, -0.2622382938861847, -0.20076079666614532, 0.24558281898498535, 1.2611017227172852, -1.181778073310852, 0.11793705821037292, 0.5664768218994141, -0.2852015495300293, 1.9883229732513428, 0.29169538617134094, 1.2233670949935913, 0.24918420612812042, -0.7364037036895752, 1.1801111698150635, -1.603468418121338, -1.0770845413208008, -0.23067843914031982, -0.5486291646957397, -1.2093302011489868, 0.7830506563186646, 0.676874577999115, -1.9401252269744873, -0.8316654562950134, -0.025729522109031677, -1.15144681930542, -1.5456911325454712, -1.7432271242141724, 0.14883890748023987, -0.9506551027297974, -1.0136916637420654, -1.0703850984573364, -2.598259925842285, -0.0888504907488823, 0.2879646122455597, -0.004688268061727285, 0.04233606532216072, 0.6562212109565735, 0.39376598596572876, -0.6998415589332581, 0.11148331314325333, -0.9442063570022583, 1.716132640838623, 0.3593176007270813, -0.5779337286949158, 0.3461696207523346, 1.1041728258132935, 0.43466562032699585, 0.43430811166763306, -1.9336975812911987, -0.06578423082828522, 0.7788234353065491, 1.055432915687561, 0.42704153060913086, 0.44367486238479614, -0.39775383472442627, 0.12584075331687927, -1.6902031898498535, 0.368705153465271, -2.1431233882904053, 0.9713314771652222, -0.5025619864463806, -1.431806206703186, 0.19900675117969513, -0.7271248698234558, -0.7923981547355652, -1.7504347562789917, -1.4718296527862549, -0.05340120941400528, -0.13360539078712463, -0.9582529067993164, 0.07700785994529724, 1.446520447731018, 0.1411450356245041, 1.3014172315597534, -1.0544087886810303, -0.41392138600349426, 0.8657196760177612, -0.10985003411769867, -0.23991142213344574, -0.07265382260084152, 1.8672715425491333, 0.30496764183044434, 1.141341209411621, -1.6556684970855713, -0.42928093671798706, 0.12702156603336334, -0.8170232176780701, 0.7311947345733643, -0.9311842918395996, 0.536224901676178, 1.0197283029556274, 0.22168225049972534, 1.1084678173065186, -0.6790544390678406, 1.0306270122528076, 2.307344675064087, 0.31302717328071594, 0.7118613123893738, -0.8413846492767334, 0.9108124375343323, -0.067422054708004, 0.5382977724075317, -1.031644344329834, -0.5102507472038269, -1.1884214878082275, -1.3714659214019775, 1.2022302150726318, -0.9298182725906372, 1.4112683534622192, 0.05768414959311485, -0.5184522271156311, 1.8794243335723877, -1.975921630859375, -0.5214003920555115, -0.7725751996040344, 0.17384040355682373, 1.4086858034133911, -2.37373948097229, -0.7132508158683777, -0.921768307685852, -0.36769941449165344, -1.051392674446106, 0.2751612961292267, -1.5243818759918213, -0.15032696723937988, -0.32536348700523376, 0.5015357136726379, 0.9438494443893433, -0.7512105107307434, -2.260009765625, 1.03689706325531, -0.11360643804073334, -1.5558644533157349, 0.3210362493991852, -1.0425031185150146, -0.17049111425876617, -0.6557384133338928, 0.901244580745697, -0.012025771662592888, -0.3256092369556427, -1.9516409635543823, -0.8479239344596863, -0.08861935883760452, 0.4649016559123993, 0.6272040009498596, -0.08226960897445679, 2.1646530628204346, -2.406806230545044, 1.4455674886703491, 1.1790573596954346, -0.1728912591934204, 1.163478970527649, 0.4693782329559326, -0.24518433213233948, -0.7382487058639526, -0.22657398879528046, -1.7766015529632568, 0.0071275485679507256, 1.0071932077407837, -0.8770388960838318, 0.7461150288581848, 0.520012378692627, 0.7000401020050049, 0.4881168305873871, -1.2555102109909058, -0.8217427730560303, 1.078136920928955, -0.05026627331972122, 0.3652766942977905, 1.597171664237976, -1.2766473293304443, -0.828491747379303, -0.008779874071478844, -1.6512919664382935, -0.7913057208061218, 0.5104649662971497, -2.674490213394165, -0.25372716784477234, 2.0323140621185303, -0.02896183356642723, 0.09512384980916977, 0.07920147478580475, -0.19999416172504425, 0.17618094384670258, -0.2877575755119324, -1.0114972591400146, -0.146914541721344, -0.24686962366104126, -0.02846217155456543, 0.2270018756389618, 1.663687825202942, -0.4886799156665802, -0.020215315744280815, -0.589968204498291, -0.545680820941925, -0.5428063869476318, 1.624910831451416, -0.4543156027793884, 0.17458941042423248, -0.5569565892219543, -0.1020064726471901, 1.4929025173187256, 1.4349271059036255, -0.5048609972000122, 1.654956340789795, 0.9243355989456177, -0.6968185901641846, -1.162055492401123, 1.2163074016571045, -1.025754690170288, 1.604661226272583, 0.5845432281494141, -0.41950923204421997, 0.7540240287780762, 0.4627843499183655, 1.2533766031265259, 0.05207169055938721, 1.382840633392334, -1.3953126668930054, -0.753861665725708, 1.2920918464660645, 0.09765571355819702, -0.26451224088668823, 0.7321093082427979, 0.28348013758659363, 0.6724259257316589, 0.3528996706008911, 1.0710983276367188, 0.2860945761203766, 0.3592107892036438, -1.0234785079956055, 0.5258581638336182, -0.7751054167747498, -1.5922447443008423, 1.5894759893417358, -0.7545779347419739, 0.8299643993377686, 0.2211485505104065, -0.027810903266072273, -0.7744901776313782, -0.6439397931098938, -1.9181994199752808, -2.5511574745178223, 0.3677214980125427, -0.6781636476516724, -1.424659013748169, -1.8442351818084717, 0.5062823295593262, -1.093697190284729, -1.1168897151947021, -0.658393144607544, -1.7467293739318848, -3.2253258228302, -1.3351655006408691, -1.6046442985534668, -1.5434963703155518, -0.9219101667404175, 0.44210538268089294, -1.0336053371429443, -0.7316529154777527, -2.4819228649139404, 0.7720922231674194, -0.12367712706327438, 0.46710649132728577, -0.39084702730178833, 0.4384767413139343, -0.39090099930763245, 1.0359033346176147, -0.6827207207679749, 1.4798747301101685, 1.5402511358261108, 1.0255939960479736, -2.0169427394866943, -0.3084312081336975, 1.0897490978240967, -0.19586867094039917, -0.5352753400802612, -0.502400815486908, 0.2449035942554474, 0.5137540102005005, 0.001084516872651875, -1.9029897451400757, 1.647063970565796, -0.5758987069129944, -0.5072796940803528, -0.710749089717865, 1.9104561805725098, -0.6564823985099792, 2.478180408477783, -1.1497801542282104, 1.0246644020080566, -2.031503200531006, -1.690760850906372, -0.6855515241622925, -0.12326868623495102, 1.3757091760635376, -0.07825954258441925, 1.168032169342041, -0.6677431464195251, -1.1637496948242188, -0.3395687937736511, -0.2490667700767517, -0.4989575445652008, 1.3773818016052246, -1.0587964057922363, 0.8282982707023621, 1.0355165004730225, 0.2612791955471039, -0.5685635209083557, 0.7491805553436279, -1.2600899934768677, 1.41669762134552, 1.6053444147109985, -0.8549354672431946, 1.0755164623260498, 0.37093794345855713, 0.23881718516349792, 1.9625905752182007, 0.8608651161193848, -0.2343454658985138, -0.6328832507133484, -0.288912296295166, -1.1193337440490723, 0.7550509572029114, -0.7290709614753723, -0.7423137426376343, -0.5423617959022522, 0.44587239623069763, -1.4283900260925293, 1.6278523206710815, 2.153589963912964, -0.34241703152656555, -1.360477328300476, 1.196045160293579, -0.12043173611164093, 0.364625483751297, -0.8791254162788391, -1.558854579925537, -0.2893969416618347, 0.46368446946144104, 1.1865384578704834, 0.6400880813598633, -1.136852502822876, 0.09193969517946243, 0.514150083065033, -1.2493826150894165, -0.6816282272338867, -0.28420114517211914, -0.21533647179603577, 0.317935049533844, -0.3454422652721405, 0.38745832443237305, 0.4531961679458618, 2.004081964492798, 0.1926150918006897, 1.1724574565887451, -0.04413168877363205, 0.6290584206581116, 1.0715559720993042, -0.9717106223106384, -1.7291080951690674, -1.7623851299285889, 1.4431312084197998, 0.6469358205795288, -1.2774467468261719, 0.8123172521591187, -1.5183053016662598, 0.8726455569267273, -2.0575573444366455, -1.829782485961914, -0.1473091095685959, 1.273706078529358, 0.25952693819999695, 0.5769439339637756, -0.8305160999298096, 0.8076685070991516, -0.09259716421365738, -1.5071699619293213, 0.5519728064537048, -1.7369128465652466, -0.4024161100387573, 0.5323233008384705, -0.16941200196743011, 0.014034171588718891, 1.2608096599578857, -0.18630042672157288, -0.1285773068666458, -0.7595137357711792, 0.39464715123176575, 0.9871556162834167, 0.05689329281449318, 1.1371495723724365, 2.203120708465576, -1.2395234107971191, -0.3652753531932831, -0.7463608980178833, -1.5968660116195679, 1.7532812356948853, 2.075826644897461, -0.0530487559735775, -1.7380034923553467, -0.3527728021144867, -2.6075549125671387, 0.1853034943342209, 1.5319215059280396, -0.4978344440460205, 1.5782206058502197, -0.5285354256629944, 1.0756886005401611, -1.3167617321014404, -1.126161813735962, -1.3011804819107056, -0.18483217060565948, 0.5673195719718933, -0.7677944898605347, -0.028630269691348076, -0.3461410105228424, 0.22722415626049042, 0.007371026091277599, 1.2879964113235474, 0.2026652991771698, 1.1075694561004639, 0.3118588328361511, 0.9976946115493774, 1.4628525972366333, -1.1301326751708984, 0.39961767196655273, 3.048877239227295, -1.2332888841629028, -0.05320127308368683, 1.3301318883895874, -0.22908636927604675, -0.1278342753648758, -0.31758230924606323, -0.00365354772657156, -0.3223832845687866, 0.5310640931129456, 2.5737764835357666, 1.1356226205825806, 0.12910142540931702, -0.4970526099205017, 0.33440592885017395, -0.05081469565629959, -1.4750136137008667, 1.7642539739608765, 1.359346866607666, -0.026020672172307968, -0.2011822760105133, -1.1607022285461426, -0.08127569407224655, -0.4792790114879608, -0.6604188084602356, -0.012111895717680454, 0.7234017252922058, -0.7262426018714905, -1.0351383686065674, 0.6479386687278748, -0.5701572895050049, 0.021690608933568, 0.7913975119590759, -1.0079164505004883, 1.9692600965499878, 0.10853581875562668, -2.4432075023651123, 0.8996294736862183, 0.2583407759666443, -0.5941535234451294, 0.6810711026191711, 0.25144729018211365, 0.5978391170501709, 1.6042439937591553, 1.02867591381073, 1.6097224950790405, -1.2503105401992798, -0.839791476726532, -1.0322175025939941, 0.13030077517032623, -0.7374500036239624, 0.07799701392650604, 1.3416528701782227, 0.2320869117975235, 2.2459144592285156, -0.45539137721061707, 0.7202131748199463, 1.035929560661316, -0.6257567405700684, 0.7272902727127075, 0.23045788705348969, -1.0428035259246826, -0.49058833718299866, 0.599782407283783, 0.8338646292686462, -2.2055623531341553, -0.056571297347545624, -0.998191237449646, 0.7970874309539795, 0.11019366979598999, -1.0380529165267944, 0.9701699614524841, -0.07379580289125443, 0.614092230796814, 0.4068514406681061, -1.7610951662063599, -0.655521810054779, 0.5606626272201538, 1.4903897047042847, -0.04496428370475769, 0.039548609405756, 0.06585098057985306, -0.42019417881965637, -0.005143263377249241, 0.8446562886238098, 0.7825310826301575, 0.05574246495962143, -0.8203749656677246, 0.0818958580493927, 0.21370597183704376, -0.6937718391418457, -1.3075931072235107, -0.6781476140022278, 0.38925790786743164, 0.15175877511501312, 0.28104501962661743, 0.6252792477607727, -0.21559973061084747, 0.3674340546131134, 0.9456662535667419, 0.6979434490203857, 0.1202559620141983, -0.16110315918922424, -0.013212785124778748, 0.5364922285079956, 0.6257641911506653, 0.35499170422554016, 1.0504710674285889, -0.18053396046161652, 0.26195624470710754, -1.4580808877944946, -0.07294423878192902, 0.010010610334575176, 0.047599535435438156, 2.275331735610962, 0.8603531122207642, 0.35820215940475464, -1.365127682685852, -0.10333198308944702, 0.9352515339851379, -1.3155438899993896, 0.5496869087219238, -0.44089293479919434, -1.1213940382003784, 0.46136602759361267, 0.9756642580032349, -0.9616740345954895, 0.2525689899921417, -0.21532726287841797, 0.4508506953716278, -0.6111485958099365, -1.1972646713256836, 0.9378208518028259, 0.035231031477451324, -0.1335645467042923, -0.2399003505706787, -1.1542015075683594, -0.03955480828881264, 0.04155511036515236, 1.3142746686935425, -0.09377110749483109, -1.6663916110992432, -2.273280143737793, -0.04171052947640419, 0.20349374413490295, -0.6752845644950867, 0.2511458992958069, 1.024307370185852, -0.5395605564117432, 0.47475531697273254, -0.5241035223007202, -0.08066967129707336, 0.4172947108745575, -1.8227856159210205, 0.43758973479270935, -2.4402828216552734, 0.8803860545158386, -0.5595996379852295, 0.7091110944747925, 1.2887473106384277, -0.28113728761672974, 1.6016730070114136, -0.03367229551076889, 0.9135454893112183, -0.5962734818458557, -0.623323380947113, -0.4967980980873108, -0.37602484226226807, 0.38961857557296753, 0.23605918884277344, -0.17551255226135254, -0.7999615669250488, 1.0826103687286377, -0.004393003415316343, 0.4000038802623749, -0.825042188167572, -1.733862042427063, 0.6226111650466919, -1.3441115617752075, 1.5149683952331543, 0.6128836870193481, 1.3930485248565674, -1.73206627368927, -0.5607749223709106, -0.9450057744979858, 0.8141813278198242, -1.4568620920181274, -0.4086124002933502, 0.6189176440238953, -0.013419883325695992, 2.1251749992370605, -0.994292676448822, 0.5504999160766602, -0.3287017345428467, -0.7187228202819824, -0.7080697417259216, -2.404585123062134, -0.6173928380012512, 1.3768349885940552, 0.6125672459602356, -1.3788197040557861, -1.8770320415496826, -0.3907935619354248, 0.5562239289283752, 1.514694094657898, 0.3921249210834503, 0.5845860242843628, -1.4037549495697021, 0.5806295871734619, -0.12984712421894073, 1.0029879808425903, -1.566949725151062, -0.9583519697189331, 0.1834239810705185, 0.0773729458451271, 0.9538422226905823, -0.011822725646197796, -2.0021212100982666, 1.5191094875335693, 0.760567843914032, -0.679918110370636, -1.604659914970398, -0.4521319568157196, -0.32662224769592285, 2.317183017730713, 0.055347707122564316, 1.871687650680542, 0.17764773964881897, -0.4106718897819519, 0.1965823918581009, 0.5311490893363953, 0.26188695430755615, 2.365863800048828, -0.9638758897781372, 0.4352722764015198, 1.2165716886520386, 1.5906307697296143, -1.3046536445617676, 0.1288677453994751, -0.31384727358818054, 0.4856037199497223, -0.3576951324939728, -0.13310599327087402, 0.9886330962181091, 0.9074572920799255, 0.7805717587471008, 0.6685136556625366, 0.9089378118515015, 0.8389811515808105, 0.3951762914657593, 0.11610236018896103, 1.4481947422027588, -0.6450306177139282, 0.23055122792720795, -0.12071231007575989, -1.3882626295089722, -0.11526539921760559, -0.542926013469696, 0.1323416531085968, 1.16434645652771, -0.12135013192892075, -0.4854840040206909, 1.0247654914855957, 0.10589069873094559, 1.422234296798706, 0.20045709609985352, -0.23381906747817993, 0.5302945971488953, 0.27619990706443787, 0.7087387442588806, -0.7842649817466736, 0.04841611161828041, -1.742090106010437, 0.649853527545929, -0.08517778664827347, 0.06184501573443413, -0.07796091586351395, 0.79177325963974, 1.0186376571655273, 0.22910521924495697, 1.2322899103164673, -2.3457465171813965, -0.19646255671977997, -1.242182970046997, -0.2746860980987549, -0.10483136028051376, 0.44348427653312683, 1.8542500734329224, -2.4501402378082275, -0.12927374243736267, 1.3946294784545898, -1.5663516521453857, 0.6898345351219177, -1.763183355331421, -1.0441315174102783, -0.3345015347003937, 0.36394423246383667, 0.9257307052612305, 0.7686364650726318, -0.5877331495285034, 0.7081171870231628, -1.4618607759475708, -0.03330961614847183, 0.707608163356781, -0.44165128469467163, 1.389755129814148, -1.980317234992981, 0.5403004288673401, -0.5289877653121948, 0.28080520033836365, 1.066017985343933, 1.6536186933517456, -0.012819662690162659, -0.023271262645721436, 0.20230546593666077, -0.9247196316719055, -0.36319682002067566, -0.6632490158081055, 0.8921712636947632, -0.27248048782348633, 0.9314182996749878, -0.9954832196235657, -0.9928410053253174, 0.06025700643658638, 1.7132307291030884, -0.2175789773464203, -1.0535423755645752, 0.9538130760192871, 0.8518080711364746, 1.1339753866195679, 0.44834521412849426, 1.1412982940673828, 0.35629475116729736, -2.722508192062378, 0.0014871052699163556, -0.4041539430618286, -0.44015973806381226, 0.9624399542808533, -1.9258098602294922, 0.16756151616573334, -1.4603084325790405, -1.0135198831558228, 0.19701090455055237, 1.8335826396942139, -2.8540310859680176, 0.5507970452308655, -0.3980768024921417, -0.051024869084358215, 0.37056228518486023, 0.49355652928352356, 1.91489577293396, -0.19138990342617035, 1.6643363237380981, 0.7073707580566406, -0.6058857440948486, 0.22207264602184296, 0.42163562774658203, -1.4097139835357666, -0.0018845237791538239, 0.6711356043815613, 0.30913645029067993, 0.14032115042209625, 1.1104711294174194, 0.33137813210487366, 1.542256236076355, 0.1608264446258545, -0.07315967231988907, -1.0157320499420166, -1.7251847982406616, -2.9565722942352295, 0.22911155223846436, -0.8031113147735596, 0.8002830147743225, -0.7127044200897217, 0.5942516922950745, -0.9621337056159973, 0.37695401906967163, -1.1216342449188232, 1.1306469440460205, 1.4435007572174072, 0.2774868309497833, -0.6310427784919739, -0.45336344838142395, 0.9841076135635376, -0.3299538493156433, 0.6594914793968201, 1.7426923513412476, 2.655322551727295, -0.02125733532011509, -0.5857188701629639, 0.9156652092933655, 0.26331764459609985, -0.8022121787071228, -0.7192347645759583, 1.7891511917114258, 0.7553418278694153, -0.8904079794883728, -0.7244842648506165, 0.8913716077804565, 0.18982547521591187, -0.32240793108940125, 0.8234705924987793, 1.6682448387145996, -1.0227614641189575, -1.4690582752227783, 1.187577247619629, -0.11482930928468704, 0.3906088173389435, 0.7784801721572876, 0.5045240521430969, -0.03568940982222557, 0.775039553642273, 0.2749512791633606, -0.9006931781768799, 1.1325831413269043, -0.0954681783914566, -1.1419227123260498, -0.5074729919433594, 0.977251410484314, 0.8255907893180847, 0.2083127796649933, -0.7248238325119019, -1.5342551469802856, -0.810269832611084, -0.030887536704540253, -0.9637706279754639, 0.5935044884681702, 1.030326247215271, -0.43385961651802063, 1.7448543310165405, 1.1415822505950928, -0.83412104845047, -2.2091546058654785, 1.8656264543533325, 0.2602231204509735, -1.6744916439056396, 0.11256664246320724, -1.205844759941101, 0.3417143225669861, 0.5514481067657471, 0.4972771108150482, -1.398919701576233, -1.5919123888015747, -0.1705598384141922, -0.12130507081747055], [0.18028461933135986, 1.372058391571045, 0.027613069862127304, 0.7396066188812256, -1.4695245027542114, 0.5618682503700256, 1.0146753787994385, -1.0060657262802124, -0.4771662652492523, -0.8312369585037231, 1.5583301782608032, 0.16833548247814178, -0.3200831711292267, 0.861463725566864, 0.20201510190963745, 0.7578237652778625, 1.127977728843689, -0.2277764230966568, 1.6390316486358643, -1.5261629819869995, -0.031166916713118553, 1.1542541980743408, -0.14445248246192932, -0.11439016461372375, 0.4626944959163666, 1.2043172121047974, -1.031826376914978, 0.25476980209350586, -0.20419757068157196, -0.6292552947998047, 1.5730401277542114, 0.3931700885295868, 1.2056947946548462, -1.2945960760116577, -0.4497981369495392, 0.06558249145746231, -0.7738535404205322, 0.29696282744407654, 0.08427112549543381, -0.24704943597316742, 0.5384365916252136, -2.332275390625, -0.6233013272285461, 0.5353899002075195, -0.6159714460372925, -0.36290955543518066, 0.34548163414001465, 0.3273252844810486, 2.819035768508911, 0.07200883328914642, 0.48866379261016846, -1.0602922439575195, -0.5067523121833801, 1.1287498474121094, -0.6045971512794495, -0.12738996744155884, -2.529348134994507, 0.8133262991905212, 0.1386508345603943, 0.15134358406066895, -0.6487664580345154, -1.457594394683838, 1.000731348991394, -0.8116076588630676, -0.5673534274101257, 1.830405592918396, -0.4064736068248749, 1.4261298179626465, -0.5527260303497314, 0.40706515312194824, 0.6793965697288513, 2.2942912578582764, -2.0268163681030273, 2.0715718269348145, -1.8497344255447388, 1.1663103103637695, 1.3227462768554688, 1.0592737197875977, 1.1021379232406616, -1.395190715789795, 0.798218309879303, -0.1407846212387085, 0.7028900980949402, 0.37318816781044006, -0.29067498445510864, -0.3261384665966034, -1.1719727516174316, 0.09464487433433533, -0.2848784029483795, 0.050566110759973526, -1.1182984113693237, -2.461010456085205, 0.08725259453058243, 1.168805480003357, -0.15678547322750092, 0.40860989689826965, -0.4966783821582794, -1.3986760377883911, -1.963714838027954, 0.8997688293457031, -1.4148117303848267, -2.203002452850342, -0.4688584506511688, -0.4865168631076813, -1.8477182388305664, 0.8374730348587036, -0.28267157077789307, 1.3070908784866333, -2.7591795921325684, 0.8974831104278564, 1.2044402360916138, -0.8560778498649597, -0.5568746328353882, 1.1542900800704956, 1.2443053722381592, 0.13688667118549347, -1.454271674156189, 0.1566048413515091, -0.13726182281970978, 1.924189567565918, -0.32146140933036804, 1.2778658866882324, -0.39048653841018677, 0.11197340488433838, -0.2851906418800354, 0.5997207760810852, -1.6627206802368164, -0.35189053416252136, -1.4261919260025024, 0.7380388975143433, 0.8929292559623718, 0.620956301689148, 0.6707119941711426, -1.488492488861084, -0.7954472303390503, -2.563941478729248, 0.2662157118320465, -0.7280665636062622, 0.2023056596517563, 1.040359377861023, -1.687664270401001, -0.489400178194046, 0.2598131597042084, -0.6683180928230286, 0.2884052097797394, -0.6869104504585266, -0.8667037487030029, 1.2731178998947144, 2.10695743560791, 0.1895899623632431, 0.5406980514526367, -0.7354074716567993, 1.0609064102172852, -0.9182056188583374, -0.2590189278125763, 1.1814619302749634, 1.9648927450180054, -0.15776848793029785, -0.4401024281978607, 0.8043526411056519, -0.2909294366836548, -1.4700831174850464, -1.3967725038528442, -1.1580710411071777, -0.1445578932762146, -0.9089667797088623, 0.5077497363090515, 0.6143773198127747, -0.7797125577926636, -0.385159432888031, 0.8841394782066345, 0.08537515997886658, -1.5128240585327148, -0.058032162487506866, 1.0960640907287598, 1.0922608375549316, 2.032134771347046, 0.5043943524360657, -0.3265575170516968, 0.3625517189502716, -0.6343279480934143, -0.2277459055185318, -0.024492138996720314, 0.9017995595932007, -0.35942578315734863, 1.5487852096557617, 0.39932316541671753, 0.5785601735115051, 0.6801356077194214, -0.29643934965133667, 0.26806434988975525, 0.8330885171890259, 0.49451524019241333, -0.6617481112480164, -0.5219815969467163, -2.014946460723877, -0.4684087038040161, -1.115772008895874, 1.7111449241638184, 0.9379448294639587, 0.651461124420166, 0.6814129948616028, 0.7726877331733704, 1.4330353736877441, 0.6711915731430054, 1.0285550355911255, -0.4677255153656006, -1.6436684131622314, -1.2631739377975464, -0.7191798686981201, -0.3914117217063904, 0.9844920039176941, 0.298086941242218, -0.7616615295410156, -0.1421527862548828, -0.7139256596565247, -0.7454758882522583, 1.0315684080123901, -0.07281836867332458, 0.32253628969192505, -0.13214099407196045, 0.7744156122207642, 0.5797536373138428, 0.4438682794570923, -0.49195507168769836, -0.41113871335983276, -2.0783703327178955, -1.3744598627090454, -0.7937335968017578, -0.2713085114955902, -0.9719486236572266, -1.1603978872299194, -1.4467607736587524, 0.14373885095119476, 0.2840508818626404, 0.7507966756820679, 1.9509811401367188, -0.5904995203018188, 0.5206798911094666, -0.31698164343833923, 1.0973968505859375, 0.9103220701217651, -0.9656418561935425, -0.7767506837844849, 0.7135247588157654, -0.9319855570793152, 0.4520018994808197, 0.13795676827430725, -1.0317376852035522, -0.16965630650520325, 0.5546602010726929, -0.4968150854110718, 0.5042563676834106, 0.6801292896270752, 0.8403850197792053, 1.2204055786132812, -0.41569286584854126, -0.8954753279685974, 0.06899832934141159, -1.4547630548477173, 1.252198576927185, 0.4286762475967407, 1.27242910861969, -0.027224993333220482, -0.6921839118003845, 0.15554368495941162, 0.14476758241653442, 0.17577654123306274, 0.2441135197877884, -0.8999252319335938, 0.7445372939109802, 0.6856333613395691, 1.051399827003479, -0.27578240633010864, 0.12665334343910217, -0.5304986834526062, 0.7808840274810791, -1.047520637512207, 1.2368723154067993, -0.016157828271389008, -1.0984864234924316, 0.9426684975624084, -1.5830312967300415, -0.7499159574508667, 0.13742521405220032, -1.1797254085540771, -0.3297281563282013, 0.1217331737279892, -1.3024299144744873, 0.4687221646308899, 0.5257679224014282, -0.8817396759986877, -0.8998021483421326, -1.1898283958435059, -1.3181265592575073, -0.9287755489349365, -0.5619696378707886, 1.2091820240020752, 1.1782628297805786, -1.557607889175415, 0.36336228251457214, -1.8374940156936646, 1.3938957452774048, 0.4166029393672943, -0.6886316537857056, 0.006413313560187817, -0.9805299043655396, -2.171295166015625, -1.4461369514465332, -1.8745206594467163, 0.5010475516319275, -2.4687490463256836, -1.553999900817871, 0.7063466906547546, -0.49482280015945435, -1.1330463886260986, -0.31838753819465637, -0.6680281162261963, 0.9826409816741943, 2.778049945831299, 1.2870584726333618, 0.5936715006828308, 0.5292128920555115, 0.7308093905448914, -0.29957932233810425, 0.49010026454925537, -0.7482884526252747, -1.5372272729873657, 0.7425310015678406, -0.25479644536972046, 0.7062282562255859, -0.24311760067939758, 0.9931870698928833, 1.238532543182373, -1.4545955657958984, 1.6484524011611938, -0.8840259909629822, 0.56755530834198, 0.9335037469863892, 0.272477388381958, 1.0372122526168823, -0.04955907166004181, 0.19167235493659973, 0.42408066987991333, -0.49348628520965576, -0.423284113407135, -0.7664165496826172, -0.6236980557441711, -0.1899956911802292, -1.240875482559204, -1.6294586658477783, 0.04197835922241211, -0.4168751835823059, -0.34758803248405457, 2.2289352416992188, -1.2794831991195679, 0.8309364318847656, -0.04188220575451851, -0.4827270805835724, 0.5459622740745544, -0.23673799633979797, 2.4531755447387695, 0.7815340757369995, -0.6696099042892456, 0.46594634652137756, -1.2793110609054565, 1.3087573051452637, -0.616416871547699, -0.3812788724899292, -0.6536248922348022, 0.04107992723584175, 0.9211676716804504, -1.2890353202819824, -1.1463487148284912, -0.6778392195701599, -0.8615908622741699, -0.15057498216629028, -0.13430708646774292, -0.6957584023475647, -1.0929745435714722, 0.8384113907814026, -0.11667005717754364, -0.05602813512086868, 1.018717646598816, -1.8048429489135742, 2.540634870529175, 1.725119948387146, 0.022680921480059624, -0.5995857119560242, -0.8750457167625427, -0.4362345337867737, -1.3154112100601196, -0.040016572922468185, -0.013530387543141842, 0.06289918720722198, 0.3066546320915222, 0.1175023689866066, 0.7626080513000488, 0.5550411343574524, 1.7974473237991333, 0.3709563612937927, -0.9259181618690491, 0.10626787692308426, -2.46768856048584, 0.6751189827919006, -0.8810123205184937, -1.144765853881836, -1.6545528173446655, 0.45985928177833557, 0.2561652958393097, 0.043406810611486435, -1.8176921606063843, -0.728936493396759, -0.4326518774032593, 0.6631148457527161, -0.9623904824256897, -0.37892022728919983, -1.1705670356750488, 0.9476684927940369, 0.5417914390563965, 0.811395525932312, 0.14772525429725647, 0.90720134973526, 1.8464070558547974, 0.7363097667694092, 1.399395227432251, -0.07730492949485779, 0.9609175324440002, 1.5928363800048828, 0.05878270044922829, 0.5627411007881165, 2.4914090633392334, -1.1935615539550781, -0.38319870829582214, 0.3282502591609955, 1.014826774597168, -0.087088443338871, 1.3089489936828613, 0.9118252396583557, -0.669208288192749, -0.6607390642166138, -1.4012913703918457, 0.03146422281861305, -0.8968841433525085, 1.7233341932296753, 1.8387839794158936, 0.22401973605155945, -1.2085691690444946, -0.42377421259880066, 0.2978428900241852, -0.8353676795959473, 0.2941945195198059, -0.8991796374320984, -0.6395888328552246, 0.2662205696105957, -0.8357576131820679, -0.04574325680732727, -0.9774373173713684, -0.038558248430490494, -0.9477714896202087, 0.353218138217926, -0.497602641582489, -0.2856725752353668, 0.5006707906723022, -0.002520884620025754, 0.7855563163757324, 0.7134275436401367, -0.6828286051750183, -1.8839266300201416, -1.1271600723266602, -0.6804039478302002, 0.4980238378047943, -1.2924354076385498, -0.4153955578804016, 1.244544506072998, -0.7650639414787292, 0.3082715570926666, 0.6601065993309021, 0.941907525062561, 1.21469247341156, -0.9658734798431396, 0.7625828981399536, 1.3361623287200928, -0.0914873257279396, -0.42386582493782043, -1.105749249458313, 0.9551212191581726, 2.001608371734619, -0.2550130784511566, 0.07696563005447388, 0.2860938310623169, 0.7387494444847107, 0.7972659468650818, 0.42670679092407227, -0.4534652531147003, 1.0435559749603271, -0.17333833873271942, 0.2031482607126236, -1.5069252252578735, 0.3029780983924866, 0.035869915038347244, 0.7206576466560364, -0.3556833863258362, -1.5347381830215454, -0.5032121539115906, -0.18261821568012238, 1.5967957973480225, 0.021162915974855423, -0.7384248375892639, -1.2958990335464478, -0.8214119076728821, -2.563455820083618, 0.5480180382728577, -0.605967104434967, 1.3456631898880005, -0.024911198765039444, -0.11562246829271317, -1.2722910642623901, -0.9158933758735657, 0.6072894334793091, -1.6717909574508667, -0.38109883666038513, -1.5064034461975098, 0.16730903089046478, 1.9984462261199951, -1.6095407009124756, 0.5345092415809631, -0.38765498995780945, -0.25628119707107544, 0.385634183883667, -0.7839213013648987, 0.5020336508750916, -0.04945339262485504, -0.4092446565628052, 0.5598620772361755, 1.9576634168624878, -0.32742762565612793, 0.2168017029762268, -2.205596446990967, 1.8113014698028564, -0.5129796862602234, -0.3360794186592102, 0.990197479724884, -0.2739103436470032, 0.28223028779029846, -0.4302731156349182, -0.5174338221549988, 0.2144245207309723, 0.4751642346382141, -1.9906057119369507, 0.7528449892997742, 0.9892178177833557, -0.5744270086288452, 1.5883231163024902, -2.0801525115966797, 1.4676697254180908, 1.9454699754714966, 0.5841719508171082, -0.11811883002519608, 0.048181965947151184, -0.4225316047668457, 0.6496787071228027, -0.792089581489563, -0.8807919025421143, -1.3866214752197266, 0.25332048535346985, -1.0123603343963623, 0.3920275568962097, -0.3201240599155426, 0.2948567569255829, 0.801188051700592, 0.5574017763137817, -0.8226049542427063, -2.18741774559021, 1.366565227508545, -0.26794278621673584, -1.05669367313385, 1.1384128332138062, -1.8040674924850464, -0.31951484084129333, 0.388568252325058, 1.1929033994674683, 0.968228816986084, 0.41870537400245667, -0.017811637371778488, 0.8985705375671387, 0.5050617456436157, 0.9192377924919128, -0.14845502376556396, -1.691001534461975, 0.565578818321228, -0.03655945137143135, 0.2902916669845581, 0.2370254248380661, 0.36182114481925964, 0.6896563768386841, -0.5854081511497498, 0.00013104514800943434, -0.9664007425308228, -0.5736792087554932, -0.2472403198480606, -0.9250641465187073, -0.23790091276168823, 1.7749685049057007, -2.2768774032592773, -1.5173418521881104, -2.385026216506958, -0.4427211582660675, 0.4546206593513489, 0.375495970249176, -0.7913075685501099, -1.669152021408081, -0.0944816991686821, 0.7655235528945923, 1.3904657363891602, -0.181028813123703, 0.8375378251075745, 0.07240647077560425, 0.41234472393989563, -0.06969989091157913, -1.5359388589859009, 1.2565863132476807, -1.1093697547912598, 0.4238152503967285, 0.10516367107629776, -0.846966564655304, -2.157270908355713, -0.06405141204595566, -0.9689051508903503, 0.3250378668308258, 0.9126907587051392, 0.6315744519233704, 0.9907671213150024, 1.1939564943313599, 1.8253499269485474, -1.0126978158950806, 0.7514832019805908, -1.068625807762146, -0.18333867192268372, -0.4219149351119995, 0.06136510521173477, 0.6882954835891724, 1.9580650329589844, 0.08388195931911469, -1.3110965490341187, -0.3219718933105469, 0.6928020119667053, -1.0340805053710938, 1.2506473064422607, -0.483900249004364, 0.9421855807304382, -0.355325847864151, 2.1014010906219482, -0.1292053759098053, 1.4275981187820435, -0.8287288546562195, 0.8620923757553101, 2.007901191711426, -0.5722222924232483, -0.2500908672809601, -1.0594444274902344, 1.8410664796829224, -1.7091120481491089, -2.4318296909332275, -1.0951707363128662, 0.5099042654037476, -0.8557953834533691, -1.3638765811920166, -1.7503427267074585, 0.2507908046245575, 0.3747333288192749, 0.13457049429416656, -1.2958052158355713, 2.0799641609191895, -0.6098824143409729, -0.9413378834724426, -1.070532202720642, 0.9794878363609314, -1.380942702293396, -1.824027180671692, -1.7373909950256348, -0.3528234362602234, 1.7540488243103027, -0.19563932716846466, -1.241281270980835, 0.8206813931465149, -0.4295523464679718, -0.1706410050392151, 0.0545828677713871, 2.2120556831359863, -0.7227362394332886, -0.14901241660118103, 0.7418830394744873, 0.9406169652938843, -0.0748489573597908, 1.3948991298675537, 0.005279276054352522, -0.0030771491583436728, -0.1829814314842224, 1.1810319423675537, 0.8654883503913879, -1.4338632822036743, 1.0602742433547974, -0.11901655048131943, -0.3471675217151642, 1.037827968597412, 0.4335894286632538, -1.0563337802886963, -0.567794144153595, 0.24734582006931305, 2.5723652839660645, -0.11724703758955002, -1.505552053451538, -0.6868165731430054, -0.7558444738388062, -0.6140048503875732, 0.6574317216873169, -0.011365316808223724, -0.7539317607879639, -1.1878780126571655, -1.2526458501815796, 0.8809294104576111, -1.3592742681503296, -0.2114928811788559, 0.29819193482398987, -0.000805032323114574, -1.0058249235153198, 0.8099303841590881, 1.8134292364120483, -0.7232610583305359, -0.7589526772499084, -0.667521595954895, -1.060760498046875, -2.1721248626708984, 0.8725326061248779, -1.5342155694961548, -0.8928056359291077, 0.8381226658821106, 1.3240922689437866, -1.0647461414337158, 1.2446681261062622, -0.015313002280890942, 1.0135873556137085, 0.05946224927902222, 1.3257765769958496, -0.7472749948501587, 0.6471025347709656, -1.6168922185897827, -0.2391517609357834, 0.9890154004096985, 0.38147348165512085, -0.8777762055397034, -0.14672140777111053, -0.32326728105545044, -0.4501952826976776, 0.06140981242060661, -2.5139825344085693, 0.16383779048919678, -0.27240288257598877, -0.4973340630531311, 0.5631448030471802, 1.7288975715637207, -0.6402236223220825, 1.856744408607483, 0.060866303741931915, -0.687210738658905, 1.2018094062805176, -0.006204143166542053, -1.5364433526992798, -1.0983202457427979, 1.5238667726516724, -1.2976948022842407, 0.4051794409751892, -0.5438076257705688, 0.6575807332992554, -0.3828527331352234, 0.6913185715675354, 1.0578547716140747, -0.25156503915786743, 0.05582752078771591, -0.21861328184604645, 0.0858471468091011, -1.2382725477218628, -1.0077385902404785, 0.8481919765472412, 0.020975559949874878, -1.0796103477478027, -1.2713558673858643, 0.3427928686141968, -0.6386515498161316, -0.1897563487291336, 2.2620697021484375, -2.181894063949585, 1.7848979234695435, 0.9963899254798889, 1.56010103225708, -1.245078206062317, -1.0537368059158325, -0.796294093132019, -1.0462431907653809, -0.26966771483421326, 1.681419849395752, 0.31167909502983093, 0.8649394512176514, 0.16138078272342682, 1.2567434310913086, 0.9099912643432617, 1.2152329683303833, -0.8146353960037231, 0.8380460739135742, 0.3601008355617523, 0.16959068179130554, -0.8767284750938416, 0.24332568049430847, 1.0903239250183105, -0.9237341284751892, -0.3093233108520508, 1.3385788202285767, 1.5693491697311401, 0.9129549860954285, -0.2559111416339874, -0.36038652062416077, -0.28293469548225403, 0.5397712588310242, 0.4591163694858551, 2.589907646179199, 0.13356634974479675, 0.135616272687912, -0.8965535759925842, -0.4019593894481659, -0.5369415283203125, 1.0076335668563843, -0.04634912684559822, 0.669231116771698, 0.20625171065330505, 0.13319379091262817, 1.0896376371383667, -0.4279586672782898, 0.540465235710144, 2.311068534851074, 0.7056781053543091, -1.2392911911010742, 0.015765175223350525, -0.05691831186413765, 1.2614784240722656, -0.8706594705581665, 0.032488126307725906, 1.0931522846221924, -0.6780477166175842, -0.48594048619270325, 1.2889668941497803, -0.40265509486198425, -1.415022373199463, 0.08251708000898361, -0.7565711140632629, -0.41527971625328064, 0.3642428517341614, 0.9924229979515076, -1.2056776285171509, -0.517827033996582, -0.516785204410553, -2.5030925273895264, 0.07531049102544785, -0.2580844461917877, 0.8185369372367859, 0.2879621088504791, -0.4824658930301666, -0.37393060326576233, 0.2771379053592682, -1.572619080543518, 1.9354146718978882, 1.3914107084274292, 2.013179302215576, 1.599174976348877, 1.2165069580078125, -0.7984330058097839, -0.6188399195671082, 1.7523062229156494, -1.2004891633987427, -0.0855676457285881, -0.8993796110153198, -0.27235451340675354, 0.7917658090591431, -0.4783010184764862, -0.26270583271980286, -0.5615472197532654, 0.646975040435791, 1.2003188133239746, 0.30399665236473083, 1.1304826736450195, -0.7654054164886475, -1.1106274127960205, 0.49053382873535156, 1.6683099269866943, -1.3517931699752808, 0.5367416739463806, 0.3957367241382599, -1.0369136333465576, -2.3109569549560547, -0.4171999394893646, 0.13978298008441925, 0.6077499389648438, -0.3834801912307739, -0.3614859879016876, 1.2641910314559937, 0.4418752193450928, -0.6632108688354492, -0.2976994812488556, -0.21127691864967346, 1.3176482915878296, 0.38163477182388306, 0.06816640496253967, 1.7137868404388428, 1.3304686546325684, 1.4227373600006104, -1.298956036567688, 1.4021316766738892, -0.9127814173698425, -0.21574445068836212, -0.39443671703338623, 1.4417657852172852, 0.20628580451011658, -0.1931871771812439, -0.11717105656862259, 0.8993074893951416, -1.8604021072387695, -0.20626644790172577, 1.0270880460739136, 1.9663965702056885, 0.5192438364028931, 1.9690306186676025, 0.18539835512638092, 0.7505872249603271, -0.03944103792309761, -0.16304799914360046, 0.8537188172340393, 1.6073592901229858, -1.2255337238311768, 0.3327839970588684, -0.7556349635124207, 0.8987202644348145, -1.223160743713379, -0.8562254309654236, 1.2037432193756104, 1.616837978363037, -1.4295177459716797, -1.1913820505142212, -0.5195331573486328, -0.7229886054992676, 0.6790899634361267, 0.11279556900262833, -0.5364556312561035, -0.2292807400226593, -0.9285616278648376, 0.6962195038795471, -0.6563453078269958, -0.07953976094722748, -0.116947241127491, -1.0197664499282837, 0.33357563614845276, -1.1801656484603882, -0.8498824238777161, 1.2194035053253174, 0.3578994572162628, 1.224768042564392, 0.7286545634269714, 0.22307778894901276, 0.12665323913097382, 0.7752053141593933, -1.709410309791565, -1.0116337537765503, -0.0911693349480629, -1.5574899911880493, 0.9312818646430969, 1.1447771787643433, -1.6434288024902344, -0.6762460470199585, -0.5605619549751282, -0.974404513835907, 0.012718885205686092, 0.18054628372192383, -0.1823047548532486, 1.0579566955566406, -0.5645778775215149, 1.845024585723877, -0.726763129234314, -1.029444932937622, 0.2378302961587906, 0.2113623023033142, -0.7935910820960999, 0.383790522813797, -0.2687264084815979, -0.6306191682815552, -0.9240751266479492, 2.1730051040649414, -1.6423324346542358, -0.7818490862846375, -0.8372628092765808, -0.8469647169113159, 0.2435270994901657, -0.5535551309585571, 0.6710613369941711, -0.4383413791656494, 0.36474838852882385, 0.535204291343689, 0.36126333475112915, -0.20973969995975494, 1.3501619100570679, 1.0657250881195068, -0.8692430853843689, 1.2341822385787964, 0.8195525407791138, -0.037623416632413864, -0.2856748104095459, -0.3285668194293976, 1.0962944030761719, -0.7382156848907471, 0.5469882488250732, 1.1238430738449097, -0.6969329714775085, 0.23134058713912964, -0.4437286853790283, 0.840628445148468, -0.029401889070868492, -0.5924325585365295]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [-0.013973673805594444, 0.024209268391132355, -0.010770206339657307, -0.012102171778678894], "dtype": "float32"}, {"__nd__": [1.0577452182769775, 0.9772136807441711, 1.0653672218322754, 0.9943830966949463], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-4.195446491241455, -2.605901002883911, -1.316453456878662, -4.889538288116455, -3.980191707611084, -4.148903846740723, 0.8272995948791504, 1.520148754119873, -2.927300453186035, -6.279682159423828, -6.737155914306641, 0.0919649600982666, -0.057018280029296875, -0.25753307342529297, -3.124281883239746, -3.8852810859680176, 0.13190317153930664, -4.130178451538086, -0.9129620790481567, -3.4799599647521973, 0.9817812442779541, -7.5197906494140625, -0.463595986366272, -0.7156229019165039, -4.006274700164795, -0.4364914894104004, -5.521112442016602, 0.7369873523712158, -5.06845760345459, -10.609549522399902, -6.298125267028809, 0.31638216972351074, -0.9936288595199585, 0.33812904357910156, -2.7868270874023438, -1.5570807456970215, -3.8828256130218506, -2.8246443271636963, -0.3746926784515381, -1.6263220310211182, -5.4080400466918945, -2.5717716217041016, 5.166337490081787, 1.6553678512573242, -2.4051504135131836, -1.4926252365112305, -4.97036075592041, -0.6916987895965576, -0.661829948425293, -0.6435446739196777, 0.578665018081665, -3.291996955871582, -3.8949296474456787, -0.3228687047958374, -3.67692232131958, -6.951972007751465, -0.9353801012039185, -5.189600944519043, 3.305485725402832, -2.7564096450805664, 0.06049966812133789, -1.062914252281189, 0.8644695281982422, -4.18798828125, -9.632902145385742, 1.361039638519287, -6.023706436157227, 0.4358367919921875, -4.759095191955566, -8.49087905883789, -5.430381774902344, -4.519721984863281, -1.9984022378921509, -5.218425750732422, 1.4434480667114258, -3.783489465713501, 0.9554884433746338, 1.6058611869812012, -1.6005045175552368, 2.8269147872924805, 2.1900429725646973, -3.6307992935180664, -0.562767505645752, -5.948708534240723, -3.152517318725586, -0.4506974220275879, 1.0176315307617188, -2.345411539077759, -0.6432759761810303, -1.0400606393814087, -4.961801528930664, 2.917797565460205, -3.997586727142334, 0.3531062602996826, -10.176159858703613, 1.3194680213928223, -3.462430715560913, -1.9377059936523438, -6.444014549255371, -6.953404903411865, -5.352780342102051, 1.1091251373291016, 1.5649771690368652, -1.5106215476989746, -4.983154296875, 0.26752758026123047, -6.437283039093018, -2.7816505432128906, -1.818462610244751, -0.521133542060852, 0.2250978946685791, -1.8726471662521362, -1.6537518501281738, -6.806604862213135, -0.8485312461853027, -5.552550315856934, -4.435114860534668, -2.5549633502960205, -2.6762051582336426, -4.55172061920166, -1.4255194664001465, -0.031239748001098633, 0.3345017433166504, 0.7345266342163086, -9.062335968017578, -5.781560897827148, 1.474120855331421, -7.559603691101074, 1.2388067245483398, -3.122915744781494, -0.995383620262146, 4.595789909362793, -1.5765299797058105, -1.6140916347503662, 1.0887460708618164, -2.8419506549835205, -5.722715377807617, -3.9154629707336426, 1.0982754230499268, -1.7295918464660645, -1.9253062009811401, 1.0600028038024902, -2.161229133605957, 4.260336875915527, -2.5970144271850586, -1.8764073848724365, -4.46605920791626, -1.8908953666687012, 2.844508171081543, -6.390235424041748, -4.286608695983887, -8.116556167602539, -2.8669447898864746, -1.9128350019454956, -1.1631102561950684, 2.93196964263916, -5.305400848388672, -0.6668336391448975, -0.8705377578735352, -0.3101985454559326, -4.79289436340332, -6.798924446105957, -9.142446517944336, -2.0539162158966064, 1.2542448043823242, -0.9728647470474243, 1.937087059020996, -1.1297909021377563, 1.9720096588134766, -3.5174989700317383, -0.27747297286987305, 0.9849700927734375, -7.655689239501953, 0.6478803157806396, -0.8813508749008179, 0.7936611175537109, 1.800173282623291, 1.8888332843780518, -3.164815664291382, 1.319751262664795, -1.8915709257125854, -2.3397674560546875, -1.6594946384429932, 0.3263731002807617, 0.9341831207275391, -3.8826980590820312, -2.851569414138794, 3.2232422828674316, -4.043371677398682, -2.8505911827087402, -7.344168663024902, -0.778788685798645, -5.443920612335205, -2.321702003479004, -1.499563455581665, -3.2777364253997803, -3.272738456726074, 1.291738510131836, -8.279595375061035, -0.11881768703460693, -0.02671074867248535, 0.6531786918640137, 3.5444250106811523, -1.567491054534912, -8.000247955322266, -2.7300169467926025, -2.9861035346984863, 0.3774120807647705, -2.703927755355835, -1.2953698635101318, -0.11223602294921875, 3.3707199096679688, -0.42473649978637695, 0.2928333282470703, -0.3381173610687256, -3.2848286628723145, -1.3147239685058594, -4.582095623016357, -0.39780962467193604, -3.0500214099884033, -2.401972770690918, 2.6556925773620605, -4.91935920715332, 3.527438163757324, -1.7400643825531006, -1.3886934518814087, -1.5275086164474487, 2.2826952934265137, 0.012344837188720703, -4.21430778503418, -1.7076609134674072, 1.9270973205566406, -4.9922380447387695, -3.3552560806274414, -3.7109386920928955, -2.2713065147399902, -3.537348747253418, -0.4165722131729126, 2.0547990798950195, -0.5197248458862305, -3.5913171768188477, -5.556530952453613, -1.006501317024231, 2.9842071533203125, 0.8360528945922852, 0.17359662055969238, -2.1266679763793945, -4.313141345977783, -3.3951172828674316, -2.983102321624756, 2.299642562866211, 3.2885069847106934, 0.7405991554260254, -8.72549819946289, -1.6991616487503052, -1.083038091659546, -3.7925195693969727, 0.7130756378173828, -0.5349514484405518, -0.9871002435684204, -2.67427134513855, -3.9941649436950684, -1.0893919467926025, -6.304325103759766, -3.535583734512329, -9.37264347076416, -0.15591192245483398, -2.8925390243530273, -4.334000110626221, -3.970879554748535, -0.9707508087158203, -4.382291793823242, -3.297764301300049, -2.5318076610565186, -2.5363430976867676, -5.807357311248779, -2.7633485794067383, -3.7931628227233887, -7.719291687011719, -2.880563259124756, -2.0909180641174316, 1.9269561767578125, -1.6073389053344727, -4.807365417480469, -0.19473052024841309, 2.126675605773926, 0.24555063247680664, 2.8454079627990723, 3.7681403160095215, 0.02256488800048828, -9.555316925048828, -1.114617109298706, -2.322840452194214, -1.2591553926467896, -0.6322780847549438, -6.021089553833008, -0.22148942947387695, -1.1250333786010742, -0.6501724720001221, -2.2733447551727295, 2.149092197418213, 1.6302490234375, -5.421616077423096, -2.858757495880127, 3.13346529006958, 0.47785234451293945, -2.168059825897217, -3.1010680198669434, -3.301912784576416, 0.6201472282409668, -3.8779075145721436, -1.6028355360031128, 2.9069557189941406, -2.113018751144409, 1.847421407699585, 2.553400993347168, -5.820451259613037, 0.6855969429016113, -8.932198524475098, 1.5335872173309326, -3.3086154460906982, -2.1088407039642334, -3.2934916019439697, 3.9086523056030273, -0.5996949672698975, -1.5374119281768799, 0.7524454593658447, -2.9739062786102295, -5.269813537597656, -2.426995038986206, 4.428074836730957, -3.9096157550811768, -1.079298734664917, -5.5059123039245605, -2.754314661026001, -7.874068260192871, 3.1480164527893066, -4.3986005783081055, 0.2469310760498047, 0.1404266357421875, -4.361082077026367, 0.9282870292663574, -3.4956748485565186, -6.147151947021484, 1.3114051818847656, -0.06702613830566406, -3.7986669540405273, 1.5278494358062744, 0.7877888679504395, -2.3455874919891357, -3.4806244373321533, -3.911621570587158, -4.635368347167969, -1.8638044595718384, 2.376859188079834, 1.0604901313781738, -2.351818323135376, -3.0933780670166016, -0.4768092632293701, -2.79195499420166, 0.2136087417602539, -2.7212071418762207, -4.415894508361816, -3.6364595890045166, 3.231781482696533, -8.965630531311035, -3.7943828105926514, 0.25365543365478516, -3.2485616207122803, -1.7141005992889404, -0.08056509494781494, 2.126309394836426, -6.846231460571289, -4.897114276885986, -0.8800656795501709, -7.137994766235352, -4.285591125488281, -4.735010147094727, -6.253192901611328, -4.123035430908203, 0.04118466377258301, -3.355966806411743, -0.1833866834640503, 4.155688285827637, -4.214639186859131, -0.5354660749435425, -3.960693597793579, -2.0305919647216797, -2.5197319984436035, -5.659833908081055, -2.6223912239074707, -1.967828631401062, 0.8138446807861328, -6.876291275024414, -8.93535041809082, -2.9638402462005615, 0.34870266914367676, -1.271935224533081, -3.111023426055908, -4.868178844451904, -1.352019190788269, 1.621293306350708, 0.1592543125152588, 0.5121579170227051, 0.19844889640808105, -1.0954045057296753, -0.46525025367736816, -2.4303691387176514, -0.85209059715271, -1.70268976688385, 2.3667359352111816, -2.0409457683563232, -3.064000129699707, -3.581737756729126, -3.062551498413086, -2.4235637187957764, 1.8891329765319824, -4.94011116027832, 2.7122902870178223, -1.4954835176467896, -0.9656132459640503, 0.6414577960968018, -4.059930324554443, -2.877077579498291, -0.6196777820587158, -2.102583646774292, 0.6637513637542725, -4.056562423706055, -4.080954074859619, -5.148774147033691, -0.47330617904663086, 0.8121728897094727, -4.009795188903809, -0.8457121849060059, 1.3601226806640625, -3.9792609214782715, -5.146474838256836, 4.8604583740234375, 2.382718563079834, 0.9673004150390625, -5.7903733253479, -6.846182346343994, -2.460670232772827, -4.750359058380127, -2.1690597534179688, -6.852555751800537, -2.895550012588501, -2.706815004348755, -1.7399147748947144, -0.24139118194580078, -1.0008375644683838, -1.2921041250228882, -0.961603045463562, -6.953761100769043, 0.6856236457824707, -6.168673038482666, 5.343214511871338, 3.290034294128418, -6.823585510253906, -2.071739673614502, -5.566884994506836, 0.029636383056640625, 2.348661422729492, 1.2227764129638672, 0.8757061958312988, -2.7944462299346924, 0.02744269371032715, 2.6414670944213867, -6.326932430267334, -0.24202704429626465, -2.8045105934143066, -0.7991242408752441, -3.789459466934204, -3.8698384761810303, 1.940676212310791, -6.804230690002441, -1.7736778259277344, -0.6040397882461548, -2.7411980628967285, -3.2776236534118652, 0.09428787231445312, -4.250611782073975, -2.3110151290893555, -3.149399757385254, -1.6952012777328491, 3.2309494018554688, 1.6155376434326172, -6.787084579467773, -2.6347150802612305, -6.308063507080078, -1.1205799579620361, -7.5903520584106445, -0.675602912902832, -3.5426299571990967, -0.6856058835983276, -7.063709259033203, 1.6139779090881348, -2.0172455310821533, -1.9343035221099854, -6.626814842224121, -4.581324577331543, 0.6201372146606445, 2.3829526901245117, 0.7607953548431396, -5.000658988952637, -2.2368013858795166, -1.3254640102386475, -7.411762237548828, -1.3868722915649414, -5.728053092956543, 2.4781508445739746, -3.2604410648345947, 0.36763453483581543, -1.875948190689087, -1.1767897605895996, 0.971592903137207, 1.060093641281128, -3.997570753097534, -1.5796462297439575, 0.8837132453918457, -4.2083563804626465, -1.836169719696045, -3.7137889862060547, -1.1902503967285156, 0.48108959197998047, -2.983969211578369, -4.38355016708374, -5.062844276428223, -0.25281262397766113, 0.25910282135009766, 3.209120750427246, -3.5011744499206543, -5.513462066650391, -1.5680997371673584, 0.9694552421569824, 1.2750537395477295, -2.6311044692993164, -5.829333782196045, -4.92015266418457, -0.8941715955734253, -2.280714750289917, -2.614516019821167, -1.8816969394683838, -2.0134057998657227, -1.0084105730056763, -0.4391155242919922, -4.665258884429932, -4.919120788574219, -4.52410888671875, 0.267972469329834, -1.9217700958251953, 0.6217207908630371, 0.0905313491821289, -9.713068008422852, -1.0817776918411255, -6.453207015991211, 0.6325604915618896, -1.5714092254638672, -1.3101563453674316, -1.933944821357727, -5.841811180114746, 0.2541508674621582, 0.1118934154510498, -9.663566589355469, -0.3334469795227051, -1.033874273300171, -3.569355010986328, -4.474294662475586, -5.435626983642578, -0.437941312789917, -1.4871010780334473, -3.5253195762634277, -6.889158248901367, 1.122323989868164, 2.2393598556518555, -1.0717726945877075, 0.6716568470001221, -4.41763973236084, -0.4131636619567871, -0.2027270793914795, 0.05022239685058594, -0.46714580059051514, 1.519571304321289, -1.2423399686813354, -4.389416217803955, -6.8437910079956055, -2.096127510070801, 0.5930004119873047, -3.921635150909424, -0.8250691890716553, 2.7415008544921875, -1.5659675598144531, -4.945746898651123, -1.9019633531570435, -5.394282341003418, -0.8543965816497803, -1.6924951076507568, -0.5137584209442139, -4.755985260009766, 0.5692620277404785, -1.3386839628219604, -8.405900001525879, -2.41171932220459, -5.360564231872559, -0.45154035091400146, 0.8258018493652344, -2.4569075107574463, -5.4563751220703125, -4.203566074371338, -2.3726561069488525, 1.1672029495239258, 4.305617332458496, -6.760685443878174, -1.2487545013427734, 0.5498604774475098, -5.083332061767578, -3.7717559337615967, 6.837553977966309, 4.089058876037598, -0.33405423164367676, -1.462788701057434, -4.394208908081055, -1.0758620500564575, -3.8613393306732178, -5.595817565917969, -4.579461574554443, -0.5168584585189819, -5.695866584777832, 0.16144442558288574, -2.02034854888916, -0.16595125198364258, 2.3347630500793457, -2.505567789077759, -4.988584995269775, -3.227023124694824, -2.122711420059204, 2.0731515884399414, 2.2120089530944824, -4.698809623718262, -3.584538221359253, 1.7841973304748535, -3.6140146255493164, -4.269812107086182, -2.0167553424835205, -5.98584508895874, -3.1333911418914795, -8.032158851623535, -0.6287291049957275, -2.021730899810791, -1.7112095355987549, -6.264918327331543, -8.26925277709961, -2.423121929168701, -2.8371572494506836, -0.23127377033233643, -2.4292025566101074, -4.051979064941406, 0.11807060241699219, -0.9604324102401733, -10.466021537780762, -1.2092140913009644, -2.3314952850341797, -1.950697898864746, -2.1244845390319824, -1.3927977085113525, 2.3448715209960938, -1.1441107988357544, 0.19398260116577148, -10.842862129211426, -2.8755784034729004, 2.7979793548583984, 6.35025691986084, 0.6925160884857178, -5.198962688446045, 1.772153615951538, -0.795622706413269, 0.4950602054595947, -2.3760664463043213, 0.4587979316711426, 1.604250192642212, -7.8507914543151855, -0.5263125896453857, -3.1012468338012695, -0.5849324464797974, -3.8033130168914795, -6.161949157714844, -1.1844903230667114, -2.957329034805298, -1.2027772665023804, -0.45702672004699707, -4.663898468017578, -2.7404825687408447, -5.57848596572876, -1.2399201393127441, -2.324000358581543, -3.5707335472106934, -5.420225143432617, -0.8677527904510498, -2.361639976501465, -1.7952821254730225, -10.753994941711426, 5.357539653778076, -3.9951000213623047, -2.1988470554351807, -0.8391516208648682, 2.076838970184326, -1.2808254957199097, -5.6341047286987305, -2.0230965614318848, -1.1774091720581055, -0.3713700771331787, -4.788730621337891, -3.7386679649353027, -2.6267523765563965, -7.753599166870117, -3.9030821323394775, -4.924870491027832, -1.4995492696762085, -2.8257155418395996, -3.6954140663146973, -8.723328590393066, 0.1088557243347168, 1.9788141250610352, -2.384488582611084, 1.7443199157714844, -2.8087801933288574, -1.7542052268981934, -3.91139554977417, -0.9086003303527832, -1.4927277565002441, -2.5516409873962402, -3.3735876083374023, -6.678704738616943, 0.761620044708252, -6.128715991973877, -3.180960178375244, 3.1965599060058594, 0.9832746982574463, -2.644711494445801, 2.7278037071228027, -1.8734397888183594, -3.578538656234741, -2.124417304992676, -1.128326177597046, -0.6294825077056885, -6.126715660095215, -5.6285929679870605, -4.353207588195801, -0.3126004934310913, -3.6616594791412354, 0.6825392246246338, -5.999619960784912, -4.336668014526367, -1.4296878576278687, -0.7381727695465088, 1.0925073623657227, -1.8425347805023193, 0.47675657272338867, -0.6265406608581543, -4.004204273223877, -1.568477749824524, -2.89727783203125, -2.4837522506713867, -4.125815391540527, -0.14444708824157715, -7.706515789031982, -5.988826751708984, 0.798642635345459, -5.023530960083008, 2.109494209289551, -2.664224147796631, 2.4957423210144043, 0.05501294136047363, -2.9212288856506348, -1.7069342136383057, -0.03811168670654297, -5.112956523895264, 0.7874770164489746, -12.44509220123291, -0.5734851360321045, 1.9819788932800293, -0.6109241247177124, -0.7179180383682251, 1.6912024021148682, -1.6476105451583862, -3.651250123977661, -1.0181156396865845, 2.3398618698120117, -2.9353890419006348, 1.6950888633728027, -2.3965978622436523, -4.205913543701172, -2.724015474319458, -3.6081056594848633, -2.9226620197296143, -5.767061233520508, -2.871828317642212, -1.278268575668335, 2.996218204498291, -1.285142421722412, -1.1589717864990234, -2.973844051361084, 2.9101881980895996, -5.887474060058594, -2.129318952560425, 3.685929298400879, -0.7600384950637817, -0.07397198677062988, -5.881141662597656, -2.6289286613464355, -4.703376293182373, 3.1884260177612305, 1.6284170150756836, -4.324436187744141, -0.8116298913955688, 3.317511558532715, -7.200164318084717, -0.09572207927703857, -3.452427864074707, -3.327134847640991, -4.796988010406494, -4.010550498962402, -2.621349573135376, -3.7895331382751465, -6.164996147155762, -0.24103641510009766, -2.094447374343872, -3.312138795852661, 0.5163660049438477, -2.71722149848938, -6.502676010131836, 1.7300565242767334, -7.3166913986206055, -5.073031425476074, 0.15744924545288086, -6.689078330993652, -3.4112777709960938, -5.922466278076172, -5.79444694519043, 0.35463380813598633, -4.184682369232178, 1.96506667137146, -1.1567082405090332, -1.1641523838043213, -0.6302231550216675, -3.761385917663574, -2.9641733169555664, 0.8653855323791504, -1.066757082939148, -7.692492485046387, -4.107660293579102, -2.503394365310669, -5.340228080749512, 0.26174449920654297, -0.35578179359436035, -4.3121795654296875, -5.14884614944458, -1.6153793334960938, -5.224210739135742, -4.695650100708008, 0.07324528694152832, 2.3247461318969727, -0.44089293479919434, -2.5385589599609375, -2.1102917194366455, -5.514225959777832, -4.041131019592285, -4.558004379272461, -0.17211341857910156, -1.1202303171157837, -7.100289344787598, -2.5755691528320312, 0.056304216384887695, -3.2211599349975586, -1.1656875610351562, 1.5554780960083008, -3.9536399841308594, 7.115297317504883, 1.0210952758789062, -3.6563096046447754, -4.20990514755249, 0.39913105964660645, -2.497070789337158, 1.921255111694336, -6.585292339324951, -0.9302903413772583, -0.9203519821166992, 1.404456377029419, -7.24169397354126, 1.7491326332092285, -2.777768611907959, -2.1859493255615234, -0.6405737400054932, 0.33044958114624023, -7.07027530670166, -7.155137062072754, 1.5190162658691406, 4.128916263580322, -2.6806790828704834, -3.2215943336486816, -4.791745185852051, -3.6393320560455322, -1.8758821487426758, -3.868560552597046, -4.220730781555176, 0.7964375019073486, -4.664299964904785, -3.939263343811035, 1.2818183898925781, -1.7375248670578003, -1.259436845779419, -4.093148231506348, -2.908345937728882, -3.3179688453674316, -6.074519157409668, -3.4765069484710693, -4.627385139465332, 0.5675809383392334, -0.21607887744903564, -3.1457576751708984, -0.8448123931884766, -2.371324062347412, 1.5353412628173828, -1.0823373794555664, -3.4902243614196777, 1.0017445087432861, 3.7240986824035645, -5.267169952392578, -2.5056166648864746, -0.02558755874633789, -0.9150981903076172, 0.03692984580993652, 0.9404175281524658, -3.057337999343872, 5.520435333251953, -2.785787582397461, -2.6358046531677246, -5.277266025543213, -1.7160829305648804, -6.090186595916748, -1.344003438949585, -1.5780229568481445, -3.0361499786376953, -4.843222618103027, 4.632900238037109, -1.499931812286377, -4.907837390899658, -3.2363953590393066, -2.530958414077759, -2.2827861309051514, -1.7579458951950073, -1.1743571758270264, -2.8949708938598633, 2.9349021911621094, 2.236175060272217, -0.773298978805542, -2.0702953338623047, -0.9835828542709351, -1.967449426651001, 0.07601165771484375, -5.001631736755371, -4.547277450561523, 4.7194976806640625, -0.537352442741394, -4.833114147186279, -0.05891680717468262, -2.23117733001709, -1.2287068367004395, -2.859334945678711, 0.4796781539916992, -2.982016086578369, -0.9345030784606934, 1.87925386428833, -2.499556541442871, -1.2452523708343506, 2.424128532409668, -4.012162208557129, -2.6982932090759277, 3.669236183166504, 1.792332649230957, -4.094182014465332, -3.625988721847534, -3.0134835243225098, -1.494396448135376, -2.962897777557373, -8.568399429321289, 1.1263463497161865, -5.997454643249512, -4.012950420379639, 2.1739282608032227, -3.5353126525878906, -3.8113861083984375, -0.2342778444290161, -2.703674793243408, -1.078029990196228, -1.6080657243728638, -1.6515856981277466, -1.1473655700683594, -4.634027004241943, -3.3003530502319336, 0.3339202404022217, 0.3118412494659424, 8.112918853759766, -1.3037835359573364, -3.0802645683288574, -4.262204647064209, -0.3133382797241211, -3.7354798316955566, 3.6513280868530273, -0.8535056114196777, -1.9177824258804321, 0.08216571807861328, -4.616812229156494, 1.2463862895965576, 0.9044029712677002, 4.499784469604492, -4.419010162353516, -1.8433727025985718, 1.6229472160339355, -3.7291691303253174, -2.833601951599121, -0.2457294464111328, -1.6903704404830933, 1.5690240859985352, 0.5759646892547607, -5.832136154174805, -3.079540729522705, -5.331658840179443, 1.758626937866211, -0.39918577671051025, -5.174881935119629, -2.1821773052215576, -3.2734785079956055, -2.389176368713379, -3.9962031841278076, -1.5237170457839966, -8.258323669433594, -3.043525457382202, -1.473549485206604, -2.674135684967041, 2.510289192199707, -2.0409295558929443, -1.63346266746521, -3.356070041656494, -3.1543850898742676, -0.6561435461044312, -2.026312828063965, -2.8436708450317383, -5.416354179382324, 0.7365179061889648, 0.09665942192077637, -0.9134564399719238, -3.8506083488464355, -0.23214423656463623, 2.0027003288269043, -2.715446949005127, -1.8646116256713867, -1.6756038665771484, -3.9216551780700684, -3.343498945236206, -0.69633948802948, 2.677891731262207, -1.4148764610290527, -2.292766809463501, -1.2897933721542358, -3.6034326553344727, -3.893125534057617, -5.758520603179932, -5.676945686340332, 0.6475718021392822, 2.358710289001465, -4.16345739364624, 4.80130672454834, 0.6399755477905273, -0.06858396530151367, -4.191676139831543, -0.700157642364502, -6.048707485198975, 1.1776587963104248, 2.561727523803711, 0.40160226821899414, -4.921672344207764, -3.036870002746582, 5.364743232727051, -1.5865495204925537, -7.021940231323242, -1.9232374429702759, -1.0305368900299072, -4.95496940612793, 0.7518668174743652, 0.7944045066833496, -7.8316426277160645, 0.26726436614990234, 1.169008493423462, -5.246509075164795, -1.5972614288330078, -6.460703372955322, -3.535900831222534, 1.3694517612457275, -1.9963520765304565, -4.309541702270508, 0.10534501075744629, -1.4188823699951172, -1.4689831733703613, 1.2661142349243164, -1.5143322944641113, -5.520317554473877, -3.7537918090820312, -1.1125926971435547, 1.2549376487731934, -1.988220453262329, 0.6702332496643066, -2.4143669605255127, -0.8243011236190796, -1.7222251892089844, 1.113637924194336, -3.323154926300049, 0.6212460994720459, -3.276146650314331, -3.22796893119812, 1.0767605304718018, 1.3261945247650146, -1.1041783094406128, -0.21548247337341309, 3.9334545135498047, -1.7506070137023926, 4.714357376098633, -1.745035171508789, 3.3161935806274414, 0.5575838088989258, -1.9498382806777954, -3.0958008766174316, -3.2983343601226807, -0.9490777254104614, -9.193492889404297, -5.289888381958008, -0.29397737979888916, -7.381022930145264, -5.49436616897583, -3.299325704574585, -4.999516010284424, -5.652844429016113, 0.8305227756500244, -0.42425990104675293, -5.888574600219727, -4.832569599151611, -1.178572177886963, 0.7196500301361084, -6.138789653778076, -1.4569344520568848, -1.5084806680679321, 1.1386961936950684, -4.646069526672363, -6.287545680999756, -3.071272611618042, -1.4182231426239014, -1.3776919841766357, 3.6039810180664062, -1.7169387340545654, -4.002560138702393, -2.1090400218963623, 1.1673002243041992, -5.007950305938721, 4.605776786804199, -3.059438705444336, -1.3636665344238281, 2.9651832580566406, -0.4452317953109741, -4.764743804931641, 0.5041382312774658, -0.8473228216171265, -4.291611671447754, -7.352024078369141, -3.725966453552246, -0.8828632831573486, -3.0840060710906982, -2.719120740890503, 1.0355489253997803, -4.3262434005737305, -2.6750035285949707, -4.50496768951416, -2.7378203868865967, -5.6986775398254395, 0.9893288612365723, -3.784071207046509, -1.148021936416626, 1.6577544212341309, -0.41769587993621826, -3.5451056957244873, -1.8452720642089844, -0.7761228084564209, -1.2296851873397827, -2.1220712661743164, -1.4640774726867676, -8.272139549255371, -3.2968201637268066, 0.26188087463378906, -7.331418037414551, -1.8201099634170532, -1.6451884508132935, -2.912065029144287, -3.0331506729125977, -6.4152374267578125, -2.5979084968566895, 4.248593330383301, -1.577479600906372, -4.16573429107666, -5.586055755615234, -1.902269959449768, -4.098348140716553, -3.10495662689209, -5.590260982513428, -0.04472792148590088, -1.6285765171051025, 3.9529523849487305, -3.1358842849731445, -5.847196578979492, -5.419104099273682, -3.5383095741271973, 1.2459256649017334, 2.2068376541137695, -6.081201076507568, -1.1847405433654785, 3.826688766479492, -1.325756549835205, -3.8475589752197266, 0.9023697376251221, 2.949862003326416, 1.4094245433807373, -5.290309429168701, -1.4687199592590332, -2.8127217292785645, -0.8805708885192871, 0.3856620788574219, -1.3151216506958008, -2.7758283615112305, -5.432891845703125, -4.678284168243408, -3.541886329650879, -0.760534405708313, -2.656291961669922, -4.887622833251953, -4.537463188171387, -4.511406421661377, -2.1272454261779785, 3.269726276397705, -1.892478108406067, -3.909121036529541, -3.3022642135620117, -2.0002291202545166, -1.8490279912948608, -0.3367478847503662, -2.3740711212158203, -3.300629138946533, -2.7544100284576416, -2.055309772491455, -3.768890380859375, -3.1446189880371094, -2.6463325023651123, -1.8402903079986572, -3.777677059173584, -7.442312717437744, -6.313851833343506, -3.7917749881744385, -5.435051918029785, -3.180757522583008, 1.8069653511047363, -8.481488227844238, 3.097079277038574, -3.9566547870635986, 1.6759889125823975, -3.133472442626953, -4.369339942932129, -1.1183966398239136, -2.2105026245117188, -2.4705348014831543, 0.1307203769683838, 0.6033492088317871, 2.926713466644287, -4.314009666442871, 2.7234597206115723, -3.779202699661255, -2.7739319801330566, -4.404045104980469, 2.2386388778686523, -3.6207032203674316, 0.6947855949401855, -3.229454517364502, -1.0612431764602661, 0.38697004318237305, -3.3770413398742676, 1.6629886627197266, 0.19439435005187988, 0.9297642707824707, 3.239161968231201, -5.224895000457764, -2.285107135772705, -0.9182591438293457, -1.7977644205093384, 3.513796329498291, -0.8671286106109619, 0.306135892868042, 1.3849570751190186, -3.1929073333740234, -3.3797149658203125, -0.738107442855835, -3.0728628635406494, -2.0880136489868164, -6.535264015197754, -3.0786094665527344, -4.33017635345459, -4.921591281890869, -1.3819384574890137, -4.684252738952637, -4.654251575469971, -5.360256671905518, -5.992085933685303, -1.5249228477478027, -0.09617841243743896, -5.7289719581604, -3.652773380279541, 1.0616700649261475, 0.9350178241729736, -2.0404818058013916, -1.9239914417266846, -1.901132583618164, 0.2681305408477783, -2.8594183921813965, -1.7641334533691406, -2.738283395767212, 1.0720911026000977, -3.6164469718933105, -6.707252025604248, 1.5964055061340332, -0.7847659587860107, -4.8422722816467285, 1.2905476093292236, -3.8487565517425537, -4.77058744430542, -6.694098949432373, -1.2403950691223145, -2.9906058311462402, -0.6045973300933838, -5.969091892242432, 2.7423086166381836, -2.022665023803711, -0.13793516159057617, -1.1007013320922852, -3.5120770931243896, -4.502764701843262, -7.296442985534668, -2.322052001953125, -2.855668783187866, -0.7018110752105713, -2.07047700881958, -6.587129592895508, 2.6845502853393555, 0.35701513290405273, -1.6558997631072998, -0.6148903369903564, -5.492451190948486, -3.4491419792175293, -0.6951885223388672, -7.823586940765381, -7.092939376831055, 1.475377082824707, -3.3699820041656494, -4.724955081939697, -1.2355798482894897, -5.903555393218994, -0.6672803163528442, 0.5812640190124512, -2.234896659851074, -3.3673110008239746, -3.751289129257202, -7.3029279708862305, -5.6082916259765625, 3.025583267211914, -3.0037574768066406, -7.883010387420654, 1.8393278121948242, -2.1642465591430664, -3.0623505115509033, -8.491291046142578, -3.5232346057891846, -0.9884687662124634, -2.4464821815490723, -1.216861367225647, 0.8489742279052734, -0.05646920204162598, 2.2717509269714355, -1.3893916606903076, -1.9523881673812866, 0.15050745010375977, 0.7511274814605713, -3.798746347427368, 0.041060447692871094, 1.4661507606506348, -4.3183794021606445, -0.7997262477874756, -0.015250444412231445, -10.360737800598145, -0.5674030780792236, -7.1146979331970215, -6.216156005859375, -2.8568832874298096, 1.9125347137451172, 0.1637098789215088, 2.4059605598449707, -5.348147869110107, -5.078319549560547, -6.936362266540527, -5.983644485473633, -3.113527297973633, -1.5991315841674805, -3.3196825981140137, -2.4512574672698975, -1.482790470123291, 4.215182304382324, -0.2983393669128418, -6.092575550079346, -5.055527687072754, -2.1333751678466797, -5.3572678565979, -4.1780924797058105, -0.2678954601287842, 1.2685399055480957, -4.160978317260742, -3.962066173553467, -3.5447702407836914, -3.955960512161255, -0.006976723670959473, 1.372002124786377, 2.8536343574523926, -0.10306262969970703, 0.22874164581298828, -4.831077575683594, -0.42871785163879395, -2.8098251819610596, -0.11783206462860107, 0.5023643970489502, 2.8809146881103516, 2.41949462890625, -3.479553461074829, 1.5286579132080078, -2.8303163051605225, -2.937661647796631, -2.4202499389648438, -8.08154296875, -1.625613808631897, 0.3093552589416504, -5.7281999588012695, -5.1139936447143555, -3.3369760513305664, 3.9563851356506348, -4.307171821594238, 0.08085441589355469, -2.477782964706421, -4.397749900817871, -1.5481252670288086, 2.592832088470459, -2.4987804889678955, -0.46968793869018555, 1.8929128646850586, 1.3788528442382812, -0.5731427669525146, -0.46270835399627686, 4.581364154815674, -0.3565964698791504, 2.671199321746826, -2.0279126167297363, -4.932455062866211, 1.6243481636047363, -3.959885597229004, -2.84019136428833, -3.690455436706543, -3.384439468383789, -3.3213794231414795, -3.0690231323242188, -4.414062023162842, -1.0204923152923584, -6.37129020690918, 1.3645820617675781, 0.4900484085083008, -3.326669454574585, 3.129230499267578, 1.5424580574035645, -3.986841917037964, 0.49497103691101074, 1.0147764682769775, 0.5704264640808105, -1.5307989120483398, -1.6972873210906982, -0.7610079050064087, -1.2833582162857056, 1.6437079906463623, -4.029840469360352, -8.098383903503418, -11.9776029586792, 0.5671718120574951, -2.3524045944213867, 0.1360306739807129, -4.160923004150391, -3.8937292098999023, -2.374382495880127, 3.0194849967956543, -2.035670518875122, 1.7301626205444336, -2.126859664916992, -6.576112270355225, -2.161729574203491, 3.47458553314209, -4.006441116333008, -5.189446449279785, -1.7323014736175537, 0.6202254295349121, -5.89808464050293, 0.8480658531188965, -2.130880832672119, -2.8195509910583496, -4.830499649047852, 5.20383358001709, -4.074273586273193, 4.294678211212158, -2.357097625732422, 2.0562362670898438, -3.2073028087615967, 1.2676482200622559, -2.5434415340423584, -2.26823353767395, -5.399070739746094, -2.81394100189209, -4.058710098266602, 2.816293716430664, -0.8654472827911377, -0.9297686815261841, -2.8922300338745117, -1.4471808671951294, 3.17264461517334, 0.2653183937072754, -6.315675735473633, -3.0289602279663086, -1.7575887441635132, -2.02022123336792, -6.303520679473877, -2.1588199138641357, -3.8731093406677246, -2.976130485534668, 0.6531434059143066, -2.0104615688323975, 0.045243263244628906, 1.354766845703125, -4.652409553527832, -0.5709608793258667, -1.9532856941223145, -8.014266014099121, -4.743899345397949, 1.5199172496795654, -8.43407917022705, -0.9855355024337769, -4.557958126068115, -5.518810272216797, -0.3157317638397217, -0.8446846008300781, -2.852379322052002, -1.6092636585235596, -2.2298197746276855, -5.060209274291992, -4.7594709396362305, -3.753063917160034, -1.3089615106582642, -3.373476982116699, -2.325359582901001, -1.730474591255188, -5.203278541564941, -3.737240791320801, 0.8005332946777344, -0.6038626432418823, -1.2153208255767822, 0.6883516311645508, -2.69858717918396, -1.4363086223602295, 1.315938949584961, -1.413349986076355, -7.494621276855469, 1.0947260856628418, -1.5642977952957153, -2.5526278018951416, -5.634639263153076, -4.91151237487793, 0.120635986328125, -2.2239136695861816, 1.057234287261963, -2.8594791889190674, -2.295849084854126, -0.4692230224609375, 2.378133773803711, -0.40396082401275635, -0.7879345417022705, -3.972921133041382, -0.7893610000610352, 1.4989428520202637, -2.949266195297241, -2.347625970840454, 0.013806581497192383, -5.387054920196533, -2.9323720932006836, -3.000002384185791, -1.0159982442855835, -4.81602144241333, -1.1202466487884521, -2.7934136390686035, -1.8842214345932007, 0.24624180793762207, -0.5985462665557861, -3.943152666091919, 0.02750849723815918, -3.3322994709014893, -1.73097562789917, -3.8794610500335693, -4.57403564453125, -3.2433576583862305, -4.053690433502197, -3.5023269653320312, -0.3097188472747803, -0.7537326812744141, 1.0022282600402832, -4.470049858093262, -1.7553271055221558, -1.4778902530670166, -4.270811080932617, -2.1139047145843506, -2.893003463745117, -2.755131721496582, -5.10296106338501, 1.2092466354370117, -4.534507751464844, -0.7768502235412598, -0.9133310317993164, 0.24532198905944824, -3.868527412414551, 2.466597557067871, -2.0520970821380615, -0.4843764305114746, -5.161348342895508, -2.6186835765838623, -5.426614284515381, -2.40779972076416, -4.959231376647949, -2.4153800010681152, 0.9292104244232178, 0.05974102020263672, -1.4901012182235718, 0.20856404304504395, -0.4141209125518799, -4.388641834259033, -5.913350582122803, 0.698462963104248, -1.437415599822998, -2.629138708114624, 1.7419867515563965, -3.890359401702881, -3.8599448204040527, -2.145850896835327, -3.214934825897217, -1.0913102626800537, -6.048938751220703, -4.74522066116333, 3.161713123321533, -5.045783519744873, 1.0752167701721191, 1.0706367492675781, 1.717444896697998, -3.0511178970336914, -6.9421916007995605, -3.1051650047302246, -4.101590156555176, -3.792874813079834, -0.49755263328552246, -5.743007183074951, -6.649669647216797, -5.849749565124512, -8.485553741455078, -3.605041980743408, 0.7542312145233154, -3.449350357055664, 1.9249706268310547, -5.712647438049316, -9.589189529418945, -0.668053150177002, 0.8665399551391602, -0.5330359935760498, 3.091691017150879, -5.540463924407959, -1.0686414241790771, -2.7729005813598633, -1.6020770072937012, -3.165508508682251, -4.479567527770996, -5.831820964813232, -1.522501826286316, -4.048886299133301, 0.4165029525756836, -5.345527648925781, -3.1034960746765137, -1.1069135665893555, -10.258426666259766, -7.251437187194824, -5.7491631507873535, 1.7855803966522217, -2.2536563873291016, -1.4817392826080322, 1.7242372035980225, -9.409347534179688, 1.171659231185913, -4.714693069458008, 1.8530025482177734, 0.7674925327301025, 1.024092674255371, 1.895423412322998, -3.5851969718933105, 0.6816864013671875, -5.0637078285217285, -1.8075170516967773, -4.848305702209473, -2.153305768966675, -3.0710737705230713, 0.6956431865692139, -3.206202507019043, -6.391473770141602, -3.336927890777588, 1.4721696376800537, -5.773891448974609, 6.872417449951172, -0.31998467445373535, -8.357880592346191, -1.9723730087280273, -2.591946601867676, -0.527534008026123, -1.9677072763442993, -3.4595835208892822, -4.812089920043945, -4.350785255432129, -1.8619599342346191, -3.193526268005371, -3.7009825706481934, -8.120882034301758, -1.37398362159729, -1.9596760272979736, 1.4739923477172852, -7.7496466636657715, -2.186572313308716, -0.9980989694595337, 0.9918973445892334, 0.8418796062469482, 0.21845746040344238, -4.004339694976807, -1.1449131965637207, -0.5027388334274292, -3.862793207168579, -0.8403371572494507, 1.7369630336761475, -0.3454408645629883, -0.7183657884597778, -7.626435279846191, -1.4406459331512451, -1.0155045986175537, -4.033191204071045, -1.85905122756958, 2.9488449096679688, -1.7876492738723755, -3.952094554901123, -5.228346347808838, 1.466841697692871, -3.476132869720459, -1.567101001739502, -2.1334242820739746, -3.894056797027588, -6.46001672744751, -5.883842468261719, 0.3841543197631836, -2.585890769958496, -4.604609489440918, -1.0159940719604492, -2.2155444622039795, -2.0364768505096436, 0.07077479362487793, -10.259772300720215, -2.9253365993499756, -3.100001811981201, -4.725643157958984, -3.2503061294555664, -6.188626766204834, -4.508947372436523, -1.0210821628570557, -5.309354305267334, -3.3435568809509277, -2.8536911010742188, 2.3723740577697754, 1.6481332778930664, -1.578558325767517, -2.438114881515503, -6.066522598266602, 1.295189380645752, 5.142070770263672, -6.747588157653809, -0.8182259798049927, -0.9156725406646729, -4.51362419128418, -1.0394854545593262, -0.6681650876998901, -5.207414150238037, 4.467077732086182, 0.12481570243835449, -5.7722249031066895, -1.8034805059432983, -0.47208285331726074, 0.971137285232544, -1.8981108665466309, -0.03792095184326172, -1.8279591798782349, -5.332516193389893, -4.140667915344238, -0.8666523694992065, 0.31932640075683594, -1.0041871070861816, 3.3677845001220703, -4.02482271194458, -3.3595261573791504, 1.9742920398712158, -0.20353198051452637, 2.7002344131469727, -5.106986999511719, -0.6032330989837646, -2.0857064723968506, -4.5830769538879395, -5.964818954467773, -0.04188680648803711, 0.5604643821716309, -0.9087231159210205, -1.0709853172302246, -0.09972381591796875, -3.32981014251709, 1.100456714630127, -5.785477638244629, 1.3693130016326904, -2.497127056121826, -0.8371448516845703, 0.2075493335723877, -1.596769094467163, 2.219662666320801, 2.343088150024414, -6.311441898345947, -4.682486534118652, -0.5517406463623047, -2.9373972415924072, -3.2155652046203613, -3.034311294555664, -2.120389461517334, -0.25263988971710205, 0.8432848453521729, -2.7833592891693115, -5.222526550292969, 0.47235822677612305, -1.603472352027893, -4.778133869171143, 0.3719494342803955, -2.3655569553375244, -1.0034263134002686, -2.784152030944824, 0.40070319175720215, 1.9642562866210938, -1.0719718933105469, -2.0964512825012207, -3.280728816986084, -0.16888201236724854, -0.010626792907714844, -1.2540135383605957, -1.3740723133087158, -2.5690526962280273, -1.5527253150939941, -3.874152898788452, -1.5281617641448975, -4.040550708770752, -5.4434638023376465, -8.790802001953125, -0.08831787109375, -1.0563510656356812, -3.162761688232422, -4.371213436126709, 0.14887642860412598, -1.6642804145812988, -1.2360432147979736, -0.6688227653503418, -0.1742534637451172, -1.7787892818450928, 1.8464922904968262, -0.12111866474151611, -2.94469952583313, -1.1034268140792847, 1.318739891052246, 0.846116304397583, -4.330513000488281, -5.538576126098633, -1.5565125942230225, -3.2403101921081543, -4.124588966369629, -3.7217016220092773, -2.8178982734680176, -3.3589375019073486, -2.4711666107177734, 1.8295714855194092, -6.087199687957764, 2.622264862060547, -5.998486518859863, 2.6289358139038086, -1.9725829362869263, -0.2506403923034668, -0.8283042907714844, -2.3934926986694336, -0.01456904411315918, -3.3552260398864746, 0.5084760189056396, -1.3883885145187378, -0.04711103439331055, -8.408599853515625, -7.708134174346924, 3.9262800216674805, 1.4127492904663086, -3.883005142211914, -10.019638061523438, -1.692033052444458, -1.3695727586746216, -3.548830032348633, 1.423715353012085, -0.5091526508331299, -2.7283236980438232, 1.6976842880249023, 0.9098172187805176, -0.8505549430847168, -12.87474536895752, -2.689934730529785, -4.996355056762695, -10.325343132019043, -3.599385976791382, -6.813945293426514, -3.056401014328003, -0.03317713737487793, -4.025747299194336, -5.1636061668396, -2.991894245147705, -0.893470287322998, -2.3486618995666504, -4.799128532409668, 2.5996570587158203, -5.966341972351074, -4.807550430297852, 0.6118888854980469, -1.5331717729568481, -3.413447141647339, -4.486320495605469, -2.3529467582702637, 0.3186779022216797, 3.501896858215332, -6.78261661529541, -2.5804712772369385, 2.6041412353515625, 5.697697639465332, -9.462903022766113, -1.8195466995239258, -3.391676425933838, 5.715967178344727, -6.549006938934326, -6.314956188201904, -0.7404648065567017, -1.1356195211410522, -3.1973276138305664, 1.6378798484802246, 3.598465919494629, 1.0766100883483887, -3.2963380813598633, -9.372708320617676, -5.776910305023193, 0.39174628257751465, -3.56380558013916, -1.343217372894287, 1.397768497467041, -0.10441744327545166, -1.6108728647232056, -3.3489603996276855, -1.0845654010772705, -1.4603233337402344, -4.832982540130615, -2.0826900005340576, 1.058304786682129, -0.9936008453369141, 1.1879172325134277, 0.9138226509094238, -0.8613934516906738, -3.1150364875793457, -2.4089479446411133, -7.11929178237915, -7.317799091339111, -9.84809398651123, -3.187161922454834, 2.281691074371338, -0.35733556747436523, -2.906744956970215, -1.6332786083221436, -0.8336551189422607, 2.8212876319885254, -0.4268369674682617, -3.7711703777313232, 0.9836668968200684, 2.98024845123291, -2.189643621444702, 3.037713050842285, -4.232133388519287, 2.3912367820739746, -0.9160586595535278, -2.3235671520233154, 3.399578094482422, 2.9455533027648926, -3.075305223464966, -1.3954663276672363, -6.376557350158691, -2.2226290702819824, -0.276763916015625, -0.8554481267929077, -1.108403205871582, -4.831979751586914, -0.34104084968566895, -5.517059326171875, -5.756666660308838, -0.7447636127471924, 2.2960824966430664, -7.565972328186035, -5.900615692138672, -7.105915546417236, -0.6278152465820312, -0.023459196090698242, -3.98827862739563, -5.347421646118164, -6.887932777404785, -4.77932596206665, -1.7333040237426758, -3.102520704269409, -4.377342224121094, -6.565673828125, 0.30057215690612793, -5.050749778747559, -1.0685365200042725, -4.203718185424805, -1.223482370376587, -5.9412102699279785, -0.4675257205963135, -4.05445671081543, 0.40473175048828125, -2.8457422256469727, -7.243616104125977, 2.0623703002929688, -1.7387492656707764, -1.545962929725647, 2.6086316108703613, -3.403785228729248, -4.908453941345215, 1.7700538635253906, 2.5335168838500977, -4.791494369506836, -5.380699634552002, -3.9965603351593018, -4.953630447387695, -1.4932224750518799, 1.0361769199371338, -2.5629525184631348, -4.04619836807251, -4.856781959533691, 1.7657604217529297, -2.893735408782959, -1.1185240745544434, -0.451122522354126, 0.3442051410675049, 2.153563976287842, 0.007296562194824219, -7.273319721221924, 4.117016792297363, -1.6726455688476562, 1.6050605773925781, -2.7640113830566406, -2.9345545768737793, -2.391634464263916, 0.6153967380523682, -10.627534866333008, -2.7124929428100586, 0.4123673439025879, -0.5541777610778809, -0.22188663482666016, -7.33258581161499, -5.331942081451416, -6.064481735229492, 3.217639923095703, 0.36655759811401367, -1.5320032835006714, -2.279932737350464, -1.9894139766693115, -2.3802685737609863, -0.6540800333023071, -1.0993008613586426, -0.6107962131500244, -2.0705583095550537, -2.3075666427612305, 1.4322640895843506, -5.321620941162109, -1.513325572013855, 0.17055892944335938, -4.984281063079834, -0.21101164817810059, -3.126692771911621, 0.6604537963867188, -1.497239351272583, -1.131712794303894, -0.6482728719711304, 4.06724739074707, -1.2059882879257202, 5.382633209228516, 3.1322622299194336, 1.4678678512573242, -3.683053731918335, -1.3298068046569824, -0.17585575580596924, 5.276582717895508, -2.2808570861816406, -2.272371768951416, -1.1723777055740356, -2.8816075325012207, -5.458780765533447, -10.884286880493164, -5.50816535949707, 0.702794075012207, -2.8716847896575928, -0.9667938947677612, -5.291621208190918, 0.21112823486328125, -1.8856717348098755, -3.632171154022217, -1.908420205116272, 1.6546835899353027, -4.388655662536621, 0.6686394214630127, -2.294718027114868, -6.646230697631836, -2.0941522121429443, -7.391072750091553, -2.3910977840423584, 0.1174314022064209, 0.972100019454956, -3.348696231842041, 0.008446216583251953, 0.48075246810913086, -0.7966747283935547, -4.52993631362915, -7.4735941886901855, 1.2693467140197754, -0.4023096561431885, 2.135593891143799, -4.375557899475098, -2.033766508102417, -0.16122090816497803, 1.2313427925109863, -7.0872344970703125, -0.13550245761871338, -2.9344143867492676, -2.2000417709350586, -1.0934321880340576, -4.268953323364258, -3.5897064208984375, -0.5738277435302734, -4.571150779724121, -2.7075717449188232, -3.1200640201568604, -1.4927245378494263, -0.12950825691223145, -6.3995466232299805, -5.0386433601379395, -1.3272459506988525, 2.1813106536865234, -2.995201587677002, 0.614403247833252, -2.1473119258880615, 0.4942901134490967, 0.26986050605773926, -3.592526912689209, -4.408102035522461, -5.46574068069458, 4.3517560958862305, 10.437847137451172, -1.8929225206375122, 1.2351222038269043, -6.71341609954834, -0.8372701406478882, -0.13316774368286133, -3.2980222702026367, 0.8671176433563232, 2.145359516143799, -5.326498985290527, 1.1946303844451904, -3.5261521339416504, -10.626729965209961, -0.3821526765823364, -0.4581536054611206, -1.51600182056427, -2.810891628265381, -11.156987190246582, -5.2636799812316895, 0.35625457763671875, -5.813120365142822, 3.660585880279541, -4.768618106842041, -0.12149190902709961, 1.6627593040466309, 3.5005245208740234, -0.7219101190567017, -0.21975314617156982, -3.0068116188049316, -1.0347404479980469, -3.3010427951812744, -10.397035598754883, 0.06611037254333496, -3.2588860988616943, -1.5785579681396484, 0.6068198680877686, -7.088050365447998, -3.8964052200317383, -5.2046027183532715, -1.5646674633026123, 3.692164421081543, -5.47545862197876, 2.6954383850097656, -5.439371109008789, -1.3743749856948853, -0.2829374074935913, -0.4354335069656372, -0.35187602043151855, 0.07062196731567383, -4.909083366394043, 1.0855507850646973, -2.0505051612854004, -2.0877749919891357, -4.546653747558594, -0.6851029396057129, -2.743332862854004, -0.33115458488464355, -2.6566309928894043, 0.7019145488739014, -2.8313021659851074, -2.899293899536133, -1.3001004457473755, -1.9991506338119507, -1.3889901638031006, -1.1411722898483276, -1.6830739974975586, -5.379700183868408, -2.988330125808716, -3.426009178161621, -2.3806233406066895, -3.762962818145752, -3.030529499053955, -2.058436870574951, 1.4452602863311768, 2.5302577018737793, 0.818981409072876, 0.6477758884429932, 0.34273338317871094, 1.1648402214050293, -0.862354040145874, 2.858579635620117, -1.8241032361984253, 1.3799805641174316, -1.9608632326126099, 2.839663505554199, -0.029448270797729492, -4.339591979980469, -5.303892612457275, -2.430975914001465, -3.1823079586029053, -1.0326707363128662, -1.5852398872375488, -7.806246280670166, -0.22305870056152344, 1.655329704284668, -3.1091301441192627, -1.7777721881866455, 0.46103358268737793, -6.691177845001221, -2.4971065521240234, -3.643841028213501, -5.115401268005371, -1.089347004890442, -3.6165518760681152, -1.8329559564590454, 2.13460636138916, 0.16962075233459473, 0.2855043411254883, 2.6411237716674805, 1.4785528182983398, 3.039303779602051, -3.6726951599121094, 0.19027328491210938, -2.9331703186035156, -1.0839335918426514, -7.121525287628174, -1.6261143684387207, 2.5140085220336914, -4.525848865509033, 1.3452560901641846, 2.518988609313965, -0.35359466075897217, -2.3028600215911865, -2.5026838779449463, -1.894107699394226, -0.4496943950653076, -0.2078721523284912, -0.13297688961029053, -3.82973051071167, 0.47466063499450684, -1.8698298931121826, -5.576670169830322, -1.8790977001190186, -2.265592098236084, -3.357379674911499, -2.1943275928497314, -1.6628466844558716, -3.748030662536621, 2.20489501953125, -4.361029148101807, -1.2156517505645752, -7.0671772956848145, -1.2103006839752197, 1.364997148513794, -4.164243698120117, -0.7372485399246216, 4.342560768127441, -1.874829649925232, 1.2637076377868652, -3.8780972957611084, -1.939918041229248, -4.744757175445557, -2.775163412094116, -1.852604866027832, -3.009718418121338, -4.951687812805176, -2.964353084564209, 0.31946635246276855, -3.2977051734924316, -3.412510395050049, -4.578244209289551, -0.8636132478713989, -3.5094587802886963, -0.22317910194396973, -4.3537445068359375, -1.7340483665466309, 0.6420912742614746, -3.9299826622009277, 3.781733512878418, -1.3077203035354614, -1.856679916381836, -0.11519980430603027, 2.0411224365234375, -6.7435221672058105, -6.203399181365967, -5.6386189460754395, 3.1248655319213867, -1.964804768562317, 0.9465980529785156, -2.7727088928222656, -0.30963134765625, -1.4729490280151367, 0.6280426979064941, -2.3921046257019043, -2.619086265563965, -4.668428897857666, 1.684887409210205, -4.331601142883301, 0.4681057929992676, -2.254652261734009, -8.380626678466797, -0.6852772235870361, -2.472573757171631, -1.7512540817260742, -6.197311878204346, -1.8783997297286987, -3.544806480407715, 0.8286640644073486, -7.658697128295898, -1.8898887634277344, -2.8723418712615967, 0.5021564960479736, -0.8688998222351074, -1.0823378562927246, 2.6849684715270996, -5.0714592933654785, -1.3048388957977295, -2.1365275382995605, -1.7156267166137695, -4.140592098236084, -3.1832001209259033, 0.00463414192199707, -4.69607400894165, -0.8820215463638306, 1.905261516571045, -2.4197170734405518, 0.2449357509613037, 0.749840259552002, -11.130453109741211, -4.0752034187316895, -6.762328147888184, -9.264131546020508, -0.16843843460083008, -4.870800971984863, -0.15774893760681152, -2.0967907905578613, 2.605203151702881, -4.46104621887207, -6.878300189971924, -2.1040329933166504, -3.1323225498199463, -2.6281561851501465, 0.15782523155212402, -4.229800701141357, -0.7801231145858765, 1.8565964698791504, -1.6882034540176392, -2.6819965839385986, 0.8007292747497559, -2.0941176414489746, 0.602994441986084, -1.6047618389129639, -0.10442495346069336, 0.7644855976104736, -1.8858497142791748, -4.172236442565918, -0.7338247299194336, -9.330819129943848, 0.765233039855957, -3.8622567653656006, 2.769071102142334, -1.8326228857040405, 2.7816367149353027, -2.07724928855896, -0.9614977836608887, -3.1706788539886475, -3.8113620281219482, -3.4587931632995605, -1.3464902639389038, -1.5483131408691406, 1.8981585502624512, 4.350276947021484, -2.685326099395752, 4.13775110244751, -1.1630678176879883, -2.5386481285095215, -0.926411509513855, -0.5389398336410522, -1.856215238571167, -8.414015769958496, -2.3036322593688965, 2.2022266387939453, 2.9883570671081543, -0.5990095138549805, -0.6906658411026001, 0.5612859725952148, 1.1528372764587402, -0.6217739582061768, -5.495279312133789, -4.1635212898254395, -0.6015584468841553, -2.5442910194396973, -2.0690505504608154, -1.259063482284546, -2.765185832977295, -5.162168979644775, -1.5483990907669067, 0.31848955154418945, 2.5510053634643555, -3.438365936279297, 1.4523429870605469, -0.3395419120788574, 3.502253532409668, -2.265561580657959, -1.4109077453613281, -2.756171941757202, 0.3996403217315674, 2.3282089233398438, -1.307633638381958, -2.467170238494873, -3.151975631713867, -2.481595993041992, -2.8381006717681885, 4.263793468475342, -3.285654067993164, -7.130191802978516, -7.031656265258789, -2.702143430709839, -1.5148248672485352, -0.7926517724990845, 3.7077083587646484, -3.946669101715088, -2.1003942489624023, 0.9855844974517822, -7.343509197235107, -0.4794175624847412, 1.6273393630981445, -0.03975403308868408, -7.328096866607666, -0.48136579990386963, -5.56256103515625, -2.765641212463379, -3.4395570755004883, 0.029880523681640625, 1.753431797027588, -3.66326904296875, -3.408871650695801, -2.142216444015503, -0.6824865341186523, 0.9252471923828125, -3.291327476501465, -0.09493601322174072, -4.34889030456543, 3.65910005569458, -3.195908546447754, 0.6159501075744629, 1.6950435638427734, 0.368119478225708, -7.587388515472412, -2.818103313446045, -1.5695747137069702, -5.551793575286865, -7.127762317657471, 2.660536766052246, 0.24178552627563477, -3.1519532203674316, -5.148426055908203, 2.1150903701782227, 0.7501430511474609, -7.883704662322998, 1.7550625801086426, -0.9647769927978516, -2.6601080894470215, 0.8180112838745117, -0.8877500295639038, 1.4380390644073486, -1.6778899431228638, -5.952352523803711, 3.7297158241271973, 4.7898125648498535, -5.827665328979492, 3.377969741821289, -2.7306180000305176, -0.8561419248580933, -9.440441131591797, -3.4202771186828613, -0.3102973699569702, -2.9171931743621826, 1.536475419998169, 0.30414581298828125, 1.6127429008483887, -3.800943374633789, -0.9104546308517456, -3.1530635356903076, 0.9603502750396729, -6.9169135093688965, -4.734921455383301, -7.0812153816223145, -2.9501423835754395, -0.17704153060913086, -2.3411026000976562, 5.7415008544921875, 3.5410985946655273, -4.5543928146362305, -2.815122127532959, -5.434335708618164, -5.828297138214111, -3.7182629108428955, -0.003966808319091797, 3.8105392456054688, -1.628102421760559, -0.9371237754821777, -2.714475631713867, -0.6758612394332886, -1.2999978065490723, -4.595113277435303, -2.963658332824707, -5.576317310333252, -5.123835563659668, -1.1201424598693848, 2.570333480834961, -1.8945386409759521, -6.555719375610352, -0.4186286926269531, 1.784985065460205, -0.704474687576294, 0.5321457386016846, -1.427781581878662, 3.4921374320983887, 0.41570496559143066, -2.990941047668457, -6.271785736083984, -2.5188210010528564, 0.38431525230407715, -1.7833372354507446, 0.8492019176483154, 0.8845911026000977, -1.013247013092041, 5.453629970550537, -5.249373435974121, -7.560213088989258, -3.4784274101257324, -2.3417770862579346, -1.7366961240768433, -4.367175102233887, -4.937229633331299, -2.3298959732055664, -4.515854358673096, -1.7495808601379395, 0.03668069839477539, 2.464718818664551, -5.708785057067871, -4.999877452850342, 1.2489519119262695, -7.524868488311768, -3.2385878562927246, -3.231205463409424, 3.3632946014404297, 1.532094955444336, -0.11456537246704102, -3.0523712635040283, -2.7550065517425537, -0.42972850799560547, 0.07602071762084961, 7.49662971496582, 0.009824037551879883, -3.8083014488220215, -4.577844619750977, -4.074305057525635, -3.6355974674224854, -5.721850872039795, 2.6570205688476562, -5.1717658042907715, -0.6234598159790039, -0.3830878734588623, -6.975851058959961, 2.5346217155456543, -1.6000466346740723, -3.1010355949401855, -1.2947486639022827, 3.3609580993652344, -1.4077014923095703, -1.6461856365203857, -4.880743026733398, -1.0470777750015259, -4.55211877822876, -2.2049031257629395, -7.634415626525879, 1.127194881439209, -5.278785228729248, -0.9068312644958496, 0.36248302459716797, -5.419478416442871, 0.3752312660217285, -5.469474792480469, -2.143698215484619, 1.5224549770355225, -1.3567219972610474, 4.501973628997803, -4.4103922843933105, -2.5807080268859863, 0.297534704208374, -0.7375081777572632, -5.771806716918945, -0.35804569721221924, -3.5337753295898438, -3.395778179168701, -2.418856620788574, -5.683886528015137, -0.6831178665161133, -3.627074718475342, -8.02243709564209, 3.494718551635742, 1.8536415100097656, -0.019803404808044434, 0.14796781539916992, -0.7314683198928833, -3.1344566345214844, -6.08953857421875, -2.1865718364715576, -1.30193030834198, -3.3814971446990967, -4.799057960510254, -8.143959999084473, -2.0530824661254883, -0.524409294128418, -4.752265930175781, -3.3688907623291016, -1.5962626934051514, 0.3374767303466797, 0.7386665344238281, -3.044585704803467, 0.5759406089782715, -1.9766383171081543, 0.8911004066467285, 1.8521828651428223, 1.329580545425415, -1.9418774843215942, -5.176069259643555, 2.084179401397705, -4.446388244628906, -0.7400459051132202, -1.4738681316375732, -5.687902450561523, 0.944739580154419, -2.526811361312866, -4.442781925201416, -1.1671316623687744, -1.0325268507003784, -7.438169002532959, -1.3161275386810303, 0.8907394409179688, -5.311073303222656, -1.1921859979629517, -0.5941760540008545, -3.3672842979431152, -1.704958200454712, -5.216490745544434, -8.627276420593262, 2.5849270820617676, -6.445971488952637, -3.2579517364501953, -1.7064299583435059, -1.4229943752288818, -4.679666519165039, 1.5673646926879883, -7.440122604370117, -1.5189838409423828, -2.019941568374634, -3.2014870643615723, -1.5920994281768799, -6.463965892791748, -3.6197335720062256, -3.0571107864379883, 0.13962411880493164, -3.219303607940674, 3.7501564025878906, -6.161159992218018, -2.1176037788391113, -0.9510371685028076, -0.28559672832489014, 1.5797858238220215, -3.449737310409546, -6.022205829620361, -4.686740398406982, -3.1841726303100586, -5.44648551940918, -0.8891738653182983, 0.7586226463317871, -3.1882898807525635, 1.7927045822143555, 0.43694376945495605, 1.5450084209442139, -5.613115310668945, 3.6868972778320312, -4.783641338348389, 0.1791691780090332, -1.6691274642944336, -3.787137508392334, -4.422483444213867, 3.2653841972351074, 1.2717981338500977, 0.5439107418060303, -3.100245237350464, -3.326962471008301, 0.32581329345703125, -5.271381855010986, -1.6678075790405273, -0.21739554405212402, -5.875309944152832, 1.8564443588256836, -3.4498109817504883, -2.104112148284912, -3.0886573791503906, -3.050518751144409, -3.908799886703491, -5.982706069946289, -1.2839018106460571, -0.7366640567779541, -8.063480377197266, -0.770034670829773, -1.879982590675354, 0.12788915634155273, -3.028109550476074, 1.7132644653320312, 0.8842463493347168, 2.4654970169067383, -1.1422626972198486, -2.2187156677246094, -0.9737316370010376, -4.511469841003418, -6.013075828552246, -3.4625184535980225, 2.0071678161621094, -0.34810078144073486, -4.633772850036621, -2.859121084213257, -1.3704348802566528, -0.1691187620162964, -4.290019512176514, 0.492551326751709, -4.245481491088867, -3.163696765899658, -2.775468349456787, -3.6990115642547607, -0.13491225242614746, -4.738630294799805, -3.3363678455352783, -2.938145875930786, 2.5807247161865234, 0.5682258605957031, -0.6902303695678711, -2.545264720916748, -0.6303212642669678, -3.7509725093841553, -2.245945930480957, -7.215781211853027, -0.5533140897750854, -2.5349650382995605, -7.770555019378662, -5.98646879196167, -7.450920581817627, 5.1554155349731445, -2.783073663711548, -2.0493509769439697, -3.328347682952881, -5.38048791885376, -3.711153984069824, -6.411879062652588, -4.048817157745361, -6.543184757232666, -4.178804397583008, -0.45268118381500244, -2.280817747116089, -0.9068197011947632, 0.05419588088989258, -5.458749771118164, -6.123270034790039, 2.0609893798828125, -2.057483196258545, -2.9382076263427734, -5.037311553955078, -1.7647438049316406, -1.4897263050079346, -2.8166141510009766, -1.0010225772857666, -0.7470881938934326, -2.1621510982513428, -4.763046741485596, 2.433238983154297, -0.2502572536468506, 2.141118049621582, 3.6676082611083984, 0.340620756149292, -3.9810352325439453, -1.7358367443084717, -0.6055370569229126, -4.172886371612549, -1.412421703338623, -1.9833325147628784, 0.35534095764160156, -2.3199524879455566, 4.297697067260742, 1.7619915008544922, -4.587915420532227, -1.2771061658859253, 0.8522322177886963, -2.5806820392608643, -1.537608027458191, -6.294335842132568, 4.4767303466796875, -2.1182332038879395, 1.4544944763183594, 0.3282196521759033, -4.963524341583252, -9.596309661865234, -2.8161678314208984, -0.8976545333862305, -2.032771348953247, -0.8083189725875854, 2.4497570991516113, -0.9624025821685791, -0.890594482421875, -1.2376042604446411, -5.774609565734863, -1.8644167184829712, 0.26180362701416016, -6.435471534729004, 0.5930249691009521, -2.779017925262451, 0.27045154571533203, -0.522179126739502, -0.09191691875457764, -3.2251219749450684, -6.674558639526367, 1.1788454055786133, 0.026686429977416992, -5.725859642028809, -3.7595536708831787, -2.7718024253845215, -2.403745412826538, -5.241952419281006, -0.1009986400604248, -3.504456043243408, 3.591646671295166, -2.397587776184082, -0.19599413871765137, -2.1837825775146484, 1.9273741245269775, 1.7035760879516602, -0.3208051919937134, -0.7084817886352539, -4.717251300811768, 1.2356438636779785, -6.96331787109375, -5.477525234222412, 2.4166998863220215, 0.22577714920043945, -1.7040528059005737, 0.4145078659057617, -0.46647143363952637, -5.3636794090271, -5.055522918701172, -1.6691348552703857, -2.9140310287475586, -1.5788767337799072, -2.8675148487091064, -6.068744659423828, -5.00316047668457, -2.2173454761505127, -5.362604141235352, -6.378464698791504, -2.909208297729492, -2.027092933654785, -0.7203019857406616, 1.0276052951812744, -3.8896288871765137, 0.6038408279418945, -1.4317512512207031, -3.637253999710083, -1.8631044626235962, 3.87673282623291, -3.62311053276062, -4.365840911865234, -0.05143129825592041, -0.5633549690246582, 2.3792433738708496, -6.275417327880859, -4.841942310333252, 2.2514166831970215, 3.6851396560668945, -0.07917451858520508, -6.1474928855896, -2.591287612915039, -1.482886791229248, -5.4725141525268555, -0.5300173759460449, 2.211118698120117, -0.0963815450668335, 1.310441017150879, 1.887777328491211, -1.9048216342926025, -4.4870381355285645, -2.0137345790863037, -2.103107213973999, -0.9961422681808472, -0.492753267288208, -1.3353090286254883, -1.6380819082260132, -5.556426525115967, -2.4114880561828613, -2.4712753295898438, -2.248321533203125, -5.979987621307373, -0.10136532783508301, -3.7762527465820312, -0.44286060333251953, -4.20176887512207, 4.429771423339844, -1.9212672710418701, -4.109391212463379, -5.046630859375, 0.21856451034545898, 0.11195802688598633, -6.358347415924072], [-2.624471426010132, -1.9672813415527344, -2.5000367164611816, -0.0430450439453125, -2.0357236862182617, -4.9625420570373535, -5.178347587585449, 0.89064621925354, -2.871210813522339, -2.9047112464904785, 0.021752119064331055, 1.7467560768127441, -4.443243980407715, -4.258502006530762, 0.12561583518981934, -5.5822272300720215, -2.014780044555664, -1.6913774013519287, -2.547880172729492, 4.500556945800781, -2.062518358230591, -0.18422424793243408, -5.836927890777588, 1.231109857559204, -1.3706978559494019, -2.5254456996917725, 0.46448373794555664, -4.251277923583984, -1.5360842943191528, -2.9513440132141113, 0.1922919750213623, 6.428228378295898, -3.1749472618103027, -6.465703010559082, -0.9471133947372437, -2.715269088745117, -0.40451622009277344, 3.021458148956299, -0.8194141387939453, -1.2343826293945312, -0.6240013837814331, -0.6023954153060913, -9.196388244628906, -2.534142017364502, -2.442556619644165, 0.5327482223510742, -5.294512748718262, -4.551961898803711, -2.6068639755249023, -4.341733455657959, -2.718108892440796, 0.3280167579650879, -1.3816137313842773, 0.029903173446655273, -1.6150530576705933, -8.444613456726074, -5.483495235443115, -4.635458946228027, -4.687743186950684, 1.8131320476531982, -1.5567586421966553, -2.4419679641723633, -1.5522834062576294, -2.5857653617858887, -5.332594871520996, -0.5163788795471191, -0.7141139507293701, -4.18649959564209, -5.164695739746094, -2.843745231628418, 1.7979516983032227, -1.206459879875183, -5.888797283172607, 2.6751108169555664, 6.615752220153809, 0.056546926498413086, -2.638162612915039, -4.720464706420898, 4.010312080383301, 0.12418460845947266, -2.5994582176208496, -2.83624267578125, 1.1352777481079102, -6.99525260925293, 3.06839656829834, 1.5731937885284424, -0.677180290222168, 0.9474425315856934, 0.5945825576782227, -2.5456697940826416, 2.3104166984558105, 2.807267189025879, -5.154940605163574, 0.6717360019683838, 0.4361844062805176, 2.7732486724853516, 0.4229164123535156, -2.9175422191619873, -1.3209843635559082, 4.311252593994141, -3.537667751312256, -4.685492515563965, -7.517939567565918, -5.554534912109375, 2.264376640319824, -5.111658096313477, -2.4566149711608887, -2.734187126159668, -3.823202133178711, -2.028085231781006, -4.877162456512451, -1.7410449981689453, 0.90159010887146, -4.369445323944092, -4.277420520782471, -5.474517345428467, -6.882946014404297, 2.37912654876709, -2.5489258766174316, -5.098511695861816, -4.79190731048584, -5.817845821380615, -5.04341983795166, -1.9432709217071533, -2.754589557647705, 2.273787021636963, 1.0916872024536133, -3.3606646060943604, -0.5633257627487183, 0.30984997749328613, -4.3104400634765625, 1.54840087890625, -1.2602250576019287, -5.111325740814209, -4.285016059875488, -2.660989284515381, -6.54154634475708, 4.613361358642578, 1.2407758235931396, -0.05388021469116211, 0.022605419158935547, -1.3888545036315918, 2.928740978240967, 6.198529243469238, 0.1733253002166748, -5.562267780303955, 0.6230430603027344, -4.4258131980896, -3.178623676300049, -2.785715341567993, -2.0565929412841797, -0.8546452522277832, -5.696078777313232, -0.3944699764251709, -2.3720409870147705, -0.7324464321136475, 2.1242151260375977, 0.3122565746307373, 0.840080738067627, 0.5505106449127197, -5.009687900543213, -6.713644981384277, -6.441072940826416, -1.8680952787399292, 0.025488853454589844, -2.583890914916992, -7.006180286407471, -8.330266952514648, -0.8164569139480591, -2.2989602088928223, -0.8212169408798218, -2.582064390182495, -6.830399513244629, -1.609161138534546, -6.90894889831543, -5.520051002502441, 0.870455265045166, -8.842781066894531, -4.690479755401611, -1.3047168254852295, -0.4256601333618164, -0.40040159225463867, -1.914237141609192, 0.012219905853271484, 3.897064208984375, -0.89463210105896, -1.1608494520187378, -4.131190776824951, 2.343369960784912, -3.8835113048553467, -3.8226418495178223, -1.7968533039093018, -3.59285831451416, 2.7370314598083496, 0.5907526016235352, -2.2444522380828857, 4.807505130767822, -6.814427375793457, 2.671687602996826, -0.07408857345581055, -7.60888147354126, -2.6190860271453857, -2.1154978275299072, -4.682423114776611, -3.945430278778076, 0.7399377822875977, -2.2220458984375, -0.21021175384521484, -1.2838135957717896, -4.651643753051758, -4.6673903465271, -4.826679229736328, -3.6307575702667236, 0.03284740447998047, -5.782061576843262, 1.4922442436218262, -0.1261153221130371, -1.5290346145629883, 0.9169983863830566, -4.47566032409668, -4.908685684204102, -7.563674449920654, 3.049348831176758, 0.43825793266296387, -3.2608487606048584, 0.49722862243652344, -4.9899797439575195, -2.0085606575012207, -2.819685935974121, -5.843345642089844, -2.0010671615600586, -4.922186851501465, -8.715532302856445, -0.6525274515151978, -2.5933804512023926, -3.1659066677093506, -0.38329362869262695, -6.626674175262451, -2.7352051734924316, -0.3526771068572998, -0.16945421695709229, -5.078444004058838, 1.075390338897705, -0.24158048629760742, -1.3070528507232666, 1.1698293685913086, 0.544921875, -6.443406105041504, -2.234290599822998, -4.19325590133667, -0.5012370347976685, -0.9830142259597778, -2.530747413635254, -2.3229498863220215, -6.136373519897461, -0.916926383972168, -0.1834113597869873, -2.0123326778411865, -6.7138166427612305, 1.8484559059143066, -3.4560647010803223, 1.5214264392852783, -9.401850700378418, -3.808817148208618, -4.409621715545654, -0.728931188583374, -3.705634355545044, -0.9302636384963989, -5.533329010009766, 0.4527599811553955, -4.50595235824585, -2.003345251083374, 3.818902015686035, -2.5420925617218018, -3.0062761306762695, -3.337472915649414, 4.094037055969238, -2.7895612716674805, -2.7936501502990723, -2.5433168411254883, -4.853795051574707, -2.9517602920532227, -2.9607133865356445, -0.6645299196243286, -6.312721252441406, -6.155795574188232, -1.8667488098144531, 1.2759778499603271, -2.7976832389831543, 2.173572063446045, -1.3357220888137817, -2.498495578765869, -1.6671099662780762, -0.35203611850738525, -5.157948970794678, -2.286874294281006, -1.3789284229278564, 1.1140623092651367, 0.5508372783660889, -3.0843539237976074, -0.34833693504333496, -4.823737144470215, -4.862004280090332, -9.20263385772705, -0.666202187538147, -4.520328521728516, -4.9590349197387695, 1.1756510734558105, -5.688011646270752, -1.9017049074172974, -0.3456261157989502, -0.36842989921569824, -4.612199306488037, -0.23538494110107422, 3.307830333709717, -3.387951612472534, -4.713062763214111, 0.8760664463043213, -7.5341997146606445, -2.6502721309661865, -1.590986728668213, -3.406989812850952, 0.5052552223205566, -2.6654438972473145, 3.951023578643799, -5.934349060058594, 0.0888667106628418, -2.0694730281829834, -3.3761563301086426, 0.6835043430328369, -2.7518036365509033, -4.268526077270508, -0.016061782836914062, -5.083695411682129, -2.4413418769836426, -1.7869378328323364, -1.1602858304977417, -4.023157119750977, -2.0534465312957764, -1.6702399253845215, -1.81015145778656, -3.3666138648986816, -2.33056640625, -3.9429545402526855, -1.637308120727539, -0.8495721817016602, 2.0850725173950195, -2.63956356048584, -6.760266304016113, -1.013208270072937, -4.977160453796387, -2.388394832611084, -3.4035863876342773, -4.334201812744141, -3.1652369499206543, 1.261976718902588, -4.2757158279418945, -2.823235034942627, -2.398346424102783, -5.661957740783691, -3.330730438232422, -2.3197150230407715, -3.678774833679199, 1.54958176612854, -7.5166425704956055, 6.810484886169434, -2.156733274459839, -6.654493808746338, -8.655847549438477, -7.344852924346924, -0.6866832971572876, 0.34374165534973145, -2.362715005874634, -0.5752605199813843, -4.524466514587402, -5.842714786529541, -0.9787135124206543, -3.576565980911255, -0.7380375862121582, -2.742354393005371, 1.4761524200439453, -5.484648704528809, -6.252188205718994, 0.39268016815185547, -3.386948585510254, -3.9637413024902344, -0.6093028783798218, -6.068835735321045, -3.921746253967285, -6.920710563659668, -0.21358847618103027, -0.04282712936401367, -4.944022178649902, -4.009938716888428, -0.4471554756164551, 0.5988187789916992, 0.7766718864440918, -7.224405288696289, 0.16579341888427734, -2.4383738040924072, -4.578930377960205, -0.48499584197998047, -3.9793434143066406, -2.6984870433807373, -4.12338924407959, -2.446798086166382, -8.275774002075195, 2.481314182281494, 0.5111453533172607, -3.0965685844421387, -1.199610948562622, -3.377017021179199, -2.15852427482605, -2.9372897148132324, 2.040982246398926, 0.9721066951751709, -6.862382888793945, 1.1496162414550781, 3.333632469177246, -4.347542762756348, -0.5041036605834961, 1.3464465141296387, -0.6579504013061523, -4.677127838134766, 0.7668516635894775, -8.862458229064941, 1.8466033935546875, 0.02234196662902832, 7.290727615356445, -4.847248554229736, -4.933746814727783, -6.621502876281738, -2.6656832695007324, -3.4998788833618164, -5.455925941467285, -9.34259033203125, -0.9639556407928467, -2.1947507858276367, -4.15792989730835, -6.859054088592529, -3.5882277488708496, -4.488969802856445, -1.0387415885925293, -4.618288040161133, -3.2775983810424805, -0.33331799507141113, -7.370927333831787, -2.3149139881134033, 1.3589134216308594, 4.530154228210449, -4.67757511138916, 0.24787116050720215, 0.7068307399749756, -5.877263069152832, -1.6497923135757446, 3.443418502807617, 2.118337631225586, -4.590246200561523, -10.980249404907227, -2.062957286834717, -3.915560007095337, -7.933868408203125, -5.439419746398926, -3.738494634628296, -5.395258903503418, -2.924896717071533, -3.419602155685425, 1.0971627235412598, -1.4745655059814453, -0.34633326530456543, 2.5737104415893555, 3.6076412200927734, -6.045960903167725, -0.6360856294631958, -2.3848962783813477, -1.1522879600524902, 6.017760276794434, 1.5814785957336426, 1.9566526412963867, 2.0304927825927734, -0.2834463119506836, 0.8791084289550781, -2.3283803462982178, -4.1588311195373535, -5.046847343444824, 2.273707389831543, 3.6422104835510254, -0.7255202531814575, -2.3749232292175293, -2.451690196990967, 3.861024856567383, -3.361959457397461, 1.4808721542358398, -2.9258880615234375, -3.3278088569641113, 1.4684417247772217, -4.421412944793701, -8.341057777404785, 0.5602242946624756, -3.4392614364624023, 1.6092298030853271, -1.1312280893325806, 0.8317115306854248, -2.3539717197418213, 1.818925380706787, 1.9998080730438232, 0.40054893493652344, -2.700761318206787, 0.5091314315795898, -0.1218254566192627, -6.301274299621582, -2.8184776306152344, -0.6812275648117065, -2.732997179031372, -5.8122968673706055, 1.1928372383117676, -0.4106454849243164, 0.04927968978881836, -6.313140392303467, 0.535675048828125, 2.2690439224243164, -3.1810128688812256, -1.583503007888794, -0.9384341239929199, 0.23593759536743164, 0.2949862480163574, -2.5031561851501465, -1.5696625709533691, -5.895226001739502, -4.042108535766602, -1.043259859085083, -3.504215955734253, -1.2227485179901123, -2.2717761993408203, 1.7444062232971191, -5.804973125457764, -5.167925834655762, 2.2018609046936035, -5.90892219543457, -6.066028594970703, -4.032652854919434, -4.169964790344238, 3.910677909851074, 0.9345071315765381, 1.9363632202148438, -2.9879536628723145, -1.181617021560669, 5.033663749694824, 0.6135625839233398, -1.5073351860046387, -3.283918857574463, 0.9403867721557617, 0.6000499725341797, -2.428039312362671, -8.10943603515625, -4.821719646453857, -5.8686394691467285, -3.774064302444458, 2.8708481788635254, -3.7544362545013428, -0.07309365272521973, -5.35860538482666, -1.22469162940979, 0.4773876667022705, 0.765535831451416, -2.517425537109375, 1.5797295570373535, -4.170686721801758, -1.9023075103759766, -7.746478080749512, -3.026566743850708, 2.557408332824707, 0.05150127410888672, -2.1318609714508057, -1.8594268560409546, -4.688067436218262, -4.32771110534668, 3.699735164642334, -6.13277530670166, 1.6007399559020996, -0.04159581661224365, -0.5085445642471313, -2.1992974281311035, -1.302551031112671, -1.9211039543151855, 1.9608454704284668, -2.5532498359680176, -0.9030953645706177, -0.10727047920227051, -0.4539012908935547, -0.2959728240966797, -7.39284610748291, 0.9992794990539551, -4.425548553466797, -5.567622184753418, 6.706674575805664, -3.9270238876342773, 3.0191688537597656, -1.0403181314468384, -6.433606147766113, -5.551645755767822, -2.380826234817505, -8.27534294128418, 1.6601438522338867, -2.6774704456329346, -4.27059268951416, -2.486508846282959, -1.2024784088134766, 0.4685025215148926, -0.17741143703460693, 0.751185417175293, -1.8602674007415771, -7.122508525848389, -5.091978073120117, -1.2162612676620483, -2.2186501026153564, -3.955454111099243, -3.393209934234619, -2.2514257431030273, -3.3019943237304688, 0.02837848663330078, 1.7439775466918945, -2.1980490684509277, 0.38489246368408203, -5.096484661102295, 1.56760835647583, -1.8117460012435913, -0.5951312780380249, 2.2083091735839844, -10.594541549682617, -7.691099166870117, 1.4530415534973145, 0.9566636085510254, -1.3836936950683594, -2.069699764251709, -2.1707959175109863, -3.9401345252990723, 0.14138293266296387, -5.489248752593994, -5.882513046264648, -3.7672948837280273, -3.28739595413208, -0.3504652976989746, -0.7473095655441284, -0.23392868041992188, -2.8903908729553223, -2.3326916694641113, -3.16420841217041, -2.369426965713501, -1.4407055377960205, -3.568443775177002, -5.861577987670898, -0.23690128326416016, -0.8906030654907227, -0.9591292142868042, -3.168348789215088, -1.5976197719573975, -3.349844455718994, -2.7912943363189697, -3.8307604789733887, -0.1282668113708496, -3.8559951782226562, -5.889232158660889, -0.8612546920776367, -3.180738687515259, -2.7830166816711426, 0.11265277862548828, -0.47419118881225586, 0.4262268543243408, -3.3186113834381104, -7.899004936218262, -1.4528629779815674, -3.934467315673828, 2.09311580657959, -1.3310198783874512, -7.8105998039245605, 1.9565582275390625, 0.6660265922546387, -0.9438153505325317, -4.0898847579956055, -0.3193296194076538, 6.735749244689941, 1.0813124179840088, -0.6548731327056885, -2.6553218364715576, -0.46674394607543945, -5.916147232055664, -3.0953001976013184, -2.960348606109619, -0.2770726680755615, -1.749698281288147, 0.03512740135192871, 3.0791211128234863, -3.8507747650146484, 0.17550086975097656, -5.064149856567383, 2.096731185913086, 4.6745991706848145, -0.9588745832443237, -0.6043689250946045, -3.852985382080078, -0.47860145568847656, -0.8968759775161743, -6.684752464294434, -1.557612657546997, -1.423192024230957, -1.8717148303985596, 2.0095434188842773, 1.0381865501403809, -6.470368385314941, -3.863220691680908, -5.024111270904541, -2.009688138961792, 3.8141074180603027, -3.0086655616760254, 1.6524243354797363, -5.397017478942871, 2.5695815086364746, -6.289785861968994, -2.8290963172912598, -0.8637968301773071, -4.554410934448242, -3.9765634536743164, -0.651702880859375, -3.6019392013549805, -0.018334627151489258, -1.438551664352417, -2.660459041595459, -3.9441938400268555, -3.758280038833618, -2.781285285949707, -2.1246349811553955, -1.6351276636123657, -2.775892734527588, -2.967315435409546, -2.1117401123046875, -0.5403463840484619, -2.0270936489105225, -0.21481561660766602, -4.0591888427734375, -1.7784595489501953, -4.236618995666504, -2.069899082183838, 0.6823372840881348, 1.0297322273254395, 0.05713033676147461, -3.1262621879577637, -0.5096333026885986, 2.641120433807373, -6.37448787689209, 3.571995735168457, 0.5916392803192139, -8.843825340270996, -1.8595367670059204, -0.8587788343429565, -3.5953445434570312, -0.6220386028289795, -0.678413987159729, -0.8193801641464233, -5.01025390625, -0.397976279258728, 2.7881555557250977, 1.570249080657959, -3.999577760696411, 1.9890995025634766, -5.587033271789551, -1.8328617811203003, 5.710077285766602, -1.2936018705368042, 0.7307529449462891, -1.421248435974121, -1.5459128618240356, -0.5340981483459473, 1.5186655521392822, -1.6119977235794067, -3.6370949745178223, 3.4644389152526855, 3.0325794219970703, -1.0064036846160889, 1.2306108474731445, 0.5712404251098633, -9.234001159667969, -4.675205707550049, -5.703094482421875, -4.089936256408691, -0.2121584415435791, -3.4735352993011475, 1.3572330474853516, -0.506162166595459, -2.6598362922668457, -2.900057315826416, 0.11181855201721191, -8.06611156463623, -2.603127956390381, -4.277161598205566, -3.7910513877868652, 0.7538270950317383, -5.158371448516846, -5.148867130279541, 1.2349121570587158, -2.9320130348205566, -0.6175076961517334, -3.6713833808898926, -4.15463924407959, -5.422605514526367, -2.1675026416778564, 2.467271327972412, -2.676511287689209, 0.34130334854125977, 2.22047758102417, -5.564907073974609, -1.0957714319229126, 1.3156075477600098, -5.699016094207764, -0.6259498596191406, -3.0593812465667725, 0.36162900924682617, -6.806121349334717, 3.5939788818359375, 0.6219677925109863, -6.112884521484375, -0.16419768333435059, -1.2719683647155762, -0.38315296173095703, -2.576435089111328, -7.542923927307129, -5.06594705581665, -0.7400304079055786, 3.7642221450805664, -1.207505702972412, -2.4133245944976807, -2.409353017807007, -3.798900604248047, -6.732679843902588, -4.988679885864258, 0.5988497734069824, -4.489213466644287, 2.4045629501342773, -1.2151293754577637, 1.8800039291381836, -3.3391671180725098, 0.03883004188537598, -3.890721082687378, -5.013749122619629, -6.652946472167969, 0.9128890037536621, -3.654188394546509, 1.1265766620635986, -6.948886871337891, -3.641023635864258, -2.5826494693756104, -2.3511037826538086, -2.558161735534668, 1.087465763092041, -3.8931334018707275, -0.57768714427948, -1.5644659996032715, 3.293710231781006, 0.8713090419769287, -0.5796934366226196, -1.6340763568878174, -1.9865838289260864, -1.2884666919708252, -2.446326732635498, -1.6008954048156738, 7.871533393859863, 4.124375343322754, -2.4390974044799805, -3.1719884872436523, -5.2111711502075195, -11.380483627319336, -6.601248264312744, 1.0205497741699219, -0.6098986864089966, -0.8304527997970581, -1.9617570638656616, -3.7767179012298584, -0.7648353576660156, 1.8405094146728516, 4.76763916015625, 3.670832633972168, -0.8257119655609131, -5.212615966796875, -3.4623312950134277, -1.7713565826416016, 0.13633108139038086, 0.6678807735443115, -3.479788303375244, -9.279459953308105, -2.6097970008850098, -2.948394298553467, -4.92803955078125, -4.952911376953125, -0.9384411573410034, -7.277218818664551, 1.1377825736999512, -1.909604787826538, -4.082395553588867, -0.930864691734314, -5.1174187660217285, -4.054318428039551, 0.5280551910400391, -5.2321696281433105, -0.9127418994903564, -5.195146560668945, -0.2336268424987793, -4.219476699829102, -9.272575378417969, -7.1006879806518555, -3.6672279834747314, -1.0272188186645508, -0.3824115991592407, -4.403703689575195, -7.145257472991943, 0.29170846939086914, -5.498575210571289, -1.4213767051696777, -1.6393733024597168, -3.3078525066375732, -5.145035743713379, -3.4075944423675537, -2.174668073654175, -1.0922292470932007, -3.386462926864624, -1.9769697189331055, -2.0114247798919678, -6.5363359451293945, 0.8100504875183105, 2.2542662620544434, 3.7389302253723145, -0.23162412643432617, -0.4081608057022095, 0.08168935775756836, -2.3579111099243164, -2.233534812927246, -1.418879508972168, -0.09350550174713135, -0.5509344339370728, 1.6322965621948242, -2.5877280235290527, -6.053934574127197, -2.8657760620117188, -5.6659440994262695, -1.4405426979064941, -0.1939537525177002, 0.5660161972045898, -2.8285741806030273, -5.8376336097717285, 0.5745596885681152, 0.6141538619995117, -0.8005542755126953, 2.1004133224487305, -4.679551601409912, 4.420490264892578, 0.2721676826477051, -5.3280253410339355, -4.2095255851745605, -3.688363552093506, -5.088563919067383, -7.42868709564209, 0.059839725494384766, -5.413285255432129, 0.6521382331848145, -3.042055606842041, -1.762458086013794, -0.8099217414855957, 0.38298916816711426, -3.3839592933654785, -0.3871297836303711, -2.6124024391174316, 1.8088769912719727, -5.145600318908691, -3.1406612396240234, -0.4490928649902344, -2.4472246170043945, -1.55469810962677, -0.7381625175476074, -3.4587838649749756, -2.195113182067871, -1.5714503526687622, -3.476776361465454, 1.151104211807251, -1.5337519645690918, -5.59464168548584, 1.5934219360351562, 3.138885974884033, -5.627326965332031, -2.8579230308532715, -0.6392961740493774, -4.05866003036499, -5.581197261810303, -5.7952189445495605, -6.133658409118652, -2.3299312591552734, -4.413626670837402, 4.441507339477539, 1.247675895690918, -2.3273820877075195, -1.9594385623931885, 2.574779987335205, -3.0988080501556396, 4.226144790649414, 4.924348831176758, 3.898077964782715, 2.1435723304748535, -1.4514590501785278, -3.6087050437927246, -2.342463493347168, -3.56394624710083, -0.05498695373535156, -6.03410530090332, -6.612737655639648, -1.1698625087738037, 0.1651747226715088, -0.4003136157989502, -0.7569842338562012, 0.5923557281494141, -0.008917927742004395, -1.3843348026275635, 0.5114004611968994, -5.297250270843506, -4.197890281677246, -4.516101360321045, -1.9893378019332886, -5.502774238586426, 4.707874298095703, -6.723265647888184, -0.4916496276855469, 1.6748645305633545, -0.6854666471481323, -1.3925302028656006, -2.577216863632202, -1.3561584949493408, -2.5317561626434326, -0.08007621765136719, -1.4781460762023926, -1.8091404438018799, -4.051845073699951, -1.7081129550933838, 3.4018149375915527, -5.245908260345459, -3.097881317138672, -4.11421012878418, -1.4093444347381592, -4.8356733322143555, -3.2578463554382324, -3.4030442237854004, 2.681485652923584, 3.540769577026367, -4.893150329589844, -1.355081558227539, 2.6347503662109375, -1.715259313583374, -0.42992615699768066, -2.4846415519714355, -2.2514524459838867, -5.253002166748047, -1.1920312643051147, -3.6139705181121826, -3.382948398590088, -3.6991348266601562, -4.0150604248046875, 1.7020392417907715, -1.5981286764144897, -3.3973639011383057, -0.4929962158203125, 0.23309111595153809, -5.244681358337402, 5.619217395782471, 1.3505220413208008, -6.4076080322265625, -0.970711350440979, -7.256832599639893, -5.129220008850098, -6.360365867614746, -3.101013422012329, -3.315375804901123, -7.5222883224487305, -2.11987042427063, -3.3465404510498047, 1.618565320968628, -3.454995632171631, -7.409223556518555, -0.7294352054595947, -2.083507776260376, -1.4545146226882935, -0.9790083169937134, -3.5848703384399414, 0.6239547729492188, -7.983272075653076, 0.7760868072509766, 1.8477602005004883, -4.409093379974365, 0.27457213401794434, 1.04095458984375, 3.1789588928222656, 0.9233498573303223, -6.913888454437256, 1.1912071704864502, -0.08623361587524414, -2.7580714225769043, -6.811489105224609, -2.9899795055389404, 6.804117202758789, -2.3511953353881836, 1.2370929718017578, -4.740050792694092, -4.120181083679199, -4.899189472198486, 0.26412296295166016, 0.04889822006225586, -2.360182523727417, 0.6840305328369141, -9.95158576965332, -3.9183030128479004, -4.828624725341797, -1.4851651191711426, -7.274311065673828, -3.1186330318450928, 2.936802864074707, 1.4881830215454102, -0.32280635833740234, -0.9923049211502075, -2.1234617233276367, -2.5979807376861572, 1.125504493713379, 4.331911563873291, -11.224416732788086, -1.479873776435852, -3.004542827606201, -10.43693733215332, 0.9485366344451904, -3.452535629272461, 0.12324309349060059, 5.779745101928711, -4.401418209075928, 1.7980716228485107, -0.16576552391052246, -0.46178317070007324, -1.6308021545410156, 1.262439250946045, 0.8019094467163086, -2.1692442893981934, -2.3389530181884766, -5.874077796936035, -0.47478199005126953, -2.874129056930542, -6.402048110961914, -1.6023681163787842, 0.28965091705322266, -1.4766013622283936, -4.2816314697265625, -4.013914585113525, 0.3416249752044678, -1.9672542810440063, -7.018278121948242, -0.0268557071685791, -3.8842082023620605, -4.734701156616211, -1.028538465499878, -0.0457911491394043, -0.008578062057495117, -1.8985663652420044, -3.66046404838562, 0.12628602981567383, -5.205617904663086, -3.06850004196167, -5.555899143218994, 2.8742408752441406, 2.8529582023620605, -4.548506736755371, 1.8825020790100098, -2.7619271278381348, -4.463625907897949, -2.8687572479248047, -4.807835578918457, -3.3744170665740967, -1.9626469612121582, -5.8931050300598145, 2.659634590148926, -3.6233983039855957, -0.44351911544799805, -1.3160529136657715, -2.232306957244873, 0.7838656902313232, -4.119780540466309, 1.8642253875732422, -1.097854733467102, 4.829900741577148, 1.9040913581848145, -1.4658441543579102, -4.827686309814453, -7.405693054199219, -1.430164098739624, -4.686558723449707, -10.904741287231445, 0.9497525691986084, -3.0015320777893066, 1.1141595840454102, -3.5312588214874268, -2.3154518604278564, -2.7219345569610596, 1.1508381366729736, -3.7926247119903564, -2.0945863723754883, 0.2740778923034668, -0.41631436347961426, -2.8138632774353027, -1.0718086957931519, -2.682262420654297, -2.1410789489746094, -7.021909236907959, -6.7076287269592285, -6.3241119384765625, -6.319797515869141, -0.8973721265792847, 2.3162155151367188, -3.4009180068969727, -0.021109342575073242, -0.8227179050445557, -3.9131906032562256, -2.1141765117645264, -3.7559831142425537, 2.6794228553771973, -4.116961479187012, -3.6403415203094482, -1.606202244758606, -0.37657225131988525, -3.423445701599121, -4.587736129760742, -0.9372431039810181, 0.601806640625, -3.3410770893096924, 4.11544132232666, -2.578007221221924, -3.1707820892333984, -3.0505292415618896, -6.443687438964844, -1.923067331314087, -3.7362241744995117, 0.039203643798828125, -1.9676982164382935, -4.308736324310303, -1.5928022861480713, -1.226791501045227, 0.11444854736328125, -2.1328723430633545, -5.714413642883301, -1.2967464923858643, -1.2336132526397705, -0.7043403387069702, -2.5518624782562256, -3.0325708389282227, 1.946601390838623, -1.78018057346344, 3.1748428344726562, -0.5928124189376831, -1.5935057401657104, 0.5894887447357178, 1.34759521484375, 0.3504486083984375, -4.75967264175415, -3.3024730682373047, -4.181031227111816, 0.6485943794250488, -3.670450210571289, -0.18012619018554688, -0.033246755599975586, -0.7546482086181641, -0.6653342247009277, -5.672876834869385, -5.946348667144775, 4.239345550537109, -2.954838275909424, -5.7201995849609375, -0.3953312635421753, 3.423105239868164, -2.6141884326934814, -3.90875506401062, -5.156200885772705, -1.8109419345855713, -1.2324211597442627, -7.274996280670166, -2.8901827335357666, -2.4450039863586426, -0.46132755279541016, -4.62467622756958, -4.60755729675293, -0.04002368450164795, -1.2850927114486694, -1.7076208591461182, 1.7060551643371582, -0.6045799255371094, -2.3616130352020264, -1.6811070442199707, -3.5743799209594727, 1.7446849346160889, -0.8561807870864868, -5.657473564147949, -1.9148800373077393, 2.0680971145629883, 4.725863456726074, -7.419898986816406, -0.6529086828231812, -3.8132214546203613, 0.33287477493286133, -1.2156753540039062, -10.172918319702148, -1.6795158386230469, -7.58745813369751, -10.201011657714844, -4.108911514282227, 0.36549830436706543, 2.588198661804199, 5.85352897644043, 0.4999198913574219, -4.239964008331299, -2.3211865425109863, -5.64693021774292, -0.9638972282409668, -8.055659294128418, 0.543114185333252, -1.1321114301681519, -6.469503402709961, -0.5952179431915283, -0.11353981494903564, -6.239966869354248, 2.69986629486084, -3.723675012588501, -2.4708876609802246, -1.5557581186294556, 1.3980720043182373, -2.7297215461730957, -1.700857162475586, -6.515307903289795, -8.368858337402344, -0.37691986560821533, 2.065715789794922, -2.2164011001586914, -2.059720754623413, -3.138000726699829, -0.283829927444458, 0.8869614601135254, -5.873544692993164, -1.619060754776001, -4.3071441650390625, -1.9227774143218994, 1.5441436767578125, -0.46708643436431885, -5.099936485290527, 2.5433053970336914, 0.08355975151062012, 0.7057127952575684, -2.7102150917053223, -3.970698118209839, -5.702721118927002, 0.6780891418457031, -6.003747463226318, -2.5716488361358643, 0.6163070201873779, -5.470576286315918, 1.9141626358032227, -0.15829193592071533, -0.24975228309631348, 3.139651298522949, -0.36629819869995117, -4.7172698974609375, 1.5751678943634033, -1.2382278442382812, -8.582674026489258, -0.05502581596374512, -5.231066703796387, -2.831428050994873, -1.5151296854019165, -5.019674777984619, -1.0036289691925049, -1.221992015838623, -2.0271108150482178, -4.696622371673584, -3.108980655670166, -0.6688230037689209, -1.9708455801010132, -0.3758840560913086, -3.486706256866455, -6.4893035888671875, -1.9208418130874634, 2.534121513366699, -0.9115543365478516, 0.7902402877807617, -4.20731258392334, -3.481541156768799, -7.31631326675415, -2.832578182220459, -1.5430798530578613, 2.35738468170166, -1.3223642110824585, -2.823805570602417, -0.645296573638916, -6.080697536468506, -5.755921363830566, 5.302864074707031, -4.005259990692139, 0.019459962844848633, -7.521770477294922, -2.2408459186553955, -3.288912773132324, 0.7191252708435059, -2.1093764305114746, -6.048336029052734, 3.8292269706726074, -1.925224781036377, 0.30326151847839355, -1.5204837322235107, -2.6962857246398926, 0.8610093593597412, -1.704227328300476, -1.4871084690093994, -1.4068996906280518, -0.5141323804855347, -0.5515539646148682, -1.1961159706115723, -2.9806675910949707, 3.7280426025390625, -3.7006945610046387, -1.0882989168167114, -4.570652484893799, -4.298469543457031, 1.6790575981140137, 1.2518196105957031, -0.9253737926483154, -0.7164831161499023, -6.335789680480957, -2.186032295227051, -2.4794905185699463, -3.0106043815612793, -2.580122470855713, -3.5906870365142822, -5.825879096984863, 2.983142852783203, 2.0060086250305176, -0.32516181468963623, -3.8810391426086426, -0.6159522533416748, -5.421672344207764, -1.9873801469802856, -4.044340133666992, -3.296060085296631, -0.7915124893188477, -5.238761901855469, -8.029702186584473, 0.30561399459838867, 5.519140243530273, -4.580479621887207, -3.427253246307373, -1.1242649555206299, 0.7303802967071533, -3.264960289001465, 0.30631184577941895, -1.5868042707443237, -2.6284985542297363, -3.652233123779297, -4.726065635681152, -0.6746343374252319, -7.594481468200684, -3.8822977542877197, -0.7645066976547241, -0.15141832828521729, -6.999462127685547, 0.2956109046936035, 0.030234336853027344, -7.093755722045898, 2.0913305282592773, -6.422918319702148, 3.078646659851074, -9.161017417907715, -2.2349584102630615, -1.0407283306121826, -4.432894706726074, -4.259787559509277, -4.019192695617676, -3.621738910675049, -4.5129499435424805, -0.4093902111053467, 0.5790653228759766, 0.52303147315979, 2.8750295639038086, 3.7150111198425293, -5.209244728088379, -3.470914125442505, 0.6151020526885986, 5.62950325012207, -4.064470291137695, -1.9130834341049194, -6.856476783752441, -1.3774067163467407, -6.999136447906494, -1.4521886110305786, -2.8494787216186523, -2.6257035732269287, -4.645193576812744, -9.236907005310059, 1.4431931972503662, -0.5009500980377197, -4.981209754943848, -1.2325797080993652, -6.387054920196533, -6.324223041534424, -1.2954161167144775, -4.308524131774902, -4.98481559753418, -6.331252574920654, -5.053517818450928, -3.566922187805176, -0.39513421058654785, -6.294031143188477, 2.057404041290283, 0.8153331279754639, -1.0419731140136719, -3.9558262825012207, -3.8240551948547363, -4.805428504943848, -4.829764366149902, -0.9449487924575806, -5.327896595001221, -0.8185645341873169, -6.885539531707764, -0.08747947216033936, -6.524700164794922, 1.700547695159912, -7.601830959320068, 0.11346435546875, -1.2191741466522217, -4.33231782913208, -2.9312939643859863, 0.20976543426513672, -0.28885531425476074, -5.717775344848633, -9.02939224243164, -1.6275808811187744, -9.74953842163086, -0.16029071807861328, -2.6503872871398926, 0.7362399101257324, -1.8966755867004395, -6.047578811645508, -6.014772415161133, -1.1121625900268555, -0.7355571985244751, 1.753629207611084, -3.5235042572021484, 0.12152647972106934, -0.4291508197784424, -5.3528971672058105, 0.9961156845092773, 3.73960542678833, 1.255305528640747, -4.395349502563477, -3.075523853302002, 1.640925407409668, -0.606997013092041, -1.1397316455841064, -1.1052454710006714, -3.777925968170166, -9.628783226013184, 5.932575225830078, -1.481261968612671, -1.646317481994629, -5.108818531036377, 0.8265500068664551, 1.2145452499389648, -4.815622329711914, -7.749813079833984, 3.2036781311035156, -0.9537062644958496, -6.352267265319824, -2.076728105545044, 0.7074213027954102, 2.46437931060791, -0.6223572492599487, -3.866209030151367, -2.539402723312378, -1.882919430732727, 2.6149821281433105, 0.891571044921875, -3.467711925506592, 1.3687658309936523, -2.9096803665161133, -0.26175665855407715, -3.0279645919799805, -4.415889739990234, 2.0875158309936523, -1.1997021436691284, -0.44031190872192383, -3.164024829864502, -0.14583659172058105, -3.5281906127929688, -3.5731899738311768, -3.274991035461426, -2.7786593437194824, -2.358003616333008, -2.542630672454834, 3.075821876525879, 6.940163612365723, -6.96040678024292, -1.3937255144119263, -4.9800872802734375, -3.370805501937866, 3.420398235321045, -3.2828750610351562, -0.6140123605728149, -4.284762382507324, -7.020569801330566, 0.10129332542419434, -1.921642780303955, 1.7282042503356934, 0.6849675178527832, 1.329284429550171, 0.16706252098083496, -7.22212553024292, -4.440886974334717, 2.477200508117676, -4.276995658874512, -5.812880992889404, -3.503268003463745, 1.432027816772461, -1.3879841566085815, -6.93439245223999, -1.495739459991455, -3.6810927391052246, 0.659736156463623, -6.049291610717773, -1.605202555656433, -3.5726211071014404, -1.020853042602539, 1.0847206115722656, -7.682163238525391, -4.5134758949279785, -1.7719932794570923, 3.233337879180908, -0.6836516857147217, -1.2906949520111084, -5.087630271911621, -7.318576335906982, -10.195448875427246, -5.637441158294678, -0.07966744899749756, 0.14804363250732422, -1.300318717956543, 1.3437740802764893, -3.495537757873535, -2.1344611644744873, -3.9874038696289062, -1.4140205383300781, -0.3906557559967041, 2.597050666809082, -2.260207176208496, -1.6388144493103027, 2.0414676666259766, -6.0388970375061035, -3.8245174884796143, -0.2576603889465332, -1.3933508396148682, 3.1012163162231445, -5.643864631652832, -3.5233466625213623, -2.96541428565979, -4.536442756652832, -1.246294617652893, -4.8115034103393555, -2.2906906604766846, 0.10042691230773926, -1.3932602405548096, -2.727370262145996, 1.746885061264038, -4.215477466583252, -4.753995895385742, -0.13447785377502441, -1.2479033470153809, 2.717252254486084, 1.1807529926300049, 4.151309490203857, -3.961519956588745, -2.7157301902770996, -2.425772190093994, 0.429746150970459, 2.3205156326293945, -0.02191948890686035, 1.056408405303955, 0.09711742401123047, -1.8619937896728516, -3.5050904750823975, -0.691056489944458, -0.9696766138076782, -6.640724182128906, -4.079468727111816, -8.563694953918457, -0.5518813133239746, -1.5610188245773315, -2.2994985580444336, 1.0325846672058105, -1.151265263557434, 0.039670705795288086, -1.5618053674697876, -0.3465256690979004, -3.5941989421844482, -3.3924129009246826, 1.7988495826721191, 0.3262007236480713, 3.8236756324768066, 0.5734398365020752, -1.5373378992080688, -1.5038414001464844, -3.852466583251953, 3.5418405532836914, -1.7420177459716797, -5.250743865966797, -3.0149683952331543, -4.464634895324707, -6.2788987159729, -5.992021083831787, -6.243288040161133, -1.775281310081482, -7.878905296325684, -6.332823753356934, -2.037736415863037, 1.8885436058044434, -5.555478096008301, -2.925881862640381, -5.357375621795654, -0.6559033393859863, 3.021390914916992, -4.383746147155762, -5.21929407119751, -8.476089477539062, -1.5295448303222656, -8.452390670776367, -1.092457890510559, -1.0679335594177246, 0.004913330078125, 2.7675695419311523, -2.9991531372070312, 2.8265976905822754, 2.2299585342407227, -0.26565825939178467, 1.956679344177246, -0.20720314979553223, -4.696630954742432, -2.4594202041625977, 0.15655517578125, 1.77370285987854, 1.7486929893493652, -1.9716867208480835, -11.057394027709961, -1.3771512508392334, 3.306652069091797, -0.931433916091919, -4.419814586639404, -3.761432409286499, -7.68950080871582, 1.3820161819458008, -0.8336206674575806, -5.862619400024414, -6.992682933807373, -2.7577075958251953, -6.351909160614014, 0.22126483917236328, -3.1746826171875, 1.2738909721374512, -2.595325469970703, -2.787702798843384, -0.925560474395752, -1.8681498765945435, -2.9452502727508545, 0.4486098289489746, -5.530239105224609, 4.720532417297363, -4.533283233642578, 1.1997404098510742, -5.219465255737305, -0.5918368101119995, 0.28187131881713867, 0.2924027442932129, -1.9950882196426392, -2.8772780895233154, -5.589972019195557, -3.6738245487213135, -6.822033405303955, -0.6734035015106201, -4.934063911437988, -4.57266902923584, -1.345578908920288, -2.6108970642089844, -3.589071035385132, -2.590646266937256, 3.9519433975219727, -1.2238550186157227, -5.042448043823242, -1.7706953287124634, 1.2874698638916016, -2.4991042613983154, -1.0045613050460815, -0.02303171157836914, -2.0008251667022705, 0.9852256774902344, 1.5353195667266846, -3.8609111309051514, 2.5488481521606445, -8.210329055786133, -7.651161193847656, -5.347898960113525, -2.941938877105713, -4.446771621704102, -2.8121211528778076, -2.608633279800415, -4.6316986083984375, -1.100160837173462, 1.1032261848449707, -0.9412585496902466, 1.9156389236450195, -1.6513055562973022, 0.3183903694152832, -3.9839603900909424, -0.44145965576171875, -3.55165958404541, -2.9987354278564453, -5.072355270385742, -3.160123348236084, -1.321303129196167, -2.060451030731201, 1.5269670486450195, -5.564566135406494, -1.0915799140930176, 0.25002622604370117, -0.7607132196426392, 1.7091460227966309, -0.5526944398880005, 2.3796329498291016, 0.3969712257385254, -0.6145081520080566, -0.028606176376342773, -3.6710588932037354, 4.206747531890869, -0.11343026161193848, 1.5725812911987305, 1.460932970046997, 1.869964838027954, 0.7602601051330566, -1.8981095552444458, 1.8012759685516357, -2.8779730796813965, -2.1641879081726074, 0.758777379989624, 1.6693570613861084, 2.862320899963379, 2.653322696685791, -2.380138874053955, 2.1631860733032227, -1.1691561937332153, -5.428879737854004, 2.770108699798584, -4.696113109588623, 0.3097705841064453, -9.027170181274414, -5.25781774520874, -2.608200788497925, -4.566548824310303, -5.3516845703125, -3.7753868103027344, 1.1417584419250488, 2.8924102783203125, -1.6827471256256104, -4.159179210662842, 0.12191915512084961, -0.07204711437225342, -0.26030707359313965, -4.037956237792969, -1.2053166627883911, 1.9511337280273438, 0.5440678596496582, 2.3315563201904297, -4.46658182144165, -0.8435578346252441, -4.38161563873291, -1.9689297676086426, 0.6213574409484863, 1.266526460647583, 0.6895427703857422, -2.6778485774993896, 0.947974681854248, -0.8216589689254761, -9.013751029968262, -2.7646260261535645, -7.315518856048584, -2.786314010620117, 2.04022216796875, -4.959984302520752, -7.057924270629883, 3.338277816772461, -7.402348518371582, -8.08827018737793, -2.4763031005859375, -6.294713020324707, 2.72860050201416, -0.548743486404419, 1.9364306926727295, -7.801760673522949, -1.4606513977050781, -3.691002130508423, -5.462974548339844, 3.0857391357421875, -1.817861795425415, -6.334316253662109, -2.946570634841919, -0.06468355655670166, 0.7363355159759521, -4.726164817810059, 4.88459587097168, -1.9074832201004028, -4.604501724243164, -0.20455551147460938, -0.8867747783660889, -6.6438679695129395, -4.395116806030273, -3.347597122192383, -0.23169207572937012, -4.549591064453125, 0.047636985778808594, 0.17024660110473633, 2.156391143798828, -3.684645891189575, -3.2942962646484375, -7.2315354347229, -5.802791118621826, -3.3200454711914062, -5.878573894500732, -2.473484754562378, 1.9201045036315918, -8.75411605834961, 3.312547206878662, -0.3787233829498291, -0.809067964553833, -3.3140430450439453, -0.8922697305679321, -5.551644325256348, -4.121053695678711, -2.774580240249634, -3.672879695892334, 3.381998062133789, -0.7404177188873291, -2.179454803466797, -1.2893234491348267, -1.282482385635376, -0.22974681854248047, -8.034902572631836, -2.3141274452209473, -1.9068689346313477, 0.24727344512939453, -1.771104335784912, -4.066580295562744, -1.6299214363098145, -0.6402320861816406, -5.209903717041016, -3.826512098312378, -1.9564138650894165, -2.4892590045928955, -5.125002861022949, 3.2581558227539062, -1.6919097900390625, 2.4577059745788574, -4.987186908721924, -2.1901051998138428, -0.912258505821228, -2.645756721496582, -6.9994306564331055, 0.6489195823669434, -4.271417617797852, -1.3489890098571777, -5.129056453704834, -0.5360736846923828, -5.663804531097412, -5.4536590576171875, -3.33803129196167, -2.973262310028076, 2.059370517730713, -2.99687123298645, -0.4961371421813965, -3.3862290382385254, 2.005000591278076, -2.5178375244140625, 0.3249073028564453, 5.925906181335449, -0.5192399024963379, -1.8388986587524414, -1.9533134698867798, 0.4307899475097656, -2.6175127029418945, 1.8632469177246094, -7.226478099822998, -3.3908886909484863, -1.7667396068572998, 4.59670352935791, -2.3848609924316406, -5.093968868255615, -4.195920944213867, -1.1828587055206299, -1.692629337310791, -4.180746078491211, -1.7563436031341553, -1.5163509845733643, -2.20986008644104, -0.7950234413146973, -1.7382134199142456, -5.709691047668457, -4.721661567687988, -0.18892383575439453, -2.014536142349243, 2.329768180847168, 1.9746196269989014, -4.3705034255981445, 1.9646787643432617, -1.7556616067886353, -0.1509244441986084, -5.784377098083496, 3.617201805114746, 1.6604580879211426, 0.09533309936523438, -2.7628273963928223, 6.711496353149414, 0.21928930282592773, -7.334522247314453, -4.038601875305176, -4.90776252746582, -3.88755464553833, 1.049034595489502, -2.7038731575012207, 2.065187931060791, 1.084073543548584, 0.6411051750183105, -4.15317440032959, -5.612689018249512, -6.754944801330566, -5.503905773162842, 1.551870584487915, -0.7220995426177979, -2.166947364807129, -5.0507025718688965, -5.02078914642334, -0.1178356409072876, -1.4573843479156494, 0.5455327033996582, -2.2560312747955322, -3.8164281845092773, -2.272094488143921, 1.465071201324463, 1.1893634796142578, -1.7666175365447998, 1.7827644348144531, -1.5298913717269897, -3.8728435039520264, -3.656815767288208, -0.8060348033905029, -1.9366447925567627, -1.7408523559570312, 3.1540660858154297, 0.6699697971343994, -2.3968617916107178, 0.31457090377807617, -4.157875061035156, 1.3992118835449219, -0.42072439193725586, -1.0224813222885132, 0.11700129508972168, -7.270445346832275, -0.1319599151611328, -1.6604633331298828, -3.477703094482422, 0.8998665809631348, 1.2178072929382324, 0.715752124786377, -5.035342693328857, -1.9431648254394531, -6.681893348693848, -7.541989326477051, -2.807223081588745, 1.342815637588501, -3.235503911972046, 3.2983059883117676, 1.6684761047363281, -1.1122101545333862, -10.500109672546387, -2.375349998474121, -7.603442192077637, -6.074528694152832, 0.9601101875305176, -5.724803447723389, 0.28499412536621094, 1.3080799579620361, -5.4999799728393555, -1.7615077495574951, -3.362583637237549, -5.3532257080078125, -7.074810028076172, -3.3080978393554688, -2.4114909172058105, 0.35123682022094727, -2.4410758018493652, -7.951294898986816, -0.7842650413513184, -1.5784573554992676, 0.09851193428039551, -4.9448347091674805, -2.601149559020996, -3.4207963943481445, -6.450491428375244, -4.537624835968018, 4.239635467529297, -5.840964317321777, -3.8392069339752197, -1.336687445640564, -0.8940939903259277, 1.2131197452545166, -4.378180503845215, 1.3923628330230713, -1.0997246503829956, 0.3912923336029053, 1.1934986114501953, -4.128823280334473, -4.628174781799316, -4.0147294998168945, 1.00272798538208, -2.5376410484313965, -3.165926218032837, -4.290055274963379, -1.179368257522583, -1.4321396350860596, 2.6686673164367676, -5.632445335388184, -3.587069034576416, 0.1036520004272461, -6.6180739402771, -1.840158462524414, -0.1111229658126831, 4.7523932456970215, -2.1429295539855957, 2.298245429992676, 0.7152056694030762, -2.951199769973755, 0.4715583324432373, 2.229827880859375, -0.17899847030639648, -4.100822925567627, -6.244952201843262, -3.0495150089263916, -6.116174697875977, -8.405317306518555, -6.689084053039551, 2.8339853286743164, -5.451068878173828, -5.041722297668457, -5.586367607116699, -3.881833076477051, -2.7889561653137207, -3.4668240547180176, -6.351131916046143, -0.9135161638259888, -5.48634147644043, -5.15848445892334, 2.0118770599365234, -5.783267974853516, 2.655226707458496, 1.0051140785217285, 0.6757268905639648, -3.3381402492523193, -6.19211483001709, 1.7979373931884766, 0.6872467994689941, -1.5066025257110596, -0.5789237022399902, -6.379998207092285, -4.026480674743652, -5.369908332824707, 0.985781192779541, -4.162005424499512, -3.554386854171753, -1.7990484237670898, 1.651604413986206, -0.41374409198760986, -2.6446666717529297, 0.055787086486816406, -3.8940205574035645, -3.113798141479492, -1.227738618850708, -0.672215461730957, 1.040501356124878, -2.129220724105835, -3.3477799892425537, -1.1964372396469116, -1.4693243503570557, -1.833081603050232, -4.320962905883789, 1.6440176963806152, -1.6317881345748901, 2.436288833618164, 0.44950437545776367, -2.7722301483154297, -4.557969093322754, 0.7700695991516113, 4.0631842613220215, -3.5175352096557617, -2.05556058883667, -1.771507978439331, -6.281555652618408, -4.036362171173096, 0.2453165054321289, -3.943687915802002, -1.3130505084991455, -1.9210776090621948, -5.476134777069092, -3.5078928470611572, -4.681110382080078, -2.981961488723755, 2.4395246505737305, -1.5484673976898193, -4.169112205505371, 1.843625545501709, -4.603514194488525, -1.7745885848999023, -1.6139615774154663, -6.510593414306641, -4.659849166870117, -1.175685167312622, 0.43419456481933594, 2.205831527709961, -1.400674819946289, -1.811205506324768, 0.4493551254272461, 3.5984392166137695, -4.035065174102783, -4.98838996887207, -0.7676568031311035, 2.3467116355895996, -5.133067607879639, 1.6721022129058838, -2.168860673904419, 1.9732398986816406, -2.831603765487671, -2.0298471450805664, -2.183859348297119, 1.4273931980133057, 0.3677849769592285, 1.7786180973052979, -0.5029008388519287, 0.8022150993347168, -3.2641663551330566, -0.041749000549316406, -3.0250697135925293, -1.89710533618927, -2.912957191467285, -5.259683132171631, -3.9909067153930664, -0.23134255409240723, -3.0993540287017822, 0.7013506889343262, -4.546866416931152, -0.16777515411376953, -3.3888468742370605, -5.687426567077637, -0.7624276876449585, -6.072544574737549, 1.1566519737243652, 1.5013065338134766, -3.039553642272949, 0.0422053337097168, -1.7059589624404907, -4.609550952911377, 2.615717887878418, -0.6835407018661499, -4.589095592498779, -5.308304786682129, -3.2401747703552246, 4.159835338592529, -3.698284864425659, -7.106351852416992, -5.946059226989746, -3.2541162967681885, 3.675192356109619, -7.8054518699646, -8.695465087890625, -7.302430152893066, -4.3626627922058105, -1.7792015075683594, -0.16929292678833008, -5.289216995239258, 0.5041205883026123, -3.572737693786621, 3.1333107948303223, 4.790759086608887, -0.9142001867294312, -4.359107971191406, 0.7211875915527344, -4.008228302001953, -1.3151700496673584, -1.3709235191345215, -0.9889059066772461, -1.3182570934295654, -4.5546417236328125, 2.3742151260375977, -6.271099090576172, 0.3071162700653076, -0.8313568830490112, -0.027306318283081055, -5.431145191192627, 1.6767191886901855, -1.368607997894287, -2.8023812770843506, -0.16204380989074707, -4.2171759605407715, 0.6844432353973389, -4.765367031097412, -1.129563331604004, 1.6477580070495605, -1.7690658569335938, -5.087184906005859, -0.7277987003326416, -3.4777724742889404, -0.7712805271148682, -4.143681049346924, -2.444906711578369, -1.4171489477157593, -0.5826056003570557, -3.0708107948303223, 1.2035510540008545, -6.866303443908691, 1.905181884765625, -3.671567678451538, -4.021536827087402, -2.4366025924682617, -8.777873039245605, -4.196953773498535, -5.75533390045166, 3.248202323913574, -0.5551496744155884, 2.070444107055664, -3.4055228233337402, -2.454329252243042, -1.8301106691360474, -6.060302734375, -0.6380788087844849, -1.9795210361480713, -5.769207000732422, -2.0144689083099365, -2.3086624145507812, -2.8072009086608887, -0.8000580072402954, -1.6487314701080322, -0.9957504272460938, -3.6727075576782227, -2.1906979084014893, -7.741244316101074, 1.9876103401184082, -1.2272613048553467, -4.7797980308532715, -1.9811606407165527, -3.815000295639038, 3.1446194648742676, -4.383749961853027, 0.34021759033203125, 2.5500388145446777, -4.755120277404785, -2.469897985458374, -1.9017915725708008, -1.6560224294662476, -6.062408447265625, -3.8526148796081543, -2.929408550262451, -5.659117698669434, -2.5719316005706787, 0.17380881309509277, -2.1198394298553467, -5.059850692749023, 1.5548896789550781, -0.5197978019714355, 2.309957504272461, -3.9311251640319824, -4.68878173828125, 0.3234391212463379, -0.30321264266967773, -3.5138208866119385, -1.3689208030700684, 2.657136917114258, -2.6761817932128906, -6.7475481033325195, -3.1129698753356934, -4.622633934020996, 0.5456070899963379, -4.7125654220581055, -3.514511823654175, -9.535171508789062, 0.546323299407959, -1.9299328327178955, -2.507266044616699, 1.7683281898498535, 0.2084360122680664, -2.2882020473480225, -4.020909309387207, -3.7977871894836426, 2.1151576042175293, -4.78759765625, -0.7337100505828857, -2.716059446334839, -5.055883407592773, -5.68217134475708, -6.263896465301514, -4.800769329071045, 0.20219707489013672, -2.27484393119812, 2.0442633628845215, -5.650697708129883, -6.183127403259277, -0.9927659034729004, -5.705794811248779, -2.6825640201568604, 0.1178886890411377, -1.3370673656463623, 1.0195372104644775, -0.686086893081665, -2.769162654876709, -1.4579637050628662, -1.765977382659912, -2.832827091217041, -3.593132972717285, -2.179262161254883, -1.5814321041107178, -0.03816866874694824, -5.9738850593566895, -3.6328370571136475, 4.489824295043945, -2.744887351989746, -2.094719648361206, -6.144036769866943, 3.6352944374084473, -0.9136444330215454, 3.7284116744995117, -1.7217050790786743, 0.09144067764282227, -5.142960548400879, 2.5078110694885254, -2.632899761199951, 0.9617948532104492, -4.189119815826416, -2.230295181274414, -2.522256851196289, -2.58430814743042, -0.12218546867370605, -5.6770806312561035, -5.5562286376953125, -6.09110164642334, -0.08856022357940674, -0.30156922340393066, -4.613212585449219, -1.6880433559417725, 3.046619415283203, -2.8131213188171387, -2.517954111099243, -5.117194175720215, -6.117526054382324, -3.8507742881774902, -0.8528242111206055, -7.689763069152832, -4.926620006561279, 1.3724844455718994, -4.302188396453857, -2.648005247116089, -1.7965766191482544, -4.440743446350098, -4.807621479034424, -0.926368236541748, -3.6717851161956787, -2.369051218032837, 1.8044157028198242, 1.1690263748168945, -0.7906759977340698, -9.468902587890625, -0.9792894124984741, -3.400510787963867, -4.92031192779541, -5.115950584411621, -4.85594367980957, -0.505217432975769, -3.215461015701294, -0.221746563911438, -6.2895097732543945, 1.2689146995544434, -4.058809280395508, 1.354395866394043, 2.0140023231506348, -0.8869820833206177, -0.4246370792388916, -0.003493070602416992, 1.1774263381958008, -2.514009475708008, -4.633217811584473, -3.6409788131713867, 2.054323196411133, 0.9464266300201416, -6.07687520980835, -2.543391227722168, -0.1580045223236084, -0.3251011371612549, -7.841500282287598, -4.208834648132324, -7.985084533691406, 0.7745511531829834, -2.322500228881836, -5.236318111419678, -2.36318302154541, -2.081329822540283, 0.7332844734191895, -1.5596235990524292, -1.9802809953689575, -2.6190547943115234, 2.312926769256592, -7.631856441497803, 1.7614407539367676, -7.625850200653076, 0.8484153747558594, -1.1498583555221558, -4.389385223388672, -4.515909194946289, -6.102886199951172, -5.9719743728637695, 0.9795098304748535, -3.9789795875549316, 1.103811502456665, -1.356398344039917, -0.812065601348877, -10.893109321594238, -7.421529769897461, -0.40723955631256104, -0.675076961517334, -0.14023947715759277, -7.399420738220215, -5.413149356842041, -6.737228870391846, -6.007416248321533, -3.840383291244507, 2.1840391159057617, 0.879359245300293, 0.527735710144043, -1.3072562217712402, -6.062268257141113, 0.003759145736694336, 0.8757016658782959, -1.8302996158599854, -0.9089505672454834, -3.901726722717285, 1.286303997039795, -1.0226417779922485, -1.6426975727081299, 5.71687126159668, -1.9360930919647217, -2.13043212890625, 2.5801048278808594, 0.5399563312530518, -1.5429739952087402, 2.8006649017333984, 0.5634677410125732, -0.830592155456543, -0.10466289520263672, -0.9739646911621094, -4.644113540649414, 0.8789768218994141, -5.4314703941345215, -5.650787353515625, 0.12737345695495605, -2.1627910137176514, -8.507415771484375, 0.08623790740966797, -1.7275563478469849, -4.30879020690918, -3.491748332977295, -1.2263615131378174, -1.09275221824646, -2.52260160446167, -3.7396953105926514, -2.6506309509277344, 0.027573108673095703, 7.733227729797363, -2.0683493614196777, -4.316160678863525, -4.631217956542969, -3.0185811519622803, -0.6083627939224243, -2.5137104988098145, -4.130929470062256, -0.8881580829620361, -0.8918511867523193, 0.6641082763671875, -1.505619764328003, -5.277291774749756, 1.3971977233886719, 4.647245407104492, -3.486424446105957, -1.242297887802124, -5.575249671936035, 1.6807007789611816, -2.201737403869629, -1.5102096796035767, -3.9328532218933105, -1.0722979307174683, 1.9197888374328613, -1.162062168121338, -3.6695032119750977, -8.598197937011719, -2.081286668777466, -5.245453834533691, -6.235645294189453, -5.125422477722168, -2.483975887298584, -2.3945705890655518, -1.8260829448699951, -5.63508939743042, -7.888883113861084, -7.503839492797852, -3.335498094558716, -2.1353347301483154, -10.643209457397461, -5.136730194091797, -1.4654488563537598, 6.471290588378906, -3.6194725036621094, -7.443498134613037, -3.5200395584106445, -1.7420893907546997, -2.8560941219329834, 3.3851118087768555, 1.250551700592041, 0.13477849960327148, -1.254224181175232, -2.1409313678741455, -1.431414008140564, -3.125667095184326, -6.674414157867432, -3.2732670307159424, -1.0041025876998901, -0.5220091342926025, 5.233790397644043, -3.423408031463623, -0.4246804714202881, -1.513017177581787, -10.045463562011719, -2.0878119468688965, -1.367663025856018, -2.6737170219421387, -0.8696160316467285, 0.6590409278869629, -1.5097897052764893, -2.4624452590942383, -2.422198534011841, -3.3253402709960938, 1.228105068206787, 0.858393669128418, -0.2641479969024658, -1.314298152923584, 3.832305908203125, -2.595716953277588, -5.251500129699707, -1.3173160552978516, -0.5955008268356323, -1.6250416040420532, -3.0223538875579834, -2.572739362716675, 1.127185344696045, -6.348598957061768, -6.34894323348999, -1.942816138267517, -0.46464014053344727, -1.789601445198059, 0.4512350559234619, -2.1615350246429443, -5.690046310424805, -5.387038230895996, -6.6372785568237305, -0.2545304298400879, -3.3179779052734375, 0.5074019432067871, -2.471470355987549, 1.9714202880859375, -1.0535637140274048, -0.9186656475067139, -3.111715316772461, -0.39125919342041016, 0.10504555702209473, -1.8194721937179565, -2.4754574298858643, -2.824580669403076, -2.9022860527038574, 0.26986122131347656, -3.408318042755127, -3.818532943725586, 4.007429122924805, -3.1313083171844482, 1.3584411144256592, -2.356229066848755, -4.569790363311768, 0.5051476955413818, -5.523662567138672, -3.8383843898773193, -4.798254489898682, -1.5291087627410889, 1.3042397499084473, -0.6714481115341187, -4.927452564239502, -5.638601779937744, -1.5403683185577393, -6.41008186340332, -2.8210411071777344, -2.995737075805664, -1.6743688583374023, 1.5816280841827393, 1.9294095039367676, -4.2922444343566895, -4.488346099853516, -0.8918964862823486, -1.4916582107543945, -1.7042670249938965, -3.284990072250366, -2.6774027347564697, 0.8638958930969238, 1.029764175415039, -0.6397972106933594, -3.123014450073242, 0.21383261680603027, 0.5153050422668457, -0.7700285911560059, -1.188708782196045, 2.0697288513183594, -1.8741726875305176, -1.9138208627700806, -5.062657356262207, -1.4568753242492676, -3.9969475269317627, -0.3815596103668213, -2.356943368911743, -4.132903099060059, -2.9383389949798584, 0.6750004291534424, 3.6126813888549805, -4.0940260887146, -5.556727886199951, -3.905330181121826, -3.4205832481384277, 0.7271139621734619, -5.611693382263184, -1.3049688339233398, -1.2568187713623047, 0.493394136428833, -2.007458448410034, 1.0788164138793945, -5.237616539001465, -0.27890753746032715, -0.24538683891296387, -2.1442461013793945, 1.274521827697754, -1.7900670766830444, -5.106810569763184, -0.9953598976135254, -5.563401699066162, -4.85731315612793, -6.1759033203125, -8.193008422851562, -7.046463489532471, 3.962766647338867, -2.603817939758301, 3.1589603424072266, 0.8683285713195801, -3.6863136291503906, -6.156879425048828, -0.6745734214782715, -4.3088531494140625, -2.3834333419799805, -0.6018208265304565, -3.8727200031280518, 0.1979389190673828, -5.429418087005615, 0.6619067192077637, 1.0522236824035645, 3.438978672027588, 2.83294677734375, -4.982919216156006, -6.143863677978516, -3.3291192054748535, -4.957033157348633, -4.35009765625, -0.8374755382537842, 0.38440942764282227, -3.646925926208496, -2.4066896438598633, -2.7304282188415527, -5.05514669418335, 0.06785345077514648, -1.54857337474823, -0.6129522323608398, 2.453795909881592, 0.6941719055175781, -0.9176058769226074, -0.8946990966796875, -4.986132621765137, -4.151679992675781, -0.1862344741821289, -5.326455116271973, -3.322516679763794, -2.7156665325164795, -1.8876579999923706, 0.1021571159362793, -0.7311910390853882, -4.99178409576416, -1.4421708583831787, -0.6431617736816406, -3.419679641723633, -5.4623494148254395, -2.1843819618225098, 4.059637069702148, -3.7446963787078857, -2.736088275909424, -2.2907726764678955, -4.991015911102295, -3.677295207977295, -2.552976608276367, -3.8683042526245117, -4.1055402755737305, -2.2109344005584717, -0.6905951499938965, 4.769203186035156, -4.173958778381348, -0.3527059555053711, -6.598082542419434, -5.2233757972717285, -3.5490753650665283, -8.430574417114258, 0.6434426307678223, -4.682682037353516, 1.2284655570983887, -2.4808566570281982, -1.8502178192138672, -0.22699511051177979, -3.3295679092407227, -2.8937149047851562, -1.5283372402191162, -0.21027755737304688, -0.292183518409729, 2.7842788696289062, 2.4574484825134277, 1.2314753532409668, -3.3752801418304443, 1.0325119495391846, -0.851836085319519, 1.9916601181030273, -2.373202085494995, -1.9539793729782104, -1.3120081424713135, 0.2019367218017578, -4.760317325592041, -3.9970173835754395, -4.9635725021362305, -0.3295978307723999, -0.8043758869171143, -5.744071006774902, -0.583417534828186, -2.8527820110321045, -0.5682427883148193, 3.143998146057129, -1.090825080871582, -0.7015472650527954, -6.161124229431152, -2.57018780708313, -2.7501635551452637, 1.9501075744628906, -1.5341944694519043, -4.842288494110107, 1.2144973278045654, 0.4923431873321533, -2.0213699340820312, -0.2081838846206665, 1.3358092308044434, -7.081439018249512, -5.899818420410156, -3.1390600204467773, -3.926102638244629, -6.221046447753906, -3.270138740539551, -1.8425089120864868, -3.0819826126098633, -4.070569038391113, -2.174623727798462, 1.4133076667785645, -4.014892101287842, -1.3741062879562378, -7.678604602813721, -8.15550422668457, 2.311978816986084, -3.132725715637207, -3.528907060623169, -1.977608561515808, 1.7165961265563965, -4.627441883087158, -3.0197737216949463, -5.214404106140137, -1.4491422176361084, 0.12275123596191406, 1.5460095405578613, 2.0858402252197266, 2.562936305999756, 0.4622340202331543, -3.288080930709839, -1.2679259777069092, -4.6454949378967285, -2.7391562461853027, -3.899620532989502, -6.564528465270996, 1.1050348281860352, -4.795173168182373, -2.1751644611358643, -1.771843433380127, -5.111175060272217, 2.376181125640869, -6.03048038482666, 0.6410176753997803, -3.1990885734558105, -1.5095711946487427, -1.7916992902755737, -1.9120744466781616, -1.3462977409362793, -0.27710485458374023, -4.474553108215332, 2.2620768547058105, 1.92059326171875, -1.3937876224517822, -0.17356085777282715, -2.2964248657226562, -1.0901672840118408, -5.244223117828369, -3.8646719455718994, -1.494124174118042, -2.0324668884277344, -2.4909720420837402, -3.9867634773254395, -0.6903215646743774, 3.7839460372924805, 1.8879179954528809, -7.972970962524414, -2.3309855461120605, -3.464139938354492, -2.910080909729004, -4.857437610626221, -7.0218186378479, -3.402048349380493, -0.9164042472839355, -6.02292537689209, -2.0710151195526123, -3.2232370376586914, -3.154536247253418, -4.00089168548584, -7.601005554199219, -0.09095263481140137, -6.654637813568115, -7.345025539398193, -4.592130661010742, -1.7363392114639282, -2.0272843837738037, 5.736368179321289, 1.007530689239502, -3.5470571517944336, 0.6874704360961914, 0.6860992908477783, -1.2325937747955322, -3.639962911605835, -0.6982938051223755, 0.8099219799041748, -1.6692873239517212, -4.598763465881348, -0.9264311790466309, 1.2491483688354492, -6.8674468994140625, -1.555078148841858, -4.944123268127441, -0.5997825860977173, 4.624715805053711, -1.1219213008880615, -1.9404547214508057, -2.782917022705078, -4.127145767211914, 2.123284339904785, -2.243971347808838, -5.906966209411621, -3.1089582443237305, 2.3513689041137695, -3.232069730758667, -3.7679860591888428, -2.101606845855713], [-6.740201473236084, -2.6380367279052734, -6.539219379425049, -4.645880699157715, -2.075371742248535, -1.9240508079528809, -2.484086751937866, 0.9102029800415039, 0.8765780925750732, -6.937856674194336, -0.6999832391738892, -4.146545886993408, -4.962604522705078, -4.288271903991699, -0.3547194004058838, -1.7705756425857544, -3.685091972351074, -4.005781650543213, 0.20762014389038086, 0.2694053649902344, 4.106274127960205, -4.733217239379883, -1.3440525531768799, -3.953345537185669, -4.435805320739746, -1.8614137172698975, -5.5164055824279785, -3.5839478969573975, -0.11837029457092285, -4.558649063110352, -0.654283881187439, -2.3041975498199463, 0.24054265022277832, -6.641651630401611, 2.6154251098632812, -3.3605377674102783, -0.21660685539245605, 1.7310106754302979, 2.5481510162353516, -13.02397346496582, -2.319763660430908, -0.906146764755249, -3.3275980949401855, -0.8188115358352661, 1.9713735580444336, -0.34546005725860596, -0.5920125246047974, 0.013774394989013672, -9.117698669433594, 1.9421873092651367, 2.453892707824707, -1.5595107078552246, -1.642193078994751, -6.597573280334473, -3.386666774749756, 0.9110965728759766, 0.33017563819885254, -4.966538429260254, -1.7865484952926636, -1.5152862071990967, -5.5458664894104, 3.982457160949707, -1.7139182090759277, 1.0118939876556396, -0.14410686492919922, 0.23997259140014648, -2.106318712234497, -7.036378860473633, 0.5947906970977783, 0.7626514434814453, -3.8471999168395996, -2.920973062515259, -6.9200758934021, -1.444155216217041, -6.290984153747559, 3.233062744140625, -1.2728512287139893, -0.6603505611419678, 1.9517192840576172, 1.533071517944336, -2.6610305309295654, -0.1668834686279297, -7.728179454803467, 2.4093570709228516, -4.860174179077148, -4.787944793701172, -5.45968770980835, -5.485502243041992, -2.3576273918151855, 2.1860337257385254, -1.2055788040161133, -2.9063830375671387, -6.3251495361328125, -3.1823570728302, 2.2147040367126465, -0.0002740621566772461, -5.49324369430542, -6.089431285858154, -4.467301368713379, -1.941555380821228, -1.5576906204223633, -2.7188258171081543, -1.5268572568893433, 4.787096977233887, -4.424893379211426, -2.0069501399993896, -3.5888137817382812, 6.0344038009643555, -5.89993953704834, -6.713686466217041, -3.947612762451172, 3.721895217895508, -1.4432342052459717, -6.363854885101318, -2.243682861328125, -2.4274675846099854, -1.2547998428344727, -4.488366603851318, -1.758962631225586, -4.989535331726074, -4.4709577560424805, -4.5334038734436035, -0.9693640470504761, 3.30056095123291, 3.6590189933776855, 2.171095848083496, -4.929404258728027, -2.299240827560425, -2.3739013671875, -2.9825916290283203, -1.7113131284713745, 3.5320839881896973, -1.9385093450546265, -1.797760248184204, -1.6769925355911255, 0.2581675052642822, -2.675628662109375, 0.9990506172180176, -2.9112226963043213, 0.29273414611816406, -0.9080932140350342, -5.927285194396973, -2.08616304397583, -4.388598442077637, 0.9963366985321045, -3.2179348468780518, -0.25972795486450195, -5.211208343505859, 1.4346489906311035, -7.951473712921143, 0.9515936374664307, -3.2197141647338867, -3.6838226318359375, 2.6896705627441406, -1.7151442766189575, -1.0758661031723022, -2.2225747108459473, -0.9563162326812744, 1.9639496803283691, -2.5886008739471436, -3.232992172241211, 0.7503376007080078, -6.855734825134277, 1.1821222305297852, -4.826042652130127, 2.661257743835449, -3.3537912368774414, 1.1735575199127197, -0.09893083572387695, -3.521205425262451, -0.5370138883590698, -0.7786000967025757, -1.3075289726257324, 2.9800591468811035, -4.656166076660156, -8.46886157989502, -3.41170072555542, -3.688016891479492, -1.6650519371032715, -2.0090370178222656, 0.6585202217102051, -8.447726249694824, -5.439176559448242, 2.73128080368042, -3.3731689453125, -2.4161641597747803, -3.4196250438690186, -7.2937140464782715, -2.691648483276367, -5.087828636169434, -3.446183681488037, 1.1401252746582031, -8.341043472290039, 0.683941125869751, -4.677464962005615, -5.696568489074707, -5.128813743591309, 1.825510025024414, -6.582805156707764, -9.735909461975098, 3.8406782150268555, -2.4233248233795166, -1.5706868171691895, -1.7548617124557495, -3.6244466304779053, -0.5640358924865723, -7.927205562591553, -5.6555705070495605, 0.015530109405517578, -1.6957308053970337, -2.7958273887634277, -5.992176532745361, 0.9709031581878662, -0.07675004005432129, -0.6845177412033081, 4.410323143005371, 0.003651857376098633, -4.812980651855469, -0.6203441619873047, -2.078537702560425, -2.2639482021331787, -2.359678268432617, 1.2292346954345703, -0.7645659446716309, 0.6958050727844238, -5.324155807495117, -5.450504779815674, -0.7659544944763184, 0.411409854888916, -2.277669668197632, -1.766426682472229, -2.7280211448669434, -2.3311758041381836, -8.628349304199219, -4.505330562591553, -1.2161403894424438, -5.490856170654297, 1.9057087898254395, 1.5665535926818848, -5.0712127685546875, -7.665270805358887, -3.6261563301086426, -0.9192460775375366, -0.41841650009155273, -1.7499533891677856, -2.044721841812134, -3.5232250690460205, 0.18801617622375488, 2.228975772857666, -3.903477668762207, -2.326201915740967, -7.3550004959106445, 0.19834518432617188, -1.0531902313232422, -5.459089279174805, -6.08351469039917, -5.576652526855469, -4.813660621643066, -1.9747263193130493, -2.400573253631592, 1.0149939060211182, -0.025458931922912598, -2.474453926086426, -10.613082885742188, -5.402425289154053, -1.7431678771972656, -0.04705047607421875, 3.7706332206726074, -3.66318678855896, -1.8330910205841064, -0.3761337995529175, -7.511247634887695, -1.1602437496185303, 0.3677060604095459, 0.5004971027374268, -0.46157968044281006, 0.9770016670227051, -3.0356884002685547, -3.3427891731262207, -0.03277158737182617, -2.2836642265319824, 1.6168956756591797, 1.3995847702026367, -1.3707795143127441, -1.2767810821533203, -0.6696091890335083, -1.1710383892059326, 0.8505110740661621, 0.5153157711029053, 0.5930540561676025, -0.2503163814544678, 0.09102845191955566, -5.843935012817383, -0.15170907974243164, -3.536088466644287, -2.266362190246582, -1.8140740394592285, -1.1128859519958496, 0.3784980773925781, -2.9893763065338135, -6.387013912200928, -3.165781259536743, -6.071161270141602, -1.9219555854797363, -0.4340859651565552, -1.138004183769226, 1.6311123371124268, -7.478187561035156, 2.097965717315674, -4.327797889709473, -0.7232928276062012, -1.9539227485656738, 0.3317136764526367, -0.4264824390411377, -2.668900489807129, -6.01214075088501, -5.629507541656494, 1.3756186962127686, -2.034472942352295, 2.9350104331970215, -5.859104156494141, 2.6230006217956543, 0.7326478958129883, -4.015161037445068, -3.8323206901550293, -4.535567283630371, -0.5600829124450684, 0.23334360122680664, -1.4894134998321533, -1.2602810859680176, 1.270927906036377, -0.40488433837890625, 4.280763626098633, -2.804912567138672, -3.942695140838623, 1.663532018661499, -0.49254369735717773, -4.480830669403076, -2.8155505657196045, -0.03143668174743652, -3.826565742492676, -1.05271577835083, -2.3449654579162598, -1.6366093158721924, 0.2307586669921875, -2.016167640686035, -1.881858468055725, -2.440101146697998, 0.9734578132629395, -6.826033592224121, 3.2553067207336426, -4.7320146560668945, -1.1064647436141968, -5.403307914733887, -0.2029252052307129, -7.3098955154418945, 0.07149410247802734, -2.981956958770752, -2.089077949523926, 2.5519046783447266, -1.2417436838150024, -0.14177155494689941, -2.125732660293579, -5.368720054626465, -4.681837558746338, -5.975797653198242, 1.5583851337432861, 1.3619685173034668, -8.025821685791016, 0.6771204471588135, -6.846602439880371, 2.269454002380371, -3.7656197547912598, -1.7452173233032227, -1.7138917446136475, -3.950140953063965, -3.829219341278076, -8.453027725219727, 4.067232608795166, -2.1403188705444336, 1.597367286682129, -3.162365674972534, -5.1134748458862305, 0.9984240531921387, 0.5503368377685547, 1.2898781299591064, -4.959477424621582, -5.3283209800720215, -1.4581489562988281, -3.14428448677063, 0.013725519180297852, -2.8022499084472656, -4.458770751953125, -2.6099627017974854, -2.8893563747406006, -6.286092281341553, -2.42392635345459, -1.5828032493591309, -5.668945789337158, -2.3573193550109863, 0.13407111167907715, -2.977731704711914, -4.193030834197998, -5.23676061630249, -1.7308374643325806, -0.5564242601394653, 1.9031620025634766, -1.6850261688232422, -3.9229540824890137, 0.6483762264251709, -1.456589937210083, -0.13202738761901855, -2.444154977798462, -0.32564544677734375, -1.292844533920288, -3.6421072483062744, -3.262794017791748, -6.960666656494141, -3.7759439945220947, -3.9032278060913086, 0.04049181938171387, 2.6996474266052246, 0.7010886669158936, 1.965531349182129, 1.0256075859069824, -4.5173797607421875, 0.0501399040222168, 1.587693691253662, -4.858029365539551, -2.8843111991882324, -0.07993388175964355, -0.519006609916687, -6.732429027557373, 2.7221574783325195, -1.8534793853759766, -0.7385740280151367, -4.429461479187012, -2.6247787475585938, -0.22494959831237793, -6.185949802398682, 0.14623713493347168, -2.172393560409546, -3.617609977722168, -1.7249712944030762, 3.4547042846679688, -6.3064398765563965, -2.038763999938965, -8.30710220336914, -2.5697946548461914, -4.42706298828125, -2.1842868328094482, 2.6334915161132812, -2.1398558616638184, -2.4183759689331055, -0.6529674530029297, -4.221523284912109, 2.785858631134033, 0.15820813179016113, -3.3589138984680176, -1.1455570459365845, 0.3729135990142822, -2.4063570499420166, 1.9177467823028564, -1.391395092010498, -3.073765754699707, 0.5111794471740723, -2.746783971786499, -3.1673731803894043, -0.5962071418762207, 0.993006706237793, -6.344857215881348, -5.679495811462402, -2.5755622386932373, 1.749739646911621, -5.6952362060546875, -7.462586402893066, 1.0139398574829102, -5.893976211547852, 1.7830169200897217, -0.47245216369628906, -0.8353474140167236, -3.7133429050445557, 0.9159293174743652, 0.7095167636871338, 7.243673324584961, -2.6076598167419434, -6.13254976272583, -2.08070969581604, -1.5813860893249512, -5.051527500152588, -9.261076927185059, -4.9947614669799805, -4.518059730529785, -3.0318641662597656, -0.25584423542022705, -3.146968364715576, -1.6846598386764526, 2.993350028991699, -6.615497589111328, -0.5184459686279297, -2.8054211139678955, -6.6042680740356445, 1.4375219345092773, -0.5247390270233154, -5.034856796264648, -7.365443706512451, -1.8562309741973877, 1.4297692775726318, 0.4305596351623535, -2.512451410293579, -2.399430513381958, -4.07294225692749, 1.4710850715637207, -1.0179482698440552, 1.6033921241760254, 0.8152494430541992, -0.4763171672821045, -4.838337421417236, -0.1708526611328125, -2.975612163543701, -3.9084620475769043, -3.142573833465576, -2.736419677734375, -6.294180870056152, 0.4475376605987549, -0.9583930969238281, 0.3985104560852051, -0.8163130283355713, -6.079533576965332, -1.0054610967636108, -4.0579142570495605, -2.730682134628296, 0.71240234375, -4.141791343688965, -2.0495481491088867, -2.004256248474121, -2.0319111347198486, -6.382127285003662, -3.5475025177001953, 2.6641407012939453, 1.4268112182617188, -1.3673617839813232, -1.1668123006820679, -6.66102409362793, -4.1461286544799805, -0.023349285125732422, 1.017432689666748, 2.2371997833251953, -3.076343059539795, 0.5546503067016602, 1.7146806716918945, -6.977423191070557, -1.0549565553665161, -2.46582293510437, -1.2881532907485962, -3.7724175453186035, -4.503640651702881, -6.5304484367370605, 1.2670722007751465, -3.8030292987823486, -3.4811742305755615, -0.5958397388458252, -6.9840874671936035, -1.3680317401885986, -1.7998653650283813, -0.39075493812561035, -5.462167263031006, 3.54524564743042, -0.7859591245651245, -6.5237884521484375, -2.7540152072906494, -0.8766028881072998, -2.0953338146209717, -3.228461742401123, 5.388598442077637, -3.962707042694092, 0.508270263671875, -3.2669425010681152, -0.026202082633972168, -3.4483914375305176, -1.2263277769088745, -2.5363688468933105, -2.0571813583374023, -0.9129115343093872, -3.5415709018707275, -0.3711247444152832, -2.1877050399780273, 1.1058216094970703, -2.4134013652801514, -5.7792439460754395, -6.49625825881958, -4.509421348571777, -4.497124195098877, 1.4147882461547852, -7.1734466552734375, -5.258816719055176, -3.86647367477417, 0.316739559173584, 1.378037691116333, -2.7193081378936768, 2.2779383659362793, -1.1723119020462036, -7.346622943878174, -3.9498891830444336, -3.406615734100342, -2.442241907119751, -5.011951446533203, -2.067495822906494, -3.381725549697876, -3.210083484649658, -6.852616310119629, -6.611578941345215, -4.090429782867432, -4.947813034057617, 1.9662976264953613, -1.7578868865966797, -1.5220921039581299, -0.5411055088043213, -0.1213994026184082, -2.7268385887145996, 0.836047887802124, 2.1105198860168457, -1.2725412845611572, -4.929512023925781, -2.1361379623413086, -0.8203961849212646, 3.6318655014038086, -6.4297895431518555, -7.838647842407227, 1.3934543132781982, 1.459580421447754, -4.80928897857666, -0.23093187808990479, -4.343245983123779, 4.79105281829834, -3.323591470718384, 1.172147274017334, 0.28835558891296387, -2.003131151199341, -1.8507425785064697, 0.39757251739501953, -0.7057929039001465, 1.4487807750701904, -7.367300987243652, 0.8182017803192139, 0.9326395988464355, 0.7540469169616699, -6.372385025024414, -4.076807022094727, -0.13471412658691406, -1.2146973609924316, -3.993537187576294, -4.9051923751831055, -3.3345212936401367, -1.5077422857284546, -0.8268877267837524, -4.148101806640625, -3.4717302322387695, -3.6777548789978027, -3.890313148498535, -2.93123459815979, -9.545000076293945, -3.7407965660095215, 1.6794610023498535, -2.8652822971343994, 2.119307041168213, -5.360722541809082, -1.086961269378662, -2.46478533744812, 2.91031551361084, -3.7273595333099365, -7.70359468460083, -2.8643150329589844, -1.5892428159713745, -4.310506820678711, -6.311509132385254, -1.1837689876556396, 0.08201122283935547, -2.3597357273101807, -2.901400089263916, -3.742933988571167, -2.9221489429473877, -2.1722934246063232, -3.032768726348877, 0.390941858291626, -3.97226881980896, 2.6026711463928223, -2.4724113941192627, -3.7437639236450195, -0.139937162399292, -1.110227346420288, 1.0296330451965332, 4.2711286544799805, 3.5852112770080566, -4.473081588745117, 1.103835105895996, 3.9079484939575195, 0.2547171115875244, 0.9997212886810303, -1.320385456085205, -0.07139086723327637, -3.1493639945983887, -0.32015466690063477, -0.7390990257263184, -2.0633203983306885, 4.008932113647461, -3.4687893390655518, 3.665910243988037, -4.419809341430664, -1.3640830516815186, -6.122169494628906, -0.8811757564544678, 0.887162446975708, 0.38703346252441406, -2.851569414138794, -0.8983954191207886, -4.917898178100586, -0.0283050537109375, -1.7535419464111328, -0.9395158290863037, -0.5726257562637329, 1.2875418663024902, -1.131659746170044, 0.9184319972991943, -7.625304222106934, 0.8995077610015869, 0.7529301643371582, -2.44590425491333, -3.445781946182251, 0.07384252548217773, 0.24954462051391602, -2.889136791229248, -3.1189935207366943, 1.3068299293518066, -0.7023632526397705, -7.445441246032715, -3.2965970039367676, -3.897282361984253, -4.195705890655518, -4.31143856048584, -5.745734691619873, -4.7324910163879395, -5.547745704650879, -1.2613089084625244, -6.616269588470459, -4.312136173248291, -0.2528707981109619, -5.6465559005737305, -3.0164687633514404, 2.304445266723633, 4.380881309509277, -5.33792781829834, 4.9625444412231445, -2.763503074645996, 1.3413128852844238, -2.836373805999756, -3.8993771076202393, 3.1669836044311523, 2.2629237174987793, 1.2763395309448242, -4.795900344848633, -1.2463291883468628, 0.3989741802215576, -1.2372891902923584, -1.363391399383545, 1.5018081665039062, 1.6590209007263184, 1.4303350448608398, 1.216407299041748, -2.8356070518493652, 0.10731053352355957, 0.6431772708892822, -2.337416648864746, -3.0339741706848145, 2.6845011711120605, -2.1202807426452637, -1.7907896041870117, -3.248652458190918, -3.8423404693603516, -2.01308536529541, -3.774554491043091, 0.019729137420654297, -0.9793932437896729, -4.197688102722168, -1.7049570083618164, -4.115127086639404, -4.125392913818359, 2.647235870361328, -1.9886107444763184, 3.736234188079834, 4.545575141906738, -1.0871117115020752, -2.1953160762786865, -1.5677611827850342, -6.229611396789551, -0.5657007694244385, -7.870514392852783, -1.1659539937973022, -4.596480369567871, 1.2451391220092773, -0.2817779779434204, -0.5527080297470093, -4.597805976867676, -2.0710432529449463, -5.22940731048584, -1.5880279541015625, 2.303415298461914, -7.434259414672852, -1.391722321510315, -0.6862715482711792, -5.278788089752197, -4.588702201843262, -3.45127534866333, -2.2163233757019043, -2.072038173675537, -2.682281017303467, -2.103266716003418, -0.6707515716552734, -1.9842183589935303, -1.434515357017517, -3.815136432647705, -2.3561782836914062, -7.520516872406006, 2.470975875854492, -7.0626606941223145, -2.533367156982422, -3.7692973613739014, -7.382280349731445, 0.8469150066375732, -0.26444482803344727, -4.143327236175537, -3.465306282043457, -4.952352523803711, -3.9827733039855957, -4.950851917266846, -0.8724688291549683, -6.034141540527344, -5.063351154327393, -0.46108460426330566, -1.4031234979629517, -1.133849859237671, -4.585663795471191, -8.288597106933594, 2.218870162963867, -4.901438236236572, -4.298336982727051, -3.8378148078918457, 1.421983003616333, 1.057309627532959, -2.778552532196045, -3.297689199447632, -3.7432973384857178, -2.8228955268859863, -2.4251022338867188, -1.3973512649536133, -5.231274604797363, 1.6227145195007324, 0.9616284370422363, 0.8532562255859375, 0.4859585762023926, -9.57951831817627, -3.6291797161102295, -2.6934866905212402, 1.821939468383789, 1.7873871326446533, -1.9304859638214111, 0.7067346572875977, 0.2480611801147461, -7.1367058753967285, -9.55987548828125, -1.668985366821289, 5.494688510894775, -0.47399795055389404, -5.660714626312256, -0.762861967086792, 0.5356695652008057, -1.4704452753067017, -2.2474050521850586, -3.8738107681274414, -1.1989517211914062, -1.147498607635498, -1.9119232892990112, -2.4664626121520996, 2.9423933029174805, -4.264631748199463, -1.9419567584991455, -4.3037919998168945, -0.34975767135620117, -0.8954088687896729, -3.0978431701660156, -4.590801239013672, -1.948974847793579, -3.2470273971557617, 1.4568562507629395, -4.487347602844238, 4.678828239440918, -0.07750380039215088, -2.6192078590393066, -3.9546947479248047, -3.6370317935943604, -0.6206028461456299, 0.771723747253418, -0.8478919267654419, -1.7546799182891846, 3.4904799461364746, -8.49006462097168, -0.5748820304870605, -3.4027364253997803, -1.4042282104492188, -3.6768298149108887, -1.6730831861495972, -6.663101673126221, 2.22200870513916, -5.2885847091674805, -5.220783233642578, -2.2387373447418213, -3.698866844177246, 1.4798917770385742, -1.5098040103912354, 0.45348405838012695, 2.5817694664001465, 6.972454071044922, -2.8720712661743164, -0.4610412120819092, 0.13490700721740723, 0.8013584613800049, -1.9577158689498901, -1.1328188180923462, -1.7212611436843872, -7.78802490234375, -1.6807459592819214, -0.2659628391265869, 0.30490994453430176, -8.08674144744873, 2.817896842956543, -1.6379767656326294, -2.944979190826416, -1.8355926275253296, -1.0082288980484009, -3.539527416229248, 4.6074724197387695, 3.2110724449157715, -0.6203197240829468, -0.3436926603317261, 1.6621155738830566, -1.4266009330749512, -2.5549662113189697, -1.1596448421478271, 0.12654638290405273, -1.1789363622665405, -2.6878509521484375, -5.5061140060424805, 0.34009242057800293, -1.2351659536361694, 1.6456236839294434, 0.11684060096740723, 2.012521266937256, -6.430796146392822, -0.2325291633605957, 4.508339881896973, 0.21260452270507812, 1.5797533988952637, -1.148108720779419, -0.877301812171936, -2.910764694213867, -0.9444715976715088, -0.5602972507476807, -0.26241564750671387, 2.7081432342529297, -0.12171590328216553, 0.6326456069946289, -3.071460485458374, -0.5940420627593994, -5.532318115234375, 0.5502614974975586, -5.019598484039307, -4.266712188720703, -3.3355801105499268, -2.7640700340270996, -2.4981276988983154, -4.741370677947998, -2.846895217895508, 2.1066837310791016, -6.109462738037109, -1.0244386196136475, -4.371744155883789, -8.781266212463379, -7.795183181762695, 4.24411678314209, -6.673805236816406, -4.456502437591553, 1.571645736694336, -2.2099742889404297, -0.586153507232666, -1.1143417358398438, 0.12000823020935059, -1.451244592666626, -0.6838841438293457, -4.728349208831787, -0.017812490463256836, -0.5861181020736694, -1.4539662599563599, 1.2307605743408203, 0.051878929138183594, -0.18557238578796387, -2.8821487426757812, -4.194424629211426, -1.7735737562179565, -1.3656996488571167, -4.564924240112305, -4.931195259094238, -2.838447332382202, -0.878140926361084, 1.560293436050415, -2.8530116081237793, -2.499002456665039, 0.898690938949585, 3.4476418495178223, -1.9777497053146362, -6.5075507164001465, -0.7031694650650024, -6.6527533531188965, -6.318326473236084, 4.013954162597656, 2.9837779998779297, -2.28426194190979, 0.9364557266235352, 0.19844460487365723, -1.4979095458984375, -4.385518550872803, -4.098267555236816, -4.079777240753174, -1.5937941074371338, -3.897993803024292, -2.2182836532592773, -7.958695888519287, -6.6402788162231445, -2.2845516204833984, 3.2638068199157715, -2.6683316230773926, -2.560340166091919, -0.09534716606140137, -3.6537466049194336, 1.2807574272155762, -5.349502086639404, -4.818047523498535, -0.035314083099365234, -2.9682862758636475, -3.989230155944824, 4.968563079833984, -1.986100435256958, -1.60019052028656, -0.7776089906692505, 2.235602855682373, -5.239261627197266, -0.5979781150817871, 1.7864818572998047, -1.2531769275665283, 3.1430325508117676, 0.16324543952941895, -2.320158004760742, -2.3878912925720215, -9.007707595825195, -1.4842097759246826, -3.009432315826416, 2.4761557579040527, -2.7203855514526367, -2.281259059906006, -5.373721122741699, -2.576498508453369, -1.1381266117095947, -4.388467788696289, -6.82311487197876, -1.5023819208145142, 0.5495903491973877, -0.987580418586731, -5.923253059387207, -0.056891441345214844, -5.473840236663818, -0.5518304109573364, -5.737298965454102, -0.7725887298583984, -3.7223453521728516, 2.485107421875, -6.195749282836914, -0.9668203592300415, -3.9951701164245605, -2.3970348834991455, 3.0953612327575684, -1.3522863388061523, -6.45803689956665, -8.813826560974121, -3.7301712036132812, 0.6061904430389404, -3.9229302406311035, -6.860588550567627, -0.8818142414093018, -1.5817208290100098, 2.2161903381347656, -5.996657848358154, -1.1130508184432983, -6.7313642501831055, -1.5784690380096436, -1.1575312614440918, 3.255704402923584, -3.270176887512207, -3.060563564300537, -9.165288925170898, -2.9018189907073975, -1.2674411535263062, 0.911250114440918, 1.7119088172912598, 0.10947751998901367, -4.5462846755981445, 1.2034292221069336, -3.2325823307037354, -7.881627082824707, -6.901492118835449, -0.8219020366668701, -8.756499290466309, -5.873495578765869, -3.016672134399414, -6.442019462585449, -3.317317247390747, 1.880772590637207, -3.413416624069214, -4.416613578796387, 0.37049317359924316, 2.257622718811035, -0.9176716804504395, 1.0972769260406494, 1.6371569633483887, -2.5186352729797363, -5.524259090423584, -2.6321897506713867, -6.28611946105957, -4.213578701019287, -4.597179412841797, 1.130547046661377, -4.200730323791504, -3.6678996086120605, -4.621318340301514, -4.141213893890381, -2.8323848247528076, -1.6794140338897705, -2.9755468368530273, -2.082939863204956, 3.194432258605957, -1.1418402194976807, -2.3223745822906494, -0.893794059753418, -2.7659268379211426, 3.186323642730713, -0.7255134582519531, 0.5434610843658447, 2.634309768676758, -1.37758469581604, -2.2809295654296875, 2.3577027320861816, -2.9821901321411133, 0.3794436454772949, -1.0233807563781738, 1.386138677597046, -4.495450973510742, 3.1627120971679688, 2.1324820518493652, -4.8968000411987305, -3.7350540161132812, 0.44041872024536133, -1.099105715751648, -5.798381805419922, -1.465773105621338, -4.1765594482421875, 0.42699456214904785, 4.629234790802002, -3.851365566253662, -2.767271041870117, 0.6519017219543457, -6.5742340087890625, -2.1154184341430664, -3.0015294551849365, -5.1655497550964355, 3.0398612022399902, -2.511888265609741, -5.218764781951904, 4.736749649047852, -7.398078918457031, -3.9737319946289062, -1.5238593816757202, -6.495543479919434, 6.9446563720703125, -1.2120368480682373, -5.275657653808594, 3.055351734161377, 3.2882018089294434, 0.5501859188079834, -1.1058437824249268, -1.9198931455612183, -5.685291767120361, -0.519174337387085, -1.676110029220581, -7.760683059692383, -1.8759244680404663, -0.940365195274353, 1.1531157493591309, -3.7296864986419678, 2.1733689308166504, -1.0084986686706543, -1.847805142402649, -2.5277066230773926, 0.1144101619720459, 1.6424202919006348, -1.7650337219238281, -3.154860019683838, 0.37424731254577637, 0.9759674072265625, 0.36675429344177246, -4.044709205627441, -2.7790627479553223, 1.1386902332305908, -4.335184574127197, -6.4335479736328125, -3.836609125137329, -4.439691543579102, 1.9343018531799316, -1.8229687213897705, -2.7378101348876953, 2.4686241149902344, -0.0764913558959961, 2.6979026794433594, -5.83782958984375, -4.528648376464844, -4.67254638671875, -9.236953735351562, 3.584054946899414, -3.8068745136260986, -0.8904683589935303, 1.9356122016906738, 0.7643759250640869, -0.8039865493774414, -4.235416412353516, -4.372951984405518, -0.6887643337249756, -4.694064140319824, -2.9452171325683594, -5.24812126159668, -0.6109309196472168, -1.1544232368469238, -2.16282320022583, -0.9710979461669922, -3.2279915809631348, -0.7935285568237305, -0.2562410831451416, -0.8848943710327148, -0.7334252595901489, -1.5736993551254272, -4.548858642578125, -2.4984171390533447, -1.5598925352096558, -3.1273415088653564, -2.7124204635620117, -2.9020161628723145, -2.316915273666382, -1.8936779499053955, -8.089746475219727, -0.8796314001083374, -0.38245534896850586, 1.0635452270507812, -2.8065505027770996, 0.6852574348449707, -7.044981002807617, 4.839405059814453, -3.2258870601654053, 0.882864236831665, -1.9442107677459717, -4.434293746948242, 1.2544207572937012, 1.957683801651001, -1.8297210931777954, -1.9262890815734863, 4.164549827575684, 2.7645883560180664, 2.4660048484802246, -0.3029444217681885, -1.448335886001587, -0.3134796619415283, -1.0333490371704102, -0.5776798725128174, 0.8991684913635254, -1.4765276908874512, 1.6506373882293701, -4.898301601409912, -2.892731189727783, -5.119030952453613, -4.8843994140625, -7.823152542114258, -2.917842388153076, -7.002924919128418, -4.135530471801758, 0.6311376094818115, -3.736182928085327, -2.915719985961914, -4.617241382598877, -3.0531368255615234, 0.18435359001159668, -1.6376075744628906, -1.3787821531295776, 3.734158515930176, -6.468951225280762, 0.9190993309020996, -4.824483871459961, -6.042746543884277, -2.216604232788086, -0.583625078201294, 0.3087804317474365, -0.8215377330780029, -3.467047691345215, -5.653023719787598, -5.891183853149414, -1.3188661336898804, 1.6620259284973145, -0.24867737293243408, 2.007331371307373, -4.979320526123047, -3.1830716133117676, -2.9254541397094727, -3.573315143585205, -2.138547658920288, 2.905461311340332, -5.557986259460449, 1.7444634437561035, -3.772003412246704, -4.147481918334961, -3.4345788955688477, -2.696059226989746, -3.8648509979248047, -2.55928373336792, -2.1561903953552246, 1.1691644191741943, -3.9799318313598633, -5.672993183135986, -4.917591094970703, -4.406797409057617, -4.853992462158203, -3.7322990894317627, -6.388064861297607, -2.8452816009521484, 2.422450065612793, -0.9647789001464844, -6.402560234069824, -0.9327890872955322, 0.6182036399841309, -0.7083816528320312, -5.576029300689697, 3.927600383758545, -2.08205246925354, -5.387704849243164, -0.6501431465148926, -2.7911112308502197, -2.274759292602539, 1.2674593925476074, 0.4961254596710205, 2.7182350158691406, -4.635745048522949, -5.069158554077148, -4.2546515464782715, 1.1318655014038086, -3.0553641319274902, -3.6368157863616943, -4.886469841003418, 1.5496869087219238, 0.33147740364074707, 2.7720131874084473, -6.1726226806640625, -4.023791313171387, -3.307161808013916, -1.3720297813415527, -1.5480053424835205, -0.35470902919769287, -2.1883938312530518, -0.5127018690109253, 5.4166083335876465, 2.193234920501709, -6.134676933288574, -4.717341423034668, -4.177968978881836, -7.171402454376221, -1.766170859336853, -1.1670119762420654, -6.1207146644592285, -4.1201629638671875, -2.009594202041626, 2.2466955184936523, -5.610526084899902, -0.5532385110855103, -0.24618005752563477, -7.261641025543213, 4.00559139251709, -3.459200859069824, 0.9609317779541016, 1.1773009300231934, 1.4536900520324707, -3.9121711254119873, -0.17055010795593262, -1.3836368322372437, -4.475817680358887, -1.512983798980713, -0.3390226364135742, -2.652846574783325, 5.164837837219238, -1.948638916015625, 3.3786725997924805, 0.8897786140441895, -3.4681100845336914, -11.426462173461914, -1.4449965953826904, 2.939493179321289, -1.9708974361419678, -5.351251602172852, -2.90956711769104, 3.683682441711426, 0.03934144973754883, -2.8638916015625, -7.009008884429932, -5.879632472991943, 1.5991196632385254, 0.5967729091644287, -2.436612367630005, -3.6816561222076416, -3.5415854454040527, -5.743216037750244, -4.0524001121521, -7.714508533477783, 1.128340721130371, -7.665355682373047, -4.861190319061279, -3.9545960426330566, 3.0563464164733887, 0.24619793891906738, 6.730722427368164, -0.020871996879577637, -2.139925479888916, -5.994895935058594, 3.0259437561035156, -3.017913341522217, 0.6648359298706055, -3.282750129699707, 0.7435977458953857, -5.7707672119140625, -2.1068599224090576, -0.9299138784408569, -2.9407854080200195, -5.118573188781738, -1.9320827722549438, -10.609759330749512, -6.178318977355957, -0.38043534755706787, -2.019806146621704, 3.5194478034973145, -1.1338030099868774, 4.627562522888184, 0.6311521530151367, -6.545998573303223, 2.167025566101074, 1.2878999710083008, -1.046347737312317, 1.2521986961364746, 4.029453277587891, 3.219027519226074, -4.541388511657715, 1.319216251373291, -6.024857521057129, -4.785430431365967, -0.7403085231781006, -3.3630049228668213, 0.5766382217407227, -1.5871219635009766, 0.9690284729003906, -0.44394993782043457, -0.3777937889099121, -4.706099033355713, 2.727491855621338, 0.06640863418579102, -4.020925998687744, -2.979855537414551, -1.4340132474899292, 3.1835951805114746, 0.19012761116027832, -0.3708462715148926, 0.030832290649414062, 4.288074493408203, -6.901119232177734, 0.9946422576904297, -7.404358386993408, -3.3646163940429688, -2.9980239868164062, -3.6865456104278564, 2.9305648803710938, -5.522587299346924, -7.467639923095703, -6.044841289520264, -2.772801399230957, -5.3564863204956055, 0.18823814392089844, 0.5129363536834717, -1.2183570861816406, -3.983741044998169, 3.690779685974121, 0.8015785217285156, -8.688909530639648, -0.06196951866149902, -0.040404319763183594, -0.780332088470459, 2.6718568801879883, 1.3529314994812012, 0.0003352165222167969, -3.392050266265869, 0.1204526424407959, -4.689341068267822, 1.4080371856689453, 0.3129303455352783, -6.308166027069092, -3.1240501403808594, -0.8448606729507446, 0.9537994861602783, 0.5682322978973389, 2.1603546142578125, -1.0454928874969482, -0.2031993865966797, -3.367105007171631, -5.214087009429932, -5.821484088897705, -1.590879201889038, 3.2788491249084473, -5.107524871826172, -5.002023696899414, 0.19481897354125977, 2.6057257652282715, -3.9472055435180664, 2.3525352478027344, -4.7962446212768555, -0.7713623046875, -2.204023599624634, -5.35468864440918, 1.507436752319336, 2.4633922576904297, -5.702155113220215, -4.267232894897461, -5.824540138244629, -0.288163423538208, -0.9097466468811035, -0.3183932304382324, 1.1878697872161865, 0.3687248229980469, 0.9295415878295898, -2.5720906257629395, 3.2094764709472656, -2.2717509269714355, -2.934368371963501, -2.770415782928467, 1.1992011070251465, -3.5988807678222656, 0.04319190979003906, -1.5126053094863892, -6.822454929351807, 1.823498249053955, -2.2978315353393555, -6.057941436767578, -2.61562180519104, 2.364516258239746, 0.20000290870666504, -2.530698299407959, -1.6965855360031128, -6.701987266540527, -4.506216526031494, -4.22937536239624, -7.320868015289307, -1.4277970790863037, -3.052302360534668, -1.463364601135254, -3.445279836654663, 1.7682127952575684, -3.504723072052002, 1.3256609439849854, -4.5681538581848145, -1.3915090560913086, -1.3745145797729492, -5.010296821594238, -3.4579615592956543, -2.0606210231781006, -0.30792784690856934, -3.0391411781311035, -0.8124581575393677, -4.0403361320495605, -0.85273277759552, -0.6273863315582275, -2.4305436611175537, -2.486961603164673, 2.1182541847229004, -1.863784670829773, -8.97812557220459, -1.5836939811706543, -0.005460500717163086, -1.4608039855957031, -3.2574679851531982, -3.813788652420044, 1.4704575538635254, -1.1502939462661743, -7.778481483459473, 0.08225154876708984, 4.952051162719727, 0.3323221206665039, -5.93412971496582, -2.8569440841674805, -1.4161041975021362, -5.4770588874816895, -4.771570205688477, -2.9340977668762207, -7.277054309844971, -1.4105918407440186, -1.423206090927124, -2.4306015968322754, 0.07991147041320801, -3.2481608390808105, -0.32831883430480957, -0.10411548614501953, -1.2554476261138916, -3.395906448364258, 1.8494410514831543, 1.0652306079864502, -4.51021671295166, 1.9670772552490234, -2.1710121631622314, 3.591902732849121, -0.06763815879821777, -5.5447893142700195, 0.1323719024658203, -1.596069097518921, -3.723328113555908, -7.997673034667969, 0.21904850006103516, -0.8083956241607666, -1.0777587890625, -4.044480323791504, 0.7498600482940674, -1.355459451675415, 0.5972728729248047, -0.7136735916137695, 1.1421892642974854, -3.155036211013794, -0.11975467205047607, -0.9453444480895996, -3.7365593910217285, 6.029925346374512, -3.618464231491089, -4.882119178771973, -3.182542324066162, 0.6398584842681885, -0.580420732498169, -3.3602261543273926, -0.5505857467651367, 1.1179978847503662, -3.868934154510498, -0.21098661422729492, 2.2795162200927734, 3.3696422576904297, -6.746448040008545, 4.735836505889893, -5.536113262176514, 1.5113210678100586, -5.337023735046387, -1.8730871677398682, 3.542886734008789, -3.3232014179229736, -1.7913342714309692, 2.4672770500183105, -0.5636124610900879, 1.4910922050476074, -8.565054893493652, 1.1584327220916748, 0.41483139991760254, 2.4622325897216797, -0.24458825588226318, -3.2610318660736084, -1.5204885005950928, -0.5464919805526733, -4.149636268615723, -1.1615543365478516, -5.466615676879883, 2.7990775108337402, -0.5074189901351929, -1.1895792484283447, 2.1223278045654297, -0.6104623079299927, -1.335582971572876, 2.029191493988037, -3.5055508613586426, -2.2274301052093506, -3.463402509689331, -2.0398762226104736, -3.612278699874878, -1.4387822151184082, 0.7503769397735596, -4.764996528625488, 0.6821980476379395, -0.4186704158782959, -3.8576910495758057, -1.2421389818191528, 0.3640141487121582, -0.9143843650817871, -9.059786796569824, -1.6892638206481934, -1.1441975831985474, -3.3487167358398438, -0.09038996696472168, -1.3784823417663574, -2.367413282394409, -4.448606014251709, -0.30128300189971924, -4.821580410003662, -1.7030277252197266, -5.165946006774902, 0.07438135147094727, -2.82283616065979, -1.887259840965271, 0.0832369327545166, -3.7274885177612305, 4.070828914642334, 2.2425971031188965, 3.665006637573242, 4.442619323730469, -2.1511685848236084, -5.170649528503418, -2.8384790420532227, 0.5645837783813477, 2.8655834197998047, -4.515463829040527, -2.112576484680176, 1.5241780281066895, 1.4032254219055176, -3.4167978763580322, -2.071855306625366, -3.5406343936920166, -3.2549121379852295, -1.9685723781585693, -2.314861536026001, -3.6611111164093018, -2.04799747467041, -0.9965019226074219, -3.442502498626709, 0.45612335205078125, -3.1380035877227783, -1.8612136840820312, -0.7350476980209351, -4.534080505371094, -3.277426242828369, 1.8810830116271973, -4.310574531555176, -3.514425039291382, -8.344010353088379, -2.3015737533569336, -1.5459723472595215, -3.409210681915283, -1.6010875701904297, -7.051489353179932, -1.844085693359375, -1.5281145572662354, -3.1727943420410156, -2.539433717727661, -6.181446552276611, -3.8864364624023438, -2.1745221614837646, -6.157005310058594, -0.8055261373519897, -5.617835521697998, -5.561834335327148, 1.8213860988616943, -0.2947409152984619, -6.980355262756348, -1.5740703344345093, -0.7754089832305908, -4.0976972579956055, -2.5358099937438965, -1.4922198057174683, -2.6552810668945312, -6.890708923339844, -4.488274574279785, -4.764638900756836, 3.0414538383483887, -2.9759252071380615, -3.3046319484710693, -0.9888712167739868, -6.337414741516113, 1.063715934753418, -0.9789189100265503, -2.3283042907714844, -3.7100296020507812, -2.68241548538208, -7.726619720458984, -3.17744779586792, -4.527393817901611, 5.082892417907715, -1.6181249618530273, 0.9527602195739746, -4.33026647567749, -5.356908321380615, 0.0643148422241211, -5.789769172668457, -3.616408586502075, -4.924940586090088, -1.0720728635787964, 2.504683494567871, 1.870950698852539, -3.3917157649993896, 3.120762348175049, -1.791457176208496, -2.1430561542510986, 0.4109938144683838, -0.010645389556884766, -1.3383145332336426, 0.5148491859436035, -4.237123012542725, -2.331498861312866, 1.7968170642852783, -2.749485492706299, -3.526378631591797, -6.579581260681152, -3.7288217544555664, 0.7863636016845703, -6.678783893585205, 2.5734634399414062, -1.2395710945129395, 1.023080587387085, -5.6640849113464355, -1.3044853210449219, 2.8968048095703125, -6.279407978057861, -5.302103042602539, -1.1038572788238525, 0.9568338394165039, -5.6337056159973145, -1.6793502569198608, -1.5590918064117432, 0.21634387969970703, -0.9473836421966553, -4.123075485229492, 0.9978094100952148, -5.690095901489258, -0.2622048854827881, -4.182950973510742, -1.2771258354187012, 3.651717185974121, -4.195220470428467, -2.0591189861297607, -2.702082633972168, -5.974328994750977, 1.8831424713134766, -0.010793209075927734, -3.187372922897339, -1.7410757541656494, -4.5763444900512695, -3.254183769226074, -6.353128433227539, -0.5976042747497559, -2.0107128620147705, -5.002392768859863, 0.9086809158325195, -10.216804504394531, -3.9499759674072266, -5.767701148986816, -2.275949716567993, -4.8020124435424805, 1.7049226760864258, -2.7033603191375732, -11.522783279418945, -0.8159133195877075, -2.618560791015625, -5.570744037628174, 0.025612831115722656, 2.2457633018493652, -1.89010488986969, 2.1966094970703125, -1.4928724765777588, -1.7115049362182617, -0.8962442874908447, -2.961214542388916, -0.8246256113052368, -3.9161980152130127, -3.344712495803833, -1.189469337463379, -2.358050584793091, 2.2419700622558594, -3.4017629623413086, -2.0356340408325195, 0.7164487838745117, -0.44319045543670654, -4.767283916473389, -2.886962413787842, -7.801840305328369, 0.6883041858673096, -2.17438006401062, -3.6181209087371826, -4.111654281616211, -5.746740341186523, -1.0061428546905518, -2.9026107788085938, -3.027475357055664, -4.474493026733398, -4.898168563842773, -2.6318609714508057, -2.8255844116210938, -4.17094612121582, -9.399041175842285, -5.950965881347656, -4.192848205566406, -5.0448784828186035, 0.15160465240478516, -3.2735629081726074, -3.002178192138672, 0.31524085998535156, -1.7495527267456055, -4.604254245758057, -0.09791433811187744, -0.4936530590057373, -6.595323085784912, -5.7908477783203125, -3.704064130783081, -1.9244489669799805, 0.6520991325378418, 2.0806360244750977, -4.424987316131592, -4.385870456695557, -2.688573122024536, -1.2580443620681763, -3.4762792587280273, -0.5494630336761475, -1.354846715927124, -2.798123836517334, -3.6174914836883545, -0.5148811340332031, 0.3873000144958496, -3.7247986793518066, -3.5397629737854004, -4.120612144470215, -1.545243263244629, 2.851938247680664, 0.28905177116394043, -4.384672164916992, -1.4711012840270996, 0.2606534957885742, -1.5105026960372925, -5.050641059875488, 3.9856438636779785, -0.6124989986419678, -3.9325966835021973, -4.483031272888184, -4.804465293884277, 1.909961223602295, -3.4117977619171143, 3.435607433319092, -0.6810671091079712, -1.036421537399292, -0.9147735834121704, 2.09682559967041, -0.15975534915924072, -5.296328544616699, 0.03667807579040527, -5.0129499435424805, -3.4149954319000244, -1.8527494668960571, 3.262838840484619, -5.362066268920898, 3.580258369445801, -1.3822009563446045, -4.681848526000977, -0.13701093196868896, -0.7949092388153076, -2.696535348892212, -6.203527450561523, -0.07998204231262207, 0.29410266876220703, -4.0797953605651855, -7.217653274536133, 5.280738353729248, 3.3619818687438965, -8.267083168029785, -0.39404773712158203, -6.963903427124023, -5.578477382659912, -0.5148745775222778, 3.8472204208374023, -6.8396806716918945, -1.3975002765655518, -1.062528371810913, 0.38907742500305176, -7.091275691986084, -0.2657310962677002, -2.7367236614227295, -5.202924728393555, -1.8541531562805176, -3.134420156478882, -5.404760360717773, -5.209129333496094, -1.361580729484558, -1.3372489213943481, 1.6258482933044434, -4.512609481811523, -5.712597370147705, -6.1232099533081055, -1.5655494928359985, -2.550309181213379, -1.8227885961532593, -3.2218685150146484, 1.3083975315093994, -3.1284799575805664, -2.914870262145996, -4.224416732788086, -5.969104766845703, -2.5170364379882812, -2.9945883750915527, -6.510083198547363, -2.6512410640716553, -1.8004357814788818, 3.970762252807617, -0.2142164707183838, 1.175379753112793, 0.2887284755706787, -3.1758203506469727, 5.002260208129883, -8.251452445983887, -3.0533695220947266, -3.1590576171875, -3.8052120208740234, 0.9761157035827637, 0.41982102394104004, -0.42224574089050293, -2.018409252166748, 2.469780921936035, 1.403334617614746, -5.415956974029541, -4.792542457580566, -3.401251792907715, -0.16019463539123535, -4.498650550842285, -3.2180821895599365, -4.251615524291992, 4.5898284912109375, -3.3426403999328613, 1.542726993560791, 0.8047747611999512, -6.173927307128906, -2.6777184009552, 0.23670101165771484, -0.9458520412445068, -3.729198455810547, -1.866997480392456, -2.9954981803894043, -6.4589715003967285, 2.4474411010742188, 2.199124336242676, -2.5614709854125977, 1.444979190826416, -0.6677892208099365, 1.1791322231292725, 3.1156792640686035, -0.06769907474517822, -0.12670409679412842, 2.4323549270629883, 0.21695280075073242, -3.445646286010742, -3.924182891845703, -1.5667012929916382, -1.892842173576355, -2.7337112426757812, -1.622338056564331, -1.9555155038833618, 3.554112434387207, -3.215589761734009, 1.8539714813232422, -2.1018459796905518, 3.816417694091797, -0.16336774826049805, 1.215688943862915, -4.312505722045898, -1.0729142427444458, -2.2664499282836914, -0.07065582275390625, 1.5179901123046875, 0.8318819999694824, -1.773140788078308, 2.8466696739196777, -3.193906784057617, -0.6382746696472168, 3.32889986038208, -1.4265491962432861, 3.4071784019470215, -7.079161167144775, -3.3593482971191406, 0.7726871967315674, -3.440791368484497, -1.197467565536499, -2.2353410720825195, -5.80719518661499, -5.132859230041504, -3.970079183578491, -2.1498866081237793, 5.007645130157471, -4.234831809997559, -1.9641213417053223, -1.7448481321334839, 0.4581265449523926, -2.1027398109436035, -1.3293704986572266, 3.0735182762145996, -3.0528838634490967, -4.465176105499268, -2.964505672454834, 3.7112488746643066, -2.8530006408691406, -0.6748858690261841, -3.479750633239746, -1.4396917819976807, -3.2498886585235596, -4.694486618041992, -5.125282287597656, -3.539318561553955, -1.669076681137085, -0.8681806325912476, -1.070813775062561, 0.5017473697662354, -5.081998348236084, -6.763965606689453, 0.9302263259887695, -1.9062702655792236, -4.7103705406188965, 1.2092366218566895, -0.06417930126190186, -1.6776010990142822, -6.110045909881592, -2.6456849575042725, -4.9531402587890625, -2.9029760360717773, -5.2659525871276855, -3.6269798278808594, -7.532581329345703, 0.809096097946167, -2.7693467140197754, 1.554675817489624, 2.5678186416625977, -3.4430184364318848, -1.0895836353302002, -5.914947032928467, -0.4471445083618164, 1.1935160160064697, 3.5261893272399902, -4.4400787353515625, -4.457530975341797, -1.5785491466522217, 1.9247407913208008, -0.966661810874939, 4.03363561630249, -2.23809814453125, -3.6480891704559326, 2.333016872406006, -3.8374505043029785, -5.8537492752075195, -0.11030220985412598, 0.49768996238708496, -5.068965435028076, -0.07522463798522949, 2.1115498542785645, -4.550430774688721, -0.13584649562835693, -0.7329468727111816, -5.781764030456543, -5.731273651123047, 4.3679890632629395, -2.5427396297454834, -2.498501777648926, -2.2950472831726074, -2.4123151302337646, -5.031853675842285, -1.591364860534668, -4.67210578918457, -1.6942387819290161, -0.7807362079620361, -0.718341588973999, -2.18414044380188, -3.7535476684570312, -7.306092262268066, 1.0166316032409668, -6.303233623504639, -5.528106689453125, -0.7784055471420288, -4.046164035797119, -4.1739501953125, -7.155458450317383, 2.836750030517578, -2.2843470573425293, -1.057233452796936, -4.411052703857422, -2.2325572967529297, 4.275681018829346, -4.999675750732422, -4.965973854064941, -3.2619259357452393, -1.4129372835159302, -1.3748033046722412, -11.259748458862305, -6.270883083343506, -2.601863384246826, -2.3666512966156006, -3.864283323287964, -1.8742594718933105, -0.4961240291595459, -3.4693822860717773, -2.024405002593994, -4.975220203399658, 1.48736572265625, -4.749739170074463, -0.008931517601013184, 3.761471748352051, -1.4472744464874268, -2.37749981880188, -0.28125, -0.607506275177002, -3.077899217605591, -2.1251327991485596, -1.064563274383545, -5.138425350189209, -6.209748268127441, -1.8604581356048584, -3.2124950885772705, -3.9310123920440674, -5.06023645401001, -4.380023002624512, -4.703397274017334, -0.8528820276260376, -3.5969672203063965, -1.1920509338378906, -4.959099769592285, -3.5042691230773926, 1.4711103439331055, -6.58372688293457, -0.5150213241577148, 3.669252395629883, -2.720588207244873, -2.534498691558838, 0.23891091346740723, -1.9435957670211792, 1.5101537704467773, -3.6177151203155518, 2.3238744735717773, -1.251622200012207, -2.0517513751983643, -4.39283561706543, -0.7553561925888062, -2.869819164276123, -0.00495457649230957, 6.6183576583862305, 0.3887746334075928, -3.6269872188568115, -1.894255518913269, -4.851442337036133, -3.337843894958496, -1.0971146821975708, -1.8193633556365967, -1.9466571807861328, -2.757768154144287, -1.6123547554016113, -1.798309087753296, -5.478933811187744, -3.2075083255767822, 3.721895217895508, -1.8260632753372192, -1.896616816520691, -1.3000547885894775, -2.6449007987976074, -1.1797266006469727, -3.2013449668884277, -5.0631303787231445, 3.213632106781006, -5.204017162322998, 2.3977017402648926, -2.594320297241211, 1.3279883861541748, -8.03051471710205, -5.143109321594238, -3.6129815578460693, -2.8941738605499268, 4.406855583190918, 0.4072284698486328, -5.334907054901123, -0.6204091310501099, -1.221908450126648, 0.07693314552307129, -2.9182307720184326, -4.932719707489014, 1.4244728088378906, -1.6743537187576294, -0.6046345233917236, -3.0033018589019775, -0.996999979019165, 0.21546530723571777, -3.3134138584136963, 1.828214168548584, -3.868283271789551, -3.1576695442199707, -0.6810343265533447, -1.9784469604492188, -2.8941588401794434, -6.5311174392700195, -1.454150676727295, -1.5702173709869385, -1.3721452951431274, -3.735171318054199, -0.44529247283935547, -3.9710586071014404, -2.1440255641937256, -6.150651454925537, -9.626692771911621, -3.628614664077759, -6.375730037689209, -2.0778913497924805, -2.996110677719116, -0.5785185098648071, 1.760655403137207, -6.3999528884887695, 0.16204547882080078, -6.6200666427612305, -2.358851909637451, 2.0496721267700195, 1.8541977405548096, -2.4047861099243164, -2.925262928009033, -6.867547988891602, 2.3109688758850098, -4.620728969573975, -5.557720184326172, -0.39380645751953125, -4.5497660636901855, -2.9152767658233643, -0.047609567642211914, -2.90496826171875, -0.9830281734466553, -0.22972369194030762, -1.739104986190796, -3.356175422668457, -3.7081868648529053, 0.8728361129760742, -7.8570990562438965, -4.845602989196777, -5.389244556427002, -2.7892613410949707, 1.2510066032409668, 0.5513291358947754, -0.6267235279083252, -1.8124744892120361, -1.6095077991485596, -1.2682288885116577, -1.7732760906219482, 1.3232841491699219, 3.6838607788085938, -2.277130126953125, -1.0463519096374512, -1.0300936698913574, 0.923095703125, -5.06340217590332, -6.279973983764648, -3.4471800327301025, 3.3844246864318848, -2.440295696258545, -1.749861717224121, -5.831212520599365, -6.863723278045654, -0.5976111888885498, 1.8603787422180176, 3.725963592529297, 2.4145679473876953, -4.464286804199219, 0.5867505073547363, 5.111315727233887, 2.584768295288086, 2.0329132080078125, -2.6298327445983887, -3.4653618335723877, 0.6761748790740967, -1.5053887367248535, -6.089043140411377, -2.5411393642425537, -0.7537513971328735, -6.08379602432251, -6.665431976318359, -4.353071212768555, -2.6384835243225098, -2.897736072540283, -2.8013925552368164, -2.667020082473755, -6.397677421569824, 0.10051202774047852, -0.4027066230773926, -2.827970266342163, -0.7083073854446411, -1.9631891250610352, 1.9899253845214844, -5.333580017089844, 4.208349227905273, -2.7222158908843994, -5.606906890869141, -0.053183794021606445, 0.5347628593444824, -1.4343290328979492, -0.7109849452972412, 2.1727914810180664, 2.715487480163574, -1.2183194160461426, -2.5441741943359375, -3.8144657611846924, -3.86944842338562, 1.9409494400024414, -2.714015007019043, -1.803446650505066, -1.732771873474121, -6.184917449951172, -1.5753675699234009, -1.5192782878875732, -4.166838645935059, 1.005211353302002, -1.7716758251190186, -5.067442893981934, -1.3930258750915527, -4.167823791503906, 1.5879340171813965, -3.7854223251342773, -2.0902771949768066, -3.173503875732422, -0.7032092809677124, -7.126701354980469, -0.3774380683898926, 2.1257143020629883, -3.8828697204589844, 1.8578040599822998, 4.011406898498535, -5.02910852432251, -4.365182399749756, -2.1979801654815674, -8.735462188720703, -1.5911223888397217, -0.6662427186965942, 0.8792545795440674, -6.0060601234436035, -0.749494194984436, -2.9328219890594482, -3.5230002403259277, -6.415426254272461, -2.923466444015503, -1.6342463493347168, -1.4544757604599, -0.08691656589508057, 0.7222869396209717, 1.1724846363067627, -0.3243255615234375, -4.323142051696777, -1.708596110343933, -0.7864580154418945, 3.6297688484191895, 0.43728184700012207, -0.12870323657989502, -2.857402801513672, 0.23555517196655273, -3.158219814300537, 3.243143081665039, -6.076749801635742, -4.191490650177002, -2.5747995376586914, -5.8523759841918945, 0.1739487648010254, -3.830850601196289, -4.394420623779297, -1.730644702911377, -4.904929161071777, 2.2799081802368164, -4.032229423522949, -7.675099849700928, -1.7578158378601074, -3.0847816467285156, -5.0107421875, -1.175632357597351, 1.4767441749572754, -2.427960157394409, 1.5272893905639648, -5.2353668212890625, -6.134519577026367, 0.1628572940826416, -7.002996921539307, -1.4261915683746338, 2.9947328567504883, 0.10350155830383301, -4.316486358642578, -0.9371762275695801, -0.394758939743042, 3.147979736328125, -0.2756812572479248, 4.373542785644531, -3.255718946456909, -5.209931373596191, -7.1668596267700195, -1.8372282981872559, -6.327358245849609, 0.7315149307250977, -2.266204833984375, -0.7782901525497437, -5.725227355957031, -5.350199222564697, -2.9131693840026855, -1.9843175411224365, -4.796239852905273, -1.53956937789917, -0.504679799079895, -1.403907060623169, 2.751361846923828, 2.373368740081787, -3.240938186645508, -2.1587376594543457, -2.5186281204223633, -0.8096175193786621, -5.897255897521973, 3.2787623405456543, -0.4075334072113037, -3.669938325881958, -0.553264856338501, -2.554903507232666, -0.8902955055236816, -7.198312282562256, -2.83879017829895, -1.5760772228240967, 4.83325719833374, -2.1890952587127686, -4.435176849365234, 0.20636606216430664, 1.112717866897583, -1.803086519241333, -0.5023318529129028, -1.2471933364868164, 0.009423255920410156, -2.1538918018341064, -5.996786594390869, 2.8669986724853516, -8.633727073669434, -7.868288040161133, -3.8519697189331055, -2.2067551612854004, -3.387833595275879, -6.106418609619141, 0.18057870864868164, -1.0835824012756348, -0.06198406219482422, -1.2178418636322021, -3.7809183597564697, -3.3378806114196777, -4.012667179107666, -1.375819206237793, -6.844837665557861, -4.974390029907227, -3.229860782623291, -0.6259641647338867, 4.030173301696777, -0.5565774440765381, 1.2967052459716797, -4.152983665466309, -1.6673870086669922, -5.24564790725708, -0.09716510772705078, -2.731827735900879, -0.7390090227127075, -0.7990880012512207, -3.4396233558654785, 1.2466950416564941, 1.6243467330932617, -7.115294933319092, -2.4929418563842773, -4.215845108032227, -7.2557783126831055, 1.858642339706421, -1.37111234664917, 0.7691981792449951, -5.578957557678223, -2.9589996337890625, 0.780541181564331, -4.227097034454346, -4.5664520263671875, -5.67794132232666, 1.5350432395935059, 0.529665470123291, -2.422574520111084, -6.125422954559326, 0.0343470573425293, -3.4616730213165283, -1.0048892498016357, -1.379075288772583, 0.8426597118377686, -5.758269309997559, -2.8734865188598633, -2.149238348007202, -5.207828998565674, -1.3819515705108643, -1.2564518451690674, -5.68678617477417, -3.734287977218628, -1.0042202472686768, 2.055959701538086, -2.4959681034088135, -1.0514296293258667, -1.1389883756637573, -1.954589605331421, -8.24974250793457, 0.09651923179626465, -2.8063859939575195, 1.1414875984191895, -4.629654407501221, -0.371021032333374, -5.181920528411865, -4.1442155838012695, 2.343893527984619, -2.4213879108428955, -2.473140239715576, -2.1174476146698, 2.813063621520996, -1.3682544231414795, -4.644754886627197, -2.7635750770568848, -3.3577022552490234, -1.1287882328033447, 0.47614312171936035, -1.7636113166809082, -6.671781539916992, 0.5437207221984863, -0.9388046264648438, -5.308178901672363, 1.122180700302124, 0.41504597663879395, -3.611006021499634, -0.16738033294677734, -1.158996343612671, -1.1234875917434692, -3.2783613204956055, 0.37670040130615234, 1.21431303024292, 3.1119117736816406, 1.7959613800048828, -0.05339038372039795, -4.426457405090332, -3.909148693084717, -7.674544334411621, -0.9612646102905273, -7.971750736236572, -0.6965882778167725, -2.6854748725891113, 1.201880931854248, 0.9976661205291748, 1.4822275638580322, 1.1159534454345703, -0.28052401542663574, -0.46362733840942383, -0.8221439123153687, -4.4490766525268555, -5.151294708251953, -2.9784560203552246, -0.003264188766479492, -3.3835511207580566, 3.7784042358398438, 1.8552520275115967, -2.128243923187256, -3.0502617359161377, -4.7798051834106445, -4.416849136352539, -1.128376841545105, 4.346623420715332, -4.392843723297119, -4.1192827224731445, -4.7309136390686035, -0.9565725326538086, -1.5268322229385376, -2.0370960235595703, -4.334340572357178, 2.943606376647949, -4.600887298583984, -4.631649971008301, -5.21639347076416, -6.160906791687012, -1.4942543506622314, -0.7340563535690308, -0.941007137298584, -8.598651885986328, -2.620591878890991, 1.0190980434417725, 3.3840551376342773, -2.710498809814453, -2.0399160385131836, -1.1338963508605957, -2.2240090370178223, -1.5118135213851929, -7.060111999511719, -0.6132974624633789, 1.5510249137878418, -4.8357391357421875, -2.352426528930664, -2.6096837520599365, -2.0107529163360596, -1.492621898651123, -8.180171966552734, -2.873950958251953, -6.378378868103027, -2.9821667671203613, 0.6828036308288574, -2.759488105773926, -6.29042911529541, -4.610280990600586, -1.6113007068634033, -3.914762496948242, -2.273689031600952, -3.4953551292419434, -2.0817370414733887, 2.143860340118408, 2.628633975982666, -1.135754108428955, 3.2058396339416504, -0.09739899635314941, -5.590515613555908, -2.3370227813720703, -4.485229969024658, -0.8850443363189697, -1.0002518892288208, -1.98086416721344, -3.3289499282836914, -0.4129800796508789, -2.770050525665283, 0.5962622165679932, -3.5883803367614746, -7.5193095207214355, 0.8861236572265625, -0.6823139190673828, -3.1980957984924316, -0.662973165512085, 1.2064766883850098, -0.3663215637207031, -1.7981404066085815, -3.8240957260131836, -8.161186218261719, -1.688499927520752, -1.9518942832946777, -2.5610899925231934, -1.5741465091705322, 0.16442632675170898, 3.0079450607299805, -2.0082318782806396, -1.501056432723999, 4.042270660400391, 0.42179298400878906, -4.129946708679199, 1.226151466369629, -3.5874626636505127, 1.6321821212768555, -3.576244592666626, 0.4953935146331787, 0.6479418277740479, -1.93784499168396, -2.293468713760376, -0.1693730354309082, 1.8294873237609863, -2.0592195987701416, 0.01240396499633789, 0.5614213943481445, -7.84757661819458, -7.901458263397217, -3.1436634063720703, -3.1961684226989746, -9.286382675170898, 0.30237483978271484, -3.6926169395446777, -5.145749568939209, 1.8202638626098633, -6.287482261657715, -1.2990690469741821, -5.614720344543457, 4.3493804931640625, -5.717865467071533, -4.292977809906006, 0.335860013961792, -2.758997678756714, 2.0505733489990234, -4.1319580078125, -1.286109447479248, 1.394202470779419, 1.0543444156646729, -1.215925931930542, -5.826731204986572, -1.7403709888458252, -7.103971481323242, -3.419287919998169, 2.7475438117980957, -3.6461808681488037, -3.981306552886963, -2.3341455459594727, 1.2133703231811523, -5.537367820739746, -3.5324807167053223, -5.476294040679932, -3.7160122394561768, -5.23670768737793, -5.773921966552734, -1.2974770069122314, 0.0596315860748291, -1.7610660791397095, 2.298884391784668, -1.167619228363037, -1.1504149436950684, 0.31537437438964844, 2.1899795532226562, -0.7915654182434082, -2.6836891174316406, 6.071952819824219, -5.247068405151367, -1.1773899793624878, 1.8842813968658447, -0.4938175678253174, -3.8176028728485107, 0.6892569065093994, 1.0222938060760498, -4.141454219818115, -4.250059604644775, -1.6835367679595947, -0.7591397762298584, -3.9421727657318115, -0.599929690361023, 2.794034004211426, -1.5920023918151855, -3.1055874824523926, -8.399600982666016, 1.0571036338806152, 2.4250998497009277, -6.671114921569824, -7.838079452514648, 0.7868976593017578, -2.6108498573303223, -9.456554412841797, 2.004899501800537, -1.5544886589050293, 1.1488494873046875, -1.1712040901184082, -0.14715313911437988, -3.1182446479797363, -3.0736560821533203, -0.6786746978759766, 2.2002687454223633, -4.391100883483887, -7.854801177978516, -4.732263565063477, -0.7234102487564087, -3.3226470947265625, -4.401972770690918, -4.854663848876953, -5.390814781188965, -2.594573497772217, -6.607541084289551, 5.066987037658691, -2.960028648376465, -3.7414705753326416, -1.7085059881210327, -1.1193525791168213, -0.5681700706481934, -3.5228464603424072, 0.8929386138916016, -2.987677574157715, 1.0736360549926758, -2.6935534477233887, -1.4053502082824707, -8.7719087600708, -2.519076347351074, -1.1808736324310303, -8.045265197753906, 2.910372734069824, -4.040498733520508, -5.713161468505859, -3.239328384399414, -3.7131152153015137, -2.028614044189453, -6.617154121398926, 1.1875581741333008, -3.5126683712005615, -4.3637518882751465, 1.0648889541625977, -2.5583083629608154, -4.107413291931152, -2.5783987045288086, -2.500699043273926, 2.1941986083984375, 0.06003308296203613, -3.6794095039367676, -4.533947944641113, -5.4915452003479, -0.25548338890075684, -2.399303913116455, -0.24300742149353027, -8.369362831115723, -1.2047969102859497, -5.057027816772461, -0.6272714138031006, -2.873073101043701, -7.321369171142578, -1.4995225667953491, 0.37891268730163574, -2.0950253009796143, -2.5511817932128906, -3.803661346435547, -3.437188148498535, -2.308537006378174, -0.39125895500183105, -2.936739921569824, -1.4640922546386719, -3.3546628952026367, -2.5673463344573975, -4.367397308349609, -5.109439849853516, -5.106554985046387, 0.6998448371887207, -0.06931483745574951, -3.519942045211792, -2.258094310760498, -0.8612779378890991, -9.640649795532227, -3.505678653717041, -3.1791582107543945, -2.97800874710083, -3.0473735332489014, -1.5519788265228271, -6.545164585113525, -1.161596417427063, -2.899766445159912, -1.3817863464355469, -0.8171951770782471, -3.403207302093506, -3.4447262287139893, -6.490047454833984, -0.5093898773193359, 1.8468503952026367, -6.834926605224609, -2.656752586364746, -3.097775459289551, -0.82073974609375, -7.870081901550293, -1.3510321378707886, 0.39667272567749023, 2.2888641357421875, -3.381620407104492, -5.405283451080322, -3.172755241394043, 1.9121947288513184, 1.6685924530029297, -2.789346694946289, -4.41824197769165, 1.035299301147461, -4.173864364624023, -3.845848560333252, -3.6818647384643555, 0.03129076957702637, -1.678903579711914, 0.8612513542175293, -8.07161808013916, -5.911172866821289, -5.955696105957031, 2.8910908699035645, -1.7224698066711426, -1.9673267602920532, -6.355437278747559, -2.885197639465332, -3.2383105754852295, -1.655810832977295, -1.2550029754638672, -8.482650756835938, -1.962395429611206, -1.7421367168426514, -1.4003517627716064, -4.215935707092285, -3.15439510345459, -0.7227797508239746, -2.2638697624206543, -4.42935848236084, -4.326243877410889, 1.8238325119018555, -1.3071564435958862, -6.883294105529785, -5.4065141677856445, -4.600321292877197, -2.168783664703369, -0.09256851673126221, 0.6211323738098145, -4.005495071411133, 1.9106042385101318, -3.4975948333740234], [-6.483878135681152, -1.163212776184082, -6.101969242095947, -1.6991045475006104, -1.2649009227752686, -2.0355794429779053, 2.828367233276367, 1.5367183685302734, -4.994906902313232, -5.8699951171875, -1.1109161376953125, -3.2607412338256836, -5.131527900695801, 0.6766800880432129, -1.479212999343872, -0.027253150939941406, -5.585883140563965, -2.2105910778045654, 0.23632287979125977, -2.320723056793213, -3.825251579284668, -3.4656848907470703, -2.1619508266448975, 4.211064338684082, -2.124997615814209, -4.0929107666015625, -1.1791532039642334, -6.021265983581543, -5.410212516784668, -0.5320694446563721, 9.229986190795898, -2.4772348403930664, -5.587770462036133, 1.0885004997253418, -2.956467628479004, -4.749143600463867, 0.7098865509033203, -1.158644676208496, 0.37425947189331055, -5.6364898681640625, -0.739973783493042, -2.9915666580200195, 3.6485328674316406, -1.6501140594482422, 0.739966630935669, -2.682892322540283, -7.888509750366211, -0.7716517448425293, -4.28772497177124, -5.213007926940918, -2.0622591972351074, -5.201602935791016, -1.6541869640350342, 0.942835807800293, -4.651433944702148, 2.9371652603149414, -7.699122428894043, -3.721571207046509, -4.938144683837891, -5.842159748077393, 2.1095080375671387, -10.704404830932617, -5.892250061035156, -2.893711805343628, -5.756704330444336, -5.501523971557617, -0.07041692733764648, 1.0652952194213867, 0.3448598384857178, -0.6490260362625122, -3.622976303100586, -1.1536763906478882, 1.5238676071166992, -3.349245071411133, -2.247495412826538, 3.314617156982422, -1.296910047531128, 0.43134236335754395, -3.535593032836914, -0.8574879169464111, -2.6050634384155273, -9.39598274230957, -6.709144592285156, -6.300196647644043, -2.927539825439453, -2.568695545196533, -0.8003736734390259, -2.742441177368164, -5.5565266609191895, -5.662142276763916, -0.3808152675628662, -7.658631324768066, 1.991243839263916, -3.9034459590911865, 1.2458643913269043, -7.91110372543335, 2.0738778114318848, 4.3216552734375, 0.8247566223144531, -0.26423561573028564, -2.678774118423462, -3.1279563903808594, -3.7862868309020996, -4.141741752624512, -0.08476543426513672, -1.2484753131866455, 0.2427234649658203, -1.5439083576202393, -7.205318927764893, -3.759897232055664, -0.1305704116821289, -2.5860683917999268, -7.6729736328125, -5.601412773132324, 2.867502212524414, -4.313741207122803, -0.2075636386871338, 1.3671374320983887, -3.9768521785736084, 2.822413444519043, -3.0502090454101562, -2.768831729888916, -7.693564414978027, -1.9946818351745605, -0.9842821359634399, -0.8164669275283813, -0.9145963191986084, -4.613000869750977, -5.0026092529296875, -3.7316932678222656, -4.475579261779785, -2.802088737487793, -3.395514488220215, -7.253878593444824, -0.3332555294036865, -3.9745547771453857, -3.056089401245117, -3.9674644470214844, -0.2355102300643921, -6.097320079803467, -6.591168403625488, -1.4165191650390625, -2.646324634552002, -3.2625598907470703, 3.1062164306640625, -4.589450836181641, -2.683612585067749, -0.16917848587036133, 3.351013660430908, -6.698559761047363, -8.069204330444336, -2.121371030807495, -3.0149426460266113, -5.48886775970459, -0.377743124961853, -1.9784858226776123, 1.1647305488586426, -4.67575740814209, -0.3495591878890991, -1.9281367063522339, -4.371180534362793, -0.7231851816177368, -1.6633782386779785, -0.8346604108810425, 0.9077081680297852, -1.7843127250671387, -3.2882957458496094, 0.4569847583770752, -2.481518268585205, -2.4550106525421143, 0.7711267471313477, 2.112828254699707, -0.7514110803604126, -1.2224693298339844, 2.31009578704834, -1.9510775804519653, -0.8685733079910278, -1.8374656438827515, -2.5438590049743652, -2.112610340118408, 1.4404449462890625, -4.3248443603515625, -3.1827383041381836, -5.857046127319336, -0.6270849704742432, -2.536404848098755, 1.8123316764831543, 0.532926082611084, -1.5990557670593262, -3.0181169509887695, -6.895149230957031, -2.6113460063934326, 6.108643531799316, -0.806881308555603, -6.578987121582031, -3.472254991531372, -1.6916433572769165, -5.507904529571533, -2.5252175331115723, -1.2893697023391724, -7.822967529296875, -1.3347612619400024, -5.13468074798584, -5.948888778686523, -3.209768056869507, -7.340240478515625, -1.0135948657989502, -2.558379650115967, -0.7183017730712891, -0.41433167457580566, 3.7965450286865234, -3.145719051361084, -7.229781150817871, 0.2544393539428711, -3.198972225189209, -3.2232580184936523, -4.169834613800049, -0.17358267307281494, -2.344351053237915, -1.7039573192596436, -4.033393859863281, 1.3262405395507812, -3.7193868160247803, -4.024899482727051, 1.0894980430603027, -6.473837852478027, -2.713495969772339, -1.0598604679107666, -1.9489719867706299, -3.307645797729492, -9.00322151184082, -5.290660381317139, -1.3740062713623047, -0.04765439033508301, -2.30415415763855, -7.420992851257324, -2.927833080291748, -6.032681465148926, -1.1561431884765625, -1.0047571659088135, 1.7639856338500977, -5.934289932250977, 2.185832977294922, 0.17301297187805176, 1.6418752670288086, -9.223808288574219, 0.4419212341308594, -5.841329097747803, -3.0406675338745117, -3.2716448307037354, 0.2603332996368408, 3.790350914001465, 0.7460207939147949, 0.1839292049407959, -3.8930442333221436, -3.318147897720337, -8.031414031982422, -3.3742661476135254, -11.003544807434082, 2.6101231575012207, -5.321463108062744, -6.3330397605896, -6.300036430358887, 4.680771827697754, 0.40917491912841797, -0.5110841989517212, -4.494211673736572, 3.718417167663574, -1.8293569087982178, 3.6694536209106445, -4.507485389709473, 3.723818778991699, -5.405466556549072, -0.1483154296875, -7.120558738708496, 0.15652847290039062, 0.370847225189209, -1.0930052995681763, -3.654801845550537, -3.227961540222168, -3.047994613647461, -2.5458905696868896, -6.089118003845215, -2.312927722930908, -2.320559501647949, -1.7778830528259277, -2.5679986476898193, -5.656813144683838, -2.7801566123962402, -3.272747755050659, -1.033260464668274, -1.2168155908584595, -1.6831132173538208, -3.329540252685547, -1.1955161094665527, -8.152986526489258, -4.437233924865723, -3.467421054840088, -2.956629753112793, -3.292983055114746, -2.1687350273132324, -5.877335071563721, -4.094200134277344, -1.3431590795516968, -0.8582701683044434, -0.6985019445419312, -5.191176414489746, 0.3076779842376709, -2.143899917602539, 4.212680816650391, -7.762840270996094, -2.42293643951416, -5.033767223358154, 0.04465055465698242, -2.7704477310180664, -3.0281805992126465, -2.388225555419922, -2.563537359237671, -3.3918960094451904, -7.6377668380737305, 0.921271562576294, -4.628123760223389, 2.9255552291870117, -4.508146286010742, 5.64412784576416, -1.9089272022247314, -0.2154541015625, 3.4491281509399414, -2.1027274131774902, -0.9225746393203735, -4.957472801208496, 1.3856072425842285, 1.8939728736877441, -8.132126808166504, 0.19398713111877441, 3.666691780090332, -1.519888162612915, -1.662365198135376, 1.2666740417480469, -0.5017013549804688, 0.8649671077728271, -4.7018280029296875, 1.5707547664642334, -2.3927886486053467, -0.28214335441589355, -7.564182281494141, 2.2582454681396484, -4.680289268493652, -4.845163345336914, -4.163729667663574, -2.2481813430786133, -2.3157293796539307, -1.054725170135498, -4.124227523803711, -5.893208980560303, -0.40118420124053955, -1.0337824821472168, -4.377984523773193, 1.2000994682312012, -4.585993766784668, 1.2092218399047852, 2.2638535499572754, -1.42608642578125, -3.5903964042663574, -0.01799798011779785, -3.268474578857422, -6.251342296600342, -0.04062914848327637, -1.108863115310669, -2.6917636394500732, -3.1647443771362305, 1.4470281600952148, -4.9816131591796875, -0.5841286182403564, 0.2333364486694336, -4.876256942749023, -1.9789271354675293, -5.482763290405273, -1.606079339981079, 1.4154777526855469, 3.8837976455688477, -0.2582583427429199, -1.3894797563552856, -3.15027117729187, 4.618407249450684, -0.23122942447662354, -0.31876397132873535, -2.5492351055145264, -5.677745819091797, -5.880527496337891, 2.766767978668213, -0.8893922567367554, -4.93214225769043, 1.9685978889465332, -6.522857666015625, -8.431940078735352, 0.7568678855895996, -5.320347785949707, -4.2721476554870605, -4.875798225402832, 2.0201048851013184, -0.9696543216705322, -0.7245838642120361, 0.5583877563476562, -4.269606113433838, -7.190208911895752, 0.30406999588012695, -9.284229278564453, -3.887928009033203, -2.672830820083618, -2.5186896324157715, 1.7057008743286133, -2.8733224868774414, -5.025223731994629, -1.4585676193237305, -9.093332290649414, -4.883697032928467, -3.3763439655303955, -2.4447247982025146, -2.514207363128662, 0.4682955741882324, 1.2172725200653076, -0.792987585067749, -1.908711314201355, -0.09717404842376709, -0.9063502550125122, -1.4488110542297363, 1.0081334114074707, 1.9154930114746094, 1.2666339874267578, -2.7179737091064453, -3.4802746772766113, 2.2017836570739746, -2.7391507625579834, 0.6720397472381592, -0.5122717618942261, -2.230372905731201, -0.8266555070877075, -1.6824204921722412, -4.587724208831787, -3.936202049255371, -2.2123544216156006, -3.835599422454834, -3.2204105854034424, -6.938826084136963, -3.1475512981414795, -1.0849852561950684, -2.9212448596954346, 1.0137090682983398, 0.2689661979675293, -8.840596199035645, 0.8671588897705078, -0.22916865348815918, -0.38251376152038574, 0.12352728843688965, 1.761599063873291, 0.2718009948730469, -8.237696647644043, 1.780832290649414, -4.3972249031066895, -7.670594215393066, 1.8949909210205078, -6.380034446716309, -1.7680667638778687, -1.7462067604064941, -2.4863457679748535, -3.7132015228271484, -1.5004781484603882, 1.1749529838562012, -4.698534965515137, -2.525641441345215, -3.4219233989715576, -4.288147926330566, 0.9364893436431885, -5.331804275512695, 0.8096635341644287, -5.954554557800293, -3.9695072174072266, -0.2216484546661377, -1.8541699647903442, -0.3199279308319092, -0.2563377618789673, -2.981559991836548, -2.237903356552124, 1.8000199794769287, 3.442826747894287, -4.860385417938232, -3.1368746757507324, 2.47412109375, -2.8828322887420654, -1.2826714515686035, -6.14892053604126, 1.0126664638519287, -0.7164405584335327, -4.802664756774902, -1.8481520414352417, -5.827702522277832, 3.8177261352539062, -6.698725700378418, -4.102115631103516, -2.7751312255859375, -1.624518871307373, -6.051142692565918, -2.866471290588379, 0.2297687530517578, -4.461376190185547, -5.952437877655029, 0.9948854446411133, -3.3901174068450928, -5.9278388023376465, 1.916062355041504, -1.1456553936004639, -1.108750820159912, -3.8665812015533447, 0.07103776931762695, -3.741213798522949, -1.0969096422195435, -3.981975555419922, -3.5898730754852295, -3.2817537784576416, 0.6251182556152344, -2.3392763137817383, -7.908413887023926, -6.1091413497924805, -1.6121119260787964, -9.238896369934082, -5.2148566246032715, -1.3459974527359009, -5.464600563049316, -1.2726163864135742, -4.658366680145264, -4.101923942565918, 2.045860767364502, 0.21947264671325684, -6.764120101928711, 0.42792510986328125, 2.9365649223327637, -4.93723726272583, -0.23657822608947754, 2.093552589416504, -3.94064998626709, 0.008867740631103516, -5.10251522064209, -2.056422233581543, -7.176706790924072, -7.228862285614014, 0.5825865268707275, 1.492492914199829, -5.265350341796875, -3.4438843727111816, -0.5433658361434937, -0.9197220802307129, -1.6909308433532715, -6.13914680480957, -0.9036298990249634, -4.012143135070801, -4.1615753173828125, -5.039511680603027, -4.652862548828125, 1.6369917392730713, 2.484893321990967, -1.6402381658554077, -0.742995023727417, -1.3635907173156738, -3.8493032455444336, -3.0066511631011963, 0.8812243938446045, 1.0854201316833496, 1.2387499809265137, 0.06943345069885254, 5.314684867858887, -2.233097553253174, -4.365777492523193, -6.382774829864502, 1.2990975379943848, -6.396812438964844, -1.7725757360458374, 0.9144613742828369, -5.014465808868408, -0.4822118282318115, -3.0410079956054688, -2.5387914180755615, 3.182478427886963, -7.346624374389648, -2.5456721782684326, -2.575967311859131, -4.1076788902282715, -2.799078941345215, -4.640267848968506, -0.872200608253479, 0.2202897071838379, -3.894157886505127, -2.5783512592315674, -4.708257675170898, -1.8756163120269775, -6.64092493057251, 1.256136417388916, -1.6270151138305664, -0.2500295639038086, -3.1383132934570312, -3.838228940963745, 0.04591226577758789, -1.9648160934448242, -3.8254661560058594, -6.0492939949035645, -4.839102745056152, -3.35733699798584, 0.04703664779663086, 2.2058944702148438, -3.110158920288086, 3.257598400115967, -0.8597252368927002, -0.5637761354446411, -4.459673881530762, 3.079195499420166, 1.4311089515686035, -2.868351936340332, 0.8961412906646729, -5.689248561859131, -1.5942981243133545, 7.274509429931641, -4.915728569030762, 1.8787970542907715, -1.9162286520004272, -2.3259811401367188, -4.7251410484313965, 3.6931724548339844, -2.4772303104400635, 0.9929680824279785, 2.6528215408325195, -0.1243743896484375, -2.1579184532165527, -0.17191863059997559, -4.340604782104492, -1.3404957056045532, -2.6600356101989746, 0.8583273887634277, -1.9554427862167358, -5.238252639770508, -1.4121060371398926, -3.9017839431762695, 4.408720016479492, -0.9570807218551636, -4.5929083824157715, 0.6000096797943115, 2.7063732147216797, -5.928016185760498, -6.861820697784424, -0.32375359535217285, -3.1567893028259277, -0.5241093635559082, -7.4781389236450195, -2.887845039367676, 0.495225191116333, -1.7185280323028564, -6.370882987976074, 3.308530807495117, -1.8918049335479736, -7.391102313995361, 2.8976621627807617, 2.743170738220215, -5.734554767608643, -2.540386915206909, 0.5300660133361816, -1.821679949760437, -4.385941505432129, -4.536193370819092, -1.1653592586517334, -0.5227879285812378, -2.0528202056884766, -2.275334596633911, -1.314323902130127, -0.8742142915725708, -2.8600337505340576, -5.058197498321533, -0.8270812034606934, 1.0610356330871582, 0.5288825035095215, 4.335086822509766, -7.2486252784729, -1.0212268829345703, 0.9453935623168945, -1.7798545360565186, -2.306913375854492, -3.069075345993042, -2.1466708183288574, -4.442493438720703, 4.150600433349609, 1.019014835357666, -3.2465267181396484, -5.824125289916992, -1.703402042388916, -2.6125659942626953, 1.811488151550293, -5.340839385986328, 0.5296697616577148, -4.1791253089904785, -3.1470298767089844, -0.30419421195983887, -0.8797804117202759, 5.115268707275391, -2.26051926612854, 1.937389850616455, -2.919121742248535, -2.282086133956909, -0.9338638782501221, -0.4214637279510498, -1.3653655052185059, 1.7775392532348633, -1.9219802618026733, 0.2805938720703125, -6.085565567016602, -1.9048993587493896, -0.8420290946960449, -1.8746354579925537, 0.37641191482543945, 3.0606675148010254, -2.071953773498535, -6.814091205596924, -4.100022792816162, -0.9364252090454102, 2.4711155891418457, -4.760735511779785, 0.9248518943786621, -8.193991661071777, -3.1129040718078613, -6.534480094909668, 0.10002899169921875, -3.5857961177825928, -5.588496208190918, -7.190624237060547, -5.976963043212891, 0.13232851028442383, -1.055582046508789, 0.7596127986907959, 1.959406852722168, -6.175319194793701, -8.332389831542969, -0.32793235778808594, -1.0221192836761475, -0.32359397411346436, -1.5036882162094116, -4.943029403686523, 1.5230789184570312, -4.362515926361084, -4.575320243835449, 1.90079665184021, -6.380291938781738, -6.047236442565918, -4.455101013183594, -0.41617631912231445, -1.6335198879241943, 2.8800621032714844, -7.572994232177734, -4.250507354736328, -3.231318950653076, -6.448503017425537, 2.486415386199951, -4.962592601776123, 2.912900924682617, -2.6464409828186035, 2.921123504638672, -2.7702672481536865, 3.0940003395080566, -2.59881329536438, -3.860499382019043, -2.125441312789917, -6.661501884460449, -4.255200386047363, -5.156419277191162, 2.0066027641296387, -1.1243826150894165, -1.813272476196289, -3.078500270843506, 3.015784740447998, 0.4518139362335205, 1.3196802139282227, -2.809920310974121, -5.849065780639648, -2.4763994216918945, -1.737246036529541, -0.37932801246643066, -2.6498584747314453, -0.47813940048217773, -2.2090847492218018, -3.100861072540283, 2.680577278137207, -9.445250511169434, -0.7253186702728271, -4.180751323699951, -1.657987356185913, 1.8199272155761719, 0.35870790481567383, -4.934012413024902, -2.1850779056549072, -4.307151794433594, -6.968863487243652, -4.902948379516602, 0.25302600860595703, -1.1043413877487183, 0.4485759735107422, -2.2995529174804688, 0.540168285369873, 0.36542534828186035, -3.5643110275268555, -6.17598819732666, -2.5089077949523926, 0.969635009765625, 0.5599515438079834, -2.6732029914855957, -5.744029998779297, -3.119015693664551, -5.991686820983887, -2.4129414558410645, 0.5503218173980713, -2.327352523803711, -5.636204719543457, -8.643623352050781, -2.125351905822754, -2.324986696243286, -4.712521076202393, -0.4492957592010498, -5.2660369873046875, -1.2416083812713623, -1.1948609352111816, -4.138430595397949, -3.453484058380127, 2.0005736351013184, -5.297429084777832, 0.15862488746643066, 0.8598761558532715, 0.1969161033630371, -6.184763431549072, -2.2166574001312256, -5.452997207641602, -0.6608119010925293, -0.5763261318206787, -0.641799807548523, -0.5267956256866455, -1.7703907489776611, 2.665717601776123, 4.686413764953613, -1.3649520874023438, -5.575852870941162, 1.8834314346313477, -1.8833675384521484, -1.7099568843841553, -0.6896179914474487, -4.461732387542725, 3.320812702178955, -3.3121182918548584, 4.145930290222168, -0.5959020853042603, 2.739285945892334, -2.088223934173584, 1.143357753753662, 2.2393393516540527, 0.18717050552368164, 0.4721951484680176, 2.2348546981811523, -1.9595680236816406, -2.0769150257110596, 0.7143306732177734, 0.1704559326171875, 0.6391153335571289, 0.30849552154541016, 0.9272375106811523, -0.4343421459197998, -3.931684970855713, 2.6861772537231445, -0.37146031856536865, -1.2597060203552246, 4.322175979614258, -0.8831160068511963, 0.3269953727722168, 0.8930351734161377, 3.2213196754455566, -0.013008475303649902, -3.098111629486084, 0.6535334587097168, -5.677330017089844, -3.854219913482666, -0.7645701169967651, -1.6409345865249634, -3.9237124919891357, -2.882261276245117, 1.6724190711975098, -3.611975908279419, -3.845099687576294, 0.5164649486541748, -1.8657386302947998, -0.2725870609283447, -3.486194133758545, 2.268362522125244, -6.422406196594238, 0.5621867179870605, -4.289523124694824, -4.862741470336914, -1.4603172540664673, -1.2764792442321777, -1.0999526977539062, -2.52091121673584, -5.313502788543701, 0.4548001289367676, -2.2338311672210693, -4.684754371643066, -2.32948899269104, -4.52845573425293, 1.8555169105529785, -1.8214136362075806, -10.317533493041992, 2.198836326599121, -3.7822794914245605, -5.038355827331543, -5.437824726104736, -8.084823608398438, -4.712008476257324, -0.440973162651062, -0.8616975545883179, -2.334597110748291, -2.152836799621582, -0.22392797470092773, -0.6594793796539307, -4.7292866706848145, -1.7689006328582764, -1.1372884511947632, 1.2511298656463623, 0.8686923980712891, -4.314446926116943, 2.7164463996887207, 0.21656036376953125, -6.038398265838623, -3.825407028198242, -1.3695858716964722, -4.392892837524414, 3.030339241027832, -1.9204834699630737, 3.2154202461242676, -1.7187086343765259, -2.6260907649993896, 0.0593717098236084, -5.4862542152404785, -0.7882890701293945, -0.26302671432495117, -2.623359203338623, -1.1494503021240234, -5.667717933654785, -3.2364351749420166, 0.5960216522216797, 0.5106313228607178, -5.568765163421631, -0.14426791667938232, -2.5942177772521973, -2.5003204345703125, -0.9582326412200928, -1.2108449935913086, -3.9843995571136475, -4.716862678527832, -2.336832284927368, -1.010333776473999, 1.4564497470855713, -7.04685640335083, -1.8723323345184326, -2.421321153640747, -3.435178279876709, -2.5681111812591553, -4.136765480041504, -0.13042688369750977, 1.4780287742614746, -3.431532859802246, -1.714746117591858, -6.621387004852295, -0.5083518028259277, -1.7706711292266846, -4.604051113128662, -0.17287302017211914, 6.350185394287109, 0.7262792587280273, -9.482115745544434, -6.002873420715332, 3.5652289390563965, -6.334859371185303, -5.447264671325684, -1.4374979734420776, -5.9005937576293945, -5.526065349578857, 0.7427706718444824, -4.746166229248047, -5.379373550415039, -0.7018609046936035, -4.051361560821533, 0.43807458877563477, -0.43355727195739746, 0.5305109024047852, -4.128676414489746, -3.802438974380493, 2.040005683898926, -3.339419364929199, 2.806209087371826, -6.208263397216797, 0.9221301078796387, -0.009720325469970703, -1.2073231935501099, -0.9254961013793945, -4.231916427612305, 4.285096645355225, -1.8964465856552124, -5.653676986694336, -0.26876091957092285, -1.7086856365203857, 2.2522058486938477, -7.648850917816162, 1.5358030796051025, -4.427960395812988, 4.690217018127441, -2.5126428604125977, -6.1036763191223145, -2.195111036300659, -2.9154443740844727, 1.6744613647460938, -5.8672027587890625, -6.052067279815674, -0.20220041275024414, 1.1178019046783447, -6.329208850860596, -1.2836536169052124, -4.106240272521973, -1.645434021949768, 7.845813751220703, -3.754096269607544, -7.911676406860352, 2.5564818382263184, 5.107460021972656, -1.5498815774917603, -0.2494802474975586, 2.157166004180908, -3.179367780685425, 8.055484771728516, -1.9669134616851807, -1.835113286972046, -3.415430784225464, 0.7052435874938965, -2.4179611206054688, 0.6577763557434082, -3.945462226867676, -3.2353787422180176, -0.17560505867004395, 0.08479070663452148, -1.03566575050354, -2.318958282470703, -4.1951799392700195, -0.4881800413131714, -3.8692619800567627, -4.517420291900635, 0.05512380599975586, 4.2292866706848145, 0.825941801071167, -2.3183093070983887, -2.054107666015625, -3.599574089050293, -1.926895260810852, -5.487582683563232, 1.7548260688781738, 1.3515567779541016, -2.7112767696380615, -3.9315145015716553, -0.022311925888061523, -0.9536226987838745, -3.4735569953918457, -0.7789373397827148, 2.7299442291259766, -6.047359943389893, -7.745956897735596, -6.4204277992248535, -1.3929188251495361, 1.130842685699463, -0.5316495895385742, -5.372379302978516, -3.5645227432250977, -1.964540958404541, -2.559434413909912, -2.7279276847839355, -2.1750741004943848, -6.062771797180176, -3.2570040225982666, 1.9748191833496094, 0.3706672191619873, 4.008819580078125, 0.060840606689453125, -4.824540138244629, -2.3705554008483887, -9.150924682617188, -2.6510348320007324, -2.2094004154205322, 1.9142770767211914, -1.871797800064087, -0.11958479881286621, -2.482023000717163, -3.267685890197754, 2.2448110580444336, -2.942817211151123, -2.0047333240509033, -2.561002254486084, -2.9604835510253906, 5.255974769592285, 0.32441043853759766, 4.328585624694824, -1.0016716718673706, -1.5523874759674072, -2.608459711074829, -1.7079881429672241, -2.7165415287017822, 2.7341480255126953, -1.4497394561767578, 0.3715682029724121, 0.27385401725769043, -3.2242050170898438, -3.591090202331543, -4.2580342292785645, -1.4087765216827393, 2.09381103515625, 1.4968631267547607, -0.6147638559341431, -4.237801551818848, -1.2225052118301392, -1.323519229888916, 1.3282930850982666, 1.4260601997375488, -7.213136672973633, 1.1399059295654297, 0.6648006439208984, -1.7055412530899048, 4.414258003234863, -3.6146273612976074, -1.8175296783447266, -3.171731948852539, -4.933680534362793, -0.3794504404067993, -2.6823763847351074, -1.5374023914337158, -1.0017307996749878, 0.8117358684539795, 1.9352219104766846, -4.4749436378479, -1.6307580471038818, -4.5763397216796875, -3.201556921005249, -1.8937640190124512, -6.237161159515381, 5.840400218963623, -8.713934898376465, -3.2484188079833984, -3.601675271987915, -0.7992981672286987, -0.12637019157409668, -3.3299953937530518, -1.2086676359176636, -3.147244453430176, -1.1728132963180542, -1.5000544786453247, -0.2588803768157959, -3.588827610015869, -3.891948938369751, -5.321702480316162, -2.953622817993164, -5.065153121948242, -5.359179973602295, -3.8518874645233154, 3.359813690185547, -3.8409805297851562, 7.357192039489746, -0.623093843460083, -2.7289090156555176, -5.788418292999268, -2.9246439933776855, -3.509120464324951, -1.5509202480316162, -4.115179061889648, 1.401200294494629, -5.120395660400391, -2.2436695098876953, -1.4647314548492432, -1.5374417304992676, 0.6693472862243652, -2.880342721939087, 0.12017655372619629, -0.6767473220825195, -3.072239875793457, -2.639413833618164, -0.10837924480438232, 0.7678828239440918, 5.069881439208984, -3.7328360080718994, -7.493008613586426, 0.49300074577331543, -1.8348199129104614, -4.455493927001953, -1.0845651626586914, -0.7432916164398193, 0.7949600219726562, -1.4335830211639404, -0.4106147289276123, 2.324723243713379, -1.476590871810913, -1.8386204242706299, -7.3192667961120605, 0.5308787822723389, -0.5490396022796631, -0.24003064632415771, -0.34740781784057617, -4.583316802978516, -2.247053384780884, -3.8650588989257812, -3.6116490364074707, 0.147064208984375, 2.1927075386047363, -1.9499659538269043, -5.652019500732422, 0.3333854675292969, -2.2795019149780273, 0.9607973098754883, -1.6645245552062988, -3.1189370155334473, -2.5122766494750977, 3.7194066047668457, 0.542478084564209, -5.879897117614746, -1.426396131515503, 0.14657998085021973, -5.449382781982422, -2.364530563354492, -0.9256196022033691, -3.838850259780884, -6.2233405113220215, 5.269561767578125, -2.3614563941955566, -2.5277833938598633, -0.7170977592468262, -0.7465906143188477, 1.2441868782043457, 1.1712629795074463, 3.632899284362793, 1.828197956085205, 0.2712831497192383, -4.436211585998535, -0.45047450065612793, -3.6439056396484375, -3.6314821243286133, -3.783939838409424, -4.066188812255859, 1.8269109725952148, 1.7514514923095703, -3.361306667327881, -3.554316997528076, 1.504486322402954, -0.1714017391204834, -6.005435943603516, -8.9600248336792, 1.1319351196289062, 0.15373897552490234, -4.241762161254883, -3.493396759033203, 0.85982346534729, -0.8454107046127319, -2.2113611698150635, -5.539220809936523, -5.972678184509277, 6.298318862915039, -1.9951823949813843, -5.293412685394287, 1.7117044925689697, -0.03864610195159912, 0.436199426651001, -5.248086929321289, -0.830639123916626, 2.096993923187256, 1.2765774726867676, -1.9044307470321655, -1.6858409643173218, 0.2935786247253418, -1.8793903589248657, -1.1722301244735718, -2.486884355545044, -2.055478572845459, -3.40289306640625, 1.4145374298095703, -4.530364513397217, 3.1989569664001465, 6.098729133605957, -0.7746964693069458, -5.116934776306152, -1.931393027305603, -4.029146194458008, -1.985867977142334, -6.0044636726379395, -0.7196189165115356, -0.3151435852050781, -5.836643695831299, -4.843967437744141, -2.4061279296875, 0.030443429946899414, -0.08583617210388184, 1.40516996383667, -0.17155194282531738, -5.505932807922363, 1.244370937347412, -2.5430643558502197, -4.005552291870117, -8.1060152053833, -0.22020673751831055, -3.2300453186035156, -4.3812103271484375, -1.615176796913147, -2.506767749786377, -2.88468074798584, -0.2990114688873291, -4.202262878417969, -0.40113067626953125, 0.847815990447998, 1.6976869106292725, -0.22630834579467773, 3.660788059234619, -2.239590644836426, 2.9297523498535156, -2.249404191970825, -0.9509282112121582, 1.9686927795410156, 0.11094307899475098, -5.034440040588379, -4.418294906616211, -7.967084884643555, 2.3707737922668457, -0.15505754947662354, -3.170989513397217, -3.7554986476898193, -5.460054397583008, -4.154446601867676, -1.5970728397369385, -2.5427136421203613, -3.4808616638183594, -2.166609287261963, -4.582924842834473, -0.881226658821106, -2.786383628845215, -4.652303695678711, -8.455626487731934, -1.2991197109222412, -2.439897298812866, -2.0743372440338135, -2.5920357704162598, -1.7521073818206787, -7.686791896820068, -1.1156113147735596, 2.3964242935180664, -3.27215313911438, -5.727753639221191, 1.2100830078125, -6.577773094177246, -1.8419440984725952, 2.160006046295166, -6.040555953979492, -5.018279552459717, 0.6584842205047607, -5.175067901611328, -4.9967193603515625, -3.3601229190826416, -5.6777753829956055, -4.085350036621094, -2.7653651237487793, -1.0551785230636597, 0.5099790096282959, -5.365119934082031, -4.2895684242248535, 0.21531152725219727, -3.5296311378479004, -2.136924982070923, -4.328295707702637, -4.981968879699707, -3.6676554679870605, -6.342458724975586, -0.8956706523895264, -3.628457546234131, -1.6687804460525513, -7.123638153076172, -3.852297306060791, -2.0275449752807617, -1.933518648147583, -5.444823741912842, 0.20384454727172852, 2.7270054817199707, -6.593353271484375, -1.8369040489196777, 2.527413845062256, -2.6172056198120117, 1.0476417541503906, -0.28924524784088135, -6.474050998687744, -1.4502450227737427, -3.9340081214904785, -1.3588931560516357, 5.591024875640869, -4.384984970092773, -4.261776924133301, -2.444706916809082, -1.901479959487915, 2.3055152893066406, -1.7679487466812134, -4.076220989227295, -0.6642715930938721, -0.24660730361938477, -4.6114301681518555, -5.40056848526001, 3.6503095626831055, -4.367335319519043, -4.5227460861206055, 3.0534939765930176, -0.63828444480896, 0.30045318603515625, -7.252737045288086, -1.8638007640838623, -0.9797223806381226, -5.946591377258301, 4.1092095375061035, -1.3611633777618408, 1.0359609127044678, -1.1731563806533813, -2.1952600479125977, -7.56492280960083, 0.3995065689086914, -3.733914613723755, -1.2750855684280396, -3.1909849643707275, 2.0119075775146484, -7.834321975708008, -1.329345464706421, -2.758528232574463, 3.494136333465576, -5.226350784301758, 1.7132244110107422, -2.950265884399414, 3.029573917388916, -3.2432668209075928, -2.492835521697998, -1.7977054119110107, -4.525468349456787, -1.8240633010864258, -3.7024638652801514, 1.898592233657837, -2.9224820137023926, -3.5179457664489746, -2.033766269683838, -7.1642746925354, -0.7210581302642822, -1.226681113243103, -0.4736747741699219, -5.115206718444824, 1.719167709350586, -1.7898447513580322, 1.9197239875793457, -3.006983518600464, -1.2945423126220703, -3.6866097450256348, -0.06752276420593262, 1.1539194583892822, 0.051164865493774414, -1.6946141719818115, -1.8924882411956787, -4.060420989990234, -0.3956761360168457, -3.7148115634918213, -3.751178503036499, -2.8208303451538086, -1.6452364921569824, -1.5169060230255127, -1.160543441772461, 3.310171127319336, -5.365788459777832, -1.787515640258789, -3.4887661933898926, -3.2664449214935303, 0.6589934825897217, 2.1988420486450195, 1.7631311416625977, 0.24535155296325684, -7.746706962585449, -3.661426067352295, -0.7883175611495972, 2.3228182792663574, -6.851855278015137, -2.2211246490478516, -0.7159472703933716, -6.908631801605225, -3.0838775634765625, -6.526829719543457, -4.00804328918457, 2.302842140197754, -3.4743618965148926, -1.7869502305984497, -1.9833295345306396, 0.24836945533752441, -3.48471736907959, 2.628035545349121, 0.4250211715698242, -11.147025108337402, -2.434018135070801, -9.859100341796875, -1.2205829620361328, -2.45623517036438, 2.1598033905029297, -2.0868117809295654, -1.752675175666809, 0.5565590858459473, -1.8051742315292358, -0.843927264213562, -0.5194385051727295, 3.31158447265625, -1.3163228034973145, 0.17217707633972168, -3.8481390476226807, -1.5193109512329102, -0.31900453567504883, 2.7957258224487305, -0.6974952220916748, 3.8590750694274902, -4.313722610473633, -2.919796943664551, -3.599283218383789, -0.12291765213012695, -1.7492902278900146, 2.4533615112304688, -1.5048198699951172, -1.0124608278274536, -4.531766891479492, -2.422769069671631, 0.02300119400024414, -5.429893493652344, 1.287581443786621, -4.080815315246582, -4.394521713256836, -4.000950813293457, -0.6026835441589355, -0.38575828075408936, -3.3279218673706055, 0.7251663208007812, -2.308804750442505, -0.05442404747009277, -1.1785911321640015, -3.872865915298462, -5.861961841583252, -10.801791191101074, -6.18972635269165, -0.33457350730895996, -2.684075355529785, -5.627187252044678, -1.7044758796691895, -2.4318952560424805, -10.042519569396973, -2.4404163360595703, -2.4156250953674316, -3.2970147132873535, -3.1755268573760986, -1.981843113899231, -1.1683874130249023, 3.6230926513671875, -12.585834503173828, 0.6005020141601562, -2.371450662612915, -3.7764997482299805, -4.839677810668945, -2.825493812561035, 0.9759228229522705, -1.488309383392334, -1.1473809480667114, 4.696865081787109, 0.09571218490600586, 1.7397782802581787, -0.2782907485961914, -2.312633514404297, -4.937438011169434, 0.2299051284790039, -2.3587965965270996, 2.4598865509033203, -2.0735573768615723, -0.689379096031189, -1.9314522743225098, -0.20854854583740234, 0.5839390754699707, -2.5653650760650635, -2.7542781829833984, -3.490931987762451, 0.317518949508667, -3.817309856414795, -2.7343263626098633, -6.315498352050781, -0.9243488311767578, -4.929681777954102, -4.815618515014648, -1.3122836351394653, -2.248073101043701, -1.4081000089645386, -8.597460746765137, 5.561826705932617, -4.991933822631836, -1.7972153425216675, 0.3249697685241699, -5.924779891967773, 0.630699634552002, -0.1639803647994995, 0.007903575897216797, 1.9367480278015137, -5.154817581176758, 1.0072834491729736, -5.165386199951172, -4.561805725097656, -3.476973295211792, -2.6155495643615723, 3.406644344329834, -4.272299766540527, -4.368936538696289, -6.489107131958008, -3.1878108978271484, -1.6168617010116577, -0.9674696922302246, -1.9085410833358765, -4.486526966094971, -0.44432127475738525, -2.0928566455841064, -3.789984941482544, -8.107771873474121, 1.123056173324585, -5.310685634613037, -5.020791530609131, 0.14130949974060059, -9.548894882202148, -4.389639854431152, -4.18205451965332, -5.327104568481445, 0.11823463439941406, -2.5382914543151855, -6.108574867248535, -3.1340177059173584, 0.5929863452911377, 1.2063934803009033, -3.0555567741394043, -4.027243137359619, -0.595689058303833, 4.162250518798828, -2.134746551513672, -3.1500134468078613, -2.402125597000122, 0.11916375160217285, -0.0721280574798584, 1.3169162273406982, 2.413337230682373, -1.764230728149414, 0.5869672298431396, -7.045052528381348, -1.26274836063385, -1.0616828203201294, -4.344593048095703, -1.768018364906311, -1.1078603267669678, 0.5147736072540283, 0.2967352867126465, -1.5536623001098633, -2.8677866458892822, 1.2611720561981201, 5.262235641479492, -2.4388492107391357, -2.3560311794281006, -6.257449150085449, -4.025286674499512, 3.455503463745117, -3.96641206741333, -3.8600544929504395, -1.3261083364486694, 0.13134241104125977, -6.076879978179932, -6.547260761260986, -2.2012295722961426, -4.849666595458984, 1.2141976356506348, -4.429085731506348, -2.7581119537353516, -1.8251667022705078, 1.4991393089294434, 0.14989185333251953, -7.16281795501709, -2.4837679862976074, -4.372699737548828, -3.9175994396209717, -8.063623428344727, 0.10859513282775879, 1.4405937194824219, -1.5966434478759766, -4.068849563598633, -5.610352516174316, -1.4990264177322388, -1.2270405292510986, 1.1377034187316895, 3.6830286979675293, -2.75248122215271, -4.472623825073242, -0.6289054155349731, 5.708215236663818, -2.3201072216033936, -3.789611339569092, 2.0199356079101562, -3.986276626586914, -1.9936751127243042, -0.19027400016784668, -2.645414113998413, -2.3868112564086914, -2.64099383354187, -3.340120553970337, -0.9489855766296387, -5.482210636138916, -1.4095098972320557, -2.7884521484375, -4.510692596435547, -1.9508708715438843, -5.743863105773926, -8.125651359558105, -1.8017675876617432, -3.5443739891052246, 3.7896852493286133, -8.476007461547852, -1.902249813079834, -3.385756254196167, -0.5468975305557251, 2.0702295303344727, -1.7353382110595703, -0.10692083835601807, -2.4216396808624268, -2.6795125007629395, -1.206324815750122, -7.300044059753418, -7.154021739959717, -1.8778340816497803, 1.378580093383789, -6.1404571533203125, -5.302978515625, -1.4091849327087402, -0.9975552558898926, 0.266495943069458, -2.823641538619995, -2.874100923538208, 1.223200798034668, 6.021167755126953, -6.558425426483154, 3.3445825576782227, 0.011430740356445312, -0.9046272039413452, -3.1797304153442383, -4.177063941955566, -6.380514144897461, -1.0094335079193115, -4.559609413146973, -2.4168756008148193, -3.396873712539673, -5.438440322875977, -3.545283555984497, 2.8675942420959473, 2.968674659729004, -10.481939315795898, -2.4032955169677734, -4.854957580566406, -1.1808817386627197, 0.784783124923706, 1.2502360343933105, -3.0636816024780273, 0.9712157249450684, -3.123861074447632, -3.332301616668701, -0.9791462421417236, 4.7341742515563965, 0.6435384750366211, -9.468437194824219, -1.043944001197815, -1.4800636768341064, -0.8505661487579346, -0.13451969623565674, -3.4314539432525635, 2.8871231079101562, -0.05355978012084961, -1.1675093173980713, -0.46451306343078613, -1.3662724494934082, -6.7621259689331055, -4.702643394470215, -1.8052079677581787, -5.809310436248779, -6.420751571655273, -1.1810100078582764, -1.7568320035934448, -3.3937244415283203, -3.9776813983917236, -3.7643566131591797, -0.07017302513122559, -3.0102787017822266, -5.401449203491211, -6.340258598327637, 2.345341682434082, 1.0030674934387207, -2.758596420288086, -5.485937118530273, -0.27024662494659424, -2.4703564643859863, -4.319404125213623, -4.710565567016602, -1.5954022407531738, -1.171022653579712, -0.5041525363922119, 0.5778114795684814, -5.099582672119141, -3.6021335124969482, -2.4905176162719727, -9.151557922363281, 0.4349634647369385, -4.665212631225586, -4.2362518310546875, -5.388749122619629, -5.1245574951171875, -0.9340915679931641, -6.239789009094238, -9.632658958435059, 3.1334338188171387, 0.5803818702697754, -0.6549257040023804, -0.8547728061676025, -0.36658215522766113, -6.724786281585693, -3.4066991806030273, 2.61678409576416, -3.2637250423431396, 0.1260209083557129, -0.14001071453094482, -1.9838786125183105, -3.478428840637207, -4.948641300201416, -4.662075042724609, 3.159514904022217, 0.0969548225402832, -4.21335506439209, -1.3007922172546387, -2.0996432304382324, -3.944253921508789, -3.47129225730896, 0.8726611137390137, -2.827562093734741, -1.8989355564117432, -3.136380195617676, -1.7921305894851685, 0.20208334922790527, -7.74688720703125, -2.0304596424102783, 0.20553874969482422, 0.16964054107666016, -0.6521239280700684, -2.79793381690979, -2.949303388595581, -3.8788886070251465, -1.8573250770568848, -2.8186464309692383, -5.624184608459473, -2.686629295349121, 1.2770578861236572, -2.5116026401519775, 1.0871875286102295, -2.598475456237793, -3.9108965396881104, -5.098407745361328, -0.9942693710327148, 0.6393313407897949, -6.327796936035156, -1.8139839172363281, 1.7879371643066406, 1.342432975769043, -5.437973499298096, -2.7473723888397217, -2.7108383178710938, -2.1968839168548584, -7.937102317810059, -4.924610614776611, -7.7108354568481445, 1.3365857601165771, 1.1898736953735352, -2.635843515396118, -1.0964456796646118, -0.15981173515319824, -2.1581904888153076, -0.7707194089889526, -2.8924002647399902, 0.5926601886749268, -6.519410133361816, -1.3903381824493408, -3.063232421875, 4.317941188812256, 0.22264623641967773, -7.61129093170166, 4.687765598297119, -4.817709445953369, 1.1235685348510742, 2.8526787757873535, -4.917991638183594, 3.727397918701172, -1.3624827861785889, 0.13524103164672852, -5.983752250671387, -2.9423837661743164, 2.93631649017334, 0.48378467559814453, -0.4719724655151367, -5.942098617553711, -1.4620752334594727, -2.140880584716797, -2.188868999481201, -0.15271830558776855, -4.144731521606445, -4.676666259765625, -4.058109283447266, -5.41243314743042, 0.25642871856689453, -3.29661226272583, 3.6683449745178223, 1.1823959350585938, 3.997142791748047, -0.19715046882629395, -3.9921748638153076, -2.4927589893341064, -3.5391950607299805, -1.9848313331604004, -0.5651298761367798, -3.220890998840332, -2.6516053676605225, 0.03061199188232422, -1.263113260269165, -0.6925337314605713, -3.817391872406006, -5.0148138999938965, -2.91403865814209, -1.3070054054260254, -6.318572998046875, -0.3376312255859375, 2.514235496520996, -0.6659059524536133, -5.939533233642578, -2.0410380363464355, -2.564849376678467, -7.019031524658203, -4.203197002410889, -1.4816937446594238, -4.224575519561768, -4.1460981369018555, 0.8721246719360352, -0.5382571220397949, -1.2100725173950195, -1.4184141159057617, -2.0377864837646484, -5.247427940368652, -2.692931652069092, -0.7074236869812012, -6.707945346832275, 1.774357795715332, -1.5417577028274536, -9.927473068237305, 3.733494281768799, -0.33237767219543457, -6.4666314125061035, -3.6978518962860107, 1.527015209197998, -2.444277763366699, -4.640109539031982, -5.055334568023682, -4.575389862060547, -1.92312753200531, 3.9064817428588867, 1.340491771697998, -1.0826547145843506, 1.048663854598999, -4.379950046539307, -4.949839115142822, -5.00827169418335, -0.3894212245941162, 3.726405620574951, 2.9900741577148438, -7.137081146240234, -1.937925934791565, -7.09701681137085, -0.9848973751068115, 2.350308895111084, -1.996140956878662, -0.337959885597229, -5.544741153717041, -4.664583206176758, 2.0701146125793457, 2.558196544647217, -2.2767751216888428, 0.4185011386871338, -5.561600685119629, -0.7363238334655762, 3.3566441535949707, -5.54884672164917, -1.1337858438491821, -1.19716477394104, -3.256826400756836, 1.7176003456115723, 1.6605873107910156, -0.06446719169616699, -0.18855011463165283, -0.245857834815979, -0.6543810367584229, -6.5118632316589355, -0.03986835479736328, -2.342679500579834, -7.060000419616699, -1.364518404006958, -0.1966996192932129, -1.6985031366348267, -8.125388145446777, -6.46275520324707, 3.1239848136901855, -4.32260799407959, -2.67211651802063, -2.948695182800293, -2.300043821334839, 1.3308191299438477, 1.0428104400634766, -4.401691913604736, 1.9921619892120361, -8.797900199890137, -0.36849355697631836, -3.2143547534942627, -4.771790504455566, -7.345074653625488, 0.6202301979064941, -4.630396842956543, -1.7101237773895264, -6.050849914550781, -0.23784708976745605, -1.0431084632873535, -2.676234245300293, 2.3147759437561035, -5.6455464363098145, -3.8334286212921143, -0.5933705568313599, 1.2964167594909668, 2.7127785682678223, -5.707271575927734, 0.059417724609375, 0.06222701072692871, -3.6399290561676025, -3.3368406295776367, -0.6778178215026855, -5.355057716369629, -3.7648611068725586, -3.601407289505005, 0.06760668754577637, -5.1702375411987305, 0.3956279754638672, -2.3264198303222656, -0.17522072792053223, -3.3608171939849854, 1.3599653244018555, -0.10954952239990234, -5.125858306884766, -2.9056713581085205, -3.627392292022705, -0.3693859577178955, -3.112245559692383, 2.101283073425293, 0.4614448547363281, -6.045987129211426, -2.903455972671509, -1.5194628238677979, -5.309995651245117, -3.6235761642456055, -3.671262264251709, 6.309603691101074, -2.652146339416504, -3.4470138549804688, 1.7213120460510254, -3.7616684436798096, -6.522924423217773, -6.203469276428223, -3.0489237308502197, -2.3458104133605957, 0.2537975311279297, 3.705099582672119, 0.37445068359375, -1.0252461433410645, -3.648284435272217, 0.889796257019043, -3.5024709701538086, -4.731802940368652, -2.454528331756592, 1.4458296298980713, -1.00761079788208, 1.9254889488220215, -10.6696195602417, -4.385827541351318, -7.070710182189941, -7.4959259033203125, -1.371049404144287, -5.851594924926758, -2.133004665374756, -3.4348974227905273, -5.525059700012207, 3.156054973602295, -2.647165060043335, -4.070567607879639, 3.040951728820801, -3.043900489807129, 0.5369162559509277, -3.5037341117858887, -3.186570644378662, -0.7742164134979248, 1.2068190574645996, -4.512611389160156, 0.30637598037719727, -2.4206578731536865, -5.6298933029174805, -2.0205883979797363, 1.4994721412658691, -3.551088333129883, -1.0017564296722412, 0.4019954204559326, 3.3677453994750977, -1.6481125354766846, -3.1706936359405518, -0.6465864181518555, 3.0982327461242676, -4.68006706237793, -1.0463813543319702, -4.263849258422852, -4.314375877380371, -0.5871013402938843, -4.65047550201416, 1.253866195678711, -2.1966919898986816, -1.9963911771774292, -2.0745158195495605, -3.9266889095306396, -2.2092270851135254, -5.552903652191162, -0.5005501508712769, -5.303472518920898, -0.17909157276153564, -0.4359097480773926, -4.648812770843506, 2.4041309356689453, -3.274458646774292, 5.600571632385254, -3.8468856811523438, 1.6478800773620605, -5.546689033508301, -0.33950912952423096, -1.5750845670700073, -0.10840082168579102, -5.374772548675537, 1.0617759227752686, -0.7921652793884277, -3.2991209030151367, 1.4367115497589111, -3.1116433143615723, -4.976559638977051, -4.01743221282959, -3.2906622886657715, 0.22996258735656738, -5.185634136199951, -2.621917724609375, 1.7333810329437256, -2.019679069519043, 0.013792991638183594, -0.2155245542526245, -4.188812255859375, -3.0738933086395264, -3.1185998916625977, 0.1248929500579834, -7.516820907592773, -0.4268862009048462, -1.2946584224700928, -1.7785611152648926, -5.676656246185303, -2.5804286003112793, -8.336907386779785, 2.060331344604492, 1.467963695526123, -2.9179611206054688, -3.326294183731079, 0.09487199783325195, -1.9093929529190063, -0.42839670181274414, -0.04533874988555908, -1.1157444715499878, -2.1860499382019043, -3.331794261932373, -2.1316850185394287, -2.0961709022521973, -2.216547966003418, -2.9721174240112305, 0.7127156257629395, 3.349943161010742, -3.9795079231262207, -2.1677942276000977, -0.3354380130767822, -0.42498934268951416, 0.8778421878814697, -0.1763678789138794, -5.3429036140441895, -4.410370826721191, 1.9692230224609375, -3.190767526626587, -0.5185763835906982, -3.3250906467437744, -4.073307514190674, -3.0135858058929443, -7.8784685134887695, -5.1374664306640625, -7.458282947540283, 3.314781665802002, -6.90386962890625, -0.7288435697555542, -0.9762264490127563, -3.646538496017456, -4.020702838897705, -2.0746705532073975, -1.558262825012207, -2.7159295082092285, 0.6851236820220947, -6.797093391418457, -0.5171653032302856, -2.41579008102417, -0.971942663192749, 0.5980865955352783, -2.803196907043457, -6.164037704467773, -0.2745213508605957, -0.45133447647094727, -4.562070846557617, -0.9349889755249023, 0.6544818878173828, -0.6776070594787598, -5.5493059158325195, 1.537269115447998, 1.5393691062927246, -3.0622751712799072, -2.0791473388671875, 2.5400867462158203, -4.930054664611816, -5.9949774742126465, -10.798954963684082, -2.255770206451416, -0.11792540550231934, 1.1550374031066895, -5.967196464538574, -0.495678186416626, -1.1633158922195435, -6.080979824066162, 0.6463799476623535, 0.9461760520935059, 1.9266748428344727, 1.3905231952667236, -2.289839506149292, -4.258996963500977, -8.054182052612305, -1.0764580965042114, -2.3668651580810547, 0.6421198844909668, -0.32620322704315186, 1.6649065017700195, -0.9610120058059692, -6.758025169372559, -0.37100839614868164, 1.8748207092285156, -6.562526702880859, -4.583469390869141, -1.1311304569244385, -6.829488754272461, -4.4120306968688965, 1.0360138416290283, 1.2867307662963867, -2.234530210494995, -2.041254997253418, -6.519196033477783, 0.3409764766693115, 1.9167461395263672, 1.394012451171875, -1.357404351234436, 1.3013215065002441, -1.0418912172317505, 2.784331798553467, 3.9856457710266113, -4.129680633544922, -4.7470502853393555, 4.417521953582764, -5.870455265045166, 0.4636044502258301, 0.6920261383056641, -3.6795125007629395, 2.89017391204834, -1.625868558883667, -6.505243301391602, -0.8643770217895508, -3.3790318965911865, -3.8171327114105225, -5.344702243804932, -4.147991180419922, -6.156747341156006, -2.313448429107666, -2.616878032684326, 0.8611724376678467, -4.349943161010742, -1.9260143041610718, 1.4048469066619873, -3.251185178756714, -0.7388861179351807, -3.0935075283050537, -1.5337684154510498, -3.261096715927124, -3.7275142669677734, 0.712608814239502, 1.3112373352050781, -3.9528961181640625, -8.536327362060547, 0.8437647819519043, -0.5099852085113525, -3.020840644836426, -3.1487412452697754, -5.149035930633545, -4.651965618133545, -1.4275367259979248, -0.06986427307128906, -3.41558575630188, -3.734553575515747, -3.5665788650512695, -4.5520339012146, -2.5708956718444824, -4.202813148498535, -1.061310887336731, -2.9646098613739014, 1.5949201583862305, -3.762213706970215, -2.746511459350586, -0.5772960186004639, -6.684713840484619, -5.091679573059082, -4.854819297790527, -6.529628276824951, 5.775007247924805, -5.716694355010986, 1.8942036628723145, 2.976346015930176, -4.4457597732543945, -3.3684940338134766, -1.8260400295257568, -2.9075117111206055, -0.7163479328155518, -2.311617374420166, 0.42982053756713867, -6.319568157196045, -1.021488904953003, -2.5381550788879395, -2.873891830444336, -0.6059550046920776, -3.88222074508667, -0.9880163669586182, -0.3743884563446045, -2.6896450519561768, -4.818574905395508, -2.0856218338012695, -1.444282054901123, -0.33969545364379883, 2.801267623901367, -6.994131565093994, -3.415235757827759, -3.994255542755127, -0.8232299089431763, -4.748119354248047, -2.2217812538146973, -1.0611872673034668, -5.064594268798828, -4.800130844116211, -7.195750713348389, -2.208447217941284, -8.481193542480469, 0.6349701881408691, -2.602139949798584, -5.053218364715576, 3.171989917755127, -2.1463122367858887, -1.7080568075180054, -6.169943809509277, 0.2322826385498047, -2.2046241760253906, -3.380249500274658, -3.08294415473938, -5.991179943084717, -6.364480972290039, -4.735995769500732, -0.3553159236907959, 1.321324348449707, -6.029852867126465, -1.724271535873413, -5.3996758460998535, -7.1751179695129395, -2.962730646133423, -3.6899940967559814, 0.5607168674468994, -1.4952772855758667, -4.303690433502197, -3.339521884918213, -2.6071760654449463, -0.3507685661315918, -1.1000813245773315, -0.3040761947631836, -3.780200719833374, -4.733793258666992, -2.2639200687408447, -3.445659637451172, -3.065655469894409, -7.54417085647583, -2.2919037342071533, 0.49102354049682617, -4.249189376831055, -1.9505189657211304, -0.921899676322937, -6.49317741394043, -5.383462429046631, -1.511649250984192, 0.8138701915740967, -0.7002271413803101, -5.024561882019043, -1.7570430040359497, -3.8053486347198486, -4.986504554748535, -0.30145955085754395, -2.7918052673339844, -4.729549407958984, -3.1558914184570312, 0.3262214660644531, -1.7262592315673828, 3.92938232421875, -1.6354924440383911, -3.1253838539123535, -2.612161874771118, -5.356717109680176, -4.386155128479004, 0.38671302795410156, -1.3693556785583496, 0.6835336685180664, -5.307811737060547, 0.4759819507598877, -2.387124538421631, -5.803536415100098, 2.6151270866394043, -3.8369691371917725, -1.9275437593460083, -1.429824709892273, -5.26859712600708, -7.216520309448242, -0.004944324493408203, -4.427164077758789, 1.3062663078308105, -4.258862495422363, -1.024526834487915, 0.5157239437103271, -1.7925441265106201, -3.1514854431152344, 1.500220775604248, 0.968400239944458, -9.730305671691895, 6.8824462890625, -1.034718632698059, -4.421945095062256, -1.103269338607788, -1.3443108797073364, 1.3728764057159424, -1.2899225950241089, 0.42153334617614746, 6.917108535766602, -2.1272778511047363, 0.13572096824645996, 0.20173144340515137, -13.41131591796875, -4.353750228881836, -2.852450370788574, -0.5262795686721802, -1.3229620456695557, -2.0808067321777344, 0.05907773971557617, -0.1012876033782959, 1.8214983940124512, -2.119906425476074, -4.093532562255859, -4.692413330078125, 3.542941093444824, 1.1699542999267578, 2.090818405151367, -4.604052543640137, -4.338581085205078, -0.36511099338531494, -0.23697996139526367, -0.8830338716506958, 0.2936882972717285, 0.1976165771484375, 0.8395283222198486, -6.356000900268555, 0.01558995246887207, -4.982561111450195, -1.6778980493545532, 1.4252145290374756, -0.8416972160339355, -0.13904023170471191, 3.980804443359375, -4.8359575271606445, -5.185978889465332, 2.523858070373535, 0.04136323928833008, -8.818401336669922, 2.948760509490967, -0.8687375783920288, -1.4605740308761597, -3.2752957344055176, -1.9027469158172607, 1.7310867309570312, -3.987013816833496, -3.004319667816162, -3.840240716934204, -4.283590316772461, -5.446619510650635, -0.5487663745880127, -4.485339164733887, -0.5649131536483765, -7.458057403564453, -3.0359389781951904, 5.499213218688965, -0.4786183834075928, 2.7907838821411133, -1.801262378692627, -2.469515323638916, 1.0071840286254883, -4.312441825866699, 1.845442295074463, 1.9680895805358887, -6.028476238250732, -2.5708675384521484, -5.252052307128906, -0.6766918897628784, -2.7436602115631104, -0.7815694808959961, -1.6440826654434204, 0.1105341911315918, -4.85929012298584, -1.0441259145736694, -6.34540319442749, -0.7134944200515747, 0.049990177154541016, -3.2661314010620117, -7.841278076171875, -5.253278732299805, 2.8861188888549805, -2.6761322021484375, -5.777615070343018, -0.38252317905426025, -4.248883247375488, 0.5248880386352539, -2.927804470062256, 0.607576847076416, -2.5161526203155518, -5.636813163757324, -6.50793981552124, -7.65471887588501, -2.038501262664795, -6.247833728790283, -2.250673770904541, -1.7099026441574097, -2.326972484588623, 1.7965002059936523, -6.636904716491699, 4.408486366271973, -9.032818794250488, 0.5571699142456055, -5.302199363708496, 2.591313362121582, 0.058442115783691406, 4.544145107269287, -6.028347015380859, -6.068073749542236, -3.556195020675659, -1.4984066486358643, -6.882758617401123, -4.709784507751465, -0.770759105682373, -1.4590098857879639, 0.47903013229370117, -0.552261471748352, 2.5138320922851562, 0.6689314842224121, 0.23127508163452148, 1.1991748809814453, -2.8505783081054688, -6.642312049865723, -1.538837194442749, -6.78707218170166, -0.7562258243560791, 0.5049738883972168, 2.3398590087890625, -1.9167988300323486, 1.298581600189209, -1.5359840393066406, 1.2606627941131592, -2.4887161254882812, -1.9631667137145996, 0.38094353675842285, 1.8949229717254639, -2.330129384994507, -3.1767311096191406, 3.1734652519226074, -2.2310667037963867, -5.789980888366699, 1.0693273544311523, 0.023186922073364258, 1.0579051971435547, -1.5008093118667603, -3.132927894592285, -0.8307375907897949, 3.7331409454345703, -4.9812517166137695, -5.659371852874756, 0.9024350643157959, -3.860107898712158, -3.7067501544952393, 1.2643935680389404, -1.3946771621704102, -2.8015007972717285, -3.6926698684692383, 0.8093276023864746, -6.596128463745117, 0.4060077667236328, -6.481478691101074, -3.8982956409454346, 4.309033393859863, -0.7646995782852173, -3.150278091430664, 0.575577974319458, -0.21536648273468018, 3.196516990661621, -5.82631778717041, -5.423340320587158, -1.8238105773925781, -2.5419816970825195, -2.3512752056121826, -3.432534694671631, -4.110696315765381, 1.7338600158691406, -1.7443571090698242, -3.373885154724121, -5.23578405380249, -2.193773031234741, -1.9772456884384155, -3.6086618900299072, 3.316070556640625, -4.507494926452637, -0.2314389944076538, 0.36827945709228516, -2.559213638305664, -3.1921234130859375, 2.044783592224121, -0.15696001052856445, -4.958532333374023, -3.3134703636169434, -6.0359625816345215, -6.519578456878662, -2.803696632385254, 3.658860206604004, -2.9121923446655273, 0.10308337211608887, 1.441701889038086, -5.026358127593994, 2.9474802017211914, -10.164673805236816, -5.965722560882568, 7.047788619995117, -6.0102128982543945, -5.133234024047852, -3.1884002685546875, -7.660080909729004, -1.582112193107605, -3.014676332473755, 4.849916934967041, -1.625319480895996, -2.958390474319458, -4.677621841430664, 4.029134750366211, -1.7549442052841187, -3.3095755577087402, -1.7369418144226074, -5.406243801116943, 0.3033125400543213, 0.5950069427490234, 0.25130152702331543, -2.094747543334961, -1.481748104095459, -3.6030731201171875, 3.8914198875427246, 0.9346175193786621, -0.18886852264404297, 0.8639628887176514, 0.9816060066223145, 2.050307273864746, -2.894556999206543, -1.9800701141357422, -1.771266222000122, -10.201354026794434, 0.5556778907775879, -6.648937225341797, -2.2495667934417725, 0.711625337600708, -1.4339640140533447, -4.793901443481445, -4.030229091644287, -2.4994804859161377, 3.56422758102417, -3.102510929107666, -5.759735107421875, -1.9964895248413086, -0.44429779052734375, -2.039552927017212, -2.091010332107544, -2.288922071456909, -3.8775064945220947, -3.0148539543151855, -3.443622589111328, -0.4425710439682007, -2.3688859939575195, 3.2407665252685547, -2.7767863273620605, -3.91465163230896, -1.3215370178222656, -4.494644641876221, 0.7997825145721436, 2.804682731628418, -5.5421624183654785, -0.288204550743103, -9.259825706481934, -3.594667911529541, 1.4013657569885254, -5.273989200592041, -3.8150551319122314, -4.560127258300781, 2.327731132507324, 2.372213363647461, -3.547818660736084, -3.453735589981079, -0.9263937473297119, 1.107041358947754, -5.80357551574707, -4.01312255859375, -0.8426085710525513, -1.7344475984573364, -3.6977949142456055, 2.11798095703125, 0.7165887355804443, -6.043190002441406, 0.978057861328125, 1.5948352813720703, -2.8911497592926025, -3.7555501461029053, -4.795949935913086, -2.540163040161133, -3.5041165351867676, -6.458606719970703, 0.5386176109313965, 1.7209267616271973, -2.71223783493042, 2.768709182739258, -1.5104546546936035, -6.074582099914551, -1.8836133480072021, -6.973554611206055, 4.24931526184082, -0.5278255939483643, 0.4129302501678467, -0.8915960788726807, -0.4384268522262573, -5.13700532913208, 0.5538859367370605, -1.1768815517425537, -1.3505436182022095, -4.679029941558838, -4.921329498291016, -4.210977077484131, -1.5095834732055664, -2.189626693725586, -2.8855581283569336, -1.460717797279358, -2.8855109214782715, -5.119634628295898, -0.9496450424194336, -5.622495651245117, 0.04930305480957031, -6.348705291748047, -2.1001079082489014, -4.790954113006592, -1.7208619117736816, -0.9599754810333252, -3.9880213737487793, 1.8550400733947754, -1.1123764514923096, -0.4748762845993042, -2.124523878097534, 2.1709914207458496, -6.153542518615723, -4.0636444091796875, -3.7429146766662598, 0.14197850227355957, -6.887184143066406, -4.363631725311279, -3.675072193145752, -2.309473991394043, -2.5331835746765137, -3.8632068634033203, -1.9422554969787598, -2.0298893451690674, -1.3358426094055176, 2.4828171730041504, 1.450270652770996, -4.527177333831787, -4.489143371582031, -4.062368392944336, -2.7022223472595215, -4.333545207977295, -2.1966118812561035, -1.725101113319397, -4.530283451080322, -2.5544137954711914, -3.724677085876465, -4.499369144439697, -1.4691004753112793, 2.143721580505371, -1.754157304763794, -5.999929428100586, -6.142831802368164, -0.588843822479248, -4.114530086517334, -5.626182556152344, 0.5799915790557861, -0.39975953102111816, -0.7079281806945801, -0.5373237133026123, 3.9736862182617188, 0.8370985984802246, -7.065129280090332, -4.4865546226501465, -6.180740833282471, -3.561420440673828, -4.943317413330078, -5.449490070343018, -1.3773013353347778, -4.9414262771606445, -1.097642183303833, -5.453957557678223, 3.6008853912353516, -3.9716038703918457, -1.3338395357131958, -7.690184116363525, -2.0776546001434326, -2.3981821537017822, -7.204440116882324, -6.784621715545654, -7.5776166915893555, -5.634119033813477, -3.129443645477295, -2.072242498397827, -4.8803181648254395, -0.13673162460327148, -3.453843593597412, 0.8587379455566406, -8.124592781066895, -3.3199706077575684, -0.35194945335388184, -5.3769426345825195, -2.9425323009490967, -4.196028709411621, -5.458468437194824, 3.1855688095092773, 1.8314976692199707, -0.8270888328552246, 3.8804898262023926, -1.4295225143432617, -2.78082275390625, -6.036808967590332, 4.235608100891113, -2.91141939163208, -1.3310952186584473, -2.2507524490356445, -2.9718220233917236, -7.096916198730469, -4.887577056884766, -1.2966980934143066, -3.499753475189209, -3.026355266571045, -1.4375195503234863, 0.9694674015045166, -5.19140625, -1.056800127029419, -2.838711977005005, 1.8805351257324219, -4.521799087524414, -3.986757516860962, -7.347896575927734, -6.992305278778076, -2.916297674179077, 0.3527507781982422, 2.410378932952881, -4.348791599273682, -2.794259786605835, 1.7380759716033936, -3.687131643295288, 0.5549523830413818, 0.8024072647094727, -5.694189548492432, -1.491326093673706, -1.544087290763855, -1.7969436645507812, 0.3615128993988037, -3.872037649154663, -0.7808352708816528, -3.013522148132324, -1.5499624013900757, 4.265119552612305, 1.1316499710083008, -1.7283804416656494, -2.1845085620880127, -3.308043956756592, -8.501646041870117, 1.665459156036377, 0.798344612121582, -4.163174629211426, -2.043684244155884, 1.573251724243164, -3.7753310203552246, 1.32590913772583, 1.2664895057678223, -1.0323073863983154, -5.668086051940918, -3.9017717838287354, -0.4147148132324219, 2.1178665161132812, 2.3646340370178223, -4.805960178375244, 2.049105167388916, -6.972922325134277, -0.6221103668212891, -7.564676284790039, -2.821199893951416, 1.2101469039916992, -2.2737655639648438, -3.707362651824951, -4.397244930267334, -1.6290606260299683, 1.8153605461120605, 2.2546000480651855, -4.387832164764404, -5.410806655883789, -2.0712828636169434, 1.8985753059387207, 0.7495720386505127, -4.300126075744629, -8.860282897949219, -4.250861167907715, -2.2213973999023438, -3.0453505516052246, -3.599870443344116, -2.3655991554260254, -5.8196210861206055, -0.09285736083984375, 1.1724131107330322, 0.2902560234069824, 1.7252516746520996, -2.5368528366088867, 0.5811922550201416, -5.5859761238098145, 4.046750545501709, 1.3923239707946777, -2.013786792755127, -4.864020347595215, -6.508575916290283, -6.287559509277344, -2.977952480316162, 0.5734210014343262, -4.222967147827148, -0.5447711944580078, -0.8792213201522827, -5.5978102684021, -5.574317932128906, -0.5182795524597168, -0.12758421897888184, -2.5170483589172363], [-2.9030895233154297, -8.64937686920166, -2.6508655548095703, -2.6878786087036133, 0.6162333488464355, -1.1958714723587036, -2.502093553543091, -2.122102737426758, -3.756380081176758, -3.419600009918213, -5.191798210144043, -4.593985557556152, -2.7938594818115234, -3.9870917797088623, -7.331182479858398, -2.971435546875, -0.06337106227874756, -0.3140918016433716, -1.1987773180007935, -2.1791129112243652, -7.302692413330078, -4.675209999084473, -5.523780345916748, 0.2207179069519043, 3.9588069915771484, -4.301074028015137, 1.2888097763061523, -2.034320592880249, -1.9087505340576172, -3.6453139781951904, -0.17228877544403076, -0.980339527130127, -2.307633399963379, 0.24269938468933105, -1.976071834564209, -0.25423121452331543, -1.613370656967163, -5.8638997077941895, 0.3187835216522217, -2.8877227306365967, -3.0869503021240234, -0.050563693046569824, -3.9214577674865723, -0.5949965715408325, 0.8858914375305176, -1.4870185852050781, 3.3120288848876953, -2.6522443294525146, -5.020809173583984, -8.85527229309082, -4.38356351852417, 1.9254331588745117, -4.509161949157715, 3.7108817100524902, 0.730881929397583, 1.2163844108581543, -5.34122371673584, -2.2515244483947754, -4.687224388122559, -8.353971481323242, -2.2848663330078125, -0.4147453308105469, 2.4026689529418945, -3.8782947063446045, -1.1685104370117188, -3.412569046020508, -0.44249916076660156, -1.1746374368667603, 0.9796817302703857, -5.417234897613525, -5.424248218536377, 1.1772794723510742, -0.3968069553375244, -5.075544834136963, -4.543237209320068, -8.01395320892334, 0.7157125473022461, -2.6380887031555176, -3.3853836059570312, -4.343347549438477, 0.2513768672943115, -3.1009445190429688, -1.264450192451477, -2.811398506164551, -1.646315097808838, 1.7942252159118652, -4.730015754699707, -3.5326426029205322, -4.169307708740234, -0.42993998527526855, -0.011317253112792969, 3.9894537925720215, -6.979202747344971, -9.992501258850098, -0.37773895263671875, -4.210591793060303, -4.636216163635254, -1.1041620969772339, -0.6711795330047607, -0.9158356189727783, -3.1598074436187744, -4.731478691101074, -3.2829723358154297, 4.833561897277832, -0.9134302139282227, -0.4745587110519409, -5.49038028717041, -1.6261322498321533, -6.774467945098877, 0.8642861843109131, 3.986769199371338, -2.038783073425293, 0.22568130493164062, -2.554853677749634, -1.8027302026748657, -3.2416553497314453, -1.0909221172332764, -3.7056424617767334, -8.135976791381836, 0.2068793773651123, -0.996645450592041, 3.1102123260498047, -2.1054487228393555, -2.92758846282959, 1.6082913875579834, -3.603107213973999, -0.9346004724502563, -1.9993258714675903, -2.2813851833343506, -2.9570746421813965, -7.06734037399292, 1.1624102592468262, 4.263101100921631, 1.54667067527771, -3.328383445739746, -2.141083002090454, 0.43347978591918945, -7.805800437927246, 1.520209789276123, 4.4651641845703125, -0.4583911895751953, -3.590884208679199, -3.107221841812134, -5.747842311859131, -1.7738999128341675, -2.7141802310943604, -1.6262942552566528, 1.5611863136291504, -0.8533915281295776, -3.776315689086914, -0.9072365760803223, -6.498661041259766, -0.7038660049438477, -3.271331310272217, 0.35411691665649414, -0.7296637296676636, 2.115224838256836, 1.211491584777832, -3.347856044769287, 2.276890754699707, 1.782203197479248, -3.5155768394470215, 2.5413618087768555, -0.6787407398223877, 2.1205453872680664, -0.045267343521118164, -3.9636220932006836, -2.622969150543213, -2.575561046600342, -7.7993621826171875, -5.2103071212768555, -1.1653497219085693, -1.441758632659912, -4.119247913360596, 0.45075345039367676, 2.450509548187256, -6.313545227050781, -4.0446882247924805, -2.1172704696655273, -3.372745990753174, -4.9205098152160645, -3.4833028316497803, 0.4293861389160156, -2.7079224586486816, 4.516279220581055, 2.0489706993103027, -2.711486339569092, -3.6714463233947754, -1.994736671447754, -3.1294760704040527, 2.0474963188171387, 2.6589040756225586, 3.2416677474975586, -2.4733803272247314, -3.629991054534912, -1.4738121032714844, -3.4798901081085205, 1.5247869491577148, -2.914942741394043, -6.585492134094238, -8.878532409667969, -6.467171669006348, -0.40735507011413574, -6.540367603302002, -1.2365117073059082, -6.124929428100586, -3.222714900970459, -0.31741416454315186, -1.41324782371521, -6.855162620544434, -4.172393321990967, 1.4019172191619873, -2.483116865158081, 2.2905735969543457, 2.076016902923584, 1.4640684127807617, -2.9000027179718018, 2.9109740257263184, -8.180150985717773, 0.18729305267333984, -0.4540609121322632, -3.423279285430908, 3.4518775939941406, -1.3058617115020752, -1.1441717147827148, -5.073812484741211, -2.8682756423950195, -0.5522469282150269, -2.2310714721679688, -4.850119590759277, -0.6553560495376587, 3.916656970977783, -4.225244522094727, -2.8645031452178955, -1.431929111480713, -5.67694616317749, 1.785109043121338, 1.8819396495819092, -5.470062732696533, -2.101998805999756, 6.109468460083008, -0.20632433891296387, -4.84860372543335, 2.8487391471862793, -2.2054851055145264, 0.03868222236633301, -3.58357572555542, 2.8387365341186523, 1.0247249603271484, -1.3463754653930664, -3.070119619369507, -3.8842480182647705, -2.1664183139801025, -1.0815074443817139, 1.0922694206237793, -3.888237237930298, -4.137355327606201, 1.2614455223083496, -3.466627836227417, -0.6470679044723511, -0.8035118579864502, -4.001564979553223, 0.3052845001220703, -0.49051499366760254, -6.140511989593506, 1.4508452415466309, -4.019079208374023, 4.91455078125, -6.7736382484436035, -1.8722190856933594, -0.48501014709472656, -7.664678573608398, -0.6273583173751831, -3.500162124633789, 3.188678741455078, 1.885664701461792, -0.0384136438369751, -6.7058281898498535, -3.1784937381744385, -4.443832874298096, -3.1838393211364746, -5.906185150146484, 0.5053040981292725, -0.02507197856903076, -2.304995059967041, 0.03867149353027344, -4.251772880554199, -2.1789050102233887, -2.454122543334961, 3.5875654220581055, -2.7574939727783203, -0.6888099908828735, -2.1181163787841797, -0.7010915279388428, 0.08909893035888672, -0.7293070554733276, -5.250153064727783, -2.7252964973449707, -0.7662205696105957, 5.388904571533203, -2.1867408752441406, 0.23634600639343262, -2.0326290130615234, -5.491695404052734, 3.16056489944458, 0.6564390659332275, -7.648964881896973, -4.017017841339111, -4.84255313873291, -1.6706570386886597, -4.9236345291137695, -2.7549967765808105, 0.22482061386108398, -1.23685622215271, -4.94657039642334, -4.496245384216309, -2.2275402545928955, -3.769653081893921, 0.36471986770629883, -0.24719548225402832, -6.161250591278076, -4.65034818649292, -3.146754741668701, -10.110522270202637, 0.39832139015197754, 0.5175604820251465, -1.645672082901001, -3.5926923751831055, -2.1732804775238037, -3.1057629585266113, -0.3478281497955322, -4.3267621994018555, -4.178539276123047, -1.903693437576294, -6.334742546081543, -1.3482747077941895, 3.035810947418213, -4.855549335479736, -2.590519428253174, -1.0797131061553955, -1.7400014400482178, -5.959385871887207, -2.346454620361328, -3.9697141647338867, -1.8228883743286133, -4.5159478187561035, -2.2206709384918213, -5.163520812988281, 1.0080945491790771, -4.066168785095215, -3.8472068309783936, -3.339977264404297, -4.314878463745117, -3.764263868331909, 0.44194507598876953, 0.8439216613769531, -0.29845237731933594, 8.856659889221191, -3.396648645401001, -2.331719160079956, -0.3520963191986084, -10.654865264892578, 3.117569923400879, 1.823807954788208, 0.3682060241699219, -0.9831326007843018, -2.5401368141174316, 5.5023932456970215, -1.6068909168243408, 0.24211549758911133, 4.281723976135254, -3.644909620285034, 1.1882944107055664, -5.917977333068848, -1.4774529933929443, -7.4998626708984375, -0.27269792556762695, -3.481971502304077, 0.17804861068725586, 3.2643632888793945, -1.081229567527771, 1.803433895111084, -2.0825371742248535, -8.376235008239746, -0.9215145111083984, -3.469696521759033, -7.409534454345703, -5.666750431060791, -1.7930777072906494, -4.1066389083862305, -2.1239233016967773, -1.290671467781067, -2.486224889755249, 1.4203181266784668, 7.020359992980957, -7.177826404571533, -3.5791983604431152, 0.5957612991333008, -3.723393678665161, -3.3313982486724854, -3.933422565460205, -3.8555214405059814, 2.0541367530822754, -3.385213851928711, 4.725961685180664, -4.8044753074646, -0.7951412200927734, -5.989527225494385, 1.3761193752288818, -3.423400402069092, -3.819918394088745, -1.4896526336669922, -4.189605712890625, -2.217133045196533, 6.095553398132324, 2.076005458831787, 0.9454262256622314, -1.8086246252059937, -6.167695999145508, -3.8919894695281982, -1.6202473640441895, -3.9230692386627197, -1.9010323286056519, -2.338564395904541, -0.7994766235351562, -2.150921583175659, -1.15406334400177, -1.774279236793518, -0.24098920822143555, -6.621932029724121, -0.6542524099349976, 0.18183016777038574, -7.52495813369751, -4.40492057800293, 1.105767011642456, -7.677422523498535, -5.190174102783203, -5.474407196044922, -1.2156989574432373, -8.73425579071045, 3.7667737007141113, -2.4738686084747314, -4.061905384063721, -2.935072898864746, 1.145841121673584, 0.8460793495178223, -3.754361629486084, -3.545319080352783, 3.972395896911621, -0.2570476531982422, 1.8682432174682617, 0.9165134429931641, -4.9490885734558105, 0.2517099380493164, -0.6027214527130127, -3.1889588832855225, -3.3442840576171875, -0.8986798524856567, -0.2949550151824951, -0.283977746963501, 0.9573488235473633, 4.644313812255859, -5.647238731384277, -1.2074499130249023, -1.1739881038665771, -1.7720733880996704, 0.7363014221191406, 0.3952760696411133, -4.749443054199219, -2.4602243900299072, -9.501798629760742, -1.2813339233398438, 1.2799363136291504, -4.7722063064575195, -0.7514017820358276, -0.19514751434326172, -2.419241428375244, -4.52904748916626, -6.76995849609375, -1.8742578029632568, 3.6379518508911133, 2.6386756896972656, -2.8511738777160645, 0.09733295440673828, 2.288426399230957, -3.889251708984375, 1.561495304107666, -5.804689407348633, -5.689513683319092, -1.7047126293182373, -1.5831159353256226, -4.530673027038574, -6.630390167236328, -6.708881855010986, -0.028377175331115723, -2.0407845973968506, -2.6865615844726562, -2.1527180671691895, -3.5751328468322754, -8.628958702087402, 2.0564794540405273, -8.188133239746094, -6.34246826171875, 0.06893730163574219, -1.5530412197113037, 1.4189224243164062, -5.430947780609131, -2.2904253005981445, -2.924938917160034, 1.794008731842041, -1.4896992444992065, -5.102687358856201, 3.248317241668701, -3.2121143341064453, -1.780236005783081, -6.316191673278809, -2.581106662750244, -0.07647597789764404, -0.2992074489593506, -2.0617287158966064, -2.2752840518951416, -4.970033168792725, 1.692004680633545, -3.0907623767852783, -2.7717957496643066, -3.4141945838928223, -4.924302101135254, 2.2507314682006836, -1.2031283378601074, -5.191565990447998, -0.9064308404922485, -3.633258819580078, -5.077329635620117, -3.514240264892578, 2.4516334533691406, -4.103596210479736, -4.888127326965332, -2.8789291381835938, 1.7191963195800781, -3.0161211490631104, -3.1308705806732178, -4.797798156738281, -7.120502471923828, -6.510680675506592, -1.580949068069458, 2.971500873565674, -6.134216785430908, 2.0104870796203613, -4.076087474822998, 0.4623241424560547, -2.7463324069976807, -0.987558126449585, 0.8588643074035645, 2.0088930130004883, -2.5432143211364746, 1.610072374343872, -4.08530330657959, 2.838226318359375, 0.8820488452911377, -4.941954612731934, 0.7597498893737793, -2.7430965900421143, -1.5326600074768066, -3.7736029624938965, 0.3972208499908447, -4.216743469238281, -5.287875175476074, -3.7972207069396973, -1.7944411039352417, -6.297266960144043, -3.503333330154419, -0.15515899658203125, -3.558497190475464, -1.059841275215149, 3.226116180419922, -4.057688236236572, -3.485959053039551, 3.775315284729004, -2.1955792903900146, -1.6392254829406738, -3.449411392211914, -9.298633575439453, -5.169612884521484, -0.8059695959091187, 0.32322096824645996, -1.7658071517944336, -2.076585531234741, -3.337299108505249, -0.36550068855285645, 0.6796741485595703, -7.332101821899414, 2.1431632041931152, -6.95146369934082, -3.203495979309082, -1.3957107067108154, 1.1950912475585938, -1.206111192703247, 2.024247646331787, -4.471593856811523, -3.9122121334075928, 6.062448501586914, -0.5458415746688843, -8.477128982543945, 0.5654737949371338, 5.689526557922363, -2.2068800926208496, -3.5192861557006836, -7.848088264465332, 2.4322872161865234, -0.08202552795410156, -3.256612539291382, -2.810559034347534, -1.7088840007781982, -6.158235549926758, -4.838730812072754, -4.1568756103515625, -2.2826669216156006, -0.5327677726745605, -4.15221643447876, -4.985806941986084, -2.5654096603393555, -1.4339969158172607, -6.436134338378906, 0.9432549476623535, 1.0732436180114746, -3.433513641357422, 0.44119763374328613, -7.287923336029053, -3.533435821533203, -1.4117416143417358, 0.13150334358215332, -4.264373779296875, -2.3763632774353027, -5.775247573852539, -1.819673776626587, -1.4225362539291382, -3.016730308532715, 3.110642433166504, 0.9120047092437744, 2.890986442565918, -1.1329233646392822, -5.002322196960449, -6.986023902893066, -3.331726551055908, 2.1982932090759277, -0.6396955251693726, -2.9866156578063965, -2.8013179302215576, -0.4238746166229248, 0.3410029411315918, -1.433698058128357, -1.4952020645141602, -8.064432144165039, 0.3352363109588623, -3.016401767730713, -7.7172532081604, 2.951511859893799, -5.764615058898926, 0.6834063529968262, -3.4686341285705566, -0.6530957221984863, 3.8418617248535156, -0.42382097244262695, 1.8462958335876465, -0.34401869773864746, 1.9124369621276855, -2.765723705291748, 3.1417007446289062, -5.090119361877441, -3.251793146133423, 2.6013755798339844, -5.379090309143066, -2.7294740676879883, -3.13424015045166, 4.732512474060059, 0.6147980690002441, -2.9641518592834473, -3.056455135345459, 4.180832386016846, -4.556885242462158, -0.7142961025238037, 0.4020500183105469, -5.357649326324463, -1.1466681957244873, 0.26062679290771484, -2.8880138397216797, -3.5297892093658447, -0.3782517910003662, -4.542179584503174, -0.7853238582611084, -7.116603851318359, 0.8808350563049316, -2.84633731842041, -3.3333568572998047, 2.668224334716797, -1.5285394191741943, 1.4971990585327148, -0.04700136184692383, 1.7324943542480469, 0.05212569236755371, -3.410308599472046, -1.5545310974121094, -0.45325422286987305, -3.8921799659729004, -0.35719025135040283, -1.675201654434204, -8.531366348266602, -2.5321357250213623, -1.977900505065918, 3.193587303161621, -1.7543864250183105, -4.280432224273682, -1.6480162143707275, 1.4126005172729492, -1.8810409307479858, -3.0513358116149902, 0.22534513473510742, -0.745598316192627, 0.03857278823852539, -7.440193176269531, -0.5059864521026611, -2.142367124557495, -2.908015251159668, -2.030913829803467, -5.386073112487793, -0.45275449752807617, -1.349583387374878, 3.067537307739258, -2.2156546115875244, -4.670536518096924, -3.87314510345459, -4.045207977294922, -6.619854927062988, -6.222813606262207, -2.8251657485961914, -1.0522979497909546, -4.93669319152832, -4.584868431091309, -5.329913139343262, -0.5622148513793945, -6.082106113433838, -2.5508952140808105, -2.883796215057373, -3.7038960456848145, -2.469207286834717, 2.3766722679138184, -2.8323609828948975, -4.687247276306152, -4.981505870819092, -3.27934193611145, -4.194467067718506, 0.26851463317871094, -1.4751216173171997, -0.23776531219482422, 0.1472775936126709, -0.6039628982543945, 2.1174674034118652, -7.685940265655518, -2.7582290172576904, 2.4043092727661133, -1.301066279411316, -2.952969551086426, -4.177175998687744, 1.5774941444396973, -4.822924613952637, 0.4896059036254883, 2.683676242828369, -0.41936302185058594, -3.288325786590576, -2.484102964401245, 3.480912685394287, -1.8662998676300049, -2.50405216217041, 0.4093630313873291, -5.688943862915039, -1.727283000946045, -3.539498805999756, -0.6213293075561523, -0.6550962924957275, 1.5930943489074707, 0.004457235336303711, -2.6462273597717285, -1.1606953144073486, -5.783097267150879, -3.3770551681518555, -1.2601869106292725, -1.1689924001693726, -5.801779270172119, -1.067541241645813, 2.503429889678955, -1.2711725234985352, -1.8735452890396118, -2.635329484939575, -6.60999059677124, -6.608023643493652, 1.1601438522338867, 0.8055739402770996, -2.7638492584228516, 2.56778621673584, -3.382347583770752, 1.7088737487792969, -5.0127692222595215, -2.4879066944122314, 1.8779582977294922, -1.612918734550476, -7.123793601989746, -4.359828948974609, 4.211270809173584, -0.131103515625, -0.40387511253356934, -0.10396039485931396, -2.993499755859375, 4.4624176025390625, 0.012038946151733398, -2.758854389190674, -2.7989652156829834, -1.4811468124389648, 1.356313705444336, -7.051602840423584, -1.024332880973816, 0.22925186157226562, 3.1704068183898926, -2.576366662979126, 0.028299331665039062, -2.1319758892059326, 1.2928948402404785, -8.269745826721191, 0.1721043586730957, -6.977880477905273, -3.5083069801330566, -4.073827266693115, -4.869767189025879, -5.83976936340332, -1.236294150352478, -3.9342267513275146, 0.11239480972290039, -4.660874366760254, -1.5247033834457397, 0.6364345550537109, -3.567959785461426, 3.49857759475708, -1.2354882955551147, 0.5503695011138916, 0.4767110347747803, 0.3885383605957031, 3.9913768768310547, -3.6687746047973633, -1.8811959028244019, -6.133135795593262, -4.45090389251709, -1.951833963394165, 3.3499794006347656, -4.341745376586914, -5.867388725280762, -5.410492897033691, 4.256945610046387, -2.2055764198303223, -8.376450538635254, 2.2852840423583984, -4.2522077560424805, -6.2054123878479, -2.4779648780822754, -2.3292362689971924, -0.41578400135040283, 0.11460471153259277, -0.8443881273269653, -2.4807050228118896, -4.042594909667969, -7.772605895996094, -2.6228179931640625, -1.4204416275024414, -7.9932169914245605, -3.710068702697754, -6.286703109741211, -2.2076058387756348, -0.3606691360473633, -2.98349666595459, -4.937231063842773, -1.5673837661743164, -5.942413330078125, -3.0298280715942383, -4.478186130523682, -2.2489817142486572, -0.331068754196167, -6.000675201416016, 0.8040742874145508, 0.9704334735870361, -4.478183269500732, -0.7145227193832397, -7.132145881652832, -5.7495293617248535, 0.2835977077484131, -0.4923286437988281, -9.651572227478027, -3.8878540992736816, -0.5007600784301758, -0.769829273223877, 1.3960199356079102, 1.3785412311553955, 5.993476867675781, 0.474606990814209, 3.389615535736084, -0.6150933504104614, -3.7592945098876953, 0.8938848972320557, -2.716264247894287, -1.3213906288146973, -7.069922924041748, 1.7457993030548096, -5.109073162078857, 2.7411251068115234, 4.211184501647949, -2.37515926361084, 0.0013892650604248047, 2.810140609741211, -6.136078834533691, -1.5324771404266357, -5.248069763183594, -3.3815274238586426, -0.3604855537414551, -3.208690881729126, 1.5120630264282227, -1.7467353343963623, 0.01810002326965332, -1.9937111139297485, -4.970282554626465, -0.4677215814590454, -3.7781968116760254, -3.226750135421753, 3.307898998260498, -1.2901484966278076, 2.5832338333129883, -1.8497204780578613, -1.7306931018829346, -4.909432411193848, 1.8280611038208008, -0.7026467323303223, 1.7337546348571777, -2.6827261447906494, -4.223647117614746, -3.604003667831421, 0.0879814624786377, -3.2478525638580322, -3.5651907920837402, -0.8229517936706543, 1.6023688316345215, -1.9551466703414917, -9.11281967163086, 0.2661254405975342, -3.3182520866394043, -2.8024067878723145, -4.9801483154296875, 5.241115570068359, 0.643834114074707, -6.86225700378418, -2.6240391731262207, -3.5162642002105713, -4.5136284828186035, -4.668614864349365, -0.5800738334655762, -1.9887479543685913, 3.204099655151367, -4.533058166503906, 0.6964895725250244, -0.8976953029632568, -2.219545602798462, -4.116618633270264, -1.953490138053894, -0.8531787395477295, -1.5955617427825928, -4.341373443603516, 1.8051199913024902, 2.5590758323669434, -0.8017189502716064, -1.368722677230835, -3.763758659362793, -5.912761688232422, -1.6310852766036987, 0.8102636337280273, -5.650455474853516, -1.317613959312439, -5.335940837860107, -4.62462043762207, 4.721992015838623, -8.513505935668945, -5.006936550140381, 2.24778413772583, -4.123728275299072, 1.329831600189209, -3.617021322250366, -3.300384998321533, -1.8837196826934814, 2.4590907096862793, -1.2157222032546997, -1.9607969522476196, -6.792542934417725, -7.6006598472595215, -2.2240471839904785, -1.3210082054138184, -3.899982213973999, -1.5941081047058105, 0.4065990447998047, -2.4992663860321045, -0.9069355726242065, -3.9123122692108154, 0.41129541397094727, -6.380399703979492, 0.6806826591491699, -5.498444080352783, -1.8530137538909912, -6.593748092651367, 2.5482029914855957, -3.9105160236358643, -3.250771999359131, 1.4162359237670898, 0.03729820251464844, -1.5991802215576172, -2.448578119277954, -9.38489818572998, -0.17029547691345215, -2.9809000492095947, 0.4685404300689697, -4.637290000915527, 1.6077933311462402, -6.461273670196533, -1.4166860580444336, -1.2437483072280884, 0.8536176681518555, -4.7420654296875, -0.41193318367004395, -2.9814605712890625, -8.502349853515625, 0.4222838878631592, -2.421997308731079, -3.876570224761963, -6.623528480529785, 7.579509735107422, 0.9504594802856445, -6.659175872802734, 0.5405902862548828, -4.243056297302246, -6.6468281745910645, 1.5714788436889648, -0.381022572517395, -0.01897573471069336, -1.6892064809799194, -4.202610969543457, -3.281942367553711, 1.5247066020965576, 0.99932861328125, -3.2092032432556152, 1.3633992671966553, -4.269730091094971, 0.8212528228759766, 0.9155733585357666, -5.37287712097168, 3.443148612976074, -7.422717094421387, -3.5458552837371826, -2.0326809883117676, -3.4183506965637207, -2.935926914215088, -2.606799602508545, -4.2657365798950195, -0.44541633129119873, -0.9055535793304443, -6.676932334899902, -1.9640363454818726, -1.9405640363693237, -3.0556211471557617, -1.611968994140625, -4.347536087036133, 4.17019510269165, -2.4673187732696533, -1.211700677871704, -0.765447735786438, -0.9379113912582397, -2.178863048553467, -1.331031322479248, -3.3861477375030518, -3.942495822906494, -4.527369499206543, 3.2559804916381836, -0.6088881492614746, -5.457340240478516, -0.06420183181762695, 4.080458641052246, 1.2232823371887207, -7.779684066772461, -3.8591346740722656, -1.9330614805221558, 0.6170840263366699, -4.065042495727539, -2.086150884628296, 2.866596221923828, 2.0871639251708984, -1.5368067026138306, -4.478515148162842, -3.577188491821289, -1.9942340850830078, 1.450502872467041, -0.22956132888793945, 0.8523759841918945, 0.5783114433288574, -4.1387176513671875, -4.170652389526367, -0.9989248514175415, -0.7477171421051025, -3.935394287109375, -2.2498061656951904, 0.37017345428466797, -0.10904955863952637, -3.9803147315979004, -5.473077774047852, -1.2080719470977783, -2.75797963142395, 2.2903318405151367, -1.0195622444152832, -9.7407808303833, -3.2926254272460938, -3.2857460975646973, -0.9363027811050415, 3.2864556312561035, 0.3170599937438965, 1.6795196533203125, 0.35765671730041504, -4.535404205322266, -5.032466411590576, -5.479085922241211, 2.0578222274780273, -5.295949935913086, -5.592828273773193, -0.7516326904296875, -1.1091114282608032, -1.2333433628082275, -4.045782089233398, -2.4040634632110596, -4.1426520347595215, 1.0591154098510742, -3.8039660453796387, -4.273491859436035, -0.35842621326446533, 2.0964555740356445, 1.8905138969421387, -4.635456085205078, -4.285257339477539, 0.945805549621582, -0.8747472763061523, 2.1460795402526855, -3.2140719890594482, -7.374704360961914, -2.4193501472473145, -3.233243465423584, -2.5963146686553955, -0.39737486839294434, -2.32608699798584, -2.6893815994262695, 6.116334915161133, -7.1125288009643555, 4.4703369140625, 4.72698450088501, 1.0102319717407227, -8.677274703979492, -1.867443561553955, 1.2746763229370117, -0.6929092407226562, -6.356837272644043, 0.6137280464172363, 1.9067778587341309, 7.576374053955078, -1.5108141899108887, 1.3282744884490967, -4.796994209289551, -1.353968620300293, -1.0923207998275757, 2.3262181282043457, -1.4031872749328613, -0.08899223804473877, -3.2767128944396973, -4.551858901977539, -8.299039840698242, 3.4500951766967773, -6.964534282684326, -1.1944806575775146, -1.0081722736358643, -0.3213651180267334, -5.374453544616699, -5.362258434295654, -0.6481091976165771, -5.494541168212891, -2.899378776550293, -2.141702651977539, -3.2271006107330322, -5.082752227783203, -2.49678897857666, 1.9328114986419678, -0.9148584604263306, -2.103545665740967, 2.1690759658813477, -3.818352222442627, -2.9905078411102295, -0.07639157772064209, -5.880841255187988, -5.279626846313477, -2.8869738578796387, 1.0125598907470703, -4.993680953979492, 1.0177862644195557, -3.301077127456665, -2.9135217666625977, -8.800683975219727, -3.209745407104492, 5.1692070960998535, -2.7594668865203857, 2.101315975189209, -2.646759033203125, -0.43003416061401367, 1.1587843894958496, -3.8267407417297363, -1.3266533613204956, -1.717012643814087, -1.7182681560516357, 3.368363857269287, 1.1580486297607422, -7.013660907745361, -3.496735095977783, 4.487188339233398, 2.349140167236328, -4.595502853393555, -3.1662402153015137, -3.1868839263916016, -1.842247724533081, -6.373469352722168, -0.8181781768798828, 0.07488131523132324, -2.299316644668579, -0.4563608169555664, 2.2385473251342773, 2.6048054695129395, -0.7025699615478516, -1.2221484184265137, 0.10010623931884766, 1.2658934593200684, 3.84847354888916, -1.6489143371582031, -5.520425319671631, -8.146675109863281, -5.3120503425598145, 1.0842976570129395, -7.4754862785339355, -1.4553292989730835, -4.658535957336426, -1.7076462507247925, 0.31891393661499023, -4.384248733520508, -8.3363037109375, 2.459535598754883, -1.8299028873443604, -2.144885301589966, 1.0453338623046875, 0.36652064323425293, 1.5272600650787354, -0.8949086666107178, 0.441436767578125, -2.178349256515503, -4.040441513061523, -0.24474668502807617, -5.5679097175598145, -3.7945361137390137, 1.4821906089782715, -2.9947736263275146, -4.959242820739746, -0.26064443588256836, 2.041003704071045, -7.349782943725586, 0.9542527198791504, -4.776385307312012, -2.0612759590148926, -3.164161205291748, -7.132099628448486, 1.9606521129608154, 0.08788061141967773, -1.2704054117202759, -3.9268076419830322, -1.5385407209396362, -3.3022942543029785, -2.455631732940674, 1.032010555267334, 0.01780223846435547, -5.9062275886535645, -1.6867727041244507, -5.2993550300598145, 0.04977750778198242, -2.317331075668335, 1.5631227493286133, -5.0836873054504395, -3.414430618286133, -6.774604797363281, -1.272840142250061, 2.037649154663086, -2.4204585552215576, -1.8275790214538574, -1.5611159801483154, -3.794384717941284, -2.815044403076172, -0.24190115928649902, -1.283189296722412, -0.7499308586120605, 2.690214157104492, -4.923811912536621, -4.616807460784912, -0.9230982065200806, -4.172660827636719, -1.9041929244995117, -0.5271899700164795, 6.0985565185546875, 1.3372068405151367, -8.602344512939453, 1.4730372428894043, -2.071108818054199, 1.6639108657836914, 3.1393914222717285, 0.31183576583862305, -0.4083120822906494, -4.301963806152344, -1.8702822923660278, -1.563016414642334, 3.4208545684814453, -1.225038766860962, -2.991426467895508, -4.32329797744751, -1.3835070133209229, -2.0191872119903564, -2.9296422004699707, -0.011718034744262695, -12.56415843963623, -2.340425729751587, -3.4937634468078613, -0.028766632080078125, -7.980159759521484, -0.02053999900817871, -3.8580408096313477, -0.20567631721496582, -6.5585808753967285, 1.2333412170410156, 3.6284637451171875, -2.9218056201934814, -1.048531413078308, -1.9616079330444336, -1.7227647304534912, -2.051231861114502, -2.240593910217285, -0.3749408721923828, -8.360931396484375, 2.6726231575012207, 0.5589196681976318, -3.6906630992889404, -0.7430980205535889, -3.49245023727417, 0.9589352607727051, 2.4220619201660156, 0.9211404323577881, -4.352343559265137, -1.8037681579589844, -1.9048234224319458, -4.175985813140869, 1.2506537437438965, -8.174269676208496, -5.362015724182129, -2.3556926250457764, -4.549689769744873, -2.8800172805786133, -3.1372909545898438, 2.1020145416259766, -4.838674545288086, -3.8901636600494385, 0.04078364372253418, -4.733958721160889, 0.6185858249664307, 3.4304614067077637, -2.395644187927246, -2.061938524246216, -3.1272547245025635, -3.9324989318847656, -5.258851528167725, -1.8639699220657349, -0.9408338069915771, 2.1041927337646484, 0.01256251335144043, -6.180184364318848, -1.9332809448242188, 1.7303838729858398, 4.836653709411621, -0.014171600341796875, 1.4021811485290527, -2.030538558959961, -4.856418609619141, -1.260087490081787, -2.410689353942871, -3.214653968811035, -0.5035802125930786, -1.813920497894287, -4.684405326843262, -2.010590076446533, -1.4268288612365723, -5.4007248878479, -2.5559465885162354, -7.564220428466797, 3.355782985687256, -0.5268094539642334, 0.3126647472381592, -0.04496121406555176, -5.023033142089844, -3.671821355819702, 3.4515204429626465, -4.671440601348877, -0.9372820854187012, 1.2892858982086182, -5.217257499694824, -0.5314873456954956, -1.9238430261611938, 1.4226527214050293, -1.828214168548584, -2.0454177856445312, -5.32108211517334, -3.1870625019073486, -5.875500679016113, -6.156820297241211, 0.7326419353485107, 2.697970390319824, -3.32358455657959, -0.7547644376754761, -0.7727593183517456, -5.618127346038818, -3.476994752883911, 0.9266502857208252, -0.023758888244628906, -1.0730390548706055, 0.576636791229248, -4.5183563232421875, -10.303531646728516, 1.035871982574463, -4.456622123718262, -2.750352144241333, -5.465132713317871, -3.2256295680999756, -1.7116491794586182, -0.1131293773651123, 1.0026369094848633, -7.681315898895264, -1.694419503211975, -0.29789483547210693, -0.8098018169403076, -3.2683825492858887, 4.022783279418945, 1.022618293762207, -0.8902838230133057, -0.09639942646026611, 1.3004131317138672, -4.582533836364746, -0.36579179763793945, -5.010898590087891, 1.04093337059021, -4.963769912719727, -2.1797375679016113, -1.401738166809082, -0.2104332447052002, -0.044530510902404785, -0.18893837928771973, -4.6186418533325195, -5.258079528808594, -10.170161247253418, -3.49111008644104, -7.699773788452148, -2.2686054706573486, -2.1333649158477783, -4.466104030609131, 0.7101821899414062, -0.7161978483200073, -1.7837607860565186, 0.7742698192596436, -3.8174843788146973, -1.6016254425048828, -4.554215908050537, 2.418593406677246, 3.231513023376465, -1.0341747999191284, -0.5108523368835449, 0.35396766662597656, -3.0681962966918945, -0.6852318048477173, -3.0537919998168945, 3.210205554962158, -8.636436462402344, -1.4845335483551025, 2.617145538330078, 1.995678186416626, -0.8834934234619141, -1.3899385929107666, -0.1978590488433838, -1.0276501178741455, 2.9453210830688477, 0.8451905250549316, 1.294739007949829, -5.597984790802002, -3.7191829681396484, -5.5933098793029785, -6.725332260131836, -1.676929235458374, 2.2093634605407715, -4.609645366668701, 0.3878493309020996, -1.5032916069030762, -3.0213212966918945, -3.3074066638946533, -3.7833542823791504, 1.7596445083618164, 0.092803955078125, -1.0256218910217285, -1.9717401266098022, 0.3984832763671875, -2.278188705444336, -0.16893815994262695, -2.5162198543548584, -1.2440365552902222, -1.4349961280822754, 0.6343152523040771, 0.043598175048828125, -5.7920823097229, 2.420053482055664, 2.389481544494629, -1.649524450302124, -5.992313385009766, -6.822971820831299, 0.6514110565185547, 0.963355302810669, -7.458433628082275, -1.0406423807144165, -3.66076922416687, -3.2670130729675293, 1.778174638748169, 0.18179750442504883, -2.260284185409546, 0.29491186141967773, 3.298675060272217, -5.689586162567139, -4.151010990142822, -3.012681484222412, 0.45609474182128906, 1.3982677459716797, -3.5753121376037598, -2.287674903869629, -3.6806962490081787, -1.0430810451507568, -4.169421195983887, -1.0045344829559326, -2.177640676498413, 0.5944080352783203, -1.0651576519012451, -7.029354572296143, -5.474569320678711, -5.569739818572998, 1.5813369750976562, 5.61386775970459, -5.597161769866943, -0.3163384199142456, 2.5231056213378906, 0.7402055263519287, -8.216541290283203, -1.3104486465454102, -3.0676393508911133, -1.2217005491256714, -0.6494030952453613, -5.948636531829834, 2.0388741493225098, 6.054653167724609, -5.306848526000977, 0.905123233795166, -1.0161885023117065, -0.28774070739746094, 1.459632396697998, 1.335824966430664, -3.2926764488220215, -1.763175368309021, -5.154177188873291, 2.7184205055236816, -1.387640118598938, -2.180227041244507, -3.7847018241882324, -2.2410521507263184, -3.3776350021362305, 0.7283482551574707, -9.147832870483398, -3.749403953552246, 0.01610279083251953, -0.9506497383117676, -1.95419442653656, -0.36542820930480957, -1.2445942163467407, -3.186643123626709, -0.6895284652709961, -1.5260558128356934, -10.443580627441406, 3.254426956176758, -4.900705337524414, -4.682351112365723, -7.788104057312012, 3.7881460189819336, -6.624947547912598, -5.681797027587891, -1.6747233867645264, -5.653756618499756, -1.7195830345153809, -3.157325267791748, -1.4650181531906128, -3.175264835357666, -4.811165809631348, -0.18230819702148438, -2.3727731704711914, -2.1252646446228027, 1.7046022415161133, -0.26903700828552246, -4.744349956512451, 0.63153076171875, -2.441377878189087, 2.9359841346740723, -1.463607668876648, -0.2914767265319824, -2.5214948654174805, -2.845728874206543, 0.6408824920654297, -1.0954172611236572, -1.952837347984314, -0.3610110282897949, 0.9221606254577637, -3.9778692722320557, -5.47593879699707, 0.4632275104522705, 4.719583511352539, -4.091657638549805, 0.2286229133605957, -3.8637382984161377, -4.7108612060546875, -2.1847498416900635, -3.6989665031433105, 1.570511817932129, 3.0192222595214844, -5.357106685638428, -2.9543473720550537, -4.676210403442383, -7.164328575134277, -2.7021381855010986, -3.5422050952911377, 0.5741755962371826, 2.4095072746276855, -3.053131103515625, -2.393012046813965, -2.6756157875061035, -1.6167738437652588, -4.047731399536133, -1.6044588088989258, -0.7967162132263184, 0.03203129768371582, 0.41271162033081055, 4.212681770324707, -3.1838364601135254, -6.049241065979004, -0.6456027030944824, -0.9857475757598877, -3.361859083175659, -4.539872646331787, -3.514679193496704, -3.0820870399475098, -4.2130632400512695, -0.7250008583068848, -5.8275556564331055, 3.1904873847961426, -3.9620344638824463, 1.3507208824157715, -5.735632419586182, -1.381475567817688, -2.801426410675049, -3.1835665702819824, 0.7248244285583496, -2.0418708324432373, -1.3478002548217773, -4.720366954803467, -0.2339770793914795, -2.2812671661376953, -6.247158050537109, 1.4808168411254883, -1.2591902017593384, -5.4606218338012695, -4.293435096740723, 3.7539825439453125, -3.9614434242248535, -2.904446840286255, -4.583290100097656, -6.410126686096191, -7.5045342445373535, -5.502199172973633, 3.402578830718994, -1.2605894804000854, 4.91298770904541, -2.2363274097442627, -3.528611183166504, -0.9975314140319824, -5.193699836730957, 4.036954402923584, -3.5343005657196045, -3.0828633308410645, -4.4704155921936035, 2.138925075531006, -3.571031093597412, -0.5306650400161743, -0.9670406579971313, -5.390960693359375, 0.09922289848327637, -6.992621421813965, -7.336930274963379, -2.4640185832977295, -1.7104215621948242, -2.7009100914001465, -4.697078704833984, -0.7247763872146606, -0.6222299337387085, -4.550064563751221, -3.110922336578369, -3.602466106414795, -4.649715900421143, -1.190218448638916, -2.922041654586792, -2.688446283340454, -1.7665654420852661, -6.529221534729004, -1.8771679401397705, -0.1858820915222168, -5.364174842834473, -1.6633307933807373, -2.287360906600952, -2.250187397003174, -4.541149139404297, 3.0276966094970703, 1.8807997703552246, -7.640296936035156, -1.5138006210327148, -1.238958477973938, -4.605495452880859, 0.43441176414489746, -1.534827709197998, 0.04081010818481445, -3.568972110748291, -0.1728067398071289, -5.3627824783325195, -6.166390895843506, -0.943317174911499, -0.1539616584777832, -6.37519645690918, -3.6303114891052246, -5.8623456954956055, 0.5969328880310059, -4.372021675109863, -2.4383761882781982, 1.9076073169708252, -4.057703971862793, -2.6225929260253906, 2.1912856101989746, -4.473207950592041, 2.4334535598754883, -5.624739646911621, 1.5343925952911377, -1.423424243927002, -5.655553340911865, -1.8900011777877808, -4.504228591918945, -1.079085350036621, -5.738067626953125, -6.780773162841797, 0.554340124130249, -0.5461702346801758, -3.0394794940948486, -4.852866172790527, -5.206653118133545, -2.5291218757629395, -2.1280245780944824, -2.789119243621826, -12.49531364440918, -5.273226261138916, -0.4628455638885498, -6.800502300262451, -4.497910022735596, -2.83756422996521, 1.724167823791504, -0.3498638868331909, -4.993861198425293, -3.482961654663086, -4.8140153884887695, -14.806779861450195, 0.04227709770202637, -5.520849704742432, 1.9040584564208984, -2.7829461097717285, -0.9458732604980469, -3.318730354309082, 1.1382741928100586, -2.6252219676971436, -0.5341333150863647, -8.229966163635254, -5.036592483520508, -0.008597135543823242, -3.3572843074798584, -2.217341184616089, 0.5480759143829346, -5.086294651031494, -4.305328845977783, 0.4205336570739746, 1.6930906772613525, -3.0908961296081543, -1.3797917366027832, -6.970277309417725, -7.133899211883545, -4.058476448059082, 0.7114009857177734, -6.170482158660889, 1.6002905368804932, -0.7468972206115723, -4.7336626052856445, -1.296583890914917, -8.044229507446289, -3.4620819091796875, 1.0683650970458984, -3.9726338386535645, -2.497986316680908, -2.938187599182129, 0.36894989013671875, -5.284377098083496, -3.263763427734375, -3.92056941986084, -7.335552215576172, -1.2068289518356323, 0.8330121040344238, -3.1989669799804688, -6.279493808746338, -7.2327775955200195, 7.888279914855957, 0.9035687446594238, -2.3451311588287354, 0.5196218490600586, -2.5854406356811523, -2.4225823879241943, -3.642369508743286, 0.11019372940063477, -3.7865712642669678, -4.666744232177734, -2.493715763092041, -1.661378026008606, -5.738801956176758, -0.3844788074493408, -1.5374798774719238, 0.5509471893310547, 2.366590976715088, -1.2105053663253784, -2.585251808166504, -0.7943689823150635, -0.10157346725463867, 0.20141386985778809, 1.9008679389953613, 1.2577903270721436, 0.10564756393432617, -1.6326305866241455, -4.828049182891846, -6.704431056976318, -5.655118465423584, -2.49495005607605, 0.4489729404449463, 1.1621589660644531, -6.695587635040283, -3.4529166221618652, -0.7324333190917969, -1.7846159934997559, 1.5216238498687744, -4.679849147796631, 0.5114805698394775, -4.2575764656066895, -1.6132334470748901, 1.096024513244629, -4.498929023742676, -2.5949454307556152, -1.714423418045044, 1.3436651229858398, -6.06617546081543, -0.5906988382339478, 0.28885340690612793, -0.16239678859710693, -2.8611583709716797, -1.7126109600067139, -3.4260635375976562, 2.1132688522338867, -0.26464295387268066, 0.6352314949035645, -1.2660815715789795, 3.1046109199523926, 0.33310890197753906, -0.6829713582992554, -2.5291695594787598, -5.780477046966553, -2.784776210784912, -3.2107667922973633, 1.7708029747009277, 0.7618706226348877, -5.171609401702881, -2.745054244995117, -4.193726539611816, 3.3056349754333496, 1.4115715026855469, -1.065400242805481, -4.71306037902832, 0.5522475242614746, -1.0101125240325928, 3.7868547439575195, -6.536385536193848, -1.0169095993041992, 0.16791796684265137, -2.9745707511901855, 0.42003941535949707, 4.002553462982178, -2.547647476196289, 0.16979742050170898, -4.005605697631836, 4.28725528717041, 2.8648757934570312, -2.1119425296783447, -6.682560920715332, -4.445876121520996, -0.5743074417114258, 0.08568906784057617, -4.031782150268555, 3.635272979736328, 2.6962637901306152, -6.332037448883057, 2.5410919189453125, 0.2388918399810791, -1.5730067491531372, -3.184049129486084, -1.7472139596939087, -4.330545902252197, 0.24282264709472656, -3.440570592880249, 6.101622581481934, -2.7918014526367188, -0.6135134696960449, -3.677877426147461, 2.2809371948242188, -2.3144381046295166, 0.3174097537994385, 0.15764236450195312, -4.507471084594727, -5.1991987228393555, 3.436709403991699, -3.9157326221466064, -3.877592086791992, -4.544137001037598, -2.9083786010742188, -2.9346437454223633, -4.074938774108887, -0.27044177055358887, -3.388319492340088, 1.6720194816589355, 1.4595551490783691, -6.9785051345825195, 0.8911280632019043, -2.153855800628662, -2.1669795513153076, 0.9425504207611084, -4.921542167663574, 1.7151033878326416, -2.112783908843994, -1.0587012767791748, -6.18942928314209, -0.3617579936981201, -0.21361851692199707, 1.7267155647277832, 0.1600484848022461, 0.3293571472167969, -7.309061050415039, -3.2790610790252686, -3.5040295124053955, 0.5175518989562988, -0.22220289707183838, -1.6510961055755615, -0.804517388343811, -3.7413883209228516, 0.866377592086792, 1.7895498275756836, -4.033052921295166, 2.201094150543213, -4.128842830657959, -0.6159716844558716, -1.3618545532226562, -3.979400873184204, -2.0260508060455322, 1.6447906494140625, 1.0876052379608154, -2.9799013137817383, -2.726044178009033, -0.41371917724609375, -1.8127809762954712, -4.184438705444336, -2.1436872482299805, -4.684057235717773, -4.247150897979736, -3.705512523651123, 4.594032287597656, -2.3305165767669678, -1.875496506690979, -1.420346975326538, -5.272990703582764, 1.236304759979248, -2.2537124156951904, 0.23236608505249023, 0.5845084190368652, -8.195040702819824, -1.7841308116912842, -6.858825206756592, -6.732230186462402, 3.200685977935791, -6.102280616760254, 2.022808074951172, -4.278031349182129, -2.563127040863037, 0.9632823467254639, -1.957383632659912, -0.8660686016082764, -1.6381343603134155, 0.3532102108001709, -2.3676412105560303, -4.496484756469727, -2.4166107177734375, -5.384296417236328, 3.7369794845581055, -6.514734745025635, -2.11116361618042, -1.9377533197402954, -0.474334716796875, -0.14529681205749512, -3.3903660774230957, -8.014448165893555, 0.8499114513397217, 0.6048884391784668, -0.8048561811447144, -4.928542613983154, 0.7296483516693115, -7.447506904602051, -0.765863299369812, -4.868290424346924, -4.781986713409424, -4.352819919586182, 0.7304816246032715, -5.936997413635254, -1.5643243789672852, -3.0797672271728516, -3.0637001991271973, -5.440073490142822, -5.608681678771973, -7.120489597320557, -0.4755406379699707, -2.6330528259277344, -0.7558960914611816, 0.285123348236084, 0.7509050369262695, 0.7658491134643555, -3.987809658050537, 2.4043831825256348, -4.917977333068848, -4.669276237487793, 1.413703441619873, 0.4408257007598877, -2.085352897644043, -1.065889596939087, -1.3568873405456543, 1.3182010650634766, -4.381850719451904, -0.06525731086730957, 0.030443668365478516, -2.470071792602539, -2.8959197998046875, 2.6657166481018066, -4.680811882019043, -4.3090667724609375, -3.620518684387207, -0.9455143213272095, -5.010262489318848, -3.241582155227661, -5.315423965454102, -2.6459596157073975, -2.0128567218780518, -0.22624123096466064, -7.745099067687988, -4.63096284866333, -4.882172584533691, -0.9860864877700806, -2.1013340950012207, -1.5260543823242188, -0.8364136219024658, 0.9446282386779785, -4.031521797180176, -6.615767478942871, -4.32320499420166, 0.7914648056030273, -1.586629867553711, 1.2856769561767578, -7.123277187347412, -2.736102342605591, -6.759913444519043, 6.294822692871094, 2.6232190132141113, -6.968750953674316, 4.111069202423096, 3.9365639686584473, -4.817451000213623, -5.682094097137451, 0.7420456409454346, 3.148733139038086, -0.7981959581375122, 0.33860182762145996, -3.302626848220825, -5.531224250793457, -5.357283592224121, -2.208723545074463, -6.272184371948242, 0.2366499900817871, -4.136783599853516, -1.5110740661621094, -3.560070753097534, 2.3225173950195312, -2.6696763038635254, -2.8371975421905518, 0.23755693435668945, -3.4852957725524902, -6.8862457275390625, -4.332478046417236, -1.9502654075622559, -3.140341281890869, -4.674994468688965, -3.45328426361084, -1.5507476329803467, 1.2199277877807617, -4.079602241516113, -1.3748323917388916, 2.546320915222168, -1.7621030807495117, 0.6291062831878662, -0.7142009735107422, -3.919841766357422, -4.0089111328125, -3.5744004249572754, -1.084378719329834, -2.022984743118286, -2.3735713958740234, -5.091228485107422, 0.5600965023040771, -1.5044171810150146, -1.1120798587799072, 0.9475877285003662, 0.08786129951477051, 4.887200355529785, 2.409132957458496, -9.16001033782959, 5.520326137542725, -0.7076535224914551, -0.505893349647522, 1.5466232299804688, -8.842421531677246, -0.15114164352416992, 0.5860409736633301, -3.983452320098877, 0.3371615409851074, -5.520491123199463, -0.8879140615463257, -6.935404300689697, 1.4583473205566406, -4.818973541259766, -0.17194795608520508, 0.18224024772644043, -10.047245025634766, -3.518855571746826, 0.5352890491485596, -5.432507514953613, -6.895837783813477, -4.429255485534668, -11.97092342376709, -0.881629467010498, -3.5732345581054688, -4.2713823318481445, -0.6402468681335449, 3.8612213134765625, -4.324383735656738, -0.9415717124938965, 0.6961417198181152, -0.14526605606079102, -6.774072647094727, -0.18485379219055176, -3.9766619205474854, -4.608301162719727, -8.224485397338867, -3.8525195121765137, -1.5684980154037476, -1.9682848453521729, -2.300823211669922, 1.2971453666687012, 2.5796971321105957, -2.2642881870269775, -4.433438301086426, 1.8427653312683105, -4.102147102355957, -1.16069495677948, -2.0589399337768555, -1.7616299390792847, -0.4550175666809082, -6.1925482749938965, -7.46803092956543, -3.7114310264587402, 2.5673303604125977, -2.354098320007324, -1.9577293395996094, 1.7178387641906738, 1.048719882965088, -3.1252758502960205, 1.7526979446411133, 0.8099372386932373, -0.7761600017547607, 0.14338064193725586, -2.908845901489258, -5.798233985900879, -4.367492198944092, -0.9010769128799438, 0.3945271968841553, -3.7037100791931152, -1.1688052415847778, -5.740251541137695, -2.4819374084472656, -0.5423891544342041, -0.672844648361206, -5.647185325622559, 1.439849853515625, -2.9967427253723145, -3.8229470252990723, -3.8587799072265625, -1.571628451347351, 2.749063014984131, -2.820936441421509, -0.8277138471603394, 0.6714391708374023, -0.4725162982940674, -2.300915479660034, 0.533613920211792, 0.12170553207397461, -2.49945330619812, -2.287642002105713, 1.829704761505127, -2.2405154705047607, -6.540977478027344, -6.157158851623535, 0.47109246253967285, -1.9431136846542358, -3.2252776622772217, -4.529458999633789, 0.43820858001708984, -7.208089351654053, -2.1912777423858643, -1.4091970920562744, 1.3759808540344238, -2.0888218879699707, -0.38773274421691895, -2.7006516456604004, -4.242877006530762, -3.1179230213165283, -0.6988544464111328, 0.09271550178527832, -2.4498486518859863, -0.6387304067611694, 0.15059566497802734, -6.860811233520508, 0.1462259292602539, -1.4665477275848389, 0.26508188247680664, -0.0006430149078369141, -7.8560967445373535, -0.25448155403137207, 1.5738248825073242, -1.0983715057373047, 1.9627327919006348, 0.9418716430664062, -1.4702091217041016, -4.887582778930664, 0.5640370845794678, -3.961064100265503, -1.9984681606292725, -0.42624616622924805, -1.400009036064148, 2.8364639282226562, -0.8925405740737915, -3.0192067623138428, 2.460366725921631, -2.2609004974365234, -0.6871166229248047, -1.8225131034851074, -4.584489822387695, -5.049905300140381, 1.8157544136047363, -2.9772086143493652, -0.8251779079437256, -1.2439422607421875, -4.240204334259033, 1.2572221755981445, -4.457923889160156, -3.903122663497925, -4.471331596374512, -2.9671812057495117, -2.7987427711486816, -7.0308332443237305, -0.08271074295043945, 2.4938507080078125, -4.560972690582275, -0.8164151906967163, -1.1022343635559082, 1.1764121055603027, -1.8103923797607422, -4.684505939483643, -1.462550401687622, -5.754021644592285, -3.606163501739502, -3.2435219287872314, -2.69161319732666, -3.784534215927124, -0.81268310546875, -0.733674168586731, 0.5681660175323486, 1.4194939136505127, 2.1700973510742188, -3.015270709991455, -6.148983001708984, -0.24460256099700928, -3.9008021354675293, 3.002480983734131, 0.13991260528564453, -7.468938827514648, -0.5146965980529785, -9.466361999511719, -0.49388206005096436, -3.472780227661133, -1.3599165678024292, -3.4062280654907227, 2.6755757331848145, -11.553404808044434, 3.532270908355713, 3.001803398132324, -5.610494136810303, -0.8940929174423218, 0.1510763168334961, -0.8680247068405151, -0.43651819229125977, -0.04641008377075195, -2.2653584480285645, 0.4400012493133545, -5.106534957885742, -0.08000898361206055, -2.3071506023406982, 1.8733506202697754, 0.08737635612487793, 2.475327491760254, -0.2940216064453125, -3.473031997680664, -1.3516819477081299, -3.0900354385375977, -5.646936416625977, -4.577263832092285, -3.3810338973999023, -3.611790180206299, -4.556865215301514, 1.1688525676727295, -1.1905385255813599, 0.920647382736206, 1.090597152709961, -5.495754718780518, 1.2222309112548828, -5.8712544441223145, -4.604598045349121, -7.923702716827393, -4.3691511154174805, 2.8652806282043457, -1.2447868585586548, -1.2678800821304321, -0.2723952531814575, -0.5657293796539307, -6.072985649108887, -6.582900524139404, -3.887094020843506, 0.9693045616149902, 0.5306081771850586, -3.551711082458496, -5.112143039703369, -0.9256236553192139, 1.658038854598999, -0.23970413208007812, 0.0070269107818603516, -5.915774345397949, -0.034230828285217285, -0.3854687213897705, -3.4895591735839844, -3.225691318511963, -1.1021394729614258, -10.933640480041504, -8.6363525390625, -0.9801150560379028, -3.574202537536621, -0.9466522932052612, 3.5194759368896484, -1.389517903327942, -1.6907962560653687, -3.838836431503296, -2.0325276851654053, -3.076284408569336, 0.9490451812744141, -5.757495403289795, -5.453383922576904, -0.3199089765548706, 1.088761329650879, 1.8507356643676758, -1.6603790521621704, -2.7872183322906494, -2.3874549865722656, -5.569485664367676, -1.9383580684661865, -1.6946210861206055, -1.0012645721435547, -3.9223620891571045, -0.5540342330932617, 2.2839484214782715, -6.6074981689453125, 2.925563335418701, -4.48714542388916, -1.9125398397445679, -1.2703559398651123, -3.6469619274139404, -3.1972646713256836, -1.4341282844543457, -2.919768810272217, -6.728048324584961, -6.167616844177246, -1.1596347093582153, -4.134337902069092, -3.3235645294189453, 0.8045806884765625, -0.2655050754547119, 0.48058366775512695, 3.359905242919922, -4.361504554748535, -5.052958965301514, -5.633938789367676, 0.8822035789489746, 1.9621081352233887, -0.8028991222381592, -3.2334530353546143, -1.5784116983413696, -3.9856085777282715, -0.3174731731414795, 2.432013511657715, -8.127305030822754, -3.09600830078125, -5.286759376525879, 2.1787476539611816, -4.135578155517578, 0.7933883666992188, -1.8048455715179443, 1.9098758697509766, -3.0185940265655518, 1.1728465557098389, -6.725961685180664, -3.7037100791931152, -1.294495940208435, -6.3035197257995605, -2.0644888877868652, -0.26665663719177246, -0.9775905609130859, -4.993556976318359, -2.2640674114227295, -1.0082472562789917, 1.3393940925598145, 0.9827148914337158, -2.031755208969116, -0.815619707107544, -3.215212821960449, -5.96842098236084, -6.660296440124512, -1.6384904384613037, -3.0246782302856445, -2.523348331451416, -3.267296075820923, -4.7531514167785645, -3.5172555446624756, -0.6044732332229614, -3.949164628982544, 6.106572151184082, -1.4772779941558838, -0.6864643096923828, 0.7462310791015625, -2.167912721633911, -4.97844934463501, -4.516222953796387, -0.38512110710144043, 3.597228527069092, -3.346752405166626, -3.140488386154175, -0.10398316383361816, 1.5547370910644531, -1.5410404205322266, -5.272088050842285, 2.383330821990967, 0.04516291618347168, -0.8204998970031738, -7.821605205535889, 6.380395889282227, -2.171647310256958, -5.999723434448242, -3.372041702270508, -2.597740650177002, 3.3793210983276367, -5.825237274169922, -4.177237510681152, 0.37073445320129395, 1.536374568939209, -7.455739974975586, -6.237017631530762, -1.4196828603744507, -0.18338322639465332, 2.2412304878234863, 1.501124620437622, -3.6880242824554443, -2.9678282737731934, -0.45047950744628906, -0.8153824806213379, 0.7571842670440674, -2.150517225265503, -2.3169243335723877, -9.424571990966797, -6.864770889282227, 4.163417816162109, 1.028174638748169, -0.11499929428100586, -1.5823006629943848, -6.738310813903809, -5.000844478607178, -3.8517374992370605, -8.541889190673828, -2.182605266571045, -0.7604973316192627, -2.90024471282959, -5.709062576293945, 0.8719851970672607, -0.8050627708435059, -5.869540214538574, -5.94415807723999, -3.818195343017578, -0.2822897434234619, -4.052075386047363, 2.048929214477539, -2.559300422668457, -6.183746337890625, 5.656181812286377, -3.8121485710144043, -4.171018600463867, -2.5211524963378906, 1.0278406143188477, -0.179040789604187, -7.468146324157715, -1.1394976377487183, -1.4015969038009644, 0.029284238815307617, -4.689546585083008, 1.1338276863098145, -0.41413676738739014, -3.416048049926758, -3.721881866455078, -4.009806156158447, -1.45424222946167, 0.5766110420227051, -0.4835629463195801, -2.5021729469299316, 4.043787956237793, 2.2524681091308594, -0.9446147680282593, 1.5922412872314453, -1.034926414489746, 0.260822057723999, -1.079691767692566, -3.231989860534668, -1.7092996835708618, -3.403890371322632, -2.7788987159729004, 2.0023698806762695, -2.3550872802734375, 0.4894134998321533, -4.342473030090332, 0.5358843803405762, -3.361405372619629, -2.2082083225250244, 0.24430322647094727, -1.1836918592453003, 1.9685602188110352, -5.4230804443359375, -6.316094398498535, -5.332997798919678, -4.982739448547363, 1.9162874221801758, -7.0309553146362305, -0.1445826292037964, -4.767754077911377, -3.104398727416992, 0.0006742477416992188, 0.5464115142822266, 3.2983179092407227, 3.958324432373047, -8.225775718688965, 1.731471061706543, -5.089174270629883, -2.2323896884918213, 5.000175476074219, -1.6561118364334106, -1.4220123291015625, 5.542622089385986, -1.4547799825668335, 1.7415077686309814, -1.4319086074829102, -0.6132136583328247, -2.8316097259521484, -1.0233488082885742, -5.119380950927734, 3.607860565185547, -1.5946991443634033, -7.371298789978027, 4.521812438964844, -1.6743824481964111, -2.2468924522399902, -4.869458198547363, -3.361496686935425, -1.246415376663208, 0.44617700576782227, -1.403196096420288, -0.37911486625671387, -3.5174973011016846, 3.8931713104248047, 0.8385004997253418, -6.009518623352051, -3.284524917602539, 1.7051997184753418, 2.716437339782715, -5.734922409057617, -2.794236183166504, -1.5717965364456177, 1.3819973468780518, -3.9493942260742188, -3.2614567279815674, -1.5245709419250488, 0.13836216926574707, 2.372162342071533, 2.97348690032959, -5.900019645690918, 4.464290618896484, 0.5261998176574707, -3.9970149993896484, 1.373960018157959, 2.8761110305786133, 0.18688249588012695, -3.17073392868042, -1.408455491065979, -2.005518913269043, -3.297236442565918, -4.539919853210449, -7.248144626617432, -1.7055890560150146, -3.234781265258789, -1.8632030487060547, -2.8390469551086426, -0.03806889057159424, -5.471427917480469, -6.643580913543701, -4.855827331542969, -2.9102420806884766, 0.11262083053588867, -8.151490211486816, -0.9202609062194824, -1.477542519569397, -7.258599758148193, -5.032865524291992, 1.2826433181762695, -1.4047561883926392, -3.8239693641662598, -4.705737113952637, 1.1499712467193604, 1.9098572731018066, -3.309514045715332, -8.11720085144043, 1.6592204570770264, -2.886697769165039, -4.931987762451172, -2.028040647506714, -7.027397155761719, -3.0719926357269287, -5.454830169677734, -5.040870189666748, -6.257437229156494, -4.1887969970703125, -3.8280444145202637, -4.895059585571289, -9.428515434265137, 0.7456960678100586, -1.3343850374221802, -5.381979942321777, -9.478034019470215, -1.4354501962661743, -6.731939315795898, -5.44696569442749, 0.12445807456970215, -3.4308371543884277, -0.25259995460510254, -0.03963673114776611, 1.5007033348083496, -7.300008296966553, 1.0743458271026611, -1.8859949111938477, -3.9616641998291016, -3.878638744354248, -3.8873863220214844, -1.9502191543579102, 0.5536055564880371, -0.44918251037597656, -2.493067741394043, -3.456103801727295, -4.450496673583984, -0.2789813280105591, -2.863497257232666, 1.2893145084381104, 2.650974750518799, -2.7941715717315674, -1.2183030843734741, -4.622743606567383, -0.437833309173584, -4.967095851898193, -0.31753575801849365, 0.43012475967407227, -5.397312164306641, -0.30877041816711426, -3.3094282150268555, -2.103574514389038, 0.06667470932006836, -1.558194875717163, -0.4489908218383789, 0.9929304122924805, -5.470190048217773, -2.7546706199645996, -2.1297953128814697, -2.860431432723999, -0.44001829624176025, -1.3914560079574585, -6.249124050140381, -1.1004674434661865, -2.281571865081787, -1.8003342151641846, -5.321749687194824, -6.021535873413086, 3.9604172706604004, -2.4718141555786133, -2.5122838020324707, 1.835820198059082, -4.190639495849609, 5.412503242492676, -1.3993339538574219, 4.5684814453125, -1.4152897596359253, -3.5846199989318848, -2.0848772525787354, -3.8466973304748535, -1.6015894412994385, -6.574969291687012, -1.6349543333053589, 5.0596184730529785, -3.1717007160186768, -4.239460468292236, 3.102294921875, 1.7442855834960938, -2.7293503284454346, 1.4501655101776123, -3.6865975856781006, -5.412283897399902, -4.297900199890137, -0.7169873714447021, 0.4482243061065674, -4.2047905921936035, 1.132009506225586, -7.1259846687316895, -0.3487112522125244, 1.177870273590088, 0.801933765411377, -0.9989421367645264, -5.390567779541016, -1.2216237783432007, -1.6661725044250488, -2.752582550048828, -0.4746429920196533, -4.680490493774414, -6.398792266845703, 1.0013461112976074, -6.756701469421387, -2.326677083969116, -0.9575803279876709, -3.3323066234588623, -1.6664115190505981, -3.3305087089538574, -1.6166479587554932, -3.594456911087036, -5.6693925857543945, 1.5493154525756836, -5.517661094665527, -6.643355846405029, -0.9800970554351807, -4.154905319213867, -6.251400470733643, 2.0973644256591797, -0.3514082431793213, 0.7240734100341797, -1.249445915222168, -1.8002017736434937, -1.6038291454315186, 2.5837159156799316, -1.9531015157699585, -10.2554349899292, -0.2037900686264038, 0.6663239002227783, -0.8440625667572021, -0.6225937604904175, -4.065400123596191, 1.54301118850708, -3.405733346939087, -3.3940861225128174, -1.1688494682312012, -6.596039772033691, -5.711414813995361, -6.960378646850586, -5.500711917877197, 0.9969475269317627, 0.688800573348999, -2.490828275680542, -0.9193310737609863, 1.5638041496276855, 0.3317742347717285, -4.075443267822266, -0.9758133888244629, -0.19421136379241943, -6.368526458740234, -1.541705846786499, -0.8457168340682983, 0.8816318511962891, -7.13439416885376, -7.970023155212402, -5.809395790100098, -0.013683080673217773, 7.440364837646484, -1.6302168369293213, -5.100472450256348, -0.7774957418441772, 2.627577781677246, -1.0526057481765747, 2.48140811920166, 3.289268970489502, -0.00467371940612793, 0.37877964973449707, -1.347278118133545, -2.3803582191467285, -4.256173610687256, -2.949190139770508, 1.5201289653778076, -0.9981366395950317, -7.560460090637207, 1.3910102844238281, -1.4087870121002197, 0.47202420234680176, 0.05931377410888672, -3.253436803817749, -1.4220364093780518, -2.445706367492676, -2.2685742378234863, -4.98162841796875, 0.5868015289306641, -5.236751556396484, -4.426149368286133, 0.763251543045044, 5.677277565002441, -1.4577386379241943, -5.119312286376953, 0.17345643043518066, 0.9684135913848877, 0.42527055740356445, 0.36700892448425293, -3.0255019664764404, -5.514479160308838, -1.147995948791504, -1.269472599029541, 2.4406728744506836, -1.3792033195495605, 1.3721094131469727, -3.9888827800750732, -2.7829809188842773, -1.1246588230133057, -7.935714244842529, 0.37442564964294434, 0.03982686996459961, -1.4146552085876465, -8.757508277893066, -1.932755947113037, -5.645590305328369, 4.7114973068237305, 1.9500584602355957, 0.996375560760498, -5.265873908996582, -3.56965970993042, 0.379225492477417, -0.4303715229034424, -0.4360775947570801, -3.469688653945923, -2.390705108642578, 2.0611867904663086, -8.394886016845703, -4.472940444946289, -3.3397891521453857, -3.277747392654419, -2.911534547805786, -4.703302383422852, -2.0606887340545654, -2.4081757068634033, -6.082242488861084, -1.784148931503296, 2.9438724517822266, 1.0405168533325195, -1.3479788303375244, 0.9903101921081543, -4.081293106079102, 3.333812713623047, 0.22545099258422852, -2.1800668239593506, -1.6751785278320312, -6.763396263122559, -3.3491382598876953, 1.004899501800537, -4.703745365142822, -3.939862012863159, -1.935861349105835, -4.323554992675781, -6.982293128967285, -7.878713130950928, -0.8792366981506348, -0.21915268898010254, -2.1471235752105713, 0.16411209106445312, -1.1455527544021606, -4.128960609436035, -2.392131805419922, -5.780505180358887, 0.07403969764709473, -6.118596076965332, -4.852179050445557, -7.614274501800537, -5.947166442871094, -4.290291786193848, -6.567733287811279, -4.01334285736084, -2.0728185176849365, -2.7811427116394043, -2.7172188758850098, -0.43784821033477783, 0.3933556079864502, -6.871315956115723, -2.566197156906128, -3.987974166870117, -1.0366871356964111, -1.1413336992263794, -1.126680612564087, -0.9848418235778809, 3.177720546722412, -2.0208241939544678, -4.258111000061035, 2.9417238235473633, -6.000311851501465, 2.6380615234375, -5.661994934082031, -4.649265289306641, -3.353804588317871, -2.796760082244873, 0.2568683624267578, -4.606960296630859], [-4.565878868103027, -2.645754337310791, 1.398637294769287, -3.3455328941345215, -0.022667407989501953, -4.247040748596191, -1.6486964225769043, -8.99770736694336, -4.426128387451172, -0.6926333904266357, 0.13053131103515625, 2.269345283508301, -1.6891461610794067, -3.0562844276428223, -3.309358596801758, 0.06483983993530273, -2.6937668323516846, -5.190043926239014, 0.3535623550415039, -3.450596570968628, -2.705406904220581, -1.7743151187896729, -3.675574779510498, 1.8681695461273193, -2.2143256664276123, -2.4490725994110107, -4.127328872680664, -0.5366642475128174, -9.077667236328125, -4.630186080932617, -6.487730979919434, -3.5968544483184814, 0.0919651985168457, 3.940342426300049, -4.4550700187683105, 0.30640482902526855, 2.5811920166015625, -1.6943929195404053, -1.6735641956329346, 0.13178801536560059, -4.000849723815918, -3.5997400283813477, 2.6513004302978516, -1.1365087032318115, -7.461680889129639, 1.413527011871338, -2.401956558227539, -5.918877601623535, -0.8596096038818359, 3.0727319717407227, -0.5524771213531494, -0.0369645357131958, -1.6872206926345825, -0.5437145233154297, 1.160975456237793, -2.5607645511627197, -3.07915997505188, -5.003782749176025, -2.420989513397217, 2.0851449966430664, -0.9467906951904297, -2.795417070388794, -4.1487016677856445, -0.44389283657073975, -1.658020257949829, -0.9989011287689209, -2.9578628540039062, -0.17205190658569336, -4.494229316711426, -3.8407018184661865, 2.0608749389648438, 2.870058536529541, -2.6074013710021973, 2.7232723236083984, -5.041584491729736, -5.894309043884277, 2.5735836029052734, -2.6424367427825928, 0.42317676544189453, -3.0543243885040283, 1.4256606101989746, 0.395885705947876, -5.648993492126465, -3.0310044288635254, -4.163556098937988, -3.847384452819824, -1.3421953916549683, -2.0056984424591064, -0.8050131797790527, -3.0548465251922607, -4.271657943725586, -1.7116615772247314, 1.6787471771240234, 2.2996950149536133, 0.13187885284423828, -9.400646209716797, -0.7113227844238281, -1.8946932554244995, -1.7980890274047852, -2.6740288734436035, -2.1677603721618652, 3.305197238922119, -0.07013940811157227, 1.4860515594482422, 0.42116451263427734, -1.0276036262512207, 1.6654739379882812, 1.8839712142944336, -9.410554885864258, -0.8222744464874268, -3.9110586643218994, 0.05386662483215332, 1.4936647415161133, -2.3228702545166016, -6.3137969970703125, -5.19985818862915, 0.5997426509857178, 0.23243093490600586, 0.575218677520752, 0.0998692512512207, -5.360212326049805, -1.4633474349975586, 0.6097378730773926, -7.210047721862793, 1.1749420166015625, -0.9103612899780273, 1.6782505512237549, 1.6425085067749023, -1.7474806308746338, 3.852485179901123, 2.899360179901123, -1.119103193283081, -3.7236578464508057, -7.330420970916748, 2.787705421447754, -0.2585386037826538, -9.362997055053711, -2.2327070236206055, -4.545282363891602, -5.369521617889404, -1.895027756690979, -7.652017116546631, -2.099029541015625, 3.4371542930603027, 4.0034990310668945, 3.971247673034668, -8.324169158935547, -7.102996826171875, -2.2237396240234375, -0.31006503105163574, -1.8598746061325073, -3.937549591064453, -4.319867134094238, -0.9292668104171753, -1.023129940032959, -3.4507875442504883, -0.08661651611328125, -3.923961639404297, 0.7870259284973145, -2.1365599632263184, -2.8182148933410645, -4.639713287353516, -1.3815804719924927, -2.9833407402038574, -0.5511431694030762, 2.661550521850586, -3.9458892345428467, 2.6227102279663086, -3.3159842491149902, -3.5969865322113037, -2.370745897293091, -0.19764995574951172, -2.4139654636383057, -0.6917455196380615, -1.0171453952789307, 0.47999000549316406, -0.4407771825790405, -0.039176106452941895, -6.484997749328613, -4.986591339111328, 1.8689932823181152, -1.3795020580291748, -5.165411949157715, -0.940893292427063, -5.6558074951171875, -5.34188175201416, -7.26665735244751, -1.3263417482376099, -2.1358530521392822, -1.6023321151733398, 0.4896249771118164, -0.557624101638794, 1.244497299194336, -0.1499030590057373, -7.984414100646973, -0.5973234176635742, -5.06138801574707, -3.5383708477020264, -2.2370123863220215, -12.092386245727539, 0.4090261459350586, -1.4350560903549194, -5.9076924324035645, -5.026811122894287, 1.798354148864746, -5.47844123840332, 2.2605247497558594, 4.331572532653809, -3.6308867931365967, -5.395840644836426, -0.16648459434509277, -1.7641031742095947, -5.319876670837402, -3.3744912147521973, 0.9515857696533203, -0.10738182067871094, -2.566535711288452, -4.781422138214111, -6.117870807647705, 1.0555715560913086, -4.516754150390625, 1.4231863021850586, -4.735230445861816, 0.000194549560546875, -1.9701669216156006, -0.44373369216918945, 0.4814624786376953, -3.695067882537842, 1.5577383041381836, -4.857406139373779, -1.5558669567108154, 1.7239274978637695, -0.9464724063873291, -3.5725295543670654, -0.862808346748352, -0.7108561992645264, -0.8295912742614746, -1.3706881999969482, -6.168805122375488, -2.370629072189331, -1.7321619987487793, -6.143001556396484, -1.4289164543151855, -2.103508949279785, 2.77938175201416, -0.9986710548400879, 4.927207946777344, -3.7978086471557617, -5.425549030303955, -4.116219520568848, -3.053788423538208, 0.5600922107696533, 3.226132392883301, -6.288846492767334, -0.6035001277923584, 1.4391818046569824, -5.195847988128662, 4.586644172668457, 0.17633605003356934, 2.447755813598633, 0.905266284942627, 0.45041394233703613, -5.634390830993652, -1.1221610307693481, 1.884185791015625, -0.7180137634277344, 1.8196296691894531, -1.8904060125350952, -0.20966029167175293, -6.097609996795654, -3.9013843536376953, 0.24564123153686523, -4.944393157958984, -4.218811988830566, -3.484323024749756, -1.5774024724960327, -7.776486396789551, -3.7565598487854004, -5.122807025909424, -0.1551663875579834, -0.004731535911560059, -1.643595576286316, -0.3215944766998291, -5.53975772857666, -0.7949151992797852, -4.728912353515625, -3.6987791061401367, -1.0423206090927124, -2.2451634407043457, -3.7220706939697266, -3.850827693939209, 1.8788037300109863, -1.8473451137542725, 1.2809514999389648, -6.478582382202148, 1.6014623641967773, 3.5384578704833984, 7.723012924194336, 1.0740766525268555, -5.0851664543151855, -4.653629302978516, -5.396098613739014, 3.6410951614379883, -0.8320890665054321, 4.693967819213867, -6.840917587280273, -3.723034620285034, -6.519052505493164, -2.381040334701538, -1.5898574590682983, 1.9829750061035156, -5.7039995193481445, 0.003174304962158203, -8.236427307128906, -1.5730273723602295, 3.2207651138305664, -1.2217450141906738, -1.722779393196106, -4.729828834533691, -3.2286603450775146, -5.028254508972168, -4.340488910675049, -0.5490951538085938, -1.5159391164779663, 1.1191253662109375, -8.851649284362793, -2.2009618282318115, -2.133071184158325, -4.209497451782227, -2.6071739196777344, -1.9022674560546875, -2.8766133785247803, -5.148288726806641, 0.290285587310791, -1.354648470878601, -4.302191257476807, 2.7479801177978516, -3.756815195083618, -2.8152003288269043, 1.9931645393371582, -1.2098032236099243, -2.823831081390381, -0.20085978507995605, -1.1782824993133545, -1.677980899810791, -7.256211757659912, 0.738422155380249, 0.32567453384399414, -0.6846432685852051, 1.2750940322875977, -7.980297088623047, -0.3534773588180542, -4.4083170890808105, 0.18007254600524902, -1.6669087409973145, -0.5250309705734253, -1.670283317565918, 0.6253321170806885, 0.25092434883117676, -4.2847981452941895, -7.081864356994629, -3.510503053665161, -2.908745288848877, -4.732489585876465, -0.6098458766937256, 1.0827140808105469, 1.8012523651123047, -0.7645297050476074, -0.2587096691131592, -3.4665255546569824, 0.7485184669494629, 0.05773115158081055, -2.082717180252075, -5.923040866851807, -4.541364669799805, -7.138835906982422, -3.16835880279541, -6.949173927307129, -2.151670217514038, -3.500023603439331, -2.1884474754333496, -3.2452139854431152, -6.546403884887695, -0.12204813957214355, -3.553630828857422, -2.979126214981079, -1.2731523513793945, 0.9932804107666016, -0.13216495513916016, -1.0260953903198242, -2.1539306640625, -1.381230354309082, -3.7469828128814697, -2.706348419189453, -4.287362098693848, -4.250548362731934, -1.4861640930175781, -2.5608253479003906, -3.883762836456299, -0.393307089805603, -5.347441673278809, -2.5989990234375, -0.1465449333190918, -1.537705659866333, -0.3314230442047119, -2.4772884845733643, -0.9682247638702393, 1.025693416595459, 5.170072555541992, -4.105641841888428, -0.30603229999542236, -2.669163703918457, -3.5597238540649414, -0.6869205236434937, -7.034666538238525, -3.512385845184326, -0.08746004104614258, -1.7412326335906982, -3.054215908050537, -5.169370651245117, -1.6553623676300049, -5.652516841888428, -0.4700596332550049, -1.1884162425994873, 0.3757455348968506, 1.1288385391235352, -0.8338208198547363, 3.71091890335083, -2.3297111988067627, -0.1934816837310791, 2.9840731620788574, -3.9705374240875244, -1.9923619031906128, 1.08219575881958, 0.3179025650024414, -5.81374454498291, -3.7354323863983154, 2.2591419219970703, -5.275589466094971, -2.394770860671997, -2.578331232070923, -4.800093650817871, 5.026572227478027, -1.7190407514572144, -0.2853548526763916, -7.749435901641846, 0.13936853408813477, -4.554409027099609, 0.6848239898681641, 1.3611435890197754, -7.369471549987793, 0.20186257362365723, -7.874994277954102, -0.7935705184936523, -2.565239906311035, -4.291082382202148, -6.488943099975586, -5.042782306671143, -4.463800430297852, -5.0430145263671875, 2.613492488861084, -4.126595497131348, -2.9256350994110107, -3.350860118865967, -4.739103317260742, -6.6861572265625, 2.5249547958374023, -6.758342742919922, -5.635526657104492, -1.270046591758728, -2.0497395992279053, -2.529103994369507, -0.9623794555664062, -6.03878116607666, -1.2067656517028809, -3.4887752532958984, -0.31773972511291504, -2.02133846282959, -1.6328703165054321, -2.177793264389038, -5.947114944458008, -7.505634307861328, -1.3374724388122559, -1.5437568426132202, 0.03106093406677246, -5.402219772338867, 0.5560767650604248, -1.7483675479888916, -1.2726237773895264, -1.515720248222351, 1.2793707847595215, 0.6738812923431396, -1.1888777017593384, -3.3920645713806152, -3.212649345397949, 0.633263111114502, 0.12659025192260742, -2.119086980819702, -1.405646800994873, -6.817314147949219, 2.121479034423828, -2.3143177032470703, -6.602046489715576, -0.6611698865890503, -3.35979962348938, -0.64674973487854, -1.3009037971496582, -1.1887078285217285, -1.568881630897522, -4.1348185539245605, -1.8717541694641113, -3.8722400665283203, -3.4050045013427734, 1.4882841110229492, 3.0952954292297363, -7.961804389953613, 0.6331677436828613, 0.28668642044067383, -1.022236704826355, -0.28381824493408203, 2.76815128326416, -3.7176461219787598, -1.010228157043457, -1.77296781539917, 2.0788965225219727, -0.13169598579406738, -0.5036262273788452, -2.2963738441467285, -4.559535026550293, -5.423298358917236, -2.9424235820770264, -1.3188066482543945, 0.6716394424438477, -0.2598867416381836, 0.8025283813476562, -8.552196502685547, -2.0106585025787354, 0.5407438278198242, -4.484072208404541, -2.7778825759887695, 0.6227302551269531, 3.863084316253662, -2.1271612644195557, -2.458916664123535, 0.21434879302978516, -4.786923408508301, -0.6289732456207275, -3.302550792694092, -0.8486572504043579, -3.3950233459472656, -6.1239752769470215, -1.7788735628128052, -0.7719234228134155, -1.614574909210205, 0.8566775321960449, -6.204794883728027, -1.2120050191879272, 3.499512195587158, -6.676591873168945, -0.5853363275527954, -4.881845951080322, -3.8426358699798584, 0.17421269416809082, 0.2723381519317627, 2.4073996543884277, -1.1882848739624023, -2.531737804412842, -1.962469458580017, 1.912689208984375, -5.682443618774414, -1.5086047649383545, -2.920961380004883, 2.482365131378174, 2.180654525756836, 0.5700492858886719, -0.04203987121582031, -4.2847795486450195, -5.594421863555908, -4.751236438751221, -0.2740771770477295, -5.428804874420166, -2.4224705696105957, -0.4962177276611328, 1.8593261241912842, -4.29340934753418, 4.044763565063477, -2.787437915802002, -0.9229663610458374, -5.975071907043457, -9.020771026611328, 0.21917486190795898, -2.7163877487182617, 4.714983940124512, 4.80887508392334, -1.8843687772750854, -2.9401590824127197, -0.21570968627929688, 5.245780944824219, 1.602890968322754, -3.600640296936035, -5.238539695739746, 2.0622644424438477, -4.797393798828125, -7.940950393676758, -5.396831512451172, -4.134747505187988, 1.283012866973877, 0.10359907150268555, -4.3914594650268555, -0.22659385204315186, 0.6962769031524658, -3.2041773796081543, -0.9466320276260376, -6.814026832580566, -5.081511974334717, -4.225987911224365, -2.6225717067718506, 4.40666389465332, -6.500587463378906, 0.4963836669921875, 1.3542962074279785, -6.672215461730957, -2.023175001144409, -2.962974786758423, -4.318398952484131, -3.8445708751678467, -9.701163291931152, -8.958574295043945, -10.046646118164062, -1.3967217206954956, -3.2591960430145264, -2.325342893600464, -4.3638081550598145, 2.501889705657959, -6.875067234039307, -1.3415390253067017, -1.651033639907837, -1.4574819803237915, -6.665097236633301, -0.33051013946533203, 1.2761940956115723, -7.103150367736816, -3.1328041553497314, -4.157825469970703, 1.470353126525879, -4.259281158447266, -5.142834663391113, 0.17463111877441406, -0.32209479808807373, 0.9677200317382812, -0.8346247673034668, -4.845484256744385, -3.934818983078003, -3.1013455390930176, -1.2516733407974243, 0.09804415702819824, -1.6917483806610107, -0.6728483438491821, 1.0681548118591309, -5.598918914794922, -0.6476947069168091, -1.4464704990386963, -0.5114874839782715, -5.497490882873535, -0.9693872928619385, -0.45595312118530273, -0.23457098007202148, -5.168123245239258, 4.562067031860352, 0.6383233070373535, -2.8691749572753906, -0.37231576442718506, -6.097067832946777, -6.245291709899902, -8.737375259399414, 1.9041180610656738, -0.27000999450683594, 0.36440587043762207, -0.4801957607269287, -1.1238486766815186, -0.6207815408706665, 3.76662540435791, -1.8829009532928467, -1.0545564889907837, 0.9825901985168457, -0.8064835071563721, -4.720391273498535, -4.326499938964844, -8.997486114501953, 7.810205459594727, -6.061585426330566, -5.6182684898376465, -4.603699684143066, -9.452921867370605, -1.194921612739563, -4.0351386070251465, -4.2174224853515625, -5.2187395095825195, -5.225888252258301, -1.0077910423278809, -6.40329647064209, -3.524965524673462, -0.7452051639556885, -7.832212924957275, -6.443081378936768, -5.563584804534912, -1.0886614322662354, 3.7151737213134766, -0.41244804859161377, -1.350185751914978, -1.9092909097671509, -6.2526535987854, -5.439385414123535, -5.047122001647949, 2.858761787414551, -3.739429473876953, 0.7378098964691162, -1.9939273595809937, -4.5431694984436035, -1.6098237037658691, 1.0716967582702637, 2.5948963165283203, -0.09780240058898926, -4.538736343383789, -2.4965755939483643, -5.666857719421387, 0.18437910079956055, -4.36490535736084, -6.380829334259033, -2.247643232345581, 1.4423737525939941, 4.716922283172607, -2.3480770587921143, -0.06951725482940674, -3.3179099559783936, -3.5869698524475098, -1.2098726034164429, -0.8506135940551758, -3.2952799797058105, 0.3075227737426758, -0.49578535556793213, -0.4474678039550781, 1.630258560180664, -1.4636633396148682, -2.0838656425476074, -3.790802478790283, -5.486515045166016, -4.2836713790893555, -2.9008865356445312, -0.8165558576583862, -4.973008632659912, -0.04246628284454346, 0.27947187423706055, -1.3906807899475098, -3.6460728645324707, -3.933161735534668, 0.0777747631072998, -3.430507183074951, 1.4284725189208984, 5.146162986755371, -7.158461570739746, 0.4794349670410156, -7.3979597091674805, -5.246359348297119, -0.20273089408874512, -2.516741991043091, 2.239523410797119, 1.1561741828918457, -2.6385388374328613, 7.7326154708862305, 0.7464621067047119, -2.4549238681793213, -1.232138752937317, -3.288665533065796, -2.284010648727417, -5.833906650543213, 0.9937973022460938, -4.413091659545898, -7.282227516174316, -2.159984827041626, -0.8944664001464844, -3.3737192153930664, -6.81083869934082, -4.989747524261475, -1.4759637117385864, -2.4635043144226074, -4.452827453613281, -4.588025093078613, -0.5891546010971069, -2.921903371810913, -2.297713041305542, -1.5405616760253906, 2.554938316345215, 1.0927848815917969, -2.505115032196045, -3.5769762992858887, -4.264304161071777, -2.6587958335876465, 0.9483323097229004, -7.667838096618652, -3.5866799354553223, -1.3276050090789795, -3.1912479400634766, 2.2675857543945312, -5.914648056030273, -9.410609245300293, -4.997655868530273, -3.786787509918213, 0.96063232421875, -2.006518602371216, -2.005079507827759, -2.5808849334716797, -1.7598280906677246, -2.025247573852539, -2.8405497074127197, 0.7269444465637207, -1.3145763874053955, -4.64939546585083, -2.7096264362335205, -4.877950668334961, -5.295358180999756, -3.9375367164611816, -6.013730049133301, -7.29689884185791, 2.007772922515869, -6.097968578338623, 0.47117018699645996, -1.6249110698699951, -0.7424143552780151, -1.0513641834259033, -0.023823261260986328, -2.413206100463867, -1.0743438005447388, 1.1672289371490479, -6.631799221038818, -2.5371289253234863, 2.6589622497558594, -3.0128633975982666, -0.5197606086730957, -4.157010555267334, -1.1701722145080566, -1.7169108390808105, -3.030020236968994, -2.76200532913208, -3.4622092247009277, -0.6897618770599365, -0.9941105842590332, -0.5793896913528442, -2.868122100830078, 0.1001596450805664, -2.819694995880127, -4.063654899597168, 3.3196067810058594, -0.3368898630142212, -5.946948051452637, -1.7362182140350342, -0.0490267276763916, -4.236480712890625, 1.696502447128296, 0.9687373638153076, -3.0616278648376465, 0.5534133911132812, 0.612929105758667, 2.263784408569336, -2.9906630516052246, -0.4600830078125, -3.142509937286377, -2.1474876403808594, -1.1204488277435303, 0.42281270027160645, -2.2385456562042236, 3.473618507385254, -4.962375640869141, 4.727469444274902, -6.69978666305542, -0.34490132331848145, -7.002669334411621, 1.2226967811584473, -7.672550201416016, 0.08671045303344727, 0.05233955383300781, -8.072000503540039, -2.8441784381866455, -5.195993423461914, -5.370122909545898, -3.2225723266601562, 2.8780317306518555, -1.3843718767166138, 2.4719204902648926, 2.649190902709961, 3.221158504486084, 1.3194351196289062, -3.8373782634735107, -6.83131742477417, -2.989051580429077, -1.176582932472229, -0.6499824523925781, -1.0666987895965576, -6.776451587677002, -4.865273952484131, -2.761671781539917, 0.6023197174072266, 0.5549581050872803, -0.1523749828338623, -1.9242231845855713, -6.851547718048096, -3.7708115577697754, -1.4431922435760498, -0.6063710451126099, 3.306227684020996, -5.595459938049316, -0.06384491920471191, -6.609671592712402, -2.3022913932800293, -6.081998825073242, -0.48031091690063477, -0.6078109741210938, -2.470437526702881, -0.1905510425567627, -0.5104572772979736, -4.461342811584473, 0.12801790237426758, -5.822155475616455, 0.2936239242553711, 1.8422036170959473, -7.068687915802002, 0.5639662742614746, 3.979428291320801, -4.715689659118652, -0.3180779218673706, -4.320234775543213, 0.8810141086578369, 1.859128475189209, -5.234745025634766, -0.45819091796875, -3.404874324798584, 2.247304916381836, -0.790874719619751, 0.72019362449646, 2.211977005004883, -2.1416049003601074, -5.091164588928223, -1.1905043125152588, 4.586227893829346, 0.31708765029907227, -2.2800240516662598, 6.932302474975586, 0.8993165493011475, -2.522153377532959, 4.953824996948242, 0.6761462688446045, -2.275786876678467, -4.321744918823242, -2.9396986961364746, -8.487889289855957, -1.024770736694336, 2.0344018936157227, 0.28720974922180176, -1.058542251586914, 0.8487663269042969, -2.330284357070923, 0.13670086860656738, 0.8099749088287354, -1.6568114757537842, -4.2126569747924805, -1.1321337223052979, 0.6201968193054199, -0.4865388870239258, -0.49323296546936035, -1.3330895900726318, -2.306313991546631, -0.1830683946609497, -6.562249660491943, -5.035702705383301, -7.861019134521484, -6.016877174377441, -1.1481757164001465, -1.8398752212524414, -1.448676347732544, 3.2164955139160156, 0.012593269348144531, -5.93125057220459, 0.8024780750274658, -3.5729727745056152, -1.2399293184280396, -7.087418556213379, -2.0865070819854736, -3.6287755966186523, -2.5009169578552246, -3.4953291416168213, 1.2546517848968506, -3.9628746509552, 2.7704391479492188, 0.49134135246276855, -5.78092098236084, -3.241903066635132, -4.145888805389404, -5.545843601226807, -4.0864129066467285, 3.402153968811035, 1.4107234477996826, -0.24904704093933105, 0.7481889724731445, -4.438713073730469, -1.0163631439208984, -5.081876277923584, -1.6717098951339722, -3.438321113586426, -1.6912100315093994, -2.2965755462646484, -4.191703796386719, -2.7920827865600586, -2.4675278663635254, 2.4112167358398438, -1.1887413263320923, -2.611114025115967, -2.0292892456054688, -2.2265233993530273, 1.9620943069458008, -4.486117362976074, 1.3139939308166504, -0.24124908447265625, -9.022109985351562, 1.896698236465454, 0.2238478660583496, -4.39678955078125, 0.36967945098876953, -7.056329250335693, -0.5723675489425659, -1.3594683408737183, -5.528111934661865, 0.63787841796875, -0.7870326042175293, -5.464205741882324, 2.4436421394348145, -4.495075225830078, -1.8156012296676636, 2.7863707542419434, 1.5149693489074707, -1.7208820581436157, 3.00508975982666, 0.1633002758026123, -3.645582675933838, 0.25987863540649414, 0.007951498031616211, -0.2558631896972656, -2.623539447784424, -0.7807211875915527, -3.418699026107788, -1.376542329788208, -5.0624799728393555, -3.145667791366577, -1.0497431755065918, -2.3019022941589355, -4.176000595092773, 1.9370708465576172, -0.07363009452819824, -5.51279354095459, -2.5411429405212402, 0.9767420291900635, -4.9973530769348145, -6.578664779663086, -0.22456037998199463, 0.0885612964630127, -3.664034843444824, -4.304444313049316, -4.486093521118164, -1.0475997924804688, -1.1957180500030518, 1.4164018630981445, -1.4963657855987549, -6.144089698791504, 0.7663779258728027, -0.2344810962677002, -0.0024176836013793945, 0.9897336959838867, -2.2354142665863037, -3.5008316040039062, -7.553426265716553, -3.7962160110473633, -6.49117374420166, 2.025075912475586, -3.8371143341064453, 2.14860200881958, -3.3440499305725098, 1.5576653480529785, -5.78173303604126, -9.885248184204102, -0.12191486358642578, -1.329663634300232, -0.4000699520111084, -5.5704569816589355, -2.708165168762207, -5.039674758911133, -3.187577247619629, -4.393204689025879, -1.1551744937896729, 3.059488296508789, 0.3987915515899658, -4.251087188720703, -5.409754753112793, -5.2419352531433105, -2.7302772998809814, -1.4093985557556152, 0.8333640098571777, -3.5241692066192627, -0.8075565099716187, -4.398390769958496, 3.5689821243286133, -7.600322723388672, 0.7992150783538818, -2.4876623153686523, -0.032024264335632324, -6.133724212646484, -4.388174057006836, -4.767848014831543, 0.4944419860839844, -3.170971632003784, 1.1468052864074707, -2.8620760440826416, 0.722597599029541, -5.6273298263549805, -0.3875575065612793, -1.801668643951416, -3.160965919494629, 1.0312213897705078, -1.4097051620483398, -6.582156658172607, 3.78731632232666, -5.33962345123291, -4.752531051635742, -6.119696617126465, -3.2133233547210693, -1.8365579843521118, 1.8927617073059082, -4.1824049949646, -2.884474277496338, -0.818879246711731, 2.430079460144043, -4.8112640380859375, -1.4927468299865723, -3.9477267265319824, -7.679182529449463, -6.134295463562012, -3.506934642791748, -2.662668228149414, -4.6008734703063965, -1.905927062034607, -0.4722597599029541, -1.5448307991027832, 4.428454399108887, 1.538024663925171, 4.015244960784912, -2.0465087890625, -1.6190537214279175, 0.7281203269958496, -5.644317626953125, -3.9293875694274902, -2.689523220062256, -0.19549667835235596, 0.30962562561035156, -3.2237744331359863, -3.090700626373291, -0.6944520473480225, 0.7479403018951416, 2.320169448852539, -7.777026653289795, -5.5078840255737305, -0.4769103527069092, 0.15461063385009766, -4.528920650482178, -3.348001480102539, -5.376562118530273, -3.0435543060302734, -0.6112886667251587, -1.0869063138961792, 0.5386176109313965, -1.6810905933380127, -5.873766899108887, -3.281813383102417, -3.847407579421997, -3.2431249618530273, -5.026473045349121, -5.3288116455078125, -5.19137716293335, -1.633814811706543, -4.047177314758301, -6.424039363861084, 0.6855432987213135, -4.949965476989746, -4.931948184967041, -2.5973896980285645, -3.2973251342773438, -0.17893528938293457, -0.47127437591552734, -5.758225917816162, -0.2831670045852661, -0.46143126487731934, -0.0782160758972168, -1.7901110649108887, -1.0383610725402832, -0.2973219156265259, -3.4533262252807617, 0.7684659957885742, 0.449507474899292, -1.6885571479797363, -3.193150281906128, 0.9303717613220215, -0.9346868991851807, -0.0614471435546875, -2.7281548976898193, -0.4134622812271118, 0.2892909049987793, -4.409110069274902, -4.826976299285889, -7.83155632019043, 2.121150493621826, 2.517897605895996, -1.3749332427978516, -2.818673849105835, -7.094541072845459, -5.183010101318359, 0.19591450691223145, 0.22626256942749023, -0.6555060148239136, -1.2213671207427979, -0.6343178749084473, -3.3680384159088135, -3.144737720489502, -4.184207916259766, -1.9073455333709717, -1.7118513584136963, -0.7652945518493652, -6.339625835418701, -3.251600503921509, -3.0505542755126953, -4.133098125457764, -0.40082013607025146, -4.137364864349365, -2.264488697052002, 1.9027292728424072, -2.382901906967163, 3.6335911750793457, -3.556623697280884, -0.0607762336730957, 0.44737720489501953, -1.9936997890472412, 2.2654056549072266, -0.2172865867614746, -1.2056390047073364, -4.505036354064941, -4.49245548248291, 1.3518478870391846, -1.4097822904586792, -4.147334575653076, 4.277609825134277, -4.526490211486816, -6.751169204711914, -5.158778190612793, -6.93815803527832, -5.089213848114014, 1.6630170345306396, -1.2867629528045654, 0.020127296447753906, -2.1632745265960693, -0.11730122566223145, 3.068326950073242, -1.4737175703048706, -0.5394763946533203, -2.184401273727417, -5.072957515716553, -3.633873701095581, 1.8881008625030518, -3.421682834625244, 1.4271550178527832, -2.13423490524292, -5.27623987197876, -1.181257963180542, -1.4119820594787598, 0.029678821563720703, 3.247178077697754, 7.07249641418457, -3.3040812015533447, -1.9111981391906738, 1.6186776161193848, -0.11677062511444092, -0.42145252227783203, 0.2723844051361084, -1.409815788269043, 0.035581350326538086, 0.7350006103515625, 2.6843695640563965, -6.358779430389404, -2.7586143016815186, -6.5002031326293945, -7.670933246612549, -1.2219810485839844, -0.5389461517333984, -0.8283292055130005, 3.054335594177246, -0.41981959342956543, -1.2617980241775513, -3.2767467498779297, 0.08975887298583984, -3.4261038303375244, 0.7478268146514893, -4.354503631591797, -0.3121528625488281, -4.470981597900391, -4.747152328491211, -3.681600570678711, 0.28055596351623535, -4.600797653198242, -3.197192668914795, -2.1508262157440186, 3.8763132095336914, -1.7773756980895996, -0.12211728096008301, -4.6074652671813965, -1.6973239183425903, -6.672167778015137, -2.0485165119171143, -4.151886940002441, -3.8657422065734863, -4.381234169006348, -2.169926881790161, -2.130652904510498, -3.6916446685791016, -2.1095175743103027, -1.4166505336761475, -2.138437509536743, -0.929131031036377, -1.8900940418243408, 0.6023919582366943, -2.8397927284240723, 2.4275097846984863, -2.2403693199157715, 2.8913984298706055, -2.2392382621765137, -5.748490810394287, 0.05819892883300781, -4.896734714508057, -1.7250560522079468, -5.79941987991333, -5.3093414306640625, -1.2264811992645264, -0.048663973808288574, -4.179213523864746, -5.59105110168457, -1.3547614812850952, 1.4860029220581055, -4.017816543579102, -6.278549671173096, -3.8171133995056152, -6.059134006500244, -0.12854743003845215, 1.530198097229004, -3.0301525592803955, 2.517360210418701, -5.3443193435668945, -2.7299070358276367, 1.3702185153961182, 1.4251174926757812, -1.8086987733840942, -5.493677139282227, -1.995619535446167, -3.6025538444519043, -2.5588841438293457, 3.5879621505737305, -2.7498626708984375, -2.2505288124084473, -7.916165351867676, -1.643662452697754, -3.948927879333496, 0.5117921829223633, 1.2431273460388184, -6.172901153564453, 4.354229927062988, -1.4985473155975342, -3.6215627193450928, 0.8065183162689209, -1.3439279794692993, -1.9082297086715698, -4.945180892944336, -5.003832817077637, -3.1845202445983887, -3.823831558227539, -3.0638160705566406, -6.373373031616211, -5.054523944854736, -7.392180442810059, 0.8122951984405518, -6.37313175201416, -9.440178871154785, -6.27754020690918, -0.9824939966201782, -1.7894536256790161, -3.027235507965088, 4.11672830581665, -1.1966307163238525, -1.2073769569396973, 3.0223517417907715, -2.2698473930358887, 0.2301023006439209, -1.0429182052612305, 1.1085586547851562, -3.551095724105835, 0.5651283264160156, 1.2630674839019775, -0.12411999702453613, -2.059499979019165, -0.555433988571167, -1.757822036743164, -7.97168493270874, -2.4790306091308594, -3.2785162925720215, -4.167806625366211, -3.8046038150787354, -3.1955349445343018, -0.11316752433776855, -4.958375930786133, -5.153664588928223, -0.8812448978424072, 3.215134620666504, -1.7068471908569336, 1.5434608459472656, -3.854966878890991, -5.97914981842041, -0.41106724739074707, -2.369555950164795, -2.0420994758605957, 0.4264659881591797, -5.39881706237793, 1.6158146858215332, -3.8617746829986572, -4.332880973815918, 2.0134968757629395, -2.2171804904937744, 1.3229658603668213, -7.472317218780518, 0.5959436893463135, -1.9000428915023804, -0.9650253057479858, -3.3103232383728027, 0.6429762840270996, -1.057248830795288, 0.7387652397155762, -5.5521063804626465, -2.9443955421447754, -3.718202590942383, 0.006566524505615234, 0.2696864604949951, -10.147414207458496, -3.7517409324645996, 1.1332406997680664, 0.6970868110656738, -0.44929659366607666, -6.589588642120361, -3.541335105895996, 0.08482217788696289, -2.6319546699523926, -4.537815093994141, 1.5890302658081055, -3.0813231468200684, -4.073921203613281, 5.848794937133789, -5.343070983886719, -2.005046844482422, -4.638515472412109, -8.080134391784668, -3.294107675552368, 0.4475245475769043, -8.281804084777832, 0.5919778347015381, -1.0634320974349976, 0.22680425643920898, 5.379166126251221, -6.814672946929932, -0.47384774684906006, -0.3076145648956299, -6.255752086639404, -0.39930450916290283, -0.14834284782409668, 2.719510555267334, 0.13016343116760254, 1.7092018127441406, -4.126919746398926, -5.541977882385254, -3.4137115478515625, -3.9037742614746094, 0.1254894733428955, -6.488562107086182, -3.816497802734375, -1.475048542022705, -6.484414100646973, -3.0749263763427734, -6.313324928283691, -3.0645713806152344, 0.1962435245513916, 1.299504280090332, 0.14461755752563477, -3.2619452476501465, -4.259903907775879, 0.6029868125915527, -0.8740881681442261, -6.167184829711914, 1.6305389404296875, -4.224393844604492, -5.479206085205078, -2.2019121646881104, -5.538772106170654, 2.3681211471557617, -3.5505454540252686, -7.2529802322387695, -8.522590637207031, -3.078126907348633, -4.597482204437256, -4.60628080368042, -1.1043622493743896, -2.0740816593170166, -4.7664794921875, -1.0902396440505981, -0.1041107177734375, -8.622549057006836, -5.041558265686035, -2.1760871410369873, -4.4524922370910645, 2.284031867980957, 4.326839447021484, 1.731130838394165, 1.3666539192199707, -2.777838706970215, 1.313392162322998, -1.740743637084961, -1.3594999313354492, -7.831927299499512, -5.563424587249756, -5.437615394592285, 3.05764102935791, -0.2609325647354126, 0.7171235084533691, -2.9170637130737305, -1.9468016624450684, -1.1767511367797852, -2.3544342517852783, 0.9056625366210938, 1.80535888671875, -3.690316915512085, -4.271526336669922, -0.702261209487915, 1.172250747680664, -9.85999870300293, 2.204026222229004, -0.41690564155578613, -1.1219682693481445, -1.7988111972808838, -5.706241130828857, -6.068955421447754, -4.491302490234375, 0.3214411735534668, -6.963037490844727, 6.317609786987305, 2.109757900238037, 0.027299880981445312, -0.0822901725769043, -2.3070108890533447, -4.853448390960693, -3.9948372840881348, -3.154086112976074, 1.9924726486206055, -1.8117737770080566, 0.7168822288513184, -2.108945608139038, -5.968955993652344, -0.26088404655456543, 5.106334686279297, -0.688620924949646, -2.1313676834106445, -5.17352294921875, 4.224056720733643, -5.802647590637207, -1.7151437997817993, -1.7416287660598755, 4.276828765869141, -1.647241234779358, -1.5997090339660645, 2.2454895973205566, -6.594891548156738, -6.019055366516113, -7.754401206970215, 1.643587589263916, -2.254873275756836, 1.1946630477905273, -6.8950276374816895, -6.4337968826293945, -1.4919769763946533, -1.460693597793579, -1.6706597805023193, -2.4731225967407227, -5.334108352661133, 2.1075820922851562, 3.0712528228759766, -1.5956027507781982, -5.461885929107666, 1.6352639198303223, -0.6267375946044922, -4.799515247344971, 0.35605692863464355, -3.688502073287964, -6.850009441375732, -5.360490798950195, 2.6145219802856445, -6.058727264404297, 0.33616161346435547, -0.6674497127532959, 5.035927772521973, -4.389513969421387, 1.7331037521362305, 0.8750550746917725, -2.406818389892578, -0.7480889558792114, -3.1959128379821777, -4.626938819885254, -2.3948276042938232, -0.453008770942688, 0.2525787353515625, -5.541009426116943, -4.315048694610596, -1.4855631589889526, -1.331527829170227, -4.267665863037109, -9.32877254486084, -0.6389484405517578, 1.1195812225341797, -9.930231094360352, -1.6394141912460327, -0.4357457160949707, -1.254920244216919, -2.141749382019043, 1.1814837455749512, -1.145275592803955, -1.3690513372421265, -8.267269134521484, -6.51515531539917, -6.723382472991943, 2.3939929008483887, -2.3218421936035156, 3.7509751319885254, -0.3056058883666992, -0.8256590366363525, -5.001196384429932, 0.05450868606567383, -3.8408448696136475, -0.15534579753875732, -4.068232536315918, -1.2555681467056274, -0.2799375057220459, -4.2997894287109375, 2.712843894958496, 1.259648323059082, -0.16342997550964355, -0.6854784488677979, -1.754637360572815, -2.1661229133605957, -0.8070158958435059, -1.6511934995651245, 2.623652458190918, 0.41995882987976074, -1.2185577154159546, -4.0216193199157715, -5.576351165771484, -2.2720954418182373, -1.6826598644256592, -2.6384353637695312, -2.7332019805908203, -4.557292938232422, -1.6228018999099731, -1.8723896741867065, 0.7195842266082764, -4.130110740661621, -1.3708637952804565, 1.3116707801818848, -2.7409324645996094, -5.834379196166992, 1.0565705299377441, -1.3188828229904175, -3.0153512954711914, -4.035324573516846, -4.731320858001709, -7.9753499031066895, 1.4129753112792969, 1.250706672668457, 2.244898796081543, -0.36977052688598633, -5.908447265625, -1.954376459121704, -5.433621406555176, -4.4769487380981445, 0.39790868759155273, -6.931515693664551, -0.02640986442565918, 0.2648453712463379, -0.07386088371276855, -1.9061951637268066, -7.681766033172607, 1.8631024360656738, -4.333550453186035, -2.7627177238464355, 0.202103853225708, -6.6639533042907715, 1.1491243839263916, 1.5706102848052979, -4.678949356079102, 3.3808436393737793, -5.136155128479004, -2.334712028503418, 2.1419477462768555, -1.635279655456543, -2.2573599815368652, -0.5332759618759155, -6.224703788757324, -2.525594711303711, -3.8624422550201416, -4.1610307693481445, -1.732597827911377, -3.136420726776123, 1.80717134475708, 1.4720311164855957, 0.7856926918029785, 0.71944260597229, -2.9819576740264893, 3.773040294647217, -0.8596621751785278, -7.5862531661987305, 0.6337251663208008, -1.2896838188171387, -5.632863521575928, -0.5551607608795166, -8.02576732635498, -7.855688095092773, -5.501655578613281, -1.947644591331482, -0.33751893043518066, -1.0937511920928955, 3.320542335510254, -10.868497848510742, -6.680855751037598, 1.1188225746154785, -6.340536594390869, 0.22450637817382812, 1.9586751461029053, -5.550081253051758, -1.8004118204116821, -5.162926197052002, -4.4993462562561035, -0.037358999252319336, -4.352431297302246, -2.288576126098633, 1.9400177001953125, 3.099517822265625, -1.3214046955108643, -2.946047067642212, 0.8458914756774902, 0.40303850173950195, 0.1341538429260254, -0.018735408782958984, 1.5120606422424316, 0.06358003616333008, -6.508993625640869, -1.8344221115112305, -1.5515564680099487, -6.2364044189453125, 2.6278305053710938, -4.32767915725708, -2.054014205932617, -5.1627373695373535, -5.40780782699585, -3.31781268119812, -5.819979190826416, 3.655470848083496, -2.2492971420288086, 0.8744287490844727, 0.5244483947753906, 1.545417308807373, 1.130289077758789, -1.1920872926712036, 1.8785507678985596, -5.033809661865234, 1.8351376056671143, -4.781515121459961, -1.2251315116882324, -4.47666072845459, -3.7863707542419434, -2.256906270980835, 2.3528614044189453, -0.09931826591491699, -2.668409824371338, -1.3541228771209717, 1.0675873756408691, -1.307626485824585, 3.080813407897949, -7.16025447845459, 2.624577522277832, -0.2960324287414551, -3.731011152267456, -2.671818971633911, -8.486542701721191, -6.3171820640563965, 0.1002969741821289, 1.1270344257354736, -4.50441837310791, -9.141105651855469, 1.9885997772216797, -3.9833061695098877, -2.8949525356292725, -1.979710340499878, -3.1200337409973145, -5.083183765411377, -2.457329273223877, -4.0495500564575195, -0.2091062068939209, -4.311563014984131, -12.074138641357422, 0.8488070964813232, -6.286891460418701, -3.638260841369629, -1.056520700454712, -4.982549667358398, 1.9322137832641602, -0.07149147987365723, 2.5711612701416016, 1.8244616985321045, -1.6691792011260986, -3.217833995819092, -4.573862075805664, -8.235577583312988, -5.321188926696777, -3.60465669631958, 2.5874719619750977, -0.7299216985702515, -1.7526777982711792, -4.156223773956299, -1.3470981121063232, -0.29947543144226074, 2.4761009216308594, -1.076708436012268, -1.4242241382598877, -4.9741668701171875, -4.387076377868652, 1.769838809967041, -3.6349709033966064, 6.3073320388793945, -1.8574469089508057, -2.7456722259521484, -2.069149971008301, 0.2528219223022461, -1.5397673845291138, -9.700789451599121, -2.9764480590820312, -0.5735936164855957, 4.2749857902526855, -1.0153590440750122, 1.4704341888427734, 2.071249485015869, 0.07629013061523438, -1.845686435699463, -3.8717520236968994, -3.2951221466064453, 3.418790817260742, -3.387767791748047, -2.2064645290374756, -3.8943324089050293, -0.5294914245605469, 1.4687104225158691, 3.7139363288879395, -3.5601797103881836, 2.2703161239624023, -3.5729966163635254, -4.153205394744873, -2.4440481662750244, -1.4555573463439941, -1.0756123065948486, -2.2647223472595215, -4.868686199188232, 0.9154036045074463, -2.37941837310791, 2.1844139099121094, -3.697871685028076, -1.542299747467041, -4.810041904449463, -3.15210223197937, 1.1709446907043457, -3.2851312160491943, -3.773983955383301, -7.388306617736816, -0.8995070457458496, -6.966124534606934, 1.392803430557251, -2.7017617225646973, 2.634798049926758, -4.159172534942627, 1.9510376453399658, -0.11830270290374756, -2.1770403385162354, 5.08165979385376, 1.9285027980804443, -7.386979103088379, -1.4043232202529907, 2.6064558029174805, 1.9661359786987305, 0.17727422714233398, -5.000823974609375, -2.7083511352539062, 0.22011637687683105, -2.71018648147583, -4.143603801727295, -2.337256669998169, -0.38306093215942383, -2.4662797451019287, -6.967966079711914, -0.6071790456771851, -2.2632808685302734, -5.376200199127197, -2.6253068447113037, -1.134928584098816, -2.7897088527679443, -1.5752334594726562, -2.3977627754211426, -2.3906219005584717, -2.833967685699463, 1.8099863529205322, 2.763453483581543, -0.1597139835357666, 4.530174732208252, -5.153360366821289, 0.601503849029541, -1.0121647119522095, -6.065034866333008, -0.9418454170227051, 1.3058111667633057, -4.934581756591797, 1.740567684173584, -4.397369384765625, -2.3411216735839844, 1.2657010555267334, -2.0645596981048584, -4.328543186187744, -3.407033681869507, 0.761343240737915, 1.7541837692260742, -2.415555477142334, 1.399566650390625, -1.5647491216659546, -5.074656963348389, -2.918059825897217, -1.5319201946258545, -0.7040356397628784, 1.7480716705322266, -5.114740371704102, -3.4491090774536133, 3.219468116760254, -4.175182342529297, 3.7919845581054688, -3.4917500019073486, -1.7211287021636963, -2.0606093406677246, -0.8607646226882935, -6.122287273406982, -8.112746238708496, -1.4716812372207642, -2.570077896118164, 0.3973977565765381, 1.5499308109283447, 0.48050546646118164, -3.5960476398468018, -5.897948265075684, 1.032320261001587, -1.4000935554504395, -3.159532070159912, 3.49296236038208, -0.4398822784423828, -3.367130756378174, -6.409013271331787, 3.837502956390381, 1.6745009422302246, 0.6744401454925537, 1.5752336978912354, 3.6766858100891113, -0.7757229804992676, -3.949599266052246, -5.830624580383301, -1.2314585447311401, -1.2763549089431763, -0.1323639154434204, -5.084985733032227, -0.45300841331481934, 2.2076282501220703, -2.743107318878174, 0.5694513320922852, -2.9834952354431152, -1.7942748069763184, -3.7743425369262695, -0.8215548992156982, -2.329414129257202, -2.779904842376709, -2.4651236534118652, 0.10885334014892578, -0.34702789783477783, 0.6969504356384277, -0.6921718120574951, -4.97734260559082, 1.9204611778259277, -5.5626420974731445, -5.803943157196045, -0.8754104375839233, -5.093149662017822, -1.996756672859192, -0.4783153533935547, -2.679874897003174, -0.8516753911972046, 1.3550610542297363, -3.6783246994018555, 3.7394256591796875, -2.7572855949401855, 2.4838204383850098, 0.7802457809448242, -0.6460516452789307, -4.49308443069458, -6.241129398345947, -2.161585569381714, -0.9713654518127441, 1.267486333847046, -5.77443265914917, -3.3234033584594727, 0.0664358139038086, -5.250222206115723, -5.437231063842773, -3.33874773979187, -2.93845272064209, 0.9676642417907715, 4.109759330749512, -4.288829326629639, 0.02709674835205078, 1.1089112758636475, 5.332012176513672, -0.1669830083847046, -1.9800333976745605, -2.7692770957946777, 1.5955438613891602, 0.8803181648254395, 1.7996325492858887, 0.028820514678955078, -5.736998081207275, -2.8225059509277344, -1.5939266681671143, 0.7229263782501221, -6.504807472229004, -6.315260887145996, 1.4262113571166992, -2.056302070617676, -5.037620544433594, -1.1044492721557617, 3.3669824600219727, 0.3306090831756592, -6.989998817443848, 1.183227777481079, -5.690335750579834, -0.028985142707824707, 1.299931526184082, -1.0935425758361816, 2.9988975524902344, -7.365507125854492, -4.831696033477783, -3.07151460647583, 4.106058120727539, -2.594902515411377, 0.33518075942993164, 1.1913065910339355, 1.8359246253967285, -1.1550472974777222, -3.794100046157837, -0.7927703857421875, -4.982844829559326, 1.4801368713378906, -3.6373353004455566, -5.904807090759277, -2.647507429122925, 0.5437817573547363, -1.0412664413452148, -5.889561653137207, -4.105372905731201, 1.4275591373443604, -4.266242027282715, 0.5043268203735352, -3.196523904800415, -4.031123161315918, -4.919137954711914, -2.054719924926758, 0.5929539203643799, 0.7060327529907227, -4.919149398803711, 0.8960244655609131, -3.5438232421875, -1.4768407344818115, 0.45529770851135254, -4.7992095947265625, -3.062185287475586, 3.412735939025879, -5.145239353179932, -0.6576762199401855, 0.9849622249603271, -5.282639503479004, -2.6659340858459473, -3.1539804935455322, -6.813303470611572, 0.9777007102966309, -1.5499299764633179, -3.359593629837036, -5.176452159881592, -4.234613418579102, 2.0022149085998535, -3.2447938919067383, -4.596043109893799, -2.1304683685302734, -4.8804521560668945, -1.0002732276916504, 0.5213265419006348, 1.153118371963501, -3.1282002925872803, -1.6577626466751099, 5.228476524353027, -4.533456325531006, -0.04424583911895752, 0.8407077789306641, -4.76237678527832, -3.9582371711730957, -1.9752041101455688, 0.008400440216064453, -5.3773040771484375, -2.6698455810546875, 0.7116436958312988, -3.3595504760742188, 2.6392998695373535, -0.8747230768203735, -2.580888032913208, -5.694050312042236, -4.478188514709473, -5.381662845611572, -2.914280891418457, -3.8382792472839355, 1.6863765716552734, -4.776989459991455, 0.12302064895629883, -4.6660027503967285, -2.2506136894226074, -3.629408359527588, 1.3650965690612793, 0.14456558227539062, -2.229313373565674, -3.4416983127593994, 1.8299689292907715, -11.917598724365234, 1.3953227996826172, -2.7211077213287354, -1.0044095516204834, -6.897629737854004, -2.378554582595825, 2.6343955993652344, -1.9427709579467773, -4.108410358428955, -1.7872364521026611, 0.3891456127166748, 1.909836769104004, -4.4311652183532715, 1.4814929962158203, 3.5827393531799316, -7.169375896453857, -0.6996654272079468, 0.9061164855957031, 0.7365789413452148, -2.104361057281494, 0.711193323135376, -4.861558437347412, 2.884955883026123, 2.144643783569336, 0.6725039482116699, 4.7303266525268555, -4.061293601989746, -3.248795986175537, -5.464860439300537, -0.5917398929595947, 0.581049919128418, -3.0492920875549316, -0.013007044792175293, -3.5114290714263916, 2.0401062965393066, -4.3715715408325195, -1.4355053901672363, -1.8393855094909668, -1.3351202011108398, -7.152522087097168, -2.8223042488098145, 0.5077886581420898, 2.636417865753174, 0.18094730377197266, -2.496072292327881, -6.997967720031738, -1.582196593284607, -4.591904640197754, -4.143905162811279, -3.0325164794921875, 0.13288068771362305, -9.01643180847168, -5.6165995597839355, -2.399801731109619, -1.2092307806015015, -10.166352272033691, -8.657395362854004, -5.636100769042969, -7.898987770080566, 0.8835563659667969, -0.982457160949707, -6.157235622406006, -0.31244921684265137, -0.5244615077972412, -2.7048118114471436, -2.7099123001098633, -1.109480381011963, 5.830888748168945, -0.8329119682312012, -1.636536955833435, -2.7844181060791016, -0.5591633319854736, -0.6097010374069214, -5.245179653167725, -6.165493011474609, -2.651296615600586, 0.26219940185546875, -1.0133508443832397, -1.0744099617004395, -1.552134394645691, -6.123006820678711, 3.4372434616088867, -4.749794960021973, 1.281151294708252, -8.353109359741211, -0.9061496257781982, -0.27619433403015137, -0.48407649993896484, -8.028182983398438, -3.023102283477783, 1.0892224311828613, -3.8751540184020996, 3.161524772644043, -6.635633945465088, -1.0054452419281006, -2.809900999069214, -5.303549766540527, -0.13694357872009277, -3.2092695236206055, -2.1230509281158447, -3.9237663745880127, -5.588388442993164, 0.5879294872283936, -9.762737274169922, -2.44081974029541, -2.897284984588623, -1.9658721685409546, -3.7621798515319824, -2.5911951065063477, -3.3553266525268555, 0.05701017379760742, -0.24138665199279785, -1.174025297164917, -2.4552383422851562, 3.2282676696777344, -0.19243741035461426, -1.169606328010559, -1.7098793983459473, 1.7192997932434082, 0.626776933670044, -4.591702461242676, -0.2381885051727295, -7.1184282302856445, -8.265806198120117, -3.461017608642578, -2.7623648643493652, -3.9077255725860596, -3.0890579223632812, -1.0574004650115967, -2.2055206298828125, -3.27248215675354, -2.8170883655548096, -1.0616008043289185, -7.045684814453125, 1.382239818572998, -0.9346441030502319, 0.8204843997955322, -1.4876399040222168, -2.0649290084838867, -1.2074618339538574, -4.134192943572998, -5.369044303894043, -4.794671535491943, -5.322332382202148, -7.144801139831543, -5.720394611358643, 1.405728816986084, -6.199076175689697, -1.1233971118927002, -2.2396368980407715, -12.039725303649902, 0.663008451461792, 5.805665016174316, 0.06859111785888672, -4.397311210632324, -3.2614552974700928, 0.2532224655151367, 0.11658716201782227, -4.647154808044434, 0.3341355323791504, 1.282186508178711, -3.1689155101776123, -7.189876556396484, -2.449000597000122, -1.8809508085250854, 1.5976476669311523, -1.7723655700683594, -2.938657522201538, -6.664790153503418, -0.8913413286209106, -3.2928411960601807, -3.8183915615081787, -1.6833041906356812, -5.654369354248047, 2.0330615043640137, -2.628427505493164, -5.689275741577148, -6.9695024490356445, -1.4469715356826782, -3.5785279273986816, 3.3973617553710938, 1.1172494888305664, 2.19460391998291, -4.885780334472656, 0.8477659225463867, -0.3533196449279785, -5.995447635650635, -7.622239112854004, -5.03525447845459, -2.448065757751465, -0.6226558685302734, -5.506319999694824, 1.5231280326843262, 0.09612178802490234, -1.0687345266342163, -3.6290767192840576, -2.957704544067383, -0.27152490615844727, -1.9801303148269653, -5.627617359161377, -7.8303141593933105, -1.778268575668335, -0.7802419662475586, -7.016963005065918, -6.419454574584961, -0.05695152282714844, -0.1966317892074585, -0.23085856437683105, -0.7117459774017334, -4.748171806335449, 0.07215666770935059, -0.30835962295532227, -2.9125008583068848, 0.357433557510376, -3.747028112411499, -9.909952163696289, -3.04050612449646, 5.908617973327637, 3.4008240699768066, -3.4002485275268555, 1.250260829925537, -0.8697030544281006, 4.186613082885742, -4.739788055419922, -1.0328912734985352, 0.3115415573120117, -2.139741897583008, -4.427846431732178, -0.34774887561798096, -1.5232858657836914, -2.1608338356018066, -0.26536011695861816, -5.309417724609375, -3.280348777770996, -4.5812602043151855, 1.1750996112823486, 0.4719679355621338, -5.731044769287109, 1.815176010131836, -2.5485007762908936, -3.0602216720581055, -1.4217088222503662, 0.20429372787475586, -4.713266372680664, -0.22336483001708984, -3.3275020122528076, 0.2470560073852539, -4.0399932861328125, -1.6495195627212524, -0.4992964267730713, -3.040255546569824, 3.5001068115234375, -4.6425604820251465, -1.3014681339263916, -4.087225914001465, -1.463599443435669, -0.8670648336410522, -2.46390700340271, -4.648172378540039, -4.126498699188232, 1.3082008361816406, -0.9600671529769897, -11.212647438049316, 1.7482848167419434, -0.10576200485229492, -4.672679901123047, -3.0620229244232178, -4.450101852416992, -2.750744342803955, 1.5388665199279785, -5.224569320678711, 1.598710060119629, -2.1490845680236816, -5.6510443687438965, -4.030477523803711, -5.65146017074585, 0.18643569946289062, 0.8844926357269287, 1.0733771324157715, 2.697077751159668, -1.3157331943511963, -1.82574462890625, -4.138200759887695, -5.081652641296387, 0.6590027809143066, -0.4474663734436035, 1.5362865924835205, -2.22719407081604, -2.354156494140625, -1.9069240093231201, -5.951601505279541, -1.0110504627227783, -5.2373552322387695, -6.635494232177734, -0.1898425817489624, -3.7799172401428223, 1.34159255027771, -5.340072154998779, -3.9369187355041504, -3.742722511291504, -6.682520866394043, 5.9688286781311035, -2.8603129386901855, 1.637021780014038, -4.704704284667969, -1.885928750038147, -4.552738189697266, 3.392737865447998, 1.4633805751800537, -4.641634464263916, -1.5224590301513672, -3.9768741130828857, 2.104426383972168, -0.8713268041610718, 0.9132754802703857, -3.8380544185638428, -0.05374574661254883, 0.2586662769317627, -1.8993892669677734, -0.981386661529541, -3.816504955291748, 3.768465042114258, 3.0600099563598633, -6.242257118225098, -2.9176149368286133, 0.03787732124328613, 1.7963318824768066, -0.31252360343933105, -5.4960761070251465, -2.5635523796081543, -2.6672725677490234, -2.677612066268921, -8.621297836303711, -0.7455371618270874, -0.5212666988372803, -5.150420665740967, 1.9323768615722656, -4.912546634674072, 1.224724292755127, -2.5188345909118652, -6.1981353759765625, -2.947434425354004, 6.652200698852539, -0.47105610370635986, -1.973054051399231, -9.249748229980469, 7.035765647888184, -3.7571091651916504, 3.537026882171631, -2.933615207672119, -3.3977646827697754, -5.71177864074707, 0.9645640850067139, 0.4157063961029053, -6.445796012878418, -6.425235748291016, -0.2734074592590332, 2.866715431213379, -1.7331558465957642, -2.371839761734009, -1.8502262830734253, -0.9260798692703247, -1.981132984161377, -1.328784465789795, -8.212329864501953, -1.2903451919555664, 2.464937210083008, -1.1759397983551025, -4.8886871337890625, -3.764920949935913, 2.4849815368652344, -0.47985029220581055, -4.042160987854004, -2.503260850906372, -1.5296379327774048, -0.030370593070983887, -0.06073462963104248, -2.6935830116271973, -2.578434705734253, -4.223444938659668, 1.5450248718261719, -2.342031717300415, -6.541096210479736, 2.0690793991088867, 0.28676843643188477, -1.614567518234253, -2.2580816745758057, 3.4626755714416504, -3.0274109840393066, -1.8967722654342651, 0.5358543395996094, -1.4666204452514648, 1.7076058387756348, -7.876458644866943, 1.1978888511657715, -4.132425785064697, -1.920971155166626, 0.8571269512176514, -5.140754222869873, -6.199650287628174, -11.487771987915039, 3.710665225982666, -3.1919291019439697, 0.008585691452026367, -2.015913248062134, 1.9454665184020996, 0.20946598052978516, -1.6251270771026611, -5.27045202255249, -5.099870681762695, -1.9182449579238892, -4.227624416351318, -3.1189956665039062, -1.976755142211914, -2.9978981018066406, -2.711794853210449, -1.5761518478393555, -6.413586139678955, 3.042046546936035, -4.266037940979004, -0.951440691947937, -4.319934368133545, -6.044829845428467, -1.8583704233169556, -3.944140911102295, -0.469903826713562, 2.063417434692383, -3.0626349449157715, -0.8357270956039429, -1.30546236038208, -3.1679444313049316, -4.4058451652526855, 0.5628821849822998, -3.059162139892578, -4.175891399383545, -2.9554290771484375, -0.7828278541564941, -3.4782376289367676, -1.7126985788345337, 0.7285313606262207, -4.296517372131348, -2.1074535846710205, -4.204684257507324, -3.605278253555298, 3.0233516693115234, 2.456094264984131, -1.6686612367630005, -5.3544020652771, -1.5500335693359375, -6.8064961433410645, -1.5743651390075684, -4.648838520050049, -1.7601162195205688, -0.5540601015090942, 2.4505739212036133, -5.0244059562683105, -3.335561752319336, -0.9222472906112671, 5.5775346755981445, -5.889969825744629, -6.063262939453125, -2.595613956451416, -2.141681671142578, -1.4993466138839722, -3.626990795135498, -1.2477073669433594, -2.3350629806518555, 0.8999719619750977, -3.1658482551574707, -1.4354448318481445, 1.8019676208496094, -2.2532551288604736, -4.165776252746582, -2.366147756576538, 1.4925415515899658, 4.730206489562988, -0.5383956432342529, -4.733548164367676, 3.91278076171875, -4.7483134269714355, 0.8167500495910645, -6.6690473556518555, 4.243651866912842, -4.968550682067871, -8.654097557067871, 2.6242504119873047, -4.48895788192749, -1.8081114292144775, 1.3074421882629395, -2.1750495433807373, 0.9415061473846436, -3.9796016216278076, -1.592879295349121, -0.789219856262207, -6.613152503967285, -1.260201334953308, -5.256464004516602, -2.8537349700927734, 0.6314811706542969, 2.9332633018493652, -2.392124652862549, 1.3463478088378906, -1.661555290222168, 0.9003868103027344, 4.233083724975586, -4.010400295257568, -1.2124102115631104, -6.895389556884766, 4.672719478607178, -0.24275648593902588, 0.39174842834472656, 2.0116829872131348, -2.1104917526245117, 0.2334599494934082, -1.8542569875717163, -3.814420223236084, -0.49941134452819824, -5.236935615539551, -8.674849510192871, -2.6978962421417236, 3.039407730102539, -1.228877305984497, 0.3428490161895752, -2.622893810272217, 2.5042991638183594, -4.453800678253174, -1.897271752357483, -5.439434051513672, -2.475653648376465, -2.832871198654175, -3.1326680183410645, -3.291368007659912, -2.198303699493408, -4.127049922943115, -5.134520530700684, -2.1244897842407227, -6.991744518280029, -6.164772987365723, -0.15491807460784912, -2.3389790058135986, -0.9711205959320068, -5.217959403991699, 2.370912551879883, -4.776717185974121, 2.230152130126953, -3.698176860809326, -5.435737609863281, -2.7447850704193115, -2.376004695892334, 2.167904853820801, -3.8438973426818848, 1.2113385200500488, 1.9935474395751953, -3.1080143451690674, -1.685311198234558, -0.7211024761199951, -1.0943493843078613, 6.581245422363281, -3.99983811378479, -2.632284641265869, -1.542389154434204, 2.1274194717407227, 0.49439477920532227, -4.010856628417969, -2.0063090324401855, -3.4073562622070312, 1.1866493225097656, -3.7516393661499023, -8.574163436889648, -1.8579145669937134, 0.4257049560546875, -4.578225135803223, -6.885849475860596, 4.559059143066406, -1.5808632373809814, -1.2206110954284668, -4.655023097991943, -1.3128385543823242, 1.7655529975891113, 2.2186660766601562, 4.648990631103516, -1.82276451587677, 0.5278282165527344, 2.045757293701172, -1.3146076202392578, 2.3880271911621094, 1.1840832233428955, -4.630087852478027, -2.349579095840454, -8.511280059814453, 0.9197969436645508, -4.451980113983154, 0.9443566799163818, 0.9995298385620117, -0.06460201740264893, -5.659633636474609, -1.3159475326538086, -6.502434730529785, -3.325075626373291, 1.0659160614013672, -1.6768826246261597, -0.6745259761810303, -3.9488508701324463, 0.8832743167877197, -1.7128796577453613, -0.711746335029602, -3.2359533309936523, 5.658285140991211, -4.373701572418213, 0.3565967082977295, 2.370121955871582, 0.43185949325561523, -1.950807809829712, -3.517350435256958, -1.0725021362304688, 0.5577535629272461, -5.2105536460876465, -3.221452236175537, -1.4005253314971924, 5.299998760223389, -1.7367950677871704, -0.2948751449584961, -6.157915115356445, -8.215105056762695, -2.084899425506592, -0.39292192459106445, -1.4874227046966553, 3.2206153869628906, 1.504652976989746, -1.1763813495635986, -1.8857219219207764, -0.17511630058288574, 1.906454086303711, -5.461725234985352, -7.045437335968018, -5.134031295776367, 0.25051259994506836, -3.4242734909057617, 0.5729513168334961, 3.487368583679199, -0.8859628438949585, -6.041616439819336, -3.915926456451416, -4.889216899871826, -1.3795313835144043, -1.762919306755066, -5.748372554779053, -1.6099823713302612, 1.6292529106140137, -1.5692718029022217, -4.336718559265137, -2.629887819290161, -0.4946908950805664, -5.827991962432861, -7.155240535736084, -5.414308071136475, -4.750743865966797, -3.6993062496185303, 4.138792991638184, -1.5621364116668701, -4.851226806640625, -2.6839938163757324, -4.380060195922852, -6.4210991859436035, -7.002908706665039, 1.0038933753967285, -4.641223907470703, -1.1542476415634155, -5.865852355957031, -1.015101432800293, 5.2466325759887695, -1.1508278846740723, -0.5350837707519531, 4.894737720489502, -4.938265323638916, -2.648275375366211, -8.001794815063477, 1.1271018981933594, 0.5322210788726807, 0.11754179000854492, -2.6762290000915527, -3.8733038902282715, -1.3071041107177734, 1.0416054725646973, -3.8107492923736572, -3.1548800468444824, 2.8924455642700195, -1.6796945333480835, -5.5636396408081055, -7.206231594085693, -3.5044140815734863, -0.8928282260894775, -0.7909650802612305, 0.5861008167266846, 0.8668220043182373, -1.3539127111434937, -0.5745683908462524, -4.016539573669434, 2.5288305282592773, 0.8450005054473877, 1.704880952835083, -1.8490703105926514, -0.7294527292251587, 0.4749886989593506, 3.2754569053649902, -4.705504417419434, -1.3944549560546875, 2.8635191917419434, 0.038031816482543945, -3.1187102794647217, -3.8581430912017822, -1.889029622077942, -7.271692752838135, -1.9237732887268066, 1.6490583419799805, -3.5543129444122314, -0.5555899143218994, -5.080112457275391, -1.9604076147079468, -1.4305397272109985, -3.9547805786132812, -0.5766555070877075, -1.214759111404419, 0.47479987144470215, -5.164444923400879, -0.02988433837890625, 1.9604249000549316, -4.587639808654785, -1.7394099235534668, -0.1981809139251709, -4.364021301269531, -5.706281661987305, -0.9017550945281982, 0.27176547050476074, -1.9097719192504883, -1.7795941829681396, -4.074844837188721, 1.3182945251464844, 1.9617972373962402, -3.736203908920288, -7.571527004241943, 1.2812838554382324, 0.06627845764160156, -3.5396227836608887, -0.7767291069030762, -5.012116432189941, -3.599435329437256, -3.038567066192627, 4.551234245300293, -2.1876320838928223, -2.8473434448242188, -6.2581987380981445, -4.872013092041016, -2.565857172012329, -2.901531457901001, -1.3311257362365723, -9.035728454589844, -4.825368881225586, -5.775838375091553, -2.4648804664611816, -5.58187198638916, 1.5853662490844727, -0.7735151052474976, -2.0622987747192383, -4.5546064376831055, 0.7881391048431396, -3.1405389308929443, 0.24023699760437012, -6.493261814117432, -5.42198371887207, 1.459770679473877, 4.594653129577637, 2.4128313064575195, -7.272712707519531, -0.4767136573791504, 0.6456832885742188, 2.811614990234375, -1.7104616165161133, -1.800628662109375, -2.387540817260742, -3.3301730155944824, -4.369264125823975, -3.1240217685699463, 1.7410035133361816, -2.390925407409668, -9.316627502441406, -3.622969388961792, -0.34142887592315674, -5.691476821899414, -1.6279128789901733, -4.564042091369629, -2.3567376136779785, -2.1704800128936768, -1.405158519744873, -2.5107827186584473, -3.6892471313476562, 0.7430398464202881, -6.070476531982422, -4.2964701652526855, -0.34013867378234863, -2.0865063667297363, -1.0904183387756348, -2.7923531532287598, -4.614645004272461, -1.2050073146820068, 0.6230103969573975, -0.5778548717498779, -1.0768362283706665, -2.9127306938171387, 1.4629549980163574, -6.686336517333984, -0.6401033401489258, -1.5881093740463257, -5.458977699279785, -4.03544282913208, 0.5612878799438477, -0.3593374490737915, -3.850322961807251, -0.7002352476119995, -4.827259540557861, -4.671292304992676, -3.121415376663208, -3.944091796875, -3.979055881500244, -7.239458084106445, 0.9536426067352295, 2.385725975036621, -7.914127826690674, -7.641219139099121, -3.4682440757751465, -0.11586308479309082, -4.822010517120361, 0.8329291343688965, -5.698859214782715, -0.5645064115524292, -2.206709146499634, -4.748675346374512, -1.485746145248413, -5.0280656814575195]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [-1.9716771841049194, -2.01712965965271, -1.9781297445297241, -2.036933422088623, -1.961519479751587, -1.9607057571411133], "dtype": "float32"}, {"__nd__": [8.57458209991455, 8.917458534240723, 8.615861892700195, 8.947592735290527, 9.150687217712402, 9.114607810974121], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0], [1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0, 1000.0, -1000.0]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [0.0, 0.0], "dtype": "float32"}, {"__nd__": [1000000.0, 1000000.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.7505407333374023, -0.3861430287361145, -0.6433624029159546, -0.16102567315101624, 0.5094521045684814, 0.6235284805297852, 0.2767280340194702, 0.6001145839691162, -0.3495651185512543, -0.9580443501472473, -0.39065423607826233, 0.06170051917433739, -0.26071277260780334, -0.1058390736579895, 0.75935298204422, -0.07343397289514542, 0.5800105333328247, -0.8950671553611755, -0.28697293996810913, 0.217561736702919, -0.9667621850967407, -0.7958259582519531, 0.676210880279541, 0.9466354846954346, 0.36638063192367554, 0.12481476366519928, 0.05676053464412689, 0.3359435796737671, -0.6619093418121338, 0.8184264302253723, 0.1416284590959549, 0.9476752877235413, -0.7128618359565735], [0.41979414224624634, 0.5416284799575806, 0.19083896279335022, -0.4227618873119354, 0.837177574634552, 0.39536410570144653, 0.13691900670528412, 0.10313396900892258, 0.17946523427963257, -0.024754997342824936, 0.7408920526504517, 0.2351219654083252, -0.32596877217292786, 0.5692952275276184, -0.6527286767959595, 0.8436508178710938, 0.8085217475891113, -0.9812211394309998, -0.5648902058601379, 0.7527550458908081, 0.10559553653001785, -0.8483712673187256, -0.521876871585846, -0.918342113494873, -0.1805073767900467, -0.9992250204086304, -0.06053007394075394, -0.5298192501068115, -0.7958636283874512, 0.14100821316242218, 0.15252149105072021, -0.563164234161377, 0.08963765949010849], [-0.3226703703403473, -0.4760860800743103, -0.015490968711674213, 0.4434339106082916, 0.7092605233192444, -0.04958333820104599, 0.4453589618206024, -0.596625030040741, 0.9548522233963013, -0.3217482268810272, 0.9563261270523071, 0.4988960027694702, 0.2574514150619507, -0.3031671941280365, 0.1416967660188675, 0.33673006296157837, -0.40165379643440247, 0.13134068250656128, -0.6404589414596558, -0.5147150158882141, 0.42764025926589966, 0.43043777346611023, 0.895318865776062, 0.18750399351119995, -0.8266898989677429, -0.9498213529586792, -0.812433660030365, -0.6374578475952148, -0.5157468318939209, 0.6716054081916809, 0.6789132356643677, 0.7789430618286133, 0.3496408462524414], [-0.8392316699028015, 0.6158154010772705, -0.7805310487747192, -0.9600892066955566, -0.8189006447792053, -0.05814329907298088, -0.9683469533920288, 0.4007595479488373, 0.43290263414382935, -0.9039888978004456, 0.04081786423921585, -0.08258941769599915, -0.8070407509803772, -0.9032189249992371, -0.9877574443817139, 0.20637601613998413, 0.9614567756652832, -0.9358760714530945, -0.24936112761497498, 0.5529629588127136, 0.19030776619911194, -0.6474354267120361, -0.3844314217567444, 0.6159969568252563, -0.9681383967399597, 0.12680208683013916, -0.876761794090271, -0.42969152331352234, -0.8371933698654175, 0.6673523783683777, 0.20761825144290924, -0.053703952580690384, -0.07088260352611542], [0.5385181903839111, -0.5840027928352356, 0.7566282749176025, 0.07359349727630615, -0.3873867392539978, 0.055515680462121964, -0.6310357451438904, 0.012360325083136559, -0.30619746446609497, 0.6539679765701294, -0.32923173904418945, -0.690733015537262, 0.16175127029418945, -0.25316381454467773, -0.26770544052124023, -0.6988637447357178, 0.43762826919555664, -0.06277477741241455, 0.8769391775131226, 0.27882951498031616, -0.5799515843391418, 0.8037962317466736, 0.3496471643447876, 0.16867351531982422, 0.7852318286895752, 0.6819756627082825, 0.3896356523036957, 0.8999016284942627, 0.9163662195205688, -0.909777045249939, -0.10740967839956284, 0.06952743232250214, -0.43523672223091125], [0.0737534686923027, -0.8157840371131897, 0.9682271480560303, -0.04398616775870323, 0.5426096320152283, 0.60997074842453, -0.39921048283576965, -0.21904002130031586, 0.04915817454457283, 0.8155059814453125, -0.5668759346008301, 0.08454511314630508, -0.4113982319831848, 0.9002673029899597, 0.699276328086853, -0.12237624078989029, 0.7632943391799927, -0.8403105139732361, -0.028230885043740273, -0.6285238862037659, 0.24978551268577576, 0.12481878697872162, 0.2912121117115021, 0.8605814576148987, 0.3728196918964386, -0.36441248655319214, 0.7197651267051697, 0.005983783397823572, -0.547432541847229, 0.2525889277458191, 0.8564914464950562, -0.9403665065765381, 0.06890236586332321], [-0.45941615104675293, -0.7961384057998657, 0.8179749846458435, 0.05708109587430954, -0.26679179072380066, 0.43036529421806335, 0.9202623963356018, 0.5148720145225525, 0.9708818197250366, 0.028977036476135254, -0.8075171709060669, -0.5858475565910339, -0.2069372534751892, -0.9833365082740784, -0.7714450359344482, -0.12907753884792328, 0.25229597091674805, 0.08258914202451706, 0.4653471112251282, 0.7374529838562012, -0.24334688484668732, -0.8620262742042542, -0.6870360970497131, -0.15953591465950012, 0.8430514335632324, 0.44051727652549744, 0.11239471286535263, -0.5465379953384399, 0.018002785742282867, 0.43260979652404785, -0.580665111541748, 0.8518643975257874, 0.5020383596420288], [-0.07923638820648193, 0.095335952937603, -0.21282018721103668, 0.1107056587934494, 0.09419748187065125, -0.17218932509422302, -0.982779324054718, 0.5774667859077454, 0.013278761878609657, -0.7404553890228271, -0.8241903185844421, 0.13291972875595093, 0.1857576221227646, -0.1354711502790451, -0.6925979852676392, -0.2704196870326996, 0.0750860795378685, -0.6640423536300659, 0.9065564870834351, 0.3884713649749756, -0.5112218856811523, -0.93292236328125, 0.6533329486846924, 0.7651616930961609, -0.9094048142433167, -0.11135898530483246, -0.7106998562812805, -0.19874732196331024, 0.5455446839332581, -0.5018913149833679, -0.9817602634429932, 0.57338547706604, 0.8842650055885315], [0.9885910153388977, -0.5660126209259033, -0.6158179640769958, 0.3874953091144562, 0.43781012296676636, 0.8300905823707581, 0.6442558765411377, -0.3665597140789032, -0.22011932730674744, 0.7754546999931335, 0.034902408719062805, -0.36612367630004883, -0.8201137185096741, 0.18741931021213531, -0.6478703022003174, -0.3836245536804199, -0.25611013174057007, -0.26264187693595886, -0.9683293104171753, 0.29425957798957825, -0.06358593702316284, -0.5380248427391052, -0.5421617031097412, 0.9108781218528748, -0.8571158647537231, -0.045750632882118225, 0.47483959794044495, 0.7501427531242371, 0.23394139111042023, -0.16305045783519745, -0.9602875113487244, 0.666195273399353, 0.32204875349998474], [-0.5232874751091003, -0.9919192790985107, -0.23707769811153412, -0.8796188235282898, -0.7215272188186646, -0.7633627653121948, 0.4607541561126709, -0.2535790503025055, 0.3709828853607178, -0.007503905799239874, 0.9784257411956787, 0.1426679491996765, -0.19898030161857605, 0.17604392766952515, 0.893358051776886, 0.905187726020813, 0.734320342540741, -0.17008617520332336, 0.32106778025627136, -0.06590860337018967, 0.9143790006637573, -0.7971604466438293, -0.9898853302001953, -0.7544662952423096, -0.11410337686538696, -0.8863288760185242, 0.5386843085289001, -0.77019202709198, -0.19583484530448914, 0.8773916959762573, -0.4440866708755493, -0.7775866389274597, -0.9891604781150818], [0.665691077709198, -0.47048038244247437, 0.8037714958190918, 0.6974964141845703, -0.49437981843948364, -0.07635293900966644, -0.2950012683868408, -0.6510520577430725, 0.47327640652656555, 0.2821011245250702, 0.7320921421051025, -0.33652305603027344, -0.037596795707941055, 0.665864109992981, 0.6609153151512146, -0.010070232674479485, 0.13103757798671722, -0.9410799741744995, 0.08540138602256775, -0.4676271975040436, -0.170511856675148, -0.21033547818660736, -0.7451234459877014, 0.9644899368286133, 0.2703040838241577, -0.9814362525939941, -0.09684789925813675, -0.3993193209171295, -0.47913363575935364, 0.9299047589302063, -0.41524338722229004, 0.7977755665779114, -0.4612504243850708], [-0.6326491236686707, -0.10425642132759094, -0.16745521128177643, -0.30662912130355835, -0.048761092126369476, 0.4010504186153412, 0.0009815971134230494, 0.0959746316075325, 0.2906283140182495, 0.3176373541355133, 0.2390536516904831, -0.4530903398990631, -0.22173282504081726, 0.7004297971725464, 0.4640985131263733, -0.2523791790008545, 0.8494277596473694, -0.24452155828475952, 0.789959728717804, 0.9103550314903259, 0.7129727005958557, 0.22973370552062988, 0.5265756249427795, -0.4384424388408661, 0.47382259368896484, 0.7499991655349731, 0.5442702174186707, 0.29385966062545776, -0.6151174306869507, -0.9551044702529907, -0.6705570220947266, -0.9099290370941162, 0.16399367153644562], [-0.9428790807723999, 0.35526472330093384, -0.1539188027381897, -0.45328155159950256, 0.7975472807884216, 0.29103410243988037, -0.9199798703193665, 0.575396716594696, -0.06301313638687134, 0.05887242779135704, 0.7094944715499878, -0.1408340334892273, -0.038771022111177444, -0.810815691947937, 0.8448033332824707, 0.8153847455978394, -0.07427014410495758, -0.29324793815612793, -0.21366280317306519, 0.727696418762207, 0.985357403755188, -0.2667364776134491, -0.2218713015317917, 0.42338478565216064, -0.630113422870636, -0.9952833652496338, -0.5004892945289612, 0.4634925425052643, -0.7992364764213562, 0.9491075873374939, -0.5278091430664062, -0.05368053540587425, -0.5750692486763], [-0.7140000462532043, 0.2270537167787552, -0.9171552658081055, -0.8896859288215637, -0.12203274667263031, 0.42030951380729675, 0.9688442349433899, -0.3241058886051178, 0.8769899010658264, -0.3714666962623596, 0.11566109210252762, 0.10144282132387161, -0.8746554255485535, 0.23468273878097534, 0.18658427894115448, -0.5519115328788757, 0.024884091690182686, -0.907130241394043, 0.7141990661621094, -0.044756654649972916, -0.043050367385149, -0.8537610769271851, -0.251801073551178, -0.6337748765945435, 0.2566240131855011, 0.8897654414176941, 0.18070600926876068, 0.5414897203445435, -0.8405620455741882, 0.8989816904067993, 0.30670803785324097, 0.9614997506141663, 0.37714943289756775], [0.48440560698509216, 0.7109265923500061, -0.964295506477356, 0.031691037118434906, -0.5915333032608032, 0.3224765658378601, -0.6339662075042725, -0.18358105421066284, 0.33280837535858154, 0.3743511736392975, 0.025181036442518234, -0.9206588268280029, 0.305467426776886, -0.3282370865345001, 0.7030081152915955, 0.629389762878418, 0.2935459017753601, -0.01605284959077835, 0.27992022037506104, 0.177620992064476, 0.8495301604270935, -0.962937593460083, 0.27552682161331177, -0.34970736503601074, 0.27419841289520264, -0.204375758767128, 0.8804506063461304, -0.6091563701629639, -0.32831576466560364, 0.40526083111763, 0.3393721580505371, -0.2896779775619507, -0.7870032787322998], [0.645144522190094, 0.8816094994544983, 0.9307428598403931, 0.5059019923210144, 0.19191154837608337, -0.8163133859634399, -0.07376257330179214, 0.8236125707626343, 0.35099080204963684, 0.7623302936553955, -0.26720207929611206, -0.0887935683131218, -0.9227167963981628, -0.21965159475803375, -0.9131326079368591, -0.31287485361099243, 0.42230769991874695, -0.7113935351371765, -0.667346179485321, 0.19045650959014893, 0.25192269682884216, 0.23271922767162323, -0.2674863636493683, -0.20475830137729645, 0.20327350497245789, 0.582883358001709, 0.9596487283706665, -0.12563657760620117, -0.31658098101615906, -0.42694225907325745, -0.16050809621810913, -0.8274659514427185, -0.11975959688425064], [0.7871767282485962, -0.19184133410453796, -0.29964810609817505, 0.39890822768211365, 0.46245622634887695, 0.1499827653169632, 0.3790889382362366, -0.33077800273895264, 0.4063882827758789, 0.32498493790626526, 0.5897381901741028, -0.9248432517051697, 0.06293720752000809, -0.49625569581985474, 0.3137769103050232, 0.19559423625469208, -0.3000164031982422, 0.5560498833656311, 0.009722825139760971, -0.47781920433044434, -0.13496679067611694, -0.8571699857711792, 0.46248748898506165, 0.34913915395736694, -0.1425107717514038, 0.7045857906341553, -0.37786099314689636, -0.9946474432945251, 0.41056475043296814, 0.9633302688598633, 0.5698910355567932, -0.6334905028343201, 0.49584871530532837]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [-0.010781689547002316, -0.034748613834381104, 0.05790913850069046, -0.25891348719596863, 0.08081870526075363, 0.10247301310300827, 0.011906755156815052, -0.11002252250909805, -0.02136288955807686, -0.12783008813858032, 0.012750181369483471, 0.0828545093536377, -0.020549297332763672, -0.0017052822513505816, 0.015928268432617188, 0.014943349175155163, 0.07366073131561279], "dtype": "float32"}, {"__nd__": [0.33575353026390076, 0.31523045897483826, 0.32014331221580505, 0.3614519238471985, 0.28466975688934326, 0.30246224999427795, 0.34213247895240784, 0.31617608666419983, 0.3319312334060669, 0.41510337591171265, 0.315026193857193, 0.26591306924819946, 0.3526899218559265, 0.360904335975647, 0.2769645154476166, 0.3040148615837097, 0.2578897476196289], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(row_mean_var, json.loads(_TESTS), 'triton', *[0.0001, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")